# ARC X79 flash duck force submit

reh2 traces (madarshbb/arc-flash-reh2-c23fp, 31.0/60): all 34 verified attempts submitted exactly once; all 29 failed
attempts submitted ZERO times and ended on the 32k token budget -- no diff, no partial program, no candidate. X79:
  1. force: once an attempt has used force_submit_tokens with no submit, the last message tells it to submit its best
     transform NOW (the diff then drives the remaining ~10k tokens of repair, as in Tufa's duck harness);
  2. rescue: an attempt that reaches the budget still without a submit gets one last short turn (low effort,
     rescue_tokens) that must submit;
  3. speed: Adarsh's 251 tok/s rung (8 at once, gpu 0.93, mamba cache 40) first; reh2's 12-at-once rung (crashed,
     cost a restart) dropped; reh2's p1 (187 tok/s) next as fallback.
Everything else is reh2 (2 attempts, transpose second view, 5.5 h / 2 = 2.75 h for 60 tasks; Adarsh's run: 2.83 h).

Report line `C23F REPORT`: `score_agent` vs `score_agent_noforce` (same run, programs from the force / rescue turns dropped), `force` {ordered, rescue_turns, submitted_after, verified_after, best_ok_after, tasks_fixed, tasks_lost, gain}; then `REHEARSAL SCORE`. Refs: reh2 31.0/60 (187 tok/s), Adarsh's fast-rung run 32.0 agent (251 tok/s).

---

# ARC flash reh2p c23f -- Qwen3.8-Flash-Next + the python-first agent (whole submission)

**What runs.** The CPU DSL solver alone for the first ~22 min (memory: Flash-Next pins ~156 of the container's ~173 GiB
once loaded), while SGLang (Pennyroyal) installs and the weights preload; then Flash-Next on 4x L4 (expert parallel 4,
MTP drafter, FP8 KV -- X60 / X62's settings, tried in order) and the python-tool agent (stateful python session,
python-first instructions, exact facts under the grids, 12k-token turns) on every task, shortest first; a second
attempt for the unverified ones while time lasts. `submission.json` is valid from minute 0 and rewritten after every
attempt; the CPU solver's verified answers fill the slots the agent leaves empty.

**Modes.** `sub`: does nothing heavy on a normal save; the scored rerun runs it all (12 h - 10 min for the
agent). `reh`: runs on save on the 120 public-eval tasks for 5.5 h of agent time and prints the `C23F REPORT` and
`REHEARSAL SCORE` lines.

Built by `agent/build_flash_sub.py` (2026-10-28). Not submitted by Claude.

**c23fp (pruned).** At minute 0, beside the CPU solver, `prune_moe.py` keeps the 384 most-used of 512 experts per layer, ranked by X72's router counts (`madarshbb/arc-x72-flash-next-expert-calibration` output, attached), and writes the checkpoint to /tmp (~11 min). The pruned model then serves with expert parallelism + the MTP drafter and up to 12 agents at once. No counts, a failed prune or no pruned rung serving -> the full model (c23f) in the same run.

10-31: the drafter rungs patch SGLang so albucino's 512-expert drafter loads beside the 384-expert target (reh2 crashed there); p3 serves the pruned model without a drafter if that fails. The second attempt (only for tasks the first did not verify) sees the task transposed.

In [ ]:
# c23f: clocks, config and the fallback submission (written before anything heavy).
import os, json, time, glob
START = time.time()
FLX = json.loads(r"""{"agent_end_margin_min": 10, "agent_kill_after_s": 240, "reh_hours": 5.5, "reh_every": 2, "attempts": 2, "temperature": 0.6, "top_p": 0.95, "top_k": 20, "reasoning_effort": "xhigh", "kv_per_agent": 24000, "min_concurrency": 2, "max_concurrency": 8, "min_ok": 1, "cpu": true, "cpu_workers_max": 12, "cpu_budget_s": 300, "cpu_alone_min": 22, "cpu_loo_s": 30, "hang_s": 600, "watch_poll_s": 15, "max_restarts": 3, "hang_restart_rung": null, "agent": {"max_turns": 30, "max_gen_tokens": 32768, "context_budget": 30000, "turn_max_tokens": 12288, "nudge_every": 6, "retain_functions": true, "trim_drain": 8000, "keep_recent": 4, "degenerate_guard": true, "timeout_as_cut": true, "keep_reasoning": true, "keep_cut_reasoning": true, "stateful": true, "python_first": true, "facts": true, "cut_hint": "act", "token_est": "calibrated", "snippet_timeout": 30, "attempt_views": [null, "transpose"], "force_submit_tokens": 22528, "rescue_submit": true, "rescue_tokens": 8192}, "flash": {"backend": "sglang", "sglang_bundle_glob": "/kaggle/input/**/pennyroyal-v253", "model_glob": "/kaggle/input/**/intel-qwen3.8-flash-next-w4a16-autoround/**/config.json", "start_timeout_s": 1500, "preload_threads": 16, "triton_bench_cache_mb": 64, "answer_concurrency": 2, "sparse_fp8_patch": true, "serve_check": true, "stage_bundle": true, "fa2_wheel_glob": "/kaggle/input/**/flash_attn-2.8.3+cu130torch2.13-cp312-cp312-linux_x86_64.whl", "require_fa2": true, "sgl_env": {"SGLANG_FLASHINFER_WORKSPACE_SIZE": "33554432"}, "draft_glob": "/kaggle/input/**/albucino-qwen3-8-flash-next-drafter/**/config.json", "token_map_glob": "/kaggle/input/**/hot_tokens_64k.pt", "dflash_glob": null, "dflash_capture_patch": false, "marlin_sum_patch": true, "marlin_ep_patch": true, "rungs": [{"chunk": 2048, "max_prefill": 4096, "max_len": 98304, "max_seqs": 4, "gpu_util": 0.955, "mamba_cache": 6, "radix": false, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 4, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "m1_ep4_mtp3_m955_4x98k", "spec": {"steps": 3, "token_map": true}}, {"chunk": 2048, "max_prefill": 4096, "max_len": 131072, "max_seqs": 4, "gpu_util": 0.955, "mamba_cache": 12, "radix": true, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 4, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "a0_ep4_mtp3_radix_m955_4x131k", "spec": {"steps": 3, "token_map": true}, "schedule_policy": "lpm"}, {"chunk": 2048, "max_prefill": 4096, "max_len": 131072, "max_seqs": 6, "gpu_util": 0.955, "mamba_cache": 24, "radix": true, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 6, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "a1_ep4_radix_m955_6x131k", "schedule_policy": "lpm"}], "compare_rungs": [], "skip_disk_speed": true}, "run_on_save": true, "prune": {"keep": 384, "counts_glob": "/kaggle/input/**/flash_calib_counts.json", "out_roots": ["/tmp", "/kaggle/tmp"], "wait_min": 30, "max_concurrency": 12, "fallback_min_h": 3.0, "hang_restart_rung": "p2_ep4_mtp3_m955_8x98k", "flash_over": {"draft_map_patch": true}}, "pruned_rungs": [{"chunk": 2048, "max_prefill": 4096, "max_len": 131072, "max_seqs": 8, "gpu_util": 0.93, "mamba_cache": 40, "radix": true, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 8, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "p0_ep4_mtp3_radix_m93_8x131k", "spec": {"steps": 3, "token_map": true}, "schedule_policy": "lpm", "needs_draft_map_patch": true}, {"chunk": 2048, "max_prefill": 4096, "max_len": 131072, "max_seqs": 8, "gpu_util": 0.955, "mamba_cache": 24, "radix": true, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 8, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "p1_ep4_mtp3_radix_m955_8x131k", "spec": {"steps": 3, "token_map": true}, "schedule_policy": "lpm", "needs_draft_map_patch": true}, {"chunk": 2048, "max_prefill": 4096, "max_len": 131072, "max_seqs": 12, "gpu_util": 0.955, "mamba_cache": 36, "radix": true, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 12, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "p3_ep4_radix_m955_12x131k", "schedule_policy": "lpm"}, {"chunk": 2048, "max_prefill": 4096, "max_len": 98304, "max_seqs": 8, "gpu_util": 0.955, "mamba_cache": 10, "radix": false, "linear_backend": "triton", "needs_sparse_patch": true, "needs_marlin_patch": true, "args": ["--disable-prefill-cuda-graph", "--ep-size", "4"], "answer_concurrency": 8, "needs_marlin_ep_patch": true, "tool_parser": "qwen3_coder", "name": "p2_ep4_mtp3_m955_8x98k", "spec": {"steps": 3, "token_map": true}, "needs_draft_map_patch": true}]}""")
HYB = FLX                                            # the CPU solver cells (c22's) read their settings from HYB
RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
RUN = RERUN or FLX["run_on_save"]
if RERUN:
    CHALLENGES = glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)[0]
    SOLUTIONS = None
    AGENT_DEADLINE = START + 12 * 3600 - FLX["agent_end_margin_min"] * 60
else:
    CHALLENGES = glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True)[0]
    SOLUTIONS = glob.glob("/kaggle/input/**/arc-agi_evaluation_solutions.json", recursive=True)[0]
    _k = int(FLX.get("reh_every", 1))
    if _k > 1:                                       # a cheaper rehearsal: every k-th eval task, time scaled by 1 / k
        _all = json.load(open(CHALLENGES))
        json.dump({t: _all[t] for t in sorted(_all)[::_k]}, open("/kaggle/working/reh_challenges.json", "w"))
        CHALLENGES = "/kaggle/working/reh_challenges.json"
    AGENT_DEADLINE = START + FLX["reh_hours"] / _k * 3600 + FLX["cpu_alone_min"] * 60 + 15 * 60
TIMING = {"start": START, "rerun": RERUN, "run": RUN}
os.makedirs("/kaggle/working/agent", exist_ok=True)
ch = json.load(open(CHALLENGES))
json.dump({t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in task["test"]] for t, task in ch.items()},
          open("/kaggle/working/submission.json", "w"))
print(f"rerun={RERUN} run={RUN} tasks={len(ch)} agent until +{(AGENT_DEADLINE - START) / 3600:.2f} h; "
      "fallback submission written")

In [ ]:
# CPU DSL solver files (cpu_solver/ of our repo: ours + Hodel's arc-dsl (MIT) + nicholashuihk's solver (MIT-0)).
import os
os.makedirs("/kaggle/working/cpu_solver/hodel", exist_ok=True)


In [ ]:
%%writefile /kaggle/working/cpu_solver/common.py
"""Shared pieces of the CPU solver (2026-10-18): grids, cross-platform time limits, candidates.

A candidate is a program that reproduces EVERY train pair; it carries its outputs for the test inputs, the engine that
found it, a name, and a prior (lower = tried/ranked first). Engines never return unverified programs."""
import json
import os
import signal
import threading
import time
from contextlib import contextmanager

import numpy as np


class Timeout(Exception):
    pass


def A(g):
    return np.array(g, dtype=np.int16)


def L(g):
    return [[int(v) for v in row] for row in np.asarray(g)]


def T(g):
    """list / array -> tuple of tuples (Hodel's Grid type)."""
    return tuple(tuple(int(v) for v in row) for row in g)


def valid_grid(g):
    try:
        h = len(g)
        if not 1 <= h <= 30:
            return False
        w = len(g[0])
        return 1 <= w <= 30 and all(len(r) == w and all(isinstance(v, (int, np.integer)) and 0 <= v <= 9 for v in r)
                                    for r in g)
    except Exception:                                                  # noqa: BLE001
        return False


_ARMED = [None]          # generation of the time_limit block now running (None: none)
_FIRED = [None]          # generation whose timer went off (Windows path)
_GEN = [0]


def _on_signal(signum, frame):
    """SIGALRM (Linux) / simulated SIGINT (Windows): raise only for the block that is still running, so a late
    interrupt (timer firing just as its block ends) is ignored instead of escaping as a stray exception."""
    if _ARMED[0] is not None and (signum == getattr(signal, "SIGALRM", None) or _FIRED[0] == _ARMED[0]):
        raise Timeout()


@contextmanager
def time_limit(seconds):
    """Interrupt pure-Python work after `seconds` (raises Timeout). Main thread only; blocks do not nest (an inner
    block replaces the outer one's timer). Linux (Kaggle): SIGALRM interval timer. Windows (local bench): a timer
    thread simulates SIGINT in the main thread."""
    if seconds is None or seconds <= 0:
        raise Timeout()
    if threading.current_thread() is not threading.main_thread():
        yield                                         # no way to interrupt: run unbounded
        return
    _GEN[0] += 1
    gen = _GEN[0]
    if hasattr(signal, "setitimer"):
        old = signal.signal(signal.SIGALRM, _on_signal)
        _ARMED[0] = gen
        signal.setitimer(signal.ITIMER_REAL, seconds)
        try:
            yield
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            _ARMED[0] = None
            signal.signal(signal.SIGALRM, old)
    else:
        import _thread
        if signal.getsignal(signal.SIGINT) is not _on_signal:
            signal.signal(signal.SIGINT, _on_signal)
        done = threading.Event()

        def _fire():
            if not done.wait(seconds):
                _FIRED[0] = gen
                _thread.interrupt_main()
        _ARMED[0] = gen
        threading.Thread(target=_fire, daemon=True).start()
        try:
            yield
        finally:
            done.set()
            _ARMED[0] = None


class Clock:
    def __init__(self, seconds):
        self.t0 = time.time()
        self.seconds = seconds

    def left(self):
        return self.seconds - (time.time() - self.t0)

    def over(self):
        return self.left() <= 0


class Cand:
    __slots__ = ("engine", "name", "outs", "prior")

    def __init__(self, engine, name, outs, prior=1.0):
        self.engine, self.name, self.outs, self.prior = engine, name, outs, prior

    def __repr__(self):
        return f"Cand({self.engine}:{self.name}, prior={self.prior})"


def load_json(path):
    with open(path) as f:
        return json.load(f)


HERE = os.path.dirname(os.path.abspath(__file__))


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners.py
"""Engine 2: our rule learners (2026-10-18). Each learner fits a parametric rule family to the train pairs and returns
candidates only for parameter settings that reproduce EVERY train pair exactly.

Families are general ARC operations chosen where token models are weakest -- big grids with simple global rules -- and
from the categories the public evaluation set uses (dev half A + training as priors; half B held out):
  symmetry   complete a pattern that is symmetric under mirror / rotation / transpose / translation with the axis or
             center ANYWHERE (not only the grid center): fill the masked cells, return the grid or the masked block
  period     repair or complete a (possibly sheared) periodic tiling, globally or inside each framed box / cell
"""
from collections import Counter
from itertools import product

import numpy as np

from common import Cand


# ------------------------------------------------------------------------------------------------ basics
def A(g):
    return np.array(g, dtype=np.int16)


_BG = [None]           # row 74: forced background color during the bg0 retry (used when the grid has it)


def bg_color(g):
    if _BG[0] is not None and (g == _BG[0]).any():
        return _BG[0]
    v, c = np.unique(g, return_counts=True)
    return int(v[np.argmax(c)])


def eq(a, b):
    return a is not None and b is not None and a.shape == b.shape and bool((a == b).all())


def same_shape(train):
    return all(x.shape == y.shape for x, y in train)


def bbox(mask):
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return None
    return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1


_STRICT = [0]          # > 0 inside compositions / meta sub-tasks: every train pair must change (pre-row-69 rule)


def changed(train):
    """Learner guard (row 69): direct learning needs at least one train pair that changes (unchanged pairs then
    constrain the rule); inside compositions and meta sub-tasks every pair must change, as before."""
    return all((x != y).any() for x, y in train) if _STRICT[0] else any((x != y).any() for x, y in train)


def fit_all(train, tests, predict, name, engine="learn", prior=1.5):
    """predict(x) -> grid or None. Candidate iff it reproduces every train pair."""
    for x, y in train:
        try:
            p = predict(x)
        except Exception:                                              # noqa: BLE001
            return None
        if p is None and not _STRICT[0] and eq(x, y):
            continue                                                   # rule does not apply and nothing changes
        if not eq(p, y):
            return None
    outs = []
    for t in tests:
        try:
            p = predict(t)
        except Exception:                                              # noqa: BLE001
            p = None
        outs.append(p.tolist() if p is not None and 1 <= p.shape[0] <= 30 and 1 <= p.shape[1] <= 30 else None)
    if all(o is None for o in outs):
        return None
    return Cand(engine, name, outs, prior)


# ------------------------------------------------------------------------------------------------ symmetry
def _affine_maps(H, W, families):
    """Candidate symmetries as affine maps (name, M, t): (i, j) -> M @ (i, j) + t."""
    if "mirror" in families:
        for a in range(1, 2 * H - 2):
            yield f"mr{a}", (-1, 0, 0, 1), (a, 0)
        for b in range(1, 2 * W - 2):
            yield f"mc{b}", (1, 0, 0, -1), (0, b)
    if "rot180" in families:
        for a in range(1, 2 * H - 2):
            for b in range(1, 2 * W - 2):
                yield f"r180_{a}_{b}", (-1, 0, 0, -1), (a, b)
    if "diag" in families:
        for d in range(-(W - 1), H):
            yield f"tr{d}", (0, 1, 1, 0), (d, -d)
        for c in range(1, H + W - 2):
            yield f"atr{c}", (0, -1, -1, 0), (c, c)
    if "rot90" in families:
        for p in range(-(W - 1), H):
            for q in range(0, H + W - 1):
                yield f"r90_{p}_{q}", (0, 1, -1, 0), (p, q)
    if "shift" in families:
        for di in range(0, H):
            for dj in range(-(W - 1), W):
                if (di, dj) > (0, 0):
                    yield f"t{di}_{dj}", (1, 0, 0, 1), (di, dj)


def _maps(H, W, families):
    """(name, mi, mj): image coordinates of every cell under each candidate symmetry (debug / analysis helper)."""
    I, J = np.indices((H, W))
    for name, M, t in _affine_maps(H, W, families):
        yield name, M[0] * I + M[1] * J + t[0], M[2] * I + M[3] * J + t[1]


FAMILIES_ALL = ("mirror", "rot180", "diag", "rot90", "shift")


def find_symmetries(g, known, families=("mirror", "rot180", "diag", "rot90"), min_frac=0.35, window=None, min_ov=6):
    """Symmetries that hold EXACTLY on the known cells (inside `window` if given: a bool mask of the cells whose
    pairs are checked) and overlap at least min_frac of them. A 48-cell sample rejects most maps cheaply."""
    H, W = g.shape
    chk = known if window is None else known & window
    ki, kj = np.nonzero(chk)
    nk = len(ki)
    if nk == 0:
        return []
    rng = np.random.default_rng(0)
    sidx = rng.choice(nk, size=min(nk, 48), replace=False)
    si, sj = ki[sidx], kj[sidx]
    sv = g[si, sj]
    I, J = np.indices((H, W))
    out = []
    for name, M, t in _affine_maps(H, W, families):
        ti = M[0] * si + M[1] * sj + t[0]
        tj = M[2] * si + M[3] * sj + t[1]
        ok = (ti >= 0) & (ti < H) & (tj >= 0) & (tj < W)
        if ok.any():
            tic, tjc = ti[ok], tj[ok]
            kk = known[tic, tjc]
            if (g[tic[kk], tjc[kk]] != sv[ok][kk]).any():
                continue
        mi = M[0] * I + M[1] * J + t[0]
        mj = M[2] * I + M[3] * J + t[1]
        inb = (mi >= 0) & (mi < H) & (mj >= 0) & (mj < W)
        mic, mjc = np.where(inb, mi, 0), np.where(inb, mj, 0)
        both = inb & chk & known[mic, mjc]
        ov = int(both.sum())
        if ov < max(min_ov, min_frac * nk):
            continue
        if (g[both] == g[mic, mjc][both]).all():
            out.append((name, mic, mjc, inb))
    return out


def sym_fill(g, unknown, syms, rounds=8):
    g = g.copy()
    unk = unknown.copy()
    for _ in range(rounds):
        if not unk.any():
            break
        moved = False
        for _, mic, mjc, inb in syms:
            sel = unk & inb & ~unk[mic, mjc]
            if sel.any():
                g[sel] = g[mic[sel], mjc[sel]]
                unk[sel] = False
                moved = True
        if not moved:
            break
    return g, unk


def local_fill(g, unk, margin=6, max_rounds=4, min_agree=0.9, lead=0.25):
    """Fallback when the global symmetries leave cells unknown (2026-10-18): in a window around the remaining unknown
    block (bbox + margin), the map that best agrees on the known cell pairs and reaches unknown cells. Accepted if
    exact, or if it agrees on >= min_agree of >= max(30, 3 x unknown) pairs and leads the runner-up by >= lead
    (0934a4d8: exact transpose; f9d67f8b: transpose at 0.92, next 0.48)."""
    H, W = g.shape
    I, J = np.indices((H, W))
    for _ in range(max_rounds):
        if not unk.any():
            break
        r0, r1, c0, c1 = bbox(unk)
        win = np.zeros_like(unk)
        win[max(0, r0 - margin):r1 + margin, max(0, c0 - margin):c1 + margin] = True
        n_unk = int(unk.sum())
        scored = []
        for name, M, t in _affine_maps(H, W, FAMILIES_ALL):
            mi = M[0] * I + M[1] * J + t[0]
            mj = M[2] * I + M[3] * J + t[1]
            inb = (mi >= 0) & (mi < H) & (mj >= 0) & (mj < W)
            mic, mjc = np.where(inb, mi, 0), np.where(inb, mj, 0)
            reach = inb & unk & ~unk[mic, mjc]
            if not reach.any():
                continue
            both = inb & ~unk & ~unk[mic, mjc] & win
            ov = int(both.sum())
            if ov < max(30, 3 * n_unk):
                continue
            agree = float((g[both] == g[mic, mjc][both]).mean())
            scored.append((agree, int(reach.sum()), name, mic, mjc, reach))
        if not scored:
            break
        scored.sort(key=lambda r: (-r[0], -r[1]))
        best = scored[0]
        second = scored[1][0] if len(scored) > 1 else 0.0
        if not (best[0] == 1.0 or (best[0] >= min_agree and best[0] - second >= lead)):
            break
        _, _, _, mic, mjc, reach = best
        g = g.copy()
        g[reach] = g[mic[reach], mjc[reach]]
        unk = unk & ~reach
    return g, unk


_SYM_CACHE = {}


def _syms_cached(x, mask_color, families):
    key = (x.tobytes(), x.shape, mask_color, families)
    if key not in _SYM_CACHE:
        if len(_SYM_CACHE) > 64:
            _SYM_CACHE.clear()
        _SYM_CACHE[key] = find_symmetries(x, x != mask_color, families)
    return _SYM_CACHE[key]


def auto_mask(x):
    """The one color whose cells form a single solid rectangle of >= 4 cells (a mask block), else None."""
    from scipy import ndimage as ndi
    hits = []
    for c in np.unique(x):
        m = x == c
        lab, n = ndi.label(m)
        if n != 1 or m.sum() < 4:
            continue
        r0, r1, c0, c1 = bbox(m)
        if m[r0:r1, c0:c1].all() and m.sum() < x.size // 2:
            hits.append(int(c))
    return hits[0] if len(hits) == 1 else None


def _sym_predict(x, mask_color, mode, families, local=True, keep_left=False):
    """keep_left (2026-10-18, 'symmetrize'): the mask is the background; cells no symmetry reaches stay background.
    mask_color 'auto': the grid's single solid-rectangle color (per grid; masks vary between pairs)."""
    if mask_color == "auto":
        mask_color = auto_mask(x)
        if mask_color is None:
            return None
    unknown = x == mask_color
    if not unknown.any():
        return None
    syms = _syms_cached(x, mask_color, families)
    if not syms:
        return None
    g, unk = sym_fill(x, unknown, syms)
    if unk.any() and local and not keep_left:
        g, unk = local_fill(g, unk)
    if unk.any() and not keep_left:
        return None
    if keep_left and (g == x).all():
        return None
    if mode == "full":
        return g
    r0, r1, c0, c1 = bbox(unknown)
    return g[r0:r1, c0:c1]


def learn_symmetry(train, tests):
    same = same_shape(train)
    masks = []
    for c in range(10):
        ok = True
        for x, y in train:
            if not (x == c).any():
                ok = False
                break
            if same:
                d = x != y
                if not d.any() or not (x[d] == c).all():
                    ok = False
                    break
            else:
                bb = bbox(x == c)
                if (bb[1] - bb[0], bb[3] - bb[2]) != y.shape:
                    ok = False
                    break
        if ok:
            masks.append(c)
    found = []
    if not masks:                                  # 2026-10-18: a mask color that changes from pair to pair
        ok = True
        for x, y in train:
            c = auto_mask(x)
            if c is None or (same and not ((x != y).any() and (x[x != y] == c).all())) or (
                    not same and (lambda bb: (bb[1] - bb[0], bb[3] - bb[2]) != y.shape)(bbox(x == c))):
                ok = False
                break
        if ok:
            masks = ["auto"]
    for c in masks:
        for fam in (("mirror",), ("mirror", "rot180", "diag", "rot90"), ("shift",),
                    ("mirror", "rot180", "diag", "rot90", "shift")):
            cand = fit_all(train, tests, lambda x, c=c, fam=fam: _sym_predict(x, c, "full" if same else "crop", fam),
                           f"symmetry[mask={c},{'+'.join(fam)}]", prior=1.2)
            if cand:
                found.append(cand)
                break
    if not found and same:                        # symmetrize: background filled from its images, the rest stays
        bgs = {bg_color(x) for x, _ in train}
        for c in [b for b in bgs if b in masks]:
            for fam in (("mirror",), ("mirror", "rot180", "diag", "rot90")):
                cand = fit_all(train, tests, lambda x, c=c, fam=fam: _sym_predict(x, c, "full", fam, keep_left=True),
                               f"symmetrize[bg={c},{'+'.join(fam)}]~weak", prior=1.4)
                if cand:
                    found.append(cand)
                    break
    return found


# ------------------------------------------------------------------------------------------------ periodic tiling
def _tile_fit(r, known, max_tol, unseen=None):
    """Smallest lattice (pi, pj, shear s) whose majority-vote tiling disagrees with at most max_tol of the known
    cells of region r and has every class seen. Returns the tiled region or None."""
    H, W = r.shape
    I, J = np.indices((H, W))
    kn = int(known.sum())
    if kn == 0:
        return None
    best = None
    opts = []
    for pi in range(1, H + 1):
        for pj in range(1, W + 1):
            if pi * pj * 2 > H * W and not (pi == H and pj < W) and not (pj == W and pi < H):
                continue
            if pi == H and pj == W:
                continue
            for s in range(0, pj if pi < H else 1):
                opts.append((pi * pj, s, pi, pj))
    opts.sort()
    for _, s, pi, pj in opts:
        cls = (I % pi) * pj + (J - (I // pi) * s) % pj
        k = pi * pj
        cnt = np.bincount(cls[known] * 10 + r[known], minlength=k * 10).reshape(k, 10)
        seen = cnt.sum(1) > 0
        if not seen.all() and unseen is None:
            continue
        maj = np.where(seen, cnt.argmax(1), unseen if unseen is not None else 0)
        bad = kn - int(cnt.max(1).sum())
        if bad <= max_tol * kn:
            # the majority must be a real majority in every class (else the 'repair' is a coin flip)
            if (cnt.max(1) * 2 <= cnt.sum(1)).any() and bad > 0:
                continue
            best = maj[cls].astype(r.dtype)
            break
    return best


def box_interiors(g):
    """Interiors of single-color rectangular outlines (innermost first), as (r0, r1, c0, c1)."""
    from scipy import ndimage as ndi
    out = []
    H, W = g.shape
    for c in np.unique(g):
        lab, n = ndi.label(g == c)
        for k in range(1, n + 1):
            m = lab == k
            r0, r1, c0, c1 = bbox(m)
            if r1 - r0 < 3 or c1 - c0 < 3:
                continue
            ring = np.zeros_like(m)
            ring[r0, c0:c1] = ring[r1 - 1, c0:c1] = True
            ring[r0:r1, c0] = ring[r0:r1, c1 - 1] = True
            if not m[ring].all():
                continue
            inner = g[r0 + 1:r1 - 1, c0 + 1:c1 - 1]
            if (inner == c).all():
                continue
            out.append((r0 + 1, r1 - 1, c0 + 1, c1 - 1))
    # keep only regions that contain no other region (innermost)
    keep = [b for b in out if not any(o != b and b[0] <= o[0] and o[1] <= b[1] and b[2] <= o[2] and o[3] <= b[3]
                                      for o in out)]
    return keep


def sep_cells(g):
    """Cells between full uniform separator rows / columns."""
    H, W = g.shape
    bg = bg_color(g)
    rows = [i for i in range(H) if (g[i] == g[i, 0]).all() and g[i, 0] != bg]
    cols = [j for j in range(W) if (g[:, j] == g[0, j]).all() and g[0, j] != bg]
    cnt = Counter([int(g[i, 0]) for i in rows] + [int(g[0, j]) for j in cols])
    if not cnt:
        return []
    sc = cnt.most_common(1)[0][0]                       # separators: the color with the most full lines
    rows = [i for i in rows if g[i, 0] == sc]
    cols = [j for j in cols if g[0, j] == sc]
    rb = [-1] + rows + [H]
    cb = [-1] + cols + [W]
    out = []
    for a, b in zip(rb, rb[1:]):
        for c, d in zip(cb, cb[1:]):
            if b - a > 1 and d - c > 1:
                out.append((a + 1, b, c + 1, d))
    return out


def _period_predict(x, regions, mask_color, tol):
    if mask_color == "auto":
        mask_color = auto_mask(x)
        if mask_color is None:
            return None
    g = x.copy()
    regs = [(0, x.shape[0], 0, x.shape[1])] if regions == "grid" else (
        box_interiors(x) if regions == "boxes" else sep_cells(x))
    if not regs:
        return None
    changed = False
    for r0, r1, c0, c1 in regs:
        r = x[r0:r1, c0:c1]
        known = np.ones(r.shape, bool) if mask_color is None else r != mask_color
        t = _tile_fit(r, known, tol)
        if t is None:
            return None
        if not (t == r).all():
            changed = True
        g[r0:r1, c0:c1] = t
    return g if changed else None


def _period_crop(x, mask_color):
    """The masked rectangle's content under the grid's (possibly sheared) periodic tiling."""
    if mask_color == "auto":
        mask_color = auto_mask(x)
        if mask_color is None:
            return None
    unknown = x == mask_color
    bb = bbox(unknown)
    if bb is None:
        return None
    t = _tile_fit(x, ~unknown, 0.0)
    if t is None:
        return None
    return t[bb[0]:bb[1], bb[2]:bb[3]]


def learn_period_crop(train, tests):
    if same_shape(train):
        return []
    for c in list(range(10)) + ["auto"]:
        if c != "auto" and not all((x == c).any() and (lambda bb: (bb[1] - bb[0], bb[3] - bb[2]) == y.shape)(bbox(x == c))
                                   for x, y in train):
            continue
        cand = fit_all(train, tests, lambda x, c=c: _period_crop(x, c), f"period_crop[mask={c}]", prior=1.3)
        if cand:
            return [cand]
    return []


def _sym_repair(x, min_agree=0.85):
    """No mask: symmetries holding on >= min_agree of the cell pairs they map; every cell takes the majority over
    itself and its images (cells off-grid under a map do not vote)."""
    H, W = x.shape
    I, J = np.indices((H, W))
    acc = []
    for name, M, t in _affine_maps(H, W, ("mirror", "rot180", "diag", "rot90")):
        mi = M[0] * I + M[1] * J + t[0]
        mj = M[2] * I + M[3] * J + t[1]
        inb = (mi >= 0) & (mi < H) & (mj >= 0) & (mj < W)
        if inb.sum() < 0.6 * H * W:
            continue
        mic, mjc = np.where(inb, mi, 0), np.where(inb, mj, 0)
        agree = (x[inb] == x[mic, mjc][inb]).mean()
        if min_agree <= agree < 1.0:
            acc.append((mic, mjc, inb))
    if not acc:
        return None
    votes = np.zeros((H, W, 10), np.int16)
    np.put_along_axis(votes, x[..., None].astype(np.int64), 1, axis=2)
    for mic, mjc, inb in acc:
        v = np.where(inb, x[mic, mjc], -1)
        for c in range(10):
            votes[..., c] += (v == c)
    g = votes.argmax(2).astype(x.dtype)
    return None if (g == x).all() else g


def learn_sym_repair(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    for a in (0.85, 0.7):
        c = fit_all(train, tests, lambda x, a=a: _sym_repair(x, a), f"sym_repair[{a}]" + ("~weak" if a < 0.85 else ""),
                    prior=1.4 if a >= 0.85 else 1.9)
        if c:
            return [c]
    return []


def learn_period(train, tests):
    if not same_shape(train):
        return []
    found = []
    # mask colors: a color that every changed cell had in the input
    masks = [None]
    for c in range(10):
        if all((x != y).any() and (x[x != y] == c).all() for x, y in train):
            masks.append(c)
    if len(masks) == 1 and all(auto_mask(x) is not None and (x != y).any() and (x[x != y] == auto_mask(x)).all()
                               for x, y in train):
        masks.append("auto")                       # 2026-10-18: mask color changes between pairs
    for regions in ("grid", "boxes", "cells"):
        for m in masks:
            for tol in ((0.0,) if m is not None else (0.12, 0.25)):  # m may be 'auto' 
                cand = fit_all(train, tests, lambda x, regions=regions, m=m, tol=tol: _period_predict(x, regions, m, tol),
                               f"period[{regions},mask={m},tol={tol}]", prior=1.3)
                if cand:
                    found.append(cand)
                    return found
    return found


LEARNERS = [learn_symmetry, learn_period, learn_period_crop, learn_sym_repair]


def all_learners():
    import learners_obj
    import learners2
    import learners3
    import learners4
    import learners5
    import learners6
    return (LEARNERS + learners_obj.LEARNERS + learners2.LEARNERS + learners3.LEARNERS + learners4.LEARNERS +
            learners5.LEARNERS + learners6.LEARNERS)


def _run_learners(train, tests, clock, skip=()):
    found = []
    for fn in all_learners():
        if clock.over():
            break
        if fn.__name__ in skip:
            continue
        try:
            found += fn(train, tests)
        except Exception:                                              # noqa: BLE001
            continue
    return found


_D4C = {"id": lambda g: g, "r90": lambda g: np.rot90(g, -1), "r180": lambda g: np.rot90(g, 2),
        "r270": lambda g: np.rot90(g, 1), "fud": lambda g: g[::-1], "flr": lambda g: g[:, ::-1],
        "tr": lambda g: g.T, "atr": lambda g: np.rot90(g, 2).T}


def engine(task, clock):
    train = [(A(p["input"]), A(p["output"])) for p in task["train"]]
    tests = [A(t["input"]) for t in task["test"]]
    return engine_arrays(train, tests, clock, cnorm=True)


def engine_arrays(train, tests, clock, cnorm=True, skip=()):
    """The learn pipeline (_engine_core); when it has no strong answer and the most-common-color background is not
    black for some train input while every grid contains black, the whole pipeline again with background = 0
    (row 74)."""
    found = _engine_core(train, tests, clock, cnorm, skip)
    weak = lambda cs: all("~weak" in c.name for c in cs)
    if (found and not weak(found)) or _BG[0] is not None or clock.over():
        return found
    grids = [x for x, _ in train] + list(tests)
    if not all((g == 0).any() for g in grids) or all(bg_color(x) == 0 for x, _ in train):
        return found
    _BG[0] = 0
    try:
        bz = _engine_core(train, tests, clock, False, skip)
    except Exception:                                                  # noqa: BLE001
        bz = []
    finally:
        _BG[0] = None
    for c in bz:
        c.name = "bg0>" + c.name
        c.prior += 0.1
    return (bz + found) if bz and not weak(bz) else (found + bz)


def _engine_core(train, tests, clock, cnorm=True, skip=()):
    """Direct learners -> (same shape) per-cell / per-object meta-learners | (other) crop compositions -> (last, only
    when nothing fits) the whole pipeline again on color-normalised pairs (meta.learn_cnorm_full)."""
    weak = lambda cs: all("~weak" in c.name for c in cs)
    direct = [c for c in _run_learners(train, tests, clock, skip)]
    if direct and not weak(direct):
        return direct
    found = []                                     # 2026-10-18 (row 55): weak-only direct answers do not stop the search
    _STRICT[0] += 1
    try:
        if same_shape(train):                      # 2026-10-18: the same learners on per-cell / per-object sub-tasks
            import meta
            for fn in meta.META:
                if clock.over():
                    break
                try:
                    found += fn(train, tests, clock)
                except Exception:                                      # noqa: BLE001
                    continue
                if found:
                    break
        else:
            found = _compose_crops(train, tests, clock, skip)
            if not found:
                found = _compose_grow(train, tests, clock, skip)
    finally:
        _STRICT[0] -= 1
    if found and not weak(found):
        return found + direct
    found = found + direct
    if (found and not weak(found)) or not cnorm or clock.over():
        return found
    import meta                                    # 2026-10-18 (row 32): normalisation over the WHOLE pipeline
    try:
        cn = meta.learn_cnorm_full(train, tests, clock)
    except Exception:                                                  # noqa: BLE001
        cn = []
    return (cn + found) if cn and not weak(cn) else (found + cn)


def _compose_crops(train, tests, clock, skip=()):
    """Composition (2026-10-18): crop first (pre.py), then a rotation / reflection or any learner."""
    import pre
    found = []
    for pname, pf in pre.PRE.items():
        if clock.over():
            break
        try:
            tr2 = [(pf(x), y) for x, y in train]
            if any(p is None for p, _ in tr2):
                continue
            te2 = [pf(t) for t in tests]
        except Exception:                                              # noqa: BLE001
            continue
        for dn, df in _D4C.items():
            if all(df(p).shape == y.shape and (df(p) == y).all() for p, y in tr2) and all(t is not None for t in te2):
                found.append(Cand("learn", f"crop[{pname}]>{dn}", [df(t).tolist() for t in te2], prior=1.5))
                break
        if found and not all('~weak' in c.name for c in found):
            break
        if any(t is None for t in te2):
            continue
        if not all(p.shape == y.shape for p, y in tr2):
            # 2026-10-18: crop, then a shape-changing learner (block_up / kron / tile + cmap / extend)
            import learners4
            for fn in (learners4.learn_block_up, learners4.learn_kron, learners4.learn_tile_cmap, learners4.learn_extend):
                try:
                    for c in fn(tr2, te2):
                        c.name = f"crop[{pname}]>" + c.name
                        c.prior += 0.2
                        found.append(c)
                except Exception:                                      # noqa: BLE001
                    continue
            if found and not all('~weak' in c.name for c in found):
                break
            continue
        for c in _run_learners(tr2, te2, clock, skip):
            c.name = f"crop[{pname}]>" + c.name
            c.prior += 0.2
            found.append(c)
        if found and not all('~weak' in c.name for c in found):
            break
    return found


def _compose_grow(train, tests, clock, skip=()):
    """Growing tasks (2026-10-18): tile / mirror-tile / upscale the input to the output's shape first, then any
    same-shape learner."""
    ratios = {(y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) for x, y in train}
    if len(ratios) != 1:
        return []
    kh, kw = ratios.pop()
    if kh != int(kh) or kw != int(kw) or kh * kw < 2 or kh > 4 or kw > 4:
        return []
    kh, kw = int(kh), int(kw)

    def mtile(g):
        rows = []
        for i in range(kh):
            rows.append([(g[::-1] if i % 2 else g)[:, ::-1] if j % 2 else (g[::-1] if i % 2 else g) for j in range(kw)])
        return np.block(rows)
    grows = {"tile": lambda g: np.tile(g, (kh, kw)), "mtile": mtile,
             "up": lambda g: np.kron(g, np.ones((kh, kw), dtype=g.dtype))}
    found = []
    for gname, gf in grows.items():
        if clock.over():
            break
        tr2 = [(gf(x), y) for x, y in train]
        te2 = [gf(t) for t in tests]
        if any(p.shape != y.shape for p, y in tr2) or any(t.shape[0] > 30 or t.shape[1] > 30 for t in te2):
            continue
        for c in _run_learners(tr2, te2, clock, skip):
            c.name = f"grow[{gname}]>" + c.name
            c.prior += 0.2
            found.append(c)
        if found and not all('~weak' in c.name for c in found):
            break
    return found


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners_obj.py
"""Object- and pixel-level learners (2026-10-18). Same contract as learners.py: a candidate only when its rule
reproduces every train pair.

  obj_recolor  recolor each object by a learned table over ONE object feature (size rank, shape, holes, ...) or a pair
  obj_delete   remove the objects a learned feature table marks
  rays         lines from source pixels in a direction set, stopping at the border / the first obstacle / never
  connect      lines between same-colored pixels on one row / column / diagonal
  stamp        a learned pattern painted around every marker pixel (absolute colors, or the marker's own color)
  outline      box / outbox / bbox interior / neighbours of each object painted in a color learned per object color
  holes        enclosed background regions filled with a color learned from the enclosure, its size, or fixed
"""
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
from scipy import ndimage as ndi

from learners import changed, A, bbox, bg_color, eq, fit_all, same_shape

S4 = ndi.generate_binary_structure(2, 1)
S8 = np.ones((3, 3), bool)
DIRS8 = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
DIRSETS = {
    "U": [(-1, 0)], "D": [(1, 0)], "L": [(0, -1)], "R": [(0, 1)],
    "UL": [(-1, -1)], "UR": [(-1, 1)], "DL": [(1, -1)], "DR": [(1, 1)],
    "V": [(-1, 0), (1, 0)], "H": [(0, -1), (0, 1)], "orth": DIRS8[:4], "diag": DIRS8[4:], "all": DIRS8,
    "d1": [(-1, -1), (1, 1)], "d2": [(-1, 1), (1, -1)],
}


# ------------------------------------------------------------------------------------------------ objects + features
class Obj:
    __slots__ = ("mask", "sl", "cells", "color", "size", "h", "w", "f")

    def __init__(self, g, mask, sl):
        self.mask, self.sl = mask, sl
        sub = g[sl][mask[sl]]
        cnt = Counter(sub.tolist())
        self.color = cnt.most_common(1)[0][0]
        self.size = int(mask[sl].sum())
        self.h, self.w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
        self.f = None


def extract(g, bg, conn=4, multicolor=False):
    st = S4 if conn == 4 else S8
    objs = []
    if multicolor == "bgreg":                         # 2026-10-18: the background's own connected regions
        lab, n = ndi.label(g == bg, structure=st)
        for k, sl in enumerate(ndi.find_objects(lab), 1):
            if sl is not None:
                objs.append(Obj(g, lab == k, sl))
        return objs
    if multicolor:
        lab, n = ndi.label(g != bg, structure=st)
        for k, sl in enumerate(ndi.find_objects(lab), 1):
            if sl is not None:
                objs.append(Obj(g, lab == k, sl))
    else:
        for c in np.unique(g):
            if c == bg:
                continue
            lab, n = ndi.label(g == c, structure=st)
            for k, sl in enumerate(ndi.find_objects(lab), 1):
                if sl is not None:
                    objs.append(Obj(g, lab == k, sl))
    return objs


def _holes(m):
    inv = ~np.pad(m, 1)
    lab, n = ndi.label(inv, structure=S4)
    return n - 1


def _canon(m):
    best = None
    for k in range(4):
        for f in (False, True):
            r = np.rot90(m, k)
            r = r[:, ::-1] if f else r
            key = (r.shape, r.tobytes())
            best = key if best is None or key < best else best
    return best


def features(g, objs, bg):
    H, W = g.shape
    sizes = sorted({o.size for o in objs})
    shapes = Counter()
    colors = Counter(o.color for o in objs)
    for o in objs:
        m = o.mask[o.sl]
        o.f = {"shape": (m.shape, m.tobytes())}
        shapes[o.f["shape"]] += 1
    tops = sorted({o.sl[0].start for o in objs})
    lefts = sorted({o.sl[1].start for o in objs})
    for o in objs:
        m = o.mask[o.sl]
        f = o.f
        f["color"] = o.color
        f["size"] = o.size
        f["h"], f["w"] = o.h, o.w
        f["dims"] = (o.h, o.w)
        f["rect"] = o.size == o.h * o.w
        f["holes"] = _holes(m)
        f["has_hole"] = f["holes"] > 0
        f["touch"] = o.sl[0].start == 0 or o.sl[1].start == 0 or o.sl[0].stop == H or o.sl[1].stop == W
        f["sides"] = (o.sl[0].start == 0, o.sl[0].stop == H, o.sl[1].start == 0, o.sl[1].stop == W)
        f["corner_tl"] = bool(o.mask[0, 0])
        f["corner_br"] = bool(o.mask[H - 1, W - 1])
        f["rank_desc"] = len(sizes) - 1 - sizes.index(o.size)
        f["rank_asc"] = sizes.index(o.size)
        f["largest"] = o.size == sizes[-1]
        f["smallest"] = o.size == sizes[0]
        f["shape_n"] = shapes[f["shape"]]
        f["shape_unique"] = shapes[f["shape"]] == 1
        f["shape_d4"] = _canon(m)
        f["color_n"] = colors[o.color]
        f["color_unique"] = colors[o.color] == 1
        f["ncolors"] = len(np.unique(g[o.sl][m]))
        f["hsym"] = bool((m == m[:, ::-1]).all() and (g[o.sl] == g[o.sl][:, ::-1]).all())
        f["vsym"] = bool((m == m[::-1]).all() and (g[o.sl] == g[o.sl][::-1]).all())
        f["aspect"] = int(np.sign(o.h - o.w))
        f["row_rank"] = tops.index(o.sl[0].start)
        f["col_rank"] = lefts.index(o.sl[1].start)
        f["parity"] = o.size % 2
        f["single"] = o.size == 1
        dil = ndi.binary_dilation(o.mask, structure=S4) & ~o.mask
        f["nbr"] = frozenset(int(v) for v in np.unique(g[dil]) if v != bg)
        f["touch_other"] = len(f["nbr"]) > 0
    return objs


FEATS = ["color", "size", "largest", "smallest", "rank_desc", "rank_asc", "shape", "shape_d4", "holes", "has_hole",
         "rect", "touch", "dims", "h", "w", "shape_unique", "shape_n", "color_unique", "color_n", "ncolors", "hsym",
         "vsym", "aspect", "parity", "single", "nbr", "touch_other", "row_rank", "col_rank", "sides", "corner_tl",
         "corner_br"]
CONFIGS = [(4, False), (8, False), (4, True), (8, True), (4, "bgreg")]


def _objs(x, conn, multi):
    bg = bg_color(x)
    return features(x, extract(x, bg, conn, multi), bg), bg


# ------------------------------------------------------------------------------------------------ recolor / delete
def _table(samples, keys):
    """samples: [(feature dict, target)]. A consistent table key -> target, or None."""
    tab = {}
    for f, t in samples:
        k = tuple(f[a] for a in keys)
        if tab.setdefault(k, t) != t:
            return None
    return tab


def _table_learner(train, tests, label_fn, apply_fn, tag, max_pairs=40):
    """Generic: label_fn(x, y, objs, bg) -> per-object targets or None; apply_fn(x, objs, bg, targets) -> grid.
    Finds the simplest feature key (one feature, then pairs) whose table is consistent and covers the test objects."""
    found = []
    for conn, multi in CONFIGS:
        samples, ok = [], True
        for x, y in train:
            objs, bg = _objs(x, conn, multi)
            if not objs or len(objs) > 60:
                ok = False
                break
            tg = label_fn(x, y, objs, bg)
            if tg is None:
                ok = False
                break
            samples += [(o.f, t) for o, t in zip(objs, tg)]
        if not ok or len({t for _, t in samples}) < 2:
            continue
        test_objs = [_objs(t, conn, multi) for t in tests]
        # 2026-10-18: every consistent single feature votes for its test prediction (v1 took the first one that fit:
        # 4 of 17 wrong on training); pairs only when no single feature works. Top 2 distinct predictions returned.
        pool = FEATS[:12] + ["shape_n", "touch_other", "col_rank", "row_rank", "nbr"]   # 2026-10-18: wider pair pool
        for keysets in ([(a,) for a in FEATS], "binary", list(combinations(pool, 2))[:max(max_pairs, 140)]):
            if keysets == "binary":
                found = _binary_rule(train, tests, samples, conn, multi, apply_fn, tag)
                if found:
                    break
                continue
            votes = {}
            for keys in keysets:
                tab = _table(samples, keys)
                if tab is None or len(tab) == len(samples) and len(samples) > 3:
                    continue                             # one key per object = memorised, not a rule
                if not all(tuple(o.f[a] for a in keys) in tab for objs, _ in test_objs for o in objs):
                    continue

                def predict(x, keys=keys, tab=tab, conn=conn, multi=multi):
                    objs, bg = _objs(x, conn, multi)
                    tg = []
                    for o in objs:
                        k = tuple(o.f[a] for a in keys)
                        if k not in tab:
                            return None
                        tg.append(tab[k])
                    return apply_fn(x, objs, bg, tg)
                c = fit_all(train, tests, predict, f"{tag}[{conn}{'m' if multi else ''},{'+'.join(keys)}]",
                            prior=1.6 + 0.1 * (len(keys) - 1))
                if c:
                    v = votes.setdefault(str(c.outs), [0, c])
                    v[0] += 1
            if votes:
                ranked = sorted(votes.values(), key=lambda v: -v[0])
                pair = isinstance(keysets, list) and keysets and len(keysets[0]) == 2
                for n, c in ranked[:2]:
                    c.prior -= 0.05 * min(n, 6)
                    c.name += f"x{n}" + ("" if n >= 2 and not pair else "~weak")   # pair tables: always weak
                    found.append(c)
                break
        if found:
            break
        if found:
            break
    return found


def _binary_rule(train, tests, samples, conn, multi, apply_fn, tag):
    """2026-10-18: target A iff one feature == one value, every other value -> B (unseen values allowed)."""
    targets = {t for _, t in samples}
    if len(targets) != 2:
        return []
    for a in FEATS:
        for va in {f[a] for f, _ in samples}:
            ins = {t for f, t in samples if f[a] == va}
            outs = {t for f, t in samples if f[a] != va}
            if len(ins) != 1 or len(outs) != 1 or ins == outs:
                continue
            ta, tb = next(iter(ins)), next(iter(outs))

            def predict(x, a=a, va=va, ta=ta, tb=tb):
                objs, bg = _objs(x, conn, multi)
                return apply_fn(x, objs, bg, [ta if o.f[a] == va else tb for o in objs])
            c = fit_all(train, tests, predict, f"{tag}[{conn}{'m' if multi else ''},{a}=={str(va)[:12]}]", prior=1.65)
            if c:
                return [c]
    return []


def learn_obj_recolor(train, tests):
    if not same_shape(train):
        return []

    def label(x, y, objs, bg):
        cover = np.zeros(x.shape, bool)
        tg = []
        for o in objs:
            v = np.unique(y[o.mask])
            if len(v) != 1:
                return None
            tg.append(int(v[0]))
            cover |= o.mask
        if ((x != y) & ~cover).any():
            return None
        return tg

    def apply(x, objs, bg, tg):
        g = x.copy()
        for o, t in zip(objs, tg):
            g[o.mask] = t
        return g
    return _table_learner(train, tests, label, apply, "recolor")


def learn_obj_delete(train, tests):
    if not same_shape(train):
        return []

    def label(x, y, objs, bg):
        cover = np.zeros(x.shape, bool)
        tg = []
        for o in objs:
            if (y[o.mask] == x[o.mask]).all():
                tg.append(True)
            elif (y[o.mask] == bg).all():
                tg.append(False)
            else:
                return None
            cover |= o.mask
        if ((x != y) & ~cover).any():
            return None
        return tg

    def apply(x, objs, bg, tg):
        g = x.copy()
        for o, t in zip(objs, tg):
            if not t:
                g[o.mask] = bg
        return g
    return _table_learner(train, tests, label, apply, "delete")


# ------------------------------------------------------------------------------------------------ rays / connect
def _toward(x, i, j, target, dirs=None):
    H, W = x.shape
    out = []
    for di, dj in (dirs or DIRS8[:4]):
        a, b = i + di, j + dj
        while 0 <= a < H and 0 <= b < W:
            if x[a, b] == target:
                out.append((di, dj))
                break
            a += di
            b += dj
    return out


def _cast(g, x, bg, srcs, dirs, stop, color, target=None):
    H, W = x.shape
    for (i, j), c in srcs:
        k = c if color is None else color
        for di, dj in (dirs if target is None else _toward(x, i, j, target, dirs)):
            a, b = i + di, j + dj
            while 0 <= a < H and 0 <= b < W:
                if x[a, b] != bg:
                    if stop == "block":
                        break
                    if stop == "over":
                        g[a, b] = k
                else:
                    g[a, b] = k
                a += di
                b += dj
    return g


def _sources(x, bg, src):
    if src == "single":
        lab, n = ndi.label(x != bg, structure=S8)
        sizes = ndi.sum(np.ones_like(lab), lab, range(n + 1))
        pts = [(i, j) for i, j in zip(*np.nonzero((x != bg) & (sizes[lab] == 1)))]
    elif src == "all":
        pts = list(zip(*np.nonzero(x != bg)))
    else:
        pts = list(zip(*np.nonzero(x == src)))
    return [((int(i), int(j)), int(x[i, j])) for i, j in pts]


def learn_rays(train, tests):
    if not same_shape(train):
        return []
    for x, y in train:
        if not (x != y).any():
            return []
    bgs = {bg_color(x) for x, _ in train}
    if len(bgs) != 1:
        return []
    bg = bgs.pop()
    diff_cols = set()
    for x, y in train:
        diff_cols |= set(np.unique(y[x != y]).tolist())
    in_cols = set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg}
    srcs = ["single", "all"] + sorted(in_cols)
    colors = [None] + sorted(diff_cols) if len(diff_cols) == 1 else [None]
    found = []
    # 2026-10-18 (Hodel d43fd935-style): single pixels shoot only toward a target color, stopping at the first obstacle
    for target in sorted(in_cols):
        for dname in ("orth", "all"):
            for col in colors:
                def predict(x, target=target, dirs=DIRSETS[dname], col=col):
                    b = bg_color(x)
                    s = [p for p in _sources(x, b, "single") if p[1] != target]
                    if not s or len(s) > 60:
                        return None
                    return _cast(x.copy(), x, b, s, dirs, "block", col, target=target)
                c = fit_all(train, tests, predict, f"rays_toward[{target},{dname},{col}]", prior=1.5)
                if c:
                    return [c]
    for src in srcs:
        for dname, dirs in DIRSETS.items():
            for stop in ("block", "pass", "over"):
                for col in colors:
                    def predict(x, src=src, dirs=dirs, stop=stop, col=col):
                        b = bg_color(x)
                        s = _sources(x, b, src)
                        if not s or len(s) > 60:
                            return None
                        return _cast(x.copy(), x, b, s, dirs, stop, col)
                    c = fit_all(train, tests, predict, f"rays[{src},{dname},{stop},{col}]", prior=1.5)
                    if c:
                        found.append(c)
                        if len(found) >= 2:
                            return found
    return found


def _connect(x, bg, colors, axes, mode, color):
    g = x.copy()
    H, W = x.shape
    for c in colors:
        pts = list(zip(*np.nonzero(x == c)))
        if len(pts) > 40:
            return None
        k = c if color is None else color
        for (i1, j1), (i2, j2) in combinations(pts, 2):
            di, dj = i2 - i1, j2 - j1
            if di == 0 and "h" in axes or dj == 0 and "v" in axes or abs(di) == abs(dj) and "d" in axes:
                n = max(abs(di), abs(dj))
                if n < 2:
                    continue
                si, sj = np.sign(di), np.sign(dj)
                cells = [(i1 + si * t, j1 + sj * t) for t in range(1, n)]
                if mode == "clear" and any(x[a, b] != bg for a, b in cells):
                    continue
                for a, b in cells:
                    if x[a, b] == bg or mode == "over":
                        g[a, b] = k
    return g


def learn_connect(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    bg = bg_color(train[0][0])
    diff_cols = set()
    for x, y in train:
        diff_cols |= set(np.unique(y[x != y]).tolist())
    in_cols = sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg})
    color_sets = [[c] for c in in_cols] + ([in_cols] if len(in_cols) > 1 else []) + [None]
    found = []
    for cs in color_sets:
        for axes in ("hv", "h", "v", "d", "hvd"):
            for mode in ("clear", "under", "over"):
                for col in [None] + (sorted(diff_cols) if len(diff_cols) == 1 else []):
                    def predict(x, cs=cs, axes=axes, mode=mode, col=col):
                        b = bg_color(x)
                        cols = cs if cs is not None else [int(v) for v in np.unique(x) if v != b]
                        return _connect(x, b, cols, axes, mode, col)
                    c = fit_all(train, tests, predict, f"connect[{cs},{axes},{mode},{col}]", prior=1.5)
                    if c:
                        found.append(c)
                        return found
    return found


# ------------------------------------------------------------------------------------------------ stamp
def learn_stamp(train, tests):
    """Every pixel of a marker color gets the same (2r+1)^2 pattern around it (cells that were background)."""
    if not same_shape(train) or not changed(train):
        return []
    bg = bg_color(train[0][0])
    found = []
    cols = sorted(set.union(*[set(np.unique(x[x != y]).tolist()) | set(np.unique(x).tolist()) for x, y in train]) - {bg})
    for rel in (False, True):
        for r in (1, 2, 3):
            pats = {}
            ok = True
            for x, y in train:
                H, W = x.shape
                for i, j in zip(*np.nonzero(x != bg)):
                    c = int(x[i, j])
                    key = "any" if rel else c
                    P = pats.setdefault(key, {})
                    for di in range(-r, r + 1):
                        for dj in range(-r, r + 1):
                            a, b = i + di, j + dj
                            if (di or dj) and 0 <= a < H and 0 <= b < W and x[a, b] == bg:
                                v = int(y[a, b])
                                v = ("src" if v == c else v) if rel else v
                                if P.setdefault((di, dj), v) != v:
                                    P[(di, dj)] = "X"
            for key, P in pats.items():
                for k in list(P):
                    if P[k] == "X":
                        ok = False
            if not ok:
                continue

            def predict(x, pats=pats, r=r, rel=rel):
                b = bg_color(x)
                g = x.copy()
                H, W = x.shape
                for i, j in zip(*np.nonzero(x != b)):
                    c = int(x[i, j])
                    P = pats.get("any" if rel else c)
                    if P is None:
                        return None
                    for (di, dj), v in P.items():
                        a, bb = i + di, j + dj
                        if 0 <= a < H and 0 <= bb < W and x[a, bb] == b:
                            g[a, bb] = c if v == "src" else v
                return g
            cand = fit_all(train, tests, predict, f"stamp[r={r},rel={rel}]", prior=1.5)
            if cand:
                found.append(cand)
                return found
    return found


# ------------------------------------------------------------------------------------------------ outline / holes
def _ring(o, kind, H, W):
    m = np.zeros((H, W), bool)
    r0, r1, c0, c1 = o.sl[0].start, o.sl[0].stop, o.sl[1].start, o.sl[1].stop
    if kind == "box":
        m[r0:r1, c0:c1] = True
        m[r0 + 1:r1 - 1, c0 + 1:c1 - 1] = False
    elif kind == "outbox":
        a, b, c, d = max(0, r0 - 1), min(H, r1 + 1), max(0, c0 - 1), min(W, c1 + 1)
        m[a:b, c:d] = True
        m[r0:r1, c0:c1] = False
    elif kind == "delta":
        m[r0:r1, c0:c1] = True
        m &= ~o.mask
    elif kind == "nbr4":
        m = ndi.binary_dilation(o.mask, structure=S4) & ~o.mask
    elif kind == "nbr8":
        m = ndi.binary_dilation(o.mask, structure=S8) & ~o.mask
    elif kind == "corners":                          # 2026-10-18, from Hodel's patch vocabulary
        for a, b in ((r0, c0), (r0, c1 - 1), (r1 - 1, c0), (r1 - 1, c1 - 1)):
            m[a, b] = True
    elif kind == "inbox":
        if r1 - r0 >= 3 and c1 - c0 >= 3:
            m[r0 + 1:r1 - 1, c0 + 1:c1 - 1] = True
            m[r0 + 2:r1 - 2, c0 + 2:c1 - 2] = False
    elif kind == "backdrop":
        m[r0:r1, c0:c1] = True
    elif kind == "inbr":
        m = ndi.binary_dilation(o.mask, structure=S8) & ~ndi.binary_dilation(o.mask, structure=S4)
    elif kind in ("hfront", "vfront", "cross"):
        ci, cj = (r0 + r1 - 1) // 2, (c0 + c1 - 1) // 2
        if kind in ("hfront", "cross"):
            m[ci, :] = True
        if kind in ("vfront", "cross"):
            m[:, cj] = True
        m &= ~o.mask
    return m


def learn_outline(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    found = []
    for conn, multi in CONFIGS[:3]:
        for kind in ("box", "outbox", "delta", "nbr4", "nbr8", "corners", "inbox", "backdrop", "inbr", "hfront",
                     "vfront", "cross"):
            for under in (True, False):
                tab, ok = {}, True
                for x, y in train:
                    objs, bg = _objs(x, conn, multi)
                    for o in objs:
                        m = _ring(o, kind, *x.shape)
                        if under:
                            m &= x == bg
                        v = np.unique(y[m])
                        if len(v) > 1:
                            ok = False
                            break
                        if len(v) == 1 and tab.setdefault(o.color, int(v[0])) != int(v[0]):
                            ok = False
                            break
                    if not ok:
                        break
                if not ok or not tab:
                    continue

                def predict(x, conn=conn, multi=multi, kind=kind, under=under, tab=tab):
                    objs, bg = _objs(x, conn, multi)
                    g = x.copy()
                    for o in objs:
                        if o.color not in tab:
                            return None
                        m = _ring(o, kind, *x.shape)
                        if under:
                            m &= x == bg
                        g[m] = tab[o.color]
                    return g
                cand = fit_all(train, tests, predict, f"outline[{conn}{'m' if multi else ''},{kind},under={under}]",
                               prior=1.5)
                if cand:
                    found.append(cand)
    # 2026-10-18: every fitting configuration votes; the two most supported distinct test grids are returned
    votes = {}
    for c in found:
        votes.setdefault(str(c.outs), [0, c])[0] += 1
    ranked = sorted(votes.values(), key=lambda v: -v[0])
    out = []
    for n, c in ranked[:2]:
        c.name += f"x{n}" + ("" if n >= 2 or len(ranked) == 1 else "~weak")
        c.prior -= 0.02 * min(n, 5)
        out.append(c)
    return out


def _enclosed(x, bg, conn):
    lab, n = ndi.label(x == bg, structure=S4 if conn == 4 else S8)
    H, W = x.shape
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
    return [(lab == k) for k in range(1, n + 1) if k not in border]


def learn_holes(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    found = []
    for conn in (4, 8):
        for keyname in ("fixed", "encl_color", "size"):
            tab, ok = {}, True
            for x, y in train:
                bg = bg_color(x)
                cover = np.zeros(x.shape, bool)
                for m in _enclosed(x, bg, conn):
                    v = np.unique(y[m])
                    if len(v) != 1:
                        ok = False
                        break
                    cover |= m
                    if keyname == "fixed":
                        k = 0
                    elif keyname == "size":
                        k = int(m.sum())
                    else:
                        ring = np.unique(x[ndi.binary_dilation(m, structure=S4) & ~m])
                        k = int(ring[0]) if len(ring) == 1 else -1
                    if tab.setdefault(k, int(v[0])) != int(v[0]):
                        ok = False
                        break
                if not ok or ((x != y) & ~cover).any():
                    ok = False
                    break
            if not ok or not tab:
                continue

            def predict(x, conn=conn, keyname=keyname, tab=tab):
                bg = bg_color(x)
                g = x.copy()
                for m in _enclosed(x, bg, conn):
                    if keyname == "fixed":
                        k = 0
                    elif keyname == "size":
                        k = int(m.sum())
                    else:
                        ring = np.unique(x[ndi.binary_dilation(m, structure=S4) & ~m])
                        k = int(ring[0]) if len(ring) == 1 else -1
                    if k not in tab:
                        return None
                    g[m] = tab[k]
                return g
            cand = fit_all(train, tests, predict, f"holes[{conn},{keyname}]", prior=1.4)
            if cand:
                found.append(cand)
                return found
    return found


LEARNERS = [learn_obj_recolor, learn_obj_delete, learn_holes, learn_outline, learn_stamp, learn_connect, learn_rays]


def _gravitate(x, pairing, drop_unmatched, cmap=None, through=False, dirs=None):
    """Single pixels slide (row or column, whichever is aligned) toward the nearest larger object of their target
    color (same color, or cmap[color]) until adjacent; pixels with no aligned target are erased or kept."""
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S4)
    sizes = ndi.sum(np.ones_like(lab), lab, range(n + 1))
    g = x.copy()
    big = (sizes[lab] > 1) & (x != bg)
    moved = False
    H, W = x.shape
    for i, j in zip(*np.nonzero((x != bg) & (sizes[lab] == 1))):
        c = int(x[i, j])
        t = c if pairing == "same" else (cmap or {}).get(c)
        best = None
        for di, dj in (dirs or DIRS8[:4]):
            a, b, k = i + di, j + dj, 1
            while 0 <= a < H and 0 <= b < W:
                if big[a, b] and (t is None or x[a, b] == t):
                    if best is None or k < best[0]:
                        best = (k, di, dj)
                    break
                if x[a, b] != bg and not through:
                    break
                a, b, k = a + di, b + dj, k + 1
        g[i, j] = bg
        if best is None:
            if not drop_unmatched:
                g[i, j] = c
            moved |= drop_unmatched
            continue
        k, di, dj = best
        g[i + di * (k - 1), j + dj * (k - 1)] = c
        moved |= k > 1
    return g if moved else None


def learn_gravitate(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    found = []
    for dirs, dn in ((DIRS8[:4], "orth"), (DIRS8, "all8")):
        for through in (False, True):
            for drop in (True, False):
                c = fit_all(train, tests, lambda x, drop=drop, th=through, dirs=dirs: _gravitate(x, "same", drop, through=th,
                                                                                              dirs=dirs),
                            f"gravitate[same,drop={drop},through={through},{dn}]", prior=1.5)
                if c:
                    return [c]
    # a color map single -> target object color, learned from which object each moved pixel ends next to
    cm = {}
    for x, y in train:
        bg = bg_color(x)
        for i, j in zip(*np.nonzero((x != bg) & (y == bg))):
            c = int(x[i, j])
            ys, xs = np.nonzero((y == c) & (x == bg))
            for a, b in zip(ys, xs):
                for di, dj in DIRS8[:4]:
                    p, q = a + di, b + dj
                    if 0 <= p < x.shape[0] and 0 <= q < x.shape[1] and x[p, q] not in (bg, c):
                        cm.setdefault(c, int(x[p, q]))
    if cm:
        for through in (False, True):
            for drop in (True, False):
                c = fit_all(train, tests, lambda x, drop=drop, th=through: _gravitate(x, "map", drop, cm, th, DIRS8),
                            f"gravitate[map,drop={drop},through={through}]~weak", prior=1.8)
                if c:
                    return [c]
    return found


LEARNERS.append(learn_gravitate)


def _auto_marker(x):
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S8)
    sizes = ndi.sum(np.ones_like(lab), lab, range(n + 1))
    singles = {int(x[i, j]) for i, j in zip(*np.nonzero((x != bg) & (sizes[lab] == 1)))}
    multi = set()
    for k in range(1, n + 1):
        m = lab == k
        if sizes[k] > 1 and len(np.unique(x[m])) >= 2:
            multi |= set(np.unique(x[m]).tolist())
    c = singles & multi
    return c.pop() if len(c) == 1 else None


def _align(x, marker, axis):
    if marker == "auto":
        marker = _auto_marker(x)
        if marker is None:
            return None
    if axis == "auto":
        a = _align(x, marker, 1)
        return a if a is not None else _align(x, marker, 0)
    """Objects (8-connected, >= 2 colors) holding a `marker` cell shift along `axis` (1: columns, 0: rows) so the
    marker sits on the line of the lone marker cell (a marker that is its own object)."""
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S8)
    objs, anchors = [], []
    for k, sl in enumerate(ndi.find_objects(lab), 1):
        m = lab == k
        if m.sum() == 1 and (x[m] == marker).all():
            anchors.append(np.argwhere(m)[0])
        elif (x[m] == marker).any() and len(np.unique(x[m])) >= 2:
            objs.append(m)
    if len(anchors) != 1 or not objs:
        return None
    a = anchors[0][axis]
    g = x.copy()
    for m in objs:
        g[m] = bg
    moved = False
    for m in objs:
        cells = np.argwhere(m)
        mk = np.argwhere(m & (x == marker))
        if len(mk) != 1:
            return None
        d = a - mk[0][axis]
        moved |= d != 0
        for (i, j) in cells:
            ni, nj = (i, j + d) if axis == 1 else (i + d, j)
            if not (0 <= ni < x.shape[0] and 0 <= nj < x.shape[1]):
                return None
            g[ni, nj] = x[i, j]
    return g if moved else None


def learn_align(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    cols = sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]))
    for marker in cols + ["auto"]:
        for axis in (1, 0, "auto"):
            c = fit_all(train, tests, lambda x, mk=marker, ax=axis: _align(x, mk, ax), f"align[marker={marker},axis={axis}]",
                        prior=1.5)
            if c:
                return [c]
    return []


LEARNERS.append(learn_align)


def _align_multi(x):
    """Every lone single-cell anchor on the grid border defines a line (left / right border: its row; top / bottom:
    its column); each multicolor object holding exactly one cell of an anchor's color shifts along that line's
    normal so the cell lands on the line."""
    bg = bg_color(x)
    H, W = x.shape
    lab, n = ndi.label(x != bg, structure=S8)
    sizes = ndi.sum(np.ones_like(lab), lab, range(n + 1))
    anchors = {}
    for i, j in zip(*np.nonzero((x != bg) & (sizes[lab] == 1))):
        c = int(x[i, j])
        if j in (0, W - 1):
            ax = 0
        elif i in (0, H - 1):
            ax = 1
        else:
            continue
        if c in anchors:
            return None
        anchors[c] = (ax, int(i) if ax == 0 else int(j), lab[i, j])
    if not anchors:
        return None
    g = x.copy()
    moves = []
    for k in range(1, n + 1):
        if any(k == a[2] for a in anchors.values()):
            continue
        m = lab == k
        hits = [(c, np.argwhere(m & (x == c))) for c in anchors if (m & (x == c)).sum() == 1]
        if len(hits) != 1:
            continue
        c, pos = hits[0]
        ax, line, _ = anchors[c]
        d = line - pos[0][ax]
        if d:
            moves.append((m, ax, d))
    if not moves:
        return None
    for m, _, _ in moves:
        g[m] = bg
    for m, ax, d in moves:
        for i, j in np.argwhere(m):
            ni, nj = (i + d, j) if ax == 0 else (i, j + d)
            if not (0 <= ni < H and 0 <= nj < W):
                return None
            g[ni, nj] = x[i, j]
    return g


def learn_align_multi(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    c = fit_all(train, tests, _align_multi, "align_multi", prior=1.5)
    return [c] if c else []


LEARNERS.append(learn_align_multi)


def _walls(x, bg):
    """Full rows / columns of one non-background color: {(side, color)} with side in U, D, L, R (border lines only)."""
    H, W = x.shape
    out = {}
    for side, line in (("U", x[0]), ("D", x[-1]), ("L", x[:, 0]), ("R", x[:, -1])):
        if (line == line[0]).all() and line[0] != bg:
            out[side] = int(line[0])
    return out


SIDE_DIR = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}


def _obj_gravity(x, mode, unit, static_color=None):
    """Objects (unit: 'cell' or 4-connected multicolor 'obj') slide until blocked, nearest to the destination first.
    mode: one of U/D/L/R; 'wall' = toward the single border wall line; 'own' = toward the border wall of the object's
    own color (objects with no such wall stay)."""
    bg = bg_color(x)
    H, W = x.shape
    walls = _walls(x, bg)
    static = np.zeros(x.shape, bool)
    for side in walls:
        if side == "U":
            static[0] = True
        elif side == "D":
            static[-1] = True
        elif side == "L":
            static[:, 0] = True
        else:
            static[:, -1] = True
    if static_color is not None:
        static |= x == static_color
    movable = (x != bg) & ~static
    if not movable.any():
        return None
    if unit == "cell":
        pieces = [np.array([[i, j]]) for i, j in zip(*np.nonzero(movable))]
    elif unit in ("obj8c", "obj4c"):                    # 8- / 4-connected, one color
        pieces = []
        for c in np.unique(x[movable]):
            lab, n = ndi.label(movable & (x == c), structure=S8 if unit == "obj8c" else S4)
            pieces += [np.argwhere(lab == k) for k in range(1, n + 1)]
    else:
        lab, n = ndi.label(movable, structure=S4)
        pieces = [np.argwhere(lab == k) for k in range(1, n + 1)]

    def direction(cells):
        if mode in SIDE_DIR:
            return SIDE_DIR[mode]
        if mode == "wall":
            return SIDE_DIR[next(iter(walls))] if len(walls) == 1 else None
        c = int(x[cells[0][0], cells[0][1]])
        sides = [s for s, wc in walls.items() if wc == c]
        return SIDE_DIR[sides[0]] if len(sides) == 1 else (0, 0)
    dirs = [direction(p) for p in pieces]
    if any(d is None for d in dirs):
        return None
    g = x.copy()
    occ = (x != bg).copy()
    for p in pieces:
        occ[p[:, 0], p[:, 1]] = False
        g[p[:, 0], p[:, 1]] = bg
    order = sorted(range(len(pieces)), key=lambda k: -max(
        (pieces[k][:, 0] * dirs[k][0] + pieces[k][:, 1] * dirs[k][1]).tolist()))
    moved = False
    for k in order:
        p, (di, dj) = pieces[k], dirs[k]
        cols = x[p[:, 0], p[:, 1]]
        s = 0
        while True:
            ni, nj = p[:, 0] + (s + 1) * di, p[:, 1] + (s + 1) * dj
            if (di, dj) == (0, 0) or (ni < 0).any() or (nj < 0).any() or (ni >= H).any() or (nj >= W).any() or occ[ni, nj].any():
                break
            s += 1
        ni, nj = p[:, 0] + s * di, p[:, 1] + s * dj
        g[ni, nj] = cols
        occ[ni, nj] = True
        moved |= s > 0
    return g if moved else None


def learn_obj_gravity(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    statics = [None] + sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg_color(train[0][0])})
    votes = {}
    for mode in ("own", "wall", "U", "D", "L", "R"):
        for unit in ("obj", "cell", "obj8c", "obj4c"):
            for sc in statics:
                c = fit_all(train, tests, lambda x, m=mode, u=unit, sc=sc: _obj_gravity(x, m, u, sc),
                            f"obj_gravity[{mode},{unit},static={sc}]", prior=1.5)
                if c:
                    votes.setdefault(str(c.outs), [0, c])[0] += 1
    out = []
    ranked = sorted(votes.values(), key=lambda v: -v[0])
    for n, c in ranked[:2]:
        c.name += f"x{n}"
        out.append(c)
    return out


LEARNERS.append(learn_obj_gravity)


def _corner_rect(x):
    bg = bg_color(x)
    H, W = x.shape
    pts = list(zip(*np.nonzero(x != bg)))
    if not pts or len(pts) > 6:
        return None
    g = x.copy()
    for i, j in pts:
        ci = 0 if i < H - 1 - i else H - 1
        cj = 0 if j < W - 1 - j else W - 1
        if i == H - 1 - i or j == W - 1 - j:
            return None                              # no nearest corner
        g[min(i, ci):max(i, ci) + 1, min(j, cj):max(j, cj) + 1] = x[i, j]
    return g


def learn_corner_rect(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    c = fit_all(train, tests, _corner_rect, "corner_rect", prior=1.6)
    return [c] if c else []


def _wall_line(x, bg):
    H, W = x.shape
    rows = [i for i in range(H) if (x[i] == x[i, 0]).all() and x[i, 0] != bg]
    cols = [j for j in range(W) if (x[:, j] == x[0, j]).all() and x[0, j] != bg]
    if len(rows) + len(cols) != 1:
        return None
    return ("row", rows[0]) if rows else ("col", cols[0])


def _rays_wall(x, modes):
    bg = bg_color(x)
    wl = _wall_line(x, bg)
    if wl is None:
        return None
    kind, k = wl
    g = x.copy()
    H, W = x.shape
    changed = False
    for i, j in zip(*np.nonzero(x != bg)):
        if (kind == "row" and i == k) or (kind == "col" and j == k):
            continue
        c = int(x[i, j])
        m = modes.get(c)
        if m is None:
            return None
        if m == "none":
            continue
        pos = i if kind == "row" else j
        toward = 1 if k > pos else -1
        d = toward if m == "toward" else -toward
        di, dj = (d, 0) if kind == "row" else (0, d)
        a, b = i + di, j + dj
        while 0 <= a < H and 0 <= b < W and x[a, b] == bg:
            g[a, b] = c
            changed = True
            a, b = a + di, b + dj
    return g if changed else None


def learn_rays_wall(train, tests):
    """One full wall line; every pixel shoots perpendicular to it -- toward it (stopping at it) or away from it (to
    the border) -- with the behaviour learned per color."""
    if not same_shape(train) or not changed(train):
        return []
    bg = bg_color(train[0][0])
    cols = sorted(set.union(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg})
    if len(cols) > 4:
        return []
    from itertools import product as iproduct
    for combo in iproduct(("toward", "away", "none"), repeat=len(cols)):
        modes = dict(zip(cols, combo))
        if all(m == "none" for m in combo):
            continue
        c = fit_all(train, tests, lambda x, modes=modes: _rays_wall(x, modes), f"rays_wall[{modes}]", prior=1.6)
        if c:
            return [c]
    return []


LEARNERS += [learn_corner_rect, learn_rays_wall]


def _moves(x, y, conn, multi):
    """Match each input object to an identical (same colored pattern) set of cells in the output; returns
    [(obj, (di, dj))] or None. The output must equal: background + every object moved."""
    objs, bg = _objs(x, conn, multi)
    if not objs or len(objs) > 20:
        return None
    H, W = x.shape
    out = []
    for o in objs:
        pat = np.where(o.mask[o.sl], x[o.sl], -1)
        h, w = pat.shape
        cells = pat >= 0
        best = None
        for i in range(H - h + 1):
            for j in range(W - w + 1):
                if (y[i:i + h, j:j + w][cells] == pat[cells]).all():
                    d = (i - o.sl[0].start, j - o.sl[1].start)
                    if best is None or abs(d[0]) + abs(d[1]) < abs(best[0]) + abs(best[1]):
                        best = d
        if best is None:
            return None
        out.append((o, best))
    return out


def _apply_moves(x, objs, bg, moves):
    g = np.full_like(x, bg)
    H, W = x.shape
    for o, (di, dj) in zip(objs, moves):
        for i, j in np.argwhere(o.mask):
            a, b = i + di, j + dj
            if not (0 <= a < H and 0 <= b < W):
                return None
            g[a, b] = x[i, j]
    return g


def learn_obj_move(train, tests):
    """Every object moves by an offset that is a learned function of ONE object feature (color, size, shape, ...)."""
    if not same_shape(train) or not changed(train):
        return []
    for conn, multi in ((8, True), (4, False), (8, False)):
        samples, ok = [], True
        for x, y in train:
            mv = _moves(x, y, conn, multi)
            if mv is None:
                ok = False
                break
            objs, bg = _objs(x, conn, multi)
            g = _apply_moves(x, objs, bg, [d for _, d in mv])
            if g is None or not (g == y).all():
                ok = False
                break
            samples += [(o.f, d) for o, d in mv]
        if not ok or len({d for _, d in samples}) < 1 or all(d == (0, 0) for _, d in samples):
            continue
        votes = {}
        # offsets proportional to the object's own size: (a * h, b * w)
        for a, b, unit in [(a, b, u) for a in (-1, 0, 1) for b in (-1, 0, 1) for u in ("hw", "size")]:
            if True:
                dim = (lambda f: (f["h"], f["w"])) if unit == "hw" else (lambda f: (f["size"], f["size"]))
                if (a, b) == (0, 0) or not all(d == (a * dim(f)[0], b * dim(f)[1]) for f, d in samples):
                    continue

                def predict(x, conn=conn, multi=multi, a=a, b=b, unit=unit):
                    objs, bg = _objs(x, conn, multi)
                    if not objs:
                        return None
                    dims = [(o.h, o.w) if unit == "hw" else (o.size, o.size) for o in objs]
                    return _apply_moves(x, objs, bg, [(a * d[0], b * d[1]) for d in dims])
                c = fit_all(train, tests, predict, f"obj_move[{conn}{'m' if multi else ''},{a},{b},{unit}]", prior=1.5)
                if c:
                    votes.setdefault(str(c.outs), [0, c])[0] += 2
        for key in ["color", "size", "shape", "shape_d4", "rank_desc", "largest", "smallest", "h", "w", "dims",
                    "touch", "single", "ncolors", "row_rank", "col_rank"]:
            tab, good = {}, True
            for f, d in samples:
                if tab.setdefault(f[key], d) != d:
                    good = False
                    break
            if not good:
                continue

            def predict(x, conn=conn, multi=multi, key=key, tab=tab):
                objs, bg = _objs(x, conn, multi)
                if not objs or any(o.f[key] not in tab for o in objs):
                    return None
                return _apply_moves(x, objs, bg, [tab[o.f[key]] for o in objs])
            c = fit_all(train, tests, predict, f"obj_move[{conn}{'m' if multi else ''},{key}]", prior=1.6)
            if c:
                votes.setdefault(str(c.outs), [0, c])[0] += 1
        if votes:
            ranked = sorted(votes.values(), key=lambda v: -v[0])
            out = []
            for n, c in ranked[:2]:
                c.name += f"x{n}" + ("" if n >= 2 else "~weak")
                out.append(c)
            return out
    return []


LEARNERS.append(learn_obj_move)


_OD4 = {"id": lambda g: g, "flr": lambda g: g[:, ::-1], "fud": lambda g: g[::-1], "r180": lambda g: g[::-1, ::-1],
        "tr": lambda g: g.T, "r90": lambda g: np.rot90(g, -1), "r270": lambda g: np.rot90(g, 1),
        "atr": lambda g: np.rot90(g, 2).T}


def learn_obj_d4(train, tests):
    """Every object (8-connected, multicolor) is replaced inside its bounding box by a rotation / reflection of itself,
    the transform chosen by a table over one object feature (or the same for all)."""
    if not same_shape(train) or not changed(train):
        return []
    for conn, multi in ((8, True), (4, True), (8, False)):
        samples, ok = [], True
        for x, y in train:
            objs, bg = _objs(x, conn, multi)
            if not objs or len(objs) > 30:
                ok = False
                break
            cover = np.zeros(x.shape, bool)
            for o in objs:
                box = x[o.sl]
                tgt = y[o.sl]
                fits = [n for n, f in _OD4.items() if f(box).shape == tgt.shape and (f(box) == tgt).all()]
                if not fits:
                    ok = False
                    break
                samples.append((o.f, frozenset(fits)))
                cover[o.sl] = True
            if not ok or ((x != y) & ~cover).any():
                ok = False
                break
        if not ok or all("id" in fs for _, fs in samples):
            continue
        for key in [None, "color", "size", "shape_d4", "h", "w", "aspect", "ncolors", "rank_desc"]:
            tab, good = {}, True
            for f, fs in samples:
                k = f[key] if key else 0
                tab[k] = tab.get(k, fs) & fs
                if not tab[k]:
                    good = False
                    break
            if not good:
                continue
            tab = {k: sorted(v)[0] if "id" not in v or len(v) == 1 else "id" for k, v in tab.items()}

            def predict(x, conn=conn, multi=multi, key=key, tab=tab):
                objs, bg = _objs(x, conn, multi)
                g = x.copy()
                for o in objs:
                    k = o.f[key] if key else 0
                    if k not in tab:
                        return None
                    t = _OD4[tab[k]](x[o.sl])
                    if t.shape != g[o.sl].shape:
                        return None
                    g[o.sl] = t
                return g
            c = fit_all(train, tests, predict, f"obj_d4[{conn}{'m' if multi else ''},{key}]", prior=1.5)
            if c:
                return [c]
    return []


def learn_recolor_relational(train, tests):
    """Objects of a source color take the color of the object they touch (or of the nearest other object)."""
    if not same_shape(train) or not changed(train):
        return []
    srcs = set()
    for x, y in train:
        d = x != y
        srcs |= set(np.unique(x[d]).tolist())
    if len(srcs) != 1:
        return []
    src = srcs.pop()
    for mode in ("touch", "nearest", "column", "row"):
        def predict(x, mode=mode):
            bg = bg_color(x)
            lab, n = ndi.label(x == src, structure=S8)
            other = (x != bg) & (x != src)
            if not other.any() or n == 0:
                return None
            g = x.copy()
            dist, (ii, jj) = ndi.distance_transform_edt(~other, return_indices=True)
            for k in range(1, n + 1):
                m = lab == k
                if mode == "touch":
                    ring = ndi.binary_dilation(m, structure=S8) & other
                    cols = np.unique(x[ring])
                    if len(cols) != 1:
                        return None
                    g[m] = cols[0]
                elif mode in ("column", "row"):         # the other-colored cells sharing its columns / rows
                    ci, cj = np.nonzero(m)
                    line = other[:, np.unique(cj)] if mode == "column" else other[np.unique(ci), :]
                    vals = x[:, np.unique(cj)][line] if mode == "column" else x[np.unique(ci), :][line]
                    vals = np.unique(vals)
                    if len(vals) != 1:
                        return None
                    g[m] = vals[0]
                else:
                    cells = np.argwhere(m)
                    best = min(cells, key=lambda c: dist[c[0], c[1]])
                    g[m] = x[ii[best[0], best[1]], jj[best[0], best[1]]]
            return g
        c = fit_all(train, tests, predict, f"recolor_rel[{src},{mode}]", prior=1.5)
        if c:
            return [c]
    return []


LEARNERS += [learn_obj_d4, learn_recolor_relational]


def _obj_repeat(x, conn, multi, step, dirs):
    objs, bg = _objs(x, conn, multi)
    if not objs or len(objs) > 10:
        return None
    H, W = x.shape
    g = x.copy()
    for o in objs:
        cells = np.argwhere(o.mask)
        vals = x[o.mask]
        for di, dj in dirs:
            si = di * (o.h + (1 if step == "gap" else 0)) if step in ("size", "gap") else di * step
            sj = dj * (o.w + (1 if step == "gap" else 0)) if step in ("size", "gap") else dj * step
            if (si, sj) == (0, 0):
                return None
            k = 1
            while k < 30:
                ci, cj = cells[:, 0] + k * si, cells[:, 1] + k * sj
                inb = (ci >= 0) & (ci < H) & (cj >= 0) & (cj < W)
                if not inb.any():
                    break
                g[ci[inb], cj[inb]] = vals[inb]
                k += 1
    return g


def learn_obj_repeat(train, tests):
    """Every object is repeated along a direction set with a step of its own size (touching / one gap) or a fixed
    step, up to the border."""
    if not same_shape(train) or not changed(train):
        return []
    dirsets = {"U": [(-1, 0)], "D": [(1, 0)], "L": [(0, -1)], "R": [(0, 1)], "V": [(-1, 0), (1, 0)],
               "H": [(0, -1), (0, 1)], "DR": [(1, 1)], "UR": [(-1, 1)]}
    for conn, multi in ((8, True), (8, False)):
        for step in ("size", "gap", 2, 3, 4):
            for dn, dirs in dirsets.items():
                c = fit_all(train, tests, lambda x, conn=conn, multi=multi, step=step, dirs=dirs:
                            _obj_repeat(x, conn, multi, step, dirs), f"obj_repeat[{conn}{'m' if multi else ''},{step},{dn}]",
                            prior=1.6)
                if c:
                    return [c]
    return []


LEARNERS.append(learn_obj_repeat)


def _shortest_path(x, end_color, path_color, order):
    """BFS (4-connected, over background cells) between the two cells of end_color; the path's cells take
    path_color. `order`: neighbour expansion order (tie-break among equal-length paths)."""
    from collections import deque
    bg = bg_color(x)
    ends = [tuple(p) for p in np.argwhere(x == end_color)]
    if len(ends) != 2:
        return None
    H, W = x.shape
    (si, sj), goal = ends
    prev = {(si, sj): None}
    dq = deque([(si, sj)])
    found = False
    while dq:
        i, j = dq.popleft()
        if (i, j) == goal:
            found = True
            break
        for di, dj in order:
            a, b = i + di, j + dj
            if 0 <= a < H and 0 <= b < W and (a, b) not in prev and (x[a, b] == bg or (a, b) == goal):
                prev[(a, b)] = (i, j)
                dq.append((a, b))
    if not found:
        return None
    g = x.copy()
    c = prev[goal]
    while c is not None and c != (si, sj):
        g[c] = path_color
        c = prev[c]
    return g


def learn_shortest_path(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    added = set()
    for x, y in train:
        added |= set(np.unique(y[x != y]).tolist())
    if len(added) != 1:
        return []
    pc = added.pop()
    from itertools import permutations
    for ec in range(10):
        if not all((x == ec).sum() == 2 for x, _ in train):
            continue
        for order in permutations(DIRS8[:4]):
            c = fit_all(train, tests, lambda x, ec=ec, order=order: _shortest_path(x, ec, pc, order),
                        f"shortest_path[{ec}->{pc},{order}]", prior=1.6)
            if c:
                return [c]
    return []


def learn_midpoint_stamp(train, tests):
    """At the midpoint of every pair of same-colored single pixels on one row / column, a learned pattern."""
    if not same_shape(train) or not changed(train):
        return []
    pats = {}
    for x, y in train:
        bg = bg_color(x)
        pts = list(zip(*np.nonzero(x != bg)))
        mids = []
        for a in range(len(pts)):
            for b in range(a + 1, len(pts)):
                (i1, j1), (i2, j2) = pts[a], pts[b]
                if x[i1, j1] == x[i2, j2] and (i1 == i2 or j1 == j2) and (i1 + i2) % 2 == 0 and (j1 + j2) % 2 == 0:
                    mids.append(((i1 + i2) // 2, (j1 + j2) // 2))
        if not mids:
            return []
        H, W = x.shape
        for mi, mj in mids:
            for di in range(-2, 3):
                for dj in range(-2, 3):
                    a, b = mi + di, mj + dj
                    if 0 <= a < H and 0 <= b < W and x[a, b] == bg:
                        v = int(y[a, b])
                        if pats.setdefault((di, dj), v) != v:
                            pats[(di, dj)] = None
    P = {k: v for k, v in pats.items() if v is not None}

    def predict(x):
        bg = bg_color(x)
        pts = list(zip(*np.nonzero(x != bg)))
        g = x.copy()
        H, W = x.shape
        hit = False
        for a in range(len(pts)):
            for b in range(a + 1, len(pts)):
                (i1, j1), (i2, j2) = pts[a], pts[b]
                if x[i1, j1] == x[i2, j2] and (i1 == i2 or j1 == j2) and (i1 + i2) % 2 == 0 and (j1 + j2) % 2 == 0:
                    mi, mj = (i1 + i2) // 2, (j1 + j2) // 2
                    for (di, dj), v in P.items():
                        p, q = mi + di, mj + dj
                        if 0 <= p < H and 0 <= q < W and x[p, q] == bg:
                            g[p, q] = v
                            hit = True
        return g if hit else None
    c = fit_all(train, tests, predict, "midpoint_stamp", prior=1.6)
    return [c] if c else []


LEARNERS += [learn_shortest_path, learn_midpoint_stamp]


def _attract(x, target_color, conn):
    """Every object not of target_color slides toward the target object (the union of target_color cells) along the
    axis on which their projections overlap, until it touches it."""
    bg = bg_color(x)
    tm = x == target_color
    if not tm.any():
        return None
    H, W = x.shape
    lab, n = ndi.label((x != bg) & ~tm, structure=S8 if conn == 8 else S4)
    if n == 0:
        return None
    g = x.copy()
    occ = x != bg
    ti, tj = np.nonzero(tm)
    moved = False
    for k in range(1, n + 1):
        m = lab == k
        ci, cj = np.nonzero(m)
        rows_overlap = set(ci) & set(ti)
        cols_overlap = set(cj) & set(tj)
        if cols_overlap and not rows_overlap:
            d = (1, 0) if ci.max() < ti.min() else (-1, 0) if ci.min() > ti.max() else None
        elif rows_overlap and not cols_overlap:
            d = (0, 1) if cj.max() < tj.min() else (0, -1) if cj.min() > tj.max() else None
        else:
            d = None
        if d is None:
            continue
        vals = x[ci, cj]
        occ[ci, cj] = False
        g[ci, cj] = bg
        s = 0
        while True:
            ni, nj = ci + (s + 1) * d[0], cj + (s + 1) * d[1]
            if (ni < 0).any() or (nj < 0).any() or (ni >= H).any() or (nj >= W).any() or occ[ni, nj].any():
                break
            s += 1
        g[ci + s * d[0], cj + s * d[1]] = vals
        occ[ci + s * d[0], cj + s * d[1]] = True
        moved |= s > 0
    return g if moved else None


def learn_attract(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    cols = sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg_color(train[0][0])})
    for tc in cols:
        for conn in (8, 4):
            c = fit_all(train, tests, lambda x, tc=tc, conn=conn: _attract(x, tc, conn), f"attract[to={tc},{conn}]",
                        prior=1.5)
            if c:
                return [c]
    return []


LEARNERS.append(learn_attract)


def learn_line_rank(train, tests):
    """Cells of a source color take a color looked up by the rank of their column (or row) among the columns (rows)
    that contain the source color."""
    if not same_shape(train) or not changed(train):
        return []
    srcs = set()
    for x, y in train:
        srcs |= set(np.unique(x[x != y]).tolist())
    if len(srcs) != 1:
        return []
    src = srcs.pop()
    for axis in (1, 0):
        tab, ok = {}, True
        for x, y in train:
            lines = sorted(set(np.nonzero(x == src)[axis].tolist()))
            for i, j in zip(*np.nonzero(x == src)):
                r = lines.index(j if axis == 1 else i)
                if tab.setdefault(r, int(y[i, j])) != int(y[i, j]):
                    ok = False
                    break
            if not ok:
                break
        if not ok or len(set(tab.values())) < 2:
            continue

        def predict(x, axis=axis, tab=tab):
            g = x.copy()
            lines = sorted(set(np.nonzero(x == src)[axis].tolist()))
            for i, j in zip(*np.nonzero(x == src)):
                r = lines.index(j if axis == 1 else i)
                if r not in tab:
                    return None
                g[i, j] = tab[r]
            return g
        c = fit_all(train, tests, predict, f"line_rank[src={src},axis={axis}]", prior=1.6)
        if c:
            return [c]
    return []


LEARNERS.append(learn_line_rank)


def _extend_lines(x, priority, under):
    """Every horizontal line object (height 1, width >= 2) fills its whole row and every vertical one (width 1,
    height >= 2) its whole column; `priority`: which orientation is painted last (wins crossings)."""
    bg = bg_color(x)
    objs = extract(x, bg, 4, False)
    hs = [o for o in objs if o.h == 1 and o.w >= 2]
    vs = [o for o in objs if o.w == 1 and o.h >= 2]
    if not hs and not vs:
        return None
    g = x.copy()
    order = (vs + hs) if priority == "h" else (hs + vs)
    for o in order:
        if o.h == 1:
            r = o.sl[0].start
            m = (g[r] == bg) if under else np.ones(x.shape[1], bool)
            g[r, m] = o.color
        else:
            c = o.sl[1].start
            m = (g[:, c] == bg) if under else np.ones(x.shape[0], bool)
            g[m, c] = o.color
    return None if (g == x).all() else g


def learn_extend_lines(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    for priority in ("h", "v"):
        for under in (False, True):
            c = fit_all(train, tests, lambda x, p=priority, u=under: _extend_lines(x, p, u),
                        f"extend_lines[{priority} wins,under={under}]", prior=1.5)
            if c:
                return [c]
    return []


def learn_align_ref(train, tests):
    """Every object (8-connected, multicolor) shifts vertically (horizontally) so its top (left) edge matches the
    reference object's -- the object of a fixed color."""
    if not same_shape(train) or not changed(train):
        return []
    cols = sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg_color(train[0][0])})
    for ref in cols:
        for axis in (0, 1):
            for edge in ("start", "center"):
                def predict(x, ref=ref, axis=axis, edge=edge):
                    objs, bg = _objs(x, 8, True)
                    refs = [o for o in objs if o.color == ref]
                    if len(refs) != 1:
                        return None
                    r = refs[0]
                    key = (lambda o: o.sl[axis].start) if edge == "start" else \
                        (lambda o: (o.sl[axis].start + o.sl[axis].stop - 1) // 2)
                    moves = [((key(r) - key(o), 0) if axis == 0 else (0, key(r) - key(o))) for o in objs]
                    return _apply_moves(x, objs, bg, moves)
                c = fit_all(train, tests, predict, f"align_ref[{ref},axis={axis},{edge}]", prior=1.5)
                if c:
                    return [c]
    return []


LEARNERS += [learn_extend_lines, learn_align_ref]


def learn_rays_per_color(train, tests):
    """Every pixel of color c becomes a full line in ITS color's direction set (H / V / both / none, learned per
    color); lines pass over everything; colors painted in a learned order (the later wins crossings)."""
    if not same_shape(train) or not changed(train):
        return []
    bg = bg_color(train[0][0])
    cols = sorted(set.union(*[set(np.unique(x).tolist()) for x, _ in train]) - {bg})
    if not 1 <= len(cols) <= 4:
        return []
    from itertools import product as iproduct, permutations
    opts = ("H", "V", "HV", "none")
    for combo in iproduct(opts, repeat=len(cols)):
        if all(o == "none" for o in combo):
            continue
        modes = dict(zip(cols, combo))
        for order in permutations([c for c in cols if modes[c] != "none"]):
            def predict(x, modes=modes, order=order):
                b = bg_color(x)
                g = x.copy()
                pts = {c: list(zip(*np.nonzero(x == c))) for c in order}
                for c in order:
                    for i, j in pts[c]:
                        if "H" in modes[c]:
                            g[i, :] = c
                        if "V" in modes[c]:
                            g[:, j] = c
                for c in cols:                      # the source pixels themselves stay on top
                    if modes.get(c) == "none":
                        g[x == c] = c
                return None if (g == x).all() else g
            c = fit_all(train, tests, predict, f"rays_per_color[{modes},{order}]", prior=1.6)
            if c:
                return [c]
    return []


LEARNERS.append(learn_rays_per_color)


def _leaky(x, c, through):
    """Rectangular outlines (one color) with exactly one gap in the border: the inside and the gap take color c, and
    a ray of c leaves through the gap (perpendicular to that side) to the edge, stopping at obstacles unless
    `through`."""
    bg = bg_color(x)
    H, W = x.shape
    g = x.copy()
    hit = False
    for col in np.unique(x):
        if col == bg:
            continue
        lab, n = ndi.label(x == col, structure=S8)
        for k, sl in enumerate(ndi.find_objects(lab), 1):
            m = lab[sl] == k
            h, w = m.shape
            if h < 3 or w < 3:
                continue
            ring = np.zeros_like(m)
            ring[0, :] = ring[-1, :] = ring[:, 0] = ring[:, -1] = True
            gaps = np.argwhere(ring & ~m)
            if len(gaps) != 1 or (m & ~ring).any():
                continue
            gi, gj = gaps[0]
            if (gi in (0, h - 1)) and (gj in (0, w - 1)):
                continue                                # a corner gap has no direction
            r0, c0 = sl[0].start, sl[1].start
            inner = np.zeros((H, W), bool)
            inner[r0 + 1:r0 + h - 1, c0 + 1:c0 + w - 1] = True
            g[inner & (x == bg)] = c
            a, b = r0 + gi, c0 + gj
            di, dj = (-1, 0) if gi == 0 else (1, 0) if gi == h - 1 else (0, -1) if gj == 0 else (0, 1)
            while 0 <= a < H and 0 <= b < W:
                if x[a, b] != bg and not ((a, b) == (r0 + gi, c0 + gj)):
                    if not through:
                        break
                else:
                    g[a, b] = c
                a, b = a + di, b + dj
            hit = True
    return g if hit else None


def learn_leaky(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    added = set()
    for x, y in train:
        added |= set(np.unique(y[x != y]).tolist())
    if len(added) != 1:
        return []
    c0 = added.pop()
    for through in (False, True):
        c = fit_all(train, tests, lambda x, t=through: _leaky(x, c0, t), f"leaky[{c0},through={through}]", prior=1.6)
        if c:
            return [c]
    return []


LEARNERS.append(learn_leaky)


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners2.py
"""Learners batch 2 (2026-10-18), chosen from the unsolved training tasks (add-only / shrinking families):

  flood      seed pixels spread through the connected region of a target color (4- or 8-connected)
  latin      fill the background so every row and column holds each color once (Latin square)
  part_pick  split the grid into equal parts (separator lines or an equal k x l division) and return the part that is
             unique under a feature (symmetry, color count, cell count, majority color, ...) -- "odd one out"
  block      output of a fixed shape: the input cut into equal blocks, each block reduced (its only non-bg color,
             majority, majority non-bg, any-non-bg flag)
  tile_one   the input is k exact copies of a tile along one axis -> return the tile
  part_d4    separator cells: the empty cells get a rotation / reflection of the one filled cell, the transform
             learned per cell position
  period_bg  periodic completion where the background is unknown and never-seen classes stay background
"""
from collections import Counter
from itertools import product

import numpy as np
from scipy import ndimage as ndi

from learners import changed, A, _tile_fit, bg_color, eq, fit_all, same_shape, sep_cells

S4 = ndi.generate_binary_structure(2, 1)
S8 = np.ones((3, 3), bool)
D4F = {"id": lambda g: g, "r90": lambda g: np.rot90(g, -1), "r180": lambda g: np.rot90(g, 2),
       "r270": lambda g: np.rot90(g, 1), "fud": lambda g: g[::-1], "flr": lambda g: g[:, ::-1],
       "tr": lambda g: g.T, "atr": lambda g: np.rot90(g, 2).T}


# ------------------------------------------------------------------------------------------------ flood fill
def _flood(x, seed_colors, target, conn):
    g = x.copy()
    region = x == target
    lab, n = ndi.label(region, structure=S4 if conn == 4 else S8)
    if n == 0:
        return None
    st = S4 if conn == 4 else S8
    changed = False
    for s in seed_colors:
        seeds = x == s
        if not seeds.any():
            continue
        touch = ndi.binary_dilation(seeds, structure=st) & region
        ids = set(np.unique(lab[touch]).tolist()) - {0}
        for k in ids:
            m = lab == k
            g[m] = s
            changed = True
    return g if changed else None


def learn_flood(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    found = []
    cols = sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]))
    added = set()
    for x, y in train:
        added |= set(np.unique(y[x != y]).tolist())
    for target in ["bg"] + cols:
        for conn in (4, 8):
            for seeds in [[c] for c in sorted(added)] + ([sorted(added)] if len(added) > 1 else []):
                def predict(x, target=target, conn=conn, seeds=seeds):
                    t = bg_color(x) if target == "bg" else target
                    return _flood(x, [s for s in seeds if s != t], t, conn)
                c = fit_all(train, tests, predict, f"flood[target={target},conn={conn},seeds={seeds}]", prior=1.4)
                if c:
                    found.append(c)
                    return found
    return found


# ------------------------------------------------------------------------------------------------ latin square
def _latin(x, hole):
    H, W = x.shape
    if H != W:
        return None
    cols = sorted(set(np.unique(x).tolist()) - {hole})
    if len(cols) != H:
        return None
    g = x.copy()

    def cand(i, j):
        return [c for c in cols if c not in g[i] and c not in g[:, j]]

    def solve():
        best = None
        for i in range(H):
            for j in range(W):
                if g[i, j] == hole:
                    cs = cand(i, j)
                    if not cs:
                        return False
                    if best is None or len(cs) < len(best[2]):
                        best = (i, j, cs)
        if best is None:
            return True
        i, j, cs = best
        if len(cs) > 1:
            return "multi"
        g[i, j] = cs[0]
        return solve()
    r = solve()
    return g if r is True else None


def learn_latin(train, tests):
    if not same_shape(train):
        return []
    for hole in range(10):
        c = fit_all(train, tests, lambda x, hole=hole: _latin(x, hole) if (x == hole).any() else None,
                    f"latin[hole={hole}]", prior=1.3)
        if c:
            return [c]
    return []


# ------------------------------------------------------------------------------------------------ parts
def parts_of(x, mode):
    """mode 'sep': cells between full separator lines; ('eq', k, l): an equal k x l division."""
    if mode == "sep":
        cells = sep_cells(x)
        if len(cells) < 2:
            return None
        shapes = {(b - a, d - c) for a, b, c, d in cells}
        if len(shapes) != 1:
            return None
        return [x[a:b, c:d] for a, b, c, d in cells]
    _, k, l = mode
    H, W = x.shape
    if H % k or W % l:
        return None
    h, w = H // k, W // l
    return [x[i * h:(i + 1) * h, j * w:(j + 1) * w] for i in range(k) for j in range(l)]


def _pfeat(p, bg):
    nb = p != bg
    cnt = Counter(p[nb].tolist())
    return {
        "sym_tr": p.shape[0] == p.shape[1] and bool((p == p.T).all()),
        "sym_lr": bool((p == p[:, ::-1]).all()), "sym_ud": bool((p == p[::-1]).all()),
        "sym_any": bool((p == p[:, ::-1]).all() or (p == p[::-1]).all() or (p.shape[0] == p.shape[1] and (p == p.T).all())),
        "ncolors": len(cnt), "n_fg": int(nb.sum()), "major": cnt.most_common(1)[0][0] if cnt else -1,
        "colors": frozenset(cnt), "pattern": (p != bg).tobytes(), "exact": p.tobytes(),
        "n_obj": int(ndi.label(nb, structure=S4)[1]),
    }


PART_MODES = ["sep"] + [("eq", k, l) for k, l in [(1, 2), (2, 1), (1, 3), (3, 1), (2, 2), (1, 4), (4, 1), (3, 3), (2, 3), (3, 2)]]
PICKS = [("unique", f) for f in ("sym_tr", "sym_lr", "sym_ud", "sym_any", "ncolors", "n_fg", "major", "colors", "pattern",
                                 "exact", "n_obj")] + \
        [(m, f) for m in ("max", "min") for f in ("ncolors", "n_fg", "n_obj")]


def _pick(parts, bg, how, f):
    fs = [_pfeat(p, bg)[f] for p in parts]
    if how == "unique":
        c = Counter(fs)
        u = [i for i, v in enumerate(fs) if c[v] == 1]
        return parts[u[0]] if len(u) == 1 and len(parts) > 2 else None
    v = max(fs) if how == "max" else min(fs)
    idx = [i for i, a in enumerate(fs) if a == v]
    return parts[idx[0]] if len(idx) == 1 else None


def learn_part_pick(train, tests):
    if same_shape(train):
        return []
    for mode in PART_MODES:
        if not all(parts_of(x, mode) is not None and any(eq(p, y) for p in parts_of(x, mode)) for x, y in train):
            continue
        for how, f in PICKS:
            def predict(x, mode=mode, how=how, f=f):
                ps = parts_of(x, mode)
                return None if ps is None else _pick(ps, bg_color(x), how, f)
            c = fit_all(train, tests, predict, f"part_pick[{mode},{how} {f}]", prior=1.5)
            if c:
                return [c]
    return []


def learn_tile_one(train, tests):
    """Input = k exact copies of a tile along rows or columns -> the tile (smallest period)."""
    if same_shape(train):
        return []

    def predict(x, axis):
        if axis == "any":
            return predict(x, 1) if predict(x, 1) is not None else predict(x, 0)
        n = x.shape[axis]
        for p in range(1, n):
            if n % p == 0 and n // p >= 2:
                t = x[:p] if axis == 0 else x[:, :p]
                reps = (n // p, 1) if axis == 0 else (1, n // p)
                if (np.tile(t, reps) == x).all():
                    return t
        return None
    for axis in (0, 1, "any"):
        c = fit_all(train, tests, lambda x, axis=axis: predict(x, axis), f"tile_one[axis={axis}]", prior=1.3)
        if c:
            return [c]
    return []


# ------------------------------------------------------------------------------------------------ block reduce
def _block(x, oh, ow, red):
    H, W = x.shape
    if H % oh or W % ow:
        return None
    bh, bw = H // oh, W // ow
    bg = bg_color(x)
    out = np.zeros((oh, ow), x.dtype)
    for i in range(oh):
        for j in range(ow):
            b = x[i * bh:(i + 1) * bh, j * bw:(j + 1) * bw]
            fg = b[b != bg]
            if red == "only":
                u = np.unique(fg)
                if len(u) > 1:
                    return None
                out[i, j] = u[0] if len(u) else bg
            elif red == "major":
                v, c = np.unique(b, return_counts=True)
                out[i, j] = v[c.argmax()]
            elif red == "major_fg":
                if len(fg):
                    v, c = np.unique(fg, return_counts=True)
                    out[i, j] = v[c.argmax()]
                else:
                    out[i, j] = bg
            elif red == "minor":
                v, c = np.unique(b, return_counts=True)
                out[i, j] = v[c.argmin()]
    return out


def learn_block(train, tests):
    if same_shape(train):
        return []
    shapes = {y.shape for _, y in train}
    found = []
    if len(shapes) == 1:
        oh, ow = shapes.pop()
        for red in ("only", "major", "major_fg", "minor"):
            c = fit_all(train, tests, lambda x, red=red: _block(x, oh, ow, red), f"block[{oh}x{ow},{red}]", prior=1.5)
            if c:
                return [c]
    # fixed block size instead of a fixed output shape
    ratios = {(x.shape[0] / y.shape[0], x.shape[1] / y.shape[1]) for x, y in train}
    if len(ratios) == 1:
        rh, rw = ratios.pop()
        if rh == int(rh) and rw == int(rw) and rh * rw > 1:
            for red in ("only", "major", "major_fg", "minor"):
                c = fit_all(train, tests, lambda x, red=red: _block(x, x.shape[0] // int(rh), x.shape[1] // int(rw), red)
                            if x.shape[0] % int(rh) == 0 and x.shape[1] % int(rw) == 0 else None,
                            f"block[/{int(rh)}x{int(rw)},{red}]", prior=1.5)
                if c:
                    return [c]
    return found


# ------------------------------------------------------------------------------------------------ separator cells <- D4 of the source
def learn_part_d4(train, tests):
    if not same_shape(train):
        return []
    table = {}
    for x, y in train:
        cells = sep_cells(x)
        if len(cells) < 2:
            return []
        bg = bg_color(x)
        filled = [k for k, (a, b, c, d) in enumerate(cells) if (x[a:b, c:d] != bg).any()]
        if len(filled) != 1:
            return []
        a, b, c, d = cells[filled[0]]
        src = x[a:b, c:d]
        for k, (a2, b2, c2, d2) in enumerate(cells):
            tgt = y[a2:b2, c2:d2]
            ok = [n for n, f in D4F.items() if f(src).shape == tgt.shape and (f(src) == tgt).all()]
            key = (k, filled[0])
            if not ok:
                return []
            table[key] = [n for n in table.get(key, ok) if n in ok]
            if not table[key]:
                return []

    def predict(x):
        cells = sep_cells(x)
        bg = bg_color(x)
        filled = [k for k, (a, b, c, d) in enumerate(cells) if (x[a:b, c:d] != bg).any()]
        if len(filled) != 1:
            return None
        a, b, c, d = cells[filled[0]]
        src = x[a:b, c:d]
        g = x.copy()
        for k, (a2, b2, c2, d2) in enumerate(cells):
            key = (k, filled[0])
            if key not in table:
                return None
            t = D4F[table[key][0]](src)
            if t.shape != (b2 - a2, d2 - c2):
                return None
            g[a2:b2, c2:d2] = t
        return g
    c = fit_all(train, tests, predict, "part_d4", prior=1.5)
    return [c] if c else []


# ------------------------------------------------------------------------------------------------ tile the pattern block
def _block_tile(x, axis, mirror):
    """Tile the rows (axis 0), columns (1) or both spanned by the non-background cells over the whole grid,
    anchored at the block; mirror: every other copy reflected."""
    from learners import bbox
    bg = bg_color(x)
    bb = bbox(x != bg)
    if bb is None:
        return None
    r0, r1, c0, c1 = bb
    H, W = x.shape
    if axis in (0, 2) and r1 - r0 == H or axis in (1, 2) and c1 - c0 == W:
        return None

    def idx(k, start, period):
        q, r = divmod(k - start, period)
        return start + (period - 1 - r if mirror and q % 2 else r)
    ph, pw = r1 - r0, c1 - c0
    if mirror == "period":                         # 2026-10-18: the block's own smallest exact period
        B = x[r0:r1, c0:c1]
        pw = next((p for p in range(1, B.shape[1] + 1) if (B[:, :B.shape[1] - p] == B[:, p:]).all()), B.shape[1])
        ph = next((p for p in range(1, B.shape[0] + 1) if (B[:B.shape[0] - p] == B[p:]).all()), B.shape[0])
        mirror = False
    I = np.array([idx(i, r0, ph) for i in range(H)]) if axis in (0, 2) else np.arange(H)
    J = np.array([idx(j, c0, pw) for j in range(W)]) if axis in (1, 2) else np.arange(W)
    return x[np.ix_(I, J)]


def learn_block_tile(train, tests):
    if not same_shape(train):
        return []
    for axis in (0, 1, 2):
        for mirror in (False, True, "period"):
            c = fit_all(train, tests, lambda x, axis=axis, mirror=mirror: _block_tile(x, axis, mirror),
                        f"block_tile[axis={axis},mirror={mirror}]", prior=1.5)
            if c:
                return [c]
    return []


LEARNERS = [learn_flood, learn_latin, learn_part_pick, learn_tile_one, learn_block, learn_part_d4, learn_block_tile]


def _broadcast(x, mode):
    cells = sep_cells(x)
    if len(cells) < 3:
        return None
    bg = bg_color(x)
    subs = [x[a:b, c:d] for a, b, c, d in cells]
    if len({s.shape for s in subs}) != 1:
        return None
    counts = [int((s != bg).sum()) for s in subs]
    top = max(counts)
    if counts.count(top) != 1 or top < 2:
        return None
    src = subs[counts.index(top)]
    g = x.copy()
    for (a, b, c, d), s, n in zip(cells, subs, counts):
        if s is src or n == 0:
            continue
        cols = set(np.unique(s[s != bg]).tolist())
        if mode == "abs":
            g[a:b, c:d] = src
        elif mode == "recolor":
            if len(cols) != 1:
                return None
            g[a:b, c:d] = np.where(src != bg, cols.pop(), bg)
        elif mode == "under":
            g[a:b, c:d] = np.where(s != bg, s, src)
    return g


def learn_broadcast(train, tests):
    """Separator cells: the one richest cell's pattern copied into every cell that holds a marker (as is, recolored
    to the marker's color, or under the marker)."""
    if not same_shape(train) or not changed(train):
        return []
    for mode in ("recolor", "abs", "under"):
        c = fit_all(train, tests, lambda x, mode=mode: _broadcast(x, mode), f"broadcast[{mode}]", prior=1.5)
        if c:
            return [c]
    return []


LEARNERS.append(learn_broadcast)


def _split(x, how):
    H, W = x.shape
    if how == "sep":
        cells = sep_cells(x)
        if len(cells) < 2:
            return None
        ps = [x[a:b, c:d] for a, b, c, d in cells]
    else:
        k, axis = how
        n = x.shape[axis]
        if n % k == 0:
            s = n // k
            ps = [x[i * s:(i + 1) * s] if axis == 0 else x[:, i * s:(i + 1) * s] for i in range(k)]
        elif (n - (k - 1)) % k == 0:                     # k parts with a one-cell divider between them
            s = (n - (k - 1)) // k
            ps = [x[i * (s + 1):i * (s + 1) + s] if axis == 0 else x[:, i * (s + 1):i * (s + 1) + s] for i in range(k)]
        else:
            return None
    if len({p.shape for p in ps}) != 1:
        return None
    return ps


BOOL2 = {"and": lambda a, b: a & b, "or": lambda a, b: a | b, "xor": lambda a, b: a ^ b, "nor": lambda a, b: ~(a | b),
         "a-b": lambda a, b: a & ~b, "b-a": lambda a, b: b & ~a, "nand": lambda a, b: ~(a & b), "xnor": lambda a, b: ~(a ^ b)}


def learn_parts_logic(train, tests):
    """Output = the parts of the input (halves / thirds / separator cells) combined cellwise: a boolean function of
    their foreground masks painted with one learned color on one learned background, or a priority overlay."""
    if same_shape(train):
        return []
    found = []
    hows = ["sep"] + [(k, a) for k in (2, 3, 4) for a in (0, 1)]
    for how in hows:
        parts = [_split(x, how) for x, _ in train]
        if any(p is None or p[0].shape != y.shape for p, (_, y) in zip(parts, train)):
            continue
        k = len(parts[0])
        for bgm in ("mode", "zero"):                     # 2026-10-18: background = most common color, or black
            bgs = [bg_color(x) if bgm == "mode" else 0 for x, _ in train]
            if bgm == "zero" and bgs == [bg_color(x) for x, _ in train]:
                continue
            if k == 2:
                for fname, f in BOOL2.items():
                    on, off, ok = None, None, True
                    for ps, (x, y), bg in zip(parts, train, bgs):
                        m = f(ps[0] != bg, ps[1] != bg)    # 2026-10-18: a pair may be all-on or all-off
                        v_on, v_off = np.unique(y[m]), np.unique(y[~m])
                        if len(v_on) > 1 or len(v_off) > 1:
                            ok = False
                            break
                        if len(v_on):
                            if on is None:
                                on = int(v_on[0])
                            ok = ok and on == int(v_on[0])
                        if len(v_off):
                            if off is None:
                                off = int(v_off[0])
                            ok = ok and off == int(v_off[0])
                        if not ok:
                            break
                    if not ok or on is None or off is None:
                        continue

                    def predict(x, how=how, f=f, on=on, off=off, bgm=bgm):
                        ps = _split(x, how)
                        if ps is None or len(ps) != 2:
                            return None
                        bg = bg_color(x) if bgm == "mode" else 0
                        return np.where(f(ps[0] != bg, ps[1] != bg), on, off).astype(x.dtype)
                    c = fit_all(train, tests, predict, f"parts_logic[{how},{fname}" + (",bg0" if bgm == "zero" else "") + "]",
                                prior=1.4)
                    if c:
                        return [c]
        # priority overlay: first non-background part wins, in one of the two reading orders
        for rev in (False, True):
            def predict(x, how=how, rev=rev):
                ps = _split(x, how)
                if ps is None:
                    return None
                bg = bg_color(x)
                g = np.full(ps[0].shape, bg, x.dtype)
                for p in (ps if rev else ps[::-1]):
                    g = np.where(p != bg, p, g)
                return g
            c = fit_all(train, tests, predict, f"parts_overlay[{how},{'first' if rev else 'last'}-wins]", prior=1.5)
            if c:
                return [c]
    return found


LEARNERS.append(learn_parts_logic)


def _odd_per_line(x, axis):
    """Separator cells in an r x c layout: per layout row (axis 0) or column (axis 1) the one cell whose content is
    unique there; the output keeps the frame of the first column (row) and holds those odd cells."""
    cells = sep_cells(x)
    if len(cells) < 4:
        return None
    rows = sorted({a for a, b, c, d in cells})
    cols = sorted({c for a, b, c, d in cells})
    if len(rows) * len(cols) != len(cells):
        return None
    grid = {(a, c): (a, b, c, d) for a, b, c, d in cells}
    picks = []
    lines = rows if axis == 0 else cols
    for L in lines:
        group = [grid[(L, c)] if axis == 0 else grid[(a, L)] for c in cols] if axis == 0 else \
            [grid[(a, L)] for a in rows]
        subs = [x[a:b, c:d] for a, b, c, d in group]
        keys = [s.tobytes() for s in subs]
        uniq = [i for i, k in enumerate(keys) if keys.count(k) == 1]
        if len(uniq) != 1 or len(subs) < 3:
            return None
        picks.append(subs[uniq[0]])
    a0, b0, c0, d0 = grid[(rows[0], cols[0])]
    if axis == 0:
        g = x[:, :d0 + 1].copy() if d0 < x.shape[1] else x[:, :d0].copy()
        for L, p in zip(rows, picks):
            a, b, c, d = grid[(L, cols[0])]
            g[a:b, c:d] = p
    else:
        g = x[:b0 + 1].copy() if b0 < x.shape[0] else x[:b0].copy()
        for L, p in zip(cols, picks):
            a, b, c, d = grid[(rows[0], L)]
            g[a:b, c:d] = p
    return g


def learn_odd_per_line(train, tests):
    if same_shape(train):
        return []
    for axis in (0, 1):
        c = fit_all(train, tests, lambda x, axis=axis: _odd_per_line(x, axis), f"odd_per_line[axis={axis}]", prior=1.5)
        if c:
            return [c]
    return []


LEARNERS.append(learn_odd_per_line)


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners3.py
"""Learners batch 3 (2026-10-18), from the unsolved recolor-only training tasks:

  local     each output cell is a function of its input neighbourhood (3x3, cross, 5x5) -- a lookup table learned
            from the train pairs, optionally invariant under rotation / reflection; every test window must have been
            seen (no guessing)
  template  every placeholder blob (one color) is overwritten by the template object (a multicolor object) whose
            shape matches it -- as is, or rotated / reflected to fit
"""
import numpy as np
from scipy import ndimage as ndi

from learners import changed, bg_color, fit_all, same_shape

S8 = np.ones((3, 3), bool)
D4 = [lambda w: w, lambda w: np.rot90(w, 1), lambda w: np.rot90(w, 2), lambda w: np.rot90(w, 3),
      lambda w: w[::-1], lambda w: w[:, ::-1], lambda w: w.T, lambda w: np.rot90(w, 2).T]


# ------------------------------------------------------------------------------------------------ local rules
def _windows(x, r, shape):
    H, W = x.shape
    p = np.pad(x, r, constant_values=-1)
    out = np.empty((H, W, 2 * r + 1, 2 * r + 1), dtype=np.int16)
    for di in range(2 * r + 1):
        for dj in range(2 * r + 1):
            out[:, :, di, dj] = p[di:di + H, dj:dj + W]
    if shape == "cross":
        m = np.zeros((2 * r + 1, 2 * r + 1), bool)
        m[r, :] = m[:, r] = True
        out = np.where(m, out, -2)
    return out


def _key(w, mode, sym, bg):
    if mode == "fgmask":                                  # colors abstracted to bg / fg / border, center exact
        c = w[w.shape[0] // 2, w.shape[1] // 2]
        w = np.where(w < 0, w, np.where(w == bg, 0, 1))
        w = w.copy()
        w[w.shape[0] // 2, w.shape[1] // 2] = 10 + c
    if sym:
        return min(f(w).tobytes() for f in D4)
    return w.tobytes()


def _local_table(train, r, shape, mode, sym):
    tab = {}
    for x, y in train:
        bg = bg_color(x)
        W = _windows(x, r, shape)
        for i in range(x.shape[0]):
            for j in range(x.shape[1]):
                k = _key(W[i, j], mode, sym, bg)
                v = int(y[i, j])
                if tab.setdefault(k, v) != v:
                    return None
    return tab


def learn_local(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    if sum(x.size for x, _ in train) > 3000:
        return []
    found = []
    for r, shape in ((1, "cross"), (1, "square"), (2, "cross"), (2, "square")):
        for mode in ("exact", "fgmask"):
            for sym in (False, True):
                tab = _local_table(train, r, shape, mode, sym)
                if tab is None:
                    continue

                def predict(x, r=r, shape=shape, mode=mode, sym=sym, tab=tab):
                    bg = bg_color(x)
                    W = _windows(x, r, shape)
                    g = x.copy()
                    for i in range(x.shape[0]):
                        for j in range(x.shape[1]):
                            k = _key(W[i, j], mode, sym, bg)
                            if k not in tab:
                                return None
                            g[i, j] = tab[k]
                    return g
                c = fit_all(train, tests, predict, f"local[r={r},{shape},{mode},sym={sym}]", prior=1.7)
                if c:
                    found.append(c)
                    return found
    return found


# ------------------------------------------------------------------------------------------------ template stamping
def _templates_and_holders(x, ph, any_color=False):
    bg = bg_color(x)
    lab, n = ndi.label((x != bg) & (x != ph), structure=S8)
    temps = []
    for k, sl in enumerate(ndi.find_objects(lab), 1):
        m = lab[sl] == k
        sub = np.where(m, x[sl], -1)
        if any_color or len(np.unique(x[sl][m])) >= 2:
            temps.append(sub)
    lab2, n2 = ndi.label(x == ph, structure=S8)
    holders = [(sl, lab2[sl] == k) for k, sl in enumerate(ndi.find_objects(lab2), 1)]
    return temps, holders, bg


def _stamp(x, ph, allow_d4, any_color=False, remove=False):
    temps, holders, bg = _templates_and_holders(x, ph, any_color)
    if not temps or not holders:
        return None
    g = x.copy()
    if remove:                                         # 2026-10-18: the template itself disappears (321b1fc6)
        lab, n = ndi.label((x != bg) & (x != ph), structure=S8)
        g[lab > 0] = bg
    for sl, m in holders:
        fit = None
        for t in temps:
            for f in (D4 if allow_d4 else D4[:1]):
                tt = f(t)
                if tt.shape == m.shape and ((tt >= 0) == m).all():
                    fit = tt
                    break
            if fit is not None:
                break
        if fit is None:
            return None
        sub = g[sl]
        sub[m] = fit[m]
        g[sl] = sub
    return g


def learn_template(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    phs = [c for c in range(10) if all((x != y).any() and (x[x != y] == c).all() for x, y in train)]
    if not phs:                                        # with the template removed its own cells change too
        bg0 = bg_color(train[0][0])
        phs = [c for c in sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train])) if c != bg0]
    for ph in phs:
        for anyc in (False, True):          # 2026-10-18: single-color keys too (recolor blobs by the key of their shape)
            for d4 in (False, True):
                for rm in (False, True):
                    c = fit_all(train, tests, lambda x, ph=ph, d4=d4, anyc=anyc, rm=rm: _stamp(x, ph, d4, anyc, rm),
                                f"template[ph={ph},d4={d4},any={anyc},remove={rm}]", prior=1.4)
                    if c:
                        return [c]
    return []


LEARNERS = [learn_template, learn_local]


# ------------------------------------------------------------------------------------------------ batch 3b (2026-10-18)
def learn_stamp_center(train, tests):
    """Every pixel of a marker color is replaced by a (2r+1)^2 pattern INCLUDING its own cell (the marker may vanish);
    only background cells around it are painted."""
    if not same_shape(train) or not changed(train):
        return []
    bg = bg_color(train[0][0])
    for r in (1, 2):
        pats, ok = {}, True
        for x, y in train:
            H, W = x.shape
            for i, j in zip(*np.nonzero(x != bg)):
                c = int(x[i, j])
                P = pats.setdefault(c, {})
                for di in range(-r, r + 1):
                    for dj in range(-r, r + 1):
                        a, b = i + di, j + dj
                        if 0 <= a < H and 0 <= b < W and (x[a, b] == bg or (di, dj) == (0, 0)):
                            v = int(y[a, b])
                            if P.setdefault((di, dj), v) != v:
                                ok = False
            if not ok:
                break
        if not ok or not any(P.get((0, 0)) not in (None, c) for c, P in pats.items()):
            continue

        def predict(x, pats=pats):
            b = bg_color(x)
            g = x.copy()
            H, W = x.shape
            for i, j in zip(*np.nonzero(x != b)):
                P = pats.get(int(x[i, j]))
                if P is None:
                    return None
                for (di, dj), v in P.items():
                    a, bb = i + di, j + dj
                    if 0 <= a < H and 0 <= bb < W and (x[a, bb] == b or (di, dj) == (0, 0)):
                        g[a, bb] = v
            return g
        c = fit_all(train, tests, predict, f"stamp_center[r={r}]", prior=1.5)
        if c:
            return [c]
    return []


def learn_line_lookup(train, tests):
    """Each output row (or column) is a learned function of the same input row (column); every test line seen."""
    if not same_shape(train):
        return []
    found = []
    for axis in (0, 1):
        tab, ok = {}, True
        for x, y in train:
            X, Y = (x, y) if axis == 0 else (x.T, y.T)
            for a, b in zip(X, Y):
                k = a.tobytes()
                if tab.setdefault(k, b.tobytes()) != b.tobytes():
                    ok = False
                    break
            if not ok:
                break
        if not ok or all(k == v for k, v in tab.items()):
            continue

        def predict(x, axis=axis, tab=tab):
            X = x if axis == 0 else x.T
            rows = []
            for a in X:
                if a.tobytes() not in tab:
                    return None
                rows.append(np.frombuffer(tab[a.tobytes()], dtype=x.dtype))
            g = np.array(rows)
            return g if axis == 0 else g.T
        c = fit_all(train, tests, predict, f"line_lookup[axis={axis}]", prior=1.8)
        if c:
            found.append(c)
            break
    return found


def _gfeat(x):
    bg = bg_color(x)
    nb = x != bg
    f = {"ncolors": len(np.unique(x)), "n_fg": int(nb.sum()), "n_obj4": int(ndi.label(nb)[1]),
         "n_obj8": int(ndi.label(nb, structure=S8)[1])}
    for c in range(10):
        f[f"count{c}"] = int((x == c).sum())
        f[f"objs{c}"] = int(ndi.label(x == c)[1]) if c != bg else 0
        if c != bg and (x == c).any():                  # 2026-10-18: objects of color c by size class
            lab, n = ndi.label(x == c)
            sz = ndi.sum(np.ones_like(lab), lab, range(1, n + 1)) if n else []
            f[f"objs{c}_single"] = int(sum(1 for v in sz if v == 1))
            f[f"objs{c}_multi"] = int(sum(1 for v in sz if v > 1))
        else:
            f[f"objs{c}_single"] = f[f"objs{c}_multi"] = 0
    v, cnt = np.unique(x[nb], return_counts=True)
    f["major_fg"] = int(v[cnt.argmax()]) if len(v) else -1
    f["minor_fg"] = int(v[cnt.argmin()]) if len(v) else -1
    return f


def learn_output_lookup(train, tests):
    """The output is one of the train outputs, chosen by one grid-level count (colors, cells, objects, ...)."""
    outs = {y.tobytes() + bytes(y.shape) for _, y in train}
    if len(outs) == len(train) or len(outs) < 2:
        return []
    feats = [_gfeat(x) for x, _ in train]
    votes = {}
    for k in feats[0]:
        tab, ok = {}, True
        for f, (_, y) in zip(feats, train):
            if tab.setdefault(f[k], y.tobytes() + bytes(y.shape)) != y.tobytes() + bytes(y.shape):
                ok = False
                break
        if not ok:
            continue
        ys = {f[k]: y for f, (_, y) in zip(feats, train)}
        c = fit_all(train, tests, lambda x, k=k, ys=ys: ys.get(_gfeat(x)[k]), f"output_lookup[{k}]", prior=2.0)
        if c:
            votes.setdefault(str(c.outs), [0, c])[0] += 1
    # 2026-10-18: v1 (first feature that fit) was 12 right / 7 wrong on training -> features vote; weak if one vote
    out = []
    for n, c in sorted(votes.values(), key=lambda v: -v[0])[:2]:
        c.name += f"x{n}~weak"                     # always weak: 8 / 16 right on training even with 2+ votes
        c.prior = 2.2 - 0.02 * min(n, 6)
        out.append(c)
    return out


LEARNERS += [learn_stamp_center, learn_line_lookup, learn_output_lookup]


def learn_count_fill(train, tests):
    """Output of a fixed shape whose first k cells (in an order learned from the train outputs) take one color, with k a
    grid count of the input (objects, cells of a color, colors, ...)."""
    shapes = {y.shape for _, y in train}
    if len(shapes) != 1:
        return []
    shp = next(iter(shapes))
    bgs = {bg_color(x) for x, _ in train}                # the input's background (an output may be mostly fill)
    if len(bgs) != 1:
        return []
    bg = bgs.pop()
    fills = [(y != bg) for _, y in train]
    cols = {int(v) for (_, y), m in zip(train, fills) for v in np.unique(y[m])}
    if len(cols) != 1:
        return []
    col = cols.pop()
    freq = sum(m.astype(int) for m in fills)
    order = sorted(range(shp[0] * shp[1]), key=lambda p: (-freq.ravel()[p], p))
    ms = [int(m.sum()) for m in fills]
    for m, n in zip(fills, ms):
        if not all(m.ravel()[p] for p in order[:n]):
            return []
    feats = [_gfeat(x) for x, _ in train]
    for k in feats[0]:
        if not all(f[k] == n for f, n in zip(feats, ms)) or len(set(ms)) < 2:
            continue

        def predict(x, k=k):
            n = _gfeat(x)[k]
            if not 0 <= n <= len(order):
                return None
            g = np.full(shp, bg, x.dtype)
            g.ravel()[order[:n]] = col
            return g
        c = fit_all(train, tests, predict, f"count_fill[{k}]", prior=1.7)
        if c:
            return [c]
    return []


LEARNERS.append(learn_count_fill)


# ------------------------------------------------------------------------------------------------ template completion
def _template_variants(T, scales):
    """T: patch with -1 for empty. Yields (name, patch) for every D4 transform x integer upscale."""
    seen = set()
    for k, f in enumerate(D4):
        P = f(T)
        for s in scales:
            Q = np.kron(P, np.ones((s, s), dtype=P.dtype)) if s > 1 else P
            key = (Q.shape, Q.tobytes())
            if key in seen:
                continue
            seen.add(key)
            yield f"d{k}s{s}", Q


def _complete(x, anchor_mode, scales, remove_src=False, multi=False):
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S8)
    temps = []
    for k, sl in enumerate(ndi.find_objects(lab), 1):
        m = lab[sl] == k
        vv, cc = np.unique(x[sl][m], return_counts=True)
        nc = len(vv)
        if nc >= 2 and m.sum() >= 3 and cc.max() >= 2:   # a bare marker group (every color once) is not a template
            temps.append((nc, int(m.sum()), k, sl, m))
    if not temps:
        return None
    temps.sort(key=lambda t: (-t[0], -t[1]))
    temps = temps[:1] if not multi else temps[:4]
    g = x.copy()
    H, W = x.shape
    painted = False
    tmask = np.zeros(x.shape, bool)
    for _, _, k, sl, m in temps:
        tmask[sl] |= m
    for _, _, k, sl, m in temps:
        T = np.where(m, x[sl], -1)
        vals, cnt = np.unique(T[T >= 0], return_counts=True)
        main = int(vals[cnt.argmax()])
        if anchor_mode == "markers":
            anchors = [int(v) for v in vals if v != main]
        elif anchor_mode == "single":                # 2026-10-18: one uniquely-colored cell marks the copies
            ones = [int(v) for v, c in zip(vals, cnt) if c == 1]
            if len(ones) != 1:
                continue
            anchors = ones
        else:
            anchors = [main]
        for name, Q in _template_variants(T, scales):
            A = np.isin(Q, anchors)
            if not A.any():
                continue
            ai, aj = np.nonzero(A)
            av = Q[ai, aj]
            # vectorised offsets (2026-10-18): (i, j) such that every anchor cell lands in bounds on its color
            i0, i1 = -int(ai.min()), H - 1 - int(ai.max())
            j0, j1 = -int(aj.min()), W - 1 - int(aj.max())
            if i1 < i0 or j1 < j0:
                continue
            valid = np.ones((i1 - i0 + 1, j1 - j0 + 1), bool)
            for a_, b_, v_ in zip(ai, aj, av):
                blk = x[a_ + i0:a_ + i1 + 1, b_ + j0:b_ + j1 + 1]
                valid &= (blk == v_) & ~tmask[a_ + i0:a_ + i1 + 1, b_ + j0:b_ + j1 + 1]
            for vi, vj in zip(*np.nonzero(valid)):
                i, j = int(vi) + i0, int(vj) + j0
                if True:
                    qi, qj = np.nonzero(Q >= 0)
                    ti, tj = qi + i, qj + j
                    ok = (ti >= 0) & (ti < H) & (tj >= 0) & (tj < W)
                    ti, tj, tv = ti[ok], tj[ok], Q[qi[ok], qj[ok]]
                    free = x[ti, tj] == bg
                    if (x[ti, tj][~free] != tv[~free]).any():
                        continue                    # the placement clashes with what is already there
                    if free.any():
                        g[ti[free], tj[free]] = tv[free]
                        painted = True
    if painted and remove_src:                      # 0e206a2e-style: the source templates are moved, not copied
        g[tmask] = bg
    return g if painted else None


def learn_template_complete(train, tests):
    """A multicolor template; partial copies (only its marker colors, or only its main color) anywhere in the grid,
    in any rotation / reflection and at scale 1-3, are completed to the full template."""
    if not same_shape(train) or not changed(train):
        return []
    scales = (1, 2, 3)
    for multi in (False, True):
        for mode in ("markers", "main", "single"):
            for rm in (False, True):
                c = fit_all(train, tests, lambda x, mode=mode, rm=rm, mu=multi: _complete(x, mode, scales, rm, mu),
                            f"template_complete[{mode},remove_src={rm},multi={multi}]", prior=1.5)
                if c:
                    return [c]
    return []


LEARNERS.append(learn_template_complete)


def learn_pattern_recolor(train, tests):
    """The changed cells are copies of one fixed shape P, all of color c in the input, recolored to c' -- learned from
    the train diffs. Every occurrence of P in color c in the test input is recolored (overlaps allowed)."""
    if not same_shape(train) or not changed(train):
        return []
    shapes, cin, cout = set(), set(), set()
    for x, y in train:
        d = x != y
        cin |= set(np.unique(x[d]).tolist())
        cout |= set(np.unique(y[d]).tolist())
        lab, n = ndi.label(d, structure=S8)
        for k, sl in enumerate(ndi.find_objects(lab), 1):
            m = lab[sl] == k
            shapes.add((m.shape, m.tobytes()))
    if len(cin) != 1 or len(cout) != 1:
        return []
    c, c2 = cin.pop(), cout.pop()
    # the smallest component shape is the pattern (bigger ones are overlapping copies)
    shp = min(shapes, key=lambda s: int(np.frombuffer(s[1], dtype=bool).sum()))
    P = np.frombuffer(shp[1], dtype=bool).reshape(shp[0])
    if P.sum() < 3:
        return []
    pi, pj = np.nonzero(P)

    def predict(x):
        H, W = x.shape
        h, w = P.shape
        g = x.copy()
        hit = False
        for i in range(H - h + 1):
            for j in range(W - w + 1):
                if (x[pi + i, pj + j] == c).all():
                    g[pi + i, pj + j] = c2
                    hit = True
        return g if hit else None
    cand = fit_all(train, tests, predict, f"pattern_recolor[{c}->{c2},{int(P.sum())}cells]", prior=1.5)
    return [cand] if cand else []


LEARNERS.append(learn_pattern_recolor)


def learn_color_list(train, tests):
    """Output = one column (or row) listing every non-background color, each repeated by its cell count (or once),
    ordered by first appearance (row-major / column-major) or by count."""
    if not all(1 in y.shape for _, y in train):
        return []

    def seq(x, order, repeat):
        bg = bg_color(x)
        cols = [int(v) for v in np.unique(x) if v != bg]
        if not cols:
            return None
        cnt = {c: int((x == c).sum()) for c in cols}
        if order == "row":
            first = {c: np.argwhere(x == c)[0].tolist() for c in cols}
            cols.sort(key=lambda c: first[c])
        elif order == "col":
            first = {c: np.argwhere(x.T == c)[0].tolist() for c in cols}
            cols.sort(key=lambda c: first[c])
        elif order == "count_desc":
            cols.sort(key=lambda c: (-cnt[c], c))
        else:
            cols.sort(key=lambda c: (cnt[c], c))
        out = [c for c in cols for _ in range(cnt[c] if repeat else 1)]
        return out if 0 < len(out) <= 30 else None
    for order in ("row", "col", "count_desc", "count_asc"):
        for repeat in (True, False):
            for vertical in (True, False):
                def predict(x, order=order, repeat=repeat, vertical=vertical):
                    s = seq(x, order, repeat)
                    if s is None:
                        return None
                    g = np.array(s, dtype=x.dtype)
                    return g[:, None] if vertical else g[None, :]
                c = fit_all(train, tests, predict,
                            f"color_list[{order},repeat={repeat},{'col' if vertical else 'row'}]~weak", prior=1.9)
                if c:
                    return [c]
    return []


LEARNERS.append(learn_color_list)


# ------------------------------------------------------------------------------------------------ ray-feature tables
def _first_hits(x, dirs):
    """For every cell and direction: the color of the first non-background cell along the ray (-1 = border)."""
    bg = bg_color(x)
    H, W = x.shape
    out = np.full((len(dirs), H, W), -1, np.int16)
    for k, (di, dj) in enumerate(dirs):
        # sweep from the far side so each cell inherits its neighbour's answer
        ii = range(H) if di <= 0 else range(H - 1, -1, -1)
        jj = range(W) if dj <= 0 else range(W - 1, -1, -1)
        for i in ii:
            for j in jj:
                a, b = i + di, j + dj
                if 0 <= a < H and 0 <= b < W:
                    out[k, i, j] = x[a, b] if x[a, b] != bg else out[k, a, b]
    return out


DIR4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]
DIR8 = DIR4 + [(-1, -1), (-1, 1), (1, -1), (1, 1)]


def learn_ray_table(train, tests):
    """Each output cell = a lookup on (own color, first non-background color hit in each of 4 / 8 directions); rays,
    connections and fills-between in one table. Every test key must have been seen."""
    if not same_shape(train) or not changed(train):
        return []
    if sum(x.size for x, _ in train) > 4000:
        return []
    for dirs, dn in ((DIR4, "4"), (DIR8, "8")):
        for sym in (False, True):
            tab, ok = {}, True
            for x, y in train:
                F = _first_hits(x, dirs)
                for i in range(x.shape[0]):
                    for j in range(x.shape[1]):
                        f = tuple(F[:, i, j].tolist())
                        k = (int(x[i, j]), tuple(sorted(f)) if sym else f)
                        if tab.setdefault(k, int(y[i, j])) != int(y[i, j]):
                            ok = False
                            break
                    if not ok:
                        break
                if not ok:
                    break
            if not ok:
                continue

            def predict(x, dirs=dirs, sym=sym, tab=tab):
                F = _first_hits(x, dirs)
                g = x.copy()
                for i in range(x.shape[0]):
                    for j in range(x.shape[1]):
                        f = tuple(F[:, i, j].tolist())
                        k = (int(x[i, j]), tuple(sorted(f)) if sym else f)
                        if k not in tab:
                            return None
                        g[i, j] = tab[k]
                return g
            c = fit_all(train, tests, predict, f"ray_table[{dn},sym={sym}]", prior=1.7)
            if c:
                return [c]
    return []


LEARNERS.append(learn_ray_table)


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners4.py
"""Learners batch 4 (2026-10-18), from the unsolved growing training tasks:

  block_up   every input cell becomes a k x l block looked up by its color (upscale with per-color blocks)
  kron       self-similar: the grid tiled by its own mask -- block (i, j) = the grid (or its inverse, or background)
             depending on whether cell (i, j) is foreground
  extend     the output (shape learned as a*H + b, c*W + d) continues the input's smallest exact period, then one
             global color map
All take an optional global color map learned from the train pairs (cmap): structure first, colors second.
"""
import numpy as np

from learners import changed, bg_color, eq, fit_all, same_shape


def learn_cmap(pairs):
    """A single color map that turns each predicted grid into its target, or None."""
    m = {}
    for p, y in pairs:
        if p is None or p.shape != y.shape:
            return None
        for a, b in zip(p.ravel().tolist(), y.ravel().tolist()):
            if m.setdefault(a, b) != b:
                return None
    return m


def apply_cmap(g, m):
    if g is None:
        return None
    out = g.copy()
    for a, b in m.items():
        out[g == a] = b
    return out


def with_cmap(train, tests, struct, name, prior):
    """struct(x) -> grid; candidate if struct alone or struct + one learned color map fits every pair."""
    try:
        preds = [struct(x) for x, _ in train]
    except Exception:                                                  # noqa: BLE001
        return None
    if any(p is None for p in preds):
        return None
    if all(eq(p, y) for p, (_, y) in zip(preds, train)):
        return fit_all(train, tests, struct, name, prior=prior)
    m = learn_cmap(list(zip(preds, [y for _, y in train])))
    if m is None or all(a == b for a, b in m.items()):
        return None
    return fit_all(train, tests, lambda x: apply_cmap(struct(x), m) if all(int(v) in m for v in np.unique(struct(x))) else None,
                   name + "+cmap", prior=prior + 0.1)


def learn_block_up(train, tests):
    rs = {(y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) for x, y in train}
    if len(rs) != 1:
        return []
    kh, kw = rs.pop()
    if kh != int(kh) or kw != int(kw) or kh * kw < 2:
        return []
    kh, kw = int(kh), int(kw)
    if kh > 5 or kw > 5:
        return []
    tab, seen = {}, 0
    for x, y in train:
        for i in range(x.shape[0]):
            for j in range(x.shape[1]):
                b = y[i * kh:(i + 1) * kh, j * kw:(j + 1) * kw]
                c = int(x[i, j])
                if c in tab and not (tab[c] == b).all():
                    return []
                tab[c] = b
                seen += 1
    if seen < 2 * len(tab):                        # each block seen twice on average, else it is memorised
        return []

    def predict(x):
        if not all(int(c) in tab for c in np.unique(x)):
            return None
        return np.block([[tab[int(x[i, j])] for j in range(x.shape[1])] for i in range(x.shape[0])])
    c = fit_all(train, tests, predict, f"block_up[{kh}x{kw}]", prior=1.4)
    return [c] if c else []


def _kron(x, mode):
    bg = bg_color(x)
    H, W = x.shape
    if H * H > 30 or W * W > 30:
        return None
    fg = x != bg
    inv = np.where(fg, bg, x[fg][0] if fg.any() else bg)
    blank = np.full_like(x, bg)
    blk_on = {"self": x, "inv": inv}[mode[0]]
    blk_off = {"blank": blank, "self": x, "inv": inv}[mode[1]]
    return np.block([[blk_on if fg[i, j] else blk_off for j in range(W)] for i in range(H)])


def learn_kron(train, tests):
    if not all(y.shape == (x.shape[0] ** 2, x.shape[1] ** 2) for x, y in train):
        return []
    for mode in (("self", "blank"), ("inv", "blank"), ("self", "inv"), ("inv", "self")):
        c = with_cmap(train, tests, lambda x, mode=mode: _kron(x, mode), f"kron[{mode[0]}/{mode[1]}]", 1.4)
        if c:
            return [c]
    return []


def _period(v, axis):
    n = v.shape[axis]
    for p in range(1, n + 1):
        a = v[:n - p] if axis == 0 else v[:, :n - p]
        b = v[p:] if axis == 0 else v[:, p:]
        if (a == b).all():
            return p
    return n


def _extend(x, oh, ow):
    if not (1 <= oh <= 30 and 1 <= ow <= 30):
        return None
    pi, pj = _period(x, 0), _period(x, 1)
    I = np.arange(oh) % pi
    J = np.arange(ow) % pj
    return x[np.ix_(I, J)]


def _shape_rule(train):
    """(a, b, c, d) with out = (a*H + b, c*W + d), from the train pairs (a, c in 1, 2, 3, 1.5)."""
    for a in (1, 2, 3, 1.5):
        for c in (1, 2, 3, 1.5):
            bs = {y.shape[0] - a * x.shape[0] for x, y in train}
            ds = {y.shape[1] - c * x.shape[1] for x, y in train}
            if len(bs) == 1 and len(ds) == 1:
                b, d = bs.pop(), ds.pop()
                if b == int(b) and d == int(d):
                    return a, int(b), c, int(d)
    return None


def learn_extend(train, tests):
    if same_shape(train):
        return []
    r = _shape_rule(train)
    if r is None:
        return []
    a, b, c, d = r

    def struct(x):
        oh, ow = a * x.shape[0] + b, c * x.shape[1] + d
        if oh != int(oh) or ow != int(ow):
            return None
        return _extend(x, int(oh), int(ow))
    cand = with_cmap(train, tests, struct, f"extend[{a}H{b:+d},{c}W{d:+d}]", 1.5)
    return [cand] if cand else []


LEARNERS = [learn_block_up, learn_kron, learn_extend]


def learn_tile_cmap(train, tests):
    """Output = the input tiled k x l (plain, or every other copy mirrored), then an optional global color map."""
    rs = {(y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) for x, y in train}
    if len(rs) != 1:
        return []
    kh, kw = rs.pop()
    if kh != int(kh) or kw != int(kw) or kh * kw < 2:
        return []
    kh, kw = int(kh), int(kw)

    def mirror_tile(x):
        rows = []
        for i in range(kh):
            r = []
            for j in range(kw):
                b = x[::-1] if i % 2 else x
                b = b[:, ::-1] if j % 2 else b
                r.append(b)
            rows.append(r)
        return np.block(rows)
    for name, f in (("plain", lambda x: np.tile(x, (kh, kw))), ("mirror", mirror_tile)):
        c = with_cmap(train, tests, f, f"tile[{kh}x{kw},{name}]", 1.5)
        if c and "+cmap" in c.name:                 # plain tiles without a color map are wwhdr's / Hodel's job
            return [c]
    return []


LEARNERS.append(learn_tile_cmap)


def _obj_sym(x, kinds, conn):
    from scipy import ndimage as ndi
    bg = bg_color(x)
    fg = x != bg
    if conn == "all":
        lab = fg.astype(int)
    elif conn == "near":                                   # pieces up to one background cell apart are one object
        lab = ndi.label(ndi.binary_dilation(fg, np.ones((3, 3), bool)), structure=np.ones((3, 3), bool))[0] * fg
    else:
        lab, n = ndi.label(fg, structure=np.ones((3, 3), bool) if conn == 8 else None)
    g = x.copy()
    changed = False
    for k, sl in enumerate(ndi.find_objects(lab), 1):
        if sl is None:
            continue
        sub = np.where(lab[sl] == k, x[sl], bg)
        out = sub.copy()
        for kind in kinds:
            img = {"lr": out[:, ::-1], "ud": out[::-1], "tr": out.T if out.shape[0] == out.shape[1] else None,
                   "r90": np.rot90(out) if out.shape[0] == out.shape[1] else None}[kind]
            if img is None:
                return None
            fill = (out == bg) & (img != bg)
            clash = (out != bg) & (img != bg) & (out != img)
            if clash.any():
                return None
            out = np.where(fill, img, out)
        region = g[sl]
        m = (region == bg) & (out != bg)
        if m.any():
            region[m] = out[m]
            changed = True
    return g if changed else None


def learn_obj_symmetrize(train, tests):
    """Each object completed to be mirror / rotation symmetric inside its own bounding box."""
    if not same_shape(train) or not changed(train):
        return []
    for conn in (8, 4, "near", "all"):
        out = []                                           # strongest symmetry group first, a weaker one as 2nd guess
        for i, kinds in enumerate((("lr", "ud", "tr"), ("r90", "r90", "r90"), ("lr", "ud"), ("tr",), ("lr",), ("ud",))):
            c = fit_all(train, tests, lambda x, kinds=kinds, conn=conn: _obj_sym(x, kinds, conn),
                        f"obj_sym[{'+'.join(kinds)},{conn}]", prior=1.5 + 0.01 * i)
            if c and all(c.outs != o.outs for o in out):
                out.append(c)
            if len(out) == 2:
                break
        if out:
            return out
    return []


LEARNERS.append(learn_obj_symmetrize)


def _progression(x, axis, marker_mode):
    """One line at a time. The step d = gcd of the gaps between the pixels of the line's main color (the color with
    >= 2 pixels). A single other-colored pixel (marker) lying on that progression: the run spans group + marker and
    takes the marker's color. Otherwise the progression continues to both borders in its own color."""
    from math import gcd
    bg = bg_color(x)
    X = x if axis == 0 else x.T
    g = X.copy()
    changed = False
    n = X.shape[1]
    for r, line in enumerate(X):
        pos = [int(p) for p in np.nonzero(line != bg)[0]]
        if len(pos) < 2:
            continue
        cols = {}
        for p in pos:
            cols.setdefault(int(line[p]), []).append(p)
        main = [c for c, ps in cols.items() if len(ps) >= 2]
        others = [c for c, ps in cols.items() if len(ps) == 1]
        if len(main) != 1 or len(others) > (1 if marker_mode else 0):
            return None
        k = main[0]
        ps = cols[k]
        d = 0
        for u, v in zip(ps, ps[1:]):
            d = gcd(d, v - u)
        if others and (cols[others[0]][0] - ps[0]) % d == 0:
            m = cols[others[0]][0]
            lo, hi, k = min(ps[0], m), max(ps[-1], m), others[0]
        else:
            lo, hi = ps[0] % d, n - 1
        for p in range(lo, hi + 1, d):
            if g[r, p] == bg or (others and k == others[0]):
                if g[r, p] != k:
                    g[r, p] = k
                    changed = True
    if not changed:
        return None
    return g if axis == 0 else g.T


def learn_progression(train, tests):
    """Per row / column: the non-background pixels define a step (gcd of their gaps); a line of one color continues to
    both borders; a line with one odd-colored end marker is filled from its first to its last pixel in that color."""
    if not same_shape(train) or not changed(train):
        return []
    for axis in (0, 1, "auto"):
        for marker in (True, False):
            def predict(x, axis=axis, marker=marker):
                if axis != "auto":
                    return _progression(x, axis, marker)
                a = _progression(x, 0, marker)
                return a if a is not None else _progression(x, 1, marker)
            c = fit_all(train, tests, predict, f"progression[axis={axis},marker={marker}]", prior=1.6)
            if c:
                return [c]
    return []


LEARNERS.append(learn_progression)


def _domino_key(x):
    """Isolated 2-cell objects of two different non-background colors: (cells, {a: b}) read left->right / top->bottom,
    in both directions as candidates."""
    from scipy import ndimage as ndi
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=np.ones((3, 3), bool))
    pairs, cells = [], np.zeros(x.shape, bool)
    for k, sl in enumerate(ndi.find_objects(lab), 1):
        m = lab[sl] == k
        if m.sum() != 2 or m.size != 2:
            continue
        v = x[sl][m]
        if v[0] == v[1]:
            continue
        pairs.append((int(v[0]), int(v[1])))
        cells[sl] |= m
    return pairs, cells


def _legend(x, direction, erase):
    pairs, cells = _domino_key(x)
    if not pairs:
        return None
    m = {}
    for a, b in pairs:
        a, b = (a, b) if direction == 0 else (b, a)
        if m.setdefault(a, b) != b:
            return None
    g = x.copy()
    rest = ~cells
    for a, b in m.items():
        g[rest & (x == a)] = b
    if erase:
        g[cells] = bg_color(x)
    return None if (g == x).all() else g


def learn_legend(train, tests):
    """Recolor by a key of two-color dominoes (a next to b: every other a becomes b); the key kept or erased."""
    if not same_shape(train) or not changed(train):
        return []
    for direction in (0, 1):
        for erase in (False, True):
            c = fit_all(train, tests, lambda x, d=direction, e=erase: _legend(x, d, e),
                        f"legend[dir={direction},erase={erase}]", prior=1.5)
            if c:
                return [c]
    return []


LEARNERS.append(learn_legend)


def _anchor_crop(x, h, w, anchor):
    H, W = x.shape
    if h > H or w > W:
        return None
    r = {"tl": (0, 0), "tr": (0, W - w), "bl": (H - h, 0), "br": (H - h, W - w),
         "c": ((H - h) // 2, (W - w) // 2)}[anchor]
    return x[r[0]:r[0] + h, r[1]:r[1] + w]


def learn_anchor_crop(train, tests):
    """Output = a fixed-size (or fixed-fraction) block at a corner or the center of the input."""
    if same_shape(train):
        return []
    shapes = {y.shape for _, y in train}
    fracs = {(x.shape[0] / y.shape[0], x.shape[1] / y.shape[1]) for x, y in train}
    rules = []
    if len(shapes) == 1:
        h, w = next(iter(shapes))
        rules.append(("fixed", lambda x, h=h, w=w: (h, w)))
    if len(fracs) == 1:
        fh, fw = next(iter(fracs))
        rules.append(("frac", lambda x, fh=fh, fw=fw: (int(round(x.shape[0] / fh)), int(round(x.shape[1] / fw)))))
    for rname, rule in rules:
        for anchor in ("tl", "tr", "bl", "br", "c"):
            c = fit_all(train, tests, lambda x, rule=rule, anchor=anchor: _anchor_crop(x, *rule(x), anchor),
                        f"anchor_crop[{rname},{anchor}]", prior=1.6)
            if c:
                return [c]
    return []


def _corner_overlay(x, h, w, order):
    H, W = x.shape
    if h > H or w > W:
        return None
    bg = bg_color(x)
    blocks = {"tl": x[:h, :w], "tr": x[:h, W - w:], "bl": x[H - h:, :w], "br": x[H - h:, W - w:]}
    g = np.full((h, w), bg, x.dtype)
    for k in order:
        b = blocks[k]
        g = np.where(b != bg, b, g)
    return g


def learn_corner_overlay(train, tests):
    """Output (fixed shape) = the four corner blocks of the input laid over each other (non-background wins)."""
    shapes = {y.shape for _, y in train}
    if same_shape(train) or len(shapes) != 1:
        return []
    h, w = next(iter(shapes))
    if h * w < 4:                                  # 1x1 "overlays" fit by accident (d9fac9be)
        return []
    for order in (("tl", "tr", "bl", "br"), ("br", "bl", "tr", "tl")):
        c = fit_all(train, tests, lambda x, order=order: _corner_overlay(x, h, w, order),
                    f"corner_overlay[{h}x{w}]", prior=1.6)
        if c:
            return [c]
    return []


LEARNERS += [learn_anchor_crop, learn_corner_overlay]


def learn_pad(train, tests):
    """Output = input padded by k cells on every side (numpy pad modes), optionally with blank corners."""
    ks = {((y.shape[0] - x.shape[0]) / 2, (y.shape[1] - x.shape[1]) / 2) for x, y in train}
    if len(ks) != 1:
        return []
    kh, kw = ks.pop()
    if kh != int(kh) or kw != int(kw) or kh < 1 or kw < 1 or kh > 5 or kw > 5:
        return []
    kh, kw = int(kh), int(kw)
    for mode in ("edge", "constant", "reflect", "symmetric", "wrap"):
        for corners in (False, True):
            def predict(x, mode=mode, corners=corners):
                if mode == "constant":
                    g = np.pad(x, ((kh, kh), (kw, kw)), mode="constant", constant_values=bg_color(x))
                else:
                    if mode == "reflect" and (x.shape[0] <= kh or x.shape[1] <= kw):
                        return None
                    g = np.pad(x, ((kh, kh), (kw, kw)), mode=mode)
                if corners:
                    for a in (slice(0, kh), slice(-kh, None)):
                        for b in (slice(0, kw), slice(-kw, None)):
                            g[a, b] = 0
                return g
            c = with_cmap(train, tests, predict, f"pad[{kh}x{kw},{mode},corners0={corners}]", 1.5)
            if c:
                return [c]
    return []


LEARNERS.append(learn_pad)


def learn_block_pattern(train, tests):
    """The grid splits into equal k x l blocks (or separator cells); every output block is one solid color looked up
    from the input block's exact pattern (as a 0/1 foreground mask)."""
    from learners import sep_cells
    if not same_shape(train):
        return []
    for bh, bw in ((3, 3), (2, 2), (4, 4), (5, 5), ("sep", None)):
        def blocks(x, bh=bh, bw=bw):
            if bh == "sep":
                cells = sep_cells(x)
                return [(slice(a, b), slice(c, d)) for a, b, c, d in cells] if len(cells) >= 2 else None
            H, W = x.shape
            if H % bh or W % bw or H * W == bh * bw:
                return None
            return [(slice(i, i + bh), slice(j, j + bw)) for i in range(0, H, bh) for j in range(0, W, bw)]
        tab, ok = {}, True
        for x, y in train:
            bs = blocks(x)
            if bs is None:
                ok = False
                break
            bg = bg_color(x)
            for sl in bs:
                v = np.unique(y[sl])
                if len(v) != 1:
                    ok = False
                    break
                k = ((x[sl] != bg).tobytes(), x[sl].shape)
                if tab.setdefault(k, int(v[0])) != int(v[0]):
                    ok = False
                    break
            if not ok:
                break
        if not ok or len(set(tab.values())) < 2:
            continue

        def predict(x, blocks=blocks, tab=tab):
            bs = blocks(x)
            if bs is None:
                return None
            bg = bg_color(x)
            g = x.copy()
            for sl in bs:
                k = ((x[sl] != bg).tobytes(), x[sl].shape)
                if k not in tab:
                    return None
                g[sl] = tab[k]
            return g
        c = fit_all(train, tests, predict, f"block_pattern[{bh}x{bw}]" if bh != "sep" else "block_pattern[sep]", prior=1.6)
        if c:
            return [c]
    return []


def learn_shift_cmap(train, tests):
    """The whole grid translated by (di, dj) (vacated cells = background), then an optional global color map."""
    if not same_shape(train) or not changed(train):
        return []
    for di in range(-3, 4):
        for dj in range(-3, 4):
            if (di, dj) == (0, 0):
                continue

            def struct(x, di=di, dj=dj):
                bg = bg_color(x)
                g = np.full_like(x, bg)
                H, W = x.shape
                src = x[max(0, -di):H - max(0, di), max(0, -dj):W - max(0, dj)]
                g[max(0, di):max(0, di) + src.shape[0], max(0, dj):max(0, dj) + src.shape[1]] = src
                return g
            c = with_cmap(train, tests, struct, f"shift[{di},{dj}]", 1.6)
            if c:
                return [c]
    return []


LEARNERS += [learn_block_pattern, learn_shift_cmap]


def _quad_key(x, mask_color, where="block"):
    """The key: a 2x2 block of non-background cells (no separator-line color, no mask color), or the grid's four
    corners. The cells of mask_color (cropped to their bbox) take, per quadrant of that bbox, the key's color for that
    quadrant."""
    bg = bg_color(x)
    H, W = x.shape
    lines = {int(x[i, 0]) for i in range(H) if (x[i] == x[i, 0]).all()} |             {int(x[0, j]) for j in range(W) if (x[:, j] == x[0, j]).all()}
    lines.discard(bg)
    if where == "corners":
        K = np.array([[x[0, 0], x[0, -1]], [x[-1, 0], x[-1, -1]]])
        if (K == bg).any() or mask_color in K:
            return None
    else:
        keys = []
        for i in range(H - 1):
            for j in range(W - 1):
                b = x[i:i + 2, j:j + 2]
                if (b != bg).all() and mask_color not in b and not (set(np.unique(b).tolist()) & lines)                         and len(np.unique(b)) >= 2:
                    keys.append(b)
        if len(keys) != 1:
            return None
        K = keys[0]
    from learners import bbox
    bb = bbox(x == mask_color)
    if bb is None:
        return None
    r0, r1, c0, c1 = bb
    M = x[r0:r1, c0:c1] == mask_color
    h, w = M.shape
    if h % 2 or w % 2:
        return None
    g = x[r0:r1, c0:c1].copy()                      # non-mask cells keep their own color
    for qi in range(2):
        for qj in range(2):
            sl = (slice(qi * h // 2, (qi + 1) * h // 2), slice(qj * w // 2, (qj + 1) * w // 2))
            g[sl] = np.where(M[sl], K[qi, qj], g[sl])
    return g


def learn_quad_key(train, tests):
    if same_shape(train):
        return []
    for c in range(10):
        if not all((x == c).any() for x, _ in train):
            continue
        for where in ("block", "corners"):
            cand = fit_all(train, tests, lambda x, c=c, w=where: _quad_key(x, c, w), f"quad_key[mask={c},{where}]",
                           prior=1.5)
            if cand:
                return [cand]
    return []


LEARNERS.append(learn_quad_key)


def _key_swap(x, pairing):
    """A 2x2 key of four non-background cells in a grid corner; its rows (or columns) are color pairs swapped both
    ways everywhere else."""
    bg = bg_color(x)
    H, W = x.shape
    for r, c in ((0, 0), (0, W - 2), (H - 2, 0), (H - 2, W - 2)):
        K = x[r:r + 2, c:c + 2]
        if (K == bg).any() or len(np.unique(K)) != 4:
            continue
        pairs = [(K[0, 0], K[0, 1]), (K[1, 0], K[1, 1])] if pairing == "rows" else [(K[0, 0], K[1, 0]), (K[0, 1], K[1, 1])]
        m = {}
        for a, b in pairs:
            m[int(a)], m[int(b)] = int(b), int(a)
        g = x.copy()
        rest = np.ones(x.shape, bool)
        rest[r:r + 2, c:c + 2] = False
        for a, b in m.items():
            g[rest & (x == a)] = b
        return None if (g == x).all() else g
    return None


def learn_key_swap(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    for pairing in ("rows", "cols"):
        c = fit_all(train, tests, lambda x, p=pairing: _key_swap(x, p), f"key_swap[{pairing}]", prior=1.5)
        if c:
            return [c]
    return []


LEARNERS.append(learn_key_swap)


def learn_shear(train, tests):
    """Output = input with row i shifted right by i (or by H-1-i), width W + H - 1, background elsewhere."""
    if not all(y.shape == (x.shape[0], x.shape[1] + x.shape[0] - 1) for x, y in train):
        return []
    pads = {int(np.bincount(y.ravel(), minlength=10).argmax()) for _, y in train} | {0}
    for rev, pad in [(r, p) for r in (False, True) for p in sorted(pads)] + [(r, "bg") for r in (False, True)]:
        def predict(x, rev=rev, pad=pad):
            H, W = x.shape
            g = np.full((H, W + H - 1), bg_color(x) if pad == "bg" else pad, x.dtype)
            for i in range(H):
                o = (H - 1 - i) if rev else i
                g[i, o:o + W] = x[i]
            return g
        c = with_cmap(train, tests, predict, f"shear[rev={rev},pad={pad}]", 1.5)
        if c:
            return [c]
    return []


_HALF_T = {"id": lambda g: g, "flr": lambda g: g[:, ::-1], "fud": lambda g: g[::-1], "r180": lambda g: g[::-1, ::-1]}


def learn_concat(train, tests):
    """Output = two copies of the input side by side (or stacked), each a reflection / rotation of it with its own
    color map."""
    for axis in (1, 0):
        if not all(y.shape == ((x.shape[0], 2 * x.shape[1]) if axis == 1 else (2 * x.shape[0], x.shape[1]))
                   for x, y in train):
            continue
        parts = []
        for half in (0, 1):
            best = None
            for tn, tf in _HALF_T.items():
                pairs = []
                for x, y in train:
                    H, W = x.shape
                    yh = y[:, half * W:(half + 1) * W] if axis == 1 else y[half * H:(half + 1) * H]
                    pairs.append((tf(x), yh))
                m = learn_cmap(pairs)
                if m is not None:
                    best = (tn, m)
                    break
            if best is None:
                break
            parts.append(best)
        if len(parts) != 2:
            continue

        def predict(x, parts=parts, axis=axis):
            hs = []
            for tn, m in parts:
                h = _HALF_T[tn](x)
                if not all(int(v) in m for v in np.unique(h)):
                    return None
                hs.append(apply_cmap(h, m))
            return np.hstack(hs) if axis == 1 else np.vstack(hs)
        c = fit_all(train, tests, predict, f"concat[axis={axis},{parts[0][0]}|{parts[1][0]}]", prior=1.5)
        if c:
            return [c]
    return []


LEARNERS += [learn_shear, learn_concat]


def _prog2d(x, c, both):
    """The non-background pixels (one color, >= 2, isolated) lie on one line with a constant step: the progression
    continues to the border (forward, or both ways) in color c (None = their own color)."""
    bg = bg_color(x)
    pts = np.argwhere(x != bg)
    if len(pts) < 2 or len(np.unique(x[x != bg])) != 1:
        return None
    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]
    d = pts[1] - pts[0]
    if not (np.diff(pts, axis=0) == d).all():
        return None
    col = int(x[tuple(pts[0])]) if c is None else c
    g = x.copy()
    H, W = x.shape
    ends = [(pts[-1], d)] + ([(pts[0], -d)] if both else [])
    hit = False
    for p, step in ends:
        q = p + step
        while 0 <= q[0] < H and 0 <= q[1] < W:
            g[q[0], q[1]] = col
            hit = True
            q = q + step
    return g if hit else None


def learn_prog2d(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    added = set()
    for x, y in train:
        added |= set(np.unique(y[x != y]).tolist())
    cs = [None] + ([added.pop()] if len(added) == 1 else [])
    for c in cs:
        for both in (False, True):
            k = fit_all(train, tests, lambda x, c=c, b=both: _prog2d(x, c, b), f"prog2d[{c},both={both}]", prior=1.6)
            if k:
                return [k]
    return []


LEARNERS.append(learn_prog2d)


def _gap_feats(x):
    """Per cell: (value, row empty, column empty, row inside the occupied row span, column inside the occupied
    column span) -- empty = all background."""
    bg = bg_color(x)
    fg = x != bg
    if not fg.any():
        return None
    re, ce = ~fg.any(1), ~fg.any(0)
    rs, cs = np.nonzero(~re)[0], np.nonzero(~ce)[0]
    ir = (np.arange(x.shape[0]) >= rs[0]) & (np.arange(x.shape[0]) <= rs[-1])
    ic = (np.arange(x.shape[1]) >= cs[0]) & (np.arange(x.shape[1]) <= cs[-1])
    H, W = x.shape
    return np.stack([x, np.repeat(re[:, None], W, 1), np.repeat(ce[None], H, 0),
                     np.repeat(ir[:, None], W, 1), np.repeat(ic[None], H, 0)], -1).reshape(-1, 5)


def learn_gap_table(train, tests):
    """Color of each cell from a table over its gap features (fills between / beside blocks along empty lines)."""
    if not same_shape(train) or not changed(train):
        return []
    tab = {}
    for x, y in train:
        f = _gap_feats(x)
        if f is None or not (f[:, 1].any() or f[:, 2].any()):
            return []
        for k, v in zip(map(tuple, f.tolist()), y.ravel().tolist()):
            if tab.setdefault(k, v) != v:
                return []
    if all(k[0] == v for k, v in tab.items()):
        return []
    keep = all(k[0] == v for k, v in tab.items() if not (k[1] or k[2]))     # cells off empty lines never change

    def predict(x):
        f = _gap_feats(x)
        if f is None:
            return None
        out = []
        for k in map(tuple, f.tolist()):
            if k not in tab:
                if k[1] or k[2] or not keep:
                    return None                            # an unseen case that train says nothing about
                out.append(k[0])
                continue
            out.append(tab[k])
        return np.array(out).reshape(x.shape)
    c = fit_all(train, tests, predict, "gap_table", prior=1.7)
    return [c] if c else []


LEARNERS.append(learn_gap_table)


def _drip(x, d, cond):
    """In each line whose far end (direction d) satisfies cond (None = every line, -1 = any non-background end, else
    that color), every background cell takes the color of the nearest non-background cell in direction d."""
    bg = bg_color(x)
    k = {"down": 0, "right": 1, "up": 2, "left": 3}[d]
    X = {0: x, 1: x.T, 2: x[::-1], 3: x.T[::-1]}[k]       # direction d becomes "down" along columns
    g = X.copy()
    hit = False
    for j in range(X.shape[1]):
        col = X[:, j]
        end = col[-1]
        if cond is not None and (end == bg if cond == -1 else end != cond):
            continue
        nxt = bg
        for i in range(len(col) - 1, -1, -1):
            if col[i] != bg:
                nxt = col[i]
            elif nxt != bg:
                g[i, j] = nxt
                hit = True
    if not hit:
        return None
    return {0: g, 1: g.T, 2: g[::-1], 3: g[::-1].T}[k]


def learn_drip(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    if not all(((x == y) | (x == bg_color(x))).all() for x, y in train):
        return []
    cols = set()
    for x, _ in train:
        cols |= set(np.unique(x).tolist())
    for d in ("down", "up", "right", "left"):
        for cond in [-1] + sorted(cols) + [None]:
            c = fit_all(train, tests, lambda x, d=d, cond=cond: _drip(x, d, cond), f"drip[{d},{cond}]", prior=1.6)
            if c:
                return [c]
    return []


LEARNERS.append(learn_drip)


def _open_lines(x, c, axes):
    """Rows / columns whose cells are all background except the two end cells are filled with c."""
    bg = bg_color(x)
    g = x.copy()
    hit = False
    if "h" in axes:
        for i in range(x.shape[0]):
            if x.shape[1] > 2 and (x[i, 1:-1] == bg).all() and x[i, 0] != bg and x[i, -1] != bg:
                g[i, 1:-1] = c
                hit = True
    if "v" in axes:
        for j in range(x.shape[1]):
            if x.shape[0] > 2 and (x[1:-1, j] == bg).all() and x[0, j] != bg and x[-1, j] != bg:
                g[1:-1, j] = np.where(g[1:-1, j] == bg, c, g[1:-1, j])
                hit = True
    return g if hit else None


def learn_open_lines(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    added = set()
    for x, y in train:
        added |= set(np.unique(y[x != y]).tolist())
    if len(added) != 1:
        return []
    c0 = added.pop()
    for axes in ("hv", "h", "v"):
        k = fit_all(train, tests, lambda x, a=axes: _open_lines(x, c0, a), f"open_lines[{c0},{axes}]", prior=1.6)
        if k:
            return [k]
    return []


LEARNERS.append(learn_open_lines)


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners5.py
"""Learners, part 5 (2026-10-18): per-cell feature tables and object picks.

cell_table: the output color of each cell comes from a table over (its value?, one or two feature atoms) chosen from a
bank of ~25 cell features (neighbour counts, mirror partner, parity, object size / border contact / bbox edge,
enclosed background, row / column counts and uniformity, inside the bbox of the rarest color, ...). Atoms are raw
values or learned thresholds; the table output is 'keep' or a color, so value-free tables carry to unseen colors.
A table that does not survive leave-one-out over the train pairs is tagged ~weak.

obj_pick: the output is the crop of one object, picked by a feature that makes it unique / max / min; every rule that
reproduces train votes, answers backed only by position rules are ~weak.

Row 74: d4_tile (output blocks = a learned rotation / reflection of the input per block position), kron_sel (input
copied into the blocks of cells with a learned color / the most / least common color / non-black), var_grow (tile or
upscale by a count read off the input), pointer (an object's single odd-colored cell shoots a ray out of the
opposite side).
"""
import json
from collections import Counter

import numpy as np
from scipy import ndimage as ndi

from common import Cand
from learners import bg_color, changed, fit_all, same_shape

S8 = np.ones((3, 3), bool)
FEATS = ["n4", "n8", "n8same", "mir_lr", "mir_ud", "pi", "pj", "pij", "size8", "size4", "border8", "bbox_edge",
         "enclosed", "row_n", "col_n", "largest", "smallest", "oi2", "oj2", "row_empty", "col_empty", "sees", "row_uni",
         "col_uni", "in_other_bbox", "in_minor_bbox", "fg", "nb_size", "nb_size4"]
CFEATS = ["row_first", "row_last", "col_first", "col_last", "nb4col", "nb8col"]   # color-valued, 10 = none


def cell_feats(x, bg=None):
    """(H*W, len(FEATS)) int array, every value clipped to 0..63."""
    bg = bg_color(x) if bg is None else bg
    H, W = x.shape
    fg = x != bg
    P = np.pad(fg, 1)
    n4 = P[:-2, 1:-1].astype(int) + P[2:, 1:-1] + P[1:-1, :-2] + P[1:-1, 2:]
    n8 = sum(np.roll(np.roll(np.pad(fg, 1), a, 0), b, 1)[1:-1, 1:-1].astype(int)
             for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0))
    Q = np.pad(x, 1, constant_values=-1)
    n8s = sum((Q[1 + a:H + 1 + a, 1 + b:W + 1 + b] == x).astype(int)
              for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0))
    mlr = (x == x[:, ::-1]).astype(int)
    mud = (x == x[::-1]).astype(int)
    I, J = np.indices(x.shape)
    size8 = np.zeros(x.shape, int)
    size4 = np.zeros(x.shape, int)
    bor8 = np.zeros(x.shape, int)
    edge = np.zeros(x.shape, int)
    oi2 = np.zeros(x.shape, int)
    oj2 = np.zeros(x.shape, int)
    larg = np.zeros(x.shape, int)
    small = np.zeros(x.shape, int)
    sizes = []
    for c in np.unique(x):
        if c == bg:
            continue
        for st, out in ((S8, size8), (None, size4)):
            lab, n = ndi.label(x == c, structure=st)
            cnt = np.bincount(lab.ravel())
            m = lab > 0
            out[m] = cnt[lab[m]]
            if st is S8:
                for k, sl in enumerate(ndi.find_objects(lab), 1):
                    mk = lab[sl] == k
                    sizes.append((cnt[k], c, k, sl, mk))
                    r0, r1, c0, c1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
                    sub = np.zeros(mk.shape, int)
                    sub[0, :] = sub[-1, :] = 1
                    sub[:, 0] = sub[:, -1] = 1
                    edge[sl][mk] = sub[mk]
                    oi2[sl][mk] = ((I[sl] - r0) % 2)[mk]
                    oj2[sl][mk] = ((J[sl] - c0) % 2)[mk]
                    if r0 == 0 or c0 == 0 or r1 == H or c1 == W:
                        bor8[sl][mk] = 1
    inob = np.zeros(x.shape, bool)
    for s_, c, k, sl, mk in sizes:
        cov = np.zeros(x.shape, bool)
        cov[sl] = True
        inob |= cov & (x != c)
    cnt_c = np.bincount(x.ravel(), minlength=10)
    present = [c for c in range(10) if cnt_c[c]]
    inmin = np.zeros(x.shape, bool)
    if len(present) > 1:
        mc = min(present, key=lambda c: (cnt_c[c], c))
        ys_, xs_ = np.nonzero(x == mc)
        inmin[ys_.min():ys_.max() + 1, xs_.min():xs_.max() + 1] = True
        inmin &= x != mc
    if sizes:
        mx, mn = max(s[0] for s in sizes), min(s[0] for s in sizes)
        for s, c, k, sl, mk in sizes:
            if s == mx:
                larg[sl][mk] = 1
            if s == mn:
                small[sl][mk] = 1
    lab, n = ndi.label(~fg)
    outside = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())
    enc = (~fg & ~np.isin(lab, list(outside))).astype(int)
    rn = np.repeat(fg.sum(1)[:, None], W, 1)
    cn = np.repeat(fg.sum(0)[None], H, 0)
    re = (rn == 0).astype(int)
    ce = (cn == 0).astype(int)
    # sees: number of the 4 directions with a non-background cell somewhere along the line
    up = np.maximum.accumulate(fg, 0)
    up = np.vstack([np.zeros((1, W), bool), up[:-1]])
    dn = np.maximum.accumulate(fg[::-1], 0)[::-1]
    dn = np.vstack([dn[1:], np.zeros((1, W), bool)])
    lf = np.maximum.accumulate(fg, 1)
    lf = np.hstack([np.zeros((H, 1), bool), lf[:, :-1]])
    rt = np.maximum.accumulate(fg[:, ::-1], 1)[:, ::-1]
    rt = np.hstack([rt[:, 1:], np.zeros((H, 1), bool)])
    sees = up.astype(int) + dn + lf + rt
    F = np.stack([n4, n8, n8s, mlr, mud, I % 2, J % 2, (I + J) % 2, size8, size4, bor8, edge, enc, rn, cn, larg,
                  small, oi2, oj2, re, ce, sees, np.repeat((x == x[:, :1]).all(1)[:, None], W, 1),
                  np.repeat((x == x[:1]).all(0)[None], H, 0), inob, inmin, fg, _nb_max(np.where(fg, size8, 0)),
                  _nb_max(np.where(fg, size4, 0))],
                 -1).reshape(-1, len(FEATS))
    return np.clip(F, 0, 63)


NUMERIC = {"n4", "n8", "n8same", "size8", "size4", "row_n", "col_n", "sees", "nb_size", "nb_size4"}


def _nb_max(v):
    """Per cell: the largest value among its 8 neighbours (0 outside the grid)."""
    P = np.pad(v, 1)
    H, W = v.shape
    return np.max([P[1 + a:H + 1 + a, 1 + b:W + 1 + b] for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0)], 0)


def _nb_color(x, bg, diag):
    """Per cell: the one non-background color among its 4 (8) neighbours, 10 if none or several."""
    H, W = x.shape
    P = np.pad(x, 1, constant_values=bg)
    offs = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diag else [])
    N = np.stack([P[1 + a:H + 1 + a, 1 + b:W + 1 + b] for a, b in offs])
    out = np.full(x.shape, 10)
    for c in range(10):
        if c == bg:
            continue
        has = (N == c).any(0)
        out = np.where(has & (out == 10), c, np.where(has, 11, out))
    return np.where(out == 11, 10, out)


def color_feats(x, bg=None):
    """(H*W, len(CFEATS)) colors a table entry may copy."""
    bg = bg_color(x) if bg is None else bg
    H, W = x.shape
    return np.stack([np.repeat(x[:, :1], W, 1), np.repeat(x[:, -1:], W, 1), np.repeat(x[:1], H, 0),
                     np.repeat(x[-1:], H, 0), _nb_color(x, bg, False), _nb_color(x, bg, True)],
                    -1).reshape(-1, len(CFEATS))


def _atoms(Ftr):
    """Raw features (exact value) and thresholds f >= t for the numeric ones (t from the train values)."""
    allF = np.concatenate(Ftr)
    at = [(f, None) for f in range(len(FEATS))]
    for f, name in enumerate(FEATS):
        if name in NUMERIC:
            vals = np.unique(allF[:, f])
            at += [(f, int(t)) for t in vals[1:][:8]]
    return at


def _col(F, a):
    f, t = a
    if t is None:
        return F[:, f]
    if isinstance(t, tuple):                             # ("eq", c): a color feature equals c
        return (F[:, f] == t[1]).astype(int)
    return (F[:, f] >= t).astype(int)


def _key(x, F, atoms, use_v):
    k = x.ravel().astype(np.int64) if use_v else np.zeros(x.size, np.int64)
    for a in atoms:
        k = k * 64 + _col(F, a)
    return k


def _aname(a):
    if a[0] >= len(FEATS):
        return f"{CFEATS[a[0] - len(FEATS)]}=={a[1][1]}"
    return FEATS[a[0]] if a[1] is None else f"{FEATS[a[0]]}>={a[1]}"


def _fit_tab(keys, xs, ys, Cs, f):
    """key -> symbol (-1 keep, 100 copy color feature f, else that color), the most specific consistent one per key;
    None if some key fits none."""
    k = np.concatenate(keys)
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    u, inv = np.unique(k, return_inverse=True)
    keep = np.ones(len(u), bool)
    np.logical_and.at(keep, inv, y == x)
    cp = np.zeros(len(u), bool)
    if f is not None:
        cf = np.concatenate([C[:, f] for C in Cs])
        cp[:] = True
        np.logical_and.at(cp, inv, y == cf)
    lo = np.full(len(u), 99)
    hi = np.full(len(u), -1)
    np.minimum.at(lo, inv, y)
    np.maximum.at(hi, inv, y)
    sym = np.where(keep, -1, np.where(cp, 100, np.where(lo == hi, lo, -9)))
    if (sym == -9).any():
        return None
    return dict(zip(u.tolist(), sym.tolist()))


def _apply_tab(tab, key, x, C, f):
    if not all(q in tab for q in key.tolist()):
        return None
    s = np.array([tab[q] for q in key.tolist()])
    out = np.where(s == -1, x.ravel(), s)
    if f is not None:
        cf = C[:, f]
        if ((s == 100) & (cf == 10)).any():
            return None
        out = np.where(s == 100, cf, out)
    return out.reshape(x.shape)


def learn_cell_table(train, tests):
    """Cell color from a table over (cell value?, one or two feature atoms); each table entry is 'keep', a color, or
    'copy color feature f' (the row's first cell, the one neighbour color, ...), so value-free tables carry over to
    colors never seen in train. Tables that do not survive leave-one-out over the train pairs are ~weak; the simplest
    consistent one that covers every test cell wins, a different 2nd one is the 2nd guess."""
    import learners
    if learners._STRICT[0] or not same_shape(train) or not changed(train):
        return []                                        # not inside compositions / meta sub-tasks (0/1 right there)
    if sum(x.size for x, _ in train) > 4000:
        return []
    bgs = {bg_color(x) for x, _ in train}
    b = bgs.pop() if len(bgs) == 1 else None          # one train background: keep it for a test grid that has it
    tbg = [b if b is not None and (t == b).any() else None for t in tests]
    Ftr = [cell_feats(x) for x, _ in train]
    Fte = [cell_feats(t, g) for t, g in zip(tests, tbg)]
    Ctr = [color_feats(x) for x, _ in train]
    Cte = [color_feats(t, g) for t, g in zip(tests, tbg)]
    atoms = _atoms(Ftr)
    cols = sorted(set().union(*[set(np.unique(x).tolist()) for x, _ in train]))
    atoms += [(len(FEATS) + f, ("eq", c)) for f in range(len(CFEATS)) for c in cols]
    Ftr = [np.hstack([F, C]) for F, C in zip(Ftr, Ctr)]          # key columns: cell features, then color features
    Fte = [np.hstack([F, C]) for F, C in zip(Fte, Cte)]
    raw = [a for a in atoms if a[1] is None]
    subsets = [(a,) for a in atoms] + [(a, b) for i, a in enumerate(raw) for b in atoms[i + 1:] if b[0] != a[0]]
    xs = [x.ravel() for x, _ in train]
    ys = [y.ravel() for _, y in train]
    found = []
    for f in [None] + list(range(len(CFEATS))):
        if f is not None and not any(((y == C[:, f]) & (y != x)).any() for x, y, C in zip(xs, ys, Ctr)):
            continue                                     # copying f never explains a changed cell
        for sub in (subsets if f is None else subsets[:len(atoms) + 300]):
            for use_v in (False, True):
                keys = [_key(x, F, sub, use_v) for (x, _), F in zip(train, Ftr)]
                tab = _fit_tab(keys, xs, ys, Ctr, f)
                if tab is None or all(v == -1 for v in tab.values()) or (f is not None and 100 not in tab.values()):
                    continue
                outs = []
                for t, F, C in zip(tests, Fte, Cte):
                    o = _apply_tab(tab, _key(t, F, sub, use_v), t, C, f)
                    if o is None:
                        outs = None
                        break
                    outs.append(o.tolist())
                if outs is None:
                    continue
                loo = len(train) >= 2
                for h in (range(len(train)) if loo else ()):
                    rest = [k for k in range(len(train)) if k != h]
                    th = _fit_tab([keys[k] for k in rest], [xs[k] for k in rest], [ys[k] for k in rest],
                                  [Ctr[k] for k in rest], f)
                    pr = None if th is None else _apply_tab(th, keys[h], train[h][0], Ctr[h], f)
                    if pr is None or not (pr == train[h][1]).all():
                        loo = False
                        break
                found.append(((not loo, len(sub), use_v, f is not None, len(tab)), sub, use_v, f, outs, loo))
                break
            if len(found) >= 8:
                break
        if len(found) >= 8:
            break
    if not found:
        return []
    found.sort(key=lambda r: r[0])
    res, seen = [], []
    for _, sub, use_v, f, outs, loo in found:
        if outs in seen:
            continue
        seen.append(outs)
        nm = ("v+" if use_v else "") + "+".join(_aname(a) for a in sub) + ("" if f is None else f">copy:{CFEATS[f]}")
        parity = any(a[0] < len(FEATS) and FEATS[a[0]] in ("pi", "pj", "pij", "oi2", "oj2") for a in sub)
        pr = (1.75 if loo else 1.9) + 0.05 * (f is not None) + 0.1 * parity + 0.02 * len(res)
        res.append(Cand("learn", f"cell_table[{nm}]" + ("" if loo else "~weak"), outs, pr))
        if len(res) == 2:
            break
    return res


SEGS = ["multi8", "multi4", "color8", "color4"]
CROPS = ["bbox", "only"]                    # whole bounding box | the object's own cells on background


def pobjects(x, seg):
    bg = bg_color(x)
    out = []
    if seg.startswith("multi"):
        lab, n = ndi.label(x != bg, structure=S8 if seg == "multi8" else None)
        labs = [(lab, k) for k in range(1, n + 1)]
    else:
        labs = []
        for c in np.unique(x):
            if c == bg:
                continue
            lab, n = ndi.label(x == c, structure=S8 if seg == "color8" else None)
            labs += [(lab, k) for k in range(1, n + 1)]
    for lab, k in labs:
        m = lab == k
        ys, xs = np.nonzero(m)
        out.append((m, (ys.min(), ys.max() + 1, xs.min(), xs.max() + 1)))
        if len(out) > 60:
            return None
    return out


def pcrop(x, o, how):
    m, (a, b, c, d) = o
    if how == "bbox":
        return x[a:b, c:d]
    return np.where(m[a:b, c:d], x[a:b, c:d], bg_color(x))


def pfeats(x, o):
    m, (a, b, c, d) = o
    sub = x[a:b, c:d]
    mm = m[a:b, c:d]
    bg = bg_color(x)
    vals = sub[mm]
    holes = ndi.binary_fill_holes(mm) & ~mm
    H, W = x.shape
    return {"size": int(mm.sum()), "area": int(mm.size), "h": int(b - a), "w": int(d - c),
            "ncolors": len(np.unique(vals)), "holes": int(ndi.label(holes)[1]), "hole_cells": int(holes.sum()),
            "border": int(a == 0 or c == 0 or b == H or d == W), "top": int(a), "left": int(c),
            "sym_lr": int((sub == sub[:, ::-1]).all()), "sym_ud": int((sub == sub[::-1]).all()),
            "dense": round(float(mm.mean()), 3), "shape": mm.tobytes() + bytes(mm.shape),
            "color": int(Counter(vals.tolist()).most_common(1)[0][0]), "pattern": sub.tobytes() + bytes(sub.shape),
            "n_bg_in": int((sub == bg).sum())}


PFEATS = ["size", "area", "h", "w", "ncolors", "holes", "hole_cells", "border", "top", "left", "sym_lr", "sym_ud",
         "dense", "shape", "color", "pattern", "n_bg_in"]
ORD = {"size", "area", "h", "w", "ncolors", "holes", "hole_cells", "top", "left", "dense", "n_bg_in"}


def _choose(fs, how, f):
    v = [r[f] for r in fs]
    if how == "unique":
        cnt = Counter(v)
        u = [i for i, a in enumerate(v) if cnt[a] == 1]
        return u[0] if len(u) == 1 and len(v) > 2 else None
    best = max(v) if how == "max" else min(v)
    idx = [i for i, a in enumerate(v) if a == best]
    return idx[0] if len(idx) == 1 else None


def learn_obj_pick(train, tests):
    """Every (segmentation, crop, feature, unique/max/min) rule that reproduces train votes for its test answer; the
    answers are ranked by votes (position-only rules count half), the best two are returned."""
    if not all(y.size < x.size for x, y in train):
        return []
    votes, names, strong = Counter(), {}, set()
    for seg in SEGS:
        objs = [pobjects(x, seg) for x, _ in train]
        if any(o is None or len(o) < 2 for o in objs):
            continue
        tobjs = [pobjects(t, seg) for t in tests]
        if any(not o or len(o) < 2 for o in tobjs):
            continue
        ftr = [[pfeats(x, o) for o in os_] for (x, _), os_ in zip(train, objs)]
        fte = [[pfeats(t, o) for o in os_] for t, os_ in zip(tests, tobjs)]
        for how_c in CROPS:
            crops = [[pcrop(x, o, how_c) for o in os_] for (x, _), os_ in zip(train, objs)]
            if not all(any(c.shape == y.shape and (c == y).all() for c in cs) for (x, y), cs in zip(train, crops)):
                continue
            for f in PFEATS:
                for how in ["unique"] + (["max", "min"] if f in ORD else []):
                    ok = True
                    for (x, y), fs, cs in zip(train, ftr, crops):
                        i = _choose(fs, how, f)
                        if i is None or cs[i].shape != y.shape or not (cs[i] == y).all():
                            ok = False
                            break
                    if not ok:
                        continue
                    outs = []
                    for t, fs, os_ in zip(tests, fte, tobjs):
                        i = _choose(fs, how, f)
                        if i is None:
                            break
                        outs.append(pcrop(t, os_[i], how_c).tolist())
                    if len(outs) != len(tests):
                        continue
                    key = json.dumps(outs)
                    votes[key] += 0.5 if f in ("top", "left") else 1.0
                    names.setdefault(key, f"obj_pick[{seg},{how_c},{how} {f}]")
                    if f not in ("top", "left"):
                        strong.add(key)
    res = []
    for k, (key, v) in enumerate(votes.most_common(2)):
        res.append(Cand("learn", names[key] + f"x{v:g}" + ("" if key in strong else "~weak"), json.loads(key),
                        1.55 + 0.02 * k))
    return res


# ------------------------------------------------------------------------------------------------ grids (row 74)
D4T = {"id": lambda g: g, "r90": lambda g: np.rot90(g, -1), "r180": lambda g: np.rot90(g, 2),
       "r270": lambda g: np.rot90(g, 1), "fud": lambda g: g[::-1], "flr": lambda g: g[:, ::-1], "tr": lambda g: g.T,
       "atr": lambda g: np.rot90(g, 2).T}


def learn_d4_tile(train, tests):
    """Output = a kh x kw arrangement of blocks, each a fixed rotation / reflection of the input (learned per block
    position; the first fitting transform in D4T order)."""
    ratios = {(y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) for x, y in train}
    if len(ratios) != 1:
        return []
    kh, kw = ratios.pop()
    if kh != int(kh) or kw != int(kw) or kh * kw < 2 or kh > 5 or kw > 5:
        return []
    kh, kw = int(kh), int(kw)
    plan = {}
    for bi in range(kh):
        for bj in range(kw):
            for name, f in D4T.items():
                ok = True
                for x, y in train:
                    h, w = x.shape
                    b = y[bi * h:(bi + 1) * h, bj * w:(bj + 1) * w]
                    fx = f(x)
                    if fx.shape != b.shape or not (fx == b).all():
                        ok = False
                        break
                if ok:
                    plan[bi, bj] = name
                    break
            if (bi, bj) not in plan:
                return []
    if len(set(plan.values())) == 1 and plan[0, 0] == "id":
        return []                                        # a plain tile: learn_tile_one's job

    def predict(x):
        rows = []
        for bi in range(kh):
            row = [D4T[plan[bi, bj]](x) for bj in range(kw)]
            if len({r.shape for r in row}) != 1:
                return None
            rows.append(row)
        return np.block(rows)
    nm = ",".join(plan[bi, bj] for bi in range(kh) for bj in range(kw))
    c = fit_all(train, tests, predict, f"d4_tile[{kh}x{kw}:{nm}]", prior=1.4)
    return [c] if c else []


def _kron_sel(x, sel, fill):
    """Block (i, j) of the output = the input where cell (i, j) is selected (sel: 'mode' / 'least' = the most / least common color,
    'nz' = not black, int c = that color), else a blank block of the fill color ('bg' = background, int = that color)."""
    H, W = x.shape
    if H * H > 30 or W * W > 30:
        return None
    v, c = np.unique(x, return_counts=True)
    if sel in ("mode", "least"):
        k = c.max() if sel == "mode" else c.min()
        if (c == k).sum() != 1:
            return None
        on = x == v[list(c).index(k)]
    elif sel == "nz":
        on = x != 0
    else:
        on = x == sel
    f = bg_color(x) if fill == "bg" else fill
    blank = np.full_like(x, f)
    return np.block([[x if on[i, j] else blank for j in range(W)] for i in range(H)])


def learn_kron_sel(train, tests):
    if not all(y.shape == (x.shape[0] ** 2, x.shape[1] ** 2) for x, y in train):
        return []
    cols = sorted(set.intersection(*[set(np.unique(x).tolist()) for x, _ in train]))
    for sel in ["mode", "least", "nz"] + cols:
        for fill in ["bg", 0]:
            c = fit_all(train, tests, lambda x, s=sel, f=fill: _kron_sel(x, s, f), f"kron_sel[{sel},{fill}]", prior=1.45)
            if c:
                return [c]
    return []


KFEATS = {
    "ncolors": lambda x: len(np.unique(x)),
    "ncolors_fg": lambda x: len(set(np.unique(x).tolist()) - {bg_color(x)}),
    "n_fg": lambda x: int((x != bg_color(x)).sum()),
    "n_bg": lambda x: int((x == bg_color(x)).sum()),
    "n_obj8": lambda x: ndi.label(x != bg_color(x), structure=np.ones((3, 3), bool))[1],
    "side": lambda x: x.shape[0] if x.shape[0] == x.shape[1] else -1,
}
GROWS = {"tile": lambda x, k: np.tile(x, (k, k)), "up": lambda x, k: np.kron(x, np.ones((k, k), x.dtype))}


def learn_var_grow(train, tests):
    """Output = the input tiled / upscaled k x k times, k a count read off the input (number of colors, cells, ...)."""
    ks = []
    for x, y in train:
        if y.shape[0] % x.shape[0] or y.shape[1] % x.shape[1] or y.shape[0] // x.shape[0] != y.shape[1] // x.shape[1]:
            return []
        ks.append(y.shape[0] // x.shape[0])
    if len(set(ks)) < 2:
        return []                                        # a fixed factor is the fixed-grow learners' job
    for fn, f in KFEATS.items():
        if [f(x) for x, _ in train] != ks:
            continue
        for gn, g in GROWS.items():
            def predict(x, f=f, g=g):
                k = f(x)
                if not 1 <= k <= 30 or x.shape[0] * k > 30 or x.shape[1] * k > 30:
                    return None
                return g(x, k)
            c = fit_all(train, tests, predict, f"var_grow[{gn},k={fn}]", prior=1.45)
            if c:
                return [c]
    return []


# ------------------------------------------------------------------------------------------------ pointer (row 74)
D4 = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def _pointer(x, colmode):
    """Multicolor objects with exactly one cell of a minority color on an outer side (exactly one of its 4
    directions leaves the object at once): a ray leaves the object on the opposite side, along the line through
    that cell, in the cell's color (colmode 'odd') or the object's main color ('main')."""
    bg = bg_color(x)
    H, W = x.shape
    lab, n = ndi.label(x != bg, structure=S8)
    g = x.copy()
    hit = False
    for k in range(1, n + 1):
        m = lab == k
        vals, cnt = np.unique(x[m], return_counts=True)
        if len(vals) != 2 or cnt.min() != 1:
            continue
        odd = vals[cnt.argmin()]
        main = vals[cnt.argmax()]
        (i, j), = np.argwhere(m & (x == odd))
        empty = [d for d in D4 if not (0 <= i + d[0] < H and 0 <= j + d[1] < W and m[i + d[0], j + d[1]])]
        if len(empty) != 1:
            continue
        di, dj = -empty[0][0], -empty[0][1]
        a, b = i, j
        while 0 <= a < H and 0 <= b < W and m[a, b]:
            a, b = a + di, b + dj
        col = odd if colmode == "odd" else main
        while 0 <= a < H and 0 <= b < W and x[a, b] == bg:
            g[a, b] = col
            a, b = a + di, b + dj
            hit = True
    return g if hit else None


def learn_pointer(train, tests):
    if not same_shape(train) or not changed(train):
        return []
    for cm in ("odd", "main"):
        c = fit_all(train, tests, lambda x, cm=cm: _pointer(x, cm), f"pointer[{cm}]", prior=1.6)
        if c:
            return [c]
    return []


# ------------------------------------------------------------------------------------------------ whole grid (row 79)
def learn_grid_d4(train, tests):
    """The whole grid rotated / reflected (optionally with one learned color map) -- same-shape square tasks never
    reach the crop>D4 compositions."""
    from learners4 import with_cmap
    for name, f in D4T.items():
        if name == "id" or not all(f(x).shape == y.shape for x, y in train):
            continue
        c = with_cmap(train, tests, f, f"grid_d4[{name}]", 1.3)
        if c:
            return [c]
    return []


LEARNERS = [learn_cell_table, learn_obj_pick, learn_d4_tile, learn_kron_sel, learn_var_grow, learn_pointer,
            learn_grid_d4]


In [ ]:
%%writefile /kaggle/working/cpu_solver/learners6.py
"""Learners batch 6 (2026-10-18, second session): fold, per-color bounding-box fill, crop + run dedupe.

  fold        the grid splits at one full non-background divider line (or exactly in half); the output overlays one
              half with the other, flipped or not (e3497940-like). Base half, flip and conflict winner are learned.
  bbox_fill   every non-background color fills the rectangle its pixels span (56ff96f3-like). Paint order (large
              first / small first) and whether existing colors are overwritten are learned.
  dedupe      crop to the non-background box (optional), then collapse runs of identical rows and columns
              (90c28cc7-like: a mosaic of solid blocks -> one cell per block).
"""
import numpy as np

from learners import bg_color, fit_all

FLIPS = {"id": lambda g: g, "lr": np.fliplr, "ud": np.flipud, "r180": lambda g: np.rot90(g, 2)}


def _split(x, axis, mode):
    """(A, B) halves of x along `axis` ("sep": at the one full line of a single non-background color that leaves two
    equal halves; "half": exactly in half), or None."""
    n = x.shape[axis]
    if mode == "half":
        if n % 2:
            return None
        k = n // 2
        return (x[:k], x[k:]) if axis == 0 else (x[:, :k], x[:, k:])
    bg = bg_color(x)
    hits = []
    for i in range(1, n - 1):
        line = x[i] if axis == 0 else x[:, i]
        if len(set(line.tolist())) == 1 and line[0] != bg and 2 * i + 1 == n:
            hits.append(i)
    if len(hits) != 1:
        return None
    i = hits[0]
    return (x[:i], x[i + 1:]) if axis == 0 else (x[:, :i], x[:, i + 1:])


def learn_fold(train, tests):
    if all(x.shape == y.shape for x, y in train):
        return []
    for axis in (1, 0):
        for mode in ("sep", "half"):
            if not all(_split(x, axis, mode) is not None for x, _ in train):
                continue
            for base in ("A", "B"):
                for fname, f in FLIPS.items():
                    for win in ("other", "base"):
                        def predict(x, axis=axis, mode=mode, base=base, f=f, win=win):
                            s = _split(x, axis, mode)
                            if s is None:
                                return None
                            a, b = s if base == "A" else s[::-1]
                            o = f(b)
                            if o.shape != a.shape:
                                return None
                            bg = bg_color(x)
                            out = a.copy()
                            m = o != bg
                            if win == "base":
                                m &= a == bg
                            out[m] = o[m]
                            return out
                        c = fit_all(train, tests, predict, f"fold[{mode}{axis},{base},{fname},{win}]", prior=1.4)
                        if c:
                            return [c]
    return []


def _bbox_fill(x, order, over):
    bg = bg_color(x)
    boxes = []
    for c in np.unique(x):
        if c == bg:
            continue
        r, k = np.nonzero(x == c)
        boxes.append(((r.max() - r.min() + 1) * (k.max() - k.min() + 1), int(c), r.min(), r.max(), k.min(), k.max()))
    if not boxes:
        return None
    # Overlapping boxes need a paint order the train pairs rarely pin down (scan: 070dd51e, 52df9849 fitted
    # train with large_first and were wrong on test) -> only conflict-free layouts.
    for i, (_, _, a0, a1, b0, b1) in enumerate(boxes):
        for _, _, c0, c1, d0, d1 in boxes[i + 1:]:
            if a0 <= c1 and c0 <= a1 and b0 <= d1 and d0 <= b1:
                return None
    boxes.sort(reverse=(order == "large_first"))
    out = x.copy()
    for _, c, r0, r1, k0, k1 in boxes:
        sub = out[r0:r1 + 1, k0:k1 + 1]
        if over:
            sub[...] = c
        else:
            sub[sub == bg] = c
    return out


def learn_bbox_fill(train, tests):
    if not all(x.shape == y.shape for x, y in train) or all((x == y).all() for x, y in train):
        return []
    for order in ("large_first", "small_first"):
        for over in (False, True):
            c = fit_all(train, tests, lambda x, o=order, v=over: _bbox_fill(x, o, v),
                        f"bbox_fill[{order},{'over' if over else 'bg_only'}]", prior=1.4)
            if c:
                return [c]
    return []


def _dedupe(g):
    keep_r = [0] + [i for i in range(1, g.shape[0]) if not (g[i] == g[i - 1]).all()]
    g = g[keep_r]
    keep_c = [0] + [j for j in range(1, g.shape[1]) if not (g[:, j] == g[:, j - 1]).all()]
    return g[:, keep_c]


def _crop(x):
    bg = bg_color(x)
    r, k = np.nonzero(x != bg)
    if not len(r):
        return None
    return x[r.min():r.max() + 1, k.min():k.max() + 1]


def learn_dedupe(train, tests):
    if not all(y.size < x.size for x, y in train):
        return []
    for crop in (True, False):
        pre = [(_crop(x) if crop else x) for x, _ in train]
        # The collapse must matter on some train pair; else a plain crop explains the task and dedupe only adds a
        # way to be wrong on test (scan: 1cf80156, crop-only, fitted and was wrong).
        if all(p is not None and p.shape == y.shape for p, (_, y) in zip(pre, train)):
            continue

        def predict(x, crop=crop):
            g = _crop(x) if crop else x
            return None if g is None else _dedupe(g)
        c = fit_all(train, tests, predict, f"dedupe[{'crop' if crop else 'whole'}]", prior=1.4)
        if c:
            return [c]
    return []


def _frames(x):
    """Single-color rectangular frames (a 4-connected one-color component whose bounding box border is all that
    color, at least 3x3): (r0, r1, c0, c1, color)."""
    from scipy import ndimage as ndi
    bg = bg_color(x)
    out = []
    for c in np.unique(x):
        if c == bg:
            continue
        lab, n = ndi.label(x == c)
        for k in range(1, n + 1):
            ys, xs = np.nonzero(lab == k)
            a, b, l, r = ys.min(), ys.max(), xs.min(), xs.max()
            if b - a < 2 or r - l < 2:
                continue
            box = x[a:b + 1, l:r + 1]
            if (box[0] == c).all() and (box[-1] == c).all() and (box[:, 0] == c).all() and (box[:, -1] == c).all():
                out.append((a, b, l, r, int(c)))
    return out


def learn_frame_crop(train, tests):
    """Output = the inside (or the whole) of one rectangular frame: the largest, the smallest, or the only one
    (1a6449f1-like)."""
    if not all(y.size < x.size for x, y in train):
        return []
    for part in ("inside", "whole"):
        for sel in ("max", "min", "only"):
            def predict(x, part=part, sel=sel):
                fr = _frames(x)
                if not fr or (sel == "only" and len(fr) != 1):
                    return None
                area = [(b - a + 1) * (r - l + 1) for a, b, l, r, _ in fr]
                best = max(area) if sel == "max" else min(area)
                pick = [f for f, s in zip(fr, area) if s == best]
                if len(pick) != 1:
                    return None
                a, b, l, r, _ = pick[0]
                return x[a + 1:b, l + 1:r] if part == "inside" else x[a:b + 1, l:r + 1]
            c = fit_all(train, tests, predict, f"frame_crop[{part},{sel}]", prior=1.45)
            if c:
                return [c]
    return []


def learn_mode_object(train, tests):
    """Output = the object pattern that occurs most often among the input's objects (39a8645d-like); obj_pick
    already covers the opposite, the one unique object."""
    from collections import Counter
    from learners5 import pobjects, pcrop
    if not all(y.size < x.size for x, y in train):
        return []
    for seg in ("color8", "color4", "multi8", "multi4"):
        for how_c in ("bbox", "only"):
            def predict(x, seg=seg, how_c=how_c):
                os_ = pobjects(x, seg)
                if not os_ or len(os_) < 3:
                    return None
                crops = [pcrop(x, o, how_c) for o in os_]
                keys = [c.tobytes() + bytes(c.shape) for c in crops]
                cnt = Counter(keys)
                best = max(cnt.values())
                top = [k for k, v in cnt.items() if v == best]
                if best < 2 or len(top) != 1:
                    return None
                return crops[keys.index(top[0])]
            c = fit_all(train, tests, predict, f"mode_object[{seg},{how_c}]", prior=1.5)
            if c:
                return [c]
    return []


LEARNERS = [learn_fold, learn_bbox_fill, learn_dedupe, learn_frame_crop, learn_mode_object]


In [ ]:
%%writefile /kaggle/working/cpu_solver/pre.py
"""Pre-transforms for composition (2026-10-18): x -> a sub-grid, so the same-shape learners can run on
(pre(x), y) for tasks whose output is smaller than the input."""
import numpy as np
from scipy import ndimage as ndi

from learners import bbox, bg_color, box_interiors

S4 = ndi.generate_binary_structure(2, 1)
S8 = np.ones((3, 3), bool)


def crop_nonbg(x):
    bb = bbox(x != bg_color(x))
    return None if bb is None else x[bb[0]:bb[1], bb[2]:bb[3]]


def _objs(x, conn, multi=True):
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S4 if conn == 4 else S8)
    return [(sl, lab[sl] == k) for k, sl in enumerate(ndi.find_objects(lab), 1) if sl is not None]


def crop_obj(x, conn, which):
    objs = _objs(x, conn)
    if not objs:
        return None
    key = (lambda o: o[1].sum()) if which in ("largest", "smallest") else (lambda o: o[1].size)
    vals = [key(o) for o in objs]
    v = max(vals) if which in ("largest", "bboxmax") else min(vals)
    pick = [o for o, a in zip(objs, vals) if a == v]
    if len(pick) != 1:
        return None
    return x[pick[0][0]]


def crop_frame(x):
    b = box_interiors(x)
    if len(b) != 1:
        return None
    r0, r1, c0, c1 = b[0]
    return x[r0:r1, c0:c1]


def crop_frame_incl(x):
    b = box_interiors(x)
    if len(b) != 1:
        return None
    r0, r1, c0, c1 = b[0]
    return x[r0 - 1:r1 + 1, c0 - 1:c1 + 1]


def crop_color(x, c):
    bb = bbox(x == c)
    return None if bb is None else x[bb[0]:bb[1], bb[2]:bb[3]]


PRE = {
    "nonbg": crop_nonbg,
    "frame": crop_frame, "frame+": crop_frame_incl,
    "obj8max": lambda x: crop_obj(x, 8, "largest"), "obj8min": lambda x: crop_obj(x, 8, "smallest"),
    "obj4max": lambda x: crop_obj(x, 4, "largest"), "obj4min": lambda x: crop_obj(x, 4, "smallest"),
    "trim": lambda x: x[1:-1, 1:-1] if min(x.shape) > 2 else None,
    "top": lambda x: x[:x.shape[0] // 2], "bottom": lambda x: x[(x.shape[0] + 1) // 2:],
    "left": lambda x: x[:, :x.shape[1] // 2], "right": lambda x: x[:, (x.shape[1] + 1) // 2:],
}
for _c in range(10):
    PRE[f"color{_c}"] = (lambda c: (lambda x: crop_color(x, c)))(_c)


def compress(x):
    """Remove rows / columns that are one color (Hodel's compress)."""
    rows = [i for i in range(x.shape[0]) if len(np.unique(x[i])) > 1]
    cols = [j for j in range(x.shape[1]) if len(np.unique(x[:, j])) > 1]
    if not rows or not cols or (len(rows) == x.shape[0] and len(cols) == x.shape[1]):
        return None
    return x[np.ix_(rows, cols)]


def dedupe(x):
    """Collapse runs of identical consecutive rows and columns (Hodel's dedupe on both axes)."""
    r = [0] + [i for i in range(1, x.shape[0]) if not (x[i] == x[i - 1]).all()]
    g = x[r]
    c = [0] + [j for j in range(1, g.shape[1]) if not (g[:, j] == g[:, j - 1]).all()]
    g = g[:, c]
    return None if g.shape == x.shape else g


PRE["compress"] = compress
PRE["dedupe"] = dedupe


def _mobjs(x):
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S8)
    return bg, lab, [(sl, lab[sl] == k, k) for k, sl in enumerate(ndi.find_objects(lab), 1) if sl is not None]


def crop_touching(x, c):
    """The object (8-connected, cells other than color c) that touches a c-colored cell; the c cells are dropped."""
    bg = bg_color(x)
    marker = x == c
    if not marker.any():
        return None
    body = (x != bg) & ~marker
    lab, n = ndi.label(body, structure=S8)
    near = set(np.unique(lab[ndi.binary_dilation(marker, structure=S8)]).tolist()) - {0}
    if len(near) != 1:
        return None
    k = near.pop()
    m = lab == k
    r0, r1, c0, c1 = bbox(m)
    return np.where(m[r0:r1, c0:c1], x[r0:r1, c0:c1], bg)


def crop_max_color(x, c, most=True, conn=8):
    """The connected multicolor object (8- or 4-connected) with the most (or fewest, > 0) cells of color c."""
    if conn == 8:
        bg, lab, objs = _mobjs(x)
    else:
        bg = bg_color(x)
        lab, n = ndi.label(x != bg, structure=S4)
        objs = [(sl, lab[sl] == k, k) for k, sl in enumerate(ndi.find_objects(lab), 1) if sl is not None]
    scored = [(int((x[sl][m] == c).sum()), sl) for sl, m, k in objs]
    scored = [s for s in scored if s[0] > 0]
    if not scored:
        return None
    v = max(s[0] for s in scored) if most else min(s[0] for s in scored)
    pick = [sl for n, sl in scored if n == v]
    return x[pick[0]] if len(pick) == 1 else None


for _c in range(10):
    PRE[f"touch{_c}"] = (lambda c: (lambda x: crop_touching(x, c)))(_c)
    PRE[f"most{_c}"] = (lambda c: (lambda x: crop_max_color(x, c, True)))(_c)
    PRE[f"fewest{_c}"] = (lambda c: (lambda x: crop_max_color(x, c, False)))(_c)
    PRE[f"most{_c}_4"] = (lambda c: (lambda x: crop_max_color(x, c, True, 4)))(_c)


def crop_unique(x, feat):
    """The one 8-connected multicolor object whose `feat` value no other object shares (odd one out)."""
    from collections import Counter
    import learners_obj as LO
    bg = bg_color(x)
    objs = LO.features(x, LO.extract(x, bg, 8, True), bg)
    if len(objs) < 3:
        return None
    vals = [o.f[feat] for o in objs]
    c = Counter(vals)
    u = [o for o, v in zip(objs, vals) if c[v] == 1]
    if len(u) != 1:
        return None
    return x[u[0].sl]


# 2026-10-18 (LOG row 48): registering uniq_<feature> crops gave 0 new training tasks at extra cost -> not registered


In [ ]:
%%writefile /kaggle/working/cpu_solver/meta.py
"""Meta-learners (2026-10-18): run every same-shape learner on SUB-TASKS built from the parts of each grid.

  per_cell  grids split by separator lines into the same layout in input and output (separators unchanged): every
            (input cell, output cell) pair across all train pairs is one sub-pair; the learner that fits all of them
            is applied to each test cell
  per_obj   every object's bounding box (8-connected, non-background) in the input and the same box in the output is a
            sub-pair; everything outside the boxes must be unchanged
A sub-task learner must reproduce every sub-pair, and the assembled grid must reproduce every train pair.
"""
import numpy as np
from scipy import ndimage as ndi

from common import Cand
from learners import bg_color, eq, same_shape, sep_cells

S8 = np.ones((3, 3), bool)


def _sub_learners():
    import learners
    import learners_obj
    import learners2
    import learners3
    import learners4
    fns = [learners.learn_symmetry, learners.learn_period, learners.learn_sym_repair] + learners_obj.LEARNERS + \
        [learners2.learn_flood, learners2.learn_latin, learners2.learn_block_tile] + \
        [f for f in learners3.LEARNERS if f.__name__ != "learn_output_lookup"] + [learners4.learn_obj_symmetrize]
    return fns


def _parts_cells(x):
    cells = sep_cells(x)
    return [(slice(a, b), slice(c, d)) for a, b, c, d in cells] if len(cells) >= 2 else None


def _parts_objs(x, pad=0):
    bg = bg_color(x)
    lab, n = ndi.label(x != bg, structure=S8)
    if n < 1 or n > 30:
        return None
    H, W = x.shape
    return [(slice(max(0, a.start - pad), min(H, a.stop + pad)), slice(max(0, b.start - pad), min(W, b.stop + pad)))
            for a, b in (sl for sl in ndi.find_objects(lab) if sl is not None)]


def _run(train, tests, partf, tag, clock):
    if not same_shape(train):
        return []
    sub_train, sub_tests, layouts = [], [], []
    for x, y in train:
        ps = partf(x)
        if ps is None:
            return []
        cover = np.zeros(x.shape, bool)
        for sl in ps:
            cover[sl] = True
            sub_train.append((x[sl], y[sl]))
        if ((x != y) & ~cover).any():
            return []
    if not any((a != b).any() for a, b in sub_train):
        return []
    for t in tests:
        ps = partf(t)
        if ps is None:
            return []
        layouts.append(ps)
        sub_tests += [t[sl] for sl in ps]
    # fit on the sub-pairs that change; the unchanged ones ride along as extra inputs that must come back unchanged
    changed = [(a, b) for a, b in sub_train if (a != b).any()]
    same = [a for a, b in sub_train if not (a != b).any()]
    if len(changed) < 2:
        return []
    found = []
    for fn in _sub_learners():
        if clock.over():
            break
        try:
            cands = fn(changed, same + sub_tests)
        except Exception:                                              # noqa: BLE001
            continue
        for c in cands[:1]:
            if any(o is None or not eq(np.array(o), a) for o, a in zip(c.outs[:len(same)], same)):
                continue
            c.outs = c.outs[len(same):]
            outs, k = [], 0
            ok = True
            for t, ps in zip(tests, layouts):
                g = t.copy()
                for sl in ps:
                    o = c.outs[k]
                    k += 1
                    if o is None or np.array(o).shape != g[sl].shape:
                        ok = False
                        break
                    g[sl] = np.array(o)
                if not ok:
                    break
                outs.append(g.tolist())
            if ok:
                found.append(Cand("learn", f"{tag}>{c.name}", outs, c.prior + 0.15))
        if found:
            break
    return found


def learn_per_cell(train, tests, clock):
    return _run(train, tests, _parts_cells, "per_cell", clock)


def learn_per_obj(train, tests, clock):
    return _run(train, tests, _parts_objs, "per_obj", clock)


def learn_per_obj_pad(train, tests, clock):
    return _run(train, tests, lambda x: _parts_objs(x, 1), "per_obj+1", clock)


META = [learn_per_cell, learn_per_obj, learn_per_obj_pad]


# ------------------------------------------------------------------------------------------------ color normalisation
def _cnorm_map(x):
    """Input colors -> canonical indices by frequency rank (background = 0, then most common first, ties by value)."""
    v, c = np.unique(x, return_counts=True)
    order = sorted(zip(v.tolist(), c.tolist()), key=lambda p: (-p[1], p[0]))
    return {col: k for k, (col, _) in enumerate(order)}


def _apply_map(g, m):
    out = g.copy()
    for a, b in m.items():
        out[g == a] = b
    return out


def learn_cnorm(train, tests, clock):
    """Every learner on color-normalised pairs (each pair relabelled by its own input's color frequency ranks); the
    prediction is mapped back through the test input's relabelling. Output colors absent from a pair's input keep
    their value if it does not collide with a canonical index. Only for tasks whose pairs use different colors."""
    palettes = {tuple(sorted(np.unique(x).tolist())) for x, _ in train}
    if len(palettes) < 2:
        return []
    tr2 = []
    for x, y in train:
        m = _cnorm_map(x)
        new = set(np.unique(y).tolist()) - set(m)
        if new & set(m.values()):
            return []
        tr2.append((_apply_map(x, m), _apply_map(y, m)))
    te_maps = [_cnorm_map(t) for t in tests]
    te2 = [_apply_map(t, m) for t, m in zip(tests, te_maps)]
    if len({tuple(sorted(np.unique(a).tolist())) for a, _ in tr2}) == len(palettes) and False:
        return []
    import learners
    found = []
    for fn in learners.all_learners():
        if clock.over():
            break
        if fn.__name__ in ("learn_output_lookup",):   # memorised outputs: 5 of cnorm's 7 wrong on training
            continue
        try:
            cands = fn(tr2, te2)
        except Exception:                                              # noqa: BLE001
            continue
        for c in cands[:1]:
            outs = []
            for o, m in zip(c.outs, te_maps):
                if o is None:
                    outs.append(None)
                    continue
                inv = {v: k for k, v in m.items()}
                g = np.array(o)
                vals = set(np.unique(g).tolist())
                if any(v not in inv and v in m.values() for v in vals):
                    outs.append(None)
                    continue
                outs.append(_apply_map(g, inv).tolist())
            if any(o is not None for o in outs):
                found.append(Cand("learn", f"cnorm>{c.name}", outs, c.prior + 0.2))
        if found:
            break
    return found


def _run_on(tr2, te2, clock, tag, post, prior_add=0.25, skip=("learn_output_lookup",)):
    import learners
    found = []
    for fn in learners.all_learners():
        if clock.over():
            break
        if fn.__name__ in skip:
            continue
        try:
            cands = fn(tr2, te2)
        except Exception:                                              # noqa: BLE001
            continue
        for c in cands[:1]:
            outs = [None if o is None else post(np.array(o), k) for k, o in enumerate(c.outs)]
            outs = [None if o is None else o.tolist() for o in outs]
            if any(o is not None for o in outs):
                found.append(Cand("learn", f"{tag}>{c.name}", outs, c.prior + prior_add))
        if found:
            break
    return found


def learn_drop_color(train, tests, clock):
    """A color present in every input and absent from every output is set to background first (noise / scaffold)."""
    bg = bg_color(train[0][0])
    gone = [c for c in range(10) if c != bg and all((x == c).any() and not (y == c).any() for x, y in train)]
    found = []
    for c in gone:
        def drop(g, c=c):
            g = g.copy()
            g[g == c] = bg_color(g)
            return g
        found = _run_on([(drop(x), y) for x, y in train], [drop(t) for t in tests], clock, f"drop{c}",
                        lambda o, k: o)
        if found:
            break
    return found


D4V = {"tr": (lambda g: g.T, lambda g: g.T), "r90": (lambda g: np.rot90(g, 1), lambda g: np.rot90(g, -1)),
       "fud": (lambda g: g[::-1], lambda g: g[::-1]), "flr": (lambda g: g[:, ::-1], lambda g: g[:, ::-1])}


def learn_d4_view(train, tests, clock):
    found = []
    for name, (f, finv) in D4V.items():
        if clock.over():
            break
        found = _run_on([(np.ascontiguousarray(f(x)), np.ascontiguousarray(f(y))) for x, y in train],
                        [np.ascontiguousarray(f(t)) for t in tests], clock, f"view-{name}",
                        lambda o, k, finv=finv: np.ascontiguousarray(finv(o)))
        if found:
            break
    return found


def learn_cnorm_full(train, tests, clock):
    """learn_cnorm over the whole learn pipeline (learners.engine_arrays without another normalisation)."""
    palettes = {tuple(sorted(np.unique(x).tolist())) for x, _ in train}
    if len(palettes) < 2:
        return []
    tr2 = []
    for x, y in train:
        m = _cnorm_map(x)
        new = set(np.unique(y).tolist()) - set(m)
        if new & set(m.values()):
            return []
        tr2.append((_apply_map(x, m), _apply_map(y, m)))
    te_maps = [_cnorm_map(t) for t in tests]
    te2 = [_apply_map(t, m) for t, m in zip(tests, te_maps)]
    import learners
    found = []
    for c in learners._engine_core(tr2, te2, clock, cnorm=False, skip=("learn_output_lookup",))[:2]:
        outs = []
        for o, m in zip(c.outs, te_maps):
            if o is None:
                outs.append(None)
                continue
            inv = {v: k for k, v in m.items()}
            g = np.array(o)
            if any(v not in inv and v in m.values() for v in np.unique(g).tolist()):
                outs.append(None)
                continue
            outs.append(_apply_map(g, inv).tolist())
        if any(o is not None for o in outs):
            found.append(Cand("learn", f"cnorm>{c.name}", outs, c.prior + 0.2))
    return found


In [ ]:
%%writefile /kaggle/working/cpu_solver/loo.py
"""Leave-one-out verification of a CPU candidate (2026-10-18).

    loo_check(task, solve_fn, top1, budget) -> (held_out_pass_rate, consistent_rate) or None

For every train pair k: re-solve with pair k removed and pair k's input as an extra test. held_out_pass: the first
answer for pair k's input equals its output. consistent: the first answers for the real test inputs equal `top1`
(the answers the full train set gave). A rule that only fits because it saw every pair (memorised tables, accidental
fits) tends to fail one of the two. Needs >= 3 train pairs (else None).

    python cpu_solver/loo.py measure [--engine wwhdr] [--n 40] [--budget 10]   # separation on wwhdr right / wrong
"""
import json
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, HERE)


def loo_check(task, solve_fn, top1, budget):
    tr = task["train"]
    if len(tr) < 3:
        return None
    held = cons = 0
    for k in range(len(tr)):
        sub = {"train": tr[:k] + tr[k + 1:], "test": [{"input": tr[k]["input"]}] + list(task["test"])}
        try:
            res = solve_fn(sub, budget)
        except Exception:                                              # noqa: BLE001
            res = []
        first = [per[0][0] if per else None for per in res] if res else []
        held += bool(first) and first[0] == tr[k]["output"]
        cons += len(first) == len(top1) + 1 and all(first[j + 1] == top1[j] for j in range(len(top1)))
    n = len(tr)
    return held / n, cons / n


def _job(args):
    tid, which, engine, budget = args
    import arcsym
    ch = json.load(open(os.path.join(ROOT, "data", f"arc-agi_{which}_challenges.json")))
    sol = json.load(open(os.path.join(ROOT, "data", f"arc-agi_{which}_solutions.json")))
    task = ch[tid]
    solve_fn = lambda t, b: arcsym.solve(t, b, engines=[engine])
    full = solve_fn(task, budget)
    top1 = [per[0][0] if per else None for per in full]
    if any(g is None for g in top1):
        return tid, which, None, None
    right = all(g == s for g, s in zip(top1, sol[tid]))
    return tid, which, right, loo_check(task, solve_fn, top1, budget)


def measure(engine="wwhdr", n=40, budget=10.0, workers=6, runfile=None):
    from concurrent.futures import ProcessPoolExecutor, as_completed
    for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[v] = "1"
    runfile = runfile or ("wwhdr_train_10s.json" if engine == "wwhdr" else "learn16_train.json")
    r = json.load(open(os.path.join(HERE, "runs", runfile)))["rows"]
    ch = json.load(open(os.path.join(ROOT, "data", "arc-agi_training_challenges.json")))
    ok3 = lambda t: len(ch[t]["train"]) >= 3
    right = [t for t, v in sorted(r.items()) if ok3(t) and v["outputs"] and all(p["hit1"] and p["n_cand"] for p in v["outputs"])][:n]
    wrong = [t for t, v in sorted(r.items()) if ok3(t) and any(p["n_cand"] and not p["hit1"] for p in v["outputs"])][:n]
    ev = (("97d7923e", "d59b0160", "e3721c99", "981571dc") if engine == "wwhdr" else
          ("0934a4d8", "135a2760", "16de56c4", "981571dc", "38007db0", "1818057f"))
    jobs = [(t, "training", engine, budget) for t in right + wrong] + [(t, "evaluation", engine, budget) for t in ev]
    rows = []
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=workers) as ex:
        for f in as_completed([ex.submit(_job, j) for j in jobs]):
            rows.append(f.result())
    out = {"engine": engine, "budget": budget, "rows": rows, "wall_s": round(time.time() - t0)}
    json.dump(out, open(os.path.join(HERE, "runs", f"loo_{engine}.json"), "w"), indent=1)
    for which in ("training", "evaluation"):
        for label in (True, False):
            sel = [x for x in rows if x[1] == which and x[2] is label and x[3] is not None]
            if not sel:
                continue
            held = sum(x[3][0] for x in sel) / len(sel)
            both = sum(1 for x in sel if x[3][0] == 1.0 and x[3][1] == 1.0)
            print(f"{which:10s} first answer {'RIGHT' if label else 'WRONG'}: n={len(sel)} mean held-out pass "
                  f"{held:.2f}; all folds pass + consistent: {both}/{len(sel)}")
    print("eval rows:", [(x[0], x[2], x[3]) for x in rows if x[1] == "evaluation"])
    print("wall", out["wall_s"], "s")


if __name__ == "__main__":
    a = sys.argv[1:]
    if a and a[0] == "measure":
        measure(a[a.index("--engine") + 1] if "--engine" in a else "wwhdr",
                int(a[a.index("--n") + 1]) if "--n" in a else 40,
                float(a[a.index("--budget") + 1]) if "--budget" in a else 10.0)


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel_engine.py
"""Engine 1: Michael Hodel's 400 hand-written ARC-1 solvers (refs/arc-dsl, MIT) as a program library (2026-10-18).

Each solver is tried as-is and under transfers that keep its logic but change the task's surface:
  cmap   the new task's colors mapped onto the solver's own task colors by frequency rank (task level), solver run,
         output mapped back -- the same rule with other colors;
  pmap   the same per pair (each input's own color ranks) -- rules whose colors change from pair to pair;
  d4     the input rotated / reflected first, the output turned back -- the same rule in another orientation.
A (solver, transfer) is a candidate only if it reproduces every train pair. Solvers are never scored on their own task
(exclude_id), so the training benchmark measures transfer, not recall."""
import os
import sys
import time
from collections import Counter

from common import HERE, Cand, Timeout, T, load_json, time_limit, valid_grid

sys.path.insert(0, os.path.join(HERE, "hodel"))
import solvers as _HS  # noqa: E402

SOLVERS = sorted((k[6:], v) for k, v in vars(_HS).items() if k.startswith("solve_") and callable(v))
_PROF = None


def color_order(grids):
    c = Counter(v for g in grids for r in g for v in r)
    return [k for k, _ in sorted(c.items(), key=lambda x: (-x[1], x[0]))]


def profile(pairs):
    ins = color_order([p[0] for p in pairs])
    outs_only = [c for c in color_order([p[1] for p in pairs]) if c not in ins]
    return ins, outs_only


def profiles():
    """Color profile of each solver's own task (from the training file, cached in hodel_profiles.json)."""
    global _PROF
    if _PROF is None:
        path = os.path.join(HERE, "hodel_profiles.json")
        if os.path.isfile(path):
            _PROF = load_json(path)
        else:
            _PROF = {}
    return _PROF


def build_profiles(training_challenges_path):
    import json
    ch = load_json(training_challenges_path)
    out = {}
    for sid, _ in SOLVERS:
        if sid in ch:
            ins, oo = profile([(p["input"], p["output"]) for p in ch[sid]["train"]])
            out[sid] = [ins, oo]
    json.dump(out, open(os.path.join(HERE, "hodel_profiles.json"), "w"))
    return out


def full_perm(pairs_map):
    """Partial color map -> a full permutation of 0..9 (unmapped colors fill the free slots in order)."""
    fwd = dict(pairs_map)
    used = set(fwd.values())
    free = [c for c in range(10) if c not in used]
    for c in range(10):
        if c not in fwd:
            fwd[c] = free.pop(0)
    inv = {v: k for k, v in fwd.items()}
    return fwd, inv


def remap(g, m):
    return tuple(tuple(m[v] for v in r) for r in g)


_D4 = {
    "rot90": (lambda g: tuple(zip(*g[::-1])), lambda g: tuple(tuple(r[::-1]) for r in zip(*g[::-1]))[::-1]),
    "rot180": (lambda g: tuple(tuple(r[::-1]) for r in g[::-1]), lambda g: tuple(tuple(r[::-1]) for r in g[::-1])),
    "rot270": (lambda g: tuple(tuple(r[::-1]) for r in zip(*g[::-1]))[::-1], lambda g: tuple(zip(*g[::-1]))),
    "flipud": (lambda g: g[::-1], lambda g: g[::-1]),
    "fliplr": (lambda g: tuple(r[::-1] for r in g), lambda g: tuple(r[::-1] for r in g)),
    "transpose": (lambda g: tuple(zip(*g)), lambda g: tuple(zip(*g))),
    "antitranspose": (lambda g: tuple(zip(*(r[::-1] for r in g[::-1]))), lambda g: tuple(zip(*(r[::-1] for r in g[::-1])))),
}


def _run(fn, g, per_call_s):
    with time_limit(per_call_s):
        o = fn(g)
    if not valid_grid(o):
        return None
    return tuple(tuple(int(v) for v in r) for r in o)


def _variants(task_pairs, test_inputs, sid):
    """Yields (variant_name, f_in(grid, k) -> grid, f_out(grid, k) -> grid); k = pair index or ('t', test index)."""
    yield "id", None, None
    prof = profiles().get(sid)
    if prof:
        ins, oo = profile(task_pairs)
        if len(ins) == len(prof[0]) and len(oo) <= len(prof[1]) and (ins != prof[0] or oo != prof[1][:len(oo)]):
            fwd, inv = full_perm(list(zip(ins, prof[0])) + list(zip(oo, prof[1])))
            yield "cmap", (lambda g, k, fwd=fwd: remap(g, fwd)), (lambda g, k, inv=inv: remap(g, inv))
        pm = {}
        for k, g in list(enumerate(p[0] for p in task_pairs)) + [(("t", i), g) for i, g in enumerate(test_inputs)]:
            gi = color_order([g])
            if len(gi) != len(prof[0]):
                pm = None
                break
            pm[k] = full_perm(list(zip(gi, prof[0])) + list(zip(oo, prof[1])))
        if pm and any(pm[k][0] != pm[0][0] for k in pm):
            yield "pmap", (lambda g, k, pm=pm: remap(g, pm[k][0])), (lambda g, k, pm=pm: remap(g, pm[k][1]))


def engine(task, clock, exclude_id=None, per_call_s=1.0, d4=True):
    pairs = [(T(p["input"]), T(p["output"])) for p in task["train"]]
    tests = [T(t["input"]) for t in task["test"]]
    found, seen = [], set()

    def attempt(sid, fn, vname, fin, fout, rank):
        for k, (x, y) in enumerate(pairs):
            xi = fin(x, k) if fin else x
            o = _run(fn, xi, per_call_s)
            if o is None:
                return
            if fout:
                o = fout(o, k)
            if o != y:
                return
        outs = []
        for i, x in enumerate(tests):
            xi = fin(x, ("t", i)) if fin else x
            try:
                o = _run(fn, xi, per_call_s * 3)
            except (Timeout, Exception):                               # noqa: BLE001
                o = None
            if o is not None and fout:
                o = fout(o, ("t", i))
            outs.append([list(r) for r in o] if o is not None else None)
        key = tuple(map(str, outs))
        if key not in seen:
            seen.add(key)
            found.append(Cand("hodel", f"{sid}:{vname}", outs, prior=rank))

    stages = [("plain", 1.0), ("color", 2.0)] + ([("d4", 3.0)] if d4 else [])
    for stage, rank in stages:
        for sid, fn in SOLVERS:
            if sid == exclude_id:
                continue
            if clock.over():
                return found
            try:
                if stage == "plain":
                    attempt(sid, fn, "id", None, None, rank)
                elif stage == "color":
                    for vname, fin, fout in _variants(pairs, tests, sid):
                        if vname != "id":
                            attempt(sid, fn, vname, fin, fout, rank)
                else:
                    for dn, (f, finv) in _D4.items():
                        attempt(sid, fn, dn, (lambda g, k, f=f: f(g)), (lambda g, k, finv=finv: finv(g)), rank)
            except (Timeout, Exception):                               # noqa: BLE001 - a crashing program is just a miss
                continue
    return found


if __name__ == "__main__":
    t0 = time.time()
    p = build_profiles(os.path.join(os.path.dirname(HERE), "data", "arc-agi_training_challenges.json"))
    print(len(p), "profiles", round(time.time() - t0, 1), "s")


In [ ]:
%%writefile /kaggle/working/cpu_solver/wwhdr_solver.py
# =====================================================================================
# ARC-AGI-2 solver v2  --  WHAT / WHY / HOW / DO / REVIEW  +  SSOR program search
# Pure python + numpy (+ scipy.ndimage if present).  Licence: MIT-0 / CC0.
# =====================================================================================
import json, glob, os, time, traceback, itertools, math, signal
from collections import Counter, defaultdict
import numpy as np
try:
    from scipy import ndimage as _ndi
except Exception:  # pragma: no cover
    _ndi = None

_S4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
_S8 = np.ones((3, 3), dtype=bool)


class Timeout(Exception):
    pass


class Budget:
    """F (feasibility) in the SSOR value gate: every search step checks the clock."""
    def __init__(self, seconds):
        self.t0 = time.time(); self.seconds = seconds
    def left(self):
        return self.seconds - (time.time() - self.t0)
    def over(self):
        return self.left() <= 0
    def check(self):
        if self.over():
            raise Timeout()


def A(g):
    return np.array(g, dtype=np.int16)


def valid(g):
    return (g is not None and isinstance(g, np.ndarray) and g.ndim == 2 and 1 <= g.shape[0] <= 30
            and 1 <= g.shape[1] <= 30 and g.min() >= 0 and g.max() <= 9)


def gkey(g):
    return (g.shape, g.tobytes())


def same(a, b):
    return a is not None and b is not None and a.shape == b.shape and bool((a == b).all())


def label(mask, conn=4):
    if _ndi is not None:
        lab, n = _ndi.label(mask, structure=_S4 if conn == 4 else _S8)
        return lab, n
    h, w = mask.shape
    lab = np.zeros((h, w), dtype=np.int32); n = 0
    nb = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if conn == 8 else [])
    for i in range(h):
        for j in range(w):
            if mask[i, j] and not lab[i, j]:
                n += 1; st = [(i, j)]; lab[i, j] = n
                while st:
                    a, b = st.pop()
                    for da, db in nb:
                        x, y = a + da, b + db
                        if 0 <= x < h and 0 <= y < w and mask[x, y] and not lab[x, y]:
                            lab[x, y] = n; st.append((x, y))
    return lab, n


D4 = {
    'id': (lambda g: g, 'id'),
    'rot90': (lambda g: np.rot90(g, 1), 'rot270'),
    'rot180': (lambda g: np.rot90(g, 2), 'rot180'),
    'rot270': (lambda g: np.rot90(g, 3), 'rot90'),
    'flipud': (lambda g: g[::-1, :], 'flipud'),
    'fliplr': (lambda g: g[:, ::-1], 'fliplr'),
    'transpose': (lambda g: g.T, 'transpose'),
    'antitranspose': (lambda g: np.rot90(g, 2).T, 'antitranspose'),
}
D4_NAMES = list(D4)


def d4(g, name):
    return np.ascontiguousarray(D4[name][0](g))


def d4_inv(g, name):
    return np.ascontiguousarray(D4[D4[name][1]][0](g))


def bg_of(g):
    v, c = np.unique(g, return_counts=True)
    if 0 in v and c[list(v).index(0)] == c.max():
        return 0
    return int(v[np.argmax(c)])# =====================================================================================
# WHAT  --  perception: multiple points of view (POVs) on every grid
# =====================================================================================
class Obj:
    __slots__ = ('mask', 'r0', 'c0', 'r1', 'c1', 'h', 'w', 'size', 'color', 'colors', 'ncol', 'sub', 'shape',
                 'touch', '_holes', '_shape_key', 'idx', 'pov')

    def __init__(self, g, mask, bg, pov):
        ys, xs = np.nonzero(mask)
        self.mask = mask; self.pov = pov
        self.r0, self.r1, self.c0, self.c1 = int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())
        self.h, self.w = self.r1 - self.r0 + 1, self.c1 - self.c0 + 1
        self.size = int(len(ys))
        vals = g[ys, xs]
        cnt = Counter(vals.tolist())
        self.color = cnt.most_common(1)[0][0]
        self.colors = frozenset(cnt)
        self.ncol = len(cnt)
        box = g[self.r0:self.r1 + 1, self.c0:self.c1 + 1]
        self.shape = mask[self.r0:self.r1 + 1, self.c0:self.c1 + 1]
        self.sub = np.where(self.shape, box, bg).astype(np.int16)
        H, W = g.shape
        self.touch = self.r0 == 0 or self.c0 == 0 or self.r1 == H - 1 or self.c1 == W - 1
        self._holes = None          # v2: lazy (count_holes / canon_shape were ~25% of v1 search time)
        self._shape_key = None
        self.idx = 0

    @property
    def holes(self):
        if self._holes is None:
            self._holes = count_holes(self.shape)
        return self._holes

    @property
    def shape_key(self):
        if self._shape_key is None:
            self._shape_key = canon_shape(self.shape)
        return self._shape_key


def count_holes(shape):
    inv = ~np.pad(shape, 1)
    lab, n = label(inv, 4)
    return int(n - 1) if n > 0 else 0


def canon_shape(shape):
    best = None
    for nm in D4_NAMES:
        s = d4(shape, nm)
        k = (s.shape, np.packbits(s).tobytes())
        if best is None or k < best:
            best = k
    return best


_OBJ_CACHE = {}


def objects(g, bg, pov='c4'):
    """pov: c4/c8 = single-colour components, m4/m8 = multicolour non-bg components, col = all cells of one colour."""
    key = (gkey(g), bg, pov)
    r = _OBJ_CACHE.get(key)
    if r is not None:
        return r
    out = []
    if pov in ('c4', 'c8'):
        conn = 4 if pov == 'c4' else 8
        for c in np.unique(g):
            if c == bg:
                continue
            lab, n = label(g == c, conn)
            for i in range(1, n + 1):
                out.append(Obj(g, lab == i, bg, pov))
    elif pov in ('m4', 'm8'):
        lab, n = label(g != bg, 4 if pov == 'm4' else 8)
        for i in range(1, n + 1):
            out.append(Obj(g, lab == i, bg, pov))
    elif pov == 'col':
        for c in np.unique(g):
            if c != bg:
                out.append(Obj(g, g == c, bg, pov))
    out.sort(key=lambda o: (o.r0, o.c0))
    for i, o in enumerate(out):
        o.idx = i
    if len(_OBJ_CACHE) > 4000:
        _OBJ_CACHE.clear()
    _OBJ_CACHE[key] = out
    return out


def enclosed_mask(g, bg, conn=4):
    """bg cells not reachable from the border through bg cells (inside/outside POV)."""
    m = g == bg
    lab, n = label(m, conn)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())
    border.discard(0)
    enc = m.copy()
    for b in border:
        enc &= lab != b
    return enc


def separators(g):
    """Grid-line POV: full rows/cols of a single colour that partition the grid."""
    H, W = g.shape
    rows = [i for i in range(H) if (g[i] == g[i, 0]).all()]
    cols = [j for j in range(W) if (g[:, j] == g[0, j]).all()]
    res = []
    for c in set([int(g[i, 0]) for i in rows] + [int(g[0, j]) for j in cols]):
        rr = [i for i in rows if g[i, 0] == c]; cc = [j for j in cols if g[0, j] == c]
        if len(rr) == H or len(cc) == W:
            continue
        res.append((c, rr, cc))
    return res


def split_by_lines(g, c, rr, cc):
    def spans(lines, n):
        out = []; prev = -1
        for x in sorted(lines) + [n]:
            if x - prev > 1:
                out.append((prev + 1, x))
            prev = x
        return out
    rs, cs = spans(rr, g.shape[0]), spans(cc, g.shape[1])
    return [[(a, b, cdx, d) for (cdx, d) in cs] for (a, b) in rs]


_PART_CACHE = {}


def partitions(g):
    """Return list of (sep_colour, cells[rows][cols] of grids) for each separator colour; also equal splits.
    v2: memoised per grid (was called ~10k times per task by crop/partition/downscale learners)."""
    key = gkey(g)
    r = _PART_CACHE.get(key)
    if r is not None:
        return r
    if len(_PART_CACHE) > 4000:
        _PART_CACHE.clear()
    r = _PART_CACHE[key] = _partitions(g)
    return r


def _partitions(g):
    res = []
    for c, rr, cc in separators(g):
        if not rr and not cc:
            continue
        lay = split_by_lines(g, c, rr, cc)
        cells = [[g[a:b, x:y] for (a, b, x, y) in row] for row in lay]
        if sum(len(r) for r in cells) >= 2:
            res.append((c, cells, lay))
    return res


def sym_flags(g):
    return (bool((g == g[:, ::-1]).all()), bool((g == g[::-1]).all()),
            g.shape[0] == g.shape[1] and bool((g == g.T).all()), bool((g == np.rot90(g, 2)).all()))


# ---------------- per-cell feature POVs (used by the cell-rule hypothesis engine) -------------
COLOR_FEATS = ['nb_u', 'nb_d', 'nb_l', 'nb_r', 'ray_u', 'ray_d', 'ray_l', 'ray_r', 'btw_h', 'btw_v',
               'row_col', 'col_col', 'mir_lr', 'mir_ud', 'mir_t', 'm_main', 'm_other', 'dray_ul', 'dray_ur',
               'dray_dl', 'dray_dr', 'btw_d1', 'btw_d2', 'enc_col', 'o_nbcol', 'bbox_col', 'n8_maj', 'n4_col',
               'rare_col', 'o_rarecol']


def _rays(g, bg):
    H, W = g.shape
    out = {}
    for name, (dy, dx) in {'u': (-1, 0), 'd': (1, 0), 'l': (0, -1), 'r': (0, 1),
                           'ul': (-1, -1), 'ur': (-1, 1), 'dl': (1, -1), 'dr': (1, 1)}.items():
        res = np.full((H, W), 10, dtype=np.int16)
        # process in order so that res[cell] = colour of first non-bg in direction
        ys = range(H - 1, -1, -1) if dy == 1 else range(H)
        if dy == 0:
            ys = range(H)
        xs_order = list(range(W - 1, -1, -1)) if dx == 1 else list(range(W))
        ys_order = list(range(H - 1, -1, -1)) if dy == 1 else list(range(H))
        for y in ys_order:
            for x in xs_order:
                py, px = y + dy, x + dx
                if 0 <= py < H and 0 <= px < W:
                    res[y, x] = g[py, px] if g[py, px] != bg else res[py, px]
        out[name] = res
    return out


_FEAT_CACHE = {}


def cell_features(g, bg):
    key = (gkey(g), bg)
    r = _FEAT_CACHE.get(key)
    if r is not None:
        return r
    H, W = g.shape
    F = {}
    F['c'] = g.astype(np.int16)
    nb = (g != bg)
    F['isbg'] = (~nb).astype(np.int16)
    P = np.pad(g, 1, constant_values=10)
    F['nb_u'] = P[0:H, 1:W + 1]; F['nb_d'] = P[2:H + 2, 1:W + 1]
    F['nb_l'] = P[1:H + 1, 0:W]; F['nb_r'] = P[1:H + 1, 2:W + 2]
    Pn = np.pad(nb, 1).astype(np.int16)
    n4 = Pn[0:H, 1:W + 1] + Pn[2:H + 2, 1:W + 1] + Pn[1:H + 1, 0:W] + Pn[1:H + 1, 2:W + 2]
    n8 = n4 + Pn[0:H, 0:W] + Pn[0:H, 2:W + 2] + Pn[2:H + 2, 0:W] + Pn[2:H + 2, 2:W + 2]
    F['n4'] = n4; F['n8'] = n8
    same4 = sum((P[1 + dy:H + 1 + dy, 1 + dx:W + 1 + dx] == g).astype(np.int16)
                for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)])
    F['same4'] = same4
    F['nb_diag'] = (P[0:H, 0:W] != bg).astype(np.int16) * 1 + (P[0:H, 2:W + 2] != bg) * 2 + \
        (P[2:H + 2, 0:W] != bg) * 4 + (P[2:H + 2, 2:W + 2] != bg) * 8
    R = _rays(g, bg)
    for k in ('u', 'd', 'l', 'r'):
        F['ray_' + k] = R[k]
    for k in ('ul', 'ur', 'dl', 'dr'):
        F['dray_' + k] = R[k]
    F['btw_h'] = np.where(R['l'] == R['r'], R['l'], 10).astype(np.int16)
    F['btw_v'] = np.where(R['u'] == R['d'], R['u'], 10).astype(np.int16)
    F['btw_d1'] = np.where(R['ul'] == R['dr'], R['ul'], 10).astype(np.int16)
    F['btw_d2'] = np.where(R['ur'] == R['dl'], R['ur'], 10).astype(np.int16)
    F['ray_any'] = ((R['u'] != 10).astype(np.int16) + (R['d'] != 10) * 2 + (R['l'] != 10) * 4 + (R['r'] != 10) * 8)
    rc = np.full((H, W), 10, dtype=np.int16); cc = np.full((H, W), 10, dtype=np.int16)
    for i in range(H):
        s = set(g[i].tolist()) - {bg}
        rc[i] = s.pop() if len(s) == 1 else (11 if s else 10)
    for j in range(W):
        s = set(g[:, j].tolist()) - {bg}
        cc[:, j] = s.pop() if len(s) == 1 else (11 if s else 10)
    F['row_col'] = rc; F['col_col'] = cc
    F['r2'] = np.repeat((np.arange(H) % 2)[:, None], W, 1).astype(np.int16)
    F['c2'] = np.repeat((np.arange(W) % 2)[None, :], H, 0).astype(np.int16)
    F['r3'] = np.repeat((np.arange(H) % 3)[:, None], W, 1).astype(np.int16)
    F['c3'] = np.repeat((np.arange(W) % 3)[None, :], H, 0).astype(np.int16)
    F['rc2'] = (F['r2'] + F['c2']) % 2
    bd = np.zeros((H, W), dtype=np.int16); bd[0, :] = 1; bd[-1, :] = 1; bd[:, 0] = 1; bd[:, -1] = 1
    F['border'] = bd
    F['mir_lr'] = g[:, ::-1].astype(np.int16); F['mir_ud'] = g[::-1].astype(np.int16)
    F['mir_t'] = g.T.astype(np.int16) if H == W else np.full((H, W), 10, dtype=np.int16)
    enc = enclosed_mask(g, bg, 4)
    F['enc'] = enc.astype(np.int16)
    F['enc8'] = enclosed_mask(g, bg, 8).astype(np.int16)
    # colour enclosing an enclosed bg region (unique colour of its 4-boundary)
    encc = np.full((H, W), 10, dtype=np.int16)
    if enc.any():
        lab, n = label(enc, 4)
        for i in range(1, n + 1):
            m = lab == i
            dil = _ndi.binary_dilation(m, _S4) if _ndi is not None else m
            s = set(g[dil & ~m].tolist()) - {bg}
            encc[m] = s.pop() if len(s) == 1 else 11
    F['enc_col'] = encc
    cnt = Counter(g.ravel().tolist())
    order = [c for c, _ in cnt.most_common()]
    F['crank'] = np.vectorize(lambda v: min(order.index(v), 4))(g).astype(np.int16)
    # single-colour objects (c4) and multicolour objects (m8)
    for pov, pre in (('c4', 'o'), ('c8', 'p'), ('m8', 'm')):
        obs = objects(g, bg, pov)
        size = np.zeros((H, W), np.int16); rank = np.full((H, W), 9, np.int16); rnk2 = np.full((H, W), 9, np.int16)
        touch = np.zeros((H, W), np.int16); holes = np.zeros((H, W), np.int16); hh = np.zeros((H, W), np.int16)
        ww = np.zeros((H, W), np.int16); rect = np.zeros((H, W), np.int16); shp = np.zeros((H, W), np.int16)
        nsame = np.zeros((H, W), np.int16); edge = np.zeros((H, W), np.int16); ncol = np.zeros((H, W), np.int16)
        main = np.full((H, W), 10, np.int16); other = np.full((H, W), 10, np.int16)
        sizes = sorted(set(o.size for o in obs), reverse=True)
        shapes = Counter(o.shape_key for o in obs)
        shape_ids = {}
        for o in obs:
            m = o.mask
            size[m] = min(o.size, 30)
            rank[m] = min(sizes.index(o.size), 8)
            rnk2[m] = min(len(sizes) - 1 - sizes.index(o.size), 8)
            touch[m] = int(o.touch); holes[m] = min(o.holes, 4); hh[m] = o.h; ww[m] = o.w
            rect[m] = int(o.size == o.h * o.w)
            shp[m] = shape_ids.setdefault(o.shape_key, len(shape_ids) + 1)
            nsame[m] = min(shapes[o.shape_key], 4)
            ncol[m] = o.ncol
            main[m] = o.color
            if o.ncol == 2:
                a, b = sorted(o.colors, key=lambda c: -int((g[m] == c).sum()))
                other[m & (g == a)] = b; other[m & (g == b)] = a
            sh = o.shape
            ip = np.pad(sh, 1)
            inner = ip[:-2, 1:-1] & ip[2:, 1:-1] & ip[1:-1, :-2] & ip[1:-1, 2:]
            e = sh & ~inner
            edge[o.r0:o.r1 + 1, o.c0:o.c1 + 1] |= e.astype(np.int16)
        F[pre + '_size'] = size; F[pre + '_rank'] = rank; F[pre + '_rrank'] = rnk2
        F[pre + '_touch'] = touch; F[pre + '_holes'] = holes
        F[pre + '_h'] = hh; F[pre + '_w'] = ww; F[pre + '_rect'] = rect; F[pre + '_shape'] = shp
        F[pre + '_nsame'] = nsame; F[pre + '_edge'] = edge
        if pre == 'm':
            F['m_ncol'] = ncol; F['m_main'] = main; F['m_other'] = other
            F['m_minor'] = ((g != main) & nb).astype(np.int16)
    # colour of the (unique) other-colour object touching this c4 object
    nbcol = np.full((H, W), 10, np.int16)
    for o in objects(g, bg, 'c4'):
        dil = _ndi.binary_dilation(o.mask, _S8) if _ndi is not None else o.mask
        s = set(g[dil & ~o.mask].tolist()) - {bg, o.color}
        nbcol[o.mask] = s.pop() if len(s) == 1 else (11 if s else 10)
    F['o_nbcol'] = nbcol
    # bbox POV: cells inside the bounding box of a multicolour object (incl. bg cells), relative position
    bbc = np.full((H, W), 10, np.int16); rel = np.zeros((H, W), np.int16); rare = np.full((H, W), 10, np.int16)
    for o in objects(g, bg, 'm8'):
        sl = (slice(o.r0, o.r1 + 1), slice(o.c0, o.c1 + 1))
        bbc[sl] = np.where(bbc[sl] == 10, o.color, 11)
        r = np.zeros((o.h, o.w), np.int16)
        r[0, :] |= 1; r[-1, :] |= 2; r[:, 0] |= 4; r[:, -1] |= 8
        rel[sl] = r
        if o.ncol >= 2:
            cnt_o = Counter(g[o.mask].tolist())
            rare[o.mask] = min(cnt_o, key=lambda c: (cnt_o[c], c))
    F['bbox_col'] = bbc; F['bbox_rel'] = rel; F['o_rarecol'] = rare
    # neighbourhood colour POVs
    stack = np.stack([P[1 + dy:H + 1 + dy, 1 + dx:W + 1 + dx] for dy in (-1, 0, 1) for dx in (-1, 0, 1)
                      if (dy, dx) != (0, 0)], 0)
    maj = np.full((H, W), 10, np.int16)
    for c in range(10):
        if c == bg:
            continue
        cntc = (stack == c).sum(0)
        best = (maj == 10) | (cntc > ((stack == maj[None]).sum(0)))
        maj = np.where((cntc > 0) & best, c, maj)
    F['n8_maj'] = maj
    st4 = np.stack([F['nb_u'], F['nb_d'], F['nb_l'], F['nb_r']], 0)
    n4c = np.full((H, W), 10, np.int16)
    for c in range(10):
        if c == bg:
            continue
        has = (st4 == c).any(0)
        n4c = np.where(has, np.where(n4c == 10, c, 11), n4c)
    F['n4_col'] = n4c
    nzc = [c for c in order if c != bg]
    F['rare_col'] = np.full((H, W), nzc[-1] if nzc else 10, np.int16)
    F['is_rare'] = (g == (nzc[-1] if nzc else -1)).astype(np.int16)
    for k in list(F):
        F[k] = np.ascontiguousarray(F[k]).astype(np.int16)
    if len(_FEAT_CACHE) > 3000:
        _FEAT_CACHE.clear()
    _FEAT_CACHE[key] = F
    return F# =====================================================================================
# WHY  --  input/output diff analysis -> ranked rule-family hypotheses (abduction + Occam)
# =====================================================================================
class TaskCtx:
    def __init__(self, train, tests):
        self.train = train                     # list of (x, y) np arrays
        self.tests = tests                     # list of x
        self.xs = [x for x, _ in train]; self.ys = [y for _, y in train]
        allin = np.concatenate([x.ravel() for x in self.xs + tests])
        cnt = Counter(allin.tolist())
        self.bg = 0 if cnt.get(0, 0) * 1.0 >= max(cnt.values()) * 0.5 else cnt.most_common(1)[0][0]
        self.in_colors = set(allin.tolist())
        self.out_colors = set(np.concatenate([y.ravel() for y in self.ys]).tolist())
        self.new_colors = self.out_colors - set(np.concatenate([x.ravel() for x in self.xs]).tolist())
        self.colors = sorted(self.in_colors | self.out_colors)
        self.analyse()

    def analyse(self):
        tr = self.train
        self.same_shape = all(x.shape == y.shape for x, y in tr)
        oshapes = [y.shape for _, y in tr]
        self.const_out_shape = oshapes[0] if len(set(oshapes)) == 1 else None
        ratios = set()
        for x, y in tr:
            ratios.add((y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]))
        self.ratio = ratios.pop() if len(ratios) == 1 else None
        self.out_smaller = all(y.size < x.size for x, y in tr)
        self.out_larger = all(y.size > x.size for x, y in tr)
        # which cells change
        self.only_add = self.only_del = self.recolor_only = False
        if self.same_shape:
            bg = self.bg
            ch = [(x != y) for x, y in tr]
            self.frac_changed = float(np.mean([c.mean() for c in ch]))
            self.only_add = all(((x[c] == bg)).all() for (x, y), c in zip(tr, ch))
            self.only_del = all(((y[c] == bg)).all() for (x, y), c in zip(tr, ch))
            self.recolor_only = all((x[c] != bg).all() and (y[c] != bg).all() for (x, y), c in zip(tr, ch))
        # output is a sub-grid of input ?
        self.out_is_subgrid = (not self.same_shape) and all(find_sub(x, y) for x, y in tr)
        self.out_in_d4_sub = self.out_is_subgrid or ((not self.same_shape) and all(
            any(find_sub(x, d4(y, n)) for n in D4_NAMES) for x, y in tr))

    def families(self):
        """Rank candidate rule families (hypotheses) by evidence from the diff analysis."""
        fam = []
        if self.same_shape:
            fam += ['cellrule', 'symmetry', 'chain']
        if self.out_in_d4_sub or self.out_smaller:
            fam += ['crop', 'partition', 'downscale', 'symcrop', 'chain']
        if self.out_larger:
            fam += ['tile', 'upscale', 'chain']
        if self.const_out_shape is not None:
            fam += ['const', 'partition']
        fam += ['chain', 'crop', 'tile', 'partition', 'downscale']
        seen = []
        for f in fam:
            if f not in seen:
                seen.append(f)
        return seen


def find_sub(x, y):
    H, W = x.shape; h, w = y.shape
    if h > H or w > W:
        return False
    if h * w > 1 and H * W * h * w > 200000:
        pass
    for i in range(H - h + 1):
        for j in range(W - w + 1):
            if x[i, j] == y[0, 0] and (x[i:i + h, j:j + w] == y).all():
                return True
    return False# =====================================================================================
# HOW (part 1)  --  typed DSL of grid->grid primitives, instantiated per task
# =====================================================================================
def crop_nonbg(g, bg):
    ys, xs = np.nonzero(g != bg)
    if len(ys) == 0:
        return None
    return g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def crop_color(g, c):
    ys, xs = np.nonzero(g == c)
    if len(ys) == 0:
        return None
    return g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def gravity_cells(g, d, bg):
    r = {'down': 0, 'left': 1, 'up': 2, 'right': 3}[d]
    x = np.rot90(g, r); out = np.full_like(x, bg)
    for j in range(x.shape[1]):
        col = x[:, j]; nb = col[col != bg]
        if len(nb):
            out[x.shape[0] - len(nb):, j] = nb
    return np.ascontiguousarray(np.rot90(out, -r))


def gravity_objects(g, d, bg, pov='m8'):
    r = {'down': 0, 'left': 1, 'up': 2, 'right': 3}[d]
    x = np.ascontiguousarray(np.rot90(g, r))
    obs = objects(x, bg, pov)
    if len(obs) > 30:
        return None
    obs = sorted(obs, key=lambda o: -o.r1)
    out = np.full_like(x, bg); H = x.shape[0]
    for o in obs:
        ys, xs = np.nonzero(o.mask); vals = x[ys, xs]
        k = 0
        while True:
            ny = ys + k + 1
            if ny.max() >= H or (out[ny, xs] != bg).any():
                break
            k += 1
        out[ys + k, xs] = vals
    return np.ascontiguousarray(np.rot90(out, -r))


def attract(g, anchor, bg):
    """Move every non-anchor object (m8) in a straight line until it touches an anchor-colour cell."""
    amask = g == anchor
    if not amask.any():
        return None
    out = g.copy(); out[(g != bg) & ~amask] = bg
    obs = [o for o in objects(g, bg, 'm8') if anchor not in o.colors]
    if not obs or len(obs) > 20:
        return None
    H, W = g.shape
    for o in obs:
        ys, xs = np.nonzero(o.mask); vals = g[ys, xs]
        best = None
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            k = 0; hit = False
            while True:
                ny, nx = ys + (k + 1) * dy, xs + (k + 1) * dx
                if ny.min() < 0 or nx.min() < 0 or ny.max() >= H or nx.max() >= W:
                    break
                if amask[ny, nx].any():
                    hit = True; break
                if (out[ny, nx] != bg).any():
                    break
                k += 1
            if hit and (best is None or k < best[0]):
                best = (k, dy, dx)
        if best is None:
            out[ys, xs] = vals
        else:
            k, dy, dx = best
            out[ys + k * dy, xs + k * dx] = vals
    return out


def fill_enclosed(g, c, bg):
    enc = enclosed_mask(g, bg, 4)
    if not enc.any():
        return None
    out = g.copy(); out[enc] = c
    return out


def remove_color(g, c, bg):
    if not (g == c).any():
        return None
    out = g.copy(); out[g == c] = bg
    return out


def keep_color(g, c, bg):
    if not (g == c).any():
        return None
    return np.where(g == c, g, bg).astype(g.dtype)


def denoise(g, bg, maxsize=1):
    out = g.copy(); ch = False
    for o in objects(g, bg, 'm8'):
        if o.size <= maxsize:
            out[o.mask] = bg; ch = True
    return out if ch else None


def dedup_rows(g):
    keep = [0] + [i for i in range(1, g.shape[0]) if not (g[i] == g[i - 1]).all()]
    return g[keep]


def remove_bg_lines(g, bg):
    r = [i for i in range(g.shape[0]) if (g[i] != bg).any()]
    c = [j for j in range(g.shape[1]) if (g[:, j] != bg).any()]
    if not r or not c:
        return None
    return g[np.ix_(r, c)]


def downsample(g, k):
    if g.shape[0] % k or g.shape[1] % k:
        return None
    return g[::k, ::k]


def largest_rect_crop(g, bg):
    obs = objects(g, bg, 'c4')
    rects = [o for o in obs if o.size == o.h * o.w and o.size >= 4]
    if not rects:
        return None
    o = max(rects, key=lambda o: o.size)
    return g[o.r0:o.r1 + 1, o.c0:o.c1 + 1]


def quad_mirror(g):
    top = np.hstack([g, g[:, ::-1]]); return np.vstack([top, top[::-1]])


def quad_rot(g):
    if g.shape[0] != g.shape[1]:
        return None
    return np.vstack([np.hstack([g, np.rot90(g, 3)]), np.hstack([np.rot90(g, 1), np.rot90(g, 2)])])


# --------------- symmetry / periodicity completion (pattern completion POV) ---------------
_SYM_CACHE = {}


def _sym_maps(H, W):
    """v2: cached per grid size, with clipped index arrays + in-bounds mask precomputed."""
    r = _SYM_CACHE.get((H, W))
    if r is None:
        r = []
        for name, mi, mj in _sym_maps_raw(H, W):
            inb = (mi >= 0) & (mi < H) & (mj >= 0) & (mj < W)
            r.append((name, np.where(inb, mi, 0), np.where(inb, mj, 0), inb))
        _SYM_CACHE[(H, W)] = r
    return r


def _sym_maps_raw(H, W):
    ii, jj = np.mgrid[0:H, 0:W]
    maps = []
    for s in range(W // 2, 2 * W - 1 - W // 2):
        maps.append(('lr%d' % s, ii, s - jj))
    for s in range(H // 2, 2 * H - 1 - H // 2):
        maps.append(('ud%d' % s, s - ii, jj))
    for k in range(-(H // 2), H // 2 + 1):
        maps.append(('tr%d' % k, jj + k, ii - k))
    for k in range(min(H, W) // 2, H + W - 1 - min(H, W) // 2):
        maps.append(('at%d' % k, k - jj, k - ii))
    for s1 in range(H // 2, 2 * H - 1 - H // 2):
        for s2 in range(W // 2, 2 * W - 1 - W // 2):
            if abs(s1 - (H - 1)) <= 3 and abs(s2 - (W - 1)) <= 3:
                maps.append(('r2_%d_%d' % (s1, s2), s1 - ii, s2 - jj))
    return maps


def sym_fill(g, noise, min_frac=0.25):
    H, W = g.shape
    known = g != noise
    nk = int(known.sum())
    if nk == g.size or nk < 4:
        return None
    good = []
    for name, si, sj, inb in _sym_maps(H, W):
        both = inb & known & known[si, sj]
        n = int(both.sum())
        if n < max(4, min_frac * nk):
            continue
        if (g[both] == g[si, sj][both]).all():
            good.append((si, sj, inb))
    if not good:
        return None
    out = g.copy(); kn = known.copy()
    for _ in range(4):
        changed = False
        for si, sj, inb in good:
            m = (~kn) & inb & kn[si, sj]
            if m.any():
                out[m] = out[si, sj][m]; kn |= m; changed = True
        if kn.all() or not changed:
            break
    if not kn.all():
        return None
    return out


def period_fill(g, noise):
    H, W = g.shape
    known = g != noise
    if known.all() or known.sum() < 4:
        return None
    cands = sorted([(py, px) for py in range(1, H + 1) for px in range(1, W + 1) if py * px < H * W],
                   key=lambda t: (t[0] * t[1], t))
    ii, jj = np.mgrid[0:H, 0:W]
    for py, px in cands:
        cls = (ii % py) * px + (jj % px)
        kv = g[known]; kc = cls[known]
        tab = np.full(py * px, -1, dtype=np.int32)
        tab[kc] = kv
        if (tab[kc] != kv).any():
            continue
        if (tab[cls[~known]] < 0).any():
            continue
        out = g.copy(); out[~known] = tab[cls[~known]]
        return out
    return None


def build_primitives(ctx):
    """Return list of (name, fn, cost, family).  cost ~ description length (Occam prior);
    the memory module (priors from training successes) lowers the cost of frequently useful primitives."""
    bg = ctx.bg
    P = []
    add = lambda n, f, c, fam: P.append((n, f, c, fam))
    for n in D4_NAMES[1:]:
        add(n, (lambda g, n=n: d4(g, n)), 1.0, 'geom')
    add('crop_nonbg', lambda g: crop_nonbg(g, bg), 1.0, 'crop')
    add('remove_bg_lines', lambda g: remove_bg_lines(g, bg), 1.5, 'crop')
    add('largest_rect', lambda g: largest_rect_crop(g, bg), 2.0, 'crop')
    for c in sorted(ctx.in_colors):
        if c != bg:
            add('crop_col%d' % c, (lambda g, c=c: crop_color(g, c)), 1.6, 'crop')
            add('remove_col%d' % c, (lambda g, c=c: remove_color(g, c, bg)), 1.6, 'color')
            add('keep_col%d' % c, (lambda g, c=c: keep_color(g, c, bg)), 1.8, 'color')
            add('attract%d' % c, (lambda g, c=c: attract(g, c, bg)), 2.2, 'move')
    for c in sorted(ctx.out_colors):
        add('fill_enc%d' % c, (lambda g, c=c: fill_enclosed(g, c, bg)), 1.6, 'fill')
    # v2 invariant: completion primitives erase colour c, so c must be absent from every train output
    gone = [c for c in sorted(ctx.in_colors) if all(not (y == c).any() for y in ctx.ys)]
    for c in gone:
        add('symfill%d' % c, (lambda g, c=c: sym_fill(g, c)), 1.8, 'pattern')
        add('perfill%d' % c, (lambda g, c=c: period_fill(g, c)), 1.8, 'pattern')
    for d in ('down', 'up', 'left', 'right'):
        add('grav_' + d, (lambda g, d=d: gravity_cells(g, d, bg)), 1.6, 'move')
        add('ograv_' + d, (lambda g, d=d: gravity_objects(g, d, bg)), 1.8, 'move')
    for k in (2, 3, 4):
        add('up%d' % k, (lambda g, k=k: np.kron(g, np.ones((k, k), dtype=g.dtype))), 1.3, 'scale')
        add('down%d' % k, (lambda g, k=k: downsample(g, k)), 1.5, 'scale')
    add('hmirror', lambda g: np.hstack([g, g[:, ::-1]]), 1.4, 'tile')
    add('vmirror', lambda g: np.vstack([g, g[::-1]]), 1.4, 'tile')
    add('quad_mirror', quad_mirror, 1.6, 'tile')
    add('quad_rot', quad_rot, 1.8, 'tile')
    add('tile2x2', lambda g: np.tile(g, (2, 2)), 1.6, 'tile')
    add('tile1x2', lambda g: np.tile(g, (1, 2)), 1.6, 'tile')
    add('tile2x1', lambda g: np.tile(g, (2, 1)), 1.6, 'tile')
    add('left_half', lambda g: g[:, :g.shape[1] // 2], 1.6, 'crop')
    add('right_half', lambda g: g[:, (g.shape[1] + 1) // 2:], 1.6, 'crop')
    add('top_half', lambda g: g[:g.shape[0] // 2], 1.6, 'crop')
    add('bottom_half', lambda g: g[(g.shape[0] + 1) // 2:], 1.6, 'crop')
    add('denoise1', lambda g: denoise(g, bg, 1), 1.8, 'clean')
    add('denoise2', lambda g: denoise(g, bg, 2), 2.0, 'clean')
    add('dedup_rows', dedup_rows, 1.8, 'crop')
    add('dedup_cols', lambda g: dedup_rows(g.T).T, 1.8, 'crop')
    return P + extra_primitives(ctx)


def mirror_about(g, c, bg, mode):
    ys, xs = np.nonzero(g == c)
    if len(ys) == 0:
        return None
    cy2, cx2 = ys.min() + ys.max(), xs.min() + xs.max()   # 2*centre
    H, W = g.shape
    out = g.copy()
    py, px = np.nonzero((g != bg) & (g != c))
    if len(py) == 0:
        return None
    vals = g[py, px]
    maps = []
    if mode in ('lr', 'both'):
        maps.append((py, cx2 - px))
    if mode in ('ud', 'both'):
        maps.append((cy2 - py, px))
    if mode == 'both':
        maps.append((cy2 - py, cx2 - px))
    for my, mx in maps:
        ok = (my >= 0) & (my < H) & (mx >= 0) & (mx < W)
        my, mx, v = my[ok], mx[ok], vals[ok]
        free = out[my, mx] == bg
        out[my[free], mx[free]] = v[free]
    return out


def obj_transform(g, bg, op):
    out = np.full_like(g, bg)
    for o in objects(g, bg, 'm8'):
        s = d4(o.sub, op)
        if s.shape != o.sub.shape:
            return None
        sl = (slice(o.r0, o.r1 + 1), slice(o.c0, o.c1 + 1))
        out[sl] = np.where(s != bg, s, out[sl])
    return out


def extra_primitives(ctx):
    bg = ctx.bg
    P = []
    for c in sorted(ctx.in_colors):
        if c == bg:
            continue
        for m in ('lr', 'ud', 'both'):
            P.append(('mirror_%s_%d' % (m, c), (lambda g, c=c, m=m: mirror_about(g, c, bg, m)), 2.0, 'pattern'))
    def pad_edge(g):
        o = np.pad(g, 1, mode='edge')
        o[0, 0] = o[0, -1] = o[-1, 0] = o[-1, -1] = bg
        return o
    P.append(('pad_edge', pad_edge, 2.0, 'scale'))
    P.append(('pad_bg', lambda g: np.pad(g, 1, constant_values=bg), 2.0, 'scale'))
    P.append(('unpad', lambda g: g[1:-1, 1:-1] if min(g.shape) > 2 else None, 1.8, 'crop'))
    for op in ('fliplr', 'flipud', 'rot180', 'transpose'):
        P.append(('objT_' + op, (lambda g, op=op: obj_transform(g, bg, op)), 1.9, 'object'))
    return P# =====================================================================================
# HOW (part 2)  --  rule-family learners: fit parameters from the train pairs, predict tests
# Each learner returns a list of Sol(name, cost, preds, q)   (q = coherence evidence, e.g. LOO)
# =====================================================================================
class Sol:
    __slots__ = ('name', 'cost', 'preds', 'q', 'fam')

    def __init__(self, name, cost, preds, q=1.0, fam=''):
        self.name, self.cost, self.preds, self.q, self.fam = name, cost, preds, q, fam

    def __repr__(self):
        return 'Sol(%s, %.2f, q=%.2f)' % (self.name, self.cost, self.q)


def learn_colormap(xs, ys, ts):
    m = {}
    for p, t in zip(xs, ys):
        if p.shape != t.shape:
            return None
        for a, b in set(zip(p.ravel().tolist(), t.ravel().tolist())):
            if m.setdefault(a, b) != b:
                return None
    if all(a == b for a, b in m.items()):
        return None
    preds = []
    for t in ts:
        lut = np.arange(10, dtype=np.int16)
        for a, b in m.items():
            lut[a] = b
        preds.append(lut[t])
    return [Sol('cmap', 1.0 + 0.1 * sum(a != b for a, b in m.items()), preds, 1.0, 'color')]


def learn_const(xs, ys, ts):
    if all(same(y, ys[0]) for y in ys):
        return [Sol('const', 2.0, [ys[0].copy() for _ in ts], 0.6, 'const')]
    return []


# ---------------------------------- cell rule engine --------------------------------------
_ACT_KEEP = -1


def _dense(a, b):
    allv = np.concatenate([a, b])
    u, inv = np.unique(allv, return_inverse=True)
    return inv[:len(a)].astype(np.int64), inv[len(a):].astype(np.int64), len(u)


def learn_cellrule(ctx, xs, ys, ts, budget, max_sols=6, triples=True):
    """Hypothesis engine for same-shape tasks: out[cell] = action(key(features of cell)),
    action in {keep input colour, constant colour, copy colour of a colour-valued feature}.
    Searches feature subsets of size 0..3 (Occam: fewest features first)."""
    if any(x.shape != y.shape for x, y in zip(xs, ys)) or not ts:
        return []
    bg = ctx.bg
    Fs = [cell_features(x, bg) for x in xs]; Ft = [cell_features(t, bg) for t in ts]
    names = [n for n in Fs[0] if n != 'c' or True]
    tr = {n: np.concatenate([F[n].ravel() for F in Fs]) for n in names}
    te = {n: np.concatenate([F[n].ravel() for F in Ft]) for n in names}
    out = np.concatenate([y.ravel() for y in ys]).astype(np.int64)
    cin = tr['c'].astype(np.int64)
    N = len(out)
    D, Dt, card = {}, {}, {}
    for n in names:
        D[n], Dt[n], card[n] = _dense(tr[n], te[n])
    names = [n for n in names if card[n] > 1]
    mis_keep = (out != cin).astype(np.float64)
    cfe = [f for f in COLOR_FEATS if f in tr]
    mis_f = {f: (out != tr[f]).astype(np.float64) for f in cfe}
    # a feature subset must be able to "see" the changed cells; quick filter
    changed = mis_keep > 0
    if not changed.any():
        return []

    out2 = out * out
    def evaluate(S, allow_unseen=False):
        K = 1
        for n in S:
            K *= card[n]
        if K > 4000000:
            return None
        k = np.zeros(N, np.int64); kt = np.zeros(len(te['c']), np.int64)
        for n in S:
            k = k * card[n] + D[n]; kt = kt * card[n] + Dt[n]
        tot = np.bincount(k, minlength=K)
        if not allow_unseen and not (tot[kt] > 0).all():
            return None
        keep_bad = np.bincount(k, mis_keep, K) > 0
        s1 = np.bincount(k, out, K); s2 = np.bincount(k, out2, K)
        totc = np.maximum(tot, 1)
        const_ok = (s2 * totc == s1 * s1)
        present = tot > 0
        need = keep_bad & ~const_ok & present
        copy_ok = {}
        if need.any():
            for f in cfe:
                ok = np.bincount(k, mis_f[f], K) == 0
                copy_ok[f] = ok
                need = need & ~ok
                if not need.any():
                    break
            if need.any():
                return None
        u = np.nonzero(present)[0]
        omin = (s1 / totc).round().astype(np.int64)
        return u, None, keep_bad, const_ok, omin, copy_ok, k, kt

    def build(S, ev, prefer_copy):
        u, inv, keep_bad, const_ok, omin, copy_ok, k, kt = ev
        if prefer_copy:
            K = len(keep_bad)
            copy_ok = dict(copy_ok)
            for f in cfe:
                if f not in copy_ok:
                    copy_ok[f] = np.bincount(k, mis_f[f], K) == 0
        acts = []
        ncopy = nconst = 0
        for gi in u.tolist():
            if not keep_bad[gi]:
                acts.append(('k',))
                continue
            cf = [f for f in cfe if f in copy_ok and copy_ok[f][gi]]
            if const_ok[gi] and (not prefer_copy or not cf):
                acts.append(('c', int(omin[gi]))); nconst += 1
            elif cf:
                acts.append(('f', cf[0])); ncopy += 1
            else:
                acts.append(('c', int(omin[gi]))); nconst += 1
        lut = dict(zip(u.tolist(), acts))
        preds = []; off = 0
        for t, F in zip(ts, Ft):
            n = t.size
            kk = kt[off:off + n]; off += n
            p = t.ravel().astype(np.int16).copy()
            for i, key in enumerate(kk.tolist()):
                a = lut.get(key, ('k',))
                if a[0] == 'c':
                    p[i] = a[1]
                elif a[0] == 'f':
                    v = F[a[1]].ravel()[i]
                    if v > 9:
                        return None
                    p[i] = v
            preds.append(p.reshape(t.shape))
        cost = 2.0 + 1.2 * len(S) + 0.05 * (nconst + ncopy)
        return Sol('cell[%s]%s' % (','.join(S), '/cp' if prefer_copy else ''), cost, preds, 1.0, 'cellrule')

    sols = []
    seen_pred = set()

    def try_S(S, unseen=False):
        ev = evaluate(S, unseen)
        if ev is None:
            return False
        for pc in (False, True):
            s = build(S, ev, pc)
            if s is None:
                continue
            if unseen:
                s.name += '/u'; s.q = 0.6; s.cost += 0.5
            key = tuple(gkey(p) for p in s.preds)
            if key in seen_pred:
                continue
            seen_pred.add(key)
            sols.append(s)
        return True

    # rank features by how well they alone separate changed from unchanged cells (purity)
    scores = []
    chf = changed.astype(np.float64)
    for n in names:
        ch = np.bincount(D[n], chf, card[n]); tot = np.bincount(D[n], None, card[n])
        pur = float(np.sum(np.minimum(ch, tot - ch))) / N
        scores.append((pur, n))
    ranked = [n for _, n in sorted(scores)]
    found_level = None
    try_S(())
    if sols:
        found_level = 0
    for n in names:
        budget.check()
        if try_S((n,)):
            found_level = 1 if found_level is None else found_level
    if found_level is None or len(sols) < 2:
        top = ranked[:45]
        prs = list(itertools.combinations(top, 2))
        base = [n for n in ('c', 'isbg', 'o_size', 'm_main') if n in names]
        prs += [(a, b) for a in base for b in names if b not in top and a != b]
        for a, b in prs:
            if budget.left() < 0.5:
                break
            if try_S((a, b)) and found_level is None:
                found_level = 2
            if len(sols) >= max_sols * 2:
                break
    if triples and found_level is None and budget.left() > 1:
        for S in itertools.combinations(ranked[:14], 3):
            if budget.left() < 0.5:
                break
            try_S(S)
            if len(sols) >= max_sols:
                break
    if not sols and budget.left() > 0.3:
        # REVIEW / repair: allow unseen feature keys on test to default to 'keep'
        for n in names:
            if budget.left() < 0.2 or len(sols) >= 4:
                break
            try_S((n,), True)
        if not sols:
            for a, b in itertools.combinations(ranked[:20], 2):
                if budget.left() < 0.2 or len(sols) >= 4:
                    break
                try_S((a, b), True)
    sols.sort(key=lambda s: s.cost)
    sols = sols[:max_sols]
    # Q: leave-one-out coherence for the top hypotheses (does the rule generalise across pairs?)
    if len(xs) >= 2:
        for s in sols[:3]:
            if s.name.endswith('/u'):
                continue
            S = tuple(s.name[5:s.name.index(']')].split(',')) if s.name[5] != ']' else ()
            ok = 0
            for i in range(len(xs)):
                if budget.left() < 0.3:
                    break
                sub = learn_cellrule_fixed(ctx, [x for j, x in enumerate(xs) if j != i],
                                           [y for j, y in enumerate(ys) if j != i], xs[i], S)
                ok += int(sub is not None and same(sub, ys[i]))
            s.q = 0.7 + 0.3 * ok / len(xs)
    return sols


def learn_cellrule_fixed(ctx, xs, ys, t, S):
    bg = ctx.bg
    Fs = [cell_features(x, bg) for x in xs]; Ft = cell_features(t, bg)
    lut = {}
    for x, y, F in zip(xs, ys, Fs):
        keys = list(zip(*[F[n].ravel().tolist() for n in S])) if S else [()] * x.size
        for kk, a, b in zip(keys, x.ravel().tolist(), y.ravel().tolist()):
            lut.setdefault(kk, set()).add((a, b))
    keys = list(zip(*[Ft[n].ravel().tolist() for n in S])) if S else [()] * t.size
    p = t.ravel().copy()
    for i, kk in enumerate(keys):
        if kk not in lut:
            return None
        pairs = lut[kk]
        if all(a == b for a, b in pairs):
            continue
        outs = set(b for _, b in pairs)
        if len(outs) == 1:
            p[i] = outs.pop()
        else:
            return None
    return p.reshape(t.shape)


# ---------------------------------- crop / select ----------------------------------------
CROP_POVS = ('c4', 'c8', 'm4', 'm8', 'col')


_RC_CACHE = {}


def region_candidates(g, bg):
    """Candidate sub-grids from several POVs, each with a property vector.  v2: memoised per (grid, bg)."""
    key = (gkey(g), bg)
    r = _RC_CACHE.get(key)
    if r is None:
        if len(_RC_CACHE) > 2000:
            _RC_CACHE.clear()
        r = _RC_CACHE[key] = _region_candidates(g, bg)
    return r


def _region_candidates(g, bg):
    cands = []
    for pov in CROP_POVS:
        obs = objects(g, bg, pov)
        if len(obs) > 40:
            continue
        shapes = Counter(o.shape_key for o in obs); cols = Counter(o.color for o in obs)
        subs = Counter(gkey(o.sub) for o in obs)
        for o in obs:
            box = g[o.r0:o.r1 + 1, o.c0:o.c1 + 1]
            props = {'size': o.size, 'area': o.h * o.w, 'h': o.h, 'w': o.w, 'ncol': o.ncol, 'color': o.color,
                     'top': o.r0, 'left': o.c0, 'bottom': -o.r1, 'right': -o.c1, 'holes': o.holes,
                     'touch': int(o.touch), 'nshape': shapes[o.shape_key], 'ncolor': cols[o.color],
                     'nsub': subs[gkey(o.sub)], 'boxcols': len(set(box.ravel().tolist())),
                     'boxnonbg': int((box != bg).sum()), 'dens': round(o.size / (o.h * o.w), 3),
                     'symlr': int((o.sub == o.sub[:, ::-1]).all()), 'symud': int((o.sub == o.sub[::-1]).all())}
            cands.append((pov, 'mask', o.sub, props))
            cands.append((pov, 'box', box, props))
            if o.h > 2 and o.w > 2:
                cands.append((pov, 'inner', box[1:-1, 1:-1], props))
    for c, cells, lay in partitions(g):
        flat = [x for row in cells for x in row]
        if len(flat) > 30:
            continue
        subs = Counter(gkey(x) for x in flat)
        ncs = Counter(len(set(x.ravel().tolist()) - {bg}) for x in flat)
        for k, x in enumerate(flat):
            nc = len(set(x.ravel().tolist()) - {bg})
            props = {'size': int((x != bg).sum()), 'ncol': nc, 'index': k, 'nsub': subs[gkey(x)],
                     'nnc': ncs[nc], 'area': x.size, 'symlr': int((x == x[:, ::-1]).all()),
                     'symud': int((x == x[::-1]).all()), 'rindex': len(flat) - 1 - k}
            cands.append(('part%d' % c, 'cell', x, props))
    return cands


def _select(cands, sel):
    kind, prop = sel[0], sel[1]
    vals = [c[3].get(prop) for c in cands]
    idx = [i for i, v in enumerate(vals) if v is not None]
    if not idx:
        return None
    if kind == 'max' or kind == 'min':
        f = max if kind == 'max' else min
        best = f(vals[i] for i in idx)
        w = [i for i in idx if vals[i] == best]
        return w[0] if len(w) == 1 else None
    if kind == 'uniq':
        cnt = Counter(vals[i] for i in idx)
        w = [i for i in idx if cnt[vals[i]] == 1]
        return w[0] if len(w) == 1 else None
    if kind == 'eq':
        w = [i for i in idx if vals[i] == sel[2]]
        return w[0] if len(w) == 1 else None


def learn_crop(ctx, xs, ys, ts, budget):
    if all(x.shape == y.shape for x, y in zip(xs, ys)):
        return []
    bg = ctx.bg
    C = [region_candidates(x, bg) for x in xs]
    Ct = [region_candidates(t, bg) for t in ts]
    sols = []
    groups = sorted(set((c[0], c[1]) for c in C[0]))
    for pov, var in groups:
        budget.check()
        cs = [[c for c in cc if c[0] == pov and c[1] == var] for cc in C]
        ct = [[c for c in cc if c[0] == pov and c[1] == var] for cc in Ct]
        if any(not c for c in cs + ct):
            continue
        for T in D4_NAMES:
            match = []
            for cc, y in zip(cs, ys):
                m = set(i for i, c in enumerate(cc) if c[2].size == y.size and same(d4(c[2], T), y))
                match.append(m)
            if any(not m for m in match):
                continue
            props = set(cs[0][next(iter(match[0]))][3])
            sels = []
            for p in sorted(props):
                sels += [('max', p), ('min', p), ('uniq', p)]
                v = cs[0][next(iter(match[0]))][3][p]
                sels.append(('eq', p, v))
            # single-candidate case
            for sel in sels:
                ok = all(_select(cc, sel) in m for cc, m in zip(cs, match))
                if not ok:
                    continue
                preds = []
                for cc in ct:
                    i = _select(cc, sel)
                    if i is None:
                        preds = None; break
                    preds.append(d4(cc[i][2], T).copy())
                if preds is None:
                    continue
                cost = 2.5 + (0.5 if sel[0] == 'eq' else 0) + (0.6 if T != 'id' else 0) + \
                    (0.3 if var != 'mask' else 0)
                sols.append(Sol('crop[%s,%s,%s,%s]' % (pov, var, '/'.join(map(str, sel)), T), cost, preds,
                                0.9, 'crop'))
            if len(sols) > 30:
                break
    sols.sort(key=lambda s: s.cost)
    return sols[:12]


# ------------------------------ partition & combine (logical ops) -------------------------
def split_parts(g, bg, mode):
    if mode[0] == 'sep':
        for c, cells, lay in partitions(g):
            flat = [x for row in cells for x in row]
            if len(flat) == mode[1] and len(set(x.shape for x in flat)) == 1:
                return flat
        return None
    _, nr, nc = mode
    H, W = g.shape
    if H % nr or W % nc:
        return None
    h, w = H // nr, W // nc
    return [g[i * h:(i + 1) * h, j * w:(j + 1) * w] for i in range(nr) for j in range(nc)]


def learn_partition(ctx, xs, ys, ts, budget):
    bg = ctx.bg
    modes = []
    for k in (2, 3, 4, 6, 9):
        modes.append(('sep', k))
    for nr, nc in ((1, 2), (2, 1), (1, 3), (3, 1), (2, 2), (1, 4), (4, 1)):
        modes.append(('eq', nr, nc))
    sols = []
    for mode in modes:
        P = [split_parts(x, bg, mode) for x in xs]
        if any(p is None or p[0].shape != y.shape for p, y in zip(P, ys)):
            continue
        Pt = [split_parts(t, bg, mode) for t in ts]
        if any(p is None for p in Pt):
            continue
        k = len(P[0])
        # local background per part set: most common colour across parts
        # (a) boolean table: key = tuple(part != bg) -> output colour
        tab = {}; ok = True
        for p, y in zip(P, ys):
            st = np.stack([(q != bg) for q in p], 0)
            keys = [tuple(st[:, i, j]) for i in range(y.shape[0]) for j in range(y.shape[1])]
            for kk, v in zip(keys, y.ravel().tolist()):
                if tab.setdefault(kk, v) != v:
                    ok = False; break
            if not ok:
                break
        if ok:
            preds = []
            for p in Pt:
                st = np.stack([(q != bg) for q in p], 0)
                h, w = p[0].shape
                o = np.zeros((h, w), np.int16)
                for i in range(h):
                    for j in range(w):
                        kk = tuple(st[:, i, j])
                        if kk not in tab:
                            # unseen key: fall back to symmetric guess (count-based)
                            cands = [v for kk2, v in tab.items() if sum(kk2) == sum(kk)]
                            if len(set(cands)) != 1:
                                o = None; break
                            tab_v = cands[0]
                        else:
                            tab_v = tab[kk]
                        o[i, j] = tab_v
                    if o is None:
                        break
                if o is None:
                    preds = None; break
                preds.append(o)
            if preds:
                sols.append(Sol('part_bool%s' % (mode,), 3.0, preds, 0.95, 'partition'))
        # (b) priority overlay
        if k <= 4:
            for perm in itertools.permutations(range(k)):
                def ov(p):
                    o = np.full(p[0].shape, bg, np.int16)
                    for idx in reversed(perm):
                        o = np.where(p[idx] != bg, p[idx], o)
                    return o
                if all(same(ov(p), y) for p, y in zip(P, ys)):
                    sols.append(Sol('part_overlay%s%s' % (mode, perm), 3.2, [ov(p) for p in Pt], 0.95, 'partition'))
                    break
    return sols


# ---------------------------------------- tiling -------------------------------------------
def learn_tile(ctx, xs, ys, ts, budget):
    bg = ctx.bg
    facs = set()
    for x, y in zip(xs, ys):
        if y.shape[0] % x.shape[0] or y.shape[1] % x.shape[1]:
            return []
        facs.add((y.shape[0] // x.shape[0], y.shape[1] // x.shape[1]))
    if len(facs) != 1:
        # fractal with variable size: factor == input size
        if all(y.shape == (x.shape[0] ** 2, x.shape[1] ** 2) for x, y in zip(xs, ys)):
            facs = {None}
        else:
            return []
    fa = facs.pop()
    sols = []
    if fa is not None and fa != (1, 1):
        a, b = fa
        ops = D4_NAMES + ['bg']
        plan = {}
        for i in range(a):
            for j in range(b):
                found = None
                for op in ops:
                    good = True
                    for x, y in zip(xs, ys):
                        h, w = x.shape
                        blk = y[i * h:(i + 1) * h, j * w:(j + 1) * w]
                        if op == 'bg':
                            r = np.full_like(x, bg)
                        else:
                            r = d4(x, op)
                            if r.shape != blk.shape:
                                good = False; break
                        if not same(r, blk):
                            good = False; break
                    if good:
                        found = op; break
                if found is None:
                    plan = None; break
                plan[(i, j)] = found
            if plan is None:
                break
        if plan:
            preds = []
            for t in ts:
                h, w = t.shape
                o = np.zeros((a * h, b * w), np.int16)
                for (i, j), op in plan.items():
                    o[i * h:(i + 1) * h, j * w:(j + 1) * w] = np.full_like(t, bg) if op == 'bg' else d4(t, op)
                preds.append(o)
            sols.append(Sol('tile%s' % (sorted(set(plan.values())),), 2.0 + 0.1 * len(set(plan.values())), preds,
                            1.0, 'tile'))
    # fractal / self-kron family: block (i,j) = content(x) if mask(x)[i,j]
    if all(y.shape == (x.shape[0] ** 2, x.shape[1] ** 2) for x, y in zip(xs, ys)):
        def masks(x):
            cnt = Counter(x.ravel().tolist())
            nz = [c for c, _ in cnt.most_common() if c != bg]
            return {'nonbg': x != bg, 'bg': x == bg,
                    'rare': (x == nz[-1]) if nz else None, 'common': (x == nz[0]) if nz else None,
                    'mode': x == cnt.most_common(1)[0][0]}
        def contents(x, cij):
            cnt = Counter(x.ravel().tolist())
            nz = [c for c, _ in cnt.most_common() if c != bg]
            inv = np.where(x == bg, nz[0] if nz else bg, bg).astype(np.int16)
            return {'x': x, 'inv': inv, 'recol': np.where(x != bg, cij, bg).astype(np.int16)}
        def frac(x, mk, ck):
            h, w = x.shape
            m = masks(x)[mk]
            if m is None:
                return None
            o = np.full((h * h, w * w), bg, np.int16)
            for i in range(h):
                for j in range(w):
                    if m[i, j]:
                        o[i * h:(i + 1) * h, j * w:(j + 1) * w] = contents(x, x[i, j])[ck]
            return o
        for mk in ('nonbg', 'bg', 'rare', 'common', 'mode'):
            for ck in ('x', 'inv', 'recol'):
                if all(same(frac(x, mk, ck), y) for x, y in zip(xs, ys)):
                    preds = [frac(t, mk, ck) for t in ts]
                    if all(p is not None for p in preds):
                        sols.append(Sol('fractal_%s_%s' % (mk, ck), 2.2, preds, 1.0, 'tile'))
        # plain tiling with factor = input size
        def tl(x):
            return np.tile(x, x.shape)
        if all(same(tl(x), y) for x, y in zip(xs, ys)):
            sols.append(Sol('tile_by_size', 2.4, [tl(t) for t in ts], 1.0, 'tile'))
    return sols


# ------------------------------------ downscale ------------------------------------------
def learn_downscale(ctx, xs, ys, ts, budget):
    bg = ctx.bg
    sols = []

    def blocks_of(x, oshape=None, fac=None):
        if fac is not None:
            a, b = fac
            if x.shape[0] % a or x.shape[1] % b:
                return None
            H, W = x.shape[0] // a, x.shape[1] // b
            return [[x[i * a:(i + 1) * a, j * b:(j + 1) * b] for j in range(W)] for i in range(H)]
        return None

    def reduce_blk(blk, mode):
        nb = blk[blk != bg]
        if mode == 'any':
            if len(nb) == 0:
                return bg
            return Counter(nb.tolist()).most_common(1)[0][0]
        if mode == 'maj':
            return Counter(blk.ravel().tolist()).most_common(1)[0][0]
        if mode == 'all':
            return Counter(nb.tolist()).most_common(1)[0][0] if len(nb) == blk.size else bg
        if mode == 'min':
            if len(nb) == 0:
                return bg
            return Counter(nb.tolist()).most_common()[-1][0]

    facs = set()
    for x, y in zip(xs, ys):
        if x.shape[0] % y.shape[0] or x.shape[1] % y.shape[1]:
            facs = None; break
        facs.add((x.shape[0] // y.shape[0], x.shape[1] // y.shape[1]))
    variants = []
    if facs and len(facs) == 1 and facs != {(1, 1)}:
        fa = facs.pop()
        variants.append(('fac', lambda x, fa=fa: blocks_of(x, fac=fa)))
    variants.append(('part', lambda x: (lambda P: P[0][1] if P else None)(partitions(x))))
    for vname, fn in variants:
        B = [fn(x) for x in xs]
        if any(b is None for b in B):
            continue
        if any((len(b), len(b[0])) != y.shape for b, y in zip(B, ys)):
            continue
        Bt = [fn(t) for t in ts]
        if any(b is None for b in Bt):
            continue
        for mode in ('any', 'maj', 'all', 'min'):
            red = lambda b: np.array([[reduce_blk(blk, mode) for blk in row] for row in b], dtype=np.int16)
            if all(same(red(b), y) for b, y in zip(B, ys)):
                sols.append(Sol('down_%s_%s' % (vname, mode), 2.5, [red(b) for b in Bt], 1.0, 'scale'))
                break
    return sols


# ------------------------- symmetry completion + crop of the hidden region ---------------
def learn_symcrop(ctx, xs, ys, ts, budget):
    sols = []
    for n in sorted(ctx.in_colors):
        ok = True
        for x, y in zip(xs, ys):
            m = x == n
            if not m.any():
                ok = False; break
            ys_, xs_ = np.nonzero(m)
            if (ys_.max() - ys_.min() + 1, xs_.max() - xs_.min() + 1) != y.shape:
                ok = False; break
        if not ok:
            continue
        budget.check()
        for fname, f in (('sym', sym_fill), ('per', period_fill)):
            def run(x):
                fx = f(x, n)
                if fx is None:
                    return None
                ys_, xs_ = np.nonzero(x == n)
                return fx[ys_.min():ys_.max() + 1, xs_.min():xs_.max() + 1]
            if all(same(run(x), y) for x, y in zip(xs, ys)):
                preds = [run(t) for t in ts]
                if all(p is not None for p in preds):
                    sols.append(Sol('symcrop_%s%d' % (fname, n), 2.0, preds, 1.0, 'pattern'))
                    break
    return sols


# --------------------------- upscale to output size by an inferred factor ---------------
def learn_upscale(ctx, xs, ys, ts, budget):
    sols = []
    # factor = number of colours (non-bg) or number of non-bg cells  (counting POV)
    for fname, fn in (('ncol', lambda x: len(set(x.ravel().tolist()) - {ctx.bg})),
                      ('nnz', lambda x: int((x != ctx.bg).sum()))):
        def run(x):
            k = fn(x)
            if k < 1 or k * max(x.shape) > 30:
                return None
            return np.kron(x, np.ones((k, k), dtype=np.int16))
        if all(same(run(x), y) for x, y in zip(xs, ys)):
            preds = [run(t) for t in ts]
            if all(p is not None for p in preds):
                sols.append(Sol('upscale_by_' + fname, 2.5, preds, 0.9, 'scale'))
    return sols


# ---------------------------- counting POV: output built from a count ---------------------
def count_features(x, bg):
    f = {}
    for pov in ('c4', 'c8', 'm4', 'm8'):
        obs = objects(x, bg, pov)
        f['n_' + pov] = len(obs)
        if obs:
            mx = max(o.size for o in obs)
            f['nmax_' + pov] = sum(1 for o in obs if o.size == mx)
            f['big_' + pov] = mx
    cnt = Counter(x.ravel().tolist())
    f['ncol'] = len([c for c in cnt if c != bg])
    for c in range(10):
        if c != bg:
            f['cells_%d' % c] = cnt.get(c, 0)
            f['obj_%d' % c] = sum(1 for o in objects(x, bg, 'c4') if o.color == c)
    nz = [c for c, _ in cnt.most_common() if c != bg]
    f['_maj'] = nz[0] if nz else bg
    f['_min'] = nz[-1] if nz else bg
    return f


def learn_count(ctx, xs, ys, ts, budget):
    bg = ctx.bg
    Fx = [count_features(x, bg) for x in xs]; Ft = [count_features(t, bg) for t in ts]
    sols = []
    # output colour candidates
    def colsrc(y):
        v = set(y.ravel().tolist()) - {bg}
        return v
    keys = [k for k in Fx[0] if not k.startswith('_') and all(k in f for f in Fx + Ft)]
    # A: shape from count, uniform colour
    for shp in ('1n', 'n1', 'nn'):
        for k in keys:
            def mk(n, col):
                if n < 1 or n > 30:
                    return None
                s = {'1n': (1, n), 'n1': (n, 1), 'nn': (n, n)}[shp]
                return np.full(s, col, np.int16)
            for csrc in ('const', '_maj', '_min'):
                ok = True; cconst = None
                for f, y in zip(Fx, ys):
                    if csrc == 'const':
                        if cconst is None:
                            cconst = int(y.ravel()[0])
                        col = cconst
                    else:
                        col = f[csrc]
                    if not same(mk(f[k], col), y):
                        ok = False; break
                if ok:
                    preds = [mk(f[k], cconst if csrc == 'const' else f[csrc]) for f in Ft]
                    if all(p is not None for p in preds):
                        sols.append(Sol('count[%s,%s,%s]' % (k, shp, csrc), 3.0, preds, 0.9, 'count'))
                        break
            if len(sols) > 6:
                break
    # B: fixed output shape, first n cells (reading order) coloured
    if ctx.const_out_shape is not None:
        S = ctx.const_out_shape
        for k in keys:
            for csrc in ('const', '_maj'):
                def mk(n, col):
                    o = np.full(S[0] * S[1], bg, np.int16)
                    if n > o.size:
                        return None
                    o[:n] = col
                    return o.reshape(S)
                ok = True; cconst = None
                for f, y in zip(Fx, ys):
                    nzv = y[y != bg]
                    if csrc == 'const':
                        if cconst is None and len(nzv):
                            cconst = int(nzv[0])
                        col = cconst if cconst is not None else bg
                    else:
                        col = f[csrc]
                    if not same(mk(f[k], col), y):
                        ok = False; break
                if ok and cconst is not None or (ok and csrc != 'const'):
                    preds = [mk(f[k], cconst if csrc == 'const' else f[csrc]) for f in Ft]
                    if all(p is not None for p in preds):
                        sols.append(Sol('countfill[%s,%s]' % (k, csrc), 3.2, preds, 0.85, 'count'))
    return sols[:6]


# ------------------------------- layout summary (object grid) -----------------------------
def _cluster(intervals):
    """group intervals (a,b) that overlap -> index per interval"""
    order = sorted(range(len(intervals)), key=lambda i: intervals[i])
    gid = [0] * len(intervals); g = -1; end = -1
    for i in order:
        a, b = intervals[i]
        if a > end:
            g += 1
        end = max(end, b); gid[i] = g
    return gid, g + 1


def layout(x, bg, sel):
    pov, minsize = sel
    obs = [o for o in objects(x, bg, pov) if o.size >= minsize]
    if not obs or len(obs) > 40:
        return None
    rg, nr = _cluster([(o.r0, o.r1) for o in obs]); cg, nc = _cluster([(o.c0, o.c1) for o in obs])
    out = np.full((nr, nc), bg, np.int16)
    for o, r, c in zip(obs, rg, cg):
        if out[r, c] != bg:
            return None
        out[r, c] = o.color
    return out


def learn_layout(ctx, xs, ys, ts, budget):
    sols = []
    for pov in ('c4', 'm8', 'c8'):
        for ms in (1, 2, 4, 6, 9):
            sel = (pov, ms)
            if all(same(layout(x, ctx.bg, sel), y) for x, y in zip(xs, ys)):
                preds = [layout(t, ctx.bg, sel) for t in ts]
                if all(p is not None for p in preds):
                    sols.append(Sol('layout[%s,%d]' % sel, 3.0, preds, 0.9, 'object'))
                    return sols
    return sols# =====================================================================================
# HOW (part 3)  --  SSOR program search
#   conventional core  = learners on the raw input + depth-1 primitives with high prior
#   sparse outliers    = 1-3 extra (rarer) primitives composed in front of a learner
#   value gate         Cr = B(N,K) * U * Q * F
#        U = exact fit on all train pairs (hard gate); partial cell accuracy guides the frontier
#        Q = coherence (leave-one-out, plausible palette/shape on test)
#        F = within budget (Budget object)
#        B = exp(-lambda * description length) * (1 + small novelty bonus)
# =====================================================================================
import heapq

PRIORS = {}          # memory of successful primitives (filled from training runs; see build/priors.json)


def prim_cost(name, fam, base):
    if not PRIORS:
        return base
    tot = PRIORS.get('_total', 1)
    n = PRIORS.get(name, 0) + 0.5 * PRIORS.get('fam:' + fam, 0)
    return max(0.6, base - 0.35 * math.log1p(n) + 0.15 * math.log1p(tot) / 3)


def _acc(xs, ys):
    if any(x.shape != y.shape for x, y in zip(xs, ys)):
        return -1.0
    return float(np.mean([(x == y).mean() for x, y in zip(xs, ys)]))


LEARNERS_ROOT = [('crop', learn_crop), ('partition', learn_partition), ('tile', learn_tile),
                 ('downscale', learn_downscale), ('symcrop', learn_symcrop), ('upscale', learn_upscale),
                 ('count', learn_count), ('layout', learn_layout)]

# v2: search configuration (overridable through env ARC_CFG='{"k1":..}' for sweeps)
CFG = dict(margin=1.5,       # stop once frontier priority > best solution cost + margin
           k1=40,            # max depth-1 nodes that get expanded (best-first order)
           k2=6,             # max depth-2 nodes that get expanded
           max_depth=3,
           d2cost=1.7,       # SSOR sparsity: only cheap primitives below depth 1
           min_sols=2,       # need this many distinct test predictions before margin-stop applies
           tta='fail')       # 'fail' = TTA views only when the original view found nothing; 'always'; 'never'
try:
    CFG.update(json.loads(os.environ.get('ARC_CFG', '{}')))
except Exception:
    pass


def _shape_gate(xs, ys):
    """WHY-level cheap invariants: which shape-changing learners can possibly fit (prunes learner calls)."""
    le = all(y.shape[0] <= x.shape[0] and y.shape[1] <= x.shape[1] for x, y in zip(xs, ys))
    le_t = le or all(y.shape[0] <= x.shape[1] and y.shape[1] <= x.shape[0] for x, y in zip(xs, ys))
    ge = all(y.shape[0] >= x.shape[0] and y.shape[1] >= x.shape[1] for x, y in zip(xs, ys))
    if le_t:   # crop output can only contain colours of the input (+ background for masked crops)
        le_t = all(set(np.unique(y).tolist()) <= set(np.unique(x).tolist()) | {ctx_bg_of(x)} for x, y in zip(xs, ys))
    return {'crop': le_t, 'partition': le, 'downscale': le, 'symcrop': le, 'tile': ge, 'upscale': ge,
            'count': True, 'layout': True}


def ctx_bg_of(x):
    return _GATE_BG[0] if _GATE_BG else bg_of(x)


_GATE_BG = []


def leaf_solutions(ctx, chain, cost, xs, ts, budget, depth, allow_cell):
    ys = ctx.ys
    out = []
    pre = '+'.join(chain)
    def wrap(s):
        s.name = (pre + '>' if pre else '') + s.name
        s.cost += cost
        return s
    if all(same(x, y) for x, y in zip(xs, ys)):
        if depth > 0:
            out.append(Sol(pre, cost, [t.copy() for t in ts], 1.0, 'chain'))
        return out, True
    cm = learn_colormap(xs, ys, ts)
    if cm:
        out += [wrap(s) for s in cm]
    shapes_match = all(x.shape == y.shape for x, y in zip(xs, ys))
    if not shapes_match:
        _GATE_BG[:] = [ctx.bg]
        gate = _shape_gate(xs, ys)
        Ls = LEARNERS_ROOT if depth <= 1 else (('tile', learn_tile), ('downscale', learn_downscale),
                                               ('partition', learn_partition))
        for nm, L in Ls:
            if budget.over():
                break
            if not gate.get(nm, True):
                continue
            try:
                out += [wrap(s) for s in L(ctx, xs, ys, ts, budget)]
            except Timeout:
                raise
            except Exception:
                pass
    if shapes_match and allow_cell:
        try:
            sub = Budget(min(budget.left() * 0.4, 6.0 if depth == 0 else 2.0))
            out += [wrap(s) for s in learn_cellrule(ctx, xs, ys, ts, sub, triples=(depth == 0))]
        except Timeout:
            pass
    return out, False


_MISS = object()


def _ok_grid(r):
    return r is not None and isinstance(r, np.ndarray) and r.ndim == 2 and 1 <= r.shape[0] <= 30 \
        and 1 <= r.shape[1] <= 30


def ssor_search(ctx, budget, max_depth=None):
    max_depth = CFG['max_depth'] if max_depth is None else max_depth
    caps = {0: 1, 1: CFG['k1'], 2: CFG['k2']}
    prims = [(n, f, prim_cost(n, fam, c), fam) for n, f, c, fam in build_primitives(ctx)]
    prims.sort(key=lambda p: p[2])
    sols = []
    seen = set()
    cnt = itertools.count()
    root = ((), 0.0, ctx.xs, ctx.tests)
    heap = [(0.0, next(cnt), root)]
    seen.add(tuple(gkey(g) for g in ctx.xs + ctx.tests))
    cell_calls = 0
    expanded = Counter()
    memo = {}                       # v2: (primitive, grid) -> result, shared by all states of this task
    pred_keys = set()
    root_acc = 0.0
    while heap and not budget.over():
        pri, _, (chain, cost, xs, ts) = heapq.heappop(heap)
        depth = len(chain)
        acc = _acc(xs, ctx.ys)
        if depth == 0:
            root_acc = acc
        allow_cell = depth == 0 or (acc >= 0.6 and acc > root_acc + 0.01 and cell_calls < 4)
        if depth > 0 and allow_cell:
            cell_calls += 1
        try:
            new, exact = leaf_solutions(ctx, chain, cost, xs, ts, budget, depth, allow_cell)
        except Timeout:
            break
        sols += new
        for s_ in new:
            pred_keys.add(tuple(gkey(p) if p is not None else None for p in s_.preds))
        if exact:
            continue
        if sols and len(pred_keys) >= CFG['min_sols']:
            best = min(s.cost for s in sols)
            if pri > best + CFG['margin']:
                break
        if depth >= max_depth or expanded[depth] >= caps.get(depth, 10 ** 9):
            continue
        expanded[depth] += 1
        # SSOR sparsity: at most one 'rare' (high-cost) primitive per chain beyond depth 1
        for name, f, c, fam in prims:
            if budget.over():
                break
            if depth >= 1 and chain[-1] == name and fam in ('geom', 'crop'):
                continue
            if depth >= 2 and c > CFG['d2cost']:
                continue
            nx = []
            for g in xs + ts:
                mk = (name, g.shape, g.tobytes())
                r = memo.get(mk, _MISS)
                if r is _MISS:
                    try:
                        r = f(g)
                    except Exception:
                        r = None
                    if not _ok_grid(r) or r.size == 0:
                        r = None
                    else:
                        r = np.ascontiguousarray(r, dtype=np.int16)
                    memo[mk] = r
                if r is None:
                    nx = None; break
                nx.append(r)
            if nx is None:
                continue
            k = tuple(gkey(g) for g in nx)
            if k in seen:
                continue
            seen.add(k)
            nxs, nts = nx[:len(xs)], nx[len(xs):]
            a2 = _acc(nxs, ctx.ys)
            guide = 1.5 * (1 - a2) if a2 >= 0 else (0.8 if all(
                x.shape == y.shape for x, y in zip(nxs, ctx.ys)) else 1.6)
            heapq.heappush(heap, (cost + c + guide, next(cnt), (chain + (name,), cost + c, nxs, nts)))
    return sols# =====================================================================================
# DO + REVIEW  --  test-time augmentation, value-gated ranking, top-2 selection
# =====================================================================================
def value_gate(s, ctx, ti, pred):
    """Cr = B * U * Q * F  (U and F are hard gates already passed)."""
    B = math.exp(-0.45 * s.cost)
    Q = s.q
    x = ctx.tests[ti]
    pal = set(np.unique(pred).tolist())
    if not pal <= (ctx.out_colors | set(np.unique(x).tolist())):
        Q *= 0.5
    if ctx.const_out_shape is not None and pred.shape != ctx.const_out_shape:
        Q *= 0.4
    if same(pred, x) and not any(same(a, b) for a, b in ctx.train):
        Q *= 0.2
    return B * Q


def color_normaliser(train, tests):
    cnt = Counter()
    for x, y in train:
        cnt.update(x.ravel().tolist())
    for t in tests:
        cnt.update(t.ravel().tolist())
    order = [c for c, _ in cnt.most_common()]
    fwd = {c: i for i, c in enumerate(order)}
    rest = [c for c in range(10) if c not in fwd]
    for c in rest:
        fwd[c] = len(fwd)
    lut = np.array([fwd[c] for c in range(10)], dtype=np.int16)
    inv = np.zeros(10, dtype=np.int16)
    for c, i in fwd.items():
        inv[i] = c
    return lut, inv


def solve_task_np(train, tests, budget_s=20.0, tta=True):
    """Returns per test input a list of (grid, score, name) sorted by score."""
    budget = Budget(budget_s)
    votes = [defaultdict(lambda: [0.0, None, '']) for _ in tests]
    views = [('id', None)]
    if tta and CFG['tta'] != 'never':
        views += [('transpose', None), ('flipud', None), ('fliplr', None), ('cnorm', None), ('rot90', None)]
    for vi, (vname, _) in enumerate(views):
        if budget.over():
            break
        if vi == 0:
            sub = Budget(budget.left() * (0.55 if tta and CFG['tta'] == 'always' else 1.0))
        else:
            sub = Budget(budget.left() / max(1, (len(views) - vi)))
        if vname == 'cnorm':
            lut, inv = color_normaliser(train, tests)
            tr = [(lut[x], lut[y]) for x, y in train]; te = [lut[t] for t in tests]
            unv = lambda g: inv[g]
        elif vname == 'id':
            tr, te, unv = train, tests, (lambda g: g)
        else:
            tr = [(d4(x, vname), d4(y, vname)) for x, y in train]; te = [d4(t, vname) for t in tests]
            unv = (lambda g, vname=vname: d4_inv(g, vname))
        ctx = TaskCtx(tr, te)
        try:
            sols = ssor_search(ctx, sub)
        except Timeout:
            sols = []
        w = 1.0 if vi == 0 else 0.8
        for s in sols:
            for ti, p in enumerate(s.preds):
                if p is None or not valid(p):
                    continue
                g = np.ascontiguousarray(unv(p)).astype(np.int16)
                if not valid(g):
                    continue
                sc = value_gate(s, ctx, ti, p) * w
                v = votes[ti][gkey(g)]
                v[0] += sc
                if v[1] is None:
                    v[1] = g; v[2] = ('%s:' % vname if vname != 'id' else '') + s.name
        if vi == 0 and tta:
            # v2: TTA only when the original view is empty/ambiguous ('fail' policy); v1 rule kept for 'always'
            if CFG['tta'] == 'fail':
                if all(len(v) > 0 for v in votes) and min(max(x[0] for x in v.values()) for v in votes) > 0.05:
                    break
            elif all(len(v) > 0 for v in votes) and min(max(x[0] for x in v.values()) for v in votes) > 0.25 \
                    and budget.left() < budget_s * 0.4:
                break
    res = []
    for v in votes:
        items = sorted(v.values(), key=lambda t: -t[0])
        res.append([(g, sc, nm) for sc, g, nm in items])
    return res# =====================================================================================
# DO  --  driver: file discovery, parallel per-task execution with hard time limits,
#          identity fallback, submission validation
# =====================================================================================
COMP_SLUG = 'arc-prize-2026-arc-agi-2'


def find_test_file(root):
    for p in (os.path.join(root, 'competitions', COMP_SLUG, 'arc-agi_test_challenges.json'),
              os.path.join(root, COMP_SLUG, 'arc-agi_test_challenges.json')):
        if os.path.isfile(p):
            return p
    cands = sorted(glob.glob(os.path.join(root, '**', '*test_challenges*.json'), recursive=True))
    if not cands:
        raise FileNotFoundError('No *test_challenges*.json under %s' % root)
    cands.sort(key=lambda p: (COMP_SLUG not in p, len(p)))
    return cands[0]


def to_list(g):
    return [[int(v) for v in row] for row in np.asarray(g)]


def _alarm(signum, frame):
    raise Timeout()


def solve_one(args):
    task_id, task, budget_s, tta = args
    t0 = time.time()
    tests = [t['input'] for t in task['test']]
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(int(budget_s * 2 + 15))
        train = [(A(p['input']), A(p['output'])) for p in task['train']]
        tn = [A(t) for t in tests]
        res = solve_task_np(train, tn, budget_s, tta)
        signal.alarm(0)
        out, names = [], []
        export = [[{'grid': to_list(c[0]), 'score': float(c[1]), 'name': c[2]} for c in cands[:4]] for cands in res]
        for ti, cands in enumerate(res):
            preds = [c[0] for c in cands[:2]]
            nm = [c[2] for c in cands[:2]]
            while len(preds) < 2:
                fb = tn[ti]
                if any(same(fb, p) for p in preds):
                    fb = preds[0]
                preds.append(fb); nm.append('identity')
            out.append({'attempt_1': to_list(preds[0]), 'attempt_2': to_list(preds[1])})
            names.append(nm)
        return task_id, out, names, time.time() - t0, None, export
    except BaseException as e:
        signal.alarm(0)
        out = []
        for t in tests:
            try:
                g = to_list(A(t))
            except Exception:
                g = [[0]]
            out.append({'attempt_1': g, 'attempt_2': g})
        if not out:
            out = [{'attempt_1': [[0]], 'attempt_2': [[0]]}]
        return (task_id, out, [['fallback', 'fallback']] * len(out), time.time() - t0, '%s: %s' % (type(e).__name__, e),
                [[] for _ in out])


def run_all(tasks, budget_s=20.0, workers=None, tta=True, verbose=True, total_budget_s=None, on_result=None):
    """info[task] = (names, seconds, error, candidates);  candidates[test] = [{'grid','score','name'}, ...] (top-4,
    every one verified on all train pairs).  on_result(task_id, out, info_tuple) is called as tasks finish."""
    import multiprocessing as mp
    workers = workers or max(1, os.cpu_count() or 1)
    t0 = time.time()
    sub, info = {}, {}
    items = [(k, v, budget_s, tta) for k, v in tasks.items()]
    if total_budget_s:
        per = total_budget_s * workers / max(1, len(items)) * 0.8
        budget_s = min(budget_s, per)
        items = [(k, v, budget_s, tta) for k, v in tasks.items()]
    if workers == 1:
        it = map(solve_one, items)
        pool = None
    else:
        pool = mp.get_context('fork').Pool(workers, maxtasksperchild=20)
        it = pool.imap_unordered(solve_one, items)
    for i, (k, out, names, dt, err, cands) in enumerate(it):
        sub[k] = out; info[k] = (names, dt, err, cands)
        if on_result is not None:
            try:
                on_result(k, out, info[k])
            except Exception:
                pass
        if verbose and (i + 1) % 20 == 0:
            print('  %d/%d tasks done, %.0fs elapsed' % (i + 1, len(items), time.time() - t0), flush=True)
    if pool is not None:
        pool.close(); pool.join()
    return sub, info


def validate_submission(sub, tasks):
    assert set(sub) == set(tasks), 'task_id mismatch'
    for k, task in tasks.items():
        assert isinstance(sub[k], list) and len(sub[k]) == len(task['test']), '%s: wrong number of outputs' % k
        for e in sub[k]:
            assert set(e) == {'attempt_1', 'attempt_2'}, '%s: bad keys' % k
            for g in e.values():
                assert isinstance(g, list) and g and all(isinstance(r, list) and len(r) == len(g[0]) for r in g)
                assert all(isinstance(v, int) and 0 <= v <= 9 for r in g for v in r)
    return True


def score_submission(sub, sols):
    hit = tot = 0
    for k, ss in sols.items():
        for i, s in enumerate(ss):
            tot += 1
            e = sub.get(k, [])
            e = e[i] if i < len(e) else {}
            hit += int(e.get('attempt_1') == s or e.get('attempt_2') == s)
    return hit, tot


In [ ]:
%%writefile /kaggle/working/cpu_solver/arcsym.py
"""Our CPU solver (2026-10-18): runs every engine on a task, keeps the programs that reproduce all train pairs, and
returns per test input up to 2 grids, best first.

    solve(task, budget_s, task_id=None) -> [[(grid, name), ...] per test input]

Engines (in order; each gets the time the earlier ones left, with a floor):
  hodel   Hodel's 400 ARC-1 solvers as a library, with color / orientation transfer (hodel_engine.py)
  wwhdr   nicholashuihk's rule learners + SSOR program search (wwhdr_solver.py, MIT-0)
Ranking: candidates whose engine prior is lower first; then the grid more programs agree on; then the engine order.
task_id is used only to keep a library program off its own task in benchmarks."""
import os
import sys
from collections import defaultdict

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from common import Clock, Cand  # noqa: E402

ENGINES = ["learn", "hodel", "wwhdr"]
SHARE = {"learn": 0.4, "hodel": 0.35, "wwhdr": 1.0}             # share of the remaining budget each engine may use
CAP_S = {"wwhdr": 90.0}          # 2026-10-18: absolute cap -- wwhdr's eval answers were 0 / 4 right at 120 s (X33)


def _wwhdr(task, clock):
    import numpy as np                          # (wwhdr touches signal.SIGALRM only in functions we never call)
    import wwhdr_solver as S
    if not getattr(S, "_pri_loaded", False):
        pri = os.path.join(HERE, "wwhdr_priors.json")
        if os.path.isfile(pri):
            import json
            S.PRIORS.update(json.load(open(pri)))
        S._pri_loaded = True
    train = [(S.A(p["input"]), S.A(p["output"])) for p in task["train"]]
    res = S.solve_task_np(train, [S.A(t["input"]) for t in task["test"]], max(1.0, clock.left()), True)
    found = []
    for rank, _ in enumerate(range(max((len(r) for r in res), default=0))):
        outs = [S.to_list(r[rank][0]) if rank < len(r) else None for r in res]
        names = [r[rank][2] for r in res if rank < len(r)]
        found.append(Cand("wwhdr", names[0] if names else "?", outs, prior=2.5 + 0.01 * rank))
    return found


def solve(task, budget_s=10.0, task_id=None, engines=None, on_partial=None, loo_s=0.0):
    """on_partial(ranked) (2026-10-18): called with the ranking so far after every engine, so a caller that may have
    to kill this process (cpu_runner's hard limit) keeps the candidates already found."""
    total = Clock(budget_s)
    cands = []
    for e in engines or ENGINES:
        if on_partial is not None and cands:
            try:
                on_partial(rank(task, cands))
            except Exception:                                          # noqa: BLE001
                pass
        left = total.left()
        if left <= 0.2:
            break
        sub = Clock(min(left * SHARE.get(e, 1.0), CAP_S.get(e, 1e9)))
        try:
            if e == "learn":
                import learners
                cands += learners.engine(task, sub)
            elif e == "hodel":
                import hodel_engine
                cands += hodel_engine.engine(task, sub, exclude_id=task_id)
            elif e == "wwhdr":
                cands += _wwhdr(task, sub)
        except Exception:                                              # noqa: BLE001 - an engine crash costs only itself
            continue
    ranked = rank(task, cands)
    if loo_s > 0:
        try:                                           # never lose the answers to a failure in the LOO step
            ranked = loo_promote(task, ranked, loo_s, task_id)
        except Exception:                                              # noqa: BLE001
            pass
    return ranked


def loo_promote(task, ranked, fold_s, task_id=None):
    """2026-10-18 (LOG row 43): a test's first answer from wwhdr, or tagged ~weak, is re-derived leaving each train
    pair out (cpu_solver/loo.py). Passing every fold with the same test answers -> tagged ~loo and the ~weak tag
    dropped. MEASURED on training: wwhdr 30/41 right vs 3/39 wrong answers pass; learn 38/60 vs 3/21."""
    import loo
    if len(task["train"]) < 3 or not ranked or any(not per for per in ranked):
        return ranked
    names = [per[0][1] for per in ranked]
    for eng in ("wwhdr", "learn"):
        idx = [j for j, n in enumerate(names) if (n.startswith("wwhdr") if eng == "wwhdr" else
                                                  (n.startswith("learn") and "~weak" in n))]
        if not idx:
            continue
        top1 = [per[0][0] for per in ranked]
        res = loo.loo_check(task, lambda t, b, eng=eng: solve(t, b, task_id=task_id, engines=[eng]), top1, fold_s)
        if res and res[0] == 1.0 and res[1] == 1.0:
            for j in idx:
                g, n = ranked[j][0]
                ranked[j][0] = (g, n.replace("~weak", "") + "~loo")
    return ranked


def rank(task, cands):
    out = []
    # 2026-10-18 plausibility gate (generic): a test output that repeats the test input when every train pair changes,
    # uses a color no train output and not the test input has, or breaks a fixed output shape is ranked last
    tr_changes = all(p["input"] != p["output"] for p in task["train"])
    out_cols = {v for p in task["train"] for r in p["output"] for v in r}
    shapes = {(len(p["output"]), len(p["output"][0])) for p in task["train"]}

    def penalty(g, j):
        x = task["test"][j]["input"]
        pen = 0.0
        if tr_changes and g == x:
            pen += 2.0
        if {v for r in g for v in r} - out_cols - {v for r in x for v in r}:
            pen += 1.0
        if len(shapes) == 1 and (len(g), len(g[0])) not in shapes:
            pen += 1.0
        return pen
    for j in range(len(task["test"])):
        score = defaultdict(lambda: [9.0, 0, None, ""])
        for c in cands:
            g = c.outs[j] if j < len(c.outs) else None
            if g is None:
                continue
            s = score[str(g)]
            pen = penalty(g, j)
            s[0] = min(s[0], c.prior + pen)
            s[1] += 1
            if s[2] is None:
                s[2], s[3] = g, f"{c.engine}:{c.name}" + ("~weak~implausible" if pen else "")
        ranked = sorted(score.values(), key=lambda s: (s[0], -s[1]))
        out.append([(s[2], s[3]) for s in ranked[:2]])
    return out


In [ ]:
%%writefile /kaggle/working/cpu_solver/cpu_runner.py
"""Background CPU solver for the submission notebook (2026-10-18).

    nice -n 15 python cpu_runner.py '<json config>'

Config: challenges, out_path, workers, budget_s (per task, soft: the engines check their clocks), hard_s (per task, the
child process is killed after it), stop_epoch (no task starts after it), log_path.
One forked child per task (Linux; spawn elsewhere), at most `workers` at a time, smallest tasks first. A child that
overruns hard_s is killed, so no learner can stall the queue. out_path is rewritten atomically after every task:
{task_id: [[[grid, program_name], ...] per test input]} -- only tasks with at least one candidate."""
import json
import multiprocessing as mp
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)


def _child(tid, task, budget, q, loo_s=0.0, hang=False):
    try:
        import arcsym
        fmt = lambda res: [[[g, n] for g, n in per] for per in res]

        def partial(r):
            q.put((tid, fmt(r), "partial"))
            if hang:                                   # test hook only: hang after the first partial result
                time.sleep(1e6)
        res = arcsym.solve(task, budget, on_partial=partial, loo_s=loo_s)
        q.put((tid, fmt(res), None))
    except BaseException as e:                                     # noqa: BLE001
        q.put((tid, None, f"{type(e).__name__}: {e}"[:200]))


def write_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


def main(cfg):
    t0 = time.time()
    for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ.setdefault(v, "1")                  # numpy single-threaded per child (the GPU phases need the rest)
    if hasattr(os, "nice"):
        try:
            os.nice(int(cfg.get("nice", 0)))
        except OSError:
            pass
    import arcsym  # noqa: F401  -- load every engine once in the parent, so forked children start warm
    import hodel_engine  # noqa: F401
    import learners  # noqa: F401
    import learners_obj  # noqa: F401
    import wwhdr_solver  # noqa: F401
    ch = json.load(open(cfg["challenges"]))
    order = sorted(ch, key=lambda t: len(json.dumps(ch[t])))
    ctx = mp.get_context("fork" if hasattr(os, "fork") else "spawn")
    q = ctx.Queue()
    out, log = {}, open(cfg.get("log_path", os.devnull), "a")
    running = {}                                       # tid -> (process, start time)
    partial = {}                                       # tid -> the latest partial result
    pending = list(order)
    done = 0
    workers, hard = int(cfg["workers"]), float(cfg["hard_s"])
    while pending or running:
        while pending and len(running) < workers and time.time() < cfg.get("stop_epoch", 1e18):
            tid = pending.pop(0)
            p = ctx.Process(target=_child, args=(tid, ch[tid], cfg["budget_s"], q, float(cfg.get("loo_s", 0.0)),
                                                 bool(cfg.get("_test_hang_after_partial"))), daemon=True)
            p.start()
            running[tid] = (p, time.time())
        if not pending and not running:
            break
        if time.time() >= cfg.get("stop_epoch", 1e18):
            pending = []
        try:
            tid, res, err = q.get(timeout=1.0)
        except Exception:                                          # noqa: BLE001 - queue.Empty
            tid = None
        if tid is not None and err == "partial":       # 2026-10-18: candidates so far, kept in case of a kill
            if tid in running and res and any(per for per in res):
                partial[tid] = res
            continue
        if tid is not None and tid in running:
            p, ts = running.pop(tid)
            p.join(timeout=5)
            done += 1
            if (not res or not any(per for per in res)) and partial.get(tid):
                res = partial.pop(tid)                 # the child failed after reporting candidates: keep them
            n = sum(1 for per in res for _ in per) if res else 0
            if res and n:
                out[tid] = res
                write_atomic(out, cfg["out_path"])
            log.write(json.dumps({"task": tid, "s": round(time.time() - ts, 1), "n_cand": n, "err": err,
                                  "names": [per[0][1] for per in res if per] if res else []}) + "\n")
            log.flush()
        now = time.time()
        for tid, (p, ts) in list(running.items()):
            if now - ts > hard or not p.is_alive() and now - ts > 5:
                if p.is_alive():
                    p.kill()
                p.join(timeout=5)
                running.pop(tid)
                if partial.get(tid):                     # the engines that finished before the kill still count
                    out[tid] = partial.pop(tid)
                    write_atomic(out, cfg["out_path"])
                done += 1
                log.write(json.dumps({"task": tid, "s": round(now - ts, 1), "n_cand": 0,
                                      "err": "killed: hard limit" if now - ts > hard else "died"}) + "\n")
                log.flush()
    write_atomic(out, cfg["out_path"])
    log.write(json.dumps({"finished": True, "tasks_done": done, "with_answers": len(out),
                          "wall_s": round(time.time() - t0)}) + "\n")
    log.close()
    return out


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


In [ ]:
%%writefile /kaggle/working/cpu_solver/cpu_merge.py
"""Merge the CPU solver's verified answers into the submission (2026-10-18).

    merge(sub, cpu, ch, nv_sub=None) -> (sub, stats)

Policy, per test output, with c1 / c2 = the CPU's first / second grid (programs that reproduce every train pair):
  * a slot still holding the fallback (the test input itself, i.e. NVARC never answered) takes c1, then c2;
  * otherwise attempt_1 (NVARC's top grid) is never touched; c1 goes to attempt_2 when it differs from both slots --
    except that an attempt_2 Qwen wrote (it differs from NVARC's own attempt_2 in nv_sub) is kept when c1 is tagged
    "~weak" (a feature table only one feature supports: 4 / 6 right on training); learn / hodel otherwise 0.88-1.0.
    Grids from the wwhdr engine (unless leave-one-out verified, tag ~loo) only ever fill fallback slots (eval: 0 unique right / 4 wrong; NVARC's own slot 2 is
    right on 7.4% of the outputs its slot 1 misses). A grid arcsym flags "~implausible"
    (repeats the input when every train pair changes, unseen color, wrong fixed shape) only ever fills a fallback slot.
"""
import json


def merge(sub, cpu, ch, nv_sub=None):
    stats = {"outputs_with_cpu": 0, "already_there": 0, "fallback_filled": 0, "slot2_replaced": 0, "kept_qwen": 0}
    for tid, per in cpu.items():
        if tid not in sub or tid not in ch:
            continue
        for j, cands in enumerate(per):
            if j >= len(sub[tid]) or not cands:
                continue
            stats["outputs_with_cpu"] += 1
            slot = sub[tid][j]
            x = ch[tid]["test"][j]["input"]
            grids = [c[0] for c in cands if c and c[0]]
            if not grids:
                continue
            if any(g in (slot["attempt_1"], slot["attempt_2"]) for g in grids[:1]):
                stats["already_there"] += 1
                continue
            if slot["attempt_1"] == x:                     # NVARC never answered this output
                rest = [g for g in grids if g != x]
                slot["attempt_1"] = rest[0]
                if len(rest) > 1 and slot["attempt_2"] == x:
                    slot["attempt_2"] = rest[1]
                stats["fallback_filled"] += 1
                continue
            c1, name1 = grids[0], cands[0][1]
            if "~implausible" in str(name1):              # 2026-10-18: flagged by arcsym's plausibility gate
                stats["skipped_implausible"] = stats.get("skipped_implausible", 0) + 1
                continue
            if str(name1).startswith("wwhdr") and "~loo" not in str(name1):             # 2026-10-18: wwhdr on eval: 0 unique right / 4 wrong (X33,
                stats["skipped_wwhdr"] = stats.get("skipped_wwhdr", 0) + 1   # all_eval runs) -> fallback slots only
                continue
            qwen_slot = nv_sub is not None and tid in nv_sub and j < len(nv_sub[tid]) and \
                slot["attempt_2"] != nv_sub[tid][j]["attempt_2"]
            if qwen_slot and (str(name1).startswith("wwhdr") or "~weak" in str(name1)):
                stats["kept_qwen"] += 1
                continue
            slot["attempt_2"] = c1
            stats["slot2_replaced"] += 1
    return sub, stats


def merge_files(sub_path, cpu_path, ch_path, nv_path=None):
    sub = json.load(open(sub_path))
    cpu = json.load(open(cpu_path))
    ch = json.load(open(ch_path))
    nv = json.load(open(nv_path)) if nv_path else None
    sub, stats = merge(sub, cpu, ch, nv)
    tmp = sub_path + ".tmp"
    json.dump(sub, open(tmp, "w"))
    import os
    os.replace(tmp, sub_path)
    return stats


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel_profiles.json
{"007bbfb7": [[0, 2, 7, 6, 4], []], "00d62c1b": [[0, 3], [4]], "017c7c7b": [[0, 1], [2]], "025d127b": [[0, 8, 6, 2], []], "045e512c": [[0, 1, 8, 5, 2, 3, 4, 6], []], "0520fde7": [[0, 1, 5], [2]], "05269061": [[0, 3, 4, 8, 1, 2], []], "05f2a901": [[0, 2, 8], []], "06df4c85": [[0, 1, 4, 8, 2, 3, 9], []], "08ed6ac7": [[0, 5], [1, 2, 3, 4]], "09629e4f": [[5, 0, 2, 3, 4, 6, 8], []], "0962bcdd": [[0, 7, 8, 2, 6], []], "0a938d79": [[0, 1, 2, 3, 4, 8], []], "0b148d64": [[0, 2, 8, 1, 3, 4], []], "0ca9ddb6": [[0, 1, 2, 6, 8], [4, 7]], "0d3d703e": [[2, 3, 8, 1, 4, 5, 6, 9], []], "0e206a2e": [[0, 8, 3, 1, 4, 2], []], "10fcaaa3": [[0, 4, 6, 2, 5], [8]], "11852cab": [[0, 8, 3, 4, 2, 1], []], "1190e5a7": [[3, 1, 7, 8], []], "137eaa0f": [[0, 5, 2, 7, 1, 3, 6, 4, 9], []], "150deff5": [[0, 5], [8, 2]], "178fcbfb": [[0, 3, 2, 1], []], "1a07d186": [[0, 4, 3, 8, 1, 2], []], "1b2d62fb": [[9, 0, 1], [8]], "1b60fb0c": [[0, 1], [2]], "1bfc4729": [[0, 1, 4, 6, 7], []], "1c786137": [[0, 3, 8, 2, 5, 1, 9, 6, 4], []], "1caeab9d": [[0, 1, 2, 4], []], "1cf80156": [[0, 2, 1, 8], []], "1e0a9b12": [[0, 4, 1, 3, 6, 7, 8, 9, 2], []], "1e32b0e9": [[0, 1, 2, 8, 3], []], "1f0c79e5": [[0, 2, 3, 4, 6, 7], []], "1f642eb9": [[0, 8, 2, 6, 3, 4, 7, 1, 9], []], "1f85a75f": [[0, 1, 2, 5, 3, 4], []], "1f876c06": [[0, 4, 6, 8, 9, 2, 3, 7], []], "1fad071e": [[0, 1, 2], []], "2013d3e2": [[0, 6, 7, 8, 1, 2, 3, 4, 5], []], "2204b7a8": [[0, 3, 1, 2, 4, 7, 8, 9], []], "22168020": [[0, 3, 8, 1, 4, 6], []], "22233c11": [[0, 3], [8]], "2281f1f4": [[0, 5], [2]], "228f6490": [[0, 5, 6, 8, 9, 4, 2, 7, 3], []], "22eb0ac0": [[0, 8, 4, 2, 3, 1, 6, 9, 5], []], "234bbc79": [[0, 5, 2, 1, 3, 6, 8], []], "23581191": [[0, 7, 8], [2]], "239be575": [[0, 8, 2], []], "23b5c85d": [[0, 2, 3, 1, 8, 6, 4, 7], []], "253bf280": [[0, 8], [3]], "25d487eb": [[0, 8, 2, 3, 1], []], "25d8a9c8": [[2, 4, 3, 9, 1, 6, 7], [0, 5]], "25ff71a9": [[0, 1, 2], []], "264363fd": [[8, 1, 3, 2, 4, 6, 5], []], "272f95fa": [[0, 8], [1, 2, 3, 6, 4]], "27a28665": [[0, 5, 8, 1, 4], [2, 3, 6]], "28bf18c6": [[0, 2, 8, 1], []], "28e73c20": [[0], [3]], "29623171": [[0, 5, 2, 3, 1], []], "29c11459": [[0, 1, 2, 3, 7], [5]], "2bcee788": [[0, 2, 4, 6, 8, 7], [3]], "2bee17df": [[0, 2, 8], [3]], "2c608aff": [[1, 2, 8, 4, 5, 3], []], "2dc579da": [[3, 4, 1, 8, 2], []], "2dd70a9a": [[0, 8, 2, 3], []], "2dee498d": [[1, 2, 4, 5, 0, 3], []], "31aa019c": [[0, 1, 3, 5, 2, 7, 9, 4, 8, 6], []], "321b1fc6": [[0, 8, 6, 7, 4, 9], []], "32597951": [[0, 1, 8], [3]], "3345333e": [[0, 2, 6, 3, 1], []], "3428a4f5": [[0, 2, 4], [3]], "3618c87e": [[0, 5, 1], []], "363442ee": [[0, 5, 1, 2, 3, 9, 4, 6, 7, 8], []], "36d67576": [[0, 4, 2, 3, 1], []], "36fdfd69": [[0, 1, 8, 3, 2], [4]], "3906de3d": [[0, 1, 2], []], "39a8645d": [[0, 8, 1, 4, 2, 6], []], "39e1d7f9": [[0, 8, 3, 4, 6, 2], []], "3aa6fb7a": [[0, 8], [1]], "3ac3eb23": [[0, 2, 4, 8], []], "3af2c5a8": [[0, 3, 8], []], "3bd67248": [[0, 6, 8, 5], [2, 4]], "3bdb4ada": [[0, 4, 8, 7, 1], []], "3befdf3e": [[0, 2, 3, 4, 1, 7, 6], []], "3c9b0459": [[2, 8, 9, 5, 1, 3, 4], []], "3de23699": [[0, 2, 4, 8, 3, 6], []], "3e980e27": [[0, 4, 1, 2, 3, 8], []], "3eda0437": [[0, 1, 5], [6]], "3f7978a0": [[0, 8, 5], []], "4093f84a": [[0, 5, 3, 1, 2], []], "41e4d17e": [[8, 1], [6]], "4258a5f9": [[0, 5], [1]], "4290ef0e": [[8, 3, 4, 1, 2, 6, 7, 0], []], "42a50994": [[0, 5, 4, 6, 8], []], "4347f46a": [[0, 3, 5, 6, 4, 2, 7, 8], []], "444801d8": [[0, 1, 2, 3, 6, 8], []], "445eab21": [[0, 7, 4, 6, 8, 2], []], "447fd412": [[0, 2, 1], []], "44d8ac46": [[0, 5], [2]], "44f52bb0": [[0, 2], [1, 7]], "4522001f": [[0, 3, 2], []], "4612dd53": [[0, 1], [2]], "46442a0e": [[8, 6, 4, 7, 9], []], "469497ad": [[0, 3, 4, 6, 1, 7, 8, 9], [2]], "46f33fce": [[0, 1, 2, 3, 4, 8], []], "47c1f68c": [[0, 2, 8, 4, 1, 3], []], "484b58aa": [[2, 1, 5, 0, 4, 3, 6, 7, 8], []], "48d8fb45": [[0, 1, 2, 4, 5], []], "4938f0c2": [[0, 2, 3], []], "496994bd": [[0, 2, 8, 3], []], "49d1d64f": [[8, 2, 1, 3, 4, 0], []], "4be741c5": [[2, 4, 8, 3, 6, 5], []], "4c4377d9": [[4, 9, 5, 1, 3], []], "4c5c2cf0": [[0, 8, 2, 1, 3, 4], []], "50846271": [[0, 5, 2], [8]], "508bd3b6": [[0, 2, 8], [3]], "50cb2852": [[0, 2, 1, 3], [8]], "5117e062": [[0, 3, 2, 1, 6, 4, 8], []], "5168d44c": [[0, 2, 3], []], "539a4f51": [[0, 6, 1, 2, 3, 4], []], "53b68214": [[0, 1, 2, 3], []], "543a7ed5": [[8, 6], [3, 4]], "54d82841": [[0, 6, 3, 8], [4]], "54d9e175": [[0, 5, 1, 2, 3, 4], [6, 7, 8, 9]], "5521c0d9": [[0, 4, 2, 1], []], "5582e5ca": [[4, 6, 8, 9, 1, 3, 0], []], "5614dbcf": [[0, 7, 5, 2, 3, 6, 8, 9], []], "56dc2b01": [[0, 3, 2], [8]], "56ff96f3": [[0, 7, 1, 2, 3, 4], []], "57aa92db": [[0, 4, 8, 1, 2, 3, 6], []], "5ad4f10b": [[0, 6, 4, 2, 1, 8, 3], []], "5bd6f4ac": [[0, 4, 7, 1, 2, 3, 8, 6, 9, 5], []], "5c0a986e": [[0, 1, 2], []], "5c2c9af4": [[0, 2, 3, 8], []], "5daaa586": [[0, 8, 2, 4, 3, 1, 6], []], "60b61512": [[0, 4], [7]], "6150a2bd": [[0, 3, 5, 1, 2, 7, 8], []], "623ea044": [[0, 2, 7, 8], []], "62c24649": [[0, 3, 2, 1], []], "63613498": [[0, 5, 7, 1, 6, 8, 3, 4, 9], []], "6430c8c4": [[0, 7, 2, 4], [3]], "6455b5f5": [[0, 2], [1, 8]], "662c240a": [[2, 1, 3, 8, 4, 5, 7, 6, 9], []], "67385a82": [[0, 3], [8]], "673ef223": [[0, 2, 8], [4]], "6773b310": [[0, 8, 6], [1]], "67a3c6ac": [[2, 7, 6, 1], []], "67a423a3": [[0, 6, 8, 1, 9, 2, 3], [4]], "67e8384a": [[2, 1, 4, 5, 3, 7, 6, 8], []], "681b3aeb": [[0, 3, 6, 1, 7, 4], []], "6855a6e4": [[0, 2, 5], []], "68b16354": [[4, 7, 2, 8, 1, 3], []], "694f12f3": [[0, 4], [2, 1]], "6a1e5592": [[0, 2, 5], [1]], "6aa20dc0": [[8, 4, 1, 2, 3, 6], []], "6b9890af": [[0, 2, 1, 4, 8], []], "6c434453": [[0, 1], [2]], "6cdd2623": [[0, 2, 4, 5, 8, 1, 3], []], "6cf79266": [[0, 5, 3, 7], [1]], "6d0160f0": [[0, 5, 3, 7, 6, 2, 8, 1, 9, 4], []], "6d0aefbc": [[1, 6, 8], []], "6d58a25d": [[0, 3, 2, 4, 7, 9, 8], []], "6d75e8bb": [[0, 8], [2]], "6e02f1e3": [[3, 4, 2], [0, 5]], "6e19193c": [[0, 7, 9], []], "6e82a1ae": [[0, 5], [1, 2, 3]], "6ecd11f4": [[0, 8, 1, 3, 4, 2, 6, 7, 9, 5], []], "6f8cd79b": [[0], [8]], "6fa7a44f": [[7, 9, 1, 2, 4, 5, 6, 8], []], "72322fa7": [[0, 8, 1, 3, 4, 6, 2], []], "72ca375d": [[0, 8, 2, 5, 6, 3, 4, 7], []], "7447852a": [[0, 2], [4]], "7468f01a": [[0, 8, 4, 6, 3, 1, 2], []], "746b3537": [[2, 3, 4, 8, 1, 6], []], "74dd1130": [[2, 5, 1, 6, 9, 8], []], "75b8110e": [[0, 6, 9, 5, 4], []], "760b3cac": [[0, 8, 4], []], "776ffc46": [[0, 5, 1, 3, 2], []], "77fdfe62": [[1, 0, 8, 2, 4, 3, 6, 7, 9], []], "780d0b14": [[0, 8, 1, 6, 2, 4, 3, 7], []], "7837ac64": [[0, 1, 4, 3, 8, 2, 6], []], "794b24be": [[0, 1], [2]], "7b6016b9": [[0, 4, 8, 1], [3, 2]], "7b7f7511": [[1, 3, 4, 2, 6, 8], []], "7c008303": [[0, 8, 3, 1, 2, 4, 6, 5], []], "7ddcd7ec": [[0, 4, 7, 3], []], "7df24a62": [[0, 4, 1], []], "7e0986d6": [[0, 2, 3, 8, 1], []], "7f4411dc": [[0, 6, 7, 5], []], "7fe24cdd": [[0, 3, 2, 8, 5, 6], []], "80af3007": [[0, 5], []], "810b9b61": [[0, 1], [3]], "82819916": [[0, 3, 1, 2, 4, 8, 6, 5], []], "83302e8f": [[0, 1, 8, 9], [4, 3]], "834ec97d": [[0, 2, 6, 9], [4]], "8403a5d5": [[0, 2, 3, 4], [5]], "846bdb03": [[0, 3, 8, 4, 2, 1, 7], []], "855e0971": [[4, 2, 8, 1, 5, 3, 0], []], "85c4e7cd": [[2, 1, 4, 7, 8, 3, 5, 6], []], "868de0fa": [[0, 1], [2, 7]], "8731374e": [[4, 8, 1, 2, 6, 3, 5, 7, 9, 0], []], "88a10436": [[0, 2, 1, 3, 5, 6], []], "88a62173": [[0, 2, 8, 1], []], "890034e9": [[0, 1, 3, 2, 4, 8], []], "8a004b2b": [[0, 1, 2, 3, 4, 8], []], "8be77c9e": [[0, 1], []], "8d5021e8": [[0, 2, 5, 8], []], "8d510a79": [[0, 5, 1, 2], []], "8e1813be": [[0, 5, 1, 2, 4, 3, 6, 8], []], "8e5a5113": [[0, 5, 1, 3, 8, 2, 4, 6, 7], []], "8eb1be9a": [[0, 8, 2], []], "8efcae92": [[0, 1, 2], []], "8f2ea7aa": [[0, 6, 7, 8], []], "90c28cc7": [[0, 8, 2, 4, 1, 3, 7, 5], []], "90f3ed37": [[0, 8], [1]], "913fb3ed": [[0, 3, 2, 8], [6, 1, 4]], "91413438": [[0, 3, 6, 4, 2], []], "91714a58": [[0, 2, 7, 6, 3, 5, 4, 9, 1, 8], []], "9172f3a0": [[0, 3, 2, 4, 7], []], "928ad970": [[0, 4, 5, 1, 3], []], "93b581b8": [[0, 2, 3, 6, 1, 4, 5, 7, 8, 9], []], "941d9a10": [[0, 5], [2, 1, 3]], "94f9d214": [[0, 1, 3], [2]], "952a094c": [[0, 7, 1, 8, 2, 3, 6, 4, 5, 9], []], "9565186b": [[2, 8, 1, 3, 4], [5]], "95990924": [[0, 5], [1, 2, 3, 4]], "963e52fc": [[0, 2, 1, 8, 3], []], "97999447": [[0, 2, 6, 3, 8], [5]], "97a05b5b": [[0, 2, 4, 3, 8, 1, 5], []], "98cf29f8": [[0, 2, 3, 4, 5], []], "995c5fa3": [[5, 0], [2, 4, 3, 8]], "99b1bc43": [[0, 2, 1, 4], [3]], "99fa7670": [[0, 6, 8, 2, 3, 5, 7], []], "9aec4887": [[0, 8, 4, 1, 2, 3, 6, 7], []], "9af7a82c": [[1, 2, 4, 3, 8], [0]], "9d9215db": [[0, 1, 3, 4, 8, 2], []], "9dfd6313": [[0, 5, 2, 6, 8, 1, 3, 4], []], "9edfc990": [[0, 8, 9, 6, 5, 4, 3, 1, 7, 2], []], "9f236235": [[0, 2, 8, 3, 1, 4], []], "a1570a43": [[0, 2, 3], []], "a2fd1cf0": [[0, 2, 3], [8]], "a3325580": [[0, 4, 6, 8, 3, 9, 2, 1, 7], []], "a416b8f3": [[0, 2, 5, 3, 8, 1, 6], []], "a48eeaf7": [[0, 2, 5], []], "a5313dff": [[0, 2], [1]], "a5f85a15": [[0, 9, 3, 2], [4]], "a61ba2ce": [[0, 1, 2, 8, 3, 4], []], "a61f2674": [[0, 5], [1, 2]], "a64e4611": [[0, 8, 1, 2], [3]], "a65b410d": [[0, 2], [3, 1]], "a68b268e": [[0, 1, 7, 4, 6, 8], []], "a699fb00": [[0, 1], [2]], "a740d043": [[1, 2, 5, 3, 6], [0]], "a78176bb": [[0, 5, 7, 2, 9], []], "a79310a0": [[0, 8], [2]], "a85d4709": [[0, 5], [3, 4, 2]], "a87f7484": [[0, 8, 4, 2, 3, 7, 1, 6], []], "a8c38be5": [[0, 5, 2, 4, 1, 3, 6, 8, 9, 7], []], "a8d7556c": [[5, 0], [2]], "a9f96cdd": [[0, 2], [3, 7, 8, 6]], "aabf363d": [[0, 2, 3, 4, 6], []], "aba27056": [[0, 7, 3, 6], [4]], "ac0a08a4": [[0, 2, 6, 7, 3, 4, 8, 9], []], "ae3edfdc": [[0, 7, 3, 1, 2], []], "ae4f1146": [[0, 8, 1], []], "aedd82e4": [[0, 2], [1]], "af902bf9": [[0, 4], [2]], "b0c4d837": [[0, 5, 8], []], "b190f7f5": [[0, 8, 4, 2, 1, 3], []], "b1948b0a": [[7, 6], [2]], "b230c067": [[0, 8], [1, 2]], "b27ca6d3": [[0, 2], [3]], "b2862040": [[9, 1], [8]], "b527c5c6": [[0, 3, 2], []], "b548a754": [[0, 2, 1, 3, 6, 8], []], "b60334d2": [[0, 5], [1]], "b6afb2da": [[0, 5], [4, 2, 1]], "b7249182": [[0, 8, 1, 2, 3, 5], []], "b775ac94": [[0, 2, 1, 8, 3, 4, 7, 6], []], "b782dc8a": [[8, 0, 3, 1, 2, 4], []], "b8825c91": [[1, 6, 9, 8, 5, 7, 3, 4, 2], []], "b8cdaf2b": [[0, 2, 6, 8, 4, 1, 3], []], "b91ae062": [[0, 6, 2, 3, 4, 8, 1, 7, 9], []], "b94a9452": [[0, 4, 2, 1, 3, 6], []], "b9b7f026": [[0, 2, 7, 3, 1, 8, 4, 5, 6], []], "ba26e723": [[4, 0], [6]], "ba97ae07": [[0, 3, 8, 1, 6, 7, 2], []], "bb43febb": [[0, 5], [2]], "bbc9ae5d": [[0, 8, 5, 1, 2, 7], []], "bc1d5164": [[0, 4, 8, 2, 3], []], "bd4472b8": [[0, 5, 1, 2, 3, 4, 8], []], "bda2d7a6": [[0, 8, 3, 2, 5, 7, 6], []], "bdad9b1f": [[0, 2, 8], [4]], "be94b721": [[0, 2, 8, 7, 3, 1, 4, 6], []], "beb8660c": [[0, 8, 2, 5, 4, 6, 1, 3], []], "c0f76784": [[0, 5], [8, 7, 6]], "c1d99e64": [[0, 3, 1, 8], [2]], "c3e719e8": [[8, 9, 3, 6, 7, 4], [0]], "c444b776": [[0, 4, 2, 1, 3, 5, 7, 8], []], "c59eb873": [[0, 3, 5, 1, 2], []], "c8cbb738": [[4, 3, 1, 8, 2, 7], []], "c8f0f002": [[1, 7, 8], [5]], "c909285e": [[8, 3, 0, 5, 1, 4, 2, 6], []], "c9e6f938": [[0, 7], []], "c9f8e694": [[0, 5, 1, 3, 4, 2], []], "caa06a1f": [[3, 6, 7, 1, 5, 4], []], "cbded52d": [[1, 0, 3, 4, 2, 7], []], "cce03e0d": [[0, 1, 2], []], "cdecee7f": [[0, 3, 1, 5, 9, 2, 4, 6, 8, 7], []], "ce22a75a": [[0, 5], [1]], "ce4f8723": [[0, 1, 2, 4], [3]], "ce602527": [[1, 8, 2, 3, 6, 4], []], "ce9e57f2": [[0, 2], [8]], "cf98881b": [[0, 4, 2, 1, 9], []], "d037b0a7": [[0, 2, 4, 3, 6, 7, 8], []], "d06dbe63": [[0, 8], [5]], "d07ae81c": [[1, 6, 2, 8, 3, 4], []], "d0f5fe59": [[0, 8], []], "d10ecb37": [[2, 4, 3, 1, 5, 0, 6], []], "d13f3404": [[0, 1, 3, 6, 2, 4, 8], []], "d22278a0": [[0, 2, 1, 8, 3, 4], []], "d23f8c26": [[0, 3, 6, 8, 4, 5, 9, 2, 7, 1], []], "d2abd087": [[0, 5], [1, 2]], "d364b489": [[0, 1], [2, 7, 8, 6]], "d406998b": [[0, 5], [3]], "d43fd935": [[0, 3, 1, 7, 8, 6], []], "d4469b4b": [[0, 2, 1, 3], [5]], "d4a91cb9": [[0, 2, 8], [4]], "d4f3cd78": [[0, 5], [8]], "d511f180": [[5, 8, 2, 3, 4, 6, 1, 7, 9], []], "d5d6de2d": [[0, 2], [3]], "d631b094": [[0, 8, 2, 1, 7], []], "d687bc17": [[0, 4, 8, 2, 3, 7, 1, 6], []], "d6ad076f": [[0, 3, 4, 7, 6, 2, 9], [8]], "d89b689b": [[0, 8, 1, 4, 6, 9, 2, 3, 5, 7], []], "d8c310e9": [[0, 2, 3, 4, 1], []], "d90796e8": [[0, 3, 2, 5], [8]], "d9f24cd1": [[0, 2, 5], []], "d9fac9be": [[0, 3, 2, 8, 1, 4], []], "dae9d2b5": [[0, 3, 4], [6]], "db3e9e38": [[0, 7], [8]], "db93a21d": [[0, 9], [3, 1]], "dbc1a6ce": [[0, 1], [8]], "dc1df850": [[0, 2, 3, 6, 8], [1]], "dc433765": [[0, 3, 4], []], "ddf7fa4f": [[0, 5, 1, 6, 7, 2, 4, 8], []], "de1cd16c": [[8, 1, 4, 0, 6, 5, 2, 3], []], "ded97339": [[0, 8], []], "e179c5f4": [[0, 1], [8]], "e21d9049": [[0, 8, 2, 3, 4], []], "e26a3af2": [[8, 2, 1, 3, 7, 9, 4, 5, 6], []], "e3497940": [[0, 5, 2, 8, 3, 4, 6, 7], []], "e40b9e2f": [[0, 8, 4, 6, 7, 3, 9], []], "e48d4e1a": [[0, 2, 3, 4, 6, 5], []], "e5062a87": [[5, 0, 2], []], "e509e548": [[0, 3], [6, 2, 1]], "e50d258f": [[0, 8, 1, 2], []], "e6721834": [[8, 0, 1, 6, 4, 3, 2], []], "e73095fd": [[0, 5], [4]], "e76a88a6": [[0, 5, 6, 8, 4, 2], []], "e8593010": [[5, 0], [2, 3, 1]], "e8dc4411": [[1, 4, 8, 0, 2, 3], []], "e9614598": [[0, 1], [3]], "e98196ab": [[0, 5, 1, 7, 2, 3, 8], []], "e9afcf9a": [[3, 4, 8, 9], []], "ea32f347": [[0, 5], [1, 4, 2]], "ea786f4a": [[3, 2, 1, 0], []], "eb281b96": [[0, 2, 8], []], "eb5a1d5d": [[3, 8, 5, 6, 2, 1], []], "ec883f72": [[0, 4, 5, 8, 9, 2, 3, 6], []], "ecdecbb3": [[0, 8, 2], []], "ed36ccf7": [[0, 9, 2, 6], []], "ef135b50": [[0, 2], [9]], "f15e1fac": [[0, 8, 2], []], "f1cefba8": [[0, 2, 8, 3, 1, 4], []], "f25fbde4": [[0, 4], []], "f25ffba3": [[0, 2, 3, 9, 8, 4], []], "f2829549": [[0, 5, 7, 1], [3]], "f35d900a": [[0, 2, 3, 8, 1, 4], [5]], "f5b8619d": [[0, 5, 2, 4], [8]], "f76d97a5": [[5, 9, 6, 4], [0]], "f8a8fe49": [[0, 2, 5], []], "f8b3ba0a": [[0, 1, 8, 3, 2, 4, 6], []], "f8c80d96": [[0, 8, 2, 1], [5]], "f8ff0b80": [[0, 2, 1, 4, 3, 7, 8], []], "f9012d9b": [[2, 5, 1, 0, 6, 8], []], "fafffa47": [[0, 9, 1], [2]], "fcb5c309": [[0, 3, 2, 1, 4], []], "fcc82909": [[0, 6, 9, 4, 7, 8, 2, 1], [3]], "feca6190": [[0, 4, 8, 1, 2, 6, 7, 9], []], "ff28f65a": [[0, 2], [1]], "ff805c23": [[0, 3, 2, 5, 8, 6, 1], []]}


In [ ]:
%%writefile /kaggle/working/cpu_solver/wwhdr_priors.json
{
"learner:tile": 20,
"learner:fractal": 4,
"learner:cell": 81,
"fill_enc4": 1,
"fam:fill": 3,
"learner:part": 31,
"attract8": 1,
"fam:move": 5,
"learner:attract8": 1,
"learner:crop": 18,
"learner:cmap": 4,
"learner:down": 5,
"remove_col8": 1,
"fam:color": 2,
"symfill0": 1,
"fam:pattern": 3,
"learner:symfill0": 1,
"grav_down": 2,
"learner:grav": 2,
"remove_col2": 1,
"learner:countfill": 1,
"crop_nonbg": 5,
"fam:crop": 18,
"quad_mirror": 1,
"fam:tile": 1,
"down4": 1,
"fam:scale": 4,
"left_half": 4,
"grav_up": 1,
"learner:count": 2,
"perfill0": 1,
"learner:perfill0": 1,
"top_half": 1,
"fill_enc2": 1,
"fliplr": 2,
"fam:geom": 3,
"dedup_rows": 3,
"dedup_cols": 3,
"learner:dedup": 3,
"up3": 1,
"learner:up3": 1,
"learner:upscale": 2,
"remove_bg_lines": 2,
"fill_enc8": 1,
"up2": 1,
"learner:up2": 1,
"ograv_right": 1,
"learner:ograv": 1,
"mirror_lr_5": 1,
"pad_edge": 1,
"rot90": 1,
"learner:rot90": 1,
"_total": 180
}


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel/dsl.py
from arc_types import *


def identity(
    x: Any
) -> Any:
    """ identity function """
    return x


def add(
    a: Numerical,
    b: Numerical
) -> Numerical:
    """ addition """
    if isinstance(a, int) and isinstance(b, int):
        return a + b
    elif isinstance(a, tuple) and isinstance(b, tuple):
        return (a[0] + b[0], a[1] + b[1])
    elif isinstance(a, int) and isinstance(b, tuple):
        return (a + b[0], a + b[1])
    return (a[0] + b, a[1] + b)


def subtract(
    a: Numerical,
    b: Numerical
) -> Numerical:
    """ subtraction """
    if isinstance(a, int) and isinstance(b, int):
        return a - b
    elif isinstance(a, tuple) and isinstance(b, tuple):
        return (a[0] - b[0], a[1] - b[1])
    elif isinstance(a, int) and isinstance(b, tuple):
        return (a - b[0], a - b[1])
    return (a[0] - b, a[1] - b)


def multiply(
    a: Numerical,
    b: Numerical
) -> Numerical:
    """ multiplication """
    if isinstance(a, int) and isinstance(b, int):
        return a * b
    elif isinstance(a, tuple) and isinstance(b, tuple):
        return (a[0] * b[0], a[1] * b[1])
    elif isinstance(a, int) and isinstance(b, tuple):
        return (a * b[0], a * b[1])
    return (a[0] * b, a[1] * b)
    

def divide(
    a: Numerical,
    b: Numerical
) -> Numerical:
    """ floor division """
    if isinstance(a, int) and isinstance(b, int):
        return a // b
    elif isinstance(a, tuple) and isinstance(b, tuple):
        return (a[0] // b[0], a[1] // b[1])
    elif isinstance(a, int) and isinstance(b, tuple):
        return (a // b[0], a // b[1])
    return (a[0] // b, a[1] // b)


def invert(
    n: Numerical
) -> Numerical:
    """ inversion with respect to addition """
    return -n if isinstance(n, int) else (-n[0], -n[1])


def even(
    n: Integer
) -> Boolean:
    """ evenness """
    return n % 2 == 0


def double(
    n: Numerical
) -> Numerical:
    """ scaling by two """
    return n * 2 if isinstance(n, int) else (n[0] * 2, n[1] * 2)


def halve(
    n: Numerical
) -> Numerical:
    """ scaling by one half """
    return n // 2 if isinstance(n, int) else (n[0] // 2, n[1] // 2)


def flip(
    b: Boolean
) -> Boolean:
    """ logical not """
    return not b


def equality(
    a: Any,
    b: Any
) -> Boolean:
    """ equality """
    return a == b


def contained(
    value: Any,
    container: Container
) -> Boolean:
    """ element of """
    return value in container


def combine(
    a: Container,
    b: Container
) -> Container:
    """ union """
    return type(a)((*a, *b))


def intersection(
    a: FrozenSet,
    b: FrozenSet
) -> FrozenSet:
    """ returns the intersection of two containers """
    return a & b


def difference(
    a: FrozenSet,
    b: FrozenSet
) -> FrozenSet:
    """ set difference """
    return type(a)(e for e in a if e not in b)


def dedupe(
    tup: Tuple
) -> Tuple:
    """ remove duplicates """
    return tuple(e for i, e in enumerate(tup) if tup.index(e) == i)


def order(
    container: Container,
    compfunc: Callable
) -> Tuple:
    """ order container by custom key """
    return tuple(sorted(container, key=compfunc))


def repeat(
    item: Any,
    num: Integer
) -> Tuple:
    """ repetition of item within vector """
    return tuple(item for i in range(num))


def greater(
    a: Integer,
    b: Integer
) -> Boolean:
    """ greater """
    return a > b


def size(
    container: Container
) -> Integer:
    """ cardinality """
    return len(container)


def merge(
    containers: ContainerContainer
) -> Container:
    """ merging """
    return type(containers)(e for c in containers for e in c)


def maximum(
    container: IntegerSet
) -> Integer:
    """ maximum """
    return max(container, default=0)


def minimum(
    container: IntegerSet
) -> Integer:
    """ minimum """
    return min(container, default=0)


def valmax(
    container: Container,
    compfunc: Callable
) -> Integer:
    """ maximum by custom function """
    return compfunc(max(container, key=compfunc, default=0))


def valmin(
    container: Container,
    compfunc: Callable
) -> Integer:
    """ minimum by custom function """
    return compfunc(min(container, key=compfunc, default=0))


def argmax(
    container: Container,
    compfunc: Callable
) -> Any:
    """ largest item by custom order """
    return max(container, key=compfunc)


def argmin(
    container: Container,
    compfunc: Callable
) -> Any:
    """ smallest item by custom order """
    return min(container, key=compfunc)


def mostcommon(
    container: Container
) -> Any:
    """ most common item """
    return max(set(container), key=container.count)


def leastcommon(
    container: Container
) -> Any:
    """ least common item """
    return min(set(container), key=container.count)


def initset(
    value: Any
) -> FrozenSet:
    """ initialize container """
    return frozenset({value})


def both(
    a: Boolean,
    b: Boolean
) -> Boolean:
    """ logical and """
    return a and b


def either(
    a: Boolean,
    b: Boolean
) -> Boolean:
    """ logical or """
    return a or b


def increment(
    x: Numerical
) -> Numerical:
    """ incrementing """
    return x + 1 if isinstance(x, int) else (x[0] + 1, x[1] + 1)


def decrement(
    x: Numerical
) -> Numerical:
    """ decrementing """
    return x - 1 if isinstance(x, int) else (x[0] - 1, x[1] - 1)


def crement(
    x: Numerical
) -> Numerical:
    """ incrementing positive and decrementing negative """
    if isinstance(x, int):
        return 0 if x == 0 else (x + 1 if x > 0 else x - 1)
    return (
        0 if x[0] == 0 else (x[0] + 1 if x[0] > 0 else x[0] - 1),
        0 if x[1] == 0 else (x[1] + 1 if x[1] > 0 else x[1] - 1)
    )


def sign(
    x: Numerical
) -> Numerical:
    """ sign """
    if isinstance(x, int):
        return 0 if x == 0 else (1 if x > 0 else -1)
    return (
        0 if x[0] == 0 else (1 if x[0] > 0 else -1),
        0 if x[1] == 0 else (1 if x[1] > 0 else -1)
    )


def positive(
    x: Integer
) -> Boolean:
    """ positive """
    return x > 0


def toivec(
    i: Integer
) -> IntegerTuple:
    """ vector pointing vertically """
    return (i, 0)


def tojvec(
    j: Integer
) -> IntegerTuple:
    """ vector pointing horizontally """
    return (0, j)


def sfilter(
    container: Container,
    condition: Callable
) -> Container:
    """ keep elements in container that satisfy condition """
    return type(container)(e for e in container if condition(e))


def mfilter(
    container: Container,
    function: Callable
) -> FrozenSet:
    """ filter and merge """
    return merge(sfilter(container, function))


def extract(
    container: Container,
    condition: Callable
) -> Any:
    """ first element of container that satisfies condition """
    return next(e for e in container if condition(e))


def totuple(
    container: FrozenSet
) -> Tuple:
    """ conversion to tuple """
    return tuple(container)


def first(
    container: Container
) -> Any:
    """ first item of container """
    return next(iter(container))


def last(
    container: Container
) -> Any:
    """ last item of container """
    return max(enumerate(container))[1]


def insert(
    value: Any,
    container: FrozenSet
) -> FrozenSet:
    """ insert item into container """
    return container.union(frozenset({value}))


def remove(
    value: Any,
    container: Container
) -> Container:
    """ remove item from container """
    return type(container)(e for e in container if e != value)


def other(
    container: Container,
    value: Any
) -> Any:
    """ other value in the container """
    return first(remove(value, container))


def interval(
    start: Integer,
    stop: Integer,
    step: Integer
) -> Tuple:
    """ range """
    return tuple(range(start, stop, step))


def astuple(
    a: Integer,
    b: Integer
) -> IntegerTuple:
    """ constructs a tuple """
    return (a, b)


def product(
    a: Container,
    b: Container
) -> FrozenSet:
    """ cartesian product """
    return frozenset((i, j) for j in b for i in a)


def pair(
    a: Tuple,
    b: Tuple
) -> TupleTuple:
    """ zipping of two tuples """
    return tuple(zip(a, b))


def branch(
    condition: Boolean,
    a: Any,
    b: Any
) -> Any:
    """ if else branching """
    return a if condition else b


def compose(
    outer: Callable,
    inner: Callable
) -> Callable:
    """ function composition """
    return lambda x: outer(inner(x))


def chain(
    h: Callable,
    g: Callable,
    f: Callable,
) -> Callable:
    """ function composition with three functions """
    return lambda x: h(g(f(x)))


def matcher(
    function: Callable,
    target: Any
) -> Callable:
    """ construction of equality function """
    return lambda x: function(x) == target


def rbind(
    function: Callable,
    fixed: Any
) -> Callable:
    """ fix the rightmost argument """
    n = function.__code__.co_argcount
    if n == 2:
        return lambda x: function(x, fixed)
    elif n == 3:
        return lambda x, y: function(x, y, fixed)
    else:
        return lambda x, y, z: function(x, y, z, fixed)


def lbind(
    function: Callable,
    fixed: Any
) -> Callable:
    """ fix the leftmost argument """
    n = function.__code__.co_argcount
    if n == 2:
        return lambda y: function(fixed, y)
    elif n == 3:
        return lambda y, z: function(fixed, y, z)
    else:
        return lambda y, z, a: function(fixed, y, z, a)


def power(
    function: Callable,
    n: Integer
) -> Callable:
    """ power of function """
    if n == 1:
        return function
    return compose(function, power(function, n - 1))


def fork(
    outer: Callable,
    a: Callable,
    b: Callable
) -> Callable:
    """ creates a wrapper function """
    return lambda x: outer(a(x), b(x))


def apply(
    function: Callable,
    container: Container
) -> Container:
    """ apply function to each item in container """
    return type(container)(function(e) for e in container)


def rapply(
    functions: Container,
    value: Any
) -> Container:
    """ apply each function in container to value """
    return type(functions)(function(value) for function in functions)


def mapply(
    function: Callable,
    container: ContainerContainer
) -> FrozenSet:
    """ apply and merge """
    return merge(apply(function, container))


def papply(
    function: Callable,
    a: Tuple,
    b: Tuple
) -> Tuple:
    """ apply function on two vectors """
    return tuple(function(i, j) for i, j in zip(a, b))


def mpapply(
    function: Callable,
    a: Tuple,
    b: Tuple
) -> Tuple:
    """ apply function on two vectors and merge """
    return merge(papply(function, a, b))


def prapply(
    function,
    a: Container,
    b: Container
) -> FrozenSet:
    """ apply function on cartesian product """
    return frozenset(function(i, j) for j in b for i in a)


def mostcolor(
    element: Element
) -> Integer:
    """ most common color """
    values = [v for r in element for v in r] if isinstance(element, tuple) else [v for v, _ in element]
    return max(set(values), key=values.count)
    

def leastcolor(
    element: Element
) -> Integer:
    """ least common color """
    values = [v for r in element for v in r] if isinstance(element, tuple) else [v for v, _ in element]
    return min(set(values), key=values.count)


def height(
    piece: Piece
) -> Integer:
    """ height of grid or patch """
    if len(piece) == 0:
        return 0
    if isinstance(piece, tuple):
        return len(piece)
    return lowermost(piece) - uppermost(piece) + 1


def width(
    piece: Piece
) -> Integer:
    """ width of grid or patch """
    if len(piece) == 0:
        return 0
    if isinstance(piece, tuple):
        return len(piece[0])
    return rightmost(piece) - leftmost(piece) + 1


def shape(
    piece: Piece
) -> IntegerTuple:
    """ height and width of grid or patch """
    return (height(piece), width(piece))


def portrait(
    piece: Piece
) -> Boolean:
    """ whether height is greater than width """
    return height(piece) > width(piece)


def colorcount(
    element: Element,
    value: Integer
) -> Integer:
    """ number of cells with color """
    if isinstance(element, tuple):
        return sum(row.count(value) for row in element)
    return sum(v == value for v, _ in element)


def colorfilter(
    objs: Objects,
    value: Integer
) -> Objects:
    """ filter objects by color """
    return frozenset(obj for obj in objs if next(iter(obj))[0] == value)


def sizefilter(
    container: Container,
    n: Integer
) -> FrozenSet:
    """ filter items by size """
    return frozenset(item for item in container if len(item) == n)


def asindices(
    grid: Grid
) -> Indices:
    """ indices of all grid cells """
    return frozenset((i, j) for i in range(len(grid)) for j in range(len(grid[0])))


def ofcolor(
    grid: Grid,
    value: Integer
) -> Indices:
    """ indices of all grid cells with value """
    return frozenset((i, j) for i, r in enumerate(grid) for j, v in enumerate(r) if v == value)


def ulcorner(
    patch: Patch
) -> IntegerTuple:
    """ index of upper left corner """
    return tuple(map(min, zip(*toindices(patch))))


def urcorner(
    patch: Patch
) -> IntegerTuple:
    """ index of upper right corner """
    return tuple(map(lambda ix: {0: min, 1: max}[ix[0]](ix[1]), enumerate(zip(*toindices(patch)))))


def llcorner(
    patch: Patch
) -> IntegerTuple:
    """ index of lower left corner """
    return tuple(map(lambda ix: {0: max, 1: min}[ix[0]](ix[1]), enumerate(zip(*toindices(patch)))))


def lrcorner(
    patch: Patch
) -> IntegerTuple:
    """ index of lower right corner """
    return tuple(map(max, zip(*toindices(patch))))


def crop(
    grid: Grid,
    start: IntegerTuple,
    dims: IntegerTuple
) -> Grid:
    """ subgrid specified by start and dimension """
    return tuple(r[start[1]:start[1]+dims[1]] for r in grid[start[0]:start[0]+dims[0]])


def toindices(
    patch: Patch
) -> Indices:
    """ indices of object cells """
    if len(patch) == 0:
        return frozenset()
    if isinstance(next(iter(patch))[1], tuple):
        return frozenset(index for value, index in patch)
    return patch


def recolor(
    value: Integer,
    patch: Patch
) -> Object:
    """ recolor patch """
    return frozenset((value, index) for index in toindices(patch))


def shift(
    patch: Patch,
    directions: IntegerTuple
) -> Patch:
    """ shift patch """
    if len(patch) == 0:
        return patch
    di, dj = directions
    if isinstance(next(iter(patch))[1], tuple):
        return frozenset((value, (i + di, j + dj)) for value, (i, j) in patch)
    return frozenset((i + di, j + dj) for i, j in patch)


def normalize(
    patch: Patch
) -> Patch:
    """ moves upper left corner to origin """
    if len(patch) == 0:
        return patch
    return shift(patch, (-uppermost(patch), -leftmost(patch)))


def dneighbors(
    loc: IntegerTuple
) -> Indices:
    """ directly adjacent indices """
    return frozenset({(loc[0] - 1, loc[1]), (loc[0] + 1, loc[1]), (loc[0], loc[1] - 1), (loc[0], loc[1] + 1)})


def ineighbors(
    loc: IntegerTuple
) -> Indices:
    """ diagonally adjacent indices """
    return frozenset({(loc[0] - 1, loc[1] - 1), (loc[0] - 1, loc[1] + 1), (loc[0] + 1, loc[1] - 1), (loc[0] + 1, loc[1] + 1)})


def neighbors(
    loc: IntegerTuple
) -> Indices:
    """ adjacent indices """
    return dneighbors(loc) | ineighbors(loc)


def objects(
    grid: Grid,
    univalued: Boolean,
    diagonal: Boolean,
    without_bg: Boolean
) -> Objects:
    """ objects occurring on the grid """
    bg = mostcolor(grid) if without_bg else None
    objs = set()
    occupied = set()
    h, w = len(grid), len(grid[0])
    unvisited = asindices(grid)
    diagfun = neighbors if diagonal else dneighbors
    for loc in unvisited:
        if loc in occupied:
            continue
        val = grid[loc[0]][loc[1]]
        if val == bg:
            continue
        obj = {(val, loc)}
        cands = {loc}
        while len(cands) > 0:
            neighborhood = set()
            for cand in cands:
                v = grid[cand[0]][cand[1]]
                if (val == v) if univalued else (v != bg):
                    obj.add((v, cand))
                    occupied.add(cand)
                    neighborhood |= {
                        (i, j) for i, j in diagfun(cand) if 0 <= i < h and 0 <= j < w
                    }
            cands = neighborhood - occupied
        objs.add(frozenset(obj))
    return frozenset(objs)


def partition(
    grid: Grid
) -> Objects:
    """ each cell with the same value part of the same object """
    return frozenset(
        frozenset(
            (v, (i, j)) for i, r in enumerate(grid) for j, v in enumerate(r) if v == value
        ) for value in palette(grid)
    )


def fgpartition(
    grid: Grid
) -> Objects:
    """ each cell with the same value part of the same object without background """
    return frozenset(
        frozenset(
            (v, (i, j)) for i, r in enumerate(grid) for j, v in enumerate(r) if v == value
        ) for value in palette(grid) - {mostcolor(grid)}
    )


def uppermost(
    patch: Patch
) -> Integer:
    """ row index of uppermost occupied cell """
    return min(i for i, j in toindices(patch))


def lowermost(
    patch: Patch
) -> Integer:
    """ row index of lowermost occupied cell """
    return max(i for i, j in toindices(patch))


def leftmost(
    patch: Patch
) -> Integer:
    """ column index of leftmost occupied cell """
    return min(j for i, j in toindices(patch))


def rightmost(
    patch: Patch
) -> Integer:
    """ column index of rightmost occupied cell """
    return max(j for i, j in toindices(patch))


def square(
    piece: Piece
) -> Boolean:
    """ whether the piece forms a square """
    return len(piece) == len(piece[0]) if isinstance(piece, tuple) else height(piece) * width(piece) == len(piece) and height(piece) == width(piece)


def vline(
    patch: Patch
) -> Boolean:
    """ whether the piece forms a vertical line """
    return height(patch) == len(patch) and width(patch) == 1


def hline(
    patch: Patch
) -> Boolean:
    """ whether the piece forms a horizontal line """
    return width(patch) == len(patch) and height(patch) == 1


def hmatching(
    a: Patch,
    b: Patch
) -> Boolean:
    """ whether there exists a row for which both patches have cells """
    return len(set(i for i, j in toindices(a)) & set(i for i, j in toindices(b))) > 0


def vmatching(
    a: Patch,
    b: Patch
) -> Boolean:
    """ whether there exists a column for which both patches have cells """
    return len(set(j for i, j in toindices(a)) & set(j for i, j in toindices(b))) > 0


def manhattan(
    a: Patch,
    b: Patch
) -> Integer:
    """ closest manhattan distance between two patches """
    return min(abs(ai - bi) + abs(aj - bj) for ai, aj in toindices(a) for bi, bj in toindices(b))


def adjacent(
    a: Patch,
    b: Patch
) -> Boolean:
    """ whether two patches are adjacent """
    return manhattan(a, b) == 1


def bordering(
    patch: Patch,
    grid: Grid
) -> Boolean:
    """ whether a patch is adjacent to a grid border """
    return uppermost(patch) == 0 or leftmost(patch) == 0 or lowermost(patch) == len(grid) - 1 or rightmost(patch) == len(grid[0]) - 1


def centerofmass(
    patch: Patch
) -> IntegerTuple:
    """ center of mass """
    return tuple(map(lambda x: sum(x) // len(patch), zip(*toindices(patch))))


def palette(
    element: Element
) -> IntegerSet:
    """ colors occurring in object or grid """
    if isinstance(element, tuple):
        return frozenset({v for r in element for v in r})
    return frozenset({v for v, _ in element})


def numcolors(
    element: Element
) -> IntegerSet:
    """ number of colors occurring in object or grid """
    return len(palette(element))


def color(
    obj: Object
) -> Integer:
    """ color of object """
    return next(iter(obj))[0]


def toobject(
    patch: Patch,
    grid: Grid
) -> Object:
    """ object from patch and grid """
    h, w = len(grid), len(grid[0])
    return frozenset((grid[i][j], (i, j)) for i, j in toindices(patch) if 0 <= i < h and 0 <= j < w)


def asobject(
    grid: Grid
) -> Object:
    """ conversion of grid to object """
    return frozenset((v, (i, j)) for i, r in enumerate(grid) for j, v in enumerate(r))


def rot90(
    grid: Grid
) -> Grid:
    """ quarter clockwise rotation """
    return tuple(row for row in zip(*grid[::-1]))


def rot180(
    grid: Grid
) -> Grid:
    """ half rotation """
    return tuple(tuple(row[::-1]) for row in grid[::-1])


def rot270(
    grid: Grid
) -> Grid:
    """ quarter anticlockwise rotation """
    return tuple(tuple(row[::-1]) for row in zip(*grid[::-1]))[::-1]


def hmirror(
    piece: Piece
) -> Piece:
    """ mirroring along horizontal """
    if isinstance(piece, tuple):
        return piece[::-1]
    d = ulcorner(piece)[0] + lrcorner(piece)[0]
    if isinstance(next(iter(piece))[1], tuple):
        return frozenset((v, (d - i, j)) for v, (i, j) in piece)
    return frozenset((d - i, j) for i, j in piece)


def vmirror(
    piece: Piece
) -> Piece:
    """ mirroring along vertical """
    if isinstance(piece, tuple):
        return tuple(row[::-1] for row in piece)
    d = ulcorner(piece)[1] + lrcorner(piece)[1]
    if isinstance(next(iter(piece))[1], tuple):
        return frozenset((v, (i, d - j)) for v, (i, j) in piece)
    return frozenset((i, d - j) for i, j in piece)


def dmirror(
    piece: Piece
) -> Piece:
    """ mirroring along diagonal """
    if isinstance(piece, tuple):
        return tuple(zip(*piece))
    a, b = ulcorner(piece)
    if isinstance(next(iter(piece))[1], tuple):
        return frozenset((v, (j - b + a, i - a + b)) for v, (i, j) in piece)
    return frozenset((j - b + a, i - a + b) for i, j in piece)


def cmirror(
    piece: Piece
) -> Piece:
    """ mirroring along counterdiagonal """
    if isinstance(piece, tuple):
        return tuple(zip(*(r[::-1] for r in piece[::-1])))
    return vmirror(dmirror(vmirror(piece)))


def fill(
    grid: Grid,
    value: Integer,
    patch: Patch
) -> Grid:
    """ fill value at indices """
    h, w = len(grid), len(grid[0])
    grid_filled = list(list(row) for row in grid)
    for i, j in toindices(patch):
        if 0 <= i < h and 0 <= j < w:
            grid_filled[i][j] = value
    return tuple(tuple(row) for row in grid_filled)


def paint(
    grid: Grid,
    obj: Object
) -> Grid:
    """ paint object to grid """
    h, w = len(grid), len(grid[0])
    grid_painted = list(list(row) for row in grid)
    for value, (i, j) in obj:
        if 0 <= i < h and 0 <= j < w:
            grid_painted[i][j] = value
    return tuple(tuple(row) for row in grid_painted)


def underfill(
    grid: Grid,
    value: Integer,
    patch: Patch
) -> Grid:
    """ fill value at indices that are background """
    h, w = len(grid), len(grid[0])
    bg = mostcolor(grid)
    g = list(list(r) for r in grid)
    for i, j in toindices(patch):
        if 0 <= i < h and 0 <= j < w:
            if g[i][j] == bg:
                g[i][j] = value
    return tuple(tuple(r) for r in g)


def underpaint(
    grid: Grid,
    obj: Object
) -> Grid:
    """ paint object to grid where there is background """
    h, w = len(grid), len(grid[0])
    bg = mostcolor(grid)
    g = list(list(r) for r in grid)
    for value, (i, j) in obj:
        if 0 <= i < h and 0 <= j < w:
            if g[i][j] == bg:
                g[i][j] = value
    return tuple(tuple(r) for r in g)


def hupscale(
    grid: Grid,
    factor: Integer
) -> Grid:
    """ upscale grid horizontally """
    g = tuple()
    for row in grid:
        r = tuple()
        for value in row:
            r = r + tuple(value for num in range(factor))
        g = g + (r,)
    return g


def vupscale(
    grid: Grid,
    factor: Integer
) -> Grid:
    """ upscale grid vertically """
    g = tuple()
    for row in grid:
        g = g + tuple(row for num in range(factor))
    return g


def upscale(
    element: Element,
    factor: Integer
) -> Element:
    """ upscale object or grid """
    if isinstance(element, tuple):
        g = tuple()
        for row in element:
            upscaled_row = tuple()
            for value in row:
                upscaled_row = upscaled_row + tuple(value for num in range(factor))
            g = g + tuple(upscaled_row for num in range(factor))
        return g
    else:
        if len(element) == 0:
            return frozenset()
        di_inv, dj_inv = ulcorner(element)
        di, dj = (-di_inv, -dj_inv)
        normed_obj = shift(element, (di, dj))
        o = set()
        for value, (i, j) in normed_obj:
            for io in range(factor):
                for jo in range(factor):
                    o.add((value, (i * factor + io, j * factor + jo)))
        return shift(frozenset(o), (di_inv, dj_inv))


def downscale(
    grid: Grid,
    factor: Integer
) -> Grid:
    """ downscale grid """
    h, w = len(grid), len(grid[0])
    g = tuple()
    for i in range(h):
        r = tuple()
        for j in range(w):
            if j % factor == 0:
                r = r + (grid[i][j],)
        g = g + (r, )
    h = len(g)
    dsg = tuple()
    for i in range(h):
        if i % factor == 0:
            dsg = dsg + (g[i],)
    return dsg


def hconcat(
    a: Grid,
    b: Grid
) -> Grid:
    """ concatenate two grids horizontally """
    return tuple(i + j for i, j in zip(a, b))


def vconcat(
    a: Grid,
    b: Grid
) -> Grid:
    """ concatenate two grids vertically """
    return a + b


def subgrid(
    patch: Patch,
    grid: Grid
) -> Grid:
    """ smallest subgrid containing object """
    return crop(grid, ulcorner(patch), shape(patch))


def hsplit(
    grid: Grid,
    n: Integer
) -> Tuple:
    """ split grid horizontally """
    h, w = len(grid), len(grid[0]) // n
    offset = len(grid[0]) % n != 0
    return tuple(crop(grid, (0, w * i + i * offset), (h, w)) for i in range(n))


def vsplit(
    grid: Grid,
    n: Integer
) -> Tuple:
    """ split grid vertically """
    h, w = len(grid) // n, len(grid[0])
    offset = len(grid) % n != 0
    return tuple(crop(grid, (h * i + i * offset, 0), (h, w)) for i in range(n))


def cellwise(
    a: Grid,
    b: Grid,
    fallback: Integer
) -> Grid:
    """ cellwise match of two grids """
    h, w = len(a), len(a[0])
    resulting_grid = tuple()
    for i in range(h):
        row = tuple()
        for j in range(w):
            a_value = a[i][j]
            value = a_value if a_value == b[i][j] else fallback
            row = row + (value,)
        resulting_grid = resulting_grid + (row, )
    return resulting_grid


def replace(
    grid: Grid,
    replacee: Integer,
    replacer: Integer
) -> Grid:
    """ color substitution """
    return tuple(tuple(replacer if v == replacee else v for v in r) for r in grid)


def switch(
    grid: Grid,
    a: Integer,
    b: Integer
) -> Grid:
    """ color switching """
    return tuple(tuple(v if (v != a and v != b) else {a: b, b: a}[v] for v in r) for r in grid)


def center(
    patch: Patch
) -> IntegerTuple:
    """ center of the patch """
    return (uppermost(patch) + height(patch) // 2, leftmost(patch) + width(patch) // 2)


def position(
    a: Patch,
    b: Patch
) -> IntegerTuple:
    """ relative position between two patches """
    ia, ja = center(toindices(a))
    ib, jb = center(toindices(b))
    if ia == ib:
        return (0, 1 if ja < jb else -1)
    elif ja == jb:
        return (1 if ia < ib else -1, 0)
    elif ia < ib:
        return (1, 1 if ja < jb else -1)
    elif ia > ib:
        return (-1, 1 if ja < jb else -1)


def index(
    grid: Grid,
    loc: IntegerTuple
) -> Integer:
    """ color at location """
    i, j = loc
    h, w = len(grid), len(grid[0])
    if not (0 <= i < h and 0 <= j < w):
        return None
    return grid[loc[0]][loc[1]] 


def canvas(
    value: Integer,
    dimensions: IntegerTuple
) -> Grid:
    """ grid construction """
    return tuple(tuple(value for j in range(dimensions[1])) for i in range(dimensions[0]))


def corners(
    patch: Patch
) -> Indices:
    """ indices of corners """
    return frozenset({ulcorner(patch), urcorner(patch), llcorner(patch), lrcorner(patch)})


def connect(
    a: IntegerTuple,
    b: IntegerTuple
) -> Indices:
    """ line between two points """
    ai, aj = a
    bi, bj = b
    si = min(ai, bi)
    ei = max(ai, bi) + 1
    sj = min(aj, bj)
    ej = max(aj, bj) + 1
    if ai == bi:
        return frozenset((ai, j) for j in range(sj, ej))
    elif aj == bj:
        return frozenset((i, aj) for i in range(si, ei))
    elif bi - ai == bj - aj:
        return frozenset((i, j) for i, j in zip(range(si, ei), range(sj, ej)))
    elif bi - ai == aj - bj:
        return frozenset((i, j) for i, j in zip(range(si, ei), range(ej - 1, sj - 1, -1)))
    return frozenset()


def cover(
    grid: Grid,
    patch: Patch
) -> Grid:
    """ remove object from grid """
    return fill(grid, mostcolor(grid), toindices(patch))


def trim(
    grid: Grid
) -> Grid:
    """ trim border of grid """
    return tuple(r[1:-1] for r in grid[1:-1])


def move(
    grid: Grid,
    obj: Object,
    offset: IntegerTuple
) -> Grid:
    """ move object on grid """
    return paint(cover(grid, obj), shift(obj, offset))


def tophalf(
    grid: Grid
) -> Grid:
    """ upper half of grid """
    return grid[:len(grid) // 2]


def bottomhalf(
    grid: Grid
) -> Grid:
    """ lower half of grid """
    return grid[len(grid) // 2 + len(grid) % 2:]


def lefthalf(
    grid: Grid
) -> Grid:
    """ left half of grid """
    return rot270(tophalf(rot90(grid)))


def righthalf(
    grid: Grid
) -> Grid:
    """ right half of grid """
    return rot270(bottomhalf(rot90(grid)))


def vfrontier(
    location: IntegerTuple
) -> Indices:
    """ vertical frontier """
    return frozenset((i, location[1]) for i in range(30))


def hfrontier(
    location: IntegerTuple
) -> Indices:
    """ horizontal frontier """
    return frozenset((location[0], j) for j in range(30))


def backdrop(
    patch: Patch
) -> Indices:
    """ indices in bounding box of patch """
    if len(patch) == 0:
        return frozenset({})
    indices = toindices(patch)
    si, sj = ulcorner(indices)
    ei, ej = lrcorner(patch)
    return frozenset((i, j) for i in range(si, ei + 1) for j in range(sj, ej + 1))


def delta(
    patch: Patch
) -> Indices:
    """ indices in bounding box but not part of patch """
    if len(patch) == 0:
        return frozenset({})
    return backdrop(patch) - toindices(patch)


def gravitate(
    source: Patch,
    destination: Patch
) -> IntegerTuple:
    """ direction to move source until adjacent to destination """
    si, sj = center(source)
    di, dj = center(destination)
    i, j = 0, 0
    if vmatching(source, destination):
        i = 1 if si < di else -1
    else:
        j = 1 if sj < dj else -1
    gi, gj = i, j
    c = 0
    while not adjacent(source, destination) and c < 42:
        c += 1
        gi += i
        gj += j
        source = shift(source, (i, j))
    return (gi - i, gj - j)


def inbox(
    patch: Patch
) -> Indices:
    """ inbox for patch """
    ai, aj = uppermost(patch) + 1, leftmost(patch) + 1
    bi, bj = lowermost(patch) - 1, rightmost(patch) - 1
    si, sj = min(ai, bi), min(aj, bj)
    ei, ej = max(ai, bi), max(aj, bj)
    vlines = {(i, sj) for i in range(si, ei + 1)} | {(i, ej) for i in range(si, ei + 1)}
    hlines = {(si, j) for j in range(sj, ej + 1)} | {(ei, j) for j in range(sj, ej + 1)}
    return frozenset(vlines | hlines)


def outbox(
    patch: Patch
) -> Indices:
    """ outbox for patch """
    ai, aj = uppermost(patch) - 1, leftmost(patch) - 1
    bi, bj = lowermost(patch) + 1, rightmost(patch) + 1
    si, sj = min(ai, bi), min(aj, bj)
    ei, ej = max(ai, bi), max(aj, bj)
    vlines = {(i, sj) for i in range(si, ei + 1)} | {(i, ej) for i in range(si, ei + 1)}
    hlines = {(si, j) for j in range(sj, ej + 1)} | {(ei, j) for j in range(sj, ej + 1)}
    return frozenset(vlines | hlines)


def box(
    patch: Patch
) -> Indices:
    """ outline of patch """
    if len(patch) == 0:
        return patch
    ai, aj = ulcorner(patch)
    bi, bj = lrcorner(patch)
    si, sj = min(ai, bi), min(aj, bj)
    ei, ej = max(ai, bi), max(aj, bj)
    vlines = {(i, sj) for i in range(si, ei + 1)} | {(i, ej) for i in range(si, ei + 1)}
    hlines = {(si, j) for j in range(sj, ej + 1)} | {(ei, j) for j in range(sj, ej + 1)}
    return frozenset(vlines | hlines)


def shoot(
    start: IntegerTuple,
    direction: IntegerTuple
) -> Indices:
    """ line from starting point and direction """
    return connect(start, (start[0] + 42 * direction[0], start[1] + 42 * direction[1]))


def occurrences(
    grid: Grid,
    obj: Object
) -> Indices:
    """ locations of occurrences of object in grid """
    occs = set()
    normed = normalize(obj)
    h, w = len(grid), len(grid[0])
    oh, ow = shape(obj)
    h2, w2 = h - oh + 1, w - ow + 1
    for i in range(h2):
        for j in range(w2):
            occurs = True
            for v, (a, b) in shift(normed, (i, j)):
                if not (0 <= a < h and 0 <= b < w and grid[a][b] == v):
                    occurs = False
                    break
            if occurs:
                occs.add((i, j))
    return frozenset(occs)


def frontiers(
    grid: Grid
) -> Objects:
    """ set of frontiers """
    h, w = len(grid), len(grid[0])
    row_indices = tuple(i for i, r in enumerate(grid) if len(set(r)) == 1)
    column_indices = tuple(j for j, c in enumerate(dmirror(grid)) if len(set(c)) == 1)
    hfrontiers = frozenset({frozenset({(grid[i][j], (i, j)) for j in range(w)}) for i in row_indices})
    vfrontiers = frozenset({frozenset({(grid[i][j], (i, j)) for i in range(h)}) for j in column_indices})
    return hfrontiers | vfrontiers


def compress(
    grid: Grid
) -> Grid:
    """ removes frontiers from grid """
    ri = tuple(i for i, r in enumerate(grid) if len(set(r)) == 1)
    ci = tuple(j for j, c in enumerate(dmirror(grid)) if len(set(c)) == 1)
    return tuple(tuple(v for j, v in enumerate(r) if j not in ci) for i, r in enumerate(grid) if i not in ri)


def hperiod(
    obj: Object
) -> Integer:
    """ horizontal periodicity """
    normalized = normalize(obj)
    w = width(normalized)
    for p in range(1, w):
        offsetted = shift(normalized, (0, -p))
        pruned = frozenset({(c, (i, j)) for c, (i, j) in offsetted if j >= 0})
        if pruned.issubset(normalized):
            return p
    return w


def vperiod(
    obj: Object
) -> Integer:
    """ vertical periodicity """
    normalized = normalize(obj)
    h = height(normalized)
    for p in range(1, h):
        offsetted = shift(normalized, (-p, 0))
        pruned = frozenset({(c, (i, j)) for c, (i, j) in offsetted if i >= 0})
        if pruned.issubset(normalized):
            return p
    return h


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel/arc_types.py
from typing import (
    List,
    Union,
    Tuple,
    Any,
    Container,
    Callable,
    FrozenSet,
    Iterable
)

Boolean = bool
Integer = int
IntegerTuple = Tuple[Integer, Integer]
Numerical = Union[Integer, IntegerTuple]
IntegerSet = FrozenSet[Integer]
Grid = Tuple[Tuple[Integer]]
Cell = Tuple[Integer, IntegerTuple]
Object = FrozenSet[Cell]
Objects = FrozenSet[Object]
Indices = FrozenSet[IntegerTuple]
IndicesSet = FrozenSet[Indices]
Patch = Union[Object, Indices]
Element = Union[Object, Grid]
Piece = Union[Grid, Patch]
TupleTuple = Tuple[Tuple]
ContainerContainer = Container[Container]


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel/constants.py
F = False
T = True

ZERO = 0
ONE = 1
TWO = 2
THREE = 3
FOUR = 4
FIVE = 5
SIX = 6
SEVEN = 7
EIGHT = 8
NINE = 9
TEN = 10

NEG_ONE = -1
NEG_TWO = -2

DOWN = (1, 0)
RIGHT = (0, 1)
UP = (-1, 0)
LEFT = (0, -1)

ORIGIN = (0, 0)
UNITY = (1, 1)
NEG_UNITY = (-1, -1)
UP_RIGHT = (-1, 1)
DOWN_LEFT = (1, -1)

ZERO_BY_TWO = (0, 2)
TWO_BY_ZERO = (2, 0)
TWO_BY_TWO = (2, 2)
THREE_BY_THREE = (3, 3)


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel/solvers.py
from dsl import *
from constants import *


def solve_67a3c6ac(I):
    O = vmirror(I)
    return O


def solve_68b16354(I):
    O = hmirror(I)
    return O


def solve_74dd1130(I):
    O = dmirror(I)
    return O


def solve_3c9b0459(I):
    O = rot180(I)
    return O


def solve_6150a2bd(I):
    O = rot180(I)
    return O


def solve_9172f3a0(I):
    O = upscale(I, THREE)
    return O


def solve_9dfd6313(I):
    O = dmirror(I)
    return O


def solve_a416b8f3(I):
    O = hconcat(I, I)
    return O


def solve_b1948b0a(I):
    O = replace(I, SIX, TWO)
    return O


def solve_c59eb873(I):
    O = upscale(I, TWO)
    return O


def solve_c8f0f002(I):
    O = replace(I, SEVEN, FIVE)
    return O


def solve_d10ecb37(I):
    O = crop(I, ORIGIN, TWO_BY_TWO)
    return O


def solve_d511f180(I):
    O = switch(I, FIVE, EIGHT)
    return O


def solve_ed36ccf7(I):
    O = rot270(I)
    return O


def solve_4c4377d9(I):
    x1 = hmirror(I)
    O = vconcat(x1, I)
    return O


def solve_6d0aefbc(I):
    x1 = vmirror(I)
    O = hconcat(I, x1)
    return O


def solve_6fa7a44f(I):
    x1 = hmirror(I)
    O = vconcat(I, x1)
    return O


def solve_5614dbcf(I):
    x1 = replace(I, FIVE, ZERO)
    O = downscale(x1, THREE)
    return O


def solve_5bd6f4ac(I):
    x1 = tojvec(SIX)
    O = crop(I, x1, THREE_BY_THREE)
    return O


def solve_5582e5ca(I):
    x1 = mostcolor(I)
    O = canvas(x1, THREE_BY_THREE)
    return O


def solve_8be77c9e(I):
    x1 = hmirror(I)
    O = vconcat(I, x1)
    return O


def solve_c9e6f938(I):
    x1 = vmirror(I)
    O = hconcat(I, x1)
    return O


def solve_2dee498d(I):
    x1 = hsplit(I, THREE)
    O = first(x1)
    return O


def solve_1cf80156(I):
    x1 = objects(I, T, T, T)
    x2 = first(x1)
    O = subgrid(x2, I)
    return O


def solve_32597951(I):
    x1 = ofcolor(I, EIGHT)
    x2 = delta(x1)
    O = fill(I, THREE, x2)
    return O


def solve_25ff71a9(I):
    x1 = objects(I, T, T, T)
    x2 = first(x1)
    O = move(I, x2, DOWN)
    return O


def solve_0b148d64(I):
    x1 = partition(I)
    x2 = argmin(x1, size)
    O = subgrid(x2, I)
    return O


def solve_1f85a75f(I):
    x1 = objects(I, T, T, T)
    x2 = argmax(x1, size)
    O = subgrid(x2, I)
    return O


def solve_23b5c85d(I):
    x1 = objects(I, T, T, T)
    x2 = argmin(x1, size)
    O = subgrid(x2, I)
    return O


def solve_9ecd008a(I):
    x1 = vmirror(I)
    x2 = ofcolor(I, ZERO)
    O = subgrid(x2, x1)
    return O


def solve_ac0a08a4(I):
    x1 = colorcount(I, ZERO)
    x2 = subtract(NINE, x1)
    O = upscale(I, x2)
    return O


def solve_be94b721(I):
    x1 = objects(I, T, F, T)
    x2 = argmax(x1, size)
    O = subgrid(x2, I)
    return O


def solve_c909285e(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    O = subgrid(x2, I)
    return O


def solve_f25ffba3(I):
    x1 = bottomhalf(I)
    x2 = hmirror(x1)
    O = vconcat(x2, x1)
    return O


def solve_c1d99e64(I):
    x1 = frontiers(I)
    x2 = merge(x1)
    O = fill(I, TWO, x2)
    return O


def solve_b91ae062(I):
    x1 = numcolors(I)
    x2 = decrement(x1)
    O = upscale(I, x2)
    return O


def solve_3aa6fb7a(I):
    x1 = objects(I, T, F, T)
    x2 = mapply(corners, x1)
    O = underfill(I, ONE, x2)
    return O


def solve_7b7f7511(I):
    x1 = portrait(I)
    x2 = branch(x1, tophalf, lefthalf)
    O = x2(I)
    return O


def solve_4258a5f9(I):
    x1 = ofcolor(I, FIVE)
    x2 = mapply(neighbors, x1)
    O = fill(I, ONE, x2)
    return O


def solve_2dc579da(I):
    x1 = vsplit(I, TWO)
    x2 = rbind(hsplit, TWO)
    x3 = mapply(x2, x1)
    O = argmax(x3, numcolors)
    return O


def solve_28bf18c6(I):
    x1 = objects(I, T, T, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    O = hconcat(x3, x3)
    return O


def solve_3af2c5a8(I):
    x1 = vmirror(I)
    x2 = hconcat(I, x1)
    x3 = hmirror(x2)
    O = vconcat(x2, x3)
    return O


def solve_44f52bb0(I):
    x1 = vmirror(I)
    x2 = equality(x1, I)
    x3 = branch(x2, ONE, SEVEN)
    O = canvas(x3, UNITY)
    return O


def solve_62c24649(I):
    x1 = vmirror(I)
    x2 = hconcat(I, x1)
    x3 = hmirror(x2)
    O = vconcat(x2, x3)
    return O


def solve_67e8384a(I):
    x1 = vmirror(I)
    x2 = hconcat(I, x1)
    x3 = hmirror(x2)
    O = vconcat(x2, x3)
    return O


def solve_7468f01a(I):
    x1 = objects(I, F, T, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    O = vmirror(x3)
    return O


def solve_662c240a(I):
    x1 = vsplit(I, THREE)
    x2 = fork(equality, dmirror, identity)
    x3 = compose(flip, x2)
    O = extract(x1, x3)
    return O


def solve_42a50994(I):
    x1 = objects(I, T, T, T)
    x2 = sizefilter(x1, ONE)
    x3 = merge(x2)
    O = cover(I, x3)
    return O


def solve_56ff96f3(I):
    x1 = fgpartition(I)
    x2 = fork(recolor, color, backdrop)
    x3 = mapply(x2, x1)
    O = paint(I, x3)
    return O


def solve_50cb2852(I):
    x1 = objects(I, T, F, T)
    x2 = compose(backdrop, inbox)
    x3 = mapply(x2, x1)
    O = fill(I, EIGHT, x3)
    return O


def solve_4347f46a(I):
    x1 = objects(I, T, F, T)
    x2 = fork(difference, toindices, box)
    x3 = mapply(x2, x1)
    O = fill(I, ZERO, x3)
    return O


def solve_46f33fce(I):
    x1 = rot180(I)
    x2 = downscale(x1, TWO)
    x3 = rot180(x2)
    O = upscale(x3, FOUR)
    return O


def solve_a740d043(I):
    x1 = objects(I, T, T, T)
    x2 = merge(x1)
    x3 = subgrid(x2, I)
    O = replace(x3, ONE, ZERO)
    return O


def solve_a79310a0(I):
    x1 = objects(I, T, F, T)
    x2 = first(x1)
    x3 = move(I, x2, DOWN)
    O = replace(x3, EIGHT, TWO)
    return O


def solve_aabf363d(I):
    x1 = leastcolor(I)
    x2 = replace(I, x1, ZERO)
    x3 = leastcolor(x2)
    O = replace(x2, x3, x1)
    return O


def solve_ae4f1146(I):
    x1 = objects(I, F, F, T)
    x2 = rbind(colorcount, ONE)
    x3 = argmax(x1, x2)
    O = subgrid(x3, I)
    return O


def solve_b27ca6d3(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, TWO)
    x3 = mapply(outbox, x2)
    O = fill(I, THREE, x3)
    return O


def solve_ce22a75a(I):
    x1 = objects(I, T, F, T)
    x2 = apply(outbox, x1)
    x3 = mapply(backdrop, x2)
    O = fill(I, ONE, x3)
    return O


def solve_dc1df850(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, TWO)
    x3 = mapply(outbox, x2)
    O = fill(I, ONE, x3)
    return O


def solve_f25fbde4(I):
    x1 = objects(I, T, T, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    O = upscale(x3, TWO)
    return O


def solve_44d8ac46(I):
    x1 = objects(I, T, F, T)
    x2 = apply(delta, x1)
    x3 = mfilter(x2, square)
    O = fill(I, TWO, x3)
    return O


def solve_1e0a9b12(I):
    x1 = rot270(I)
    x2 = rbind(order, identity)
    x3 = apply(x2, x1)
    O = rot90(x3)
    return O


def solve_0d3d703e(I):
    x1 = switch(I, THREE, FOUR)
    x2 = switch(x1, EIGHT, NINE)
    x3 = switch(x2, TWO, SIX)
    O = switch(x3, ONE, FIVE)
    return O


def solve_3618c87e(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = merge(x2)
    O = move(I, x3, TWO_BY_ZERO)
    return O


def solve_1c786137(I):
    x1 = objects(I, T, F, F)
    x2 = argmax(x1, height)
    x3 = subgrid(x2, I)
    O = trim(x3)
    return O


def solve_8efcae92(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ONE)
    x3 = compose(size, delta)
    x4 = argmax(x2, x3)
    O = subgrid(x4, I)
    return O


def solve_445eab21(I):
    x1 = objects(I, T, F, T)
    x2 = fork(multiply, height, width)
    x3 = argmax(x1, x2)
    x4 = color(x3)
    O = canvas(x4, TWO_BY_TWO)
    return O


def solve_6f8cd79b(I):
    x1 = asindices(I)
    x2 = apply(initset, x1)
    x3 = rbind(bordering, I)
    x4 = mfilter(x2, x3)
    O = fill(I, EIGHT, x4)
    return O


def solve_2013d3e2(I):
    x1 = objects(I, F, T, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    x4 = lefthalf(x3)
    O = tophalf(x4)
    return O


def solve_41e4d17e(I):
    x1 = objects(I, T, F, T)
    x2 = fork(combine, vfrontier, hfrontier)
    x3 = compose(x2, center)
    x4 = mapply(x3, x1)
    O = underfill(I, SIX, x4)
    return O


def solve_9565186b(I):
    x1 = shape(I)
    x2 = objects(I, T, F, F)
    x3 = argmax(x2, size)
    x4 = canvas(FIVE, x1)
    O = paint(x4, x3)
    return O


def solve_aedd82e4(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, TWO)
    x3 = sizefilter(x2, ONE)
    x4 = merge(x3)
    O = fill(I, ONE, x4)
    return O


def solve_bb43febb(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, FIVE)
    x3 = compose(backdrop, inbox)
    x4 = mapply(x3, x2)
    O = fill(I, TWO, x4)
    return O


def solve_e98196ab(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = objects(x1, T, F, T)
    x4 = merge(x3)
    O = paint(x2, x4)
    return O


def solve_f76d97a5(I):
    x1 = palette(I)
    x2 = first(x1)
    x3 = last(x1)
    x4 = switch(I, x2, x3)
    O = replace(x4, FIVE, ZERO)
    return O


def solve_ce9e57f2(I):
    x1 = objects(I, T, F, T)
    x2 = fork(connect, ulcorner, centerofmass)
    x3 = mapply(x2, x1)
    x4 = fill(I, EIGHT, x3)
    O = switch(x4, EIGHT, TWO)
    return O


def solve_22eb0ac0(I):
    x1 = fgpartition(I)
    x2 = fork(recolor, color, backdrop)
    x3 = apply(x2, x1)
    x4 = mfilter(x3, hline)
    O = paint(I, x4)
    return O


def solve_9f236235(I):
    x1 = compress(I)
    x2 = objects(I, T, F, F)
    x3 = vmirror(x1)
    x4 = valmin(x2, width)
    O = downscale(x3, x4)
    return O


def solve_a699fb00(I):
    x1 = ofcolor(I, ONE)
    x2 = shift(x1, RIGHT)
    x3 = shift(x1, LEFT)
    x4 = intersection(x2, x3)
    O = fill(I, TWO, x4)
    return O


def solve_46442a0e(I):
    x1 = rot90(I)
    x2 = rot180(I)
    x3 = rot270(I)
    x4 = hconcat(I, x1)
    x5 = hconcat(x3, x2)
    O = vconcat(x4, x5)
    return O


def solve_7fe24cdd(I):
    x1 = rot90(I)
    x2 = rot180(I)
    x3 = rot270(I)
    x4 = hconcat(I, x1)
    x5 = hconcat(x3, x2)
    O = vconcat(x4, x5)
    return O


def solve_0ca9ddb6(I):
    x1 = ofcolor(I, ONE)
    x2 = ofcolor(I, TWO)
    x3 = mapply(dneighbors, x1)
    x4 = mapply(ineighbors, x2)
    x5 = fill(I, SEVEN, x3)
    O = fill(x5, FOUR, x4)
    return O


def solve_543a7ed5(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, SIX)
    x3 = mapply(outbox, x2)
    x4 = fill(I, THREE, x3)
    x5 = mapply(delta, x2)
    O = fill(x4, FOUR, x5)
    return O


def solve_0520fde7(I):
    x1 = vmirror(I)
    x2 = lefthalf(x1)
    x3 = righthalf(x1)
    x4 = vmirror(x3)
    x5 = cellwise(x2, x4, ZERO)
    O = replace(x5, ONE, TWO)
    return O


def solve_dae9d2b5(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = ofcolor(x1, FOUR)
    x4 = ofcolor(x2, THREE)
    x5 = combine(x3, x4)
    O = fill(x1, SIX, x5)
    return O


def solve_8d5021e8(I):
    x1 = vmirror(I)
    x2 = hconcat(x1, I)
    x3 = hmirror(x2)
    x4 = vconcat(x2, x3)
    x5 = vconcat(x4, x2)
    O = hmirror(x5)
    return O


def solve_928ad970(I):
    x1 = ofcolor(I, FIVE)
    x2 = subgrid(x1, I)
    x3 = trim(x2)
    x4 = leastcolor(x3)
    x5 = inbox(x1)
    O = fill(I, x4, x5)
    return O


def solve_b60334d2(I):
    x1 = ofcolor(I, FIVE)
    x2 = replace(I, FIVE, ZERO)
    x3 = mapply(dneighbors, x1)
    x4 = mapply(ineighbors, x1)
    x5 = fill(x2, ONE, x3)
    O = fill(x5, FIVE, x4)
    return O


def solve_b94a9452(I):
    x1 = objects(I, F, F, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    x4 = leastcolor(x3)
    x5 = mostcolor(x3)
    O = switch(x3, x4, x5)
    return O


def solve_d037b0a7(I):
    x1 = objects(I, T, F, T)
    x2 = rbind(shoot, DOWN)
    x3 = compose(x2, center)
    x4 = fork(recolor, color, x3)
    x5 = mapply(x4, x1)
    O = paint(I, x5)
    return O


def solve_d0f5fe59(I):
    x1 = objects(I, T, F, T)
    x2 = size(x1)
    x3 = astuple(x2, x2)
    x4 = canvas(ZERO, x3)
    x5 = shoot(ORIGIN, UNITY)
    O = fill(x4, EIGHT, x5)
    return O


def solve_e3497940(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = vmirror(x2)
    x4 = objects(x3, T, F, T)
    x5 = merge(x4)
    O = paint(x1, x5)
    return O


def solve_e9afcf9a(I):
    x1 = astuple(TWO, ONE)
    x2 = crop(I, ORIGIN, x1)
    x3 = hmirror(x2)
    x4 = hconcat(x2, x3)
    x5 = hconcat(x4, x4)
    O = hconcat(x5, x4)
    return O


def solve_48d8fb45(I):
    x1 = objects(I, T, T, T)
    x2 = matcher(size, ONE)
    x3 = extract(x1, x2)
    x4 = lbind(adjacent, x3)
    x5 = extract(x1, x4)
    O = subgrid(x5, I)
    return O


def solve_d406998b(I):
    x1 = vmirror(I)
    x2 = ofcolor(x1, FIVE)
    x3 = compose(even, last)
    x4 = sfilter(x2, x3)
    x5 = fill(x1, THREE, x4)
    O = vmirror(x5)
    return O


def solve_5117e062(I):
    x1 = objects(I, F, T, T)
    x2 = matcher(numcolors, TWO)
    x3 = extract(x1, x2)
    x4 = subgrid(x3, I)
    x5 = mostcolor(x3)
    O = replace(x4, EIGHT, x5)
    return O


def solve_3906de3d(I):
    x1 = rot270(I)
    x2 = rbind(order, identity)
    x3 = switch(x1, ONE, TWO)
    x4 = apply(x2, x3)
    x5 = switch(x4, ONE, TWO)
    O = cmirror(x5)
    return O


def solve_00d62c1b(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = rbind(bordering, I)
    x4 = compose(flip, x3)
    x5 = mfilter(x2, x4)
    O = fill(I, FOUR, x5)
    return O


def solve_7b6016b9(I):
    x1 = objects(I, T, F, F)
    x2 = rbind(bordering, I)
    x3 = compose(flip, x2)
    x4 = mfilter(x1, x3)
    x5 = fill(I, TWO, x4)
    O = replace(x5, ZERO, THREE)
    return O


def solve_67385a82(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, THREE)
    x3 = sizefilter(x2, ONE)
    x4 = difference(x2, x3)
    x5 = merge(x4)
    O = fill(I, EIGHT, x5)
    return O


def solve_a5313dff(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = rbind(bordering, I)
    x4 = compose(flip, x3)
    x5 = mfilter(x2, x4)
    O = fill(I, ONE, x5)
    return O


def solve_ea32f347(I):
    x1 = objects(I, T, F, T)
    x2 = replace(I, FIVE, FOUR)
    x3 = argmin(x1, size)
    x4 = argmax(x1, size)
    x5 = fill(x2, ONE, x4)
    O = fill(x5, TWO, x3)
    return O


def solve_d631b094(I):
    x1 = palette(I)
    x2 = other(x1, ZERO)
    x3 = ofcolor(I, x2)
    x4 = size(x3)
    x5 = astuple(ONE, x4)
    O = canvas(x2, x5)
    return O


def solve_10fcaaa3(I):
    x1 = leastcolor(I)
    x2 = hconcat(I, I)
    x3 = vconcat(x2, x2)
    x4 = ofcolor(x3, x1)
    x5 = mapply(ineighbors, x4)
    O = underfill(x3, EIGHT, x5)
    return O


def solve_007bbfb7(I):
    x1 = hupscale(I, THREE)
    x2 = vupscale(x1, THREE)
    x3 = hconcat(I, I)
    x4 = hconcat(x3, I)
    x5 = vconcat(x4, x4)
    x6 = vconcat(x5, x4)
    O = cellwise(x2, x6, ZERO)
    return O


def solve_496994bd(I):
    x1 = width(I)
    x2 = height(I)
    x3 = halve(x2)
    x4 = astuple(x3, x1)
    x5 = crop(I, ORIGIN, x4)
    x6 = hmirror(x5)
    O = vconcat(x5, x6)
    return O


def solve_1f876c06(I):
    x1 = fgpartition(I)
    x2 = compose(last, first)
    x3 = power(last, TWO)
    x4 = fork(connect, x2, x3)
    x5 = fork(recolor, color, x4)
    x6 = mapply(x5, x1)
    O = paint(I, x6)
    return O


def solve_05f2a901(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, TWO)
    x3 = first(x2)
    x4 = colorfilter(x1, EIGHT)
    x5 = first(x4)
    x6 = gravitate(x3, x5)
    O = move(I, x3, x6)
    return O


def solve_39a8645d(I):
    x1 = objects(I, T, T, T)
    x2 = totuple(x1)
    x3 = apply(color, x2)
    x4 = mostcommon(x3)
    x5 = matcher(color, x4)
    x6 = extract(x1, x5)
    O = subgrid(x6, I)
    return O


def solve_1b2d62fb(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = ofcolor(x1, ZERO)
    x4 = ofcolor(x2, ZERO)
    x5 = intersection(x3, x4)
    x6 = replace(x1, NINE, ZERO)
    O = fill(x6, EIGHT, x5)
    return O


def solve_90c28cc7(I):
    x1 = objects(I, F, F, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    x4 = dedupe(x3)
    x5 = rot90(x4)
    x6 = dedupe(x5)
    O = rot270(x6)
    return O


def solve_b6afb2da(I):
    x1 = objects(I, T, F, F)
    x2 = replace(I, FIVE, TWO)
    x3 = colorfilter(x1, FIVE)
    x4 = mapply(box, x3)
    x5 = fill(x2, FOUR, x4)
    x6 = mapply(corners, x3)
    O = fill(x5, ONE, x6)
    return O


def solve_b9b7f026(I):
    x1 = objects(I, T, F, F)
    x2 = argmin(x1, size)
    x3 = rbind(adjacent, x2)
    x4 = remove(x2, x1)
    x5 = extract(x4, x3)
    x6 = color(x5)
    O = canvas(x6, UNITY)
    return O


def solve_ba97ae07(I):
    x1 = objects(I, T, F, T)
    x2 = totuple(x1)
    x3 = apply(color, x2)
    x4 = mostcommon(x3)
    x5 = ofcolor(I, x4)
    x6 = backdrop(x5)
    O = fill(I, x4, x6)
    return O


def solve_c9f8e694(I):
    x1 = height(I)
    x2 = width(I)
    x3 = ofcolor(I, ZERO)
    x4 = astuple(x1, ONE)
    x5 = crop(I, ORIGIN, x4)
    x6 = hupscale(x5, x2)
    O = fill(x6, ZERO, x3)
    return O


def solve_d23f8c26(I):
    x1 = asindices(I)
    x2 = width(I)
    x3 = halve(x2)
    x4 = matcher(last, x3)
    x5 = compose(flip, x4)
    x6 = sfilter(x1, x5)
    O = fill(I, ZERO, x6)
    return O


def solve_d5d6de2d(I):
    x1 = objects(I, T, F, T)
    x2 = sfilter(x1, square)
    x3 = difference(x1, x2)
    x4 = compose(backdrop, inbox)
    x5 = mapply(x4, x3)
    x6 = replace(I, TWO, ZERO)
    O = fill(x6, THREE, x5)
    return O


def solve_dbc1a6ce(I):
    x1 = ofcolor(I, ONE)
    x2 = product(x1, x1)
    x3 = fork(connect, first, last)
    x4 = apply(x3, x2)
    x5 = fork(either, vline, hline)
    x6 = mfilter(x4, x5)
    O = underfill(I, EIGHT, x6)
    return O


def solve_ded97339(I):
    x1 = ofcolor(I, EIGHT)
    x2 = product(x1, x1)
    x3 = fork(connect, first, last)
    x4 = apply(x3, x2)
    x5 = fork(either, vline, hline)
    x6 = mfilter(x4, x5)
    O = underfill(I, EIGHT, x6)
    return O


def solve_ea786f4a(I):
    x1 = width(I)
    x2 = shoot(ORIGIN, UNITY)
    x3 = decrement(x1)
    x4 = tojvec(x3)
    x5 = shoot(x4, DOWN_LEFT)
    x6 = combine(x2, x5)
    O = fill(I, ZERO, x6)
    return O


def solve_08ed6ac7(I):
    x1 = objects(I, T, F, T)
    x2 = totuple(x1)
    x3 = order(x1, height)
    x4 = size(x2)
    x5 = interval(x4, ZERO, NEG_ONE)
    x6 = mpapply(recolor, x5, x3)
    O = paint(I, x6)
    return O


def solve_40853293(I):
    x1 = partition(I)
    x2 = fork(recolor, color, backdrop)
    x3 = apply(x2, x1)
    x4 = mfilter(x3, hline)
    x5 = mfilter(x3, vline)
    x6 = paint(I, x4)
    O = paint(x6, x5)
    return O


def solve_5521c0d9(I):
    x1 = objects(I, T, F, T)
    x2 = merge(x1)
    x3 = cover(I, x2)
    x4 = chain(toivec, invert, height)
    x5 = fork(shift, identity, x4)
    x6 = mapply(x5, x1)
    O = paint(x3, x6)
    return O


def solve_f8ff0b80(I):
    x1 = objects(I, T, T, T)
    x2 = order(x1, size)
    x3 = apply(color, x2)
    x4 = rbind(canvas, UNITY)
    x5 = apply(x4, x3)
    x6 = merge(x5)
    O = hmirror(x6)
    return O


def solve_85c4e7cd(I):
    x1 = objects(I, T, F, F)
    x2 = compose(invert, size)
    x3 = order(x1, size)
    x4 = order(x1, x2)
    x5 = apply(color, x4)
    x6 = mpapply(recolor, x5, x3)
    O = paint(I, x6)
    return O


def solve_d2abd087(I):
    x1 = objects(I, T, F, T)
    x2 = matcher(size, SIX)
    x3 = compose(flip, x2)
    x4 = mfilter(x1, x2)
    x5 = mfilter(x1, x3)
    x6 = fill(I, TWO, x4)
    O = fill(x6, ONE, x5)
    return O


def solve_017c7c7b(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = equality(x1, x2)
    x4 = crop(I, TWO_BY_ZERO, THREE_BY_THREE)
    x5 = branch(x3, x2, x4)
    x6 = vconcat(I, x5)
    O = replace(x6, ONE, TWO)
    return O


def solve_363442ee(I):
    x1 = ofcolor(I, ONE)
    x2 = crop(I, ORIGIN, THREE_BY_THREE)
    x3 = asobject(x2)
    x4 = lbind(shift, x3)
    x5 = compose(x4, decrement)
    x6 = mapply(x5, x1)
    O = paint(I, x6)
    return O


def solve_5168d44c(I):
    x1 = ofcolor(I, THREE)
    x2 = height(x1)
    x3 = equality(x2, ONE)
    x4 = branch(x3, ZERO_BY_TWO, TWO_BY_ZERO)
    x5 = ofcolor(I, TWO)
    x6 = recolor(TWO, x5)
    O = move(I, x6, x4)
    return O


def solve_e9614598(I):
    x1 = ofcolor(I, ONE)
    x2 = fork(add, first, last)
    x3 = x2(x1)
    x4 = halve(x3)
    x5 = dneighbors(x4)
    x6 = insert(x4, x5)
    O = fill(I, THREE, x6)
    return O


def solve_d9fac9be(I):
    x1 = palette(I)
    x2 = objects(I, T, F, T)
    x3 = argmax(x2, size)
    x4 = color(x3)
    x5 = remove(ZERO, x1)
    x6 = other(x5, x4)
    O = canvas(x6, UNITY)
    return O


def solve_e50d258f(I):
    x1 = width(I)
    x2 = astuple(NINE, x1)
    x3 = canvas(ZERO, x2)
    x4 = vconcat(I, x3)
    x5 = objects(x4, F, F, T)
    x6 = rbind(colorcount, TWO)
    x7 = argmax(x5, x6)
    O = subgrid(x7, I)
    return O


def solve_810b9b61(I):
    x1 = objects(I, T, T, T)
    x2 = apply(toindices, x1)
    x3 = fork(either, vline, hline)
    x4 = sfilter(x2, x3)
    x5 = difference(x2, x4)
    x6 = fork(equality, identity, box)
    x7 = mfilter(x5, x6)
    O = fill(I, THREE, x7)
    return O


def solve_54d82841(I):
    x1 = height(I)
    x2 = objects(I, T, F, T)
    x3 = compose(last, center)
    x4 = apply(x3, x2)
    x5 = decrement(x1)
    x6 = lbind(astuple, x5)
    x7 = apply(x6, x4)
    O = fill(I, FOUR, x7)
    return O


def solve_60b61512(I):
    x1 = objects(I, T, T, T)
    x2 = mapply(delta, x1)
    O = fill(I, SEVEN, x2)
    return O


def solve_25d8a9c8(I):
    x1 = asindices(I)
    x2 = objects(I, T, F, F)
    x3 = sizefilter(x2, THREE)
    x4 = mfilter(x3, hline)
    x5 = toindices(x4)
    x6 = difference(x1, x5)
    x7 = fill(I, FIVE, x5)
    O = fill(x7, ZERO, x6)
    return O


def solve_239be575(I):
    x1 = objects(I, F, T, T)
    x2 = lbind(contained, TWO)
    x3 = compose(x2, palette)
    x4 = sfilter(x1, x3)
    x5 = size(x4)
    x6 = greater(x5, ONE)
    x7 = branch(x6, ZERO, EIGHT)
    O = canvas(x7, UNITY)
    return O


def solve_67a423a3(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, T)
    x3 = colorfilter(x2, x1)
    x4 = merge(x3)
    x5 = delta(x4)
    x6 = first(x5)
    x7 = neighbors(x6)
    O = fill(I, FOUR, x7)
    return O


def solve_5c0a986e(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, ONE)
    x3 = lrcorner(x1)
    x4 = ulcorner(x2)
    x5 = shoot(x3, UNITY)
    x6 = shoot(x4, NEG_UNITY)
    x7 = fill(I, TWO, x5)
    O = fill(x7, ONE, x6)
    return O


def solve_6430c8c4(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = astuple(FOUR, FOUR)
    x4 = ofcolor(x1, ZERO)
    x5 = ofcolor(x2, ZERO)
    x6 = intersection(x4, x5)
    x7 = canvas(ZERO, x3)
    O = fill(x7, THREE, x6)
    return O


def solve_94f9d214(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = ofcolor(x1, ZERO)
    x4 = ofcolor(x2, ZERO)
    x5 = astuple(FOUR, FOUR)
    x6 = canvas(ZERO, x5)
    x7 = intersection(x3, x4)
    O = fill(x6, TWO, x7)
    return O


def solve_a1570a43(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, THREE)
    x3 = recolor(TWO, x1)
    x4 = ulcorner(x2)
    x5 = ulcorner(x1)
    x6 = subtract(x4, x5)
    x7 = increment(x6)
    O = move(I, x3, x7)
    return O


def solve_ce4f8723(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = ofcolor(x1, ZERO)
    x4 = ofcolor(x2, ZERO)
    x5 = intersection(x3, x4)
    x6 = astuple(FOUR, FOUR)
    x7 = canvas(THREE, x6)
    O = fill(x7, ZERO, x5)
    return O


def solve_d13f3404(I):
    x1 = objects(I, T, F, T)
    x2 = rbind(shoot, UNITY)
    x3 = compose(x2, center)
    x4 = fork(recolor, color, x3)
    x5 = mapply(x4, x1)
    x6 = astuple(SIX, SIX)
    x7 = canvas(ZERO, x6)
    O = paint(x7, x5)
    return O


def solve_dc433765(I):
    x1 = ofcolor(I, THREE)
    x2 = ofcolor(I, FOUR)
    x3 = first(x1)
    x4 = first(x2)
    x5 = subtract(x4, x3)
    x6 = sign(x5)
    x7 = recolor(THREE, x1)
    O = move(I, x7, x6)
    return O


def solve_f2829549(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = ofcolor(x1, ZERO)
    x4 = ofcolor(x2, ZERO)
    x5 = intersection(x3, x4)
    x6 = shape(x1)
    x7 = canvas(ZERO, x6)
    O = fill(x7, THREE, x5)
    return O


def solve_fafffa47(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = shape(x2)
    x4 = ofcolor(x1, ZERO)
    x5 = ofcolor(x2, ZERO)
    x6 = intersection(x4, x5)
    x7 = canvas(ZERO, x3)
    O = fill(x7, TWO, x6)
    return O


def solve_fcb5c309(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, T)
    x3 = colorfilter(x2, x1)
    x4 = difference(x2, x3)
    x5 = argmax(x4, size)
    x6 = color(x5)
    x7 = subgrid(x5, I)
    O = replace(x7, x6, x1)
    return O


def solve_ff805c23(I):
    x1 = hmirror(I)
    x2 = vmirror(I)
    x3 = ofcolor(I, ONE)
    x4 = subgrid(x3, x1)
    x5 = subgrid(x3, x2)
    x6 = palette(x4)
    x7 = contained(ONE, x6)
    O = branch(x7, x5, x4)
    return O


def solve_e76a88a6(I):
    x1 = objects(I, F, F, T)
    x2 = argmax(x1, numcolors)
    x3 = normalize(x2)
    x4 = remove(x2, x1)
    x5 = apply(ulcorner, x4)
    x6 = lbind(shift, x3)
    x7 = mapply(x6, x5)
    O = paint(I, x7)
    return O


def solve_7c008303(I):
    x1 = ofcolor(I, THREE)
    x2 = subgrid(x1, I)
    x3 = ofcolor(x2, ZERO)
    x4 = replace(I, THREE, ZERO)
    x5 = replace(x4, EIGHT, ZERO)
    x6 = compress(x5)
    x7 = upscale(x6, THREE)
    O = fill(x7, ZERO, x3)
    return O


def solve_7f4411dc(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = rbind(difference, x2)
    x4 = rbind(greater, TWO)
    x5 = chain(x4, size, x3)
    x6 = compose(x5, dneighbors)
    x7 = sfilter(x2, x6)
    O = fill(I, ZERO, x7)
    return O


def solve_b230c067(I):
    x1 = objects(I, T, T, T)
    x2 = totuple(x1)
    x3 = apply(normalize, x2)
    x4 = leastcommon(x3)
    x5 = matcher(normalize, x4)
    x6 = extract(x1, x5)
    x7 = replace(I, EIGHT, ONE)
    O = fill(x7, TWO, x6)
    return O


def solve_e8593010(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = sizefilter(x1, TWO)
    x4 = merge(x2)
    x5 = fill(I, THREE, x4)
    x6 = merge(x3)
    x7 = fill(x5, TWO, x6)
    O = replace(x7, ZERO, ONE)
    return O


def solve_6d75e8bb(I):
    x1 = objects(I, T, F, T)
    x2 = first(x1)
    x3 = ulcorner(x2)
    x4 = subgrid(x2, I)
    x5 = replace(x4, ZERO, TWO)
    x6 = asobject(x5)
    x7 = shift(x6, x3)
    O = paint(I, x7)
    return O


def solve_3f7978a0(I):
    x1 = fgpartition(I)
    x2 = matcher(color, FIVE)
    x3 = extract(x1, x2)
    x4 = ulcorner(x3)
    x5 = subtract(x4, DOWN)
    x6 = shape(x3)
    x7 = add(x6, TWO_BY_ZERO)
    O = crop(I, x5, x7)
    return O


def solve_1190e5a7(I):
    x1 = mostcolor(I)
    x2 = frontiers(I)
    x3 = sfilter(x2, vline)
    x4 = difference(x2, x3)
    x5 = astuple(x4, x3)
    x6 = apply(size, x5)
    x7 = increment(x6)
    O = canvas(x1, x7)
    return O


def solve_6e02f1e3(I):
    x1 = numcolors(I)
    x2 = canvas(ZERO, THREE_BY_THREE)
    x3 = equality(x1, THREE)
    x4 = equality(x1, TWO)
    x5 = branch(x3, TWO_BY_ZERO, ORIGIN)
    x6 = branch(x4, TWO_BY_TWO, ZERO_BY_TWO)
    x7 = connect(x5, x6)
    O = fill(x2, FIVE, x7)
    return O


def solve_a61f2674(I):
    x1 = objects(I, T, F, T)
    x2 = argmax(x1, size)
    x3 = argmin(x1, size)
    x4 = replace(I, FIVE, ZERO)
    x5 = recolor(ONE, x2)
    x6 = recolor(TWO, x3)
    x7 = combine(x5, x6)
    O = paint(x4, x7)
    return O


def solve_fcc82909(I):
    x1 = objects(I, F, T, T)
    x2 = rbind(add, DOWN)
    x3 = compose(x2, llcorner)
    x4 = compose(toivec, numcolors)
    x5 = fork(add, lrcorner, x4)
    x6 = fork(astuple, x3, x5)
    x7 = compose(box, x6)
    x8 = mapply(x7, x1)
    O = fill(I, THREE, x8)
    return O


def solve_72ca375d(I):
    x1 = objects(I, T, T, T)
    x2 = totuple(x1)
    x3 = rbind(subgrid, I)
    x4 = apply(x3, x2)
    x5 = apply(vmirror, x4)
    x6 = papply(equality, x4, x5)
    x7 = pair(x4, x6)
    x8 = extract(x7, last)
    O = first(x8)
    return O


def solve_253bf280(I):
    x1 = ofcolor(I, EIGHT)
    x2 = prapply(connect, x1, x1)
    x3 = rbind(greater, ONE)
    x4 = compose(x3, size)
    x5 = sfilter(x2, x4)
    x6 = fork(either, vline, hline)
    x7 = mfilter(x5, x6)
    x8 = fill(I, THREE, x7)
    O = fill(x8, EIGHT, x1)
    return O


def solve_694f12f3(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, FOUR)
    x3 = compose(backdrop, inbox)
    x4 = argmin(x2, size)
    x5 = argmax(x2, size)
    x6 = x3(x4)
    x7 = x3(x5)
    x8 = fill(I, ONE, x6)
    O = fill(x8, TWO, x7)
    return O


def solve_1f642eb9(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = difference(x1, x2)
    x4 = first(x3)
    x5 = rbind(gravitate, x4)
    x6 = compose(crement, x5)
    x7 = fork(shift, identity, x6)
    x8 = mapply(x7, x2)
    O = paint(I, x8)
    return O


def solve_31aa019c(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = first(x2)
    x4 = neighbors(x3)
    x5 = astuple(TEN, TEN)
    x6 = canvas(ZERO, x5)
    x7 = initset(x3)
    x8 = fill(x6, x1, x7)
    O = fill(x8, TWO, x4)
    return O


def solve_27a28665(I):
    x1 = objects(I, T, F, F)
    x2 = valmax(x1, size)
    x3 = equality(x2, ONE)
    x4 = equality(x2, FOUR)
    x5 = equality(x2, FIVE)
    x6 = branch(x3, TWO, ONE)
    x7 = branch(x4, THREE, x6)
    x8 = branch(x5, SIX, x7)
    O = canvas(x8, UNITY)
    return O


def solve_7ddcd7ec(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = difference(x1, x2)
    x4 = first(x3)
    x5 = color(x4)
    x6 = lbind(position, x4)
    x7 = fork(shoot, center, x6)
    x8 = mapply(x7, x2)
    O = fill(I, x5, x8)
    return O


def solve_3bd67248(I):
    x1 = height(I)
    x2 = decrement(x1)
    x3 = decrement(x2)
    x4 = astuple(x3, ONE)
    x5 = astuple(x2, ONE)
    x6 = shoot(x4, UP_RIGHT)
    x7 = shoot(x5, RIGHT)
    x8 = fill(I, TWO, x6)
    O = fill(x8, FOUR, x7)
    return O


def solve_73251a56(I):
    x1 = dmirror(I)
    x2 = papply(pair, I, x1)
    x3 = lbind(apply, maximum)
    x4 = apply(x3, x2)
    x5 = mostcolor(x4)
    x6 = replace(x4, ZERO, x5)
    x7 = index(x6, ORIGIN)
    x8 = shoot(ORIGIN, UNITY)
    O = fill(x6, x7, x8)
    return O


def solve_25d487eb(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, T)
    x3 = ofcolor(I, x1)
    x4 = center(x3)
    x5 = merge(x2)
    x6 = center(x5)
    x7 = subtract(x6, x4)
    x8 = shoot(x4, x7)
    O = underfill(I, x1, x8)
    return O


def solve_8f2ea7aa(I):
    x1 = objects(I, T, F, T)
    x2 = merge(x1)
    x3 = subgrid(x2, I)
    x4 = upscale(x3, THREE)
    x5 = hconcat(x3, x3)
    x6 = hconcat(x5, x3)
    x7 = vconcat(x6, x6)
    x8 = vconcat(x7, x6)
    O = cellwise(x4, x8, ZERO)
    return O


def solve_b8825c91(I):
    x1 = replace(I, FOUR, ZERO)
    x2 = dmirror(x1)
    x3 = papply(pair, x1, x2)
    x4 = lbind(apply, maximum)
    x5 = apply(x4, x3)
    x6 = cmirror(x5)
    x7 = papply(pair, x5, x6)
    x8 = apply(x4, x7)
    O = cmirror(x8)
    return O


def solve_cce03e0d(I):
    x1 = upscale(I, THREE)
    x2 = hconcat(I, I)
    x3 = hconcat(x2, I)
    x4 = vconcat(x3, x3)
    x5 = vconcat(x4, x3)
    x6 = ofcolor(x1, ZERO)
    x7 = ofcolor(x1, ONE)
    x8 = combine(x6, x7)
    O = fill(x5, ZERO, x8)
    return O


def solve_d364b489(I):
    x1 = ofcolor(I, ONE)
    x2 = shift(x1, DOWN)
    x3 = fill(I, EIGHT, x2)
    x4 = shift(x1, UP)
    x5 = fill(x3, TWO, x4)
    x6 = shift(x1, RIGHT)
    x7 = fill(x5, SIX, x6)
    x8 = shift(x1, LEFT)
    O = fill(x7, SEVEN, x8)
    return O


def solve_a5f85a15(I):
    x1 = objects(I, T, T, T)
    x2 = interval(ONE, NINE, ONE)
    x3 = apply(double, x2)
    x4 = apply(decrement, x3)
    x5 = papply(astuple, x4, x4)
    x6 = apply(ulcorner, x1)
    x7 = lbind(shift, x5)
    x8 = mapply(x7, x6)
    O = fill(I, FOUR, x8)
    return O


def solve_3ac3eb23(I):
    x1 = objects(I, T, F, T)
    x2 = chain(ineighbors, last, first)
    x3 = fork(recolor, color, x2)
    x4 = mapply(x3, x1)
    x5 = paint(I, x4)
    x6 = vsplit(x5, THREE)
    x7 = first(x6)
    x8 = vconcat(x7, x7)
    O = vconcat(x7, x8)
    return O


def solve_444801d8(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, ONE)
    x3 = rbind(toobject, I)
    x4 = chain(leastcolor, x3, delta)
    x5 = rbind(shift, UP)
    x6 = compose(x5, backdrop)
    x7 = fork(recolor, x4, x6)
    x8 = mapply(x7, x2)
    O = underpaint(I, x8)
    return O


def solve_22168020(I):
    x1 = palette(I)
    x2 = remove(ZERO, x1)
    x3 = lbind(ofcolor, I)
    x4 = lbind(prapply, connect)
    x5 = fork(x4, x3, x3)
    x6 = compose(merge, x5)
    x7 = fork(recolor, identity, x6)
    x8 = mapply(x7, x2)
    O = paint(I, x8)
    return O


def solve_6e82a1ae(I):
    x1 = objects(I, T, F, T)
    x2 = lbind(sizefilter, x1)
    x3 = compose(merge, x2)
    x4 = x3(TWO)
    x5 = x3(THREE)
    x6 = x3(FOUR)
    x7 = fill(I, THREE, x4)
    x8 = fill(x7, TWO, x5)
    O = fill(x8, ONE, x6)
    return O


def solve_b2862040(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, NINE)
    x3 = colorfilter(x1, ONE)
    x4 = rbind(bordering, I)
    x5 = compose(flip, x4)
    x6 = mfilter(x2, x5)
    x7 = rbind(adjacent, x6)
    x8 = mfilter(x3, x7)
    O = fill(I, EIGHT, x8)
    return O


def solve_868de0fa(I):
    x1 = objects(I, T, F, F)
    x2 = sfilter(x1, square)
    x3 = compose(even, height)
    x4 = sfilter(x2, x3)
    x5 = difference(x2, x4)
    x6 = merge(x4)
    x7 = merge(x5)
    x8 = fill(I, TWO, x6)
    O = fill(x8, SEVEN, x7)
    return O


def solve_681b3aeb(I):
    x1 = rot270(I)
    x2 = objects(x1, T, F, T)
    x3 = argmax(x2, size)
    x4 = argmin(x2, size)
    x5 = color(x4)
    x6 = canvas(x5, THREE_BY_THREE)
    x7 = normalize(x3)
    x8 = paint(x6, x7)
    O = rot90(x8)
    return O


def solve_8e5a5113(I):
    x1 = crop(I, ORIGIN, THREE_BY_THREE)
    x2 = rot90(x1)
    x3 = rot180(x1)
    x4 = astuple(x2, x3)
    x5 = astuple(FOUR, EIGHT)
    x6 = apply(tojvec, x5)
    x7 = apply(asobject, x4)
    x8 = mpapply(shift, x7, x6)
    O = paint(I, x8)
    return O


def solve_025d127b(I):
    x1 = objects(I, T, F, T)
    x2 = apply(color, x1)
    x3 = merge(x1)
    x4 = lbind(colorfilter, x1)
    x5 = rbind(argmax, rightmost)
    x6 = compose(x5, x4)
    x7 = mapply(x6, x2)
    x8 = difference(x3, x7)
    O = move(I, x8, RIGHT)
    return O


def solve_2281f1f4(I):
    x1 = ofcolor(I, FIVE)
    x2 = product(x1, x1)
    x3 = power(first, TWO)
    x4 = power(last, TWO)
    x5 = fork(astuple, x3, x4)
    x6 = apply(x5, x2)
    x7 = urcorner(x1)
    x8 = remove(x7, x6)
    O = underfill(I, TWO, x8)
    return O


def solve_cf98881b(I):
    x1 = hsplit(I, THREE)
    x2 = first(x1)
    x3 = remove(x2, x1)
    x4 = first(x3)
    x5 = last(x3)
    x6 = ofcolor(x4, NINE)
    x7 = ofcolor(x2, FOUR)
    x8 = fill(x5, NINE, x6)
    O = fill(x8, FOUR, x7)
    return O


def solve_d4f3cd78(I):
    x1 = ofcolor(I, FIVE)
    x2 = delta(x1)
    x3 = fill(I, EIGHT, x2)
    x4 = box(x1)
    x5 = difference(x4, x1)
    x6 = position(x4, x5)
    x7 = first(x5)
    x8 = shoot(x7, x6)
    O = fill(x3, EIGHT, x8)
    return O


def solve_bda2d7a6(I):
    x1 = partition(I)
    x2 = order(x1, size)
    x3 = apply(color, x2)
    x4 = last(x2)
    x5 = remove(x4, x2)
    x6 = repeat(x4, ONE)
    x7 = combine(x6, x5)
    x8 = mpapply(recolor, x3, x7)
    O = paint(I, x8)
    return O


def solve_137eaa0f(I):
    x1 = objects(I, F, T, T)
    x2 = matcher(first, FIVE)
    x3 = rbind(sfilter, x2)
    x4 = chain(invert, center, x3)
    x5 = fork(shift, identity, x4)
    x6 = canvas(ZERO, THREE_BY_THREE)
    x7 = mapply(x5, x1)
    x8 = shift(x7, UNITY)
    O = paint(x6, x8)
    return O


def solve_6455b5f5(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = argmax(x1, size)
    x4 = valmin(x1, size)
    x5 = sizefilter(x2, x4)
    x6 = recolor(ONE, x3)
    x7 = merge(x5)
    x8 = paint(I, x6)
    O = fill(x8, EIGHT, x7)
    return O


def solve_b8cdaf2b(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = shift(x2, UP)
    x4 = ulcorner(x3)
    x5 = urcorner(x3)
    x6 = shoot(x4, NEG_UNITY)
    x7 = shoot(x5, UP_RIGHT)
    x8 = combine(x6, x7)
    O = underfill(I, x1, x8)
    return O


def solve_bd4472b8(I):
    x1 = width(I)
    x2 = astuple(TWO, x1)
    x3 = crop(I, ORIGIN, x2)
    x4 = tophalf(x3)
    x5 = dmirror(x4)
    x6 = hupscale(x5, x1)
    x7 = repeat(x6, TWO)
    x8 = merge(x7)
    O = vconcat(x3, x8)
    return O


def solve_4be741c5(I):
    x1 = portrait(I)
    x2 = branch(x1, dmirror, identity)
    x3 = branch(x1, height, width)
    x4 = x3(I)
    x5 = astuple(ONE, x4)
    x6 = x2(I)
    x7 = crop(x6, ORIGIN, x5)
    x8 = apply(dedupe, x7)
    O = x2(x8)
    return O


def solve_bbc9ae5d(I):
    x1 = width(I)
    x2 = palette(I)
    x3 = halve(x1)
    x4 = vupscale(I, x3)
    x5 = rbind(shoot, UNITY)
    x6 = other(x2, ZERO)
    x7 = ofcolor(x4, x6)
    x8 = mapply(x5, x7)
    O = fill(x4, x6, x8)
    return O


def solve_d90796e8(I):
    x1 = objects(I, F, F, T)
    x2 = sizefilter(x1, TWO)
    x3 = lbind(contained, TWO)
    x4 = compose(x3, palette)
    x5 = mfilter(x2, x4)
    x6 = cover(I, x5)
    x7 = matcher(first, THREE)
    x8 = sfilter(x5, x7)
    O = fill(x6, EIGHT, x8)
    return O


def solve_2c608aff(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, T)
    x3 = argmax(x2, size)
    x4 = toindices(x3)
    x5 = ofcolor(I, x1)
    x6 = prapply(connect, x4, x5)
    x7 = fork(either, vline, hline)
    x8 = mfilter(x6, x7)
    O = underfill(I, x1, x8)
    return O


def solve_f8b3ba0a(I):
    x1 = compress(I)
    x2 = astuple(THREE, ONE)
    x3 = palette(x1)
    x4 = lbind(colorcount, x1)
    x5 = compose(invert, x4)
    x6 = order(x3, x5)
    x7 = rbind(canvas, UNITY)
    x8 = apply(x7, x6)
    x9 = merge(x8)
    O = crop(x9, DOWN, x2)
    return O


def solve_80af3007(I):
    x1 = objects(I, T, T, T)
    x2 = first(x1)
    x3 = subgrid(x2, I)
    x4 = upscale(x3, THREE)
    x5 = hconcat(x3, x3)
    x6 = hconcat(x5, x3)
    x7 = vconcat(x6, x6)
    x8 = vconcat(x7, x6)
    x9 = cellwise(x4, x8, ZERO)
    O = downscale(x9, THREE)
    return O


def solve_83302e8f(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = sfilter(x2, square)
    x4 = difference(x2, x3)
    x5 = merge(x3)
    x6 = recolor(THREE, x5)
    x7 = merge(x4)
    x8 = recolor(FOUR, x7)
    x9 = paint(I, x6)
    O = paint(x9, x8)
    return O


def solve_1fad071e(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, ONE)
    x3 = sizefilter(x2, FOUR)
    x4 = size(x3)
    x5 = subtract(FIVE, x4)
    x6 = astuple(ONE, x4)
    x7 = canvas(ONE, x6)
    x8 = astuple(ONE, x5)
    x9 = canvas(ZERO, x8)
    O = hconcat(x7, x9)
    return O


def solve_11852cab(I):
    x1 = objects(I, T, T, T)
    x2 = merge(x1)
    x3 = hmirror(x2)
    x4 = vmirror(x2)
    x5 = dmirror(x2)
    x6 = cmirror(x2)
    x7 = paint(I, x3)
    x8 = paint(x7, x4)
    x9 = paint(x8, x5)
    O = paint(x9, x6)
    return O


def solve_3428a4f5(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = astuple(SIX, FIVE)
    x4 = ofcolor(x1, TWO)
    x5 = ofcolor(x2, TWO)
    x6 = combine(x4, x5)
    x7 = intersection(x4, x5)
    x8 = difference(x6, x7)
    x9 = canvas(ZERO, x3)
    O = fill(x9, THREE, x8)
    return O


def solve_178fcbfb(I):
    x1 = objects(I, T, F, T)
    x2 = ofcolor(I, TWO)
    x3 = mapply(vfrontier, x2)
    x4 = fill(I, TWO, x3)
    x5 = colorfilter(x1, TWO)
    x6 = difference(x1, x5)
    x7 = compose(hfrontier, center)
    x8 = fork(recolor, color, x7)
    x9 = mapply(x8, x6)
    O = paint(x4, x9)
    return O


def solve_3de23699(I):
    x1 = fgpartition(I)
    x2 = sizefilter(x1, FOUR)
    x3 = first(x2)
    x4 = difference(x1, x2)
    x5 = first(x4)
    x6 = color(x3)
    x7 = color(x5)
    x8 = subgrid(x3, I)
    x9 = trim(x8)
    O = replace(x9, x7, x6)
    return O


def solve_54d9e175(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = compose(neighbors, center)
    x4 = fork(recolor, color, x3)
    x5 = mapply(x4, x2)
    x6 = paint(I, x5)
    x7 = replace(x6, ONE, SIX)
    x8 = replace(x7, TWO, SEVEN)
    x9 = replace(x8, THREE, EIGHT)
    O = replace(x9, FOUR, NINE)
    return O


def solve_5ad4f10b(I):
    x1 = objects(I, T, T, T)
    x2 = argmax(x1, size)
    x3 = color(x2)
    x4 = subgrid(x2, I)
    x5 = leastcolor(x4)
    x6 = replace(x4, x5, ZERO)
    x7 = replace(x6, x3, x5)
    x8 = height(x7)
    x9 = divide(x8, THREE)
    O = downscale(x7, x9)
    return O


def solve_623ea044(I):
    x1 = objects(I, T, F, T)
    x2 = first(x1)
    x3 = center(x2)
    x4 = color(x2)
    x5 = astuple(UNITY, NEG_UNITY)
    x6 = astuple(UP_RIGHT, DOWN_LEFT)
    x7 = combine(x5, x6)
    x8 = lbind(shoot, x3)
    x9 = mapply(x8, x7)
    O = fill(I, x4, x9)
    return O


def solve_6b9890af(I):
    x1 = objects(I, T, T, T)
    x2 = ofcolor(I, TWO)
    x3 = argmin(x1, size)
    x4 = subgrid(x2, I)
    x5 = width(x4)
    x6 = divide(x5, THREE)
    x7 = upscale(x3, x6)
    x8 = normalize(x7)
    x9 = shift(x8, UNITY)
    O = paint(x4, x9)
    return O


def solve_794b24be(I):
    x1 = ofcolor(I, ONE)
    x2 = size(x1)
    x3 = decrement(x2)
    x4 = canvas(ZERO, THREE_BY_THREE)
    x5 = tojvec(x3)
    x6 = connect(ORIGIN, x5)
    x7 = equality(x2, FOUR)
    x8 = insert(UNITY, x6)
    x9 = branch(x7, x8, x6)
    O = fill(x4, TWO, x9)
    return O


def solve_88a10436(I):
    x1 = objects(I, F, F, T)
    x2 = colorfilter(x1, FIVE)
    x3 = first(x2)
    x4 = center(x3)
    x5 = difference(x1, x2)
    x6 = first(x5)
    x7 = normalize(x6)
    x8 = shift(x7, x4)
    x9 = shift(x8, NEG_UNITY)
    O = paint(I, x9)
    return O


def solve_88a62173(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = tophalf(x1)
    x4 = tophalf(x2)
    x5 = bottomhalf(x1)
    x6 = bottomhalf(x2)
    x7 = astuple(x3, x4)
    x8 = astuple(x5, x6)
    x9 = combine(x7, x8)
    O = leastcommon(x9)
    return O


def solve_890034e9(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = inbox(x2)
    x4 = recolor(ZERO, x3)
    x5 = occurrences(I, x4)
    x6 = normalize(x2)
    x7 = shift(x6, NEG_UNITY)
    x8 = lbind(shift, x7)
    x9 = mapply(x8, x5)
    O = fill(I, x1, x9)
    return O


def solve_99b1bc43(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = ofcolor(x1, ZERO)
    x4 = ofcolor(x2, ZERO)
    x5 = combine(x3, x4)
    x6 = intersection(x3, x4)
    x7 = difference(x5, x6)
    x8 = shape(x1)
    x9 = canvas(ZERO, x8)
    O = fill(x9, THREE, x7)
    return O


def solve_a9f96cdd(I):
    x1 = ofcolor(I, TWO)
    x2 = replace(I, TWO, ZERO)
    x3 = shift(x1, NEG_UNITY)
    x4 = fill(x2, THREE, x3)
    x5 = shift(x1, UP_RIGHT)
    x6 = fill(x4, SIX, x5)
    x7 = shift(x1, DOWN_LEFT)
    x8 = fill(x6, EIGHT, x7)
    x9 = shift(x1, UNITY)
    O = fill(x8, SEVEN, x9)
    return O


def solve_af902bf9(I):
    x1 = ofcolor(I, FOUR)
    x2 = prapply(connect, x1, x1)
    x3 = fork(either, vline, hline)
    x4 = mfilter(x2, x3)
    x5 = underfill(I, NEG_ONE, x4)
    x6 = objects(x5, F, F, T)
    x7 = compose(backdrop, inbox)
    x8 = mapply(x7, x6)
    x9 = fill(x5, TWO, x8)
    O = replace(x9, NEG_ONE, ZERO)
    return O


def solve_b548a754(I):
    x1 = objects(I, T, F, T)
    x2 = replace(I, EIGHT, ZERO)
    x3 = leastcolor(x2)
    x4 = replace(x2, x3, ZERO)
    x5 = leastcolor(x4)
    x6 = merge(x1)
    x7 = backdrop(x6)
    x8 = box(x6)
    x9 = fill(I, x3, x7)
    O = fill(x9, x5, x8)
    return O


def solve_bdad9b1f(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, EIGHT)
    x3 = center(x1)
    x4 = center(x2)
    x5 = hfrontier(x3)
    x6 = vfrontier(x4)
    x7 = intersection(x5, x6)
    x8 = fill(I, TWO, x5)
    x9 = fill(x8, EIGHT, x6)
    O = fill(x9, FOUR, x7)
    return O


def solve_c3e719e8(I):
    x1 = mostcolor(I)
    x2 = hconcat(I, I)
    x3 = upscale(I, THREE)
    x4 = ofcolor(x3, x1)
    x5 = asindices(x3)
    x6 = difference(x5, x4)
    x7 = hconcat(x2, I)
    x8 = vconcat(x7, x7)
    x9 = vconcat(x8, x7)
    O = fill(x9, ZERO, x6)
    return O


def solve_de1cd16c(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, F)
    x3 = sizefilter(x2, ONE)
    x4 = difference(x2, x3)
    x5 = rbind(subgrid, I)
    x6 = apply(x5, x4)
    x7 = rbind(colorcount, x1)
    x8 = argmax(x6, x7)
    x9 = mostcolor(x8)
    O = canvas(x9, UNITY)
    return O


def solve_d8c310e9(I):
    x1 = objects(I, F, F, T)
    x2 = first(x1)
    x3 = hperiod(x2)
    x4 = multiply(x3, THREE)
    x5 = tojvec(x3)
    x6 = tojvec(x4)
    x7 = shift(x2, x5)
    x8 = shift(x2, x6)
    x9 = paint(I, x7)
    O = paint(x9, x8)
    return O


def solve_a3325580(I):
    x1 = objects(I, T, F, T)
    x2 = valmax(x1, size)
    x3 = sizefilter(x1, x2)
    x4 = order(x3, leftmost)
    x5 = apply(color, x4)
    x6 = astuple(ONE, x2)
    x7 = rbind(canvas, x6)
    x8 = apply(x7, x5)
    x9 = merge(x8)
    O = dmirror(x9)
    return O


def solve_8eb1be9a(I):
    x1 = objects(I, T, T, T)
    x2 = first(x1)
    x3 = interval(NEG_TWO, FOUR, ONE)
    x4 = lbind(shift, x2)
    x5 = height(x2)
    x6 = rbind(multiply, x5)
    x7 = apply(x6, x3)
    x8 = apply(toivec, x7)
    x9 = mapply(x4, x8)
    O = paint(I, x9)
    return O


def solve_321b1fc6(I):
    x1 = objects(I, F, F, T)
    x2 = colorfilter(x1, EIGHT)
    x3 = difference(x1, x2)
    x4 = first(x3)
    x5 = cover(I, x4)
    x6 = normalize(x4)
    x7 = lbind(shift, x6)
    x8 = apply(ulcorner, x2)
    x9 = mapply(x7, x8)
    O = paint(x5, x9)
    return O


def solve_1caeab9d(I):
    x1 = objects(I, T, T, T)
    x2 = ofcolor(I, ONE)
    x3 = lowermost(x2)
    x4 = lbind(subtract, x3)
    x5 = chain(toivec, x4, lowermost)
    x6 = fork(shift, identity, x5)
    x7 = merge(x1)
    x8 = cover(I, x7)
    x9 = mapply(x6, x1)
    O = paint(x8, x9)
    return O


def solve_77fdfe62(I):
    x1 = ofcolor(I, EIGHT)
    x2 = subgrid(x1, I)
    x3 = replace(I, EIGHT, ZERO)
    x4 = replace(x3, ONE, ZERO)
    x5 = compress(x4)
    x6 = width(x2)
    x7 = halve(x6)
    x8 = upscale(x5, x7)
    x9 = ofcolor(x2, ZERO)
    O = fill(x8, ZERO, x9)
    return O


def solve_c0f76784(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = sfilter(x2, square)
    x4 = sizefilter(x3, ONE)
    x5 = merge(x4)
    x6 = argmax(x3, size)
    x7 = merge(x3)
    x8 = fill(I, SEVEN, x7)
    x9 = fill(x8, EIGHT, x6)
    O = fill(x9, SIX, x5)
    return O


def solve_1b60fb0c(I):
    x1 = rot90(I)
    x2 = ofcolor(I, ONE)
    x3 = ofcolor(x1, ONE)
    x4 = neighbors(ORIGIN)
    x5 = mapply(neighbors, x4)
    x6 = lbind(shift, x3)
    x7 = apply(x6, x5)
    x8 = lbind(intersection, x2)
    x9 = compose(size, x8)
    x10 = argmax(x7, x9)
    O = underfill(I, TWO, x10)
    return O


def solve_ddf7fa4f(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = colorfilter(x1, FIVE)
    x4 = product(x2, x3)
    x5 = fork(vmatching, first, last)
    x6 = sfilter(x4, x5)
    x7 = compose(color, first)
    x8 = fork(recolor, x7, last)
    x9 = mapply(x8, x6)
    O = paint(I, x9)
    return O


def solve_47c1f68c(I):
    x1 = leastcolor(I)
    x2 = vmirror(I)
    x3 = objects(I, T, T, T)
    x4 = merge(x3)
    x5 = mostcolor(x4)
    x6 = cellwise(I, x2, x1)
    x7 = hmirror(x6)
    x8 = cellwise(x6, x7, x1)
    x9 = compress(x8)
    O = replace(x9, x1, x5)
    return O


def solve_6c434453(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, EIGHT)
    x3 = dneighbors(UNITY)
    x4 = insert(UNITY, x3)
    x5 = merge(x2)
    x6 = cover(I, x5)
    x7 = apply(ulcorner, x2)
    x8 = lbind(shift, x4)
    x9 = mapply(x8, x7)
    O = fill(x6, TWO, x9)
    return O


def solve_23581191(I):
    x1 = objects(I, T, T, T)
    x2 = fork(combine, vfrontier, hfrontier)
    x3 = compose(x2, center)
    x4 = fork(recolor, color, x3)
    x5 = mapply(x4, x1)
    x6 = paint(I, x5)
    x7 = fork(intersection, first, last)
    x8 = apply(x3, x1)
    x9 = x7(x8)
    O = fill(x6, TWO, x9)
    return O


def solve_c8cbb738(I):
    x1 = mostcolor(I)
    x2 = fgpartition(I)
    x3 = valmax(x2, shape)
    x4 = canvas(x1, x3)
    x5 = apply(normalize, x2)
    x6 = lbind(subtract, x3)
    x7 = chain(halve, x6, shape)
    x8 = fork(shift, identity, x7)
    x9 = mapply(x8, x5)
    O = paint(x4, x9)
    return O


def solve_3eda0437(I):
    x1 = interval(TWO, TEN, ONE)
    x2 = prapply(astuple, x1, x1)
    x3 = lbind(canvas, ZERO)
    x4 = lbind(occurrences, I)
    x5 = lbind(lbind, shift)
    x6 = fork(apply, x5, x4)
    x7 = chain(x6, asobject, x3)
    x8 = mapply(x7, x2)
    x9 = argmax(x8, size)
    O = fill(I, SIX, x9)
    return O


def solve_dc0a314f(I):
    x1 = ofcolor(I, THREE)
    x2 = replace(I, THREE, ZERO)
    x3 = dmirror(x2)
    x4 = papply(pair, x2, x3)
    x5 = lbind(apply, maximum)
    x6 = apply(x5, x4)
    x7 = cmirror(x6)
    x8 = papply(pair, x6, x7)
    x9 = apply(x5, x8)
    O = subgrid(x1, x9)
    return O


def solve_d4469b4b(I):
    x1 = palette(I)
    x2 = other(x1, ZERO)
    x3 = equality(x2, ONE)
    x4 = equality(x2, TWO)
    x5 = branch(x3, UNITY, TWO_BY_TWO)
    x6 = branch(x4, RIGHT, x5)
    x7 = fork(combine, vfrontier, hfrontier)
    x8 = x7(x6)
    x9 = canvas(ZERO, THREE_BY_THREE)
    O = fill(x9, FIVE, x8)
    return O


def solve_6ecd11f4(I):
    x1 = objects(I, F, T, T)
    x2 = argmax(x1, size)
    x3 = argmin(x1, size)
    x4 = subgrid(x2, I)
    x5 = subgrid(x3, I)
    x6 = width(x4)
    x7 = width(x5)
    x8 = divide(x6, x7)
    x9 = downscale(x4, x8)
    x10 = ofcolor(x9, ZERO)
    O = fill(x5, ZERO, x10)
    return O


def solve_760b3cac(I):
    x1 = ofcolor(I, FOUR)
    x2 = ofcolor(I, EIGHT)
    x3 = ulcorner(x1)
    x4 = index(I, x3)
    x5 = equality(x4, FOUR)
    x6 = branch(x5, NEG_ONE, ONE)
    x7 = multiply(x6, THREE)
    x8 = tojvec(x7)
    x9 = vmirror(x2)
    x10 = shift(x9, x8)
    O = fill(I, EIGHT, x10)
    return O


def solve_c444b776(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = argmin(x2, size)
    x4 = backdrop(x3)
    x5 = toobject(x4, I)
    x6 = normalize(x5)
    x7 = lbind(shift, x6)
    x8 = compose(x7, ulcorner)
    x9 = mapply(x8, x2)
    O = paint(I, x9)
    return O


def solve_d4a91cb9(I):
    x1 = ofcolor(I, EIGHT)
    x2 = ofcolor(I, TWO)
    x3 = first(x1)
    x4 = first(x2)
    x5 = last(x3)
    x6 = first(x4)
    x7 = astuple(x6, x5)
    x8 = connect(x7, x3)
    x9 = connect(x7, x4)
    x10 = combine(x8, x9)
    O = underfill(I, FOUR, x10)
    return O


def solve_eb281b96(I):
    x1 = height(I)
    x2 = width(I)
    x3 = decrement(x1)
    x4 = astuple(x3, x2)
    x5 = crop(I, ORIGIN, x4)
    x6 = hmirror(x5)
    x7 = vconcat(I, x6)
    x8 = double(x3)
    x9 = astuple(x8, x2)
    x10 = crop(x7, DOWN, x9)
    O = vconcat(x7, x10)
    return O


def solve_ff28f65a(I):
    x1 = objects(I, T, F, T)
    x2 = size(x1)
    x3 = double(x2)
    x4 = interval(ZERO, x3, TWO)
    x5 = apply(tojvec, x4)
    x6 = astuple(ONE, NINE)
    x7 = canvas(ZERO, x6)
    x8 = fill(x7, ONE, x5)
    x9 = hsplit(x8, THREE)
    O = merge(x9)
    return O


def solve_7e0986d6(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = replace(I, x1, ZERO)
    x4 = leastcolor(x3)
    x5 = rbind(colorcount, x4)
    x6 = chain(positive, decrement, x5)
    x7 = rbind(toobject, x3)
    x8 = chain(x6, x7, dneighbors)
    x9 = sfilter(x2, x8)
    O = fill(x3, x4, x9)
    return O


def solve_09629e4f(I):
    x1 = objects(I, F, T, T)
    x2 = argmin(x1, numcolors)
    x3 = normalize(x2)
    x4 = upscale(x3, FOUR)
    x5 = paint(I, x4)
    x6 = ofcolor(I, FIVE)
    O = fill(x5, FIVE, x6)
    return O


def solve_a85d4709(I):
    x1 = ofcolor(I, FIVE)
    x2 = lbind(matcher, last)
    x3 = lbind(sfilter, x1)
    x4 = lbind(mapply, hfrontier)
    x5 = chain(x4, x3, x2)
    x6 = x5(ZERO)
    x7 = x5(TWO)
    x8 = x5(ONE)
    x9 = fill(I, TWO, x6)
    x10 = fill(x9, THREE, x7)
    O = fill(x10, FOUR, x8)
    return O


def solve_feca6190(I):
    x1 = objects(I, T, F, T)
    x2 = size(x1)
    x3 = multiply(x2, FIVE)
    x4 = astuple(x3, x3)
    x5 = canvas(ZERO, x4)
    x6 = rbind(shoot, UNITY)
    x7 = compose(x6, center)
    x8 = fork(recolor, color, x7)
    x9 = mapply(x8, x1)
    x10 = paint(x5, x9)
    O = hmirror(x10)
    return O


def solve_a68b268e(I):
    x1 = tophalf(I)
    x2 = bottomhalf(I)
    x3 = lefthalf(x1)
    x4 = righthalf(x1)
    x5 = lefthalf(x2)
    x6 = righthalf(x2)
    x7 = ofcolor(x4, FOUR)
    x8 = ofcolor(x3, SEVEN)
    x9 = ofcolor(x5, EIGHT)
    x10 = fill(x6, EIGHT, x9)
    x11 = fill(x10, FOUR, x7)
    O = fill(x11, SEVEN, x8)
    return O


def solve_beb8660c(I):
    x1 = shape(I)
    x2 = objects(I, T, F, T)
    x3 = compose(invert, size)
    x4 = order(x2, x3)
    x5 = apply(normalize, x4)
    x6 = size(x5)
    x7 = interval(ZERO, x6, ONE)
    x8 = apply(toivec, x7)
    x9 = mpapply(shift, x5, x8)
    x10 = canvas(ZERO, x1)
    x11 = paint(x10, x9)
    O = rot180(x11)
    return O


def solve_913fb3ed(I):
    x1 = ofcolor(I, THREE)
    x2 = ofcolor(I, EIGHT)
    x3 = ofcolor(I, TWO)
    x4 = mapply(neighbors, x1)
    x5 = mapply(neighbors, x2)
    x6 = mapply(neighbors, x3)
    x7 = fill(I, SIX, x4)
    x8 = fill(x7, FOUR, x5)
    O = fill(x8, ONE, x6)
    return O


def solve_0962bcdd(I):
    x1 = leastcolor(I)
    x2 = replace(I, ZERO, x1)
    x3 = leastcolor(x2)
    x4 = ofcolor(I, x3)
    x5 = mapply(dneighbors, x4)
    x6 = fill(I, x3, x5)
    x7 = objects(x6, F, T, T)
    x8 = fork(connect, ulcorner, lrcorner)
    x9 = fork(connect, llcorner, urcorner)
    x10 = fork(combine, x8, x9)
    x11 = mapply(x10, x7)
    O = fill(x6, x1, x11)
    return O


def solve_3631a71a(I):
    x1 = shape(I)
    x2 = replace(I, NINE, ZERO)
    x3 = lbind(apply, maximum)
    x4 = dmirror(x2)
    x5 = papply(pair, x2, x4)
    x6 = apply(x3, x5)
    x7 = subtract(x1, TWO_BY_TWO)
    x8 = crop(x6, TWO_BY_TWO, x7)
    x9 = vmirror(x8)
    x10 = objects(x9, T, F, T)
    x11 = merge(x10)
    x12 = shift(x11, TWO_BY_TWO)
    O = paint(x6, x12)
    return O


def solve_05269061(I):
    x1 = objects(I, T, T, T)
    x2 = neighbors(ORIGIN)
    x3 = mapply(neighbors, x2)
    x4 = rbind(multiply, THREE)
    x5 = apply(x4, x3)
    x6 = merge(x1)
    x7 = lbind(shift, x6)
    x8 = mapply(x7, x5)
    x9 = shift(x8, UP_RIGHT)
    x10 = shift(x8, DOWN_LEFT)
    x11 = paint(I, x8)
    x12 = paint(x11, x9)
    O = paint(x12, x10)
    return O


def solve_95990924(I):
    x1 = objects(I, T, F, T)
    x2 = apply(outbox, x1)
    x3 = apply(ulcorner, x2)
    x4 = apply(urcorner, x2)
    x5 = apply(llcorner, x2)
    x6 = apply(lrcorner, x2)
    x7 = fill(I, ONE, x3)
    x8 = fill(x7, TWO, x4)
    x9 = fill(x8, THREE, x5)
    O = fill(x9, FOUR, x6)
    return O


def solve_e509e548(I):
    x1 = objects(I, T, F, T)
    x2 = rbind(subgrid, I)
    x3 = chain(palette, trim, x2)
    x4 = lbind(contained, THREE)
    x5 = compose(x4, x3)
    x6 = fork(add, height, width)
    x7 = compose(decrement, x6)
    x8 = fork(equality, size, x7)
    x9 = mfilter(x1, x5)
    x10 = mfilter(x1, x8)
    x11 = replace(I, THREE, SIX)
    x12 = fill(x11, TWO, x9)
    O = fill(x12, ONE, x10)
    return O


def solve_d43fd935(I):
    x1 = objects(I, T, F, T)
    x2 = ofcolor(I, THREE)
    x3 = sizefilter(x1, ONE)
    x4 = rbind(vmatching, x2)
    x5 = rbind(hmatching, x2)
    x6 = fork(either, x4, x5)
    x7 = sfilter(x3, x6)
    x8 = rbind(gravitate, x2)
    x9 = fork(add, center, x8)
    x10 = fork(connect, center, x9)
    x11 = fork(recolor, color, x10)
    x12 = mapply(x11, x7)
    O = paint(I, x12)
    return O


def solve_db3e9e38(I):
    x1 = ofcolor(I, SEVEN)
    x2 = lrcorner(x1)
    x3 = shoot(x2, UP_RIGHT)
    x4 = shoot(x2, NEG_UNITY)
    x5 = combine(x3, x4)
    x6 = rbind(shoot, UP)
    x7 = mapply(x6, x5)
    x8 = last(x2)
    x9 = rbind(subtract, x8)
    x10 = chain(even, x9, last)
    x11 = fill(I, EIGHT, x7)
    x12 = sfilter(x7, x10)
    O = fill(x11, SEVEN, x12)
    return O


def solve_e73095fd(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = fork(equality, toindices, backdrop)
    x4 = sfilter(x2, x3)
    x5 = lbind(mapply, dneighbors)
    x6 = chain(x5, corners, outbox)
    x7 = fork(difference, x6, outbox)
    x8 = ofcolor(I, FIVE)
    x9 = rbind(intersection, x8)
    x10 = matcher(size, ZERO)
    x11 = chain(x10, x9, x7)
    x12 = mfilter(x4, x11)
    O = fill(I, FOUR, x12)
    return O


def solve_1bfc4729(I):
    x1 = asindices(I)
    x2 = tophalf(I)
    x3 = bottomhalf(I)
    x4 = leastcolor(x2)
    x5 = leastcolor(x3)
    x6 = hfrontier(TWO_BY_ZERO)
    x7 = box(x1)
    x8 = combine(x6, x7)
    x9 = fill(x2, x4, x8)
    x10 = hmirror(x9)
    x11 = replace(x10, x4, x5)
    O = vconcat(x9, x11)
    return O


def solve_93b581b8(I):
    x1 = fgpartition(I)
    x2 = chain(cmirror, dmirror, merge)
    x3 = x2(x1)
    x4 = upscale(x3, THREE)
    x5 = astuple(NEG_TWO, NEG_TWO)
    x6 = shift(x4, x5)
    x7 = underpaint(I, x6)
    x8 = toindices(x3)
    x9 = fork(combine, hfrontier, vfrontier)
    x10 = mapply(x9, x8)
    x11 = difference(x10, x8)
    O = fill(x7, ZERO, x11)
    return O


def solve_9edfc990(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = ofcolor(I, ONE)
    x4 = rbind(adjacent, x3)
    x5 = mfilter(x2, x4)
    x6 = recolor(ONE, x5)
    O = paint(I, x6)
    return O


def solve_a65b410d(I):
    x1 = ofcolor(I, TWO)
    x2 = urcorner(x1)
    x3 = shoot(x2, UP_RIGHT)
    x4 = shoot(x2, DOWN_LEFT)
    x5 = underfill(I, THREE, x3)
    x6 = underfill(x5, ONE, x4)
    x7 = rbind(shoot, LEFT)
    x8 = mapply(x7, x3)
    x9 = mapply(x7, x4)
    x10 = underfill(x6, ONE, x9)
    O = underfill(x10, THREE, x8)
    return O


def solve_7447852a(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = compose(last, center)
    x4 = order(x2, x3)
    x5 = size(x4)
    x6 = interval(ZERO, x5, THREE)
    x7 = rbind(contained, x6)
    x8 = compose(x7, last)
    x9 = interval(ZERO, x5, ONE)
    x10 = pair(x4, x9)
    x11 = sfilter(x10, x8)
    x12 = mapply(first, x11)
    O = fill(I, FOUR, x12)
    return O


def solve_97999447(I):
    x1 = objects(I, T, F, T)
    x2 = apply(toindices, x1)
    x3 = rbind(shoot, RIGHT)
    x4 = compose(x3, center)
    x5 = fork(recolor, color, x4)
    x6 = mapply(x5, x1)
    x7 = paint(I, x6)
    x8 = interval(ZERO, FIVE, ONE)
    x9 = apply(double, x8)
    x10 = apply(increment, x9)
    x11 = apply(tojvec, x10)
    x12 = prapply(shift, x2, x11)
    x13 = merge(x12)
    O = fill(x7, FIVE, x13)
    return O


def solve_91714a58(I):
    x1 = shape(I)
    x2 = asindices(I)
    x3 = objects(I, T, F, T)
    x4 = argmax(x3, size)
    x5 = mostcolor(x4)
    x6 = canvas(ZERO, x1)
    x7 = paint(x6, x4)
    x8 = rbind(toobject, x7)
    x9 = rbind(colorcount, x5)
    x10 = chain(x9, x8, neighbors)
    x11 = lbind(greater, THREE)
    x12 = compose(x11, x10)
    x13 = sfilter(x2, x12)
    O = fill(x7, ZERO, x13)
    return O


def solve_a61ba2ce(I):
    x1 = objects(I, T, F, T)
    x2 = lbind(index, I)
    x3 = matcher(x2, ZERO)
    x4 = lbind(extract, x1)
    x5 = rbind(subgrid, I)
    x6 = lbind(compose, x3)
    x7 = chain(x5, x4, x6)
    x8 = x7(ulcorner)
    x9 = x7(urcorner)
    x10 = x7(llcorner)
    x11 = x7(lrcorner)
    x12 = hconcat(x11, x10)
    x13 = hconcat(x9, x8)
    O = vconcat(x12, x13)
    return O


def solve_8e1813be(I):
    x1 = replace(I, FIVE, ZERO)
    x2 = objects(x1, T, T, T)
    x3 = first(x2)
    x4 = vline(x3)
    x5 = branch(x4, dmirror, identity)
    x6 = x5(x1)
    x7 = objects(x6, T, T, T)
    x8 = order(x7, uppermost)
    x9 = apply(color, x8)
    x10 = dedupe(x9)
    x11 = size(x10)
    x12 = rbind(repeat, x11)
    x13 = apply(x12, x10)
    O = x5(x13)
    return O


def solve_bc1d5164(I):
    x1 = leastcolor(I)
    x2 = crop(I, ORIGIN, THREE_BY_THREE)
    x3 = crop(I, TWO_BY_ZERO, THREE_BY_THREE)
    x4 = tojvec(FOUR)
    x5 = crop(I, x4, THREE_BY_THREE)
    x6 = astuple(TWO, FOUR)
    x7 = crop(I, x6, THREE_BY_THREE)
    x8 = canvas(ZERO, THREE_BY_THREE)
    x9 = rbind(ofcolor, x1)
    x10 = astuple(x2, x3)
    x11 = astuple(x5, x7)
    x12 = combine(x10, x11)
    x13 = mapply(x9, x12)
    O = fill(x8, x1, x13)
    return O


def solve_ce602527(I):
    x1 = vmirror(I)
    x2 = fgpartition(x1)
    x3 = order(x2, size)
    x4 = last(x3)
    x5 = remove(x4, x3)
    x6 = compose(toindices, normalize)
    x7 = rbind(upscale, TWO)
    x8 = chain(toindices, x7, normalize)
    x9 = x6(x4)
    x10 = rbind(intersection, x9)
    x11 = chain(size, x10, x8)
    x12 = argmax(x5, x11)
    x13 = subgrid(x12, x1)
    O = vmirror(x13)
    return O


def solve_5c2c9af4(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = center(x2)
    x4 = ulcorner(x2)
    x5 = subtract(x3, x4)
    x6 = multiply(NEG_ONE, NINE)
    x7 = interval(ZERO, NINE, ONE)
    x8 = interval(ZERO, x6, NEG_ONE)
    x9 = lbind(multiply, x5)
    x10 = apply(x9, x7)
    x11 = apply(x9, x8)
    x12 = pair(x10, x11)
    x13 = mapply(box, x12)
    x14 = shift(x13, x3)
    O = fill(I, x1, x14)
    return O


def solve_75b8110e(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = tophalf(x1)
    x4 = bottomhalf(x1)
    x5 = tophalf(x2)
    x6 = bottomhalf(x2)
    x7 = rbind(ofcolor, ZERO)
    x8 = fork(difference, asindices, x7)
    x9 = fork(toobject, x8, identity)
    x10 = x9(x5)
    x11 = x9(x4)
    x12 = x9(x6)
    x13 = paint(x3, x12)
    x14 = paint(x13, x11)
    O = paint(x14, x10)
    return O


def solve_941d9a10(I):
    x1 = shape(I)
    x2 = objects(I, T, F, F)
    x3 = colorfilter(x2, ZERO)
    x4 = apply(toindices, x3)
    x5 = lbind(lbind, contained)
    x6 = lbind(extract, x4)
    x7 = compose(x6, x5)
    x8 = decrement(x1)
    x9 = astuple(FIVE, FIVE)
    x10 = x7(ORIGIN)
    x11 = x7(x8)
    x12 = x7(x9)
    x13 = fill(I, ONE, x10)
    x14 = fill(x13, THREE, x11)
    O = fill(x14, TWO, x12)
    return O


def solve_c3f564a4(I):
    x1 = asindices(I)
    x2 = dmirror(I)
    x3 = invert(NINE)
    x4 = papply(pair, I, x2)
    x5 = lbind(apply, maximum)
    x6 = apply(x5, x4)
    x7 = ofcolor(x6, ZERO)
    x8 = difference(x1, x7)
    x9 = toobject(x8, x6)
    x10 = interval(x3, NINE, ONE)
    x11 = interval(NINE, x3, NEG_ONE)
    x12 = pair(x10, x11)
    x13 = lbind(shift, x9)
    x14 = mapply(x13, x12)
    O = paint(x6, x14)
    return O


def solve_1a07d186(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = difference(x1, x2)
    x4 = apply(color, x3)
    x5 = rbind(contained, x4)
    x6 = compose(x5, color)
    x7 = sfilter(x2, x6)
    x8 = lbind(colorfilter, x3)
    x9 = chain(first, x8, color)
    x10 = fork(gravitate, identity, x9)
    x11 = fork(shift, identity, x10)
    x12 = mapply(x11, x7)
    x13 = merge(x2)
    x14 = cover(I, x13)
    O = paint(x14, x12)
    return O


def solve_d687bc17(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = difference(x1, x2)
    x4 = apply(color, x3)
    x5 = rbind(contained, x4)
    x6 = compose(x5, color)
    x7 = sfilter(x2, x6)
    x8 = lbind(colorfilter, x3)
    x9 = chain(first, x8, color)
    x10 = fork(gravitate, identity, x9)
    x11 = fork(shift, identity, x10)
    x12 = merge(x2)
    x13 = mapply(x11, x7)
    x14 = cover(I, x12)
    O = paint(x14, x13)
    return O


def solve_9af7a82c(I):
    x1 = objects(I, T, F, F)
    x2 = order(x1, size)
    x3 = valmax(x1, size)
    x4 = rbind(astuple, ONE)
    x5 = lbind(subtract, x3)
    x6 = compose(x4, size)
    x7 = chain(x4, x5, size)
    x8 = fork(canvas, color, x6)
    x9 = lbind(canvas, ZERO)
    x10 = compose(x9, x7)
    x11 = fork(vconcat, x8, x10)
    x12 = compose(cmirror, x11)
    x13 = apply(x12, x2)
    x14 = merge(x13)
    O = cmirror(x14)
    return O


def solve_6e19193c(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, T)
    x3 = rbind(toobject, I)
    x4 = compose(first, delta)
    x5 = rbind(colorcount, x1)
    x6 = matcher(x5, TWO)
    x7 = chain(x6, x3, dneighbors)
    x8 = rbind(sfilter, x7)
    x9 = chain(first, x8, toindices)
    x10 = fork(subtract, x4, x9)
    x11 = fork(shoot, x4, x10)
    x12 = mapply(x11, x2)
    x13 = fill(I, x1, x12)
    x14 = mapply(delta, x2)
    O = fill(x13, ZERO, x14)
    return O


def solve_ef135b50(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, ZERO)
    x3 = product(x1, x1)
    x4 = power(first, TWO)
    x5 = compose(first, last)
    x6 = fork(equality, x4, x5)
    x7 = sfilter(x3, x6)
    x8 = fork(connect, first, last)
    x9 = mapply(x8, x7)
    x10 = intersection(x9, x2)
    x11 = fill(I, NINE, x10)
    x12 = trim(x11)
    x13 = asobject(x12)
    x14 = shift(x13, UNITY)
    O = paint(I, x14)
    return O


def solve_cbded52d(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = product(x2, x2)
    x4 = fork(vmatching, first, last)
    x5 = fork(hmatching, first, last)
    x6 = fork(either, x4, x5)
    x7 = sfilter(x3, x6)
    x8 = compose(center, first)
    x9 = compose(center, last)
    x10 = fork(connect, x8, x9)
    x11 = chain(initset, center, x10)
    x12 = compose(color, first)
    x13 = fork(recolor, x12, x11)
    x14 = mapply(x13, x7)
    O = paint(I, x14)
    return O


def solve_8a004b2b(I):
    x1 = objects(I, F, T, T)
    x2 = ofcolor(I, FOUR)
    x3 = subgrid(x2, I)
    x4 = argmax(x1, lowermost)
    x5 = normalize(x4)
    x6 = replace(x3, FOUR, ZERO)
    x7 = objects(x6, T, F, T)
    x8 = merge(x7)
    x9 = width(x8)
    x10 = ulcorner(x8)
    x11 = width(x4)
    x12 = divide(x9, x11)
    x13 = upscale(x5, x12)
    x14 = shift(x13, x10)
    O = paint(x3, x14)
    return O


def solve_e26a3af2(I):
    x1 = rot90(I)
    x2 = apply(mostcommon, I)
    x3 = apply(mostcommon, x1)
    x4 = repeat(x2, ONE)
    x5 = repeat(x3, ONE)
    x6 = compose(size, dedupe)
    x7 = x6(x2)
    x8 = x6(x3)
    x9 = greater(x8, x7)
    x10 = branch(x9, height, width)
    x11 = x10(I)
    x12 = rot90(x4)
    x13 = branch(x9, x5, x12)
    x14 = branch(x9, vupscale, hupscale)
    O = x14(x13, x11)
    return O


def solve_6cf79266(I):
    x1 = ofcolor(I, ZERO)
    x2 = astuple(ZERO, ORIGIN)
    x3 = initset(x2)
    x4 = upscale(x3, THREE)
    x5 = toindices(x4)
    x6 = lbind(shift, x5)
    x7 = rbind(difference, x1)
    x8 = chain(size, x7, x6)
    x9 = matcher(x8, ZERO)
    x10 = lbind(add, NEG_UNITY)
    x11 = chain(flip, x9, x10)
    x12 = fork(both, x9, x11)
    x13 = sfilter(x1, x12)
    x14 = mapply(x6, x13)
    O = fill(I, ONE, x14)
    return O


def solve_a87f7484(I):
    x1 = numcolors(I)
    x2 = dmirror(I)
    x3 = portrait(I)
    x4 = branch(x3, dmirror, identity)
    x5 = x4(I)
    x6 = decrement(x1)
    x7 = hsplit(x5, x6)
    x8 = rbind(ofcolor, ZERO)
    x9 = apply(x8, x7)
    x10 = leastcommon(x9)
    x11 = matcher(x8, x10)
    x12 = extract(x7, x11)
    O = x4(x12)
    return O


def solve_4093f84a(I):
    x1 = leastcolor(I)
    x2 = replace(I, x1, FIVE)
    x3 = ofcolor(I, FIVE)
    x4 = portrait(x3)
    x5 = branch(x4, identity, dmirror)
    x6 = x5(x2)
    x7 = lefthalf(x6)
    x8 = righthalf(x6)
    x9 = rbind(order, identity)
    x10 = rbind(order, invert)
    x11 = apply(x9, x7)
    x12 = apply(x10, x8)
    x13 = hconcat(x11, x12)
    O = x5(x13)
    return O


def solve_ba26e723(I):
    x1 = rbind(divide, THREE)
    x2 = rbind(multiply, THREE)
    x3 = compose(x2, x1)
    x4 = fork(equality, identity, x3)
    x5 = compose(x4, last)
    x6 = ofcolor(I, FOUR)
    x7 = sfilter(x6, x5)
    O = fill(I, SIX, x7)
    return O


def solve_4612dd53(I):
    x1 = ofcolor(I, ONE)
    x2 = box(x1)
    x3 = fill(I, TWO, x2)
    x4 = subgrid(x1, x3)
    x5 = ofcolor(x4, ONE)
    x6 = mapply(vfrontier, x5)
    x7 = mapply(hfrontier, x5)
    x8 = size(x6)
    x9 = size(x7)
    x10 = greater(x8, x9)
    x11 = branch(x10, x7, x6)
    x12 = fill(x4, TWO, x11)
    x13 = ofcolor(x12, TWO)
    x14 = ulcorner(x1)
    x15 = shift(x13, x14)
    O = underfill(I, TWO, x15)
    return O


def solve_29c11459(I):
    x1 = lefthalf(I)
    x2 = righthalf(I)
    x3 = objects(x2, T, F, T)
    x4 = objects(x1, T, F, T)
    x5 = compose(hfrontier, center)
    x6 = fork(recolor, color, x5)
    x7 = mapply(x6, x4)
    x8 = paint(x1, x7)
    x9 = mapply(x6, x3)
    x10 = paint(I, x9)
    x11 = objects(x8, T, F, T)
    x12 = apply(urcorner, x11)
    x13 = shift(x12, RIGHT)
    x14 = merge(x11)
    x15 = paint(x10, x14)
    O = fill(x15, FIVE, x13)
    return O


def solve_963e52fc(I):
    x1 = width(I)
    x2 = asobject(I)
    x3 = hperiod(x2)
    x4 = height(x2)
    x5 = astuple(x4, x3)
    x6 = ulcorner(x2)
    x7 = crop(I, x6, x5)
    x8 = rot90(x7)
    x9 = double(x1)
    x10 = divide(x9, x3)
    x11 = increment(x10)
    x12 = repeat(x8, x11)
    x13 = merge(x12)
    x14 = rot270(x13)
    x15 = astuple(x4, x9)
    O = crop(x14, ORIGIN, x15)
    return O


def solve_ae3edfdc(I):
    x1 = objects(I, T, F, T)
    x2 = replace(I, THREE, ZERO)
    x3 = replace(x2, SEVEN, ZERO)
    x4 = lbind(colorfilter, x1)
    x5 = lbind(rbind, gravitate)
    x6 = chain(x5, first, x4)
    x7 = x6(TWO)
    x8 = x6(ONE)
    x9 = x4(THREE)
    x10 = x4(SEVEN)
    x11 = fork(shift, identity, x7)
    x12 = fork(shift, identity, x8)
    x13 = mapply(x11, x9)
    x14 = mapply(x12, x10)
    x15 = paint(x3, x13)
    O = paint(x15, x14)
    return O


def solve_1f0c79e5(I):
    x1 = ofcolor(I, TWO)
    x2 = replace(I, TWO, ZERO)
    x3 = leastcolor(x2)
    x4 = ofcolor(x2, x3)
    x5 = combine(x1, x4)
    x6 = recolor(x3, x5)
    x7 = compose(decrement, double)
    x8 = ulcorner(x5)
    x9 = invert(x8)
    x10 = shift(x1, x9)
    x11 = apply(x7, x10)
    x12 = interval(ZERO, NINE, ONE)
    x13 = prapply(multiply, x11, x12)
    x14 = lbind(shift, x6)
    x15 = mapply(x14, x13)
    O = paint(I, x15)
    return O


def solve_56dc2b01(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, THREE)
    x3 = first(x2)
    x4 = ofcolor(I, TWO)
    x5 = gravitate(x3, x4)
    x6 = first(x5)
    x7 = equality(x6, ZERO)
    x8 = branch(x7, width, height)
    x9 = x8(x3)
    x10 = gravitate(x4, x3)
    x11 = sign(x10)
    x12 = multiply(x11, x9)
    x13 = crement(x12)
    x14 = recolor(EIGHT, x4)
    x15 = shift(x14, x13)
    x16 = paint(I, x15)
    O = move(x16, x3, x5)
    return O


def solve_e48d4e1a(I):
    x1 = shape(I)
    x2 = ofcolor(I, FIVE)
    x3 = fill(I, ZERO, x2)
    x4 = leastcolor(x3)
    x5 = size(x2)
    x6 = ofcolor(I, x4)
    x7 = rbind(toobject, I)
    x8 = rbind(colorcount, x4)
    x9 = chain(x8, x7, dneighbors)
    x10 = matcher(x9, FOUR)
    x11 = extract(x6, x10)
    x12 = multiply(DOWN_LEFT, x5)
    x13 = add(x12, x11)
    x14 = canvas(ZERO, x1)
    x15 = fork(combine, vfrontier, hfrontier)
    x16 = x15(x13)
    O = fill(x14, x4, x16)
    return O


def solve_6773b310(I):
    x1 = compress(I)
    x2 = neighbors(ORIGIN)
    x3 = insert(ORIGIN, x2)
    x4 = rbind(multiply, THREE)
    x5 = apply(x4, x3)
    x6 = astuple(FOUR, FOUR)
    x7 = shift(x5, x6)
    x8 = fork(insert, identity, neighbors)
    x9 = apply(x8, x7)
    x10 = rbind(toobject, x1)
    x11 = apply(x10, x9)
    x12 = rbind(colorcount, SIX)
    x13 = matcher(x12, TWO)
    x14 = mfilter(x11, x13)
    x15 = fill(x1, ONE, x14)
    x16 = replace(x15, SIX, ZERO)
    O = downscale(x16, THREE)
    return O


def solve_780d0b14(I):
    x1 = asindices(I)
    x2 = objects(I, T, T, T)
    x3 = rbind(greater, TWO)
    x4 = compose(x3, size)
    x5 = sfilter(x2, x4)
    x6 = totuple(x5)
    x7 = apply(color, x6)
    x8 = apply(center, x6)
    x9 = pair(x7, x8)
    x10 = fill(I, ZERO, x1)
    x11 = paint(x10, x9)
    x12 = rbind(greater, ONE)
    x13 = compose(dedupe, totuple)
    x14 = chain(x12, size, x13)
    x15 = sfilter(x11, x14)
    x16 = rot90(x15)
    x17 = sfilter(x16, x14)
    O = rot270(x17)
    return O


def solve_2204b7a8(I):
    x1 = objects(I, T, F, T)
    x2 = lbind(sfilter, x1)
    x3 = compose(size, x2)
    x4 = x3(vline)
    x5 = x3(hline)
    x6 = greater(x4, x5)
    x7 = branch(x6, lefthalf, tophalf)
    x8 = branch(x6, righthalf, bottomhalf)
    x9 = branch(x6, hconcat, vconcat)
    x10 = x7(I)
    x11 = x8(I)
    x12 = index(x10, ORIGIN)
    x13 = shape(x11)
    x14 = decrement(x13)
    x15 = index(x11, x14)
    x16 = replace(x10, THREE, x12)
    x17 = replace(x11, THREE, x15)
    O = x9(x16, x17)
    return O


def solve_d9f24cd1(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, FIVE)
    x3 = prapply(connect, x1, x2)
    x4 = mfilter(x3, vline)
    x5 = underfill(I, TWO, x4)
    x6 = matcher(numcolors, TWO)
    x7 = objects(x5, F, F, T)
    x8 = sfilter(x7, x6)
    x9 = difference(x7, x8)
    x10 = colorfilter(x9, TWO)
    x11 = mapply(toindices, x10)
    x12 = apply(urcorner, x8)
    x13 = shift(x12, UNITY)
    x14 = rbind(shoot, UP)
    x15 = mapply(x14, x13)
    x16 = fill(x5, TWO, x15)
    x17 = mapply(vfrontier, x11)
    O = fill(x16, TWO, x17)
    return O


def solve_b782dc8a(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, F)
    x3 = ofcolor(I, x1)
    x4 = first(x3)
    x5 = dneighbors(x4)
    x6 = toobject(x5, I)
    x7 = mostcolor(x6)
    x8 = ofcolor(I, x7)
    x9 = colorfilter(x2, ZERO)
    x10 = rbind(adjacent, x8)
    x11 = mfilter(x9, x10)
    x12 = toindices(x11)
    x13 = rbind(manhattan, x3)
    x14 = chain(even, x13, initset)
    x15 = sfilter(x12, x14)
    x16 = difference(x12, x15)
    x17 = fill(I, x1, x15)
    O = fill(x17, x7, x16)
    return O


def solve_673ef223(I):
    x1 = objects(I, T, F, T)
    x2 = ofcolor(I, EIGHT)
    x3 = replace(I, EIGHT, FOUR)
    x4 = colorfilter(x1, TWO)
    x5 = argmin(x1, uppermost)
    x6 = apply(uppermost, x4)
    x7 = fork(subtract, maximum, minimum)
    x8 = x7(x6)
    x9 = toivec(x8)
    x10 = leftmost(x5)
    x11 = equality(x10, ZERO)
    x12 = branch(x11, LEFT, RIGHT)
    x13 = rbind(shoot, x12)
    x14 = mapply(x13, x2)
    x15 = underfill(x3, EIGHT, x14)
    x16 = shift(x2, x9)
    x17 = mapply(hfrontier, x16)
    O = underfill(x15, EIGHT, x17)
    return O


def solve_f5b8619d(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = mapply(vfrontier, x2)
    x4 = underfill(I, EIGHT, x3)
    x5 = hconcat(x4, x4)
    O = vconcat(x5, x5)
    return O


def solve_f8c80d96(I):
    x1 = leastcolor(I)
    x2 = objects(I, T, F, F)
    x3 = colorfilter(x2, x1)
    x4 = argmax(x3, size)
    x5 = argmin(x2, width)
    x6 = size(x5)
    x7 = equality(x6, ONE)
    x8 = branch(x7, identity, outbox)
    x9 = chain(outbox, outbox, x8)
    x10 = power(x9, TWO)
    x11 = power(x9, THREE)
    x12 = x9(x4)
    x13 = x10(x4)
    x14 = x11(x4)
    x15 = fill(I, x1, x12)
    x16 = fill(x15, x1, x13)
    x17 = fill(x16, x1, x14)
    O = replace(x17, ZERO, FIVE)
    return O


def solve_ecdecbb3(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, TWO)
    x3 = colorfilter(x1, EIGHT)
    x4 = product(x2, x3)
    x5 = fork(gravitate, first, last)
    x6 = compose(crement, x5)
    x7 = compose(center, first)
    x8 = fork(add, x7, x6)
    x9 = fork(connect, x7, x8)
    x10 = apply(x9, x4)
    x11 = lbind(greater, EIGHT)
    x12 = compose(x11, size)
    x13 = mfilter(x10, x12)
    x14 = fill(I, TWO, x13)
    x15 = apply(x8, x4)
    x16 = intersection(x13, x15)
    x17 = mapply(neighbors, x16)
    O = fill(x14, EIGHT, x17)
    return O


def solve_e5062a87(I):
    x1 = ofcolor(I, TWO)
    x2 = recolor(ZERO, x1)
    x3 = normalize(x2)
    x4 = occurrences(I, x2)
    x5 = lbind(shift, x3)
    x6 = apply(x5, x4)
    x7 = astuple(ONE, THREE)
    x8 = astuple(FIVE, ONE)
    x9 = astuple(TWO, SIX)
    x10 = initset(x7)
    x11 = insert(x8, x10)
    x12 = insert(x9, x11)
    x13 = rbind(contained, x12)
    x14 = chain(flip, x13, ulcorner)
    x15 = sfilter(x6, x14)
    x16 = merge(x15)
    x17 = recolor(TWO, x16)
    O = paint(I, x17)
    return O


def solve_a8d7556c(I):
    x1 = initset(ORIGIN)
    x2 = recolor(ZERO, x1)
    x3 = upscale(x2, TWO)
    x4 = occurrences(I, x3)
    x5 = lbind(shift, x3)
    x6 = mapply(x5, x4)
    x7 = fill(I, TWO, x6)
    x8 = add(SIX, SIX)
    x9 = astuple(EIGHT, x8)
    x10 = index(x7, x9)
    x11 = equality(x10, TWO)
    x12 = initset(x9)
    x13 = add(x9, DOWN)
    x14 = insert(x13, x12)
    x15 = toobject(x14, x7)
    x16 = toobject(x14, I)
    x17 = branch(x11, x16, x15)
    O = paint(x7, x17)
    return O


def solve_4938f0c2(I):
    x1 = objects(I, T, T, T)
    x2 = ofcolor(I, TWO)
    x3 = vmirror(x2)
    x4 = height(x2)
    x5 = width(x2)
    x6 = toivec(x4)
    x7 = tojvec(x5)
    x8 = add(x7, ZERO_BY_TWO)
    x9 = add(x6, TWO_BY_ZERO)
    x10 = shift(x3, x8)
    x11 = fill(I, TWO, x10)
    x12 = ofcolor(x11, TWO)
    x13 = hmirror(x12)
    x14 = shift(x13, x9)
    x15 = fill(x11, TWO, x14)
    x16 = size(x1)
    x17 = greater(x16, FOUR)
    O = branch(x17, I, x15)
    return O


def solve_834ec97d(I):
    x1 = asindices(I)
    x2 = objects(I, T, F, T)
    x3 = first(x2)
    x4 = shift(x3, DOWN)
    x5 = fill(I, ZERO, x3)
    x6 = paint(x5, x4)
    x7 = uppermost(x4)
    x8 = leftmost(x4)
    x9 = subtract(x8, TEN)
    x10 = add(x8, TEN)
    x11 = interval(x9, x10, TWO)
    x12 = lbind(greater, x7)
    x13 = compose(x12, first)
    x14 = rbind(contained, x11)
    x15 = compose(x14, last)
    x16 = sfilter(x1, x13)
    x17 = sfilter(x16, x15)
    O = fill(x6, FOUR, x17)
    return O


def solve_846bdb03(I):
    x1 = objects(I, F, F, T)
    x2 = rbind(colorcount, FOUR)
    x3 = matcher(x2, ZERO)
    x4 = extract(x1, x3)
    x5 = remove(x4, x1)
    x6 = merge(x5)
    x7 = subgrid(x6, I)
    x8 = index(x7, DOWN)
    x9 = subgrid(x4, I)
    x10 = lefthalf(x9)
    x11 = palette(x10)
    x12 = other(x11, ZERO)
    x13 = equality(x8, x12)
    x14 = branch(x13, identity, vmirror)
    x15 = x14(x4)
    x16 = normalize(x15)
    x17 = shift(x16, UNITY)
    O = paint(x7, x17)
    return O


def solve_90f3ed37(I):
    x1 = objects(I, T, T, T)
    x2 = order(x1, uppermost)
    x3 = first(x2)
    x4 = remove(x3, x2)
    x5 = normalize(x3)
    x6 = lbind(shift, x5)
    x7 = compose(x6, ulcorner)
    x8 = interval(TWO, NEG_ONE, NEG_ONE)
    x9 = apply(tojvec, x8)
    x10 = rbind(apply, x9)
    x11 = lbind(compose, size)
    x12 = lbind(lbind, intersection)
    x13 = compose(x11, x12)
    x14 = lbind(lbind, shift)
    x15 = chain(x10, x14, x7)
    x16 = fork(argmax, x15, x13)
    x17 = mapply(x16, x4)
    O = underfill(I, ONE, x17)
    return O


def solve_8403a5d5(I):
    x1 = asindices(I)
    x2 = objects(I, T, F, T)
    x3 = first(x2)
    x4 = color(x3)
    x5 = leftmost(x3)
    x6 = interval(x5, TEN, TWO)
    x7 = rbind(contained, x6)
    x8 = compose(x7, last)
    x9 = sfilter(x1, x8)
    x10 = increment(x5)
    x11 = add(x5, THREE)
    x12 = interval(x10, TEN, FOUR)
    x13 = interval(x11, TEN, FOUR)
    x14 = lbind(astuple, NINE)
    x15 = apply(tojvec, x12)
    x16 = apply(x14, x13)
    x17 = fill(I, x4, x9)
    x18 = fill(x17, FIVE, x15)
    O = fill(x18, FIVE, x16)
    return O


def solve_91413438(I):
    x1 = colorcount(I, ZERO)
    x2 = subtract(NINE, x1)
    x3 = multiply(x1, THREE)
    x4 = multiply(x3, x1)
    x5 = subtract(x4, THREE)
    x6 = astuple(THREE, x5)
    x7 = canvas(ZERO, x6)
    x8 = hconcat(I, x7)
    x9 = objects(x8, T, T, T)
    x10 = first(x9)
    x11 = lbind(shift, x10)
    x12 = compose(x11, tojvec)
    x13 = interval(ZERO, x2, ONE)
    x14 = rbind(multiply, THREE)
    x15 = apply(x14, x13)
    x16 = mapply(x12, x15)
    x17 = paint(x8, x16)
    x18 = hsplit(x17, x1)
    O = merge(x18)
    return O


def solve_539a4f51(I):
    x1 = shape(I)
    x2 = index(I, ORIGIN)
    x3 = colorcount(I, ZERO)
    x4 = decrement(x1)
    x5 = positive(x3)
    x6 = branch(x5, x4, x1)
    x7 = crop(I, ORIGIN, x6)
    x8 = width(x7)
    x9 = astuple(ONE, x8)
    x10 = crop(x7, ORIGIN, x9)
    x11 = vupscale(x10, x8)
    x12 = dmirror(x11)
    x13 = hconcat(x7, x11)
    x14 = hconcat(x12, x7)
    x15 = vconcat(x13, x14)
    x16 = asobject(x15)
    x17 = multiply(UNITY, TEN)
    x18 = canvas(x2, x17)
    O = paint(x18, x16)
    return O


def solve_5daaa586(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = rbind(bordering, I)
    x4 = compose(flip, x3)
    x5 = extract(x2, x4)
    x6 = outbox(x5)
    x7 = subgrid(x6, I)
    x8 = fgpartition(x7)
    x9 = argmax(x8, size)
    x10 = color(x9)
    x11 = toindices(x9)
    x12 = prapply(connect, x11, x11)
    x13 = mfilter(x12, vline)
    x14 = mfilter(x12, hline)
    x15 = size(x13)
    x16 = size(x14)
    x17 = greater(x15, x16)
    x18 = branch(x17, x13, x14)
    O = fill(x7, x10, x18)
    return O


def solve_3bdb4ada(I):
    x1 = objects(I, T, F, T)
    x2 = totuple(x1)
    x3 = compose(increment, ulcorner)
    x4 = compose(decrement, lrcorner)
    x5 = apply(x3, x2)
    x6 = apply(x4, x2)
    x7 = papply(connect, x5, x6)
    x8 = apply(last, x5)
    x9 = compose(last, first)
    x10 = power(last, TWO)
    x11 = fork(subtract, x9, x10)
    x12 = compose(even, x11)
    x13 = lbind(rbind, astuple)
    x14 = lbind(compose, x12)
    x15 = compose(x14, x13)
    x16 = fork(sfilter, first, x15)
    x17 = pair(x7, x8)
    x18 = mapply(x16, x17)
    O = fill(I, ZERO, x18)
    return O


def solve_ec883f72(I):
    x1 = palette(I)
    x2 = objects(I, T, T, T)
    x3 = fork(multiply, height, width)
    x4 = argmax(x2, x3)
    x5 = color(x4)
    x6 = remove(ZERO, x1)
    x7 = other(x6, x5)
    x8 = lrcorner(x4)
    x9 = llcorner(x4)
    x10 = urcorner(x4)
    x11 = ulcorner(x4)
    x12 = shoot(x8, UNITY)
    x13 = shoot(x9, DOWN_LEFT)
    x14 = shoot(x10, UP_RIGHT)
    x15 = shoot(x11, NEG_UNITY)
    x16 = combine(x12, x13)
    x17 = combine(x14, x15)
    x18 = combine(x16, x17)
    O = underfill(I, x7, x18)
    return O


def solve_2bee17df(I):
    x1 = height(I)
    x2 = rot90(I)
    x3 = subtract(x1, TWO)
    x4 = interval(ZERO, x1, ONE)
    x5 = rbind(colorcount, ZERO)
    x6 = matcher(x5, x3)
    x7 = rbind(vsplit, x1)
    x8 = lbind(apply, x6)
    x9 = compose(x8, x7)
    x10 = x9(I)
    x11 = pair(x4, x10)
    x12 = sfilter(x11, last)
    x13 = mapply(hfrontier, x12)
    x14 = x9(x2)
    x15 = pair(x14, x4)
    x16 = sfilter(x15, first)
    x17 = mapply(vfrontier, x16)
    x18 = astuple(x13, x17)
    x19 = merge(x18)
    O = underfill(I, THREE, x19)
    return O


def solve_e8dc4411(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, ZERO)
    x3 = ofcolor(I, x1)
    x4 = position(x2, x3)
    x5 = fork(connect, ulcorner, lrcorner)
    x6 = x5(x2)
    x7 = intersection(x2, x6)
    x8 = equality(x6, x7)
    x9 = fork(subtract, identity, crement)
    x10 = fork(add, identity, x9)
    x11 = branch(x8, identity, x10)
    x12 = shape(x2)
    x13 = multiply(x12, x4)
    x14 = apply(x11, x13)
    x15 = interval(ONE, FIVE, ONE)
    x16 = lbind(multiply, x14)
    x17 = apply(x16, x15)
    x18 = lbind(shift, x2)
    x19 = mapply(x18, x17)
    O = fill(I, x1, x19)
    return O


def solve_e40b9e2f(I):
    x1 = objects(I, F, T, T)
    x2 = neighbors(ORIGIN)
    x3 = mapply(neighbors, x2)
    x4 = first(x1)
    x5 = lbind(intersection, x4)
    x6 = compose(hmirror, vmirror)
    x7 = x6(x4)
    x8 = lbind(shift, x7)
    x9 = apply(x8, x3)
    x10 = argmax(x9, x5)
    x11 = paint(I, x10)
    x12 = objects(x11, F, T, T)
    x13 = first(x12)
    x14 = compose(size, x5)
    x15 = compose(vmirror, dmirror)
    x16 = x15(x13)
    x17 = lbind(shift, x16)
    x18 = apply(x17, x3)
    x19 = argmax(x18, x14)
    O = paint(x11, x19)
    return O


def solve_29623171(I):
    x1 = leastcolor(I)
    x2 = interval(ZERO, NINE, FOUR)
    x3 = product(x2, x2)
    x4 = rbind(add, THREE)
    x5 = rbind(interval, ONE)
    x6 = fork(x5, identity, x4)
    x7 = compose(x6, first)
    x8 = compose(x6, last)
    x9 = fork(product, x7, x8)
    x10 = rbind(colorcount, x1)
    x11 = rbind(toobject, I)
    x12 = compose(x10, x11)
    x13 = apply(x9, x3)
    x14 = valmax(x13, x12)
    x15 = matcher(x12, x14)
    x16 = compose(flip, x15)
    x17 = mfilter(x13, x15)
    x18 = mfilter(x13, x16)
    x19 = fill(I, x1, x17)
    O = fill(x19, ZERO, x18)
    return O


def solve_a2fd1cf0(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, THREE)
    x3 = uppermost(x1)
    x4 = leftmost(x1)
    x5 = uppermost(x2)
    x6 = leftmost(x2)
    x7 = astuple(x3, x5)
    x8 = minimum(x7)
    x9 = maximum(x7)
    x10 = astuple(x8, x6)
    x11 = astuple(x9, x6)
    x12 = connect(x10, x11)
    x13 = astuple(x4, x6)
    x14 = minimum(x13)
    x15 = maximum(x13)
    x16 = astuple(x3, x14)
    x17 = astuple(x3, x15)
    x18 = connect(x16, x17)
    x19 = combine(x12, x18)
    O = underfill(I, EIGHT, x19)
    return O


def solve_b0c4d837(I):
    x1 = ofcolor(I, FIVE)
    x2 = ofcolor(I, EIGHT)
    x3 = height(x1)
    x4 = decrement(x3)
    x5 = height(x2)
    x6 = subtract(x4, x5)
    x7 = astuple(ONE, x6)
    x8 = canvas(EIGHT, x7)
    x9 = subtract(SIX, x6)
    x10 = astuple(ONE, x9)
    x11 = canvas(ZERO, x10)
    x12 = hconcat(x8, x11)
    x13 = hsplit(x12, TWO)
    x14 = first(x13)
    x15 = last(x13)
    x16 = vmirror(x15)
    x17 = vconcat(x14, x16)
    x18 = astuple(ONE, THREE)
    x19 = canvas(ZERO, x18)
    O = vconcat(x17, x19)
    return O


def solve_8731374e(I):
    x1 = objects(I, T, F, F)
    x2 = argmax(x1, size)
    x3 = subgrid(x2, I)
    x4 = height(x3)
    x5 = width(x3)
    x6 = vsplit(x3, x4)
    x7 = lbind(greater, FOUR)
    x8 = compose(x7, numcolors)
    x9 = sfilter(x6, x8)
    x10 = merge(x9)
    x11 = rot90(x10)
    x12 = vsplit(x11, x5)
    x13 = sfilter(x12, x8)
    x14 = merge(x13)
    x15 = rot270(x14)
    x16 = leastcolor(x15)
    x17 = ofcolor(x15, x16)
    x18 = fork(combine, vfrontier, hfrontier)
    x19 = mapply(x18, x17)
    O = fill(x15, x16, x19)
    return O


def solve_272f95fa(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = apply(toindices, x2)
    x4 = rbind(bordering, I)
    x5 = compose(flip, x4)
    x6 = extract(x3, x5)
    x7 = remove(x6, x3)
    x8 = lbind(vmatching, x6)
    x9 = lbind(hmatching, x6)
    x10 = sfilter(x7, x8)
    x11 = sfilter(x7, x9)
    x12 = argmin(x10, uppermost)
    x13 = argmax(x10, uppermost)
    x14 = argmin(x11, leftmost)
    x15 = argmax(x11, leftmost)
    x16 = fill(I, SIX, x6)
    x17 = fill(x16, TWO, x12)
    x18 = fill(x17, ONE, x13)
    x19 = fill(x18, FOUR, x14)
    O = fill(x19, THREE, x15)
    return O


def solve_db93a21d(I):
    x1 = objects(I, T, T, T)
    x2 = ofcolor(I, NINE)
    x3 = colorfilter(x1, NINE)
    x4 = rbind(shoot, DOWN)
    x5 = mapply(x4, x2)
    x6 = underfill(I, ONE, x5)
    x7 = compose(halve, width)
    x8 = rbind(greater, ONE)
    x9 = compose(x8, x7)
    x10 = matcher(x7, THREE)
    x11 = power(outbox, TWO)
    x12 = power(outbox, THREE)
    x13 = mapply(outbox, x3)
    x14 = sfilter(x3, x9)
    x15 = sfilter(x3, x10)
    x16 = mapply(x11, x14)
    x17 = mapply(x12, x15)
    x18 = fill(x6, THREE, x13)
    x19 = fill(x18, THREE, x16)
    O = fill(x19, THREE, x17)
    return O


def solve_53b68214(I):
    x1 = width(I)
    x2 = objects(I, T, T, T)
    x3 = first(x2)
    x4 = vperiod(x3)
    x5 = toivec(x4)
    x6 = interval(ZERO, NINE, ONE)
    x7 = lbind(multiply, x5)
    x8 = apply(x7, x6)
    x9 = lbind(shift, x3)
    x10 = mapply(x9, x8)
    x11 = astuple(x1, x1)
    x12 = portrait(x3)
    x13 = shape(x3)
    x14 = add(DOWN, x13)
    x15 = decrement(x14)
    x16 = shift(x3, x15)
    x17 = branch(x12, x10, x16)
    x18 = canvas(ZERO, x11)
    x19 = paint(x18, x3)
    O = paint(x19, x17)
    return O


def solve_d6ad076f(I):
    x1 = objects(I, T, F, T)
    x2 = argmin(x1, size)
    x3 = argmax(x1, size)
    x4 = vmatching(x2, x3)
    x5 = branch(x4, DOWN, RIGHT)
    x6 = branch(x4, uppermost, leftmost)
    x7 = valmax(x1, x6)
    x8 = x6(x2)
    x9 = equality(x7, x8)
    x10 = branch(x9, NEG_ONE, ONE)
    x11 = multiply(x5, x10)
    x12 = inbox(x2)
    x13 = rbind(shoot, x11)
    x14 = mapply(x13, x12)
    x15 = underfill(I, EIGHT, x14)
    x16 = objects(x15, T, F, T)
    x17 = colorfilter(x16, EIGHT)
    x18 = rbind(bordering, I)
    x19 = mfilter(x17, x18)
    O = cover(x15, x19)
    return O


def solve_6cdd2623(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = prapply(connect, x2, x2)
    x4 = fgpartition(I)
    x5 = merge(x4)
    x6 = cover(I, x5)
    x7 = fork(either, hline, vline)
    x8 = box(x5)
    x9 = rbind(difference, x8)
    x10 = chain(positive, size, x9)
    x11 = fork(both, x7, x10)
    x12 = mfilter(x3, x11)
    O = fill(x6, x1, x12)
    return O


def solve_a3df8b1e(I):
    x1 = shape(I)
    x2 = ofcolor(I, ONE)
    x3 = first(x2)
    x4 = shoot(x3, UP_RIGHT)
    x5 = fill(I, ONE, x4)
    x6 = ofcolor(x5, ONE)
    x7 = urcorner(x6)
    x8 = shoot(x7, NEG_UNITY)
    x9 = fill(x5, ONE, x8)
    x10 = objects(x9, T, T, T)
    x11 = first(x10)
    x12 = subgrid(x11, x9)
    x13 = shape(x12)
    x14 = subtract(x13, DOWN)
    x15 = crop(x12, DOWN, x14)
    x16 = vconcat(x15, x15)
    x17 = vconcat(x16, x16)
    x18 = vconcat(x17, x17)
    x19 = hmirror(x18)
    x20 = crop(x19, ORIGIN, x1)
    O = hmirror(x20)
    return O


def solve_8d510a79(I):
    x1 = ofcolor(I, ONE)
    x2 = ofcolor(I, TWO)
    x3 = ofcolor(I, FIVE)
    x4 = uppermost(x3)
    x5 = chain(toivec, decrement, double)
    x6 = lbind(greater, x4)
    x7 = compose(x6, first)
    x8 = chain(invert, x5, x7)
    x9 = fork(shoot, identity, x8)
    x10 = compose(x5, x7)
    x11 = fork(shoot, identity, x10)
    x12 = lbind(matcher, x7)
    x13 = compose(x12, x7)
    x14 = fork(sfilter, x11, x13)
    x15 = mapply(x9, x1)
    x16 = mapply(x14, x2)
    x17 = underfill(I, TWO, x16)
    O = fill(x17, ONE, x15)
    return O


def solve_cdecee7f(I):
    x1 = objects(I, T, F, T)
    x2 = astuple(ONE, THREE)
    x3 = size(x1)
    x4 = order(x1, leftmost)
    x5 = apply(color, x4)
    x6 = rbind(canvas, UNITY)
    x7 = apply(x6, x5)
    x8 = merge(x7)
    x9 = dmirror(x8)
    x10 = subtract(NINE, x3)
    x11 = astuple(ONE, x10)
    x12 = canvas(ZERO, x11)
    x13 = hconcat(x9, x12)
    x14 = hsplit(x13, THREE)
    x15 = merge(x14)
    x16 = crop(x15, ORIGIN, x2)
    x17 = crop(x15, DOWN, x2)
    x18 = crop(x15, TWO_BY_ZERO, x2)
    x19 = vmirror(x17)
    x20 = vconcat(x16, x19)
    O = vconcat(x20, x18)
    return O


def solve_3345333e(I):
    x1 = leastcolor(I)
    x2 = ofcolor(I, x1)
    x3 = cover(I, x2)
    x4 = leastcolor(x3)
    x5 = ofcolor(x3, x4)
    x6 = neighbors(ORIGIN)
    x7 = mapply(neighbors, x6)
    x8 = vmirror(x5)
    x9 = lbind(shift, x8)
    x10 = apply(x9, x7)
    x11 = rbind(intersection, x5)
    x12 = compose(size, x11)
    x13 = argmax(x10, x12)
    O = fill(x3, x4, x13)
    return O


def solve_b190f7f5(I):
    x1 = portrait(I)
    x2 = branch(x1, vsplit, hsplit)
    x3 = x2(I, TWO)
    x4 = argmin(x3, numcolors)
    x5 = argmax(x3, numcolors)
    x6 = width(x5)
    x7 = rbind(repeat, x6)
    x8 = chain(dmirror, merge, x7)
    x9 = upscale(x5, x6)
    x10 = x8(x4)
    x11 = x8(x10)
    x12 = ofcolor(x11, ZERO)
    O = fill(x9, ZERO, x12)
    return O


def solve_caa06a1f(I):
    x1 = asobject(I)
    x2 = shape(I)
    x3 = decrement(x2)
    x4 = index(I, x3)
    x5 = double(x2)
    x6 = canvas(x4, x5)
    x7 = paint(x6, x1)
    x8 = objects(x7, F, F, T)
    x9 = first(x8)
    x10 = shift(x9, LEFT)
    x11 = vperiod(x10)
    x12 = hperiod(x10)
    x13 = neighbors(ORIGIN)
    x14 = lbind(mapply, neighbors)
    x15 = power(x14, TWO)
    x16 = x15(x13)
    x17 = astuple(x11, x12)
    x18 = lbind(multiply, x17)
    x19 = apply(x18, x16)
    x20 = lbind(shift, x10)
    x21 = mapply(x20, x19)
    O = paint(I, x21)
    return O


def solve_e21d9049(I):
    x1 = asindices(I)
    x2 = leastcolor(I)
    x3 = objects(I, T, F, T)
    x4 = ofcolor(I, x2)
    x5 = merge(x3)
    x6 = shape(x5)
    x7 = neighbors(ORIGIN)
    x8 = lbind(mapply, neighbors)
    x9 = power(x8, TWO)
    x10 = x9(x7)
    x11 = lbind(multiply, x6)
    x12 = lbind(shift, x5)
    x13 = apply(x11, x10)
    x14 = mapply(x12, x13)
    x15 = lbind(hmatching, x4)
    x16 = lbind(vmatching, x4)
    x17 = fork(either, x15, x16)
    x18 = compose(x17, initset)
    x19 = paint(I, x14)
    x20 = sfilter(x1, x18)
    x21 = difference(x1, x20)
    O = cover(x19, x21)
    return O


def solve_d89b689b(I):
    x1 = objects(I, T, F, T)
    x2 = ofcolor(I, EIGHT)
    x3 = sizefilter(x1, ONE)
    x4 = apply(initset, x2)
    x5 = lbind(argmin, x4)
    x6 = lbind(rbind, manhattan)
    x7 = compose(x5, x6)
    x8 = fork(recolor, color, x7)
    x9 = mapply(x8, x3)
    x10 = merge(x3)
    x11 = cover(I, x10)
    O = paint(x11, x9)
    return O


def solve_746b3537(I):
    x1 = chain(size, dedupe, first)
    x2 = x1(I)
    x3 = equality(x2, ONE)
    x4 = branch(x3, dmirror, identity)
    x5 = x4(I)
    x6 = objects(x5, T, F, F)
    x7 = order(x6, leftmost)
    x8 = apply(color, x7)
    x9 = repeat(x8, ONE)
    O = x4(x9)
    return O


def solve_63613498(I):
    x1 = crop(I, ORIGIN, THREE_BY_THREE)
    x2 = ofcolor(x1, ZERO)
    x3 = asindices(x1)
    x4 = difference(x3, x2)
    x5 = normalize(x4)
    x6 = objects(I, T, F, T)
    x7 = compose(toindices, normalize)
    x8 = matcher(x7, x5)
    x9 = mfilter(x6, x8)
    x10 = fill(I, FIVE, x9)
    x11 = asobject(x1)
    O = paint(x10, x11)
    return O


def solve_06df4c85(I):
    x1 = partition(I)
    x2 = mostcolor(I)
    x3 = ofcolor(I, x2)
    x4 = colorfilter(x1, ZERO)
    x5 = argmax(x1, size)
    x6 = difference(x1, x4)
    x7 = remove(x5, x6)
    x8 = merge(x7)
    x9 = product(x8, x8)
    x10 = power(first, TWO)
    x11 = compose(first, last)
    x12 = fork(equality, x10, x11)
    x13 = sfilter(x9, x12)
    x14 = compose(last, first)
    x15 = power(last, TWO)
    x16 = fork(connect, x14, x15)
    x17 = fork(recolor, color, x16)
    x18 = apply(x17, x13)
    x19 = fork(either, vline, hline)
    x20 = mfilter(x18, x19)
    x21 = paint(I, x20)
    O = fill(x21, x2, x3)
    return O


def solve_f9012d9b(I):
    x1 = objects(I, T, F, F)
    x2 = ofcolor(I, ZERO)
    x3 = lbind(contained, ZERO)
    x4 = chain(flip, x3, palette)
    x5 = mfilter(x1, x4)
    x6 = vsplit(I, TWO)
    x7 = hsplit(I, TWO)
    x8 = extract(x6, x4)
    x9 = extract(x7, x4)
    x10 = asobject(x8)
    x11 = asobject(x9)
    x12 = vperiod(x10)
    x13 = hperiod(x11)
    x14 = neighbors(ORIGIN)
    x15 = mapply(neighbors, x14)
    x16 = astuple(x12, x13)
    x17 = rbind(multiply, x16)
    x18 = apply(x17, x15)
    x19 = lbind(shift, x5)
    x20 = mapply(x19, x18)
    x21 = paint(I, x20)
    O = subgrid(x2, x21)
    return O


def solve_4522001f(I):
    x1 = objects(I, F, F, T)
    x2 = first(x1)
    x3 = toindices(x2)
    x4 = contained(ZERO_BY_TWO, x3)
    x5 = contained(TWO_BY_TWO, x3)
    x6 = contained(TWO_BY_ZERO, x3)
    x7 = astuple(NINE, NINE)
    x8 = canvas(ZERO, x7)
    x9 = astuple(THREE, ORIGIN)
    x10 = initset(x9)
    x11 = upscale(x10, TWO)
    x12 = upscale(x11, TWO)
    x13 = shape(x12)
    x14 = shift(x12, x13)
    x15 = combine(x12, x14)
    x16 = paint(x8, x15)
    x17 = rot90(x16)
    x18 = rot180(x16)
    x19 = rot270(x16)
    x20 = branch(x4, x17, x16)
    x21 = branch(x5, x18, x20)
    O = branch(x6, x19, x21)
    return O


def solve_a48eeaf7(I):
    x1 = ofcolor(I, TWO)
    x2 = outbox(x1)
    x3 = apply(initset, x2)
    x4 = ofcolor(I, FIVE)
    x5 = lbind(argmin, x3)
    x6 = lbind(lbind, manhattan)
    x7 = compose(x6, initset)
    x8 = compose(x5, x7)
    x9 = mapply(x8, x4)
    x10 = cover(I, x4)
    O = fill(x10, FIVE, x9)
    return O


def solve_eb5a1d5d(I):
    x1 = compose(dmirror, dedupe)
    x2 = x1(I)
    x3 = x1(x2)
    x4 = fork(remove, last, identity)
    x5 = compose(hmirror, x4)
    x6 = fork(vconcat, identity, x5)
    x7 = x6(x3)
    x8 = dmirror(x7)
    O = x6(x8)
    return O


def solve_e179c5f4(I):
    x1 = height(I)
    x2 = ofcolor(I, ONE)
    x3 = first(x2)
    x4 = shoot(x3, UP_RIGHT)
    x5 = fill(I, ONE, x4)
    x6 = ofcolor(x5, ONE)
    x7 = urcorner(x6)
    x8 = shoot(x7, NEG_UNITY)
    x9 = fill(x5, ONE, x8)
    x10 = ofcolor(x9, ONE)
    x11 = subgrid(x10, x9)
    x12 = height(x11)
    x13 = width(x11)
    x14 = decrement(x12)
    x15 = astuple(x14, x13)
    x16 = ulcorner(x10)
    x17 = crop(x9, x16, x15)
    x18 = repeat(x17, NINE)
    x19 = merge(x18)
    x20 = astuple(x1, x13)
    x21 = crop(x19, ORIGIN, x20)
    x22 = hmirror(x21)
    O = replace(x22, ZERO, EIGHT)
    return O


def solve_228f6490(I):
    x1 = objects(I, T, F, F)
    x2 = colorfilter(x1, ZERO)
    x3 = rbind(bordering, I)
    x4 = compose(flip, x3)
    x5 = sfilter(x2, x4)
    x6 = first(x5)
    x7 = last(x5)
    x8 = difference(x1, x2)
    x9 = compose(normalize, toindices)
    x10 = x9(x6)
    x11 = x9(x7)
    x12 = matcher(x9, x10)
    x13 = matcher(x9, x11)
    x14 = extract(x8, x12)
    x15 = extract(x8, x13)
    x16 = ulcorner(x6)
    x17 = ulcorner(x7)
    x18 = ulcorner(x14)
    x19 = ulcorner(x15)
    x20 = subtract(x16, x18)
    x21 = subtract(x17, x19)
    x22 = move(I, x14, x20)
    O = move(x22, x15, x21)
    return O


def solve_995c5fa3(I):
    x1 = hsplit(I, THREE)
    x2 = astuple(TWO, ONE)
    x3 = rbind(ofcolor, ZERO)
    x4 = compose(ulcorner, x3)
    x5 = compose(size, x3)
    x6 = matcher(x5, ZERO)
    x7 = matcher(x4, UNITY)
    x8 = matcher(x4, DOWN)
    x9 = matcher(x4, x2)
    x10 = rbind(multiply, THREE)
    x11 = power(double, TWO)
    x12 = compose(double, x6)
    x13 = chain(x11, double, x7)
    x14 = compose(x10, x8)
    x15 = compose(x11, x9)
    x16 = fork(add, x12, x13)
    x17 = fork(add, x14, x15)
    x18 = fork(add, x16, x17)
    x19 = rbind(canvas, UNITY)
    x20 = compose(x19, x18)
    x21 = apply(x20, x1)
    x22 = merge(x21)
    O = hupscale(x22, THREE)
    return O


def solve_d06dbe63(I):
    x1 = ofcolor(I, EIGHT)
    x2 = center(x1)
    x3 = connect(ORIGIN, DOWN)
    x4 = connect(ORIGIN, ZERO_BY_TWO)
    x5 = combine(x3, x4)
    x6 = subtract(x2, TWO_BY_ZERO)
    x7 = shift(x5, x6)
    x8 = astuple(NEG_TWO, TWO)
    x9 = interval(ZERO, FIVE, ONE)
    x10 = lbind(multiply, x8)
    x11 = apply(x10, x9)
    x12 = lbind(shift, x7)
    x13 = mapply(x12, x11)
    x14 = fill(I, FIVE, x13)
    x15 = rot180(x14)
    x16 = ofcolor(x15, EIGHT)
    x17 = center(x16)
    x18 = subtract(x17, x6)
    x19 = shift(x13, x18)
    x20 = toivec(NEG_TWO)
    x21 = shift(x19, x20)
    x22 = fill(x15, FIVE, x21)
    O = rot180(x22)
    return O


def solve_36fdfd69(I):
    x1 = upscale(I, TWO)
    x2 = objects(x1, T, T, T)
    x3 = colorfilter(x2, TWO)
    x4 = fork(manhattan, first, last)
    x5 = lbind(greater, FIVE)
    x6 = compose(x5, x4)
    x7 = product(x3, x3)
    x8 = sfilter(x7, x6)
    x9 = apply(merge, x8)
    x10 = mapply(delta, x9)
    x11 = fill(x1, FOUR, x10)
    x12 = merge(x3)
    x13 = paint(x11, x12)
    O = downscale(x13, TWO)
    return O


def solve_0a938d79(I):
    x1 = portrait(I)
    x2 = branch(x1, dmirror, identity)
    x3 = x2(I)
    x4 = fgpartition(x3)
    x5 = merge(x4)
    x6 = chain(double, decrement, width)
    x7 = x6(x5)
    x8 = compose(vfrontier, tojvec)
    x9 = lbind(mapply, x8)
    x10 = rbind(interval, x7)
    x11 = width(x3)
    x12 = rbind(x10, x11)
    x13 = chain(x9, x12, leftmost)
    x14 = fork(recolor, color, x13)
    x15 = mapply(x14, x4)
    x16 = paint(x3, x15)
    O = x2(x16)
    return O


def solve_045e512c(I):
    x1 = objects(I, T, T, T)
    x2 = argmax(x1, size)
    x3 = remove(x2, x1)
    x4 = lbind(shift, x2)
    x5 = lbind(mapply, x4)
    x6 = double(TEN)
    x7 = interval(FOUR, x6, FOUR)
    x8 = rbind(apply, x7)
    x9 = lbind(position, x2)
    x10 = lbind(rbind, multiply)
    x11 = chain(x8, x10, x9)
    x12 = compose(x5, x11)
    x13 = fork(recolor, color, x12)
    x14 = mapply(x13, x3)
    O = paint(I, x14)
    return O


def solve_82819916(I):
    x1 = objects(I, F, T, T)
    x2 = argmax(x1, size)
    x3 = remove(x2, x1)
    x4 = normalize(x2)
    x5 = compose(last, last)
    x6 = rbind(argmin, x5)
    x7 = compose(first, x6)
    x8 = fork(other, palette, x7)
    x9 = x7(x4)
    x10 = matcher(first, x9)
    x11 = sfilter(x4, x10)
    x12 = difference(x4, x11)
    x13 = compose(toivec, uppermost)
    x14 = lbind(shift, x11)
    x15 = lbind(shift, x12)
    x16 = compose(x14, x13)
    x17 = compose(x15, x13)
    x18 = fork(recolor, x7, x16)
    x19 = fork(recolor, x8, x17)
    x20 = fork(combine, x18, x19)
    x21 = mapply(x20, x3)
    O = paint(I, x21)
    return O


def solve_99fa7670(I):
    x1 = shape(I)
    x2 = objects(I, T, F, T)
    x3 = rbind(shoot, RIGHT)
    x4 = compose(x3, center)
    x5 = fork(recolor, color, x4)
    x6 = mapply(x5, x2)
    x7 = paint(I, x6)
    x8 = add(x1, DOWN_LEFT)
    x9 = initset(x8)
    x10 = recolor(ZERO, x9)
    x11 = objects(x7, T, F, T)
    x12 = insert(x10, x11)
    x13 = order(x12, uppermost)
    x14 = first(x13)
    x15 = remove(x10, x13)
    x16 = remove(x14, x13)
    x17 = compose(lrcorner, first)
    x18 = compose(lrcorner, last)
    x19 = fork(connect, x17, x18)
    x20 = compose(color, first)
    x21 = fork(recolor, x20, x19)
    x22 = pair(x15, x16)
    x23 = mapply(x21, x22)
    O = underpaint(x7, x23)
    return O


def solve_72322fa7(I):
    x1 = objects(I, F, T, T)
    x2 = matcher(numcolors, ONE)
    x3 = sfilter(x1, x2)
    x4 = difference(x1, x3)
    x5 = lbind(matcher, first)
    x6 = compose(x5, mostcolor)
    x7 = fork(sfilter, identity, x6)
    x8 = fork(difference, identity, x7)
    x9 = lbind(occurrences, I)
    x10 = compose(x9, x7)
    x11 = compose(x9, x8)
    x12 = compose(ulcorner, x8)
    x13 = fork(subtract, ulcorner, x12)
    x14 = lbind(rbind, add)
    x15 = compose(x14, x13)
    x16 = fork(apply, x15, x11)
    x17 = lbind(lbind, shift)
    x18 = compose(x17, normalize)
    x19 = fork(mapply, x18, x10)
    x20 = fork(mapply, x18, x16)
    x21 = mapply(x19, x4)
    x22 = mapply(x20, x4)
    x23 = paint(I, x21)
    O = paint(x23, x22)
    return O


def solve_855e0971(I):
    x1 = rot90(I)
    x2 = frontiers(I)
    x3 = sfilter(x2, hline)
    x4 = size(x3)
    x6 = positive(x4)
    x7 = branch(x6, identity, dmirror)
    x8 = x7(I)
    x9 = rbind(subgrid, x8)
    x10 = matcher(color, ZERO)
    x11 = compose(flip, x10)
    x12 = partition(x8)
    x13 = sfilter(x12, x11)
    x14 = rbind(ofcolor, ZERO)
    x15 = lbind(mapply, vfrontier)
    x16 = chain(x15, x14, x9)
    x17 = fork(shift, x16, ulcorner)
    x18 = fork(intersection, toindices, x17)
    x19 = mapply(x18, x13)
    x20 = fill(x8, ZERO, x19)
    O = x7(x20)
    return O


def solve_a78176bb(I):
    x1 = palette(I)
    x2 = objects(I, T, F, T)
    x3 = remove(ZERO, x1)
    x4 = other(x3, FIVE)
    x5 = colorfilter(x2, FIVE)
    x6 = lbind(index, I)
    x7 = compose(x6, urcorner)
    x8 = matcher(x7, FIVE)
    x9 = sfilter(x5, x8)
    x10 = difference(x5, x9)
    x11 = apply(urcorner, x9)
    x12 = apply(llcorner, x10)
    x13 = rbind(add, UP_RIGHT)
    x14 = rbind(add, DOWN_LEFT)
    x15 = apply(x13, x11)
    x16 = apply(x14, x12)
    x17 = rbind(shoot, UNITY)
    x18 = rbind(shoot, NEG_UNITY)
    x19 = fork(combine, x17, x18)
    x20 = mapply(x19, x15)
    x21 = mapply(x19, x16)
    x22 = combine(x20, x21)
    x23 = fill(I, x4, x22)
    O = replace(x23, FIVE, ZERO)
    return O


def solve_952a094c(I):
    x1 = objects(I, T, F, T)
    x2 = sizefilter(x1, ONE)
    x3 = argmax(x1, size)
    x4 = outbox(x3)
    x5 = corners(x4)
    x6 = lbind(rbind, manhattan)
    x7 = lbind(argmax, x2)
    x8 = chain(x7, x6, initset)
    x9 = compose(color, x8)
    x10 = fork(astuple, x9, identity)
    x11 = apply(x10, x5)
    x12 = merge(x2)
    x13 = cover(I, x12)
    O = paint(x13, x11)
    return O


def solve_6d58a25d(I):
    x1 = objects(I, T, T, T)
    x2 = argmax(x1, size)
    x3 = remove(x2, x1)
    x4 = merge(x3)
    x5 = color(x4)
    x6 = uppermost(x2)
    x7 = rbind(greater, x6)
    x8 = compose(x7, uppermost)
    x9 = rbind(vmatching, x2)
    x10 = fork(both, x9, x8)
    x11 = sfilter(x3, x10)
    x12 = increment(x6)
    x13 = rbind(greater, x12)
    x14 = compose(x13, first)
    x15 = rbind(sfilter, x14)
    x16 = chain(x15, vfrontier, center)
    x17 = mapply(x16, x11)
    O = underfill(I, x5, x17)
    return O


def solve_6aa20dc0(I):
    x1 = objects(I, F, T, T)
    x2 = argmax(x1, numcolors)
    x3 = normalize(x2)
    x4 = lbind(matcher, first)
    x5 = compose(x4, mostcolor)
    x6 = fork(sfilter, identity, x5)
    x7 = fork(difference, identity, x6)
    x8 = lbind(rbind, upscale)
    x9 = interval(ONE, FOUR, ONE)
    x10 = apply(x8, x9)
    x11 = initset(identity)
    x12 = insert(vmirror, x11)
    x13 = insert(hmirror, x12)
    x14 = insert(cmirror, x13)
    x15 = insert(dmirror, x14)
    x16 = fork(compose, first, last)
    x17 = lbind(occurrences, I)
    x18 = lbind(lbind, shift)
    x19 = compose(x17, x7)
    x20 = product(x15, x10)
    x21 = apply(x16, x20)
    x22 = rapply(x21, x3)
    x23 = fork(mapply, x18, x19)
    x24 = mapply(x23, x22)
    O = paint(I, x24)
    return O


def solve_e6721834(I):
    x1 = portrait(I)
    x2 = branch(x1, vsplit, hsplit)
    x3 = x2(I, TWO)
    x4 = order(x3, numcolors)
    x5 = first(x4)
    x6 = last(x4)
    x7 = objects(x6, F, F, T)
    x8 = merge(x7)
    x9 = mostcolor(x8)
    x10 = matcher(first, x9)
    x11 = compose(flip, x10)
    x12 = rbind(sfilter, x11)
    x13 = lbind(occurrences, x5)
    x14 = compose(x13, x12)
    x15 = chain(positive, size, x14)
    x16 = sfilter(x7, x15)
    x17 = chain(first, x13, x12)
    x18 = compose(ulcorner, x12)
    x19 = fork(subtract, x17, x18)
    x20 = fork(shift, identity, x19)
    x21 = apply(x20, x16)
    x22 = compose(decrement, width)
    x23 = chain(positive, decrement, x22)
    x24 = mfilter(x21, x23)
    O = paint(x5, x24)
    return O


def solve_447fd412(I):
    x1 = objects(I, F, T, T)
    x2 = argmax(x1, numcolors)
    x3 = normalize(x2)
    x4 = lbind(matcher, first)
    x5 = compose(x4, mostcolor)
    x6 = fork(sfilter, identity, x5)
    x7 = fork(difference, identity, x6)
    x8 = lbind(rbind, upscale)
    x9 = interval(ONE, FOUR, ONE)
    x10 = apply(x8, x9)
    x11 = lbind(recolor, ZERO)
    x12 = compose(x11, outbox)
    x13 = fork(combine, identity, x12)
    x14 = lbind(occurrences, I)
    x15 = lbind(rbind, subtract)
    x16 = lbind(apply, increment)
    x17 = lbind(lbind, shift)
    x18 = chain(x15, ulcorner, x7)
    x19 = chain(x14, x13, x7)
    x20 = fork(apply, x18, x19)
    x21 = compose(x16, x20)
    x22 = fork(mapply, x17, x21)
    x23 = rapply(x10, x3)
    x24 = mapply(x22, x23)
    O = paint(I, x24)
    return O


def solve_2bcee788(I):
    x1 = mostcolor(I)
    x2 = objects(I, T, F, T)
    x3 = replace(I, x1, THREE)
    x4 = argmax(x2, size)
    x5 = argmin(x2, size)
    x6 = position(x4, x5)
    x7 = first(x6)
    x8 = last(x6)
    x9 = subgrid(x4, x3)
    x10 = hline(x5)
    x11 = hmirror(x9)
    x12 = vmirror(x9)
    x13 = branch(x10, x11, x12)
    x14 = branch(x10, x7, ZERO)
    x15 = branch(x10, ZERO, x8)
    x16 = asobject(x13)
    x17 = matcher(first, THREE)
    x18 = compose(flip, x17)
    x19 = sfilter(x16, x18)
    x20 = ulcorner(x4)
    x21 = shape(x4)
    x22 = astuple(x14, x15)
    x23 = multiply(x21, x22)
    x24 = add(x20, x23)
    x25 = shift(x19, x24)
    O = paint(x3, x25)
    return O


def solve_776ffc46(I):
    x1 = objects(I, T, F, T)
    x2 = colorfilter(x1, FIVE)
    x3 = fork(equality, toindices, box)
    x4 = extract(x2, x3)
    x5 = inbox(x4)
    x6 = subgrid(x5, I)
    x7 = asobject(x6)
    x8 = matcher(first, ZERO)
    x9 = compose(flip, x8)
    x10 = sfilter(x7, x9)
    x11 = normalize(x10)
    x12 = toindices(x11)
    x13 = compose(toindices, normalize)
    x14 = matcher(x13, x12)
    x15 = mfilter(x1, x14)
    x16 = color(x11)
    O = fill(I, x16, x15)
    return O


def solve_f35d900a(I):
    x1 = objects(I, T, F, T)
    x2 = palette(I)
    x3 = remove(ZERO, x2)
    x4 = lbind(other, x3)
    x5 = compose(x4, color)
    x6 = fork(recolor, x5, outbox)
    x7 = mapply(x6, x1)
    x8 = mapply(toindices, x1)
    x9 = box(x8)
    x10 = difference(x9, x8)
    x11 = lbind(argmin, x8)
    x12 = rbind(compose, initset)
    x13 = lbind(rbind, manhattan)
    x14 = chain(x12, x13, initset)
    x15 = chain(initset, x11, x14)
    x16 = fork(manhattan, initset, x15)
    x17 = compose(even, x16)
    x18 = sfilter(x10, x17)
    x19 = paint(I, x7)
    O = fill(x19, FIVE, x18)
    return O


def solve_0dfd9992(I):
    x1 = height(I)
    x2 = width(I)
    x3 = partition(I)
    x4 = colorfilter(x3, ZERO)
    x5 = difference(x3, x4)
    x6 = merge(x5)
    x7 = astuple(x1, ONE)
    x8 = astuple(ONE, x2)
    x9 = decrement(x1)
    x10 = decrement(x2)
    x11 = toivec(x10)
    x12 = tojvec(x9)
    x13 = crop(I, x11, x8)
    x14 = crop(I, x12, x7)
    x15 = asobject(x14)
    x16 = asobject(x13)
    x17 = vperiod(x15)
    x18 = hperiod(x16)
    x19 = astuple(x17, x18)
    x20 = lbind(multiply, x19)
    x21 = neighbors(ORIGIN)
    x22 = mapply(neighbors, x21)
    x23 = apply(x20, x22)
    x24 = lbind(shift, x6)
    x25 = mapply(x24, x23)
    O = paint(I, x25)
    return O


def solve_29ec7d0e(I):
    x1 = height(I)
    x2 = width(I)
    x3 = partition(I)
    x4 = colorfilter(x3, ZERO)
    x5 = difference(x3, x4)
    x6 = merge(x5)
    x7 = astuple(x1, ONE)
    x8 = astuple(ONE, x2)
    x9 = decrement(x1)
    x10 = decrement(x2)
    x11 = toivec(x10)
    x12 = tojvec(x9)
    x13 = crop(I, x11, x8)
    x14 = crop(I, x12, x7)
    x15 = asobject(x14)
    x16 = asobject(x13)
    x17 = vperiod(x15)
    x18 = hperiod(x16)
    x19 = astuple(x17, x18)
    x20 = lbind(multiply, x19)
    x21 = neighbors(ORIGIN)
    x22 = mapply(neighbors, x21)
    x23 = apply(x20, x22)
    x24 = lbind(shift, x6)
    x25 = mapply(x24, x23)
    O = paint(I, x25)
    return O


def solve_36d67576(I):
    x1 = objects(I, F, F, T)
    x2 = argmax(x1, numcolors)
    x3 = astuple(TWO, FOUR)
    x4 = rbind(contained, x3)
    x5 = compose(x4, first)
    x6 = rbind(sfilter, x5)
    x7 = lbind(rbind, subtract)
    x8 = lbind(occurrences, I)
    x9 = lbind(lbind, shift)
    x10 = compose(x7, ulcorner)
    x11 = chain(x10, x6, normalize)
    x12 = chain(x8, x6, normalize)
    x13 = fork(apply, x11, x12)
    x14 = compose(x9, normalize)
    x15 = fork(mapply, x14, x13)
    x16 = astuple(cmirror, dmirror)
    x17 = astuple(hmirror, vmirror)
    x18 = combine(x16, x17)
    x19 = product(x18, x18)
    x20 = fork(compose, first, last)
    x21 = apply(x20, x19)
    x22 = totuple(x21)
    x23 = combine(x18, x22)
    x24 = rapply(x23, x2)
    x25 = mapply(x15, x24)
    O = paint(I, x25)
    return O


def solve_98cf29f8(I):
    x1 = fgpartition(I)
    x2 = fork(multiply, height, width)
    x3 = fork(equality, size, x2)
    x4 = extract(x1, x3)
    x5 = other(x1, x4)
    x6 = color(x5)
    x7 = rbind(greater, THREE)
    x8 = rbind(toobject, I)
    x9 = rbind(colorcount, x6)
    x10 = chain(x8, ineighbors, last)
    x11 = chain(x7, x9, x10)
    x12 = sfilter(x5, x11)
    x13 = outbox(x12)
    x14 = backdrop(x13)
    x15 = cover(I, x5)
    x16 = gravitate(x14, x4)
    x17 = shift(x14, x16)
    O = fill(x15, x6, x17)
    return O


def solve_469497ad(I):
    x1 = numcolors(I)
    x2 = decrement(x1)
    x3 = upscale(I, x2)
    x4 = objects(x3, F, F, T)
    x5 = argmin(x4, size)
    x6 = ulcorner(x5)
    x7 = llcorner(x5)
    x8 = shoot(x6, NEG_UNITY)
    x9 = shoot(x6, UNITY)
    x10 = shoot(x7, DOWN_LEFT)
    x11 = shoot(x7, UP_RIGHT)
    x12 = combine(x8, x9)
    x13 = combine(x10, x11)
    x14 = combine(x12, x13)
    x15 = underfill(x3, TWO, x14)
    x16 = objects(x15, T, F, T)
    x17 = argmax(x16, lrcorner)
    O = paint(x15, x17)
    return O


def solve_39e1d7f9(I):
    x1 = fgpartition(I)
    x2 = objects(I, T, F, T)
    x3 = order(x1, height)
    x4 = last(x3)
    x5 = remove(x4, x3)
    x6 = last(x5)
    x7 = color(x6)
    x8 = colorfilter(x2, x7)
    x9 = power(outbox, TWO)
    x10 = rbind(toobject, I)
    x11 = mostcolor(I)
    x12 = lbind(remove, x11)
    x13 = chain(size, x12, palette)
    x14 = chain(x13, x10, x9)
    x15 = argmax(x8, x14)
    x16 = ulcorner(x15)
    x17 = shape(x15)
    x18 = subtract(x16, x17)
    x19 = decrement(x18)
    x20 = multiply(x17, THREE)
    x21 = add(x20, TWO_BY_TWO)
    x22 = crop(I, x19, x21)
    x23 = asobject(x22)
    x24 = apply(ulcorner, x8)
    x25 = increment(x17)
    x26 = rbind(subtract, x25)
    x27 = apply(x26, x24)
    x28 = lbind(shift, x23)
    x29 = mapply(x28, x27)
    O = paint(I, x29)
    return O


def solve_484b58aa(I):
    x1 = height(I)
    x2 = width(I)
    x3 = partition(I)
    x4 = colorfilter(x3, ZERO)
    x5 = difference(x3, x4)
    x6 = merge(x5)
    x7 = astuple(x1, TWO)
    x8 = astuple(TWO, x2)
    x9 = power(decrement, TWO)
    x10 = x9(x1)
    x11 = x9(x2)
    x12 = toivec(x11)
    x13 = tojvec(x10)
    x14 = crop(I, x12, x8)
    x15 = crop(I, x13, x7)
    x16 = asobject(x15)
    x17 = asobject(x14)
    x18 = vperiod(x16)
    x19 = hperiod(x17)
    x20 = astuple(x18, x19)
    x21 = lbind(multiply, x20)
    x22 = neighbors(ORIGIN)
    x23 = mapply(neighbors, x22)
    x24 = apply(x21, x23)
    x25 = lbind(shift, x6)
    x26 = mapply(x25, x24)
    O = paint(I, x26)
    return O


def solve_3befdf3e(I):
    x1 = objects(I, F, F, T)
    x2 = leastcolor(I)
    x3 = palette(I)
    x4 = remove(ZERO, x3)
    x5 = other(x4, x2)
    x6 = switch(I, x2, x5)
    x7 = compose(width, inbox)
    x8 = lbind(power, outbox)
    x9 = compose(x8, x7)
    x10 = initset(x9)
    x11 = lbind(rapply, x10)
    x12 = chain(initset, first, x11)
    x13 = fork(rapply, x12, identity)
    x14 = compose(first, x13)
    x15 = compose(backdrop, x14)
    x16 = lbind(chain, backdrop)
    x17 = lbind(x16, inbox)
    x18 = compose(x17, x9)
    x19 = lbind(apply, initset)
    x20 = chain(x19, corners, x15)
    x21 = fork(mapply, x18, x20)
    x22 = fork(intersection, x15, x21)
    x23 = mapply(x15, x1)
    x24 = mapply(x22, x1)
    x25 = underfill(x6, x5, x23)
    O = fill(x25, ZERO, x24)
    return O


def solve_9aec4887(I):
    x1 = objects(I, F, T, T)
    x2 = argmin(x1, numcolors)
    x3 = other(x1, x2)
    x4 = subgrid(x3, I)
    x5 = normalize(x2)
    x6 = shift(x5, UNITY)
    x7 = toindices(x6)
    x8 = normalize(x3)
    x9 = lbind(argmin, x8)
    x11 = lbind(rbind, manhattan)
    x12 = rbind(compose, initset)
    x13 = chain(x12, x11, initset)
    x14 = chain(first, x9, x13)
    x15 = fork(astuple, x14, identity)
    x16 = apply(x15, x7)
    x17 = paint(x4, x16)
    x18 = fork(connect, ulcorner, lrcorner)
    x19 = x18(x7)
    x20 = fork(combine, identity, vmirror)
    x21 = x20(x19)
    x22 = intersection(x7, x21)
    O = fill(x17, EIGHT, x22)
    return O


def solve_49d1d64f(I):
    x1 = shape(I)
    x2 = add(x1, TWO)
    x3 = canvas(ZERO, x2)
    x4 = asobject(I)
    x5 = shift(x4, UNITY)
    x6 = paint(x3, x5)
    x7 = asindices(x3)
    x8 = fork(difference, box, corners)
    x9 = x8(x7)
    x10 = lbind(lbind, manhattan)
    x11 = rbind(compose, initset)
    x12 = chain(x11, x10, initset)
    x13 = lbind(argmin, x5)
    x14 = chain(first, x13, x12)
    x15 = fork(astuple, x14, identity)
    x16 = apply(x15, x9)
    O = paint(x6, x16)
    return O


def solve_57aa92db(I):
    x1 = objects(I, F, T, T)
    x2 = objects(I, T, F, T)
    x3 = lbind(lbind, colorcount)
    x4 = fork(apply, x3, palette)
    x5 = compose(maximum, x4)
    x6 = compose(minimum, x4)
    x7 = fork(subtract, x5, x6)
    x8 = argmax(x1, x7)
    x9 = leastcolor(x8)
    x10 = normalize(x8)
    x11 = matcher(first, x9)
    x12 = sfilter(x10, x11)
    x13 = ulcorner(x12)
    x14 = colorfilter(x2, x9)
    x15 = rbind(toobject, I)
    x16 = lbind(remove, ZERO)
    x17 = chain(first, x16, palette)
    x18 = chain(x17, x15, outbox)
    x19 = lbind(multiply, x13)
    x20 = compose(x19, width)
    x21 = fork(subtract, ulcorner, x20)
    x22 = lbind(shift, x10)
    x23 = compose(x22, x21)
    x24 = fork(upscale, x23, width)
    x25 = fork(recolor, x18, x24)
    x26 = mapply(x25, x14)
    x27 = paint(I, x26)
    x28 = merge(x2)
    O = paint(x27, x28)
    return O


def solve_aba27056(I):
    x1 = objects(I, T, F, T)
    x2 = mapply(toindices, x1)
    x3 = box(x2)
    x4 = difference(x3, x2)
    x5 = delta(x2)
    x6 = position(x5, x4)
    x7 = interval(ZERO, NINE, ONE)
    x8 = lbind(multiply, x6)
    x9 = apply(x8, x7)
    x10 = lbind(shift, x4)
    x11 = mapply(x10, x9)
    x12 = fill(I, FOUR, x5)
    x13 = fill(x12, FOUR, x11)
    x14 = corners(x4)
    x15 = ofcolor(x13, ZERO)
    x16 = rbind(toobject, x13)
    x17 = rbind(colorcount, ZERO)
    x18 = chain(x17, x16, dneighbors)
    x19 = matcher(x18, TWO)
    x20 = rbind(adjacent, x2)
    x21 = rbind(adjacent, x11)
    x22 = fork(both, x20, x21)
    x23 = compose(x22, initset)
    x24 = sfilter(x15, x19)
    x25 = sfilter(x24, x23)
    x26 = product(x14, x25)
    x27 = fork(subtract, last, first)
    x28 = fork(shoot, first, x27)
    x29 = mapply(x28, x26)
    O = fill(x13, FOUR, x29)
    return O


def solve_f1cefba8(I):
    x1 = palette(I)
    x2 = objects(I, F, F, T)
    x3 = ofcolor(I, ZERO)
    x4 = first(x2)
    x5 = ulcorner(x4)
    x6 = subgrid(x4, I)
    x7 = power(trim, TWO)
    x8 = x7(x6)
    x9 = asindices(x8)
    x10 = shift(x9, TWO_BY_TWO)
    x11 = fill(x6, ZERO, x10)
    x12 = leastcolor(x11)
    x13 = remove(ZERO, x1)
    x14 = other(x13, x12)
    x15 = ofcolor(x11, x12)
    x16 = shift(x15, x5)
    x17 = ofcolor(I, x12)
    x18 = uppermost(x17)
    x19 = lowermost(x17)
    x20 = matcher(first, x18)
    x21 = matcher(first, x19)
    x22 = fork(either, x20, x21)
    x23 = sfilter(x16, x22)
    x24 = difference(x16, x23)
    x25 = mapply(vfrontier, x23)
    x26 = mapply(hfrontier, x24)
    x27 = combine(x25, x26)
    x28 = intersection(x3, x27)
    x29 = fill(I, x14, x27)
    O = fill(x29, x12, x28)
    return O


def solve_1e32b0e9(I):
    x1 = height(I)
    x2 = mostcolor(I)
    x3 = asobject(I)
    x4 = subtract(x1, TWO)
    x5 = divide(x4, THREE)
    x6 = astuple(x5, x5)
    x7 = crop(I, ORIGIN, x6)
    x8 = partition(x7)
    x9 = matcher(color, ZERO)
    x10 = compose(flip, x9)
    x11 = extract(x8, x10)
    x12 = initset(x2)
    x13 = palette(x3)
    x14 = palette(x11)
    x15 = difference(x13, x14)
    x16 = difference(x15, x12)
    x17 = first(x16)
    x18 = interval(ZERO, THREE, ONE)
    x19 = product(x18, x18)
    x20 = totuple(x19)
    x21 = apply(first, x20)
    x22 = apply(last, x20)
    x23 = lbind(multiply, x5)
    x24 = apply(x23, x21)
    x25 = apply(x23, x22)
    x26 = papply(add, x24, x21)
    x27 = papply(add, x25, x22)
    x28 = papply(astuple, x26, x27)
    x29 = lbind(shift, x11)
    x30 = mapply(x29, x28)
    O = underfill(I, x17, x30)
    return O


def solve_28e73c20(I):
    x1 = width(I)
    x2 = astuple(ONE, TWO)
    x3 = astuple(TWO, TWO)
    x4 = astuple(TWO, ONE)
    x5 = astuple(THREE, ONE)
    x6 = canvas(THREE, UNITY)
    x7 = upscale(x6, FOUR)
    x8 = initset(DOWN)
    x9 = insert(UNITY, x8)
    x10 = insert(x2, x9)
    x11 = insert(x3, x10)
    x12 = fill(x7, ZERO, x11)
    x13 = vupscale(x6, FIVE)
    x14 = hupscale(x13, THREE)
    x15 = insert(x4, x9)
    x16 = insert(x5, x15)
    x17 = fill(x14, ZERO, x16)
    x18 = even(x1)
    x19 = branch(x18, x12, x17)
    x20 = canvas(ZERO, UNITY)
    x21 = lbind(hupscale, x20)
    x22 = chain(x21, decrement, height)
    x23 = rbind(hconcat, x6)
    x24 = compose(x23, x22)
    x25 = lbind(hupscale, x6)
    x26 = compose(x25, height)
    x27 = fork(vconcat, x24, rot90)
    x28 = fork(vconcat, x26, x27)
    x29 = subtract(x1, FOUR)
    x30 = power(x28, x29)
    O = x30(x19)
    return O


def solve_4c5c2cf0(I):
    x1 = objects(I, T, T, T)
    x2 = objects(I, F, T, T)
    x3 = first(x2)
    x4 = rbind(subgrid, I)
    x5 = fork(equality, identity, rot90)
    x6 = compose(x5, x4)
    x7 = extract(x1, x6)
    x8 = center(x7)
    x9 = subgrid(x3, I)
    x10 = hmirror(x9)
    x11 = objects(x10, F, T, T)
    x12 = first(x11)
    x13 = objects(x10, T, T, T)
    x14 = rbind(subgrid, x10)
    x15 = compose(x5, x14)
    x16 = extract(x13, x15)
    x17 = center(x16)
    x18 = subtract(x8, x17)
    x19 = shift(x12, x18)
    x20 = paint(I, x19)
    x21 = objects(x20, F, T, T)
    x22 = first(x21)
    x23 = subgrid(x22, x20)
    x24 = vmirror(x23)
    x25 = objects(x24, F, T, T)
    x26 = first(x25)
    x27 = objects(x24, T, T, T)
    x28 = color(x7)
    x29 = matcher(color, x28)
    x30 = extract(x27, x29)
    x31 = center(x30)
    x32 = subtract(x8, x31)
    x33 = shift(x26, x32)
    O = paint(x20, x33)
    return O


def solve_508bd3b6(I):
    x1 = width(I)
    x2 = objects(I, T, T, T)
    x3 = argmin(x2, size)
    x4 = argmax(x2, size)
    x5 = ulcorner(x3)
    x6 = urcorner(x3)
    x7 = index(I, x5)
    x8 = equality(x7, EIGHT)
    x9 = branch(x8, x5, x6)
    x10 = branch(x8, UNITY, DOWN_LEFT)
    x11 = multiply(x10, x1)
    x12 = double(x11)
    x13 = add(x9, x12)
    x14 = subtract(x9, x12)
    x15 = connect(x13, x14)
    x16 = fill(I, THREE, x15)
    x17 = paint(x16, x4)
    x18 = objects(x17, T, F, T)
    x19 = rbind(adjacent, x4)
    x20 = extract(x18, x19)
    x21 = first(x20)
    x22 = last(x21)
    x23 = flip(x8)
    x24 = branch(x23, UNITY, DOWN_LEFT)
    x25 = multiply(x24, x1)
    x26 = double(x25)
    x27 = add(x22, x26)
    x28 = subtract(x22, x26)
    x29 = connect(x27, x28)
    x30 = fill(x17, THREE, x29)
    x31 = paint(x30, x3)
    O = paint(x31, x4)
    return O


def solve_6d0160f0(I):
    x1 = ofcolor(I, FOUR)
    x2 = first(x1)
    x3 = first(x2)
    x4 = last(x2)
    x5 = greater(x3, THREE)
    x6 = greater(x3, SEVEN)
    x7 = greater(x4, THREE)
    x8 = greater(x4, SEVEN)
    x9 = branch(x5, FOUR, ZERO)
    x10 = branch(x6, EIGHT, x9)
    x11 = branch(x7, FOUR, ZERO)
    x12 = branch(x8, EIGHT, x11)
    x13 = astuple(x10, x12)
    x14 = initset(ZERO)
    x15 = insert(FOUR, x14)
    x16 = insert(EIGHT, x15)
    x17 = product(x16, x16)
    x18 = crop(I, ORIGIN, THREE_BY_THREE)
    x19 = asindices(x18)
    x20 = recolor(ZERO, x19)
    x21 = lbind(shift, x20)
    x22 = mapply(x21, x17)
    x23 = paint(I, x22)
    x24 = crop(I, x13, THREE_BY_THREE)
    x25 = replace(x24, FIVE, ZERO)
    x26 = ofcolor(x25, FOUR)
    x27 = first(x26)
    x28 = asindices(x25)
    x29 = toobject(x28, x25)
    x30 = multiply(x27, FOUR)
    x31 = shift(x29, x30)
    O = paint(x23, x31)
    return O


def solve_f8a8fe49(I):
    x1 = objects(I, T, F, T)
    x2 = replace(I, FIVE, ZERO)
    x3 = colorfilter(x1, TWO)
    x4 = first(x3)
    x5 = portrait(x4)
    x6 = branch(x5, hsplit, vsplit)
    x7 = branch(x5, vmirror, hmirror)
    x8 = ofcolor(I, TWO)
    x9 = subgrid(x8, I)
    x10 = trim(x9)
    x11 = x7(x10)
    x12 = x6(x11, TWO)
    x13 = compose(normalize, asobject)
    x14 = apply(x13, x12)
    x15 = last(x14)
    x16 = first(x14)
    x17 = ulcorner(x8)
    x18 = increment(x17)
    x19 = shift(x15, x18)
    x20 = shift(x16, x18)
    x21 = branch(x5, width, height)
    x22 = branch(x5, tojvec, toivec)
    x23 = x21(x15)
    x24 = double(x23)
    x25 = compose(x22, increment)
    x26 = x25(x23)
    x27 = invert(x26)
    x28 = x25(x24)
    x29 = shift(x19, x27)
    x30 = shift(x20, x28)
    x31 = paint(x2, x29)
    O = paint(x31, x30)
    return O


def solve_d07ae81c(I):
    x1 = objects(I, T, F, F)
    x2 = sizefilter(x1, ONE)
    x3 = apply(color, x2)
    x4 = difference(x1, x2)
    x5 = apply(color, x4)
    x6 = first(x5)
    x7 = last(x5)
    x8 = ofcolor(I, x6)
    x9 = ofcolor(I, x7)
    x10 = rbind(shoot, UNITY)
    x11 = rbind(shoot, NEG_UNITY)
    x12 = rbind(shoot, DOWN_LEFT)
    x13 = rbind(shoot, UP_RIGHT)
    x14 = fork(combine, x10, x11)
    x15 = fork(combine, x12, x13)
    x16 = fork(combine, x14, x15)
    x17 = compose(x16, center)
    x18 = mapply(x17, x2)
    x19 = intersection(x8, x18)
    x20 = intersection(x9, x18)
    x21 = first(x2)
    x22 = color(x21)
    x23 = center(x21)
    x24 = neighbors(x23)
    x25 = toobject(x24, I)
    x26 = mostcolor(x25)
    x27 = other(x3, x22)
    x28 = equality(x26, x6)
    x29 = branch(x28, x22, x27)
    x30 = branch(x28, x27, x22)
    x31 = fill(I, x29, x19)
    O = fill(x31, x30, x20)
    return O


def solve_6a1e5592(I):
    x1 = width(I)
    x2 = objects(I, T, F, T)
    x3 = astuple(FIVE, x1)
    x4 = crop(I, ORIGIN, x3)
    x5 = colorfilter(x2, FIVE)
    x6 = merge(x5)
    x7 = cover(I, x6)
    x8 = compose(toindices, normalize)
    x9 = apply(x8, x5)
    x10 = asindices(x4)
    x11 = ofcolor(x4, ZERO)
    x12 = ofcolor(x4, TWO)
    x13 = rbind(multiply, TEN)
    x14 = rbind(multiply, FIVE)
    x15 = rbind(intersection, x12)
    x16 = rbind(intersection, x11)
    x17 = rbind(intersection, x10)
    x18 = chain(x13, size, x15)
    x19 = chain(size, x16, delta)
    x20 = compose(x14, uppermost)
    x21 = chain(size, x16, outbox)
    x22 = chain(x13, size, x17)
    x23 = compose(invert, x18)
    x24 = fork(add, x22, x23)
    x25 = fork(subtract, x24, x21)
    x26 = fork(subtract, x25, x20)
    x27 = fork(subtract, x26, x19)
    x28 = rbind(apply, x10)
    x29 = lbind(lbind, shift)
    x30 = rbind(argmax, x27)
    x31 = chain(x30, x28, x29)
    x32 = mapply(x31, x9)
    O = fill(x7, ONE, x32)
    return O


def solve_0e206a2e(I):
    x1 = palette(I)
    x2 = objects(I, F, F, T)
    x3 = rbind(greater, ONE)
    x4 = compose(x3, numcolors)
    x5 = sfilter(x2, x4)
    x6 = remove(ZERO, x1)
    x7 = lbind(colorcount, I)
    x8 = argmax(x6, x7)
    x9 = remove(x8, x6)
    x10 = rbind(contained, x9)
    x11 = compose(x10, first)
    x12 = rbind(sfilter, x11)
    x13 = lbind(rbind, subtract)
    x14 = lbind(occurrences, I)
    x15 = lbind(lbind, shift)
    x16 = compose(x13, ulcorner)
    x17 = chain(x16, x12, normalize)
    x18 = chain(x14, x12, normalize)
    x19 = fork(apply, x17, x18)
    x20 = compose(x15, normalize)
    x21 = fork(mapply, x20, x19)
    x22 = astuple(cmirror, dmirror)
    x23 = astuple(hmirror, vmirror)
    x24 = combine(x22, x23)
    x25 = product(x24, x24)
    x26 = fork(compose, first, last)
    x27 = apply(x26, x25)
    x28 = totuple(x27)
    x29 = combine(x24, x28)
    x30 = lbind(rapply, x29)
    x31 = mapply(x30, x5)
    x32 = mapply(x21, x31)
    x33 = paint(I, x32)
    x34 = merge(x5)
    O = cover(x33, x34)
    return O


def solve_d22278a0(I):
    x1 = asindices(I)
    x2 = objects(I, T, F, T)
    x3 = fork(multiply, sign, identity)
    x4 = lbind(apply, x3)
    x5 = chain(even, maximum, x4)
    x6 = lbind(sfilter, x1)
    x7 = fork(add, first, last)
    x8 = rbind(remove, x2)
    x9 = compose(center, last)
    x10 = fork(subtract, first, x9)
    x11 = compose(x5, x10)
    x12 = lbind(rbind, equality)
    x13 = lbind(argmin, x2)
    x14 = chain(x7, x4, x10)
    x15 = lbind(lbind, astuple)
    x16 = lbind(rbind, astuple)
    x17 = lbind(compose, x11)
    x18 = lbind(compose, x14)
    x19 = compose(x18, x15)
    x20 = compose(x18, x16)
    x21 = compose(x13, x19)
    x22 = rbind(compose, x21)
    x23 = lbind(lbind, valmin)
    x24 = rbind(compose, x19)
    x25 = chain(x24, x23, x8)
    x26 = lbind(fork, greater)
    x27 = fork(x26, x25, x20)
    x28 = chain(x6, x17, x16)
    x29 = chain(x6, x22, x12)
    x30 = fork(intersection, x28, x29)
    x31 = compose(x6, x27)
    x32 = fork(intersection, x30, x31)
    x33 = fork(recolor, color, x32)
    x34 = mapply(x33, x2)
    O = paint(I, x34)
    return O


def solve_4290ef0e(I):
    x1 = mostcolor(I)
    x2 = fgpartition(I)
    x3 = objects(I, T, F, T)
    x4 = rbind(valmax, width)
    x5 = lbind(colorfilter, x3)
    x6 = compose(x5, color)
    x7 = compose(double, x4)
    x8 = lbind(prapply, manhattan)
    x9 = fork(x8, identity, identity)
    x10 = lbind(remove, ZERO)
    x11 = compose(x10, x9)
    x12 = rbind(branch, NEG_TWO)
    x13 = fork(x12, positive, decrement)
    x14 = chain(x13, minimum, x11)
    x15 = fork(add, x14, x7)
    x16 = compose(x15, x6)
    x17 = compose(invert, x16)
    x18 = order(x2, x17)
    x19 = rbind(argmin, centerofmass)
    x20 = compose(initset, vmirror)
    x21 = fork(insert, dmirror, x20)
    x22 = fork(insert, cmirror, x21)
    x23 = fork(insert, hmirror, x22)
    x24 = compose(x19, x23)
    x25 = apply(x24, x18)
    x26 = size(x2)
    x27 = apply(size, x2)
    x28 = contained(ONE, x27)
    x29 = increment(x26)
    x30 = branch(x28, x26, x29)
    x31 = double(x30)
    x32 = decrement(x31)
    x33 = apply(normalize, x25)
    x34 = interval(ZERO, x30, ONE)
    x35 = pair(x34, x34)
    x36 = mpapply(shift, x33, x35)
    x37 = astuple(x32, x32)
    x38 = canvas(x1, x37)
    x39 = paint(x38, x36)
    x40 = rot90(x39)
    x41 = paint(x40, x36)
    x42 = rot90(x41)
    x43 = paint(x42, x36)
    x44 = rot90(x43)
    O = paint(x44, x36)
    return O


def solve_50846271(I):
    x1 = ofcolor(I, TWO)
    x2 = prapply(connect, x1, x1)
    x3 = lbind(greater, SIX)
    x4 = compose(x3, size)
    x5 = fork(either, vline, hline)
    x6 = fork(both, x4, x5)
    x7 = mfilter(x2, x6)
    x8 = fill(I, TWO, x7)
    x9 = objects(x8, T, F, F)
    x10 = colorfilter(x9, TWO)
    x11 = valmax(x10, width)
    x12 = halve(x11)
    x13 = toivec(x12)
    x14 = tojvec(x12)
    x15 = rbind(add, ZERO_BY_TWO)
    x16 = rbind(add, TWO_BY_ZERO)
    x17 = rbind(subtract, ZERO_BY_TWO)
    x18 = rbind(subtract, TWO_BY_ZERO)
    x19 = rbind(colorcount, TWO)
    x20 = rbind(toobject, x8)
    x21 = compose(initset, x15)
    x22 = fork(insert, x16, x21)
    x23 = fork(insert, x17, x22)
    x24 = fork(insert, x18, x23)
    x25 = fork(combine, dneighbors, x24)
    x26 = chain(x19, x20, x25)
    x27 = rbind(argmax, x26)
    x28 = compose(x27, toindices)
    x29 = apply(x28, x10)
    x30 = rbind(add, x13)
    x31 = rbind(subtract, x13)
    x32 = rbind(add, x14)
    x33 = rbind(subtract, x14)
    x34 = fork(connect, x30, x31)
    x35 = fork(connect, x32, x33)
    x36 = fork(combine, x34, x35)
    x37 = mapply(x36, x29)
    x38 = fill(x8, EIGHT, x37)
    O = fill(x38, TWO, x1)
    return O


def solve_b527c5c6(I):
    x1 = objects(I, F, F, T)
    x2 = matcher(first, TWO)
    x3 = rbind(sfilter, x2)
    x4 = compose(lowermost, x3)
    x5 = compose(rightmost, x3)
    x6 = compose(uppermost, x3)
    x7 = compose(leftmost, x3)
    x8 = fork(equality, x4, lowermost)
    x9 = fork(equality, x5, rightmost)
    x10 = fork(equality, x6, uppermost)
    x11 = fork(equality, x7, leftmost)
    x12 = compose(invert, x10)
    x13 = compose(invert, x11)
    x14 = fork(add, x12, x8)
    x15 = fork(add, x13, x9)
    x16 = fork(astuple, x14, x15)
    x17 = compose(center, x3)
    x18 = fork(shoot, x17, x16)
    x19 = mapply(x18, x1)
    x20 = fill(I, TWO, x19)
    x21 = compose(vline, x18)
    x22 = sfilter(x1, x21)
    x23 = difference(x1, x22)
    x24 = chain(decrement, minimum, shape)
    x25 = compose(increment, x24)
    x26 = compose(invert, x24)
    x27 = rbind(interval, ONE)
    x28 = fork(x27, x26, x25)
    x29 = lbind(apply, toivec)
    x30 = lbind(apply, tojvec)
    x31 = lbind(lbind, shift)
    x32 = compose(x31, x18)
    x33 = compose(x29, x28)
    x34 = compose(x30, x28)
    x35 = fork(mapply, x32, x33)
    x36 = fork(mapply, x32, x34)
    x37 = mapply(x35, x23)
    x38 = mapply(x36, x22)
    x39 = combine(x37, x38)
    O = underfill(x20, THREE, x39)
    return O


def solve_150deff5(I):
    x1 = canvas(FIVE, TWO_BY_TWO)
    x2 = asobject(x1)
    x3 = occurrences(I, x2)
    x4 = lbind(shift, x2)
    x5 = mapply(x4, x3)
    x6 = fill(I, EIGHT, x5)
    x7 = canvas(FIVE, UNITY)
    x8 = astuple(TWO, ONE)
    x9 = canvas(EIGHT, x8)
    x10 = vconcat(x9, x7)
    x11 = asobject(x10)
    x12 = occurrences(x6, x11)
    x13 = lbind(shift, x11)
    x14 = mapply(x13, x12)
    x15 = fill(x6, TWO, x14)
    x16 = astuple(ONE, THREE)
    x17 = canvas(FIVE, x16)
    x18 = asobject(x17)
    x19 = occurrences(x15, x18)
    x20 = lbind(shift, x18)
    x21 = mapply(x20, x19)
    x22 = fill(x15, TWO, x21)
    x23 = hmirror(x10)
    x24 = asobject(x23)
    x25 = occurrences(x22, x24)
    x26 = lbind(shift, x24)
    x27 = mapply(x26, x25)
    x28 = fill(x22, TWO, x27)
    x29 = dmirror(x10)
    x30 = asobject(x29)
    x31 = occurrences(x28, x30)
    x32 = lbind(shift, x30)
    x33 = mapply(x32, x31)
    x34 = fill(x28, TWO, x33)
    x35 = vmirror(x29)
    x36 = asobject(x35)
    x37 = occurrences(x34, x36)
    x38 = lbind(shift, x36)
    x39 = mapply(x38, x37)
    O = fill(x34, TWO, x39)
    return O


def solve_b7249182(I):
    x1 = objects(I, T, F, T)
    x2 = merge(x1)
    x3 = portrait(x2)
    x4 = branch(x3, identity, dmirror)
    x5 = x4(I)
    x6 = objects(x5, T, F, T)
    x7 = order(x6, uppermost)
    x8 = first(x7)
    x9 = last(x7)
    x10 = color(x8)
    x11 = color(x9)
    x12 = compose(first, toindices)
    x13 = x12(x8)
    x14 = x12(x9)
    x15 = connect(x13, x14)
    x16 = centerofmass(x15)
    x17 = connect(x13, x16)
    x18 = fill(x5, x11, x15)
    x19 = fill(x18, x10, x17)
    x20 = add(x16, DOWN)
    x21 = initset(x16)
    x22 = insert(x20, x21)
    x23 = toobject(x22, x19)
    x24 = astuple(ZERO, NEG_TWO)
    x25 = shift(x23, ZERO_BY_TWO)
    x26 = shift(x23, x24)
    x27 = combine(x25, x26)
    x28 = ulcorner(x27)
    x29 = urcorner(x27)
    x30 = connect(x28, x29)
    x31 = shift(x30, UP)
    x32 = llcorner(x27)
    x33 = lrcorner(x27)
    x34 = connect(x32, x33)
    x35 = shift(x34, DOWN)
    x36 = paint(x19, x27)
    x37 = fill(x36, x10, x31)
    x38 = fill(x37, x11, x35)
    x39 = cover(x38, x22)
    O = x4(x39)
    return O


def solve_9d9215db(I):
    x1 = rot90(I)
    x2 = rot180(I)
    x3 = rot270(I)
    x4 = initset(I)
    x5 = chain(numcolors, lefthalf, tophalf)
    x6 = insert(x1, x4)
    x7 = insert(x2, x6)
    x8 = insert(x3, x7)
    x9 = argmax(x8, x5)
    x10 = vmirror(x9)
    x11 = papply(pair, x9, x10)
    x12 = lbind(apply, maximum)
    x13 = apply(x12, x11)
    x14 = partition(x13)
    x15 = sizefilter(x14, FOUR)
    x16 = apply(llcorner, x15)
    x17 = apply(lrcorner, x15)
    x18 = combine(x16, x17)
    x19 = cover(x13, x18)
    x20 = tojvec(NEG_TWO)
    x21 = rbind(add, ZERO_BY_TWO)
    x22 = rbind(add, x20)
    x23 = compose(x21, ulcorner)
    x24 = compose(x22, urcorner)
    x25 = fork(connect, x23, x24)
    x26 = compose(even, last)
    x27 = rbind(sfilter, x26)
    x28 = chain(normalize, x27, x25)
    x29 = fork(shift, x28, x23)
    x30 = fork(recolor, color, x29)
    x31 = mapply(x30, x15)
    x32 = paint(x19, x31)
    x33 = rot90(x32)
    x34 = rot180(x32)
    x35 = rot270(x32)
    x36 = papply(pair, x32, x33)
    x37 = apply(x12, x36)
    x38 = papply(pair, x37, x34)
    x39 = apply(x12, x38)
    x40 = papply(pair, x39, x35)
    O = apply(x12, x40)
    return O


def solve_6855a6e4(I):
    x1 = fgpartition(I)
    x2 = rot90(I)
    x3 = colorfilter(x1, TWO)
    x4 = first(x3)
    x5 = portrait(x4)
    x6 = branch(x5, I, x2)
    x7 = objects(x6, T, F, T)
    x8 = colorfilter(x7, FIVE)
    x9 = apply(center, x8)
    x10 = valmin(x9, first)
    x11 = compose(first, center)
    x12 = matcher(x11, x10)
    x13 = compose(flip, x12)
    x14 = extract(x8, x12)
    x15 = extract(x8, x13)
    x16 = ulcorner(x14)
    x17 = ulcorner(x15)
    x18 = subgrid(x14, x6)
    x19 = subgrid(x15, x6)
    x20 = hmirror(x18)
    x21 = hmirror(x19)
    x22 = ofcolor(x20, FIVE)
    x23 = recolor(FIVE, x22)
    x24 = ofcolor(x21, FIVE)
    x25 = recolor(FIVE, x24)
    x26 = height(x23)
    x27 = height(x25)
    x28 = add(THREE, x26)
    x29 = add(THREE, x27)
    x30 = toivec(x28)
    x31 = toivec(x29)
    x32 = add(x16, x30)
    x33 = subtract(x17, x31)
    x34 = shift(x23, x32)
    x35 = shift(x25, x33)
    x36 = merge(x8)
    x37 = cover(x6, x36)
    x38 = paint(x37, x34)
    x39 = paint(x38, x35)
    x40 = rot270(x39)
    O = branch(x5, x39, x40)
    return O


def solve_264363fd(I):
    x1 = objects(I, F, F, T)
    x2 = argmin(x1, size)
    x3 = normalize(x2)
    x4 = height(x2)
    x5 = width(x2)
    x6 = equality(x4, FIVE)
    x7 = equality(x5, FIVE)
    x8 = astuple(x6, x7)
    x9 = add(UNITY, x8)
    x10 = invert(x9)
    x11 = center(x2)
    x12 = index(I, x11)
    x13 = branch(x6, UP, RIGHT)
    x14 = add(x13, x11)
    x15 = index(I, x14)
    x16 = astuple(x12, ORIGIN)
    x17 = initset(x16)
    x18 = cover(I, x2)
    x19 = mostcolor(x18)
    x20 = ofcolor(x18, x19)
    x21 = occurrences(x18, x17)
    x22 = objects(x18, F, F, T)
    x23 = rbind(occurrences, x17)
    x24 = rbind(subgrid, x18)
    x25 = compose(x23, x24)
    x26 = lbind(mapply, vfrontier)
    x27 = lbind(mapply, hfrontier)
    x28 = compose(x26, x25)
    x29 = compose(x27, x25)
    x30 = branch(x6, x28, x29)
    x31 = branch(x7, x29, x28)
    x32 = fork(combine, x30, x31)
    x33 = lbind(recolor, x15)
    x34 = compose(x33, x32)
    x35 = fork(paint, x24, x34)
    x36 = compose(asobject, x35)
    x37 = fork(shift, x36, ulcorner)
    x38 = mapply(x37, x22)
    x39 = paint(x18, x38)
    x40 = shift(x3, x10)
    x41 = lbind(shift, x40)
    x42 = mapply(x41, x21)
    x43 = paint(x39, x42)
    O = fill(x43, x19, x20)
    return O


def solve_7df24a62(I):
    x1 = height(I)
    x2 = width(I)
    x3 = ofcolor(I, ONE)
    x4 = ofcolor(I, FOUR)
    x5 = ulcorner(x3)
    x6 = subgrid(x3, I)
    x7 = rot90(x6)
    x8 = rot180(x6)
    x9 = rot270(x6)
    x10 = matcher(size, ZERO)
    x11 = rbind(ofcolor, ONE)
    x12 = compose(normalize, x11)
    x13 = rbind(ofcolor, FOUR)
    x14 = rbind(shift, x5)
    x15 = compose(x14, x13)
    x16 = lbind(subtract, x1)
    x17 = chain(increment, x16, height)
    x18 = lbind(subtract, x2)
    x19 = chain(increment, x18, width)
    x20 = rbind(interval, ONE)
    x21 = lbind(x20, ZERO)
    x22 = compose(x21, x17)
    x23 = compose(x21, x19)
    x24 = fork(product, x22, x23)
    x25 = rbind(shift, NEG_UNITY)
    x26 = lbind(lbind, shift)
    x27 = chain(x26, x25, x12)
    x28 = astuple(x6, x7)
    x29 = astuple(x8, x9)
    x30 = combine(x28, x29)
    x31 = apply(x15, x30)
    x32 = lbind(difference, x4)
    x33 = apply(x32, x31)
    x34 = apply(normalize, x31)
    x35 = apply(x24, x34)
    x36 = lbind(rbind, difference)
    x37 = apply(x26, x34)
    x38 = apply(x36, x33)
    x39 = papply(compose, x38, x37)
    x40 = lbind(compose, x10)
    x41 = apply(x40, x39)
    x42 = papply(sfilter, x35, x41)
    x43 = apply(x27, x30)
    x44 = mpapply(mapply, x43, x42)
    O = fill(I, ONE, x44)
    return O


def solve_f15e1fac(I):
    x1 = ofcolor(I, TWO)
    x2 = portrait(x1)
    x3 = branch(x2, identity, dmirror)
    x4 = x3(I)
    x5 = leftmost(x1)
    x6 = equality(x5, ZERO)
    x7 = branch(x6, identity, vmirror)
    x8 = x7(x4)
    x9 = ofcolor(x8, EIGHT)
    x10 = uppermost(x9)
    x11 = equality(x10, ZERO)
    x12 = branch(x11, identity, hmirror)
    x13 = x12(x8)
    x14 = ofcolor(x13, EIGHT)
    x15 = ofcolor(x13, TWO)
    x16 = rbind(shoot, DOWN)
    x17 = mapply(x16, x14)
    x18 = height(x13)
    x19 = apply(first, x15)
    x20 = insert(ZERO, x19)
    x21 = insert(x18, x19)
    x22 = apply(decrement, x21)
    x23 = order(x20, identity)
    x24 = order(x22, identity)
    x25 = size(x15)
    x26 = increment(x25)
    x27 = interval(ZERO, x26, ONE)
    x28 = apply(tojvec, x27)
    x29 = pair(x23, x24)
    x30 = lbind(sfilter, x17)
    x31 = compose(first, last)
    x32 = chain(decrement, first, first)
    x33 = fork(greater, x31, x32)
    x34 = chain(increment, last, first)
    x35 = fork(greater, x34, x31)
    x36 = fork(both, x33, x35)
    x37 = lbind(lbind, astuple)
    x38 = lbind(compose, x36)
    x39 = chain(x30, x38, x37)
    x40 = apply(x39, x29)
    x41 = papply(shift, x40, x28)
    x42 = merge(x41)
    x43 = fill(x13, EIGHT, x42)
    x44 = chain(x3, x7, x12)
    O = x44(x43)
    return O


def solve_234bbc79(I):
    x1 = objects(I, F, F, T)
    x2 = rbind(other, FIVE)
    x3 = compose(x2, palette)
    x4 = fork(recolor, x3, identity)
    x5 = apply(x4, x1)
    x6 = order(x5, leftmost)
    x7 = compose(last, last)
    x8 = lbind(matcher, x7)
    x9 = compose(x8, leftmost)
    x10 = compose(x8, rightmost)
    x11 = fork(sfilter, identity, x9)
    x12 = fork(sfilter, identity, x10)
    x13 = compose(dneighbors, last)
    x14 = rbind(chain, x13)
    x15 = lbind(x14, size)
    x16 = lbind(rbind, intersection)
    x17 = chain(x15, x16, toindices)
    x18 = fork(argmin, x11, x17)
    x19 = fork(argmin, x12, x17)
    x20 = compose(last, x18)
    x21 = compose(last, x19)
    x22 = astuple(ZERO, DOWN_LEFT)
    x23 = initset(x22)
    x24 = lbind(add, RIGHT)
    x25 = chain(x20, first, last)
    x26 = compose(x21, first)
    x27 = fork(subtract, x26, x25)
    x28 = compose(first, last)
    x29 = compose(x24, x27)
    x30 = fork(shift, x28, x29)
    x31 = fork(combine, first, x30)
    x32 = fork(remove, x28, last)
    x33 = fork(astuple, x31, x32)
    x34 = size(x1)
    x35 = power(x33, x34)
    x36 = astuple(x23, x6)
    x37 = x35(x36)
    x38 = first(x37)
    x39 = width(x38)
    x40 = decrement(x39)
    x41 = astuple(THREE, x40)
    x42 = canvas(ZERO, x41)
    O = paint(x42, x38)
    return O


def solve_22233c11(I):
    x1 = objects(I, T, T, T)
    x2 = rbind(upscale, TWO)
    x3 = chain(invert, halve, shape)
    x4 = fork(combine, hfrontier, vfrontier)
    x5 = compose(x2, vmirror)
    x6 = fork(shift, x5, x3)
    x7 = compose(toindices, x6)
    x8 = lbind(mapply, x4)
    x9 = compose(x8, toindices)
    x10 = fork(difference, x7, x9)
    x11 = mapply(x10, x1)
    O = fill(I, EIGHT, x11)
    return O


def solve_2dd70a9a(I):
    x1 = ofcolor(I, TWO)
    x2 = ofcolor(I, THREE)
    x3 = vline(x1)
    x4 = vline(x2)
    x5 = center(x1)
    x6 = branch(x4, uppermost, rightmost)
    x7 = x6(x1)
    x8 = x6(x2)
    x9 = greater(x7, x8)
    x10 = both(x4, x9)
    x11 = branch(x10, lowermost, uppermost)
    x12 = x11(x2)
    x13 = branch(x4, leftmost, rightmost)
    x14 = x13(x2)
    x15 = astuple(x12, x14)
    x16 = other(x2, x15)
    x17 = subtract(x15, x16)
    x18 = shoot(x15, x17)
    x19 = underfill(I, ONE, x18)
    x20 = objects(x19, T, F, F)
    x21 = colorfilter(x20, ONE)
    x22 = rbind(adjacent, x2)
    x23 = sfilter(x21, x22)
    x24 = difference(x21, x23)
    x25 = merge(x24)
    x26 = cover(x19, x25)
    x27 = shoot(x5, DOWN)
    x28 = shoot(x5, UP)
    x29 = shoot(x5, LEFT)
    x30 = shoot(x5, RIGHT)
    x31 = combine(x27, x28)
    x32 = combine(x29, x30)
    x33 = branch(x3, x31, x32)
    x34 = ofcolor(x26, ONE)
    x35 = initset(x15)
    x36 = rbind(manhattan, x35)
    x37 = compose(x36, initset)
    x38 = argmax(x34, x37)
    x39 = initset(x38)
    x40 = gravitate(x39, x33)
    x41 = crement(x40)
    x42 = add(x38, x41)
    x43 = connect(x38, x42)
    x44 = fill(x26, ONE, x43)
    x45 = connect(x42, x5)
    x46 = underfill(x44, ONE, x45)
    O = replace(x46, ONE, THREE)
    return O


def solve_a64e4611(I):
    x1 = asindices(I)
    x2 = fork(product, identity, identity)
    x3 = lbind(canvas, ZERO)
    x4 = compose(asobject, x3)
    x5 = fork(multiply, first, last)
    x6 = compose(positive, size)
    x7 = lbind(lbind, shift)
    x8 = rbind(fork, x5)
    x9 = lbind(x8, multiply)
    x10 = lbind(chain, x6)
    x11 = rbind(x10, x4)
    x12 = lbind(lbind, occurrences)
    x13 = chain(x9, x11, x12)
    x14 = compose(x2, first)
    x15 = compose(x13, last)
    x16 = fork(argmax, x14, x15)
    x17 = chain(x7, x4, x16)
    x18 = compose(x4, x16)
    x19 = fork(occurrences, last, x18)
    x20 = fork(mapply, x17, x19)
    x21 = multiply(TWO, SIX)
    x22 = interval(THREE, x21, ONE)
    x23 = astuple(x22, I)
    x24 = x20(x23)
    x25 = fill(I, THREE, x24)
    x26 = interval(THREE, TEN, ONE)
    x27 = astuple(x26, x25)
    x28 = x20(x27)
    x29 = fill(x25, THREE, x28)
    x30 = astuple(x26, x29)
    x31 = x20(x30)
    x32 = fill(x29, THREE, x31)
    x33 = rbind(toobject, x32)
    x34 = rbind(colorcount, THREE)
    x35 = chain(x34, x33, neighbors)
    x36 = matcher(x35, EIGHT)
    x37 = sfilter(x1, x36)
    x38 = fill(I, THREE, x37)
    x39 = ofcolor(x38, ZERO)
    x40 = rbind(bordering, x38)
    x41 = compose(x40, initset)
    x42 = lbind(contained, THREE)
    x43 = rbind(toobject, x38)
    x44 = chain(x42, palette, x43)
    x45 = compose(x44, dneighbors)
    x46 = fork(both, x45, x41)
    x47 = sfilter(x39, x46)
    O = fill(x38, THREE, x47)
    return O


def solve_7837ac64(I):
    x1 = fgpartition(I)
    x2 = argmax(x1, size)
    x3 = remove(x2, x1)
    x4 = merge(x3)
    x5 = subgrid(x4, I)
    x6 = chain(color, merge, frontiers)
    x7 = x6(I)
    x8 = objects(x5, T, F, F)
    x9 = colorfilter(x8, ZERO)
    x10 = rbind(toobject, x5)
    x11 = chain(x10, corners, outbox)
    x12 = lbind(contained, x7)
    x13 = chain(x12, palette, x11)
    x14 = compose(numcolors, x11)
    x15 = compose(flip, x13)
    x16 = matcher(x14, ONE)
    x17 = fork(both, x15, x16)
    x18 = sfilter(x9, x17)
    x19 = compose(color, x11)
    x20 = fork(recolor, x19, identity)
    x21 = mapply(x20, x18)
    x22 = paint(x5, x21)
    x23 = first(x9)
    x24 = height(x23)
    x25 = height(x5)
    x26 = increment(x24)
    x27 = interval(ZERO, x25, x26)
    x28 = interval(ZERO, x25, ONE)
    x29 = rbind(contained, x27)
    x30 = chain(flip, x29, last)
    x31 = lbind(apply, first)
    x32 = rbind(sfilter, x30)
    x33 = rbind(pair, x28)
    x34 = chain(x31, x32, x33)
    x35 = compose(dmirror, x34)
    x36 = power(x35, TWO)
    x37 = x36(x22)
    O = downscale(x37, x24)
    return O


def solve_a8c38be5(I):
    x1 = replace(I, FIVE, ZERO)
    x2 = objects(x1, T, F, T)
    x3 = apply(normalize, x2)
    x4 = astuple(NINE, NINE)
    x5 = canvas(FIVE, x4)
    x6 = asindices(x5)
    x7 = box(x6)
    x8 = center(x6)
    x9 = lbind(contained, ZERO)
    x10 = rbind(subtract, x8)
    x11 = compose(x9, x10)
    x12 = chain(outbox, outbox, initset)
    x13 = corners(x6)
    x14 = mapply(x12, x13)
    x15 = difference(x7, x14)
    x16 = inbox(x7)
    x17 = sfilter(x16, x11)
    x18 = combine(x15, x17)
    x19 = fill(x5, ONE, x18)
    x20 = objects(x19, T, F, T)
    x21 = apply(toindices, x20)
    x22 = lbind(matcher, normalize)
    x23 = lbind(extract, x21)
    x24 = chain(ulcorner, x23, x22)
    x25 = compose(x24, toindices)
    x26 = fork(shift, identity, x25)
    x27 = mapply(x26, x3)
    O = paint(x5, x27)
    return O


def solve_b775ac94(I):
    x1 = objects(I, F, T, T)
    x2 = lbind(rbind, equality)
    x3 = rbind(compose, first)
    x4 = chain(x3, x2, mostcolor)
    x5 = fork(sfilter, identity, x4)
    x6 = fork(difference, identity, x5)
    x7 = lbind(rbind, adjacent)
    x8 = rbind(compose, initset)
    x9 = chain(x8, x7, x6)
    x10 = fork(extract, x5, x9)
    x11 = fork(insert, x10, x6)
    x12 = lbind(recolor, ZERO)
    x13 = chain(x12, delta, x11)
    x14 = fork(combine, x11, x13)
    x15 = fork(position, x5, x6)
    x16 = chain(toivec, first, x15)
    x17 = chain(tojvec, last, x15)
    x18 = fork(multiply, shape, x16)
    x19 = fork(multiply, shape, x17)
    x20 = fork(multiply, shape, x15)
    x21 = fork(shift, hmirror, x18)
    x22 = fork(shift, vmirror, x19)
    x23 = compose(hmirror, vmirror)
    x24 = fork(shift, x23, x20)
    x25 = lbind(compose, x5)
    x26 = x25(x21)
    x27 = x25(x22)
    x28 = x25(x24)
    x29 = compose(crement, invert)
    x30 = lbind(compose, x29)
    x31 = x30(x16)
    x32 = x30(x17)
    x33 = x30(x15)
    x34 = fork(shift, x26, x31)
    x35 = fork(shift, x27, x32)
    x36 = fork(shift, x28, x33)
    x37 = lbind(index, I)
    x38 = lbind(compose, toindices)
    x39 = x38(x14)
    x40 = x38(x34)
    x41 = x38(x35)
    x42 = x38(x36)
    x43 = fork(intersection, x39, x40)
    x44 = fork(intersection, x39, x41)
    x45 = fork(intersection, x39, x42)
    x46 = chain(x37, first, x43)
    x47 = chain(x37, first, x44)
    x48 = chain(x37, first, x45)
    x49 = fork(recolor, x46, x34)
    x50 = fork(recolor, x47, x35)
    x51 = fork(recolor, x48, x36)
    x52 = mapply(x49, x1)
    x53 = mapply(x50, x1)
    x54 = mapply(x51, x1)
    x55 = paint(I, x52)
    x56 = paint(x55, x53)
    O = paint(x56, x54)
    return O


def solve_97a05b5b(I):
    x1 = objects(I, F, T, T)
    x2 = argmax(x1, size)
    x3 = subgrid(x2, I)
    x4 = rbind(greater, ONE)
    x5 = compose(x4, numcolors)
    x6 = sfilter(x1, x5)
    x7 = lbind(rbind, subtract)
    x8 = switch(x3, TWO, ZERO)
    x9 = lbind(occurrences, x8)
    x10 = lbind(lbind, shift)
    x11 = compose(x7, ulcorner)
    x12 = matcher(first, TWO)
    x13 = compose(flip, x12)
    x14 = rbind(sfilter, x12)
    x15 = rbind(sfilter, x13)
    x16 = lbind(recolor, ZERO)
    x17 = compose(x16, x15)
    x18 = fork(combine, x17, x14)
    x19 = chain(x11, x18, normalize)
    x20 = objects(x8, T, T, T)
    x21 = apply(toindices, x20)
    x22 = chain(x9, x18, normalize)
    x23 = rbind(colorcount, TWO)
    x24 = lbind(sfilter, x21)
    x25 = chain(size, first, x24)
    x26 = compose(positive, size)
    x27 = lbind(lbind, contained)
    x28 = chain(x26, x24, x27)
    x29 = compose(x25, x27)
    x30 = rbind(sfilter, x28)
    x31 = compose(x30, x22)
    x32 = lbind(rbind, equality)
    x33 = rbind(compose, x29)
    x34 = chain(x33, x32, x23)
    x35 = fork(sfilter, x31, x34)
    x36 = fork(apply, x19, x35)
    x37 = compose(x10, normalize)
    x38 = fork(mapply, x37, x36)
    x39 = astuple(cmirror, dmirror)
    x40 = astuple(hmirror, vmirror)
    x41 = combine(x39, x40)
    x42 = product(x41, x41)
    x43 = fork(compose, first, last)
    x44 = apply(x43, x42)
    x45 = lbind(rapply, x44)
    x46 = mapply(x45, x6)
    x47 = mapply(x38, x46)
    x48 = paint(x3, x47)
    x49 = palette(x47)
    x50 = lbind(remove, TWO)
    x51 = x50(x49)
    x52 = chain(first, x50, palette)
    x53 = rbind(contained, x51)
    x54 = chain(flip, x53, x52)
    x55 = sfilter(x6, x54)
    x56 = fork(apply, x19, x22)
    x57 = fork(mapply, x37, x56)
    x58 = mapply(x45, x55)
    x59 = mapply(x57, x58)
    O = paint(x48, x59)
    return O


def solve_3e980e27(I):
    x1 = objects(I, F, T, T)
    x2 = astuple(TEN, TEN)
    x3 = invert(x2)
    x4 = astuple(TWO, x3)
    x5 = astuple(THREE, x3)
    x6 = initset(x4)
    x7 = insert(x5, x6)
    x8 = insert(x7, x1)
    x9 = lbind(contained, TWO)
    x10 = lbind(contained, THREE)
    x11 = compose(invert, ulcorner)
    x12 = lbind(compose, x11)
    x13 = lbind(rbind, sfilter)
    x14 = compose(x12, x13)
    x15 = rbind(compose, center)
    x16 = lbind(lbind, shift)
    x17 = x14(x9)
    x18 = x14(x10)
    x19 = fork(shift, identity, x17)
    x20 = fork(shift, identity, x18)
    x21 = compose(x9, palette)
    x22 = compose(x10, palette)
    x23 = sfilter(x8, x21)
    x24 = argmax(x23, size)
    x25 = remove(x24, x23)
    x26 = vmirror(x24)
    x27 = chain(x15, x16, x19)
    x28 = x27(x26)
    x29 = mapply(x28, x25)
    x30 = sfilter(x8, x22)
    x31 = argmax(x30, size)
    x32 = remove(x31, x30)
    x33 = chain(x15, x16, x20)
    x34 = x33(x31)
    x35 = mapply(x34, x32)
    x36 = combine(x29, x35)
    O = paint(I, x36)
    return O


In [ ]:
%%writefile /kaggle/working/cpu_solver/hodel/LICENSE
MIT License

Copyright (c) 2023 Michael Hodel

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


In [ ]:
# The CPU DSL solver ALONE from minute 0 (c23f): all its workers until +cpu_alone_min, then flash_smoke kills it before
# Flash-Next pins its RAM. Answers -> /kaggle/working/cpu_answers.json (atomic, after every task).
import json, os, subprocess, sys
CPU_PROC = None
TIMING["cpu_workers"] = 0
if RUN and FLX.get("cpu"):
    try:
        _n = max(4, min(FLX["cpu_workers_max"], (os.cpu_count() or 8) - 8))
        _ccfg = {"challenges": CHALLENGES, "out_path": "/kaggle/working/cpu_answers.json", "workers": _n,
                 "budget_s": FLX["cpu_budget_s"], "hard_s": FLX["cpu_budget_s"] * 1.5 + 60, "nice": 15,
                 "stop_epoch": START + FLX["cpu_alone_min"] * 60, "log_path": "/kaggle/working/cpu_log.jsonl",
                 "loo_s": FLX.get("cpu_loo_s", 30)}
        CPU_PROC = subprocess.Popen([sys.executable, "/kaggle/working/cpu_solver/cpu_runner.py", json.dumps(_ccfg)],
                                    stdout=open("/kaggle/working/cpu_runner.out", "w"), stderr=subprocess.STDOUT,
                                    start_new_session=True)
        open("/kaggle/working/cpu_solver.pid", "w").write(str(CPU_PROC.pid))
        TIMING["cpu_workers"] = _n
        print(f"CPU solver started: {_n} workers of {os.cpu_count()} CPUs, until +{FLX['cpu_alone_min']} min")
    except Exception as e:
        print("CPU solver NOT started (the agent is unaffected):", repr(e))

In [ ]:
%%writefile /kaggle/working/agent/grids.py
"""Grid <-> text, task prompts and failure feedback for the program-synthesis agent.

Grids are shown as rows of digits with no separators, the same layout NVARC's
formatter uses, plus explicit dimensions -- reasoning models miscount rows and
columns of long digit strings otherwise.
"""

SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles by writing Python programs. "
    "Each puzzle has a few input->output example pairs that share one hidden "
    "transformation rule. Grids are 2D lists of integers 0-9 (colours); 0 is "
    "usually the background."
)

INSTRUCTIONS = """Find the transformation rule that maps EVERY example input to its output, then write it as a Python function.

Requirements:
- Define `def transform(grid: list[list[int]]) -> list[list[int]]:`.
- Use only the Python standard library and numpy (already importable as `np` if you `import numpy as np`).
- The function must work for every example input AND the test input(s); do not hard-code example outputs.
- Return a new 2D list of ints 0-9 (numpy arrays are accepted).

Reason about the rule first, then give the final program in ONE ```python code block."""


def grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in g)


def shape(g):
    return (len(g), len(g[0]) if g else 0)


def show(g, name):
    r, c = shape(g)
    return f"{name} ({r}x{c}):\n{grid_text(g)}"


def task_prompt(task):
    parts = [INSTRUCTIONS, ""]
    for i, pair in enumerate(task["train"], 1):
        parts.append(f"## Example {i}")
        parts.append(show(pair["input"], "Input"))
        parts.append(show(pair["output"], "Output"))
        parts.append("")
    for j, t in enumerate(task["test"], 1):
        parts.append(f"## Test input {j}")
        parts.append(show(t["input"], "Input"))
        parts.append("")
    return "\n".join(parts).rstrip()


def diff_text(expected, got, max_cells=900):
    """Expected grid with wrong cells marked, so the model sees WHERE it failed.

    '.' marks a correct cell; a wrong cell shows `expected/got` inline is too
    wide, so two aligned grids are given: the expected output, and the
    program's output with correct cells replaced by '.'.
    """
    re_, ce = shape(expected)
    rg, cg = shape(got)
    if (re_, ce) != (rg, cg):
        return (f"Wrong output SIZE: expected {re_}x{ce}, your program returned {rg}x{cg}.\n"
                + show(expected, "Expected") + "\n" + show(got, "Your output"))
    if re_ * ce > max_cells:
        wrong = sum(expected[r][c] != got[r][c] for r in range(re_) for c in range(ce))
        return f"{wrong} of {re_ * ce} cells are wrong (grid too large to print the diff)."
    wrong = 0
    rows = []
    for r in range(re_):
        s = []
        for c in range(ce):
            if expected[r][c] == got[r][c]:
                s.append(".")
            else:
                s.append(str(int(got[r][c])))
                wrong += 1
        rows.append("".join(s))
    return (f"{wrong} of {re_ * ce} cells are wrong.\n"
            + show(expected, "Expected") + "\n"
            + f"Your output, with CORRECT cells shown as '.' ({re_}x{ce}):\n" + "\n".join(rows))


def feedback_prompt(results, train, max_examples=3):
    """Tell the model which examples failed and how; ask for a corrected program."""
    lines = []
    n_ok = sum(r["ok"] for r in results)
    lines.append(f"Your program reproduced {n_ok} of {len(results)} examples.")
    shown = 0
    for i, (r, pair) in enumerate(zip(results, train), 1):
        if r["ok"]:
            continue
        if shown >= max_examples:
            break
        shown += 1
        lines.append(f"\n## Example {i} failed")
        if r.get("error"):
            lines.append(f"Error: {r['error']}")
        else:
            lines.append(diff_text(pair["output"], r["output"]))
    lines.append("\nRe-examine the rule against ALL examples, then give the corrected "
                 "complete program in ONE ```python code block.")
    return "\n".join(lines)

In [ ]:
%%writefile /kaggle/working/agent/client.py
"""Minimal OpenAI-compatible chat client for a local vLLM server (stdlib only).

Returns a normalised dict: content, reasoning, tool_calls [{id, name, arguments}],
finish_reason, prompt_tokens, completion_tokens. If the server's tool parser
misses a call that the model wrote as qwen3_coder markup in its text, the call
is recovered from the markup, as the Tufa harness does.
"""
import json
import re
import urllib.error
import urllib.request

TOOL_BLOCK = re.compile(r"<tool_call>\s*<function=([\w\-]+)>(.*?)</function>\s*</tool_call>", re.S)
PARAM = re.compile(r"<parameter=([\w\-]+)>\n?(.*?)\n?</parameter>", re.S)


def recover_markup(text):
    calls = []
    for name, body in TOOL_BLOCK.findall(text or ""):
        args = {k: v for k, v in PARAM.findall(body)}
        calls.append({"id": f"markup-{len(calls) + 1}", "name": name, "arguments": args})
    return calls


def strip_markup(text):
    return TOOL_BLOCK.sub("", text or "").strip()


class ChatClient:
    def __init__(self, base_url, model, temperature=0.6, top_p=0.95, top_k=20,
                 max_tokens=16384, thinking=True, timeout=1800, reasoning_effort=None,
                 thinking_token_budget=None, template_kwargs=None, presence_penalty=None, bad_words=None, logit_bias=None):
        self.url = base_url.rstrip("/") + "/chat/completions"
        self.model = model
        # top_k=None leaves it to the server (gpt-oss wants no top-k). presence_penalty (X23, 2026-10-13)
        # is sent only when set: Qwen recommends ~1.5 against repetition in long thinking.
        # bad_words (X37, 2026-10-18): vLLM bans these words (with and without a leading space). Unlike logit_bias,
        # which vLLM 0.19 rejects whenever MTP speculative decoding is on, the spec-decode sampler applies it.
        # logit_bias (X37s): {token id (str): bias}, a soft penalty -- only on a server WITHOUT MTP.
        self.sampling = {k: v for k, v in {"temperature": temperature, "top_p": top_p,
                                           "top_k": top_k, "presence_penalty": presence_penalty,
                                           "bad_words": bad_words or None, "logit_bias": logit_bias or None}.items()
                         if v is not None}
        self.reasoning_effort = reasoning_effort
        # vLLM >= 0.19 forces </think> after this many reasoning tokens (needs the server
        # flag --reasoning-config). Qwen3.8 otherwise spends whole 16k turns thinking.
        self.thinking_token_budget = thinking_token_budget
        # Extra chat-template kwargs, e.g. Qwen3.8's reasoning_effort (xhigh default /
        # medium / low) -- the template itself turns it into a system instruction.
        self.template_kwargs = template_kwargs or {}
        self.max_tokens = max_tokens
        self.thinking = thinking
        self.timeout = timeout

    def chat(self, messages, tools=None, max_tokens=None, seed=None, timeout=None,
             tool_choice=None, template_kwargs=None):
        # template_kwargs: per-request overrides on top of the client's own, e.g. the
        # reasoning-effort ladder stepping one turn down to "medium".
        payload = {"model": self.model, "messages": messages,
                   "max_tokens": max_tokens or self.max_tokens,
                   "chat_template_kwargs": {"enable_thinking": self.thinking, **self.template_kwargs,
                                            **(template_kwargs or {})},
                   **self.sampling}
        if seed is not None:
            payload["seed"] = seed
        if self.reasoning_effort:          # gpt-oss: low / medium / high
            payload["reasoning_effort"] = self.reasoning_effort
        if self.thinking_token_budget:
            payload["thinking_token_budget"] = self.thinking_token_budget
        if tools:
            payload["tools"] = tools
            # A named tool forces that call (vLLM guided decoding); default lets it choose.
            payload["tool_choice"] = ({"type": "function", "function": {"name": tool_choice}}
                                      if tool_choice else "auto")
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=timeout or self.timeout) as r:
                body = json.loads(r.read().decode())
            return normalise(body)
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:500]
            overflow = "maximum context length" in detail or "context length" in detail
            if (e.code == 400 and not overflow and "thinking_token_budget" in payload
                    and any(w in detail.lower() for w in ("thinking", "budget", "reasoning"))):
                # A server started without --reasoning-config (the last rungs of the startup
                # ladder) may refuse the budget: drop it for every later call, not fail them all.
                self.thinking_token_budget = None
                return self.chat(messages, tools, max_tokens, seed, timeout, tool_choice, template_kwargs)
            return {"error": f"HTTP {e.code}: {detail}", "context_overflow": overflow}
        except Exception as e:
            # probe24 lost every result to one uncaught socket TimeoutError.
            text = f"{type(e).__name__}: {e}"
            return {"error": text[:300], "context_overflow": False,
                    "timeout": isinstance(e, TimeoutError) or "timed out" in text}

    COMMIT_TRANSITION = ("\n\nI have to stop analysing now and commit: I will apply the rule I was converging on "
                         "and give the final answer directly.\n")

    def commit(self, messages, reasoning, max_tokens=8192, seed=None, timeout=None, template_kwargs=None,
               transition=None):
        """Forced commit for a reply cut while still thinking (2026-10-12). vLLM's thinking_token_budget
        never fired for Qwen3.8 on our stack: every cut reply had empty answer text. So close the
        thinking ourselves: the SAME chat prompt (tokenized by the server with the same template kwargs)
        + the reply's own reasoning + a transition + </think>, sent as token ids to /v1/completions, and
        the model writes the answer with its whole reasoning in context. Returns normalise()'s keys (the
        answer in "content") plus commit_prompt_tokens, or {"error": ...}."""
        root = self.url.rsplit("/v1/", 1)[0]
        kw = {"enable_thinking": self.thinking, **self.template_kwargs, **(template_kwargs or {})}
        to = timeout or self.timeout
        try:
            p = _post(root + "/tokenize", {"model": self.model, "messages": messages, "add_generation_prompt": True,
                                           "chat_template_kwargs": kw}, to)
            ids = p["tokens"]
            # Qwen3.8's generation prompt already ends with "<think>\n"; add it only if it does not.
            tail = _post(root + "/detokenize", {"model": self.model, "tokens": ids[-8:]}, to).get("prompt", "")
            head = "" if "<think>" in tail else "<think>\n"
            text = head + (reasoning or "").rstrip() + (transition or self.COMMIT_TRANSITION) + "</think>\n\n"
            cont = _post(root + "/tokenize", {"model": self.model, "prompt": text, "add_special_tokens": False},
                         to)["tokens"]
            room = (p.get("max_model_len") or 10 ** 9) - len(ids) - 16
            if len(cont) + 512 > room:               # keep the END of the reasoning (closest to a commit)
                cont = cont[-(room - 512):]
            payload = {"model": self.model, "prompt": ids + cont,
                       "max_tokens": max(64, min(max_tokens, room - len(cont))), **self.sampling}
            if seed is not None:
                payload["seed"] = seed
            body = _post(root + "/v1/completions", payload, to)
            ch, usage = body["choices"][0], body.get("usage") or {}
            return {"content": ch.get("text") or "", "reasoning": "", "tool_calls": [],
                    "finish_reason": ch.get("finish_reason"), "prompt_tokens": usage.get("prompt_tokens", 0),
                    "completion_tokens": usage.get("completion_tokens", 0), "commit_prompt_tokens": len(ids) + len(cont)}
        except urllib.error.HTTPError as e:
            return {"error": f"HTTP {e.code}: {e.read().decode(errors='replace')[:300]}"}
        except Exception as e:
            return {"error": f"{type(e).__name__}: {e}"[:300]}


def _post(url, payload, timeout):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())


def normalise(body):
    ch = body["choices"][0]
    msg = ch.get("message", {}) or {}
    content = msg.get("content") or ""
    reasoning = msg.get("reasoning_content") or msg.get("reasoning") or ""
    calls = []
    for tc in msg.get("tool_calls") or []:
        fn = tc.get("function", {})
        try:
            args = json.loads(fn.get("arguments") or "{}")
        except json.JSONDecodeError:
            args = {"_raw": fn.get("arguments")}
        calls.append({"id": tc.get("id") or f"call-{len(calls) + 1}",
                      "name": fn.get("name"), "arguments": args})
    if not calls:
        calls = recover_markup(content) or recover_markup(reasoning)
        content = strip_markup(content)
    usage = body.get("usage") or {}
    # X63: prompt tokens served from the prefix cache (SGLang --enable-cache-report; 0 when the server omits it)
    cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
    return {"content": content, "reasoning": reasoning, "tool_calls": calls,
            "finish_reason": ch.get("finish_reason"),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0), "cached_tokens": cached}

In [ ]:
%%writefile /kaggle/working/agent/repl.py
"""The `python` tool: run an exploratory snippet with the task preloaded.

Each call is a fresh subprocess (nothing persists between calls, as in the
Tufa duck harness), so a crash or an infinite loop costs one call. The task is
injected as numpy arrays plus a few ARC helpers; stdout is captured and capped.

Optional function retention (dfranzen's ARC-AGI-3 fork of the duck harness): with a
store, plain top-level functions a call defines are kept as SOURCE and replayed into
later calls; variables never persist. Old transcripts show why: 30-38% of all
function code the model wrote re-defined a function it had already written, and
19 of 20 ab3q36 submits re-typed functions first written in a python call.
"""
import json
import os
import subprocess
import sys

HELPERS = r'''
import numpy as np
from collections import Counter, defaultdict, deque
import itertools, math, functools, copy, re

def show(g):
    """Digit rows, one line per row."""
    return "\n".join("".join(str(int(v)) for v in row) for row in np.asarray(g))

def diff(a, b):
    """Cells where a and b differ, as (row, col, a_value, b_value); size mismatch -> string."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shape mismatch {a.shape} vs {b.shape}"
    return [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in zip(*np.nonzero(a != b))]

def objects(g, bg=None, diagonal=False, multicolor=False):
    """Connected components. bg defaults to the most common colour.
    Returns dicts: color (or colors), cells [(r,c)], bbox (r0,c0,r1,c1), size, shape (cropped mask)."""
    g = np.asarray(g)
    if bg is None:
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
    H, W = g.shape
    seen = np.zeros_like(g, dtype=bool)
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] + ([(-1,-1),(-1,1),(1,-1),(1,1)] if diagonal else [])
    out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = g[r, c]
            q, cells = deque([(r, c)]), []
            seen[r, c] = True
            while q:
                y, x = q.popleft(); cells.append((y, x))
                for dy, dx in nbrs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] and g[ny, nx] != bg \
                            and (multicolor or g[ny, nx] == col):
                        seen[ny, nx] = True; q.append((ny, nx))
            ys, xs = zip(*cells)
            r0, c0, r1, c1 = min(ys), min(xs), max(ys), max(xs)
            mask = np.zeros((r1 - r0 + 1, c1 - c0 + 1), dtype=int)
            for y, x in cells:
                mask[y - r0, x - c0] = g[y, x]
            d = {"cells": cells, "bbox": (r0, c0, r1, c1), "size": len(cells), "shape": mask}
            if multicolor:
                d["colors"] = sorted({int(g[y, x]) for y, x in cells})
            else:
                d["color"] = int(col)
            out.append(d)
    return out
'''

RUNNER = r'''
import sys, json, io, contextlib, traceback
job = json.loads(sys.stdin.read())
ns = {"__name__": "__arc_repl__"}
exec(job["helpers"], ns)
import numpy as np
ns["train_inputs"] = [np.array(g) for g in job["train_inputs"]]
ns["train_outputs"] = [np.array(g) for g in job["train_outputs"]]
ns["test_inputs"] = [np.array(g) for g in job["test_inputs"]]
ns["train"] = list(zip(ns["train_inputs"], ns["train_outputs"]))
keep = job.get("retained")                # None: retention off (each call starts fresh)
preloaded = set(ns)
restore_failed = []
if keep is not None:
    # Replay what earlier calls kept: their imports, then their function definitions.
    # Source only -- no variable or array is ever carried over.
    for src in keep["imports"]:
        try:
            exec(compile(src, "<kept>", "exec"), ns)
        except Exception:
            pass
    for name, src in keep["defs"].items():
        try:
            exec(compile(src, "<kept>", "exec"), ns)
        except Exception:
            restore_failed.append(name)
buf = io.StringIO()
try:
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        exec(compile(job["code"], "<snippet>", "exec"), ns)
        if "result" in ns and ns["result"] is not None:
            print(repr(ns["result"]))
except Exception:
    tb = traceback.format_exc().strip().splitlines()
    buf.write("\n" + "\n".join(tb[-4:]))
sys.stdout.write("@@REPL@@" + json.dumps(buf.getvalue()) + "\n")
if keep is not None:
    import ast, builtins, dis, types
    defs, imports, rejected = {}, [], {}
    try:
        tree = ast.parse(job["code"])
    except SyntaxError:
        tree = ast.Module(body=[], type_ignores=[])
    bound = set()
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)) and not getattr(node, "level", 0):
            src = ast.get_source_segment(job["code"], node)
            if src:
                imports.append(src)
                bound |= {(a.asname or a.name).split(".")[0] for a in node.names}
    new = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
    available = (set(dir(builtins)) | preloaded | set(keep["defs"]) | bound
                 | {n.name for n in new} | set(keep.get("bound", [])))
    for node in new:
        fn = ns.get(node.name)
        # only a def that actually ran and still holds the name (not reassigned later)
        if not (isinstance(fn, types.FunctionType) and fn.__code__.co_filename == "<snippet>"
                and fn.__code__.co_firstlineno == node.lineno):
            continue
        reads, todo = set(), [fn.__code__]
        while todo:
            co = todo.pop()
            reads |= {i.argval for i in dis.get_instructions(co)
                      if i.opname in ("LOAD_GLOBAL", "LOAD_NAME")}
            todo += [c for c in co.co_consts if isinstance(c, types.CodeType)]
        missing = sorted(r for r in reads if isinstance(r, str) and r not in available)
        if missing:
            rejected[node.name] = missing
        else:
            defs[node.name] = ast.get_source_segment(job["code"], node)
    sys.stdout.write("@@KEPT@@" + json.dumps({"defs": defs, "imports": imports, "bound": sorted(bound),
                                             "rejected": rejected,
                                             "restore_failed": restore_failed}) + "\n")
'''

PREAMBLE_DOC = (
    "Preloaded in every `python` call (each call starts fresh; nothing persists): "
    "`np`, `train_inputs`, `train_outputs`, `test_inputs` (lists of numpy int arrays), "
    "`train` (list of (input, output) pairs), and helpers `show(g)` (digit rows), "
    "`diff(a, b)` (differing cells as (row, col, a, b)), "
    "`objects(g, bg=None, diagonal=False, multicolor=False)` (connected components with "
    "color, cells, bbox, size, shape). Print compact summaries, never whole large grids."
)


RETAIN_DOC = (
    "Variables do not persist between calls, but every function you define with a plain "
    "top-level `def` is kept: later `python` calls and `submit` can call it without "
    "redefining it. A kept function may use builtins, the preloaded names, imports and "
    "other kept functions; pass anything else in as an argument.")


def new_store():
    """Per-attempt record of what python calls have kept (source only)."""
    return {"defs": {}, "imports": [], "bound": []}


# X69 (2026-10-28, user: "retain the variables from earlier calls"): X63's 248 python calls had 36 NameErrors, 17 of
# them `ex1_in` -- the model reuses variables from earlier calls, as in a notebook (Qwen-Agent's code interpreter is a
# stateful Jupyter kernel). A Session is one python process per attempt that keeps its namespace between calls. A
# timeout or a crash kills it; the next call starts a fresh process (task + helpers reloaded, kept functions replayed)
# and says so. The model's code never sees the protocol pipe: stdin is emptied and stdout is captured per call.
SESSION_DOC = (
    "The python session is STATEFUL, like a notebook: variables, functions and imports you define persist across "
    "`python` calls, and `submit` runs in the same session (your transform may use anything you defined). The value "
    "of a final bare expression is printed. Preloaded: `np`, `train_inputs`, `train_outputs`, `test_inputs` (lists of "
    "numpy int arrays), `train` (list of (input, output) pairs), and helpers `show(g)` (digit rows), `diff(a, b)` "
    "(differing cells as (row, col, a, b)), `objects(g, bg=None, diagonal=False, multicolor=False)` (connected "
    "components with color, cells, bbox, size, shape). Print compact summaries, never whole large grids.")

SESSION_RUNNER = r'''
import sys, json, io, contextlib, traceback, copy, ast
_proto_in, _proto_out = sys.stdin, sys.stdout
sys.stdin = io.StringIO("")                      # the model's code must not read the protocol pipe
job = json.loads(_proto_in.readline())
ns = {"__name__": "__arc_repl__"}
exec(job["helpers"], ns)
import numpy as np
ns["train_inputs"] = [np.array(g) for g in job["train_inputs"]]
ns["train_outputs"] = [np.array(g) for g in job["train_outputs"]]
ns["test_inputs"] = [np.array(g) for g in job["test_inputs"]]
ns["train"] = list(zip(ns["train_inputs"], ns["train_outputs"]))
restored = []
for src in job.get("replay", []):                # kept functions from before a restart
    try:
        exec(compile(src, "<kept>", "exec"), ns)
        restored.append(src)
    except BaseException:
        pass

def clean(res):
    if isinstance(res, np.ndarray):
        res = res.tolist()
    if isinstance(res, tuple):
        res = list(res)
    if not isinstance(res, list) or not res:
        return None, "output is not a non-empty list of rows"
    rows = []
    for row in res:
        if hasattr(row, "tolist"):
            row = row.tolist()
        if not isinstance(row, (list, tuple)) or not row:
            return None, "output rows must be non-empty lists"
        out = []
        for v in row:
            try:
                iv = int(v)
            except Exception:
                return None, f"non-integer cell {v!r}"
            if iv != v or not 0 <= iv <= 9:
                return None, f"cell value {v!r} outside 0-9"
            out.append(iv)
        rows.append(out)
    w = len(rows[0])
    if any(len(r) != w for r in rows):
        return None, "output is not rectangular"
    if not (1 <= len(rows) <= 30 and 1 <= w <= 30):
        return None, f"output size {len(rows)}x{w} outside 1..30"
    return rows, None

def run_code(code):
    buf = io.StringIO()
    try:
        tree = ast.parse(code)
    except SyntaxError:
        tb = traceback.format_exc().strip().splitlines()
        return "\n".join(tb[-4:])
    last = tree.body[-1] if tree.body and isinstance(tree.body[-1], ast.Expr) else None
    body = ast.Module(body=tree.body[:-1] if last else tree.body, type_ignores=[])
    assigns_result = any(isinstance(n, ast.Name) and n.id == "result" and isinstance(n.ctx, ast.Store)
                         for n in ast.walk(tree))
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            exec(compile(body, "<snippet>", "exec"), ns)
            if last is not None:
                val = eval(compile(ast.Expression(last.value), "<snippet>", "eval"), ns)
                if val is not None:
                    print(repr(val))
            elif assigns_result and ns.get("result") is not None:
                print(repr(ns["result"]))
    except SystemExit:
        buf.write("\n[exit() ignored: the session keeps running]")
    except BaseException:
        tb = traceback.format_exc().strip().splitlines()
        buf.write("\n" + "\n".join(tb[-4:]))
    return buf.getvalue()

def run_transform(code, inputs):
    sink = io.StringIO()
    try:
        with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
            exec(compile(code, "<program>", "exec"), ns)
        fn = ns.get("transform")
        if not callable(fn):
            raise NameError("no function named transform")
    except BaseException as e:
        msg = "".join(traceback.format_exception_only(type(e), e)).strip()
        return [{"error": "program failed to load: " + msg[-400:]} for _ in inputs]
    results = []
    for grid in inputs:
        try:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                res = fn(copy.deepcopy(grid))
            rows, err = clean(res)
            results.append({"output": rows} if err is None else {"error": err})
        except BaseException as e:
            tb = traceback.format_exc().strip().splitlines()
            results.append({"error": " | ".join(tb[-3:])[-400:]})
    return results

_proto_out.write("@@SESSION@@" + json.dumps({"ready": True, "restored": len(restored)}) + "\n")
_proto_out.flush()
for line in _proto_in:
    req = json.loads(line)
    if req["op"] == "exec":
        rep = {"out": run_code(req["code"])}
    else:
        rep = {"results": run_transform(req["code"], req["inputs"])}
    _proto_out.write("@@SESSION@@" + json.dumps(rep) + "\n")
    _proto_out.flush()
'''


def remember_defs(store, code):
    """Record a snippet's top-level imports and function / class definitions (source) in the store, so a Session
    restarted after a timeout or crash gets them back. Variables cannot be replayed safely; the restart note says so."""
    import ast
    if store is None:
        return
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return
    for node in tree.body:
        src = ast.get_source_segment(code, node)
        if not src:
            continue
        if isinstance(node, (ast.Import, ast.ImportFrom)) and not getattr(node, "level", 0):
            if src not in store["imports"]:
                store["imports"].append(src)
        elif isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            store["defs"].pop(node.name, None)          # newest definition last
            store["defs"][node.name] = src


class Session:
    """One stateful python process per attempt (see SESSION_RUNNER). Not thread-safe: one attempt, one thread."""

    def __init__(self, task, store=None, mem_bytes=2 << 30, start_timeout=60.0):
        self.task, self.store, self.mem_bytes, self.start_timeout = task, store, mem_bytes, start_timeout
        self.proc, self.lines, self.restarts, self.starts = None, None, 0, 0

    def _start(self):
        import queue
        import threading
        pre = None
        if os.name == "posix":
            mem = self.mem_bytes

            def pre():
                try:
                    import resource
                    resource.setrlimit(resource.RLIMIT_AS, (mem, mem))
                except Exception:
                    pass
        self.proc = subprocess.Popen([sys.executable, "-c", SESSION_RUNNER], stdin=subprocess.PIPE,
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1,
                                     preexec_fn=pre,
                                     env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1"))
        self.lines = queue.Queue()
        proc, q = self.proc, self.lines

        def pump():
            for ln in proc.stdout:              # only marked lines are replies; anything else is the model's noise
                if ln.startswith("@@SESSION@@"):
                    q.put(ln[len("@@SESSION@@"):])
            q.put(None)
        threading.Thread(target=pump, daemon=True).start()
        replay = list(self.store["imports"]) + list(self.store["defs"].values()) if self.store else []
        job = {"helpers": HELPERS, "replay": replay,
               "train_inputs": [p["input"] for p in self.task["train"]],
               "train_outputs": [p["output"] for p in self.task["train"]],
               "test_inputs": [t["input"] for t in self.task["test"]]}
        self.proc.stdin.write(json.dumps(job) + "\n")
        self.proc.stdin.flush()
        return self._reply(self.start_timeout)

    def _reply(self, timeout):
        import queue
        try:
            ln = self.lines.get(timeout=timeout)
        except queue.Empty:
            return None
        return json.loads(ln) if ln is not None else None

    def _request(self, req, timeout):
        """(reply or None, note): a fresh process if needed; on timeout / crash the process is killed."""
        note = ""
        if self.proc is None or self.proc.poll() is not None:
            started = self.starts > 0
            self.starts += 1
            if self._start() is None:
                self.close()
                return None, "[the python session could not start]"
            if started:
                self.restarts += 1
                note = ("[the python session had stopped and was restarted: variables from earlier calls are gone"
                        + ("; kept functions were restored" if self.store and self.store["defs"] else "") + "]\n")
        try:
            self.proc.stdin.write(json.dumps(req) + "\n")
            self.proc.stdin.flush()
        except (BrokenPipeError, OSError):
            self.close()
            return None, note + "[the python session crashed]"
        rep = self._reply(timeout)
        if rep is None:
            crashed = self.proc.poll() is not None
            self.close()
            return None, note + (f"[the python session crashed (out of memory?)]" if crashed else
                                 f"[timed out after {timeout:.0f}s; the python session was stopped -- the next call "
                                 "starts a fresh one without your variables]")
        return rep, note

    def run(self, code, timeout=30.0, max_chars=4000):
        rep, note = self._request({"op": "exec", "code": code}, timeout)
        if rep is None:
            return note.strip()
        out = rep["out"].strip() or "[no output -- print() what you want to see, or end with an expression]"
        if len(out) > max_chars:
            out = out[:max_chars] + f"\n... [truncated {len(out) - max_chars} chars]"
        return note + out

    def run_transform(self, code, inputs, timeout=30.0):
        """sandbox.run_one's result shape, computed inside the session."""
        rep, note = self._request({"op": "transform", "code": code, "inputs": inputs}, timeout)
        if rep is None:
            return [{"error": note.strip()} for _ in inputs]
        return rep["results"]

    def close(self):
        if self.proc is not None:
            try:
                self.proc.kill()
                self.proc.wait(timeout=5)
            except Exception:
                pass
        self.proc = None


def run_snippet(code, task, timeout=30.0, max_chars=4000, mem_bytes=2 << 30, store=None):
    """Run one snippet. With `store` (from new_store()), functions defined by earlier calls
    are replayed first and the ones this call defines are added to the store; a line at
    the end of the output says what was kept."""
    job = {"code": code, "helpers": HELPERS,
           "train_inputs": [p["input"] for p in task["train"]],
           "train_outputs": [p["output"] for p in task["train"]],
           "test_inputs": [t["input"] for t in task["test"]]}
    if store is not None:
        job["retained"] = store
    pre = None
    if os.name == "posix":
        def pre():
            try:
                import resource
                resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
            except Exception:
                pass
    try:
        p = subprocess.run([sys.executable, "-c", RUNNER], input=json.dumps(job),
                           capture_output=True, text=True, timeout=timeout, preexec_fn=pre,
                           env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1"))
    except subprocess.TimeoutExpired:
        return f"[timed out after {timeout:.0f}s]"
    out, kept = None, None
    for line in reversed(p.stdout.splitlines()):
        if line.startswith("@@KEPT@@") and kept is None:
            kept = json.loads(line[len("@@KEPT@@"):])
        if line.startswith("@@REPL@@"):
            out = json.loads(line[len("@@REPL@@"):])
            break
    if out is None:
        out = "[snippet crashed] " + " | ".join((p.stderr or "").strip().splitlines()[-2:])
    out = out.strip() or "[no output -- print() what you want to see]"
    if len(out) > max_chars:
        out = out[:max_chars] + f"\n... [truncated {len(out) - max_chars} chars]"
    if store is not None and kept is not None:
        out += _absorb(store, kept)
    return out


def _absorb(store, kept):
    """Merge one call's kept definitions into the store; return the note for the model."""
    for name in kept["restore_failed"]:
        store["defs"].pop(name, None)
    for src in kept["imports"]:
        if src not in store["imports"]:
            store["imports"].append(src)
    store["bound"] = sorted(set(store["bound"]) | set(kept["bound"]))
    notes = []
    for name, src in kept["defs"].items():
        if store["defs"].get(name) != src:
            store["defs"].pop(name, None)            # re-insert: newest definition last
            store["defs"][name] = src
    if kept["defs"]:
        notes.append("kept for later calls: " + ", ".join(sorted(kept["defs"])))
    for name, missing in kept["rejected"].items():
        notes.append(f"not kept: {name} (uses {', '.join(missing[:4])} from this snippet -- "
                     "pass it as an argument to keep the function)")
    if kept["restore_failed"]:
        notes.append("dropped (failed to reload): " + ", ".join(kept["restore_failed"]))
    return ("\n[" + "; ".join(notes) + "]") if notes else ""


def with_kept(code, store):
    """A submitted program plus the kept functions (and imports) it relies on.

    Returns (full_source, names_added). The program's own definitions win; kept
    functions are added only if the program uses them, transitively, and transform()
    itself is taken from the store if the program does not define one."""
    import ast
    if not store or not store["defs"]:
        return code, []
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code, []
    defined = set()
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            defined.add(node.name)
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            defined |= {(a.asname or a.name).split(".")[0] for a in node.names}
        elif isinstance(node, ast.Assign):
            defined |= {t.id for t in node.targets if isinstance(t, ast.Name)}

    def used(src):
        try:
            return {n.id for n in ast.walk(ast.parse(src)) if isinstance(n, ast.Name)}
        except SyntaxError:
            return set()

    need = (used(code) & set(store["defs"])) - defined
    if "transform" not in defined and "transform" in store["defs"]:
        need.add("transform")
    todo = list(need)
    while todo:
        for n in (used(store["defs"][todo.pop()]) & set(store["defs"])) - defined - need:
            need.add(n)
            todo.append(n)
    if not need:
        return code, []
    # Kept functions were written against the python tool's namespace (np, objects(), ...),
    # so the helper preamble goes in first; the program's own definitions still win.
    parts = ([HELPERS.strip()] + list(store["imports"])
             + [src for n, src in store["defs"].items() if n in need])
    return "\n".join(parts) + "\n\n" + code, sorted(need)

In [ ]:
%%writefile /kaggle/working/agent/repl_agent.py
"""REPL tool agent for ARC-AGI-2 tasks, modelled on the Tufa Labs duck harness.

The model gets two tools:
  python(code)  explore the task: train/test grids are preloaded as numpy
                arrays with ARC helpers; stdout comes back, capped.
  submit(code)  a program defining transform(grid). It is run on every train
                input; the reply says which examples it reproduced and shows a
                cell diff for the rest. Reproducing ALL of them verifies the
                program, records its test outputs and ends the attempt.

Many attempts run concurrently against one vLLM OpenAI server, which batches
them. Old turns are evicted when the context budget is exceeded; the system
prompt and the task stay. Answers are aggregated exactly as in agent.py.

Optional, off by default, from dfranzen's ARC-AGI-3 fork of the duck harness (see
run_attempt): reasoning-effort ladder on cut turns, function retention, kept
reasoning, hysteresis trimming, repetition-loop guard, timeout-as-cut.
"""
import json
import threading
import time
import zlib
from concurrent.futures import ThreadPoolExecutor

from agent import aggregate
from backend import extract_code
from grids import SYSTEM, show, diff_text
from repl import run_snippet, PREAMBLE_DOC, RETAIN_DOC, SESSION_DOC, Session, new_store, with_kept, remember_defs
from sandbox import run_one

TOOLS = [
    {"type": "function", "function": {
        "name": "python",
        "description": "Run Python to inspect the task. " + PREAMBLE_DOC,
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Python code; print() what you need."}},
            "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "submit",
        "description": ("Submit a complete program defining transform(grid) -> grid. It is run on "
                        "every example input; you are told which examples it reproduces. "
                        "When it reproduces all of them it is accepted and you are done."),
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Full Python source defining transform(grid)."}},
            "required": ["code"]}}},
]

REPL_INSTRUCTIONS = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

Work like a scientist: use the `python` tool to measure the examples (objects, colours, sizes, positions, differences between input and output), form a hypothesis, and test it in code. Then call `submit` with the full source defining `transform(grid)`. Use only the standard library and numpy.

Submit EARLY and often: submitting is free, and a failing submission tells you exactly which examples and which cells are wrong -- that is the fastest way to find the rule. Do not wait for certainty. If a submission fails, study the diff and fix the rule rather than special-casing examples."""

# probe24: every failed attempt spent 30-73k tokens exploring and never submitted
# once, so it never saw a diff. After this many python calls without a submit,
# the tool output carries a reminder.
NUDGE_EVERY = 6
NUDGE = ("\n\n[harness] You have run {n} python calls without submitting. Submitting is free "
         "and returns a cell-level diff against every example. Submit your current best "
         "transform now, then refine it from the diff.")

# probe9q38: every failed attempt had 1-3 turns cut at 16k tokens mid-thought, while
# solves used 2-3k tokens a turn. The thinking-cap treatment lowers the per-turn cap
# and, when a turn is cut, orders the model to act instead of resuming the analysis.
THINKCAP_HINT = ("[harness] You hit the per-turn thinking limit. Stop analysing. In your next "
                 "reply, call `python` to test your current best hypothesis in code, or "
                 "`submit` a transform implementing it -- a failing submit shows exactly "
                 "which cells are wrong.")

# X44 (2026-10-19): X40 threw a cut turn's thinking away and the next turn restarted the analysis from scratch (70% of
# xhigh's tokens). With keep_cut_reasoning the cut turn's reasoning stays in the history (rendered by the template's
# preserve_thinking) and the model is asked to continue it, not to restart or to stop.
CONTINUE_HINT = ("[harness] Your previous turn reached the per-turn length limit while you were still thinking. Your "
                 "thinking so far is kept above -- do not start over. Continue from exactly where you stopped, and "
                 "end this turn with a `python` call or a `submit`.")


# The probe24 wording, kept verbatim so a run with the nudge off matches it exactly.
REPL_INSTRUCTIONS_V1 = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

Work like a scientist: use the `python` tool to measure the examples (objects, colours, sizes, positions, differences between input and output), form a hypothesis, and test it in code. When you have a program that you believe reproduces EVERY example, call `submit` with the full source defining `transform(grid)`. Use only the standard library and numpy. If a submission fails, study the diff and fix the rule rather than special-casing examples."""


# probe9q38 transcripts: failures parse and count grids IN THE REASONING, cell by cell,
# often filling a whole 16k-token turn before any tool call ("col 7=8, 8=0, 9=0 ...").
# Tufa's harness says the same thing to its model. Optional, its own arm.
IN_PYTHON_RULE = """

Never read, count, index or transcribe grid cells in your head. Do ALL of that with the `python` tool -- the grids are already loaded as numpy arrays. Keep your own reasoning to hypotheses and plans; let code do the looking."""


# X69 (2026-10-28, user: "how can we make the model use python more aggressively?"). X44 / X63 thought a median
# 25.6k / 33k tokens before their first python call, re-typing preloaded grids and listing coordinates by hand; the
# trace review found hand bookkeeping is ~half of every trace. This wording goes with the stateful session (variables
# persist) and the exact facts block under the grids.
PYTHON_FIRST_INSTRUCTIONS = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

You have a stateful Python session, like a notebook: the grids are already loaded as numpy arrays, and every variable and function you define stays available in later calls. Use it for ALL looking: reading cells, counting, locating and comparing objects, measuring sizes, checking a rule against the examples, and building output grids. Never transcribe, count or index grid cells in your head, and never re-type a grid -- your thinking is for hypotheses and plans; the code does the looking. Call `python` early, after a short first look, and keep each turn's thinking short: think, run code, read the result, repeat.

Exact facts about every grid (sizes, colour counts, objects, which cells change) are listed after the grids; use them instead of counting.

When you have a hypothesis, write `transform(grid)` and `submit` it: submitting is free and returns a cell-level diff for every example it gets wrong. Do not wait for certainty. If a submission fails, study the diff and fix the rule rather than special-casing examples. Use only the standard library and numpy."""

# X69: a cut turn keeps its reasoning (X44's keep_cut_reasoning) but is told to act, not to keep thinking.
ACT_HINT = ("[harness] You reached the per-turn thinking limit. Your thinking so far is kept above -- do not repeat "
            "it. Stop analysing in your head now: in this reply, call `python` to check your current idea on the "
            "examples, or `submit` a transform.")


def repl_prompt(task, early_submit=True, in_python=False, python_first=False, facts=False):
    if python_first:
        head = PYTHON_FIRST_INSTRUCTIONS
    else:
        head = (REPL_INSTRUCTIONS if early_submit else REPL_INSTRUCTIONS_V1) + (IN_PYTHON_RULE if in_python else "")
    parts = [head, ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", show(p["input"], "Input"), show(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", show(t["input"], "Input"), ""]
    if facts:
        from task_facts import facts as task_facts      # lazy: older notebooks ship no task_facts.py
        parts += [task_facts(task), ""]
    return "\n".join(parts).rstrip()


def verify(code, task, timeout=10.0, runner=None):
    """Run a submitted program; return (n_ok, test_outputs, feedback_text). runner(code, inputs) defaults to a fresh
    sandbox; X69's stateful session passes its own (the program then sees the session's names)."""
    tr = task["train"]
    inputs = [p["input"] for p in tr] + [t["input"] for t in task["test"]]
    res = runner(code, inputs) if runner else run_one(code, inputs, timeout=timeout)
    n_ok, lines, shown = 0, [], 0
    for i, (r, p) in enumerate(zip(res[:len(tr)], tr), 1):
        if "output" in r and r["output"] == p["output"]:
            n_ok += 1
            continue
        if shown < 2:
            shown += 1
            lines.append(f"## Example {i} failed")
            lines.append(f"Error: {r['error']}" if "error" in r else diff_text(p["output"], r["output"]))
    test_out = [r.get("output") for r in res[len(tr):]]
    test_err = [r.get("error") for r in res[len(tr):] if "error" in r]
    if n_ok == len(tr):
        if test_err:
            return n_ok, test_out, ("Reproduces all examples but FAILS on a test input: "
                                    + test_err[0] + " -- fix it and submit again.")
        return n_ok, test_out, f"Accepted: reproduces all {len(tr)} examples."
    return n_ok, test_out, (f"Reproduced {n_ok} of {len(tr)} examples.\n" + "\n".join(lines)
                            + "\nFix the rule and submit again.")


def _est_tokens(messages):
    return len(json.dumps(messages)) // 3


# X63 (2026-10-26): len(json) / 3 undercounts -- X44 sent 14 requests over the 131,072-token context while estimated
# under its 90k budget. lordhansolo's ARC-AGI-3 harness prices text per kind (characters per token, Flash-Next tokenizer,
# set at the 5th percentile so the count is an upper bound for ~95% of requests); Qwen also splits numbers into single
# digits, and our grids are rows of digits, so every digit counts as one token. run_attempt(token_est="calibrated")
# rescales this by the server's own prompt_tokens for the request just sent (dfranzen's calibration).
_CPT = {"system": 4.3, "tool": 1.8, "reasoning": 2.5, "text": 2.9}
_NO_DIGITS = str.maketrans("", "", "0123456789")


def _text_tokens(text, cpt):
    text = text or ""
    digits = len(text) - len(text.translate(_NO_DIGITS))
    return digits + (len(text) - digits) / cpt


def _est_kinds(messages):
    n = 0.0
    for m in messages:
        role = m.get("role")
        c = m.get("content") or ""
        n += 4 + _text_tokens(c if isinstance(c, str) else json.dumps(c),
                              _CPT["system"] if role == "system" else _CPT["tool"] if role == "tool" else _CPT["text"])
        for k in ("reasoning", "reasoning_content"):
            if m.get(k):
                n += _text_tokens(m[k], _CPT["reasoning"])
        if m.get("tool_calls"):
            n += _text_tokens(json.dumps(m["tool_calls"]), _CPT["text"])
    return int(n)


def trim(messages, budget, keep_recent=6, drain=0, state=None, est=_est_tokens):
    """Drop the oldest assistant/tool/user groups after the task prompt until the
    estimate fits. A group starts at an assistant message, so tool results are
    never separated from the call that produced them.

    With `drain` and a per-attempt `state` dict the cut has hysteresis (dfranzen's
    ARC-AGI-3 harness): it only moves when the estimate exceeds `budget`, and then
    drops down to `budget - drain`. Between those moments the retained history is
    byte-identical from turn to turn, so the server's prefix cache keeps serving it;
    without it, every turn past the budget shifts the cut by one group and
    re-prefills everything after the task prompt."""
    if drain and state is not None:
        return _trim_hysteresis(messages, budget, keep_recent, drain, state, est)
    head, rest = messages[:2], messages[2:]
    starts = [i for i, m in enumerate(rest) if m["role"] == "assistant"]
    dropped = 0
    while est(head + rest) > budget and len(starts) > keep_recent:
        cut = starts[1]
        rest = rest[cut:]
        starts = [i - cut for i in starts[1:]]
        dropped += 1
    if dropped:
        note = {"role": "user", "content": f"[{dropped} older turns were removed to save context.]"}
        rest = [note] + rest
    return head + rest


def _trim_hysteresis(messages, budget, keep_recent, drain, state, est=_est_tokens):
    head, rest = messages[:2], messages[2:]
    starts = [i for i, m in enumerate(rest) if m["role"] == "assistant"]
    max_cut = max(0, len(starts) - keep_recent)

    def view(cut):
        if cut <= 0:
            return head + rest
        note = {"role": "user", "content": f"[{cut} older turns were removed to save context.]"}
        return head + [note] + rest[starts[cut]:]

    cut = min(state.get("cut", 0), max_cut)
    if est(view(cut)) > budget:
        while cut < max_cut and est(view(cut)) > budget - drain:
            cut += 1
        state["evictions"] = state.get("evictions", 0) + 1
    state["cut"] = cut
    return view(cut)


def degenerate(text, min_chars=500, ratio=0.01):
    """A repetition loop rather than text: zlib compresses it below `ratio`. dfranzen saw a
    reasoning-only reply of 6,144 '!' under speculative decoding; prose compresses to
    0.2-0.3, an ASCII grid to ~0.02, so 0.01 only catches genuine loops."""
    if not text or len(text) < min_chars:
        return False
    raw = text.encode("utf-8", "replace")
    return len(zlib.compress(raw, 6)) / len(raw) < ratio


DEGENERATE_STUB = ("(previous reply was a repetition loop and has been omitted; do not continue "
                   "that pattern)")


def make_tools(retain=False, stateful=False):
    """The two tools; with function retention their descriptions say what persists."""
    if stateful:
        py = dict(TOOLS[0]["function"], description="Run Python in your stateful session. " + SESSION_DOC)
        sub = dict(TOOLS[1]["function"], description=(
            "Submit a complete program defining transform(grid) -> grid. It runs in your python session (it may use "
            "functions and variables you defined there) on every example input; you are told which examples it "
            "reproduces, with a cell diff for the others. When it reproduces all of them it is accepted and you are "
            "done."))
        return [{"type": "function", "function": py}, {"type": "function", "function": sub}]
    if not retain:
        return TOOLS
    py = dict(TOOLS[0]["function"], description=(
        "Run Python to inspect the task. "
        + PREAMBLE_DOC.replace(" (each call starts fresh; nothing persists)", "") + " " + RETAIN_DOC))
    sub = dict(TOOLS[1]["function"], description=(
        TOOLS[1]["function"]["description"] + " The program may call functions kept from your "
        "python calls (they are added automatically); if it defines no transform(), the kept "
        "transform() is submitted."))
    return [{"type": "function", "function": py}, {"type": "function", "function": sub}]


class _Shared:
    def __init__(self):
        self.lock = threading.Lock()
        self.verified = {}


class Attempt:
    def __init__(self, task_id, idx):
        self.task_id, self.idx = task_id, idx
        self.programs, self.status, self.note = [], "active", ""
        self.turns = self.python_calls = self.submits = 0
        self.python_at_submit = self.nudges = self.cuts = self.forced = 0
        self.rescued = self.force_turn = 0           # X79
        self.gen_tokens = self.prompt_tokens = 0
        # ARC-AGI-3 takeaways (all zero unless their option is on)
        self.effort_steps = self.timeouts = self.degenerate = self.kept_used = 0
        self.evictions = 0
        self.cached_tokens = self.overflows = 0      # X63: prefix-cache hits, context-length rejections
        self.name_errors = self.session_restarts = 0  # X69: NameErrors in python output; stateful-session restarts
        self._session = None
        self.wall = 0.0
        # Everything the model thought, ran and saw -- so a failure can be read as
        # "right idea, out of budget" vs "never found the rule".
        self.transcript = []


TIMEOUT_HINT = ("[harness] Your previous reply took too long and was abandoned. Keep your "
                "reasoning short: call `python` to test your current best hypothesis, or `submit`.")


FORCE_HINT = ("[harness] You have used {used:,} of your {budget:,} tokens and have not submitted yet. Your NEXT action "
              "must be `submit` with your current best transform(grid), even if you are unsure. The reply shows "
              "exactly which examples and cells are wrong; use your remaining tokens to fix it from that diff.")
RESCUE_HINT = ("[harness] Your token budget is used up and you never submitted. Do not analyse further: call `submit` "
               "now with the most complete transform(grid) you have. This is your last turn.")


def _harness_note(messages, text):
    """X79: add a harness instruction after the latest tool result / user message (never to the task prompt)."""
    m = messages[-1]
    if len(messages) > 2 and m["role"] in ("user", "tool") and isinstance(m.get("content"), str):
        m["content"] += "\n\n" + text
    else:
        messages.append({"role": "user", "content": text})


def run_attempt(att, task, client, shared, deadline, max_turns=40, max_gen_tokens=120000,
                context_budget=28000, stop_after=1, seed=0, snippet_timeout=30.0,
                nudge_every=NUDGE_EVERY, turn_max_tokens=None, in_python=False,
                force_submit_after=0, effort_ladder=None, retain_functions=False,
                keep_reasoning=False, reasoning_key="reasoning_content", keep_recent=6,
                trim_drain=0, degenerate_guard=False, timeout_as_cut=False, keep_cut_reasoning=False,
                token_est="json3", stateful=False, python_first=False, facts=False, cut_hint="continue",
                force_submit_tokens=0, rescue_submit=False, rescue_tokens=8192):
    """One attempt at one task. The options after force_submit_after are the ARC-AGI-3
    (dfranzen / duck harness) takeaways, each off by default so the baseline is unchanged:

    effort_ladder     e.g. ["medium", "low"]: after a turn is cut at max_tokens (or times
                      out, with timeout_as_cut) the next turns run one reasoning_effort rung
                      lower; a submit that reproduces more examples than before restores
                      the client's own effort.
    retain_functions  python-defined functions persist (as source) into later python calls
                      and into submits that use them.
    keep_reasoning    send each completed turn's reasoning back under `reasoning_key`
                      (pair with chat-template kwarg preserve_thinking=True). Cut turns and
                      repetition loops are never sent back.
    trim_drain        hysteresis for the context trim (see trim()); keep_recent groups are
                      never evicted.
    degenerate_guard  a repetition-loop reply is replaced by a stub in the history.
                      Loops are COUNTED in every arm (att.degenerate).
    timeout_as_cut    a request timeout before the deadline is treated like a cut turn
                      (retry, one rung lower) instead of failing the attempt.
    keep_cut_reasoning  (X44, needs keep_reasoning) a turn cut at the per-turn cap keeps its
                      reasoning in the history and the next turn is told to continue it
                      (CONTINUE_HINT) instead of restarting or stopping.
    token_est         (X63) "json3" = len(json) / 3 (X44); "calibrated" = _est_kinds scaled by the
                      server's prompt_tokens / _est_kinds of the request just sent (0.9-3.0).
    stateful          (X69) one python process per attempt keeps variables between calls (repl.Session); submit
                      runs in it too. Functions are still recorded so a restarted session gets them back.
    python_first      (X69) PYTHON_FIRST_INSTRUCTIONS instead of REPL_INSTRUCTIONS (+ in_python).
    facts             (X69) the exact facts block (task_facts.facts) after the grids.
    cut_hint          (X69) "act": a cut turn that keeps its reasoning gets ACT_HINT (stop thinking, run code)
                      instead of CONTINUE_HINT."""
    t0 = time.time()
    # The early-submit wording and the nudge are ONE treatment: off means probe24's harness.
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": repl_prompt(task, early_submit=bool(nudge_every),
                                                        in_python=in_python, python_first=python_first,
                                                        facts=facts)}]
    tools = make_tools(retain_functions, stateful)
    store = new_store() if (retain_functions or stateful) else None
    session = att._session = Session(task, store=store) if stateful else None
    trim_state = {} if trim_drain else None
    rung, best_ok = -1, -1                   # rung -1 = the client's own reasoning effort
    scale = [1.0]
    est = (lambda ms: int(_est_kinds(ms) * scale[0])) if token_est == "calibrated" else _est_tokens

    def step_down():
        nonlocal rung
        if effort_ladder and rung < len(effort_ladder) - 1:
            rung += 1
            att.effort_steps += 1

    while att.turns < max_turns:
        with shared.lock:
            if shared.verified.get(att.task_id, 0) >= stop_after:
                att.status, att.note = "stopped", "task verified by a sibling"
                break
        if time.time() > deadline:
            att.status, att.note = "stopped", "deadline"
            break
        rescue = False
        if att.gen_tokens >= max_gen_tokens:
            if not (rescue_submit and att.submits == 0 and not att.rescued):
                att.status, att.note = "failed", "token budget"
                break
            att.rescued, rescue = 1, True               # X79: one last short turn that must submit
            att.force_turn = att.force_turn or att.turns + 1
            _harness_note(messages, RESCUE_HINT)
        elif force_submit_tokens and att.submits == 0 and not att.force_turn and att.gen_tokens >= force_submit_tokens:
            att.force_turn = att.turns + 1              # X79: programs from this turn on came after the order
            _harness_note(messages, FORCE_HINT.format(used=att.gen_tokens, budget=max_gen_tokens))
        att.turns += 1
        # A request may not outlive the deadline by more than a minute.
        req_timeout = max(60.0, min(3600.0, deadline - time.time() + 60.0))
        # mini2arm: the text nudge was read and ignored. The forcesubmit treatment
        # makes the NEXT call a submit after this many python calls without one.
        force = (force_submit_after
                 and att.python_calls - att.python_at_submit >= force_submit_after)
        if force:
            att.forced += 1
        extra = {}
        if rung >= 0:            # only then, so clients without the kwarg keep working
            extra["template_kwargs"] = {"reasoning_effort": effort_ladder[rung]}
        sent = trim(messages, context_budget, keep_recent, trim_drain, trim_state, est)
        r = client.chat(sent, tools=tools, seed=seed + att.turns,
                        timeout=req_timeout, max_tokens=(rescue_tokens if rescue else turn_max_tokens),
                        tool_choice="submit" if force else None,
                        **({"template_kwargs": {"reasoning_effort": "low"}} if rescue else extra))
        if trim_state is not None:
            att.evictions = trim_state.get("evictions", 0)
        if "error" in r:
            att.overflows += bool(r.get("context_overflow"))
            if r.get("context_overflow") and keep_reasoning:
                # Kept reasoning can make even the protected recent turns too long for
                # the model; shed it (oldest first, newest last) before shrinking.
                held = [m for m in messages if reasoning_key in m]
                if held:
                    for m in held[:max(1, len(held) - 1)]:
                        m.pop(reasoning_key)
                    continue
            if r.get("context_overflow") and context_budget > 8000:
                context_budget = int(context_budget * 0.7)
                continue
            if time.time() >= deadline:
                # Requests are timed to end a minute after the deadline: this is the run
                # ending, not the attempt failing (ab3q36 logged 21 of these as failures).
                att.status, att.note = "stopped", "deadline (request cut)"
                break
            if timeout_as_cut and r.get("timeout"):
                att.timeouts += 1
                step_down()
                if messages[-1]["role"] != "user":
                    messages.append({"role": "user", "content": TIMEOUT_HINT})
                continue
            att.status, att.note = "failed", r["error"][:200]
            break
        att.gen_tokens += r["completion_tokens"]
        att.prompt_tokens += r["prompt_tokens"]
        att.cached_tokens += r.get("cached_tokens", 0)
        est_sent = est(sent)                  # what the trim believed, before this reply recalibrates it
        if token_est == "calibrated" and r["prompt_tokens"]:
            raw = _est_kinds(sent)
            if raw > 0:
                scale[0] = min(3.0, max(0.9, r["prompt_tokens"] / raw))
        cut = r["finish_reason"] == "length"
        loops = [k for k in ("reasoning", "content") if degenerate(r[k])]
        att.degenerate += bool(loops)
        att.transcript.append({"turn": att.turns, "t": round(time.time() - t0, 1),
                               "reasoning": r["reasoning"], "content": r["content"],
                               "calls": [c["name"] for c in r["tool_calls"]],
                               "finish": r["finish_reason"], "tokens": r["completion_tokens"],
                               "prompt": r["prompt_tokens"], "cached": r.get("cached_tokens", 0),
                               "est": est_sent,
                               **({"effort": effort_ladder[rung]} if rung >= 0 else {}),
                               **({"loop": loops} if loops else {})})
        if cut:
            step_down()
        content = r["content"] or ""
        if degenerate_guard and "content" in loops:
            content = DEGENERATE_STUB
        msg = {"role": "assistant", "content": content}
        if (keep_reasoning and r["reasoning"] and (not cut or keep_cut_reasoning)
                and "reasoning" not in loops):
            msg[reasoning_key] = r["reasoning"]
        calls = r["tool_calls"]
        if not calls:
            code = extract_code(r["content"])
            if code:                       # answered with a code block instead of submit
                calls = [{"id": f"implicit-{att.turns}", "name": "submit",
                          "arguments": {"code": code}, "implicit": True}]
            else:
                messages.append(msg)
                if r["finish_reason"] == "length":
                    # The thinking-cap treatment answers a cut with an order to act; the
                    # baseline keeps probe24's wording.
                    kept = keep_cut_reasoning and keep_reasoning and r["reasoning"]
                    hint = (ACT_HINT if kept and cut_hint == "act" else CONTINUE_HINT if kept
                            else THINKCAP_HINT if turn_max_tokens else "Your reply was cut off. Be more concise.")
                    att.cuts += 1
                else:
                    hint = "Use the `python` tool to explore, or `submit` your transform program."
                messages.append({"role": "user", "content": hint})
                continue
        if not any(c.get("implicit") for c in calls):
            msg["tool_calls"] = [{"id": c["id"], "type": "function",
                                  "function": {"name": c["name"],
                                               "arguments": json.dumps(c["arguments"])}}
                                 for c in calls]
        messages.append(msg)
        done = False
        for c in calls:
            code = (c["arguments"] or {}).get("code") or ""
            if c["name"] == "python":
                att.python_calls += 1
                if session is not None:
                    out = session.run(code, timeout=snippet_timeout)
                    remember_defs(store, code)
                else:
                    out = run_snippet(code, task, timeout=snippet_timeout, store=store)
                att.name_errors += "NameError" in out
                since = att.python_calls - att.python_at_submit
                if nudge_every and since and since % nudge_every == 0:
                    out += NUDGE.format(n=since)
                    att.nudges += 1
            elif c["name"] == "submit":
                att.submits += 1
                att.python_at_submit = att.python_calls
                if session is not None:            # the program sees the session's names; nothing to splice in
                    full, added = code, []
                    n_ok, test_out, out = verify(code, task, runner=lambda c_, i_: session.run_transform(
                        c_, i_, timeout=snippet_timeout))
                else:
                    full, added = with_kept(code, store) if store else (code, [])
                    n_ok, test_out, out = verify(full, task)
                if added:
                    att.kept_used += 1
                    out += f"\n(Ran with your kept functions: {', '.join(added)}.)"
                ntr = len(task["train"])
                att.programs.append({"turn": att.turns, "code": full, "n_ok": n_ok,
                                     "n_train": ntr, "test": test_out,
                                     **({"stateful": True} if session is not None else {})})
                if n_ok > best_ok:
                    if best_ok >= 0 or n_ok > 0:
                        rung = -1            # progress: back to the client's own effort
                    best_ok = n_ok
                if n_ok == ntr and all(o is not None for o in test_out):
                    done = True
            else:
                out = f"Unknown tool {c['name']!r}. Tools: python, submit."
            att.transcript.append({"turn": att.turns, "tool": c["name"], "code": code,
                                   "output": out})
            if c.get("implicit"):
                messages.append({"role": "user", "content": out})
            else:
                messages.append({"role": "tool", "tool_call_id": c["id"], "content": out})
        if done:
            att.status = "verified"
            with shared.lock:
                shared.verified[att.task_id] = shared.verified.get(att.task_id, 0) + 1
            break
    else:
        att.status, att.note = "failed", "turn budget"
    if session is not None:
        att.session_restarts = session.restarts
        session.close()
    att.wall = time.time() - t0
    return att


# run_attempt options a probe or submission config may set for the whole run
# (probe arms can set any run_attempt keyword).
AGENT_OPTS = ("turn_max_tokens", "in_python", "force_submit_after", "effort_ladder",
              "retain_functions", "keep_reasoning", "reasoning_key", "keep_recent", "trim_drain",
              "degenerate_guard", "timeout_as_cut", "snippet_timeout", "keep_cut_reasoning", "token_est",
              "stateful", "python_first", "facts", "cut_hint", "force_submit_tokens", "rescue_submit",
              "rescue_tokens")


def attempt_record(a, task):
    return {"task_id": a.task_id, "idx": a.idx, "status": a.status,
            "note": a.note, "turns": a.turns, "python_calls": a.python_calls,
            "submits": a.submits, "nudges": a.nudges, "cuts": a.cuts, "forced": a.forced, "gen_tokens": a.gen_tokens,
            "prompt_tokens": a.prompt_tokens, "wall": a.wall,
            "effort_steps": a.effort_steps, "timeouts": a.timeouts, "degenerate": a.degenerate,
            "kept_used": a.kept_used, "evictions": a.evictions,
            "cached_tokens": a.cached_tokens, "overflows": a.overflows,
            "name_errors": a.name_errors, "session_restarts": a.session_restarts,
            "best_ok": max((p["n_ok"] for p in a.programs), default=-1),
            "force_turn": getattr(a, "force_turn", 0), "rescued": getattr(a, "rescued", 0),
            "n_train": len(task["train"]),
            **({"view": a.view} if getattr(a, "view", None) else {}),
            "programs": a.programs, "transcript": a.transcript}


def _view_back(att, view):
    """Map an attempt's test grids from `view` back to the task's own orientation / colours (None if not a grid)."""
    import augment
    back = augment.back(view, att.task_id)
    for p in att.programs:
        out = []
        for g in p.get("test") or []:
            try:
                ok = isinstance(g, list) and g and all(isinstance(r, list) and len(r) == len(g[0]) for r in g)
                out.append(back(g) if ok else None)
            except Exception:                   # a malformed grid is no answer, never a crash
                out.append(None)
        p["test"] = out


def solve_repl(tasks, client, attempts=2, concurrency=32, deadline=None, stop_after=1,
               log=print, on_done=None, task_kw=None, attempt_views=None, **kw):
    """Same return shape as agent.solve: (answers, report).

    `on_done(record)` is called as each attempt finishes, so a caller can persist
    results incrementally -- probe24 lost everything to one late exception.

    `attempt_views` (c23fp, 10-31 after X70): attempt k sees the task in view attempt_views[k] (agent/augment.py, e.g.
    [None, "transpose"]: the second attempt, which only runs on tasks the first did not verify, sees it transposed);
    its test grids are mapped back before on_done or the answers read them."""
    deadline = deadline or float("inf")
    shared = _Shared()
    views = list(attempt_views or [])
    # attempt-major order: every task gets its first attempt before any gets a second
    atts = [Attempt(tid, k) for k in range(attempts) for tid in tasks]
    t0 = time.time()
    done_n = [0]

    def work(att):
        view = views[att.idx] if att.idx < len(views) else None
        try:
            # task_kw lets arms of an A/B share one run: key -> per-arm overrides. An arm
            # may bring its own client (different sampling / chat-template kwargs).
            over = dict(kw, **(task_kw or {}).get(att.task_id, {}))
            arm_client = over.pop("client", client)
            task = tasks[att.task_id]
            if view:
                import augment
                att.view, task = view, augment.task_view(task, view, att.task_id)
            run_attempt(att, task, arm_client, shared, deadline,
                        stop_after=stop_after, seed=1000 * att.idx, **over)
        except Exception as e:                  # one attempt must never sink the run
            import traceback
            att.status, att.note = "crashed", f"{type(e).__name__}: {e}"[:300]
            att.transcript.append({"crash": traceback.format_exc()[-2000:]})
        finally:                                # X69: never leave a stateful python process behind
            if getattr(att, "_session", None) is not None:
                att._session.close()
            if view:
                _view_back(att, view)
        if on_done:
            try:
                on_done(attempt_record(att, tasks[att.task_id]))
            except Exception as e:
                log(f"on_done failed: {e}")
        with shared.lock:
            done_n[0] += 1
            n_ver = sum(1 for v in shared.verified.values() if v)
        log(f"[{time.time() - t0:6.0f}s] {att.task_id}#{att.idx} {att.status:8s} "
            f"turns {att.turns:2d} py {att.python_calls:2d} sub {att.submits} "
            f"tok {att.gen_tokens:6d} | done {done_n[0]}/{len(atts)} tasks verified {n_ver}")
        return att

    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        list(ex.map(work, atts))

    answers = {tid: aggregate(tid, tasks[tid], atts) for tid in tasks}
    report = {"wall_s": time.time() - t0,
              "attempts": [attempt_record(a, tasks[a.task_id]) for a in atts]}
    return answers, report

In [ ]:
%%writefile /kaggle/working/agent/agent.py
"""Batched program-synthesis agent for ARC tasks.

Every task gets K independent attempts (conversations). Each ROUND sends every
live conversation to the model in ONE batch -- that is what keeps the GPUs
busy -- then runs each extracted `transform` on the task's train and test inputs
in the sandbox. An attempt whose program reproduces every train output is
VERIFIED and stops. Otherwise the model is shown where its program failed and
asked again, up to R rounds.

The verification against the demos is the point of the whole approach: a
program that reproduces all examples is a far stronger selector than any
likelihood score (exp06: the NLL scorer prefers a wrong grid over the injected
truth for 83 % of missed outputs).

Answers per test output are voted over verified programs; if fewer than two
distinct verified answers exist, the gap is filled from the best unverified
programs (most train examples right), and the source of each answer is kept so
a fallback solver can be preferred over weak guesses.
"""
import time
from collections import Counter, defaultdict

from grids import SYSTEM, task_prompt, feedback_prompt
from backend import extract_code
from sandbox import run_many


def visible_reply(text, keep=6000):
    """What goes back into the conversation: the answer without the reasoning.

    Qwen3-style models wrap reasoning in <think>; gpt-oss via raw harmony output
    puts the answer after a 'final' channel marker. Either way the program is in
    the tail, and re-sending long reasoning would crowd out the context.
    """
    t = text or ""
    if "</think>" in t:
        t = t.split("</think>")[-1]
    for marker in ("<|channel|>final<|message|>", "assistantfinal"):
        if marker in t:
            t = t.split(marker)[-1]
    t = t.strip()
    return t[-keep:] if len(t) > keep else t


def _key(g):
    return tuple(tuple(r) for r in g)


class Attempt:
    __slots__ = ("task_id", "idx", "messages", "rounds", "status", "programs",
                 "tokens", "note")

    def __init__(self, task_id, idx, prompt):
        self.task_id, self.idx = task_id, idx
        self.messages = [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": prompt}]
        self.rounds = 0
        self.status = "active"          # active | verified | failed | dropped
        self.programs = []              # dicts: round, code, n_ok, n_train, test
        self.tokens = 0
        self.note = ""


def solve(tasks, backend, samples=4, rounds=3, max_tokens=None, deadline=None,
          stop_after=2, max_prompt_tokens=None, sandbox_timeout=10.0,
          sandbox_workers=None, log=print):
    """tasks: {task_id: {"train": [...], "test": [...]}}.

    Returns (answers, report). answers[task_id] is a list over test inputs of
    [(grid, source), ...] best first; report holds per-attempt diagnostics.
    """
    t0 = time.time()
    deadline = deadline or float("inf")
    attempts = [Attempt(tid, i, task_prompt(task))
                for tid, task in tasks.items() for i in range(samples)]
    verified_count = Counter()
    rounds_log = []

    for rnd in range(1, rounds + 1):
        live = [a for a in attempts if a.status == "active"
                and verified_count[a.task_id] < stop_after]
        for a in attempts:
            if a.status == "active" and verified_count[a.task_id] >= stop_after:
                a.status, a.note = "failed", "task already verified"
        if not live:
            break
        if time.time() > deadline:
            log(f"round {rnd}: deadline reached with {len(live)} attempts live")
            break
        if max_prompt_tokens:
            keep = []
            for a in live:
                if backend.prompt_tokens(a.messages) > max_prompt_tokens:
                    a.status, a.note = "dropped", "prompt too long"
                else:
                    keep.append(a)
            live = keep
            if not live:
                break

        tg = time.time()
        replies = backend.chat([a.messages for a in live], max_tokens=max_tokens,
                               metas=[{"task_id": a.task_id, "attempt": a.idx} for a in live])
        gen_s = time.time() - tg

        jobs, owners = [], []
        for a, (text, ntok, finish) in zip(live, replies):
            a.rounds += 1
            a.tokens += ntok
            code = extract_code(text)
            a.messages.append({"role": "assistant", "content": visible_reply(text)})
            if code is None:
                why = ("Your reply was cut off before the program." if finish == "length"
                       else "No ```python block defining transform() was found.")
                a.messages.append({"role": "user", "content":
                                   why + " Give the complete program in ONE ```python code block."})
                if a.rounds >= rounds:
                    a.status, a.note = "failed", "no code"
                continue
            task = tasks[a.task_id]
            inputs = [p["input"] for p in task["train"]] + [t["input"] for t in task["test"]]
            jobs.append((code, inputs))
            owners.append((a, code))

        ts = time.time()
        results = run_many(jobs, timeout=sandbox_timeout, workers=sandbox_workers)
        run_s = time.time() - ts

        n_ver = 0
        for (a, code), res in zip(owners, results):
            task = tasks[a.task_id]
            ntr = len(task["train"])
            train_res = []
            for r, pair in zip(res[:ntr], task["train"]):
                ok = "output" in r and r["output"] == pair["output"]
                train_res.append({"ok": ok, "output": r.get("output"), "error": r.get("error")})
            n_ok = sum(t["ok"] for t in train_res)
            test_out = [r.get("output") for r in res[ntr:]]
            a.programs.append({"round": a.rounds, "code": code, "n_ok": n_ok,
                               "n_train": ntr, "test": test_out,
                               "test_errors": [r.get("error") for r in res[ntr:]]})
            if n_ok == ntr and all(o is not None for o in test_out):
                a.status = "verified"
                verified_count[a.task_id] += 1
                n_ver += 1
            elif a.rounds >= rounds:
                a.status, a.note = "failed", f"best {max(p['n_ok'] for p in a.programs)}/{ntr}"
            else:
                a.messages.append({"role": "user",
                                   "content": feedback_prompt(train_res, task["train"])})

        toks = sum(r[1] for r in replies)
        rounds_log.append({"round": rnd, "live": len(live), "programs": len(jobs),
                           "verified_now": n_ver, "gen_s": gen_s, "run_s": run_s,
                           "tokens": toks, "tok_per_s": toks / max(gen_s, 1e-9)})
        log(f"round {rnd}: {len(live)} live, {len(jobs)} programs, {n_ver} newly verified, "
            f"tasks verified {sum(1 for t in tasks if verified_count[t])}/{len(tasks)}, "
            f"gen {gen_s:.0f}s ({toks / max(gen_s, 1e-9):.0f} tok/s) run {run_s:.1f}s")

    answers = {tid: aggregate(tid, tasks[tid], attempts) for tid in tasks}
    report = {
        "rounds": rounds_log,
        "wall_s": time.time() - t0,
        "attempts": [{"task_id": a.task_id, "idx": a.idx, "status": a.status,
                      "rounds": a.rounds, "tokens": a.tokens, "note": a.note,
                      "best_ok": max((p["n_ok"] for p in a.programs), default=-1),
                      "n_train": len(tasks[a.task_id]["train"]),
                      "programs": a.programs} for a in attempts],
    }
    return answers, report


def aggregate(tid, task, attempts):
    """Per test input: candidate grids best first, each tagged with its source.

    Verified programs vote; unverified programs are ranked by train examples
    solved, then by how many programs agree.
    """
    mine = [a for a in attempts if a.task_id == tid]
    out = []
    for j in range(len(task["test"])):
        ver = Counter()
        unv = defaultdict(lambda: [0, 0])          # key -> [best n_ok, votes]
        grids = {}
        for a in mine:
            for p in a.programs:
                g = p["test"][j] if j < len(p["test"]) else None
                if g is None:
                    continue
                k = _key(g)
                grids[k] = g
                if p["n_ok"] == p["n_train"]:
                    ver[k] += 1
                else:
                    u = unv[k]
                    u[0] = max(u[0], p["n_ok"])
                    u[1] += 1
        ranked = [(grids[k], f"verified x{n}") for k, n in ver.most_common()]
        seen = {k for k in ver}
        rest = sorted((k for k in unv if k not in seen),
                      key=lambda k: (-unv[k][0], -unv[k][1]))
        ranked += [(grids[k], f"unverified {unv[k][0]}/{len(task['train'])}") for k in rest]
        out.append(ranked)
    return out


def to_submission(answers, tasks, fallback=None):
    """Kaggle format. `fallback(task_id, j)` supplies grids when the agent has
    fewer than two; otherwise the test input is echoed."""
    sub = {}
    for tid, task in tasks.items():
        rows = []
        for j, t in enumerate(task["test"]):
            cands = [g for g, _ in answers.get(tid, [[]] * len(task["test"]))[j]]
            if fallback and len(cands) < 2:
                for g in fallback(tid, j):
                    if g not in cands:
                        cands.append(g)
            while len(cands) < 2:
                cands.append(t["input"])
            rows.append({"attempt_1": cands[0], "attempt_2": cands[1]})
        sub[tid] = rows
    return sub


def score(sub, solutions):
    """ARC Prize scoring: a task scores the fraction of its test outputs matched
    by either attempt; the total is the sum over tasks (and / n as a percent)."""
    total, per = 0.0, {}
    for tid, sols in solutions.items():
        rows = sub.get(tid, [])
        hit = 0
        for j, sol in enumerate(sols):
            if j < len(rows) and sol in (rows[j]["attempt_1"], rows[j]["attempt_2"]):
                hit += 1
        per[tid] = hit / len(sols)
        total += per[tid]
    return total, 100.0 * total / max(len(solutions), 1), per

In [ ]:
%%writefile /kaggle/working/agent/sandbox.py
"""Run model-written `transform` programs in isolated subprocesses.

Each job is (code, inputs). The program runs in a fresh interpreter with a
wall-clock timeout (and, on Linux, an address-space cap), so an infinite loop,
a crash or a memory blow-up costs one job, never the run. Anything the program
prints is swallowed; the result travels back as one marked JSON line.

Per input the result is {"output": grid} or {"error": message}. An output is
only accepted if it is a 2D rectangular grid of ints 0-9, 1..30 on each side.
"""
import json
import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

MARK = "@@ARC_RESULT@@"

RUNNER = r'''
import sys, json, io, contextlib, copy, traceback
MARK = "@@ARC_RESULT@@"
job = json.loads(sys.stdin.read())

def clean(res):
    try:
        import numpy as np
        if isinstance(res, np.ndarray):
            res = res.tolist()
    except Exception:
        pass
    if isinstance(res, tuple):
        res = list(res)
    if not isinstance(res, list) or not res:
        return None, "output is not a non-empty list of rows"
    rows = []
    for row in res:
        if hasattr(row, "tolist"):
            row = row.tolist()
        if not isinstance(row, (list, tuple)) or not row:
            return None, "output rows must be non-empty lists"
        out = []
        for v in row:
            try:
                iv = int(v)
            except Exception:
                return None, f"non-integer cell {v!r}"
            if iv != v or not 0 <= iv <= 9:
                return None, f"cell value {v!r} outside 0-9"
            out.append(iv)
        rows.append(out)
    w = len(rows[0])
    if any(len(r) != w for r in rows):
        return None, "output is not rectangular"
    if not (1 <= len(rows) <= 30 and 1 <= w <= 30):
        return None, f"output size {len(rows)}x{w} outside 1..30"
    return rows, None

results = []
sink = io.StringIO()
ns = {"__name__": "__arc_program__"}
try:
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        exec(compile(job["code"], "<program>", "exec"), ns)
    fn = ns.get("transform")
    if not callable(fn):
        raise NameError("no function named transform")
except Exception as e:
    msg = "".join(traceback.format_exception_only(type(e), e)).strip()
    results = [{"error": "program failed to load: " + msg[-400:]} for _ in job["inputs"]]
else:
    for grid in job["inputs"]:
        try:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                res = fn(copy.deepcopy(grid))
            rows, err = clean(res)
            results.append({"output": rows} if err is None else {"error": err})
        except Exception as e:
            tb = traceback.format_exc().strip().splitlines()
            results.append({"error": " | ".join(tb[-3:])[-400:]})
sys.stdout.write(MARK + json.dumps(results) + "\n")
sys.stdout.flush()
'''


def _limits(mem_bytes):
    def apply():
        try:
            import resource
            resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
        except Exception:
            pass
    return apply if os.name == "posix" else None


def run_one(code, inputs, timeout=10.0, mem_bytes=2 << 30):
    """Run one program on a list of input grids; never raises."""
    try:
        p = subprocess.run(
            [sys.executable, "-c", RUNNER],
            input=json.dumps({"code": code, "inputs": inputs}),
            capture_output=True, text=True, timeout=timeout,
            preexec_fn=_limits(mem_bytes),
            env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
                     MKL_NUM_THREADS="1", PYTHONHASHSEED="0"))
    except subprocess.TimeoutExpired:
        return [{"error": f"timed out after {timeout:.0f}s"} for _ in inputs]
    for line in reversed(p.stdout.splitlines()):
        if line.startswith(MARK):
            try:
                res = json.loads(line[len(MARK):])
                if len(res) == len(inputs):
                    return res
            except Exception:
                break
    tail = (p.stderr or "").strip().splitlines()[-2:]
    why = " | ".join(tail)[-300:] or f"exit code {p.returncode}"
    return [{"error": "program crashed: " + why} for _ in inputs]


def run_many(jobs, timeout=10.0, workers=None, mem_bytes=2 << 30):
    """jobs: list of (code, inputs). Returns results in the same order."""
    workers = workers or max(1, (os.cpu_count() or 2) - 1)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(run_one, c, i, timeout, mem_bytes) for c, i in jobs]
        return [f.result() for f in futs]

In [ ]:
%%writefile /kaggle/working/agent/backend.py
"""LLM backends. The agent only needs one call: complete a batch of chats.

`VLLMBackend` runs offline on the Kaggle GPUs. `MockBackend` replays scripted
programs so the whole harness can be exercised on CPU with no model.
"""
import re
import time

CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)
THINK_RE = re.compile(r"<think>.*?</think>", re.S)


def extract_code(text):
    """The last fenced block that defines transform(); falls back to any block
    containing it, then to a bare `def transform` in unfenced text."""
    if not text:
        return None
    body = THINK_RE.sub("", text)
    if "<think>" in body and "</think>" not in body:
        return None                      # truncated inside the reasoning
    blocks = [b for b in CODE_RE.findall(body) if "def transform" in b]
    if blocks:
        return blocks[-1].strip()
    i = body.rfind("def transform")
    if i >= 0:                           # unfenced: take the function to the end
        return "import numpy as np\n" + body[i:].strip()
    return None


class VLLMBackend:
    """Offline vLLM chat. One engine over all GPUs (tensor parallel)."""

    def __init__(self, model, tp=4, max_model_len=32768, gpu_mem=0.90,
                 max_tokens=16384, temperature=0.8, top_p=0.95, seed=0,
                 chat_kwargs=None, quantization=None, enforce_eager=False):
        from vllm import LLM, SamplingParams
        self.SamplingParams = SamplingParams
        self.llm = LLM(model=model, tensor_parallel_size=tp, max_model_len=max_model_len,
                       gpu_memory_utilization=gpu_mem, seed=seed, trust_remote_code=True,
                       quantization=quantization, enforce_eager=enforce_eager)
        self.max_tokens, self.temperature, self.top_p = max_tokens, temperature, top_p
        self.chat_kwargs = chat_kwargs or {}
        self.seed = seed
        self.tokenizer = self.llm.get_tokenizer()

    def prompt_tokens(self, messages):
        ids = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                                 **self.chat_kwargs)
        return len(ids)

    def chat(self, conversations, max_tokens=None, metas=None):
        """Returns [(text, n_generated_tokens, finish_reason)] in order."""
        params = [self.SamplingParams(temperature=self.temperature, top_p=self.top_p,
                                      max_tokens=max_tokens or self.max_tokens,
                                      seed=self.seed + i)
                  for i in range(len(conversations))]
        self.seed += len(conversations)
        outs = self.llm.chat(conversations, params, use_tqdm=False,
                             chat_template_kwargs=self.chat_kwargs)
        res = []
        for o in outs:
            c = o.outputs[0]
            res.append((c.text, len(c.token_ids), c.finish_reason))
        return res


class MockBackend:
    """Replays `script(task_id, round, attempt, conversation) -> text`.

    `metas` (task_id, attempt) travel beside the conversations, never inside
    them, so the vLLM path sees plain chat messages.
    """

    def __init__(self, script, delay=0.0):
        self.script = script
        self.delay = delay
        self.calls = 0

    def prompt_tokens(self, messages):
        return sum(len(m["content"]) for m in messages) // 3

    def chat(self, conversations, max_tokens=None, metas=None):
        self.calls += 1
        time.sleep(self.delay)
        out = []
        for conv, meta in zip(conversations, metas or [{}] * len(conversations)):
            rnd = sum(m["role"] == "assistant" for m in conv)
            text = self.script(meta.get("task_id"), rnd, meta.get("attempt", 0), conv)
            out.append((text, len(text) // 3, "stop"))
        return out

In [ ]:
%%writefile /kaggle/working/agent/task_facts.py
"""X69 (2026-10-28): exact facts about every grid of a task, computed by a program, for the agent's first message.

The trace review (analysis/TRACE-REVIEW-2026-10-27.md) found ~half of every Qwen trace is hand bookkeeping -- re-typed
rows and coordinate lists -- and X63's agent spent a median 33k tokens re-typing preloaded grids before its first python
call. This block hands over the measurements it was rebuilding: sizes, colour counts, objects (X48's grid_objects), and
for same-size examples which cells change and how. Pure Python, no dependencies.
"""
from collections import Counter

from grid_objects import describe


def _size(g):
    return f"{len(g)}x{len(g[0])}"


def _colours(g):
    c = Counter(v for row in g for v in row)
    return "{" + ", ".join(f"{k}: {c[k]}" for k in sorted(c)) + "}"


def _objects(g):
    """grid_objects.describe without its long header (the block's own header says what an object is)."""
    head, _, rest = describe(g).partition("\n")
    lines = [f"objects: {head.rsplit(': ', 1)[-1]}"] + [ln for ln in rest.splitlines() if ln]
    return "\n".join("  " + ln for ln in lines)


def changes(a, b, max_kinds=8):
    """Same-size grids: how many cells differ and the colour transitions, most common first."""
    moves = Counter((x, y) for ra, rb in zip(a, b) for x, y in zip(ra, rb) if x != y)
    n = sum(moves.values())
    if not n:
        return "no cells differ (output = input)"
    kinds = ", ".join(f"{x}->{y}: {k}" for (x, y), k in moves.most_common(max_kinds))
    more = f", ... {len(moves) - max_kinds} more kinds" if len(moves) > max_kinds else ""
    return f"{n} cells differ ({kinds}{more})"


def facts(task):
    lines = ["## Exact facts computed by python ((row, col) from 0; objects = same-colour cells connected through any "
             "of the 8 neighbours, background = the most common colour)"]
    for i, p in enumerate(task["train"], 1):
        a, b = p["input"], p["output"]
        same = len(a) == len(b) and len(a[0]) == len(b[0])
        lines.append(f"Example {i}: input {_size(a)} -> output {_size(b)}"
                     + (f", same size; {changes(a, b)}" if same else ""))
        lines.append(f"  input colours {_colours(a)}; output colours {_colours(b)}")
        lines.append("  input " + _objects(a).lstrip())
        lines.append("  output " + _objects(b).lstrip())
    for j, t in enumerate(task["test"], 1):
        g = t["input"]
        lines.append(f"Test input {j}: {_size(g)}; colours {_colours(g)}")
        lines.append("  " + _objects(g).lstrip())
    return "\n".join(lines)

In [ ]:
%%writefile /kaggle/working/agent/grid_objects.py
"""X48 (2026-10-21): an exact object list under each grid of the direct prompt, so the model does not have to count,
locate or measure objects by eye (the deep-dive's failures: 58490d8a recounted objects and got one too many;
c4d067a0 mis-did block-position arithmetic while re-checking an example).

An object = a set of same-colour, non-background cells connected through any of the 8 neighbours. The background is
the most frequent colour (0 on ties). Coordinates are (row, column) from 0. Per colour: its cell count and objects, each
with its size, bounding box and -- when the box is at most 5x5 -- its shape (# = the colour, . = not). Long lists are
summarised so a noisy grid cannot flood the prompt. Pure Python, no dependencies.
"""
from collections import Counter

MAX_PER_COLOUR = 10          # objects listed per colour; the rest are summarised by size
MAX_SHAPE = 5                # shape drawn when the bounding box is at most this many rows and columns


def background(grid):
    c = Counter(v for row in grid for v in row)
    top = max(c.values())
    return 0 if c.get(0) == top else min(k for k, v in c.items() if v == top)


def objects(grid, bg=None):
    """[(colour, [(r, c), ...]), ...] in reading order of each object's first cell."""
    bg = background(grid) if bg is None else bg
    h, w = len(grid), len(grid[0])
    seen = [[False] * w for _ in range(h)]
    out = []
    for r in range(h):
        for c in range(w):
            if seen[r][c] or grid[r][c] == bg:
                continue
            col, stack, cells = grid[r][c], [(r, c)], []
            seen[r][c] = True
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        yy, xx = y + dy, x + dx
                        if 0 <= yy < h and 0 <= xx < w and not seen[yy][xx] and grid[yy][xx] == col:
                            seen[yy][xx] = True
                            stack.append((yy, xx))
            out.append((col, sorted(cells)))
    return out


def _one(cells):
    rs, cs = [r for r, _ in cells], [c for _, c in cells]
    r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)
    if len(cells) == 1:
        return f"1 cell at ({r0},{c0})"
    s = f"{len(cells)} cells, rows {r0}-{r1}, cols {c0}-{c1}"
    hh, ww = r1 - r0 + 1, c1 - c0 + 1
    if len(cells) == hh * ww:
        return s + f" (solid {hh}x{ww} rectangle)"
    if hh <= MAX_SHAPE and ww <= MAX_SHAPE:
        on = set(cells)
        s += " shape " + "/".join("".join("#" if (r, c) in on else "." for c in range(c0, c1 + 1))
                                  for r in range(r0, r1 + 1))
    return s


def describe(grid):
    """The object list as text lines (no trailing newline)."""
    bg = background(grid)
    objs = objects(grid, bg)
    counts = Counter(v for row in grid for v in row)
    lines = [f"Objects (one colour, 8-connected; background {bg}; (row,col) from 0): {len(objs)}"]
    for col in sorted({c for c, _ in objs}):
        mine = [cells for c, cells in objs if c == col]
        head = f"  colour {col}: {counts[col]} cells in {len(mine)} object{'s' if len(mine) != 1 else ''}"
        items = [_one(cells) for cells in mine[:MAX_PER_COLOUR]]
        rest = mine[MAX_PER_COLOUR:]
        if rest:
            sizes = Counter(len(cells) for cells in rest)
            items.append(f"... and {len(rest)} more (sizes " + ", ".join(f"{k}x{v}" for k, v in sorted(sizes.items())) + ")")
        lines.append(head + " -- " + "; ".join(items))
    return "\n".join(lines)

In [ ]:
%%writefile /kaggle/working/agent/run_direct.py
"""Direct-answer mode: the model reasons (e.g. Qwen3.8 at reasoning_effort xhigh) and
writes the test output grid(s) itself -- no tools. Two samples per task give the two
attempts. This is how headline ARC-AGI-2 numbers for chat models are usually measured,
so it is the comparison the REPL agent has to beat.

    python run_direct.py '<json config>'

Config: base_url, model, task_ids, samples (2), concurrency, max_tokens, deadline_epoch,
temperature/top_p/top_k, template_kwargs, bad_words, penalize_words / penalty, images, out,
data_dir.
Writes <out> (JSON) and <out>.jsonl (one line per finished sample).
"""
import json
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                       # noqa: E402
from grids import show                              # noqa: E402

# grids.SYSTEM tells the model it solves puzzles "by writing Python programs" -- right for
# the REPL agent, wrong when the answer is the grid itself.
DIRECT_SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles. Each puzzle has a few input->output "
    "example pairs that share one hidden transformation rule. Grids are 2D arrays of digits "
    "0-9 (colours); 0 is usually the background."
)

# A reply cut off before any grid (the thinking budget missing or ignored, or the answer
# itself cut) gets ONE short follow-up without thinking: the tail of its own reasoning,
# and an order to commit. An unanswered task scores nothing; a committed guess may score.
RESCUE = """{prompt}

## Your notes so far (they were cut off)
{notes}

Stop analysing now. Apply the rule you were converging on and give the final test output grid(s): each as rows of digits in its own ```grid code block, in test order, with nothing else in the block."""

DIRECT_INSTRUCTIONS = """Solve this ARC puzzle. The examples share one hidden transformation rule. Work out the rule, check it against every example, then apply it to the test input(s).

At the very end of your answer, give each test output grid as rows of digits inside its own ```grid code block, in test order, with nothing else in the block."""

# X46 (2026-10-20): the same answer, plus the rule as a program after the grids. The program is run on every example
# input: one wrong example proves the rule wrong; passing all of them makes its test output the stronger candidate.
PROGRAM_INSTRUCTIONS = """Solve this ARC puzzle. The examples share one hidden transformation rule. Work out the rule, check it against every example, then apply it to the test input(s).

At the end of your answer, give each test output grid as rows of digits inside its own ```grid code block, in test order, with nothing else in the block. After the grid block(s), write the rule as a Python function `transform(grid)` in one ```python code block: it takes one input grid (a list of rows of ints 0-9) and returns its output grid in the same form; numpy may be used. It will be run on every example input and on the test input(s), so it must implement the general rule, not copy an example's output."""


# X48 (2026-10-21): an exact object list under every grid (grid_objects.describe), so the model need not count, locate or
# measure objects by eye -- the trace deep-dive's failures. Off unless objects=True; the prompt is then unchanged.
OBJECTS_NOTE = ("Under each grid is a list of its objects, computed exactly by a program: an object is a group of "
                "same-colour cells (not the background colour) connected through any of their 8 neighbours, and "
                "(row,col) count from 0. Use these counts, positions and sizes instead of counting cells by hand.")


# X31 (2026-10-17): NVARC's top candidates as hints. Off unless a hints list is passed; the prompt is then unchanged.
HINTS_INTRO = """## Candidate outputs from another solver
A separate model trained only on ARC grids proposed the output grids below. On puzzles like this one it is often wrong. Work out the rule from the examples yourself. Use a candidate only if the rule you found produces exactly that grid; otherwise write your own."""


WAIT_WORDS = re.compile(r"\b(?:wait|hmm|alternatively)\b", re.I)


def direct_prompt(task, hints=None, program=False, objects=False):
    parts = [PROGRAM_INSTRUCTIONS if program else DIRECT_INSTRUCTIONS, ""] + ([OBJECTS_NOTE, ""] if objects else [])
    if objects:
        from grid_objects import describe        # lazy: the hybrid notebook ships run_direct without grid_objects.py
        g = lambda grid, name: show(grid, name) + "\n" + describe(grid)
    else:
        g = show
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", g(p["input"], "Input"), g(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", g(t["input"], "Input"), ""]
    if hints and any(hints):
        parts += [HINTS_INTRO, ""]
        for j, cands in enumerate(hints, 1):
            for c, h in zip("ABCDEFGH", cands):
                parts += [show(h["grid"], f"Test {j}, candidate {c} (found in {h['views']} of its 16 views)"), ""]
    return "\n".join(parts).rstrip()


# X38 (2026-10-18): each grid also as an image, right after its digits. One added note says how to read them.
VISION_NOTE = ("Each grid is shown twice: as rows of digits and, right after them, as an image (colours: {colours}). "
               "Use the images to see shapes and objects, and the digits for exact cells.")


def direct_content(task, hints=None):
    """direct_prompt's text with every grid followed by its PNG: OpenAI content parts (text / image_url)."""
    assert not hints, "hints are not combined with images"
    from grid_image import data_url, COLOUR_NOTE
    parts, buf = [], [DIRECT_INSTRUCTIONS, "", VISION_NOTE.format(colours=COLOUR_NOTE), ""]

    def flush():
        text = "\n".join(buf)
        if text.strip():
            parts.append({"type": "text", "text": text})
        buf.clear()

    def grid(g, name):
        buf.append(show(g, name))
        flush()
        parts.append({"type": "image_url", "image_url": {"url": data_url(g)}})
        buf.append("")                    # the next text part starts on a new line

    for i, p in enumerate(task["train"], 1):
        buf.append(f"## Example {i}")
        grid(p["input"], "Input")
        grid(p["output"], "Output")
    for j, t in enumerate(task["test"], 1):
        buf.append(f"## Test input {j}")
        grid(t["input"], "Input")
    flush()
    return parts


def parse_grids(text, n_tests):
    """The last n_tests digit-only code blocks (```grid preferred, any fence accepted)."""
    blocks = re.findall(r"```(?:grid)?[^\n]*\n(.*?)```", text or "", re.S)
    grids = []
    for b in blocks:
        rows = [r.strip() for r in b.strip().splitlines() if r.strip()]
        rows = [r.replace(" ", "").replace(",", "") for r in rows]
        if rows and all(r.isdigit() for r in rows) and len({len(r) for r in rows}) == 1:
            grids.append([[int(c) for c in r] for r in rows])
    return grids[-n_tests:] if len(grids) >= n_tests else None


def parse_program(text):
    """X46: the last ```python block of the answer that defines transform, or None."""
    blocks = re.findall(r"```(?:python|py)[^\n]*\n(.*?)```", text or "", re.S)
    progs = [b for b in blocks if re.search(r"^\s*def transform\s*\(", b, re.M)]
    return progs[-1] if progs else None


def check_program(code, task, timeout=20.0):
    """X46: run the program on every example input and every test input in one sandboxed subprocess. prog_n_ran counts
    the examples that returned a valid grid (no crash): n_ran == n_train with n_ok < n_train is a clean falsification."""
    from sandbox import run_one                     # lazy: the hybrid notebook ships run_direct without sandbox.py
    n = len(task["train"])
    res = run_one(code, [p["input"] for p in task["train"]] + [t["input"] for t in task["test"]], timeout=timeout)
    errs = [r["error"] for r in res if "error" in r]
    return {"prog_n_train": n, "prog_n_ran": sum("output" in r for r in res[:n]),
            "prog_n_ok": sum(r.get("output") == p["output"] for r, p in zip(res[:n], task["train"])),
            "prog_test": [r.get("output") for r in res[n:]], "prog_error": errs[0][:300] if errs else None}


# X47 / hybrid option (2026-10-21): after a natural answer, ask for the rule as a program with thinking OFF (no earlier
# reasoning sent). Word for word X47's follow-up arm (agent/x47_run.FOLLOWUP; agent/test_followup.py checks they match).
FOLLOWUP_PROGRAM = ("Write the rule you applied above as a Python function `transform(grid)` in one ```python code block: "
                    "it takes one input grid (a list of rows of ints 0-9) and returns its output grid in the same form; "
                    "numpy may be used. It will be run on every example input and on the test input(s), so it must "
                    "implement the general rule, not copy an example's output. Reply with the code block only.")


def followup_program(client, task, prompt, answer_text, seed, timeout, max_tokens=8192):
    """One thinking-off follow-up after an answer: its transform(grid), run on the examples and test inputs."""
    r = client.chat([{"role": "system", "content": DIRECT_SYSTEM}, {"role": "user", "content": prompt},
                     {"role": "assistant", "content": answer_text}, {"role": "user", "content": FOLLOWUP_PROGRAM}],
                    seed=seed, max_tokens=max_tokens, timeout=timeout, template_kwargs={"enable_thinking": False})
    if "error" in r:
        return {"followup_error": r["error"][:200], "followup_tokens": 0}
    out = {"followup_tokens": r.get("completion_tokens", 0),
           "program": parse_program(r.get("content")) or parse_program(r.get("reasoning"))}
    if out["program"]:
        out.update(check_program(out["program"], task))
    return out


def answer(client, task, seed, deadline, max_tokens=None, rescue_tokens=8192, notes_chars=16000, commit="rescue",
           keep_full=False, hints=None, images=False, program=False, objects=False, followup=False):
    """One direct answer to `task`: reason, then write the test grid(s). Returns a record;
    rec["grids"] is the list of test grids, or None when nothing parseable came back.
    commit="continue": a reply cut while still thinking first gets client.commit() -- the same
    prompt + its own reasoning + </think>, answered in context (status "committed"); the old
    rescue call remains the fallback. commit="rescue" (default) is the pre-2026-10-12 behaviour.
    images=True (X38): the main call shows every grid as digits AND an image; the rescue stays text-only.
    program=True (X46): the prompt also asks for transform(grid) after the grids; the record gets the whole answer
    text, the program, and check_program's results on the examples and test inputs.
    objects=True (X48): every grid in the prompt is followed by its exact object list (grid_objects.describe).
    followup=True (X47 in the hybrid): a natural answer gets followup_program(); when that program reproduces every
    example and returns a grid for every test input, its outputs replace the hand-written grids (kept as grids_hand)."""
    n = len(task["test"])
    prompt = direct_prompt(task, hints, program, objects)
    assert not (images and (program or objects)), "the image prompt has no program instructions or object lists"
    assert not (images and commit == "continue"), "the forced commit re-tokenizes a text-only prompt"
    user = direct_content(task, hints) if images else prompt

    def timeout():
        return max(60.0, min(4 * 3600, deadline - time.time() + 60))

    mt = max_tokens or client.max_tokens
    for _ in range(3):
        r = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                         {"role": "user", "content": user}],
                        seed=seed, max_tokens=mt, timeout=timeout())
        if not (r.get("context_overflow") and mt > 8192):
            break
        mt = int(mt * 0.75)                    # a long prompt: shrink the reply, never give up
    if "error" in r:
        return {"status": "error: " + r["error"][:200], "grids": None, "gen_tokens": 0}
    rec = {"finish": r["finish_reason"], "gen_tokens": r["completion_tokens"],
           "reasoning_chars": len(r["reasoning"] or ""),
           # X37: how often the reasoning second-guesses itself (should be ~0 when these words are banned)
           "wait_words": len(WAIT_WORDS.findall(r["reasoning"] or "")),
           "content_tail": (r["content"] or "")[-1500:],
           "reasoning_tail": (r["reasoning"] or "")[-3000:]}
    if keep_full:                     # 2026-10-15: whole trace for offline analysis (popped and saved by the caller)
        rec["_full"] = (r["reasoning"] or "") + "\n\n=== ANSWER ===\n" + (r["content"] or "")
    grids = parse_grids(r["content"], n)
    if program:                       # X46: checked whether or not the grids parsed
        rec["content"] = r["content"] or ""
        rec["program"] = parse_program(r["content"])
        if rec["program"]:
            rec.update(check_program(rec["program"], task))
    if grids and followup and deadline - time.time() > 120:
        f = followup_program(client, task, prompt, r["content"] or "", seed + 3, timeout())
        rec.update({k: v for k, v in f.items() if k != "prog_test"})
        rec["gen_tokens"] += f.get("followup_tokens", 0)
        pt = f.get("prog_test") or []
        if (f.get("program") and f.get("prog_n_ok") == f.get("prog_n_train") and len(pt) == n
                and all(g is not None for g in pt)):
            rec["grids_hand"], rec["grids_source"] = grids, "followup_program"
            grids = pt
    if grids:
        return dict(rec, status="answered", grids=grids)
    if commit == "none":
        # 2026-10-15: no forced answer at all (no commit, no rescue). The commit sub scored 13.61 vs 21.53 / 24.86
        # without it, and a forced grid must never replace NVARC's slot 2 in the hybrid.
        return dict(rec, status="no grid", grids=None)
    if (commit == "continue" and r["finish_reason"] == "length" and (r["reasoning"] or "").strip()
            and deadline - time.time() > 120):
        r1 = client.commit([{"role": "system", "content": DIRECT_SYSTEM}, {"role": "user", "content": prompt}],
                           r["reasoning"], max_tokens=rescue_tokens, seed=seed + 2, timeout=timeout())
        if "error" in r1:
            rec["commit_error"] = r1["error"][:200]
        else:
            rec["gen_tokens"] += r1["completion_tokens"]
            rec.update({"commit_finish": r1["finish_reason"], "commit_tail": (r1["content"] or "")[-1500:],
                        "commit_prompt_tokens": r1.get("commit_prompt_tokens")})
            grids = parse_grids(r1["content"], n)
            if grids:
                return dict(rec, status="committed", grids=grids)
    notes =((r["reasoning"] or "") + "\n" + (r["content"] or "")).strip()
    if notes and deadline - time.time() > 120:
        r2 = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                          {"role": "user", "content": RESCUE.format(prompt=prompt,
                                                                    notes=notes[-notes_chars:])}],
                         seed=seed + 1, max_tokens=rescue_tokens, timeout=timeout(),
                         template_kwargs={"enable_thinking": False, "reasoning_effort": "low"})
        if "error" in r2:
            rec["rescue_error"] = r2["error"][:200]
        else:
            rec["gen_tokens"] += r2["completion_tokens"]
            rec["rescue_finish"] = r2["finish_reason"]
            rec["rescue_tail"] = ((r2["content"] or "") or (r2["reasoning"] or ""))[-1500:]
            # Without thinking, a reasoning parser may still file the reply as reasoning.
            grids = parse_grids(r2["content"], n) or parse_grids(r2["reasoning"], n)
            if grids:
                return dict(rec, status="rescued", grids=grids)
    return dict(rec, status="no grid", grids=None)


def budget_probe(cfg, budget=256):
    """Does the SERVER's thinking_token_budget fire? (It never did through 2026-10-11.) One question that
    invites long thinking, with a 256-token budget: if it fires, the reply has answer text after ~256
    reasoning tokens; if not, it runs to max_tokens (4096) with no answer text."""
    c = ChatClient(cfg["base_url"], cfg["model"], max_tokens=4096, temperature=1.0, top_p=0.95, top_k=20,
                   thinking=True, thinking_token_budget=budget, template_kwargs=cfg.get("template_kwargs"),
                   timeout=1800)
    msgs = [{"role": "user", "content": "Find every pair of primes (p, q) with p < q and p + q = 200. "
                                        "Check each pair carefully, then list them."}]
    r = c.chat(msgs, seed=1)
    out = {"budget": budget, "error": r.get("error"), "finish": r.get("finish_reason"),
           "completion_tokens": r.get("completion_tokens"), "reasoning_chars": len(r.get("reasoning") or ""),
           "content_chars": len(r.get("content") or ""), "budget_still_sent": c.thinking_token_budget is not None}
    out["fired"] = bool(out["content_chars"]) and (out["completion_tokens"] or 0) < 4096
    if not out["fired"] and (r.get("reasoning") or "").strip():
        # The client-side commit on this cut reply: proves /tokenize + /v1/completions work before the run.
        r1 = c.commit(msgs, r["reasoning"], max_tokens=512, seed=2)
        out["commit"] = {"error": r1.get("error"), "finish": r1.get("finish_reason"),
                         "content_head": (r1.get("content") or "")[:300], "prompt_tokens": r1.get("commit_prompt_tokens")}
    print("BUDGET PROBE", json.dumps(out), flush=True)
    return out


def word_bias(cfg):
    """X37s (2026-10-18): {token id: penalty} for every word in cfg["penalize_words"] that the SERVED tokenizer maps to
    exactly one token (multi-token words are skipped: biasing their first piece would hit unrelated words). Uses
    vLLM's /tokenize, so no tokenizer is loaded here. Printed loudly: an empty map means the run has no penalty."""
    import urllib.request
    root = cfg["base_url"].rsplit("/v1", 1)[0]
    bias, skipped = {}, []
    for w in cfg["penalize_words"]:
        try:
            req = urllib.request.Request(root + "/tokenize", data=json.dumps(
                {"model": cfg["model"], "prompt": w, "add_special_tokens": False}).encode(),
                headers={"Content-Type": "application/json"})
            toks = json.loads(urllib.request.urlopen(req, timeout=60).read())["tokens"]
        except Exception as e:                                   # noqa: BLE001
            skipped.append(f"{w!r}: {type(e).__name__}")
            continue
        if len(toks) == 1:
            bias[str(toks[0])] = float(cfg.get("penalty", -3.0))
        else:
            skipped.append(f"{w!r}: {len(toks)} tokens")
    print(f"LOGIT BIAS {len(bias)} tokens at {cfg.get('penalty', -3.0)}: {bias} | skipped {skipped}", flush=True)
    if not bias:
        raise SystemExit("LOGIT BIAS EMPTY -- run stopped: it would only repeat the control")
    return bias


def main(cfg):
    t0 = time.time()
    root = cfg.get("data_dir", "/kaggle/input")
    import glob
    ch = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_challenges.json", recursive=True)[0]))
    sol = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_solutions.json", recursive=True)[0]))
    ids = cfg["task_ids"]
    bias = word_bias(cfg) if cfg.get("penalize_words") else None
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 49152),
                        temperature=cfg.get("temperature", 1.0), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"), timeout=4 * 3600,
                        presence_penalty=cfg.get("presence_penalty"), bad_words=cfg.get("bad_words"),
                        logit_bias=bias)
    if cfg.get("bad_words") or bias:
        # X37 / X37s: one tiny request first -- a server that refuses bad_words / logit_bias (vLLM 0.19 refuses
        # logit_bias whenever MTP is on) stops the run here, not every answer for hours.
        probe = client.chat([{"role": "user", "content": "Say OK."}], max_tokens=4, timeout=300,
                            template_kwargs={"enable_thinking": False})
        if "error" in probe:
            raise SystemExit(f"BAD_WORDS / LOGIT BIAS REJECTED BY THE SERVER -- run stopped: {probe['error'][:300]}")
        print(f"accepted: bad_words {cfg.get('bad_words')} | logit_bias {bias}", flush=True)
    out = cfg.get("out", "/kaggle/working/direct_results.json")
    jsonl = out.replace(".json", ".jsonl")
    deadline = cfg.get("deadline_epoch", float("inf"))
    lock = threading.Lock()
    recs = []
    # X70: cfg["views"] (e.g. ["transpose", "colour"]) answers every task once per view (agent/augment.py) and maps
    # the grids back before scoring; per-task scores are then keyed "<task>@<view>". Without it: unchanged.
    views = cfg.get("views") or [None]
    if views != [None] and cfg.get("smoke"):
        # X70 smoke: the smallest task in the last view, thinking off and a short cap -- the server answers and the
        # grid maps back to the task's own orientation / colours before hours of xhigh answers are spent.
        import augment
        t = min(ids, key=lambda i: len(json.dumps(ch[i])))
        v = views[-1]
        sc = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg["smoke"].get("max_tokens", 4096),
                        thinking=False, template_kwargs=cfg.get("template_kwargs"), timeout=1800)
        r = answer(sc, augment.task_view(ch[t], v, t), 7, deadline, commit="none")
        g = [augment.back(v, t)(x) for x in r["grids"]] if r.get("grids") else None
        ok = bool(g) and all(len(a) == len(s) and len(a[0]) == len(s[0]) for a, s in zip(g, sol[t]))
        print(f"SMOKE {'ok' if ok else 'FAILED'}: {t}@{v} status {r.get('status')} tokens {r.get('gen_tokens')} "
              f"shapes back {[(len(a), len(a[0])) for a in g] if g else None} truth {[(len(s), len(s[0])) for s in sol[t]]} "
              f"right {g == sol[t] if g else None}", flush=True)
        if "error" in str(r.get("status")):
            raise SystemExit("smoke request failed -- run stopped")
    jobs = [(t, k, v) for k in range(cfg.get("samples", 2)) for t in ids for v in views]   # sample-major
    probe = budget_probe(cfg) if cfg.get("budget_probe") else None
    hints = json.load(open(cfg["hints_path"])) if cfg.get("hints_path") else {}   # X31: {task: [[cand, ...] per test]}

    def work(job):
        t, k, v = job
        rec = {"task_id": t, "sample": k, **({"view": v} if v else {})}
        if time.time() > deadline:
            rec["status"] = "skipped: deadline"
        else:
            ts = time.time()
            task = ch[t]
            if v:
                import augment
                task = augment.task_view(ch[t], v, t)
            rec.update(answer(client, task, 1000 * k + 7, deadline,
                              rescue_tokens=cfg.get("rescue_tokens", 8192), commit=cfg.get("commit", "rescue"),
                              keep_full=bool(cfg.get("save_full_dir")), hints=hints.get(t),
                              images=cfg.get("images", False)))
            rec["wall"] = round(time.time() - ts, 1)
            full = rec.pop("_full", None)
            if full is not None:
                import gzip
                os.makedirs(cfg["save_full_dir"], exist_ok=True)
                path = os.path.join(cfg["save_full_dir"], f"{t}_{v}_{k}.txt.gz" if v else f"{t}_{k}.txt.gz")
                with gzip.open(path, "wt", encoding="utf-8") as f:
                    f.write(full)
                rec["full_trace"] = os.path.basename(path)
            if v and rec["grids"]:
                import augment
                rec["grids_view"] = rec["grids"]
                rec["grids"] = [augment.back(v, t)(g) for g in rec["grids"]]
            grids = rec["grids"]
            rec["correct"] = [g == s for g, s in zip(grids, sol[t])] if grids else None
            if t in hints:                           # X31: which shown candidate (if any) the answer copied, per test
                rec["copied"] = ([next(("ABCDEFGH"[i] for i, h in enumerate(hints[t][j]) if h["grid"] == g), None)
                                  for j, g in enumerate(grids)] if grids else None)
        with lock:
            recs.append(rec)
            with open(jsonl, "a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"[{time.time() - t0:6.0f}s] {t}{'@' + v if v else ''}#{k} {rec['status']:10s} "
                  f"tok {rec.get('gen_tokens', 0):6d} finish {rec.get('finish')} "
                  f"correct {rec.get('correct')}", flush=True)

    with ThreadPoolExecutor(max_workers=cfg.get("concurrency", 16)) as ex:
        list(ex.map(work, jobs))

    # pass@2 per task: any test output matched by either sample (fractional per test).
    per = {}
    for t in ids:
        for v in views:
            n = len(sol[t])
            hit = [False] * n
            for r in recs:
                if r["task_id"] == t and r.get("view") == v and r.get("correct"):
                    hit = [h or c for h, c in zip(hit, r["correct"])]
            per[f"{t}@{v}" if v else t] = sum(hit) / n
    tok = [r["gen_tokens"] for r in recs if "gen_tokens" in r]
    waits = [r["wait_words"] for r in recs if "wait_words" in r]
    if views != [None]:
        import augment
        rep = augment.report(recs, ids, views, sol, cfg.get("groups") or {"all": ids})
        print("X70 REPORT", json.dumps(rep), flush=True)
    summary = {"config": cfg, "logit_bias": bias, "per_task": per, "score": sum(per.values()), "n_tasks": len(ids),
               "samples": len(recs), "mean_gen_tokens": sum(tok) / max(len(tok), 1),
               "mean_wait_words": sum(waits) / max(len(waits), 1),
               "cut_at_max_tokens": sum(1 for r in recs if r.get("finish") == "length"),
               "wall_s": time.time() - t0, "records": recs,
               "statuses": {s: sum(1 for r in recs if r.get("status") == s) for s in {r.get("status") for r in recs}},
               "budget_probe": probe}
    json.dump(summary, open(out, "w"))
    print(f"\nDIRECT SCORE {summary['score']:.2f} / {len(ids)}  mean tokens {summary['mean_gen_tokens']:.0f}  "
          f"cut {summary['cut_at_max_tokens']}  wall {summary['wall_s']:.0f}s", flush=True)
    print("per task:", {t: round(v, 2) for t, v in per.items()}, flush=True)


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))

In [ ]:
%%writefile /kaggle/working/agent/run_probe.py
"""Run the LLM-agent probe against a live vLLM OpenAI server (inside Kaggle).

    python run_probe.py '<json config>'

Config keys: base_url, model, task_ids | n_tasks, attempts, concurrency,
max_turns, max_gen_tokens, context_budget, max_tokens, deadline_epoch,
bench_requests, out.

Writes one JSON with: throughput benchmark, per-attempt report, submission,
score against the evaluation solutions, and per-task results.
"""
import glob
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient, _post               # noqa: E402
from repl_agent import solve_repl, AGENT_OPTS      # noqa: E402
from agent import to_submission, score             # noqa: E402


def find(name, root="/kaggle/input"):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def bench(client, n, max_tokens=512):
    """Aggregate decode throughput with n concurrent plain requests."""
    t = time.time()
    with ThreadPoolExecutor(max_workers=n) as ex:
        rs = list(ex.map(lambda i: client.chat(
            [{"role": "user", "content": f"Write a detailed story about grid puzzle number {i}."}],
            max_tokens=max_tokens, seed=i), range(n)))
    dt = time.time() - t
    toks = sum(r.get("completion_tokens", 0) for r in rs)
    errs = [r["error"] for r in rs if "error" in r]
    return {"requests": n, "seconds": dt, "tokens": toks, "tok_per_s": toks / dt,
            "errors": errs[:3]}


def make_client(cfg):
    return ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 16384),
                      temperature=cfg.get("temperature", 0.6), top_p=cfg.get("top_p", 0.95),
                      top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                      reasoning_effort=cfg.get("reasoning_effort"),
                      thinking_token_budget=cfg.get("thinking_token_budget"),
                      template_kwargs=cfg.get("template_kwargs"))


def think_check(base_url, model, template_kwargs, timeout=120):
    """X44 (2026-10-19): does the server's chat template render an EARLIER assistant turn's reasoning? X40 ran without
    it, so every turn started without its earlier thinking. /tokenize one conversation without and with ~300 words of
    reasoning on its first assistant turn -- once followed by a user message, once by a tool result -- and return the
    first message key whose reasoning adds >= 80% of its own token count in both (or None), with the counts. No
    generation: a few seconds."""
    root = base_url.rsplit("/v1", 1)[0]
    kw = {"enable_thinking": True, **(template_kwargs or {})}
    words = " ".join(f"note{i}: the red square moves one cell right." for i in range(40))
    n_words = len(_post(root + "/tokenize", {"model": model, "prompt": words, "add_special_tokens": False},
                        timeout)["tokens"])
    convs = {"user_after": [{"role": "user", "content": "Solve the puzzle."},
                            {"role": "assistant", "content": "Let me test it."},
                            {"role": "user", "content": "Continue."}],
             "tool_after": [{"role": "user", "content": "Solve the puzzle."},
                            {"role": "assistant", "content": "", "tool_calls": [{
                                "id": "c1", "type": "function",
                                "function": {"name": "python", "arguments": json.dumps({"code": "print(1)"})}}]},
                            {"role": "tool", "tool_call_id": "c1", "content": "1"}]}

    def count(msgs):
        return len(_post(root + "/tokenize", {"model": model, "messages": msgs, "add_generation_prompt": True,
                                              "chat_template_kwargs": kw}, timeout)["tokens"])
    res = {"reasoning_tokens": n_words}
    for key in ("reasoning_content", "reasoning"):
        ok = True
        for name, msgs in convs.items():
            with_r = [dict(m) for m in msgs]
            with_r[1][key] = words
            try:                          # X62: SGLang may reject a key its message schema lacks -- try the next
                res[f"{key}:{name}"] = count(with_r) - count(msgs)
            except Exception as e:        # noqa: BLE001
                res[f"{key}:{name}"] = f"error: {type(e).__name__}: {e}"[:200]
                ok = False
                break
            ok = ok and res[f"{key}:{name}"] >= 0.8 * n_words
        if ok:
            return key, res
    return None, res


def main(cfg):
    t0 = time.time()
    root = cfg.get("data_dir", "/kaggle/input")
    ch = json.load(open(find("arc-agi_evaluation_challenges.json", root)))
    sol = json.load(open(find("arc-agi_evaluation_solutions.json", root)))
    ids = cfg.get("task_ids") or sorted(ch)[:cfg.get("n_tasks", 2)]
    tasks = {t: ch[t] for t in ids}
    print(f"probe on {len(tasks)} tasks: {ids}", flush=True)

    client = make_client(cfg)
    out = {"config": cfg, "task_ids": ids}
    if cfg.get("bench_requests"):
        for n in (1, cfg["bench_requests"]):
            b = bench(client, n)
            out.setdefault("bench", []).append(b)
            print(f"bench {n:3d} concurrent: {b['tok_per_s']:.0f} tok/s "
                  f"({b['tokens']} tok in {b['seconds']:.0f}s) errors {b['errors']}", flush=True)

    path = cfg.get("out", "/kaggle/working/probe_results.json")
    jsonl = path.replace(".json", "_attempts.jsonl")

    # Arms of an A/B run side by side on the same server: every task is duplicated
    # as "<task>@<arm>" with that arm's overrides, so the comparison is paired.
    arms = cfg.get("arms") or {"": {}}
    run_tasks, task_kw = {}, {}
    # Task-major order: a task's arms are queued together, so no arm systematically
    # starts later and loses more attempts to the deadline.
    # An arm's "client" dict overrides the client settings (sampling, template_kwargs,
    # thinking budget) -- e.g. a reasoning-effort sweep on one server.
    arm_clients = {}
    for arm, over in arms.items():
        if "client" in over:
            arm_clients[arm] = make_client(dict(cfg, **over["client"]))
    for t in ids:
        for arm, over in arms.items():
            key = f"{t}@{arm}" if arm else t
            kw = {k: v for k, v in over.items() if k != "client"}
            if arm in arm_clients:
                kw["client"] = arm_clients[arm]
            run_tasks[key], task_kw[key] = tasks[t], kw
    print(f"arms: {arms}", flush=True)

    # X44: an arm that keeps reasoning must have it rendered by the template, or it would repeat X40's blind turns.
    if cfg.get("think_check"):
        for arm, over in arms.items():
            if not over.get("keep_reasoning"):
                continue
            tk = dict(cfg.get("template_kwargs") or {}, **((over.get("client") or {}).get("template_kwargs") or {}))
            key, res = think_check(cfg["base_url"], cfg["model"], tk)
            out.setdefault("think_check", {})[arm or "(single)"] = dict(res, key=key)
            print(f"THINK CHECK arm {arm or '(single)'}: key {key} | {res}", flush=True)
            if key is None:
                out["error"] = f"arm {arm}: the chat template does not render earlier reasoning -- run stopped"
                json.dump(out, open(path, "w"))
                print("THINK CHECK FAILED -- earlier thinking would not reach the model; stopping.", flush=True)
                return out
            for k in task_kw:
                if (k.split("@", 1)[1] if "@" in k else "") == arm:
                    task_kw[k]["reasoning_key"] = key

    # X69: a stateful arm needs a working python session (checked without the model), then a tiny end-to-end attempt
    # (first task, first arm, a few turns) must reach a python call before the full run spends hours.
    if any(over.get("stateful") for over in arms.values()):
        from repl import Session
        s = Session(next(iter(tasks.values())))
        got = [s.run("x = 41\nprint(len(train) > 0)"), s.run("x + 1")]
        s.close()
        out["session_check"] = got
        print(f"SESSION CHECK {'ok' if got == ['True', '42'] else 'FAILED'}: {got}", flush=True)
        if got != ["True", "42"]:
            out["error"] = "stateful python session check failed -- run stopped"
            json.dump(out, open(path, "w"))
            return out
    if cfg.get("smoke"):
        key = next(iter(run_tasks))
        kw = dict(task_kw[key], **{k: v for k, v in cfg["smoke"].items() if k not in ("max_turns", "max_gen_tokens")})
        _, rep = solve_repl({key: run_tasks[key]}, client, attempts=1, concurrency=1, deadline=cfg.get("deadline_epoch"),
                            max_turns=cfg["smoke"].get("max_turns", 3),
                            max_gen_tokens=cfg["smoke"].get("max_gen_tokens", 8000),
                            context_budget=cfg.get("context_budget", 28000), nudge_every=cfg.get("nudge_every", 6),
                            log=lambda m: print("  smoke " + m, flush=True), task_kw={key: kw},
                            **{k: cfg[k] for k in AGENT_OPTS if k in cfg})
        a = rep["attempts"][0]
        sm = {k: a.get(k) for k in ("task_id", "status", "note", "turns", "python_calls", "submits", "gen_tokens",
                                    "name_errors", "session_restarts")}
        sm["first_tool_output"] = next((e["output"][:300] for e in a["transcript"] if "tool" in e), None)
        out["smoke"] = sm
        ok = a["status"] != "crashed" and a["python_calls"] + a["submits"] > 0
        print(f"SMOKE {'ok' if ok else 'FAILED'}: {json.dumps(sm)}", flush=True)
        if not ok:
            out["error"] = "smoke attempt made no tool call or crashed -- run stopped"
            json.dump(out, open(path, "w"))
            return out

    # Colab (2026-10-20): a rerun after a disconnect skips every task-arm that already has an attempt record in the
    # attempts JSONL -- that file, not this run's final JSON, is then the complete result (agent/x44_report.py).
    if cfg.get("resume") and os.path.exists(jsonl):
        have = set()
        for line in open(jsonl, encoding="utf-8"):
            try:
                have.add(json.loads(line)["task_id"])
            except (ValueError, KeyError):
                pass                                    # a line cut by a disconnect: that task runs again
        for k in [k for k in run_tasks if k in have]:
            run_tasks.pop(k)
            task_kw.pop(k)
        print(f"resume: {len(have)} task-arms already in {jsonl}; {len(run_tasks)} to run", flush=True)

    trace_dir = cfg.get("trace_dir")
    if trace_dir:
        os.makedirs(trace_dir, exist_ok=True)

    def on_done(rec):
        """Persist each attempt the moment it ends, with test correctness."""
        truth = sol[rec["task_id"].split("@")[0]]
        for p in rec["programs"]:
            p["test_correct"] = [o == s for o, s in zip(p["test"], truth)]
        with open(jsonl, "a") as f:
            f.write(json.dumps(rec) + "\n")
        if trace_dir:                     # X62 (user): every attempt's reasoning, replies and code, one gzip each
            import gzip
            name = f"{rec['task_id'].replace('@', '_')}_{rec.get('idx', 0)}.json.gz"
            with gzip.open(os.path.join(trace_dir, name), "wt", encoding="utf-8") as g:
                json.dump({k: rec.get(k) for k in ("task_id", "idx", "status", "turns", "gen_tokens", "prompt_tokens",
                                                   "cached_tokens", "overflows", "python_calls", "submits", "best_ok",
                                                   "n_train", "programs", "transcript")}, g)
        ver = [p for p in rec["programs"] if p["n_ok"] == p["n_train"]]
        if ver:
            print(f"    {rec['task_id']}: verified program test_correct={ver[-1]['test_correct']}",
                  flush=True)

    answers, report = solve_repl(
        run_tasks, client, attempts=cfg.get("attempts", 2), concurrency=cfg.get("concurrency", 16),
        deadline=cfg.get("deadline_epoch"), stop_after=cfg.get("stop_after", 1),
        max_turns=cfg.get("max_turns", 30), max_gen_tokens=cfg.get("max_gen_tokens", 120000),
        context_budget=cfg.get("context_budget", 28000), nudge_every=cfg.get("nudge_every", 6),
        log=lambda m: print(m, flush=True), on_done=on_done, task_kw=task_kw,
        **{k: cfg[k] for k in AGENT_OPTS if k in cfg})
    sub = to_submission(answers, run_tasks)
    total, pct, per = score(sub, {k: sol[k.split("@")[0]] for k in run_tasks})
    by_arm = {arm: sum(per.get(f"{t}@{arm}" if arm else t, 0.0) for t in ids) for arm in arms}   # resume: some absent
    gen = sum(a["gen_tokens"] for a in report["attempts"])
    out.update({"report": report, "submission": sub, "score_tasks": total, "score_pct": pct,
                "score_by_arm": by_arm,
                "per_task": per, "wall_s": time.time() - t0, "gen_tokens": gen,
                "answer_sources": {k: [[s for _, s in c[:2]] for c in answers[k]] for k in run_tasks}})
    # Transcripts go to their own file: they are large and only read for diagnosis.
    tx = {f"{a['task_id']}#{a['idx']}": a.pop("transcript", []) for a in report["attempts"]}
    json.dump(tx, open(path.replace(".json", "_transcripts.json"), "w"))
    json.dump(out, open(path, "w"))

    print("\n" + "=" * 72)
    for arm, v in by_arm.items():
        print(f"ARM {arm or '(single)'}: {v:.2f} / {len(ids)} tasks")
    print(f"SCORE {total:.2f} / {len(run_tasks)} task-arms = {pct:.1f} %   wall {out['wall_s']:.0f}s   "
          f"generated {gen} tokens ({gen / max(report['wall_s'], 1):.0f} tok/s agent-side)")
    ver = sorted({a['task_id'] for a in report['attempts'] if a['status'] == 'verified'})
    print(f"verified tasks {len(ver)}: {ver}")
    print(f"verified but wrong on test: {[t for t in ver if per[t] < 1]}")
    for t in run_tasks:
        atts = [a for a in report["attempts"] if a["task_id"] == t]
        print(f"  {t}  score {per[t]:.2f}  " + "  ".join(
            f"#{a['idx']}:{a['status']}/{a['best_ok']}of{a['n_train']}/t{a['turns']}/"
            f"py{a['python_calls']}/sub{a['submits']}/{a['gen_tokens']}tok" for a in atts))
    print(f"wrote {path} ({os.path.getsize(path)} bytes)")
    return out     # X63: flash_smoke.agent_eval read the score from this; without it X62 / X63's summary said 0 attempted


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))

In [ ]:
%%writefile /kaggle/working/agent/metrics.py
"""Sample a vLLM server's Prometheus /metrics while a probe runs (stdlib only).

dfranzen's ARC-AGI-3 write-up tuned serving against cache numbers -- 93% of prompt
tokens reused, prefill 15% of time -- and our own logs show why that matters here:
at 24 concurrent attempts (ab3q36) the server pushed 3.9 prompt tokens through
prefill per generated token and the prefix-cache hit rate decayed to ~1%, while at
8-9 (probe9q38, mini2arm) it was 0.2-1.0 with hits near 70%, for the same ~140
generated tok/s. Every probe now records these, so a concurrency or context change
can be judged on them.

    s = Sampler("http://127.0.0.1:1234/metrics", every=30); s.start()
    ...; s.stop(); summary = s.summary()
"""
import json
import re
import threading
import time
import urllib.request

# metric-name suffixes worth keeping (vLLM V1 names; older names are aliases)
KEEP = ("prompt_tokens_total", "generation_tokens_total", "num_preemptions_total",
        "prefix_cache_queries_total", "prefix_cache_hits_total", "kv_cache_usage_perc",
        "gpu_cache_usage_perc", "num_requests_running", "num_requests_waiting",
        "spec_decode_num_draft_tokens_total", "spec_decode_num_accepted_tokens_total",
        "request_success_total", "spec_accept_length")
LINE = re.compile(r"^([a-zA-Z_:][a-zA-Z0-9_:]*)(\{[^}]*\})?\s+([-+0-9.eEnaNinf]+)")
# llama.cpp's llama-server (--metrics) names for the same quantities (X20, 2026-10-11); SGLang's (X68, 2026-10-27:
# its prompt / generation counters only move when a request FINISHES -- version_bench counts tokens from the stream)
ALIAS = {"tokens_predicted_total": "generation_tokens_total", "requests_processing": "num_requests_running",
         "requests_deferred": "num_requests_waiting", "kv_cache_usage_ratio": "kv_cache_usage_perc",
         "num_running_reqs": "num_requests_running", "num_queue_reqs": "num_requests_waiting",
         "token_usage": "kv_cache_usage_perc", "num_retracted_requests_total": "num_preemptions_total"}


def parse(text):
    """Prometheus text -> {short_name: value}: counters (*_total) summed over label sets, gauges their max (X68: a
    gauge exported once per rank must not count 4x)."""
    out = {}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        m = LINE.match(line)
        if not m:
            continue
        name = m.group(1).split(":")[-1]
        name = ALIAS.get(name, name)
        if not name.endswith(KEEP):
            continue
        for k in KEEP:
            if name.endswith(k):
                name = k
        try:
            v = float(m.group(3))
        except ValueError:
            continue
        if name.endswith("_total"):
            out[name] = out.get(name, 0.0) + v
        else:
            out[name] = max(out.get(name, v), v)
    return out


def fetch(url, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return parse(r.read().decode(errors="replace"))


class Sampler(threading.Thread):
    def __init__(self, url, every=30.0):
        super().__init__(daemon=True)
        self.url, self.every = url, every
        self.samples, self.errors = [], 0
        self._stop_ev = threading.Event()

    def run(self):
        while True:
            try:
                self.samples.append({"t": time.time(), **fetch(self.url)})
            except Exception:
                self.errors += 1
            if self._stop_ev.wait(self.every):
                break

    def stop(self):
        self._stop_ev.set()
        self.join(timeout=30)
        try:                                     # one last sample so deltas cover the end
            self.samples.append({"t": time.time(), **fetch(self.url)})
        except Exception:
            self.errors += 1

    def summary(self):
        return summarise(self.samples, self.errors)


def summarise(samples, errors=0):
    if len(samples) < 2:
        return {"samples": len(samples), "errors": errors}
    a, b = samples[0], samples[-1]
    d = lambda k: b.get(k, 0.0) - a.get(k, 0.0)
    gen, prompt = d("generation_tokens_total"), d("prompt_tokens_total")
    q, h = d("prefix_cache_queries_total"), d("prefix_cache_hits_total")
    kv = [s.get("kv_cache_usage_perc", s.get("gpu_cache_usage_perc", 0.0)) for s in samples]
    kv = [x * 100 if x <= 1.0 else x for x in kv]           # vLLM reports a 0-1 fraction
    run = [s.get("num_requests_running", 0.0) for s in samples]
    wait = [s.get("num_requests_waiting", 0.0) for s in samples]
    drafts, acc = d("spec_decode_num_draft_tokens_total"), d("spec_decode_num_accepted_tokens_total")
    secs = b["t"] - a["t"]
    out = {"samples": len(samples), "errors": errors, "seconds": round(secs),
           "generated_tokens": int(gen), "prompt_tokens": int(prompt),
           "gen_tok_per_s": round(gen / max(secs, 1), 1),
           "prompt_per_generated": round(prompt / max(gen, 1), 2),
           "prefix_cache_hit_rate": round(h / q, 3) if q else None,
           # prompt tokens that had to be computed (not served from the prefix cache)
           "prefill_computed_per_generated": round((q - h) / max(gen, 1), 2) if q else None,
           "preemptions": int(d("num_preemptions_total")),
           "kv_usage_peak_pct": round(max(kv), 1), "kv_usage_mean_pct": round(sum(kv) / len(kv), 1),
           "running_mean": round(sum(run) / len(run), 1), "waiting_peak": int(max(wait)),
           "requests_done": int(d("request_success_total"))}
    if drafts:
        out["spec_acceptance"] = round(acc / drafts, 3)
    return out


def write(sampler, path):
    data = {"summary": sampler.summary(), "samples": sampler.samples}
    json.dump(data, open(path, "w"))
    return data["summary"]

In [ ]:
%%writefile /kaggle/working/agent/hyperqwen_bench.py
"""X8: HyperQwen (github.com/syv-ai/HyperQwen, vLLM 0.30 patches) on 4x L4.

    python hyperqwen_bench.py '<json>'     json: {"site", "deadline_epoch", "out", "configs": [...]}

Steps, each logged:
  1. apply HyperQwen's vLLM patch series to the installed vLLM 0.30.0 (GNU patch, fuzz 0);
  2. prepare the INT4 W4A16 AutoRound checkpoint (dbirks, on Kaggle as aihelpme/
     kcc-qwen38-27b-w4a16) with HyperQwen's CPU scripts: int8 lm_head / embed_tokens / MTP.
     The input is read-only, so the model dir is symlinks plus real copies of exactly the
     files the scripts rewrite;
  3. for each config: start their throughput ("batch") server split over 4 GPUs, run the
     agent-like load (real ARC prompts, thinking on, 2048 tokens) at several concurrencies,
     record tok/s, KV capacity and log facts, stop.
HyperQwen tests RTX 3090 (sm86) and 5090 (sm120); L4 (sm89) is untested upstream.
"""
import glob
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)


def one(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])


cfg_cu13_nvcc = False          # opt back in only with a matching nvcc / runtime header pair


def env_for(site, extra=None):
    e = dict(os.environ, PYTHONPATH=str(site), USE_TF="0", TRANSFORMERS_NO_TF="1",
             TRANSFORMERS_NO_TORCHVISION="1", VLLM_NO_USAGE_STATS="1",
             FLASHINFER_DISABLE_VERSION_CHECK="1", VLLM_USE_FLASHINFER_SAMPLER="0",
             # custom all-reduce cannot export a VMM graph buffer over CUDA IPC with TP>1 (#163)
             PYTORCH_CUDA_ALLOC_CONF="expandable_segments:False")
    # NOT the wheelhouse's CUDA 13 nvcc: it is nvidia-cuda-nvcc 13.4.92 while the CUDA headers
    # beside it come from nvidia-cuda-runtime 13.0.96 (torch's pin), and FlashInfer's CCCL stops
    # with "CUDA compiler and CUDA toolkit headers are incompatible" (X8 v1 and v3, 2026-10-09/10).
    # The image's own /usr/local/cuda nvcc compiled FlashInfer 0.6.18's sampler fine in X6 v1,
    # which then only failed to link -lcuda (fixed by cuda_link_env below).
    if cfg_cu13_nvcc and glob.glob(f"{site}/nvidia/cu13/bin/nvcc"):
        cu13 = f"{site}/nvidia/cu13"
        e["CUDA_HOME"] = cu13
        e["PATH"] = f"{cu13}/bin:" + e.get("PATH", "")
    # X8 v1 (2026-10-09): every config died linking FlashInfer's JIT kernels (`cannot find -lcuda`).
    from bench_serving import cuda_link_env
    e.update(cuda_link_env())
    e.update(extra or {})
    return e


def apply_patches(repo, site):
    if not shutil.which("patch"):
        raise RuntimeError("GNU patch is not installed in this image")
    sp = site / "vllm"
    r = subprocess.run(["bash", str(repo / "patches" / "apply.sh"), str(sp)], cwd=repo,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    if r.returncode != 0:
        raise RuntimeError(f"patches/apply.sh failed with {r.returncode}")


def prepare_model(repo, site, base, work):
    """Symlink the read-only checkpoint; copy the files the prepare scripts rewrite."""
    d = work / "Qwen3.8-27B-W4A16-AutoRound"
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)
    idx = json.load(open(base / "model.safetensors.index.json"))
    rewrite = {"config.json", "model.safetensors.index.json"}
    for k, shard in idx["weight_map"].items():
        if "lm_head" in k or "embed_tokens" in k or k.startswith("mtp") or ".mtp." in k:
            rewrite.add(shard)
    for f in base.iterdir():
        (shutil.copy2(f, d / f.name) if f.name in rewrite else os.symlink(f, d / f.name))
    print("copied for rewrite:", sorted(rewrite), flush=True)
    py = [sys.executable]
    e = env_for(site)
    for script in (["prepare/quant_lm_head.py", str(d)], ["prepare/quant_embed.py", str(d)],
                   ["prepare/quant_mtp.py", str(d)],
                   ["prepare/build_draft_vocab.py", str(d), "--ids", "prepare/draft_vocab_ids.json"]):
        t = time.time()
        r = subprocess.run(py + script, cwd=repo, env=e, capture_output=True, text=True)
        print(f"{script[0]}: rc {r.returncode} in {time.time() - t:.0f}s\n{r.stdout[-1500:]}{r.stderr[-1500:]}",
              flush=True)
        if r.returncode != 0:
            raise RuntimeError(f"{script[0]} failed")
    return d


def start(model, site, cfg, log):
    args = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model),
            "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", "1234",
            "--tensor-parallel-size", "4", "--gpu-memory-utilization", str(cfg.get("gpu_util", 0.92)),
            "--max-model-len", str(cfg.get("max_len", 65536)), "--max-num-seqs", str(cfg.get("max_seqs", 64)),
            "--language-model-only", "--kv-cache-dtype", "fp8", "--mamba-ssm-cache-dtype", "float16",
            "--async-scheduling", "--max-num-batched-tokens", "2048",
            "--compilation-config",
            json.dumps({"max_cudagraph_capture_size": 64, "custom_ops": ["+rms_norm", "+silu_and_mul"]}),
            "--reasoning-parser", "qwen3", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
            *cfg.get("extra", [])]
    extra_env = {}
    if cfg.get("int8_act", True):
        extra_env = {"VLLM_MARLIN_INPUT_DTYPE": "int8", "VLLM_MARLIN_INT8_INCLUDE_RE": "mlp"}
    print(" ".join(args), extra_env, flush=True)
    t = time.time()
    proc = subprocess.Popen(args, env=env_for(site, extra_env), stdout=open(log, "w"),
                            stderr=subprocess.STDOUT, start_new_session=True)
    while time.time() - t < 1800:
        text = Path(log).read_text(errors="replace")
        if proc.poll() is not None or "EngineCore failed" in text:
            return proc, False, time.time() - t
        try:
            urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=5)
            return proc, True, time.time() - t
        except Exception:
            time.sleep(10)
    return proc, False, time.time() - t


def stop(proc):
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        proc.kill()
    proc.wait(timeout=60)
    for _ in range(36):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.split()
        if used and all(int(u) < 1500 for u in used if u.strip().isdigit()):
            return
        time.sleep(5)


PREFLIGHT = r'''
import time, torch, flashinfer
t = time.time()
probs = torch.softmax(torch.randn(4, 1000, device="cuda"), -1)
out = flashinfer.sampling.top_p_sampling_from_probs(probs, 0.9)
torch.cuda.synchronize()
print("FLASHINFER_JIT_OK", tuple(out.shape), f"{time.time() - t:.0f}s", flush=True)
'''
TOOLCHAIN_ERRORS = ("Ninja build failed", "headers are incompatible", "cannot find -l")


def jit_preflight(site, timeout=900):
    """Build one FlashInfer JIT module (sampling) with the servers' exact env BEFORE the 6-minute
    model preparation and three server starts. X8 v1 and v3 spent ~2 h each to learn that
    FlashInfer could not build at all. Returns (ok, fatal, tail): fatal only for toolchain errors,
    so a mere API change in this snippet never blocks the bench."""
    try:
        r = subprocess.run([sys.executable, "-c", PREFLIGHT], env=env_for(site), capture_output=True,
                           text=True, timeout=timeout)
        tail = (r.stdout + r.stderr)[-3000:]
    except subprocess.TimeoutExpired:
        return False, False, f"preflight timed out after {timeout}s"
    ok = "FLASHINFER_JIT_OK" in tail
    return ok, (not ok and any(k in tail for k in TOOLCHAIN_ERRORS)), tail


def setup(site):
    """Patch vLLM, check the FlashInfer toolchain, prepare the INT4 checkpoint. Returns
    (model_dir, steps); raises on any failure, with the reason in steps["error"]."""
    steps = {}
    # The dataset may hold the repo at its root or under HyperQwen/: find it by its patch series.
    src = one("/kaggle/input/**/patches/series").parent.parent
    # Inputs are read-only; some scripts write beside themselves. 15 MB, so copy it.
    repo = Path("/kaggle/working/HyperQwen")
    shutil.rmtree(repo, ignore_errors=True)
    shutil.copytree(src, repo)
    base = one("/kaggle/input/**/qwen3.8-27b-w4a16/**/config.json").parent
    work = Path(max(["/kaggle/working", "/tmp"], key=lambda p: shutil.disk_usage(p).free))
    commit = (repo / "COMMIT").read_text().strip() if (repo / "COMMIT").exists() else "?"
    print("repo", repo, commit, "| base", base, "| work", work,
          {p: shutil.disk_usage(p).free // 2**30 for p in ("/kaggle/working", "/tmp")}, flush=True)
    try:
        t = time.time(); apply_patches(repo, site); steps["patches_s"] = round(time.time() - t)
        t = time.time(); ok, fatal, tail = jit_preflight(site)
        steps.update({"jit_preflight_s": round(time.time() - t), "jit_preflight_ok": ok,
                      "jit_preflight_tail": tail[-1500:]})
        print(f"FlashInfer JIT preflight ok={ok} fatal={fatal} in {time.time() - t:.0f}s\n{tail[-1500:]}",
              flush=True)
        if fatal:
            raise RuntimeError("FlashInfer cannot build kernels in this environment (see jit_preflight_tail)")
        t = time.time(); model = prepare_model(repo, site, base, work / "models")
        steps["prepare_s"] = round(time.time() - t)
    except Exception as e:
        steps["error"] = f"{type(e).__name__}: {e}"
        raise
    return model, steps


def serve(cfg):
    """X15: set up as main() does, then start ONE HyperQwen server and leave it running for the
    direct-answer runner. cfg: {"site", "server": {max_len, max_seqs, int8_act, gpu_util},
    "ladder": [[extra flags], ...]} -- the first option set that starts wins, as in the probes.
    Writes /kaggle/working/server_options.json and vllm-server.pid; exits 1 if nothing starts."""
    site = Path(cfg["site"])
    work = Path(cfg.get("work_dir", "/kaggle/working"))
    try:
        model, steps = setup(site)
    except Exception as e:
        print("SETUP FAILED:", e, flush=True)
        return 1
    print("setup steps:", json.dumps(steps)[:2000], flush=True)
    log = work / "vllm-server.log"
    for i, extra in enumerate(cfg.get("ladder", [[]])):
        proc, ok, secs = start(model, site, dict(cfg.get("server", {}), extra=extra), log)
        if ok:
            print(f"server ready after {secs:.0f}s\nSERVER OPTION SET {i}: {extra}", flush=True)
            (work / "server_options.json").write_text(json.dumps(
                {"index": i, "extra": extra, "server": cfg.get("server", {}), "steps": steps}))
            (work / "vllm-server.pid").write_text(str(proc.pid))
            lines = log.read_text(errors="replace").splitlines()
            print("\n".join(l[-220:] for l in lines if any(k in l for k in (
                "KV cache size", "Maximum concurrency", "took", "Marlin", "int8", "Using")))[-4000:], flush=True)
            return 0
        print(f"option set {i} failed after {secs:.0f}s: {extra}\n"
              + "\n".join(log.read_text(errors="replace").splitlines()[-30:]), flush=True)
        stop(proc)
    return 1


def fp8_bench(cfg):
    """X18 (2026-10-11): HyperQwen's patches on the FP8 model, against the same server flags on plain
    vLLM 0.30. Configs with "patched": false run first, on the unpatched install; then the patch
    series is applied and the "patched": true configs run. No INT4 prepare step (it rewrites an
    AutoRound checkpoint). Each config: short replies at `concurrency` (2,048 tokens) and a long
    load `long` = [n, max_tokens] (closer to real xhigh answers). cfg: site, model_glob, configs,
    deadline_epoch, out."""
    from bench_serving import load
    from repl_agent import repl_prompt
    site = Path(cfg["site"])
    out = cfg.get("out", "/kaggle/working/bench_results.json")
    results = {"steps": {}, "configs": []}
    src = one("/kaggle/input/**/patches/series").parent.parent
    work = cfg.get("work_dir", "/kaggle/working")
    repo = Path(work) / "HyperQwen"
    shutil.rmtree(repo, ignore_errors=True)
    shutil.copytree(src, repo)
    model = one(cfg["model_glob"]).parent
    print("repo", repo, "| model", model, flush=True)
    t = time.time(); ok, fatal, tail = jit_preflight(site)
    results["steps"].update({"jit_preflight_s": round(time.time() - t), "jit_preflight_ok": ok})
    print(f"FlashInfer JIT preflight ok={ok} fatal={fatal}\n{tail[-1500:]}", flush=True)
    if fatal:
        results["steps"]["error"] = "FlashInfer toolchain"
        json.dump(results, open(out, "w"), indent=1)
        return
    ch = json.load(open(one("/kaggle/input/**/arc-agi_evaluation_challenges.json")))
    prompts = [repl_prompt(ch[t]) for t in sorted(ch)[:64]]
    patched = False
    for c in sorted(cfg["configs"], key=lambda c: bool(c.get("patched"))):   # unpatched first
        if c.get("patched") and not patched:
            try:
                t = time.time(); apply_patches(repo, site); results["steps"]["patches_s"] = round(time.time() - t)
                patched = True
            except Exception as e:
                results["steps"]["patch_error"] = f"{type(e).__name__}: {e}"
                print("PATCHES FAILED:", e, flush=True)
                break
        if time.time() > cfg["deadline_epoch"] - 1500:
            results["configs"].append({"name": c["name"], "skipped": "deadline"})
            continue
        log = f"{work}/server_{c['name']}.log"
        rec = {"name": c["name"], "cfg": c, "patched": patched}
        proc, ok, secs = start(model, site, dict(c, int8_act=False), log)
        lines = Path(log).read_text(errors="replace").splitlines()
        rec.update({"started": ok, "start_s": round(secs),
                    "log_facts": [l[-220:] for l in lines if any(k in l for k in (
                        "KV cache size", "Maximum concurrency", "took", "backend", "Using"))][-25:]})
        if ok:
            rec["bench"] = [load(n, prompts) for n in c.get("concurrency", [16, 32, 64])]
            if c.get("long"):
                n, mt = c["long"]
                rec["long"] = load(n, prompts, max_tokens=mt)
        else:
            rec["error_tail"] = lines[-40:]
        stop(proc)
        results["configs"].append(rec)
        json.dump(results, open(out, "w"), indent=1)
        print(json.dumps({k: rec[k] for k in rec if k != "log_facts"}, indent=1)[:2500], flush=True)
    json.dump(results, open(out, "w"), indent=1)          # also records a patch failure
    print("\n" + "=" * 72)
    for r in results["configs"]:
        b = " | ".join(f"N={x['n']}: {x['tok_per_s']} tok/s" for x in r.get("bench", []))
        lg = r.get("long")
        lg = f" | long N={lg['n']}x{lg['tokens'] // max(1, lg['n'])}: {lg['tok_per_s']} tok/s" if lg else ""
        print(f"{r['name']:22s} patched={r.get('patched')} started={r.get('started')} {b or r.get('skipped', '')}{lg}",
              flush=True)


def main(cfg):
    from bench_serving import load
    from repl_agent import repl_prompt
    site = Path(cfg["site"])
    out = cfg.get("out", "/kaggle/working/bench_results.json")
    results = {"steps": {}, "configs": []}
    # The dataset may hold the repo at its root or under HyperQwen/: find it by its patch series.
    src = one("/kaggle/input/**/patches/series").parent.parent
    # Inputs are read-only; some scripts write beside themselves. 15 MB, so copy it.
    repo = Path("/kaggle/working/HyperQwen")
    shutil.rmtree(repo, ignore_errors=True)
    shutil.copytree(src, repo)
    base = one("/kaggle/input/**/qwen3.8-27b-w4a16/**/config.json").parent
    work = Path(max(["/kaggle/working", "/tmp"], key=lambda p: shutil.disk_usage(p).free))
    commit = (repo / "COMMIT").read_text().strip() if (repo / "COMMIT").exists() else "?"
    print("repo", repo, commit, "| base", base, "| work", work,
          {p: shutil.disk_usage(p).free // 2**30 for p in ("/kaggle/working", "/tmp")}, flush=True)
    try:
        t = time.time(); apply_patches(repo, site); results["steps"]["patches_s"] = round(time.time() - t)
        t = time.time(); ok, fatal, tail = jit_preflight(site)
        results["steps"].update({"jit_preflight_s": round(time.time() - t), "jit_preflight_ok": ok,
                                 "jit_preflight_tail": tail[-1500:]})
        print(f"FlashInfer JIT preflight ok={ok} fatal={fatal} in {time.time() - t:.0f}s\n{tail[-1500:]}",
              flush=True)
        if fatal:
            raise RuntimeError("FlashInfer cannot build kernels in this environment (see jit_preflight_tail)")
        t = time.time(); model = prepare_model(repo, site, base, work / "models")
        results["steps"]["prepare_s"] = round(time.time() - t)
    except Exception as e:
        results["steps"]["error"] = f"{type(e).__name__}: {e}"
        json.dump(results, open(out, "w"), indent=1)
        print("SETUP FAILED:", e, flush=True)
        return
    ch = json.load(open(one("/kaggle/input/**/arc-agi_evaluation_challenges.json")))
    prompts = [repl_prompt(ch[t]) for t in sorted(ch)[:64]]
    for c in cfg["configs"]:
        if time.time() > cfg["deadline_epoch"] - 1200:
            results["configs"].append({"name": c["name"], "skipped": "deadline"})
            break
        log = f"/kaggle/working/server_{c['name']}.log"
        rec = {"name": c["name"], "cfg": c}
        proc, ok, secs = start(model, site, c, log)
        lines = Path(log).read_text(errors="replace").splitlines()
        rec.update({"started": ok, "start_s": round(secs),
                    "log_facts": [l[-220:] for l in lines if any(k in l for k in (
                        "KV cache size", "Maximum concurrency", "took", "Marlin", "int8", "Using"))][-25:]})
        if ok:
            rec["bench"] = [load(n, prompts) for n in c["concurrency"]]
        else:
            rec["error_tail"] = lines[-40:]
        stop(proc)
        results["configs"].append(rec)
        json.dump(results, open(out, "w"), indent=1)
        print(json.dumps({k: rec[k] for k in rec if k != "log_facts"}, indent=1)[:2500], flush=True)
    print("\n" + "=" * 72)
    for r in results["configs"]:
        b = " | ".join(f"N={x['n']}: {x['tok_per_s']} tok/s" for x in r.get("bench", []))
        print(f"{r['name']:22s} started={r.get('started')} {b or r.get('skipped', '')}", flush=True)


if __name__ == "__main__":
    _cfg = json.loads(sys.argv[1])
    if _cfg.get("mode") == "serve":
        sys.exit(serve(_cfg))
    if _cfg.get("mode") == "fp8_bench":
        sys.exit(fp8_bench(_cfg))
    main(_cfg)

In [ ]:
%%writefile /kaggle/working/agent/flash_smoke.py
"""X28: can Qwen3.8-Flash-Next (125B MoE, ~5-6B active, 51B n-gram "PLE" table) be served on 4x L4? (2026-10-17)

    python flash_smoke.py '<json config>'      (site, model_glob, rungs, deadline_epoch, out)

Smoke test, not a benchmark. Steps, each written to <out> as it finishes:
  1. machine: RAM (and the container's cgroup limit), CPUs, GPU memory, disk read speed of the PLE files
     (sequential MB/s and random 4 KB reads/s -- is reading the table from disk on demand viable?);
  2. checkpoint: config summary, bytes per tensor group from the safetensors headers (what must sit on the
     GPUs vs the PLE table), chat-template flags;
  3. vLLM 0.30: version, whether the architecture is registered, every PLE-related env var / code line;
  4. FlashInfer JIT preflight (the X8/X18 toolchain);
  5. server ladder (one rung after another until one starts): RAM / server RSS (anon vs file-backed) sampled
     during the load -> is the PLE table copied into RAM or read from the mounted files;
  6. if a server is up: smoke reply; tok/s at 1 / mid / max at once (2,048-token replies, real ARC prompts);
     4 real direct xhigh answers at a 32,768 cap on X13 tasks Qwen3.8-27B solved quickly.
"""
import glob
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from hyperqwen_bench import env_for, jit_preflight, stop   # noqa: E402  (the vLLM 0.30 toolchain env of X8 / X18)

PORT = 1234
TASKS = ["2ba387bc", "7b5033c1", "31f7f899", "1ae2feb7"]   # X13 (Qwen3.8-27B FP8): solved in 14.7k / 22.3k / 33.6k / 39.1k


def meminfo():
    m = {}
    if not os.path.exists("/proc/meminfo"):
        return {"MemTotal": None, "MemAvailable": None}
    for line in open("/proc/meminfo"):
        k, v = line.split(":", 1)
        m[k] = int(v.split()[0]) // 1024                       # MiB
    out = {k: m.get(k) for k in ("MemTotal", "MemAvailable", "Cached", "SwapTotal")}
    for p in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v = open(p).read().strip()
            out["cgroup_limit_mib"] = v if v == "max" else int(v) // 2**20
            break
        except Exception:
            pass
    return out


def gpu_mem():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True).stdout.split("\n")
    return [l.strip() for l in r if l.strip()]


def group_rss(pgid):
    """Resident memory of every process in the server's process group, split anon (copied into RAM) vs file
    (pages of mapped files, e.g. a memory-mapped PLE table)."""
    anon = file_ = 0
    for d in glob.glob("/proc/[0-9]*"):
        try:
            if os.getpgid(int(d[6:])) != pgid:
                continue
            for line in open(d + "/status"):
                if line.startswith("RssAnon:"):
                    anon += int(line.split()[1])
                elif line.startswith("RssFile:"):
                    file_ += int(line.split()[1])
        except Exception:
            pass
    return {"rss_anon_gib": round(anon / 2**20, 1), "rss_file_gib": round(file_ / 2**20, 1)}


def disk_speed(files, seq_bytes=2 << 30, n_random=2000):
    out = {}
    if not files:
        return {"error": "no PLE files found"}
    t = time.time()
    n = 0
    with open(files[0], "rb", buffering=0) as f:
        while n < seq_bytes:
            b = f.read(64 << 20)
            if not b:
                break
            n += len(b)
    out["seq_mb_s"] = round(n / 2**20 / (time.time() - t), 1)
    rnd = random.Random(0)
    fhs = [open(p, "rb", buffering=0) for p in files[1:] or files]
    sizes = [os.fstat(f.fileno()).st_size for f in fhs]
    t = time.time()
    for _ in range(n_random):
        i = rnd.randrange(len(fhs))
        fhs[i].seek(rnd.randrange(0, max(1, sizes[i] - 4096)) // 4096 * 4096)
        fhs[i].read(4096)
    dt = time.time() - t
    for f in fhs:
        f.close()
    out.update(random_reads_s=round(n_random / dt), random_read_ms=round(1000 * dt / n_random, 3))
    return out


def tensor_groups(model):
    """Bytes per tensor group, from the safetensors headers only (8-byte length + JSON)."""
    groups = {}
    for p in sorted(model.glob("*.safetensors")):
        with open(p, "rb") as f:
            hdr = json.loads(f.read(int.from_bytes(f.read(8), "little")))
        for name, meta in hdr.items():
            if name == "__metadata__":
                continue
            n = meta["data_offsets"][1] - meta["data_offsets"][0]
            low = name.lower()
            g = ("ple/ngram" if any(k in low for k in ("ple", "per_layer", "ngram", "n_gram")) else
                 "mtp" if "mtp" in low else "vision" if any(k in low for k in ("visual", "vision")) else
                 "experts" if "expert" in low else "embed/lm_head" if any(k in low for k in ("embed", "lm_head")) else
                 "other")
            gg = groups.setdefault(g, {"bytes": 0, "tensors": 0, "example": name})
            gg["bytes"] += n
            gg["tensors"] += 1
    return {g: dict(v, gib=round(v["bytes"] / 2**30, 2)) for g, v in groups.items()}


def config_summary(model):
    c = json.load(open(model / "config.json"))
    tc = c.get("text_config", c)
    keys = {k: tc[k] for k in tc if any(s in k.lower() for s in ("layer", "hidden", "expert", "ple", "ngram",
                                                                   "head", "vocab", "mtp", "context", "position"))
            and not isinstance(tc[k], (list, dict))}
    q = c.get("quantization_config") or {}
    if (model / "quantization_config.json").exists() and not q:
        q = json.load(open(model / "quantization_config.json"))
    tmpl = (model / "chat_template.jinja").read_text(errors="replace") if (model / "chat_template.jinja").exists() else ""
    return {"architectures": c.get("architectures"), "model_type": c.get("model_type"), "text": keys,
            "quant": {k: q.get(k) for k in ("quant_method", "bits", "group_size", "sym", "packing_format",
                                            "data_type") if k in q},
            "template_flags": sorted(set(re.findall(r"(reasoning_effort|enable_thinking|thinking_budget|"
                                                    r"preserve_thinking)", tmpl)))}


def vllm_facts(site):
    code = ("import vllm, json\nfrom vllm.model_executor.models.registry import ModelRegistry\n"
            "a = ModelRegistry.get_supported_archs()\n"
            "print(json.dumps({'vllm': vllm.__version__, 'qwen4_archs': [x for x in a if 'qwen4' in x.lower()]}))")
    r = subprocess.run([sys.executable, "-c", code], env=env_for(site), capture_output=True, text=True, timeout=600)
    out = {"probe": (r.stdout.strip().splitlines() or [""])[-1][:2000], "stderr_tail": r.stderr[-800:]}
    hits = []
    for p in glob.glob(f"{site}/vllm/**/*.py", recursive=True):
        try:
            for i, line in enumerate(open(p, errors="replace")):
                if "PLE" in line and ("environ" in line or "envs." in line or "OFFLOAD" in line):
                    hits.append(f"{p[len(str(site)) + 1:]}:{i + 1}: {line.strip()[:160]}")
        except Exception:
            pass
    out["ple_code"] = hits[:40]
    out["ple_env_names"] = sorted(set(re.findall(r"VLLM_[A-Z0-9_]*PLE[A-Z0-9_]*", "\n".join(hits))))
    return out


SGL = {}            # set by main() when cfg backend is "sglang" (X29): {"bin": ..., "env": ...}


def start(model, site, rung, log, env_extra, timeout):
    if SGL:
        from sglang_launch import serve_cmd
        args, env = serve_cmd(SGL["bin"], model, rung, PORT), dict(SGL["env"], **rung.get("env", {}))   # X59: rung env
    else:
        args = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model),
                "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", str(PORT),
                "--tensor-parallel-size", "4", "--gpu-memory-utilization", str(rung["gpu_util"]),
                "--max-model-len", str(rung["max_len"]), "--max-num-seqs", str(rung["max_seqs"]),
                "--language-model-only", "--kv-cache-dtype", "fp8", "--trust-remote-code",
                "--reasoning-parser", "qwen3", "--generation-config", "vllm", *rung.get("extra", [])]
        env = env_for(site, env_extra)
    print(" ".join(map(str, args)), env_extra, flush=True)
    t = time.time()
    proc = subprocess.Popen(args, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    trace, ok = [], False
    while time.time() - t < timeout:
        trace.append(dict(t=round(time.time() - t), mem_avail_mib=meminfo()["MemAvailable"], **group_rss(proc.pid)))
        text = Path(log).read_text(errors="replace")
        if proc.poll() is not None or "EngineCore failed" in text:
            break
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5)
            ok = True
            break
        except Exception:
            time.sleep(20)
    trace.append(dict(t=round(time.time() - t), mem_avail_mib=meminfo()["MemAvailable"], **group_rss(proc.pid)))
    return proc, ok, round(time.time() - t), trace


def kv_facts(lines):
    """X41: the numbers that decide whether Flash-Next fits -- answer-context tokens the KV pool holds, memory left
    after the weights, the linear-attention state per request -- parsed from the SGLang / vLLM log."""
    text = "\n".join(lines)
    out = {}
    for key, pat in (("kv_tokens", r"max_total_num_tokens=(\d+)"), ("kv_tokens", r"#tokens: (\d+)"),
                     ("kv_tokens", r"GPU KV cache size: ([\d,]+) tokens"),
                     ("mamba_state_mb", r"mamba_cache_per_req=([\d.]+) MB"),
                     ("mamba_states", r"max_mamba_cache_size=(\d+)")):
        if key not in out:
            m = re.findall(pat, text)
            if m:
                out[key] = float(m[-1].replace(",", "")) if "." in m[-1] else int(m[-1].replace(",", ""))
    avail = re.findall(r"avail mem=([\d.]+) GB", text)
    if avail:
        out["avail_mem_gb"] = [float(a) for a in avail[-2:]]          # before the weights, after the weights
    return out


def spec_setup(cfg, model):
    """X59, once per run: the MTP drafter's SGLang view + FR-Spec's hot-token map (in the Pennyroyal bundle), and the
    DFlash2 drafter's directory (BF16, used as is)."""
    import sglang_launch as sl
    out = {"draft_files": []}
    if cfg.get("draft_glob"):
        try:
            hits = sorted(glob.glob(cfg["draft_glob"], recursive=True))
            if not hits:
                raise FileNotFoundError(f"no MTP drafter at {cfg['draft_glob']}")
            maps = sorted(glob.glob(cfg.get("token_map_glob", "/kaggle/input/**/hot_tokens_64k.pt"), recursive=True))
            view = sl.prepare_draft_view(model, Path(hits[0]).parent, Path(cfg.get("sglang_prefix", sl.PREFIX)))
            out.update(draft_view=str(view), token_map=maps[0] if maps else None)
            out["draft_files"] += sorted(str(p) for p in Path(hits[0]).parent.glob("*.safetensors"))
        except Exception as e:
            out["mtp_why"] = f"{type(e).__name__}: {e}"[:600]
    if cfg.get("dflash_glob"):
        hits = sorted(glob.glob(cfg["dflash_glob"], recursive=True))
        if hits:
            out["dflash_dir"] = str(Path(hits[0]).parent)
            out["draft_files"] += sorted(str(p) for p in Path(hits[0]).parent.glob("*.safetensors"))
        else:
            out["dflash_why"] = f"no DFlash drafter at {cfg['dflash_glob']}"
    out["ok"] = bool(out.get("draft_view") or out.get("dflash_dir"))
    return out


def spec_facts(lines):
    """X59: speculative acceptance as SGLang logs it (mean accepted tokens per verify step)."""
    v = [float(x) for l in lines for x in re.findall(r"accept len(?:gth)?[:=]\s*([\d.]+)", l)]
    return {"accept_len_mean": round(sum(v) / len(v), 2), "accept_len_n": len(v)} if v else {}


def serve_check(port, think_tokens=512, timeout=900):
    """X56 (2026-10-24): a server counts only once it really generates -- X41b's g1 answered /v1/models, then died on
    its first request. A short reply without thinking, then `think_tokens` of decode with thinking on."""
    url = f"http://127.0.0.1:{port}/v1/chat/completions"

    def post(payload):
        req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
        return json.loads(urllib.request.urlopen(req, timeout=timeout).read())
    out = {}
    try:
        r = post({"model": "qwen", "messages": [{"role": "user", "content": "What is 2+2? One word."}], "max_tokens": 64,
                  "chat_template_kwargs": {"enable_thinking": False}})
        out["smoke"] = (r["choices"][0]["message"].get("content") or "")[:200]
        t = time.time()
        r = post({"model": "qwen", "messages": [{"role": "user", "content": "List the primes below 60, checking each."}],
                  "max_tokens": think_tokens, "temperature": 1.0, "top_p": 0.95, "top_k": 20,
                  "chat_template_kwargs": {"enable_thinking": True}})
        dt = time.time() - t
        n = (r.get("usage") or {}).get("completion_tokens", 0)
        out.update(gen_tokens=n, seconds=round(dt, 1), tok_s=round(n / max(dt, 1e-6), 1))
        msg = r["choices"][0]["message"]
        think = ((msg.get("reasoning_content") or "") + (msg.get("content") or "")).strip()
        out["think_sample"] = think[:160]
        # a prompt longer than one prefill chunk (4,096): chunked prefill reads the cache it just wrote (#43741)
        grid = "\n".join(f"row {i}: " + " ".join(str((i * j + j) % 10) for j in range(30)) for i in range(200))
        r = post({"model": "qwen", "messages": [{"role": "user", "content": grid + "\nHow many rows are there? Reply "
                                                                                "with only the number."}],
                  "max_tokens": 64, "chat_template_kwargs": {"enable_thinking": False}})
        u = r.get("usage") or {}
        out.update(long_prompt_tokens=u.get("prompt_tokens", 0), long_reply=(r["choices"][0]["message"].get("content")
                                                                              or "")[:80])
        out["long_recall_ok"] = "200" in out["long_reply"]     # X57: positions intact over 6k tokens
        # X60: X59's DFlash rung "passed" with every reply "!!!!..." -- the text must make sense, not just exist
        out["coherent"] = (("4" in out["smoke"] or "four" in out["smoke"].lower()) and len(set(think)) > 8
                           and out["long_recall_ok"])
        out["ok"] = (n >= think_tokens // 2 and bool(out["smoke"].strip()) and u.get("completion_tokens", 0) >= 1
                     and u.get("prompt_tokens", 0) > 4096 and out["coherent"])
    except Exception as e:
        out.update(ok=False, error=f"{type(e).__name__}: {e}"[:400])
    return out


INC_NAMES = ("INCConfig", "AutoRoundConfig", "GPTQMarlinConfig", "GPTQConfig")


def patch_ple_inc(site):
    """X28 v1 (2026-10-17) died at model init: `Qwen4Exp PLE embedding does not support quantization config INCConfig`.
    vLLM 0.30 accepts only None / FP8 / ModelOpt for the PLE table, but Intel's AutoRound checkpoint keeps that table in
    BF16 -- i.e. unquantized. Insert, at the top of Qwen4ExpPLEEmbeddingMethod.from_quant_config, "an INC/AutoRound/GPTQ
    config counts as none for the PLE table". Text patch with checks; returns what it did."""
    hits = [p for p in glob.glob(f"{site}/vllm/**/qwen4_exp/**/*.py", recursive=True)
            if "def from_quant_config(" in open(p, errors="replace").read()
            and "does not support quantization config" in open(p, errors="replace").read()]
    if len(hits) != 1:
        return {"ok": False, "why": f"{len(hits)} candidate files", "files": hits}
    p = hits[0]
    lines = open(p, encoding="utf-8").read().split("\n")
    i = next(k for k, l in enumerate(lines) if "def from_quant_config(" in l)
    sig, j, depth = "", i, 0
    while True:                                          # the signature may span several lines
        sig += lines[j] + "\n"
        depth += lines[j].count("(") - lines[j].count(")")
        if depth <= 0 and lines[j].rstrip().endswith(":"):
            break
        j += 1
    params = re.findall(r"[(,]\s*\*{0,2}\s*(\w+)\s*(?=[:=,)])", sig.replace("\n", " "))
    qname = next((n for n in params if "quant" in n.lower()), None)
    if qname is None:
        return {"ok": False, "why": "no quant parameter", "signature": sig}
    k = j + 1
    while not lines[k].strip():
        k += 1
    ind = lines[k][:len(lines[k]) - len(lines[k].lstrip())]
    if "X28 v2 patch" in "\n".join(lines):
        return {"ok": True, "already": True, "file": p}
    lines[j + 1:j + 1] = [f"{ind}# X28 v2 patch: Intel AutoRound keeps the PLE table in BF16 -> serve it unquantized",
                          f"{ind}if {qname} is not None and type({qname}).__name__ in {INC_NAMES!r}:",
                          f"{ind}    {qname} = None"]
    src = "\n".join(lines)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", os.path.basename(p)[:-3] + ".*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p[len(str(site)) + 1:], "param": qname,
            "patched": "\n".join(lines[i:j + 5])[:1200]}


def preload(files, threads=16):
    """Read files into the OS page cache with many parallel streams (X28 v1: one stream from the model mount ran at
    41 MB/s; X26 read 30.9 GB with 8 threads in 53 s). Returns GB and MB/s."""
    def read(p):
        n = 0
        with open(p, "rb") as f:
            while True:
                b = f.read(64 << 20)
                if not b:
                    return n
                n += len(b)
    t = time.time()
    with ThreadPoolExecutor(threads) as ex:
        n = sum(ex.map(read, files))
    dt = time.time() - t
    return {"files": len(files), "gb": round(n / 1e9, 1), "s": round(dt), "mb_s": round(n / 1e6 / max(dt, 1e-6), 1)}


def bench_load(base, n, prompts, max_tokens=2048):
    """bench_serving.load against `base` (that one is fixed to port 1234): n requests at once, real ARC prompts,
    thinking on, 2,048 tokens each -> aggregate generated tokens per second."""
    from client import ChatClient
    from grids import SYSTEM
    client = ChatClient(base, "qwen", max_tokens=max_tokens, timeout=1800)
    msgs = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompts[i % len(prompts)]}]
            for i in range(n)]
    t = time.time()
    with ThreadPoolExecutor(max_workers=n) as ex:
        rs = list(ex.map(lambda m: client.chat(m[1], seed=m[0]), enumerate(msgs)))
    dt = time.time() - t
    tok = sum(r.get("completion_tokens", 0) for r in rs)
    return {"n": n, "seconds": round(dt, 1), "tokens": tok, "tok_per_s": round(tok / dt, 1),
            "per_stream": round(tok / dt / n, 1), "errors": [r["error"][:120] for r in rs if "error" in r][:3]}


def main(cfg):
    out = {"config": {k: v for k, v in cfg.items() if k != "site"}}
    work = Path(cfg.get("work_dir", "/kaggle/working"))
    data = cfg.get("data_root", "/kaggle/input")
    path = cfg.get("out", str(work / "flash_smoke.json"))
    save = lambda: json.dump(out, open(path, "w"), indent=1)
    site = Path(cfg["site"])
    hits = sorted(glob.glob(cfg["model_glob"], recursive=True))
    assert hits, f"no model at {cfg['model_glob']}"
    model = Path(hits[0]).parent
    out["model_dir"] = str(model)
    ple_files = sorted(str(p) for p in model.glob("*ple*.safetensors"))
    out["machine"] = {"mem": meminfo(), "cpus": os.cpu_count(), "gpu_mem": gpu_mem(),
                      "disk_ple": disk_speed(ple_files) if not cfg.get("skip_disk_speed") else {"skipped": True},
                      "ple_files": len(ple_files)}
    print("MACHINE", json.dumps(out["machine"]), flush=True)
    save()
    try:
        out["checkpoint"] = {"config": config_summary(model), "groups": tensor_groups(model)}
    except Exception as e:
        out["checkpoint"] = {"error": f"{type(e).__name__}: {e}"[:400]}
    print("CHECKPOINT", json.dumps(out["checkpoint"])[:3000], flush=True)
    save()
    if cfg.get("backend") == "sglang":               # X29: dfranzen's SGLang bundle instead of vLLM 0.30
        import sglang_launch as sl
        t = time.time()
        stage = None
        try:
            bundle = sl.find_bundle(cfg["sglang_bundle_glob"])
            out["machine"]["mounts"] = {"bundle": sl.mount_of(bundle), "model": sl.mount_of(model)}
            if cfg.get("stage_bundle"):                  # X56 v4: local copy first -- v3's install timed out on the input
                sdir = cfg.get("stage_dir", "/tmp/sgl-bundle")
                stage = sl.stage_bundle(bundle, sdir, cfg.get("stage_threads", 16))
                print("STAGED", json.dumps(stage), flush=True)
                bundle = Path(sdir)
            inst = sl.install(bundle, cfg.get("sglang_prefix", sl.PREFIX))
            if stage is not None:
                shutil.rmtree(cfg.get("stage_dir", "/tmp/sgl-bundle"), ignore_errors=True)   # installed: free the disk
            cuda_home, cxx = sl.toolchain(inst["venv"])
            env = sl.env_for(inst["venv"], cuda_home, cxx, cfg.get("sglang_prefix", sl.PREFIX))
            out["sglang"] = dict(sl.facts(inst["python"], env, inst["venv"]), install=inst, cuda_home=cuda_home,
                                 cxx=cxx, install_s=round(time.time() - t), stage=stage)
            env.update(cfg.get("sgl_env") or {})         # X57: e.g. the unused FlashInfer workspace cut to 32 MiB
            out["sglang"]["sgl_env"] = cfg.get("sgl_env") or {}
            SGL.update(bin=inst["sglang"], env=env)
            if cfg.get("mem_report_patch"):              # X57: per-group GPU weight bytes in the server log
                out["sglang"]["mem_report_patch"] = sl.patch_mem_report(inst["venv"])
            if cfg.get("dflash_capture_patch"):          # X59: Flash-Next hands DFlash2 its 5 target-layer features
                out["sglang"]["dflash_patch"] = sl.patch_dflash_capture(inst["venv"])
            if cfg.get("marlin_sum_patch"):              # X59: Marlin MoE's last reduction without sgl_kernel (SM89)
                out["sglang"]["marlin_patch"] = sl.patch_marlin_sum(inst["venv"])
            if cfg.get("marlin_ep_patch"):               # X60: Marlin gets the EP mapping (skip / zero -1 experts)
                out["sglang"]["marlin_ep_patch"] = sl.patch_marlin_ep(inst["venv"])
            if cfg.get("early_head_patch"):              # X75: drafter shares embed + lm_head before the KV pool is sized
                out["sglang"]["early_head_patch"] = sl.patch_early_draft_head(inst["venv"])
            if cfg.get("draft_map_patch"):               # c23fp: a 512-expert drafter beside a 384-expert target (EP)
                out["sglang"]["draft_map_patch"] = sl.patch_draft_expert_map(inst["venv"])
                print("DRAFT MAP PATCH", json.dumps(out["sglang"]["draft_map_patch"]), flush=True)
            if cfg.get("triton_bench_cache_mb"):         # X41 v2: the autotuner's 256 MiB flush buffer OOMed v1
                out["sglang"]["triton_patch"] = sl.patch_triton_bench_cache(inst["venv"], cfg["triton_bench_cache_mb"])
            if cfg.get("sparse_source"):                 # X56: the decode path's code, kept whatever happens next
                out["sparse_src"] = sl.sparse_source(inst["venv"])
            if cfg.get("sparse_fp8_patch"):              # X56: an FP8 cache packs into q-dtype decode scratch
                out["sglang"]["sparse_patch"] = sl.patch_sparse_fp8(inst["venv"])
            if cfg.get("fa2_wheel_glob"):                # X56: classic FA2, so SM89's QSA decode avoids FA4 cute
                out["sglang"]["fa2_install"] = sl.install_fa2(inst["venv"], cfg["fa2_wheel_glob"],
                                                              cfg.get("sglang_prefix", sl.PREFIX))
                out["sglang"]["fa2_check"] = sl.fa2_check(inst["python"], env)
        except Exception as e:
            out["sglang"] = {"error": f"{type(e).__name__}: {e}"[-2500:], "install_s": round(time.time() - t),
                             "stage": stage}
        print("SGLANG", json.dumps(out["sglang"])[:3500], flush=True)
        save()
        fatal = "error" in out["sglang"] or out["sglang"].get("flashinfer_jit") not in (None, "ok")
        if not fatal and cfg.get("require_fa2") and not (out["sglang"].get("fa2_check") or {}).get("ok"):
            # X56: without FA2 the decode falls to FA4 cute, which fails on SM89 (#36968) -- no rung can serve
            out["summary"] = {"served": False, "reason": "classic FA2 not usable for the QSA decode on this GPU",
                              "fa2": out["sglang"].get("fa2_install"), "fa2_check": out["sglang"].get("fa2_check")}
            save()
            print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
            return out
    else:
        out["vllm"] = vllm_facts(site)
        print("VLLM", json.dumps(out["vllm"])[:3000], flush=True)
        save()
        ok, fatal, tail = jit_preflight(site)
        out["jit_preflight"] = {"ok": ok, "fatal": fatal, "tail": tail[-600:]}
        print("JIT", ok, fatal, flush=True)
        save()
    if fatal:                                        # FlashInfer cannot build: no server can start (X8 v1 / v3)
        out["summary"] = {"served": False, "reason": ("SGLang install / toolchain failed" if SGL == {} and
                                                       cfg.get("backend") == "sglang" else
                                                       "FlashInfer toolchain error in the JIT preflight")}
        save()
        print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
        return out
    if cfg.get("patch_ple_inc"):
        out["ple_patch"] = patch_ple_inc(site)
        print("PLE PATCH", json.dumps(out["ple_patch"])[:1500], flush=True)
        save()
        if not out["ple_patch"]["ok"]:
            out["summary"] = {"served": False, "reason": "PLE quantization patch did not apply"}
            save()
            print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
            return out
    if SGL and any(r.get("spec") and not r["spec"].get("self_draft") for r in cfg["rungs"]):   # X59: drafter view + map
        out["spec"] = spec_setup(cfg, model)
        print("SPEC", json.dumps(out["spec"])[:1500], flush=True)
        save()
    if cfg.get("preload_threads"):                   # GPU-side weights only: vLLM copies the PLE table itself
        gpu_files = sorted(str(p) for p in model.glob("*.safetensors") if "ple" not in p.name)
        gpu_files += (out.get("spec") or {}).get("draft_files", [])
        out["preload"] = dict(preload(gpu_files, cfg["preload_threads"]), mem_after=meminfo())
        print("PRELOAD", json.dumps(out["preload"]), flush=True)
        save()
    env_extra = {} if SGL else {"VLLM_PLE_CPU_OFFLOAD": "1"}   # vLLM: the documented switch only
    out["env_extra"] = env_extra
    ev = glob.glob(f"{data}/**/arc-agi_evaluation_challenges.json", recursive=True)
    es = glob.glob(f"{data}/**/arc-agi_evaluation_solutions.json", recursive=True)
    # c23f: the scored rerun may not carry the eval files -- the bench prompts then come from the submission's tasks
    ch = json.load(open(ev[0])) if ev else json.load(open(cfg["submission"]["challenges"]))
    sol = json.load(open(es[0])) if es else {}
    from repl_agent import repl_prompt
    prompts = [repl_prompt(ch[t]) for t in sorted(ch)[:48]]
    base = f"http://127.0.0.1:{PORT}/v1"

    def prepared(rung):
        """The rung with its runtime paths filled in, or (None, why it cannot run)."""
        if rung.get("needs_sparse_patch") and not (out.get("sglang", {}).get("sparse_patch") or {}).get("ok"):
            return None, "sparse FP8 patch not applied"
        if rung.get("needs_dflash_patch") and not (out.get("sglang", {}).get("dflash_patch") or {}).get("ok"):
            return None, "DFlash capture patch not applied"
        if rung.get("needs_marlin_patch") and not (out.get("sglang", {}).get("marlin_patch") or {}).get("ok"):
            return None, "Marlin sum patch not applied"
        if rung.get("needs_marlin_ep_patch") and not (out.get("sglang", {}).get("marlin_ep_patch") or {}).get("ok"):
            return None, "Marlin EP patch not applied"
        if rung.get("needs_early_head_patch") and not (out.get("sglang", {}).get("early_head_patch") or {}).get("ok"):
            return None, "early draft-head patch not applied"
        if rung.get("needs_draft_map_patch") and not (out.get("sglang", {}).get("draft_map_patch") or {}).get("ok"):
            return None, "draft expert-map patch not applied"
        if not rung.get("spec"):
            return rung, None
        if rung["spec"].get("self_draft"):               # X63: the model's own MTP head, read from its own checkpoint
            return dict(rung, spec=dict(rung["spec"], draft_view=str(model))), None
        sp = out.get("spec") or {}                       # X59: the runtime drafter paths, or skip without them
        if rung["spec"].get("algo") == "DFLASH":
            if not sp.get("dflash_dir"):
                return None, f"no DFlash drafter: {sp.get('dflash_why')}"
            return dict(rung, spec=dict(rung["spec"], draft_path=sp["dflash_dir"])), None
        if not sp.get("draft_view"):
            return None, f"no drafter: {sp.get('mtp_why')}"
        return dict(rung, spec=dict(rung["spec"], draft_view=sp["draft_view"],
                                    token_map=sp["token_map"] if rung["spec"].get("token_map") else None)), None

    def bench_suite(proc, rung):
        res = []
        ms = rung["max_seqs"]
        for n in sorted({1, max(1, ms // 2), ms}):
            if time.time() > cfg["deadline_epoch"] - 1500:
                break
            res.append(dict(bench_load(base, n, prompts), mem_avail_mib=meminfo()["MemAvailable"], **group_rss(proc.pid)))
            print("BENCH", rung["name"], res[-1], flush=True)
            save()
        return res

    def try_rung(rung, tag=""):
        log = str(work / f"server_{rung['name']}{tag}.log")
        proc, up, secs, trace = start(model, site, rung, log, env_extra, cfg.get("start_timeout_s", 2400))
        check, came_up = None, up
        if up and cfg.get("serve_check"):                # X56: up is not enough -- it must generate
            check = serve_check(PORT, cfg.get("serve_check_tokens", 512))
            up = bool(check.get("ok")) and getattr(proc, "poll", lambda: None)() is None
            print("SERVE CHECK", rung["name"], json.dumps(check), flush=True)
        lines = Path(log).read_text(errors="replace").splitlines()
        rec = {"name": rung["name"] + tag, "rung": rung, "started": up, "start_s": secs, "gpu_mem": gpu_mem(),
               **({"came_up": came_up, "serve_check": check} if check is not None else {}),
               "mem_trace": trace[::max(1, len(trace) // 40)] + trace[-1:],
               "log_facts": [l[-220:] for l in lines if any(k in l for k in (
                   "KV cache size", "Maximum concurrency", "took", "PLE", "ple", "offload", "Offload",
                   "memory", "Loading weights", "KV Cache", "max_total_num_tokens", "avail mem", "mamba_cache",
                   "Load weight end", "DFLASH_CAPTURE", "X75 EARLY HEAD"))][-30:]}
        rec.update(kv_facts(lines))
        rec.update(spec_facts(lines))
        rep = [l.split("X57 MEMREPORT ", 1)[1] for l in lines if "X57 MEMREPORT {" in l]
        if rep:
            try:
                rec["mem_report"] = json.loads(rep[0])
            except Exception:
                rec["mem_report"] = rep[0][:4000]
        rec["mem_lines"] = [l[-200:] for l in lines if "TP0" in l and any(k in l for k in (
            "avail mem=", "Memory pool end", "KV Cache is allocated", "Mamba Cache is allocated", "Reserving"))][:12]
        if not up:
            rec["error_tail"] = [l[-300:] for l in lines[-40 if check is not None else -30:]]
        out["rungs"].append(rec)
        print("RUNG", json.dumps(rec)[:4000], flush=True)
        save()
        return proc, up, rec

    if cfg.get("cpu_wait"):                          # c23f: the CPU solver must be gone before Flash-Next pins its RAM
        out["cpu_wait"] = wait_cpu(cfg["cpu_wait"])
        print("CPU WAIT", json.dumps(out["cpu_wait"]), flush=True)
        save()
    # X59: rungs named in cfg compare_rungs are each benchmarked (tok/s at 1 / half / max) and stopped while another
    # compared rung is still to come; the fastest at max is then served for the accuracy run (restarted if needed).
    # Fallback rungs after them run only if no compared rung served. Without compare_rungs: the plain ladder.
    compare = [r["name"] for r in cfg["rungs"] if r["name"] in set(cfg.get("compare_rungs", []))]
    server, rung_used, benched = None, None, {}
    out["rungs"] = []
    if cfg.get("mix_bench"):                         # X68: every rung in turn on X54's decode-only load
        return mix_all(cfg, out, save, prepared, try_rung, ch)
    for i, rung0 in enumerate(cfg["rungs"]):
        if time.time() > cfg["deadline_epoch"] - 1800:
            out["rungs"].append({"name": rung0["name"], "skipped": "deadline"})
            break
        if benched and rung0["name"] not in compare:
            break                                        # a compared rung served: no fallback needed
        rung, why = prepared(rung0)
        if rung is None:
            out["rungs"].append({"name": rung0["name"], "skipped": why})
            save()
            continue
        proc, up, rec = try_rung(rung)
        if not up:
            stop(proc)
            continue
        if rung["name"] in compare:
            rec["bench"] = bench_suite(proc, rung)
            benched[rung["name"]] = (rung, rec["bench"])
            save()
            later = [r["name"] for r in cfg["rungs"][i + 1:] if r["name"] in compare]
            if later and time.time() < cfg["deadline_epoch"] - cfg.get("compare_reserve_s", 7200):
                stop(proc)
                continue
        server, rung_used = proc, rung
        break
    if benched:
        best = max(benched, key=lambda n: benched[n][1][-1]["tok_per_s"] if benched[n][1] else 0.0)
        out["compare"] = {"best": best, "tok_s": {n: {b["n"]: b["tok_per_s"] for b in v[1]} for n, v in benched.items()}}
        print("COMPARE", json.dumps(out["compare"]), flush=True)
        save()
        if rung_used is None or rung_used["name"] != best:
            if server is not None:
                stop(server)
            server, rung_used = None, None
            proc, up, rec = try_rung(benched[best][0], tag="_restart")
            if up:
                server, rung_used = proc, benched[best][0]
            else:
                stop(proc)
    if server is not None:
        (work / "vllm-server.pid").write_text(str(server.pid))
    if server is None:
        out["summary"] = {"served": False}
        save()
        print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
        return out
    served_log = str(work / f"server_{out['rungs'][-1]['name']}.log")
    current = {"proc": server}

    def restart(rung_name=None):
        """c23f watchdog (X62 hung 9.3 h): stop the hung / dead server, start the named rung (else the one that was
        serving), serve check; the other one if that fails. Returns (proc, rung name, log path) or None."""
        stop(current["proc"])
        named = [r for r in cfg["rungs"] if r["name"] == rung_name]
        same = [r for r in cfg["rungs"] if r["name"] == rung_used["name"]]
        for r0 in named + [r for r in same if r not in named]:
            r, why = prepared(r0)
            if r is None:
                continue
            tag = f"_wd{len(out['rungs'])}"
            proc, up, rec = try_rung(r, tag=tag)
            if up:
                current["proc"] = proc
                (work / "vllm-server.pid").write_text(str(proc.pid))
                return proc, r["name"], str(work / f"server_{r['name']}{tag}.log")
            stop(proc)
        return None

    if cfg.get("submission"):                        # c23f: the agent over every task; submission.json kept valid
        import flash_sub
        return flash_sub.run(cfg, out, save, base, rung_used, server, restart=restart, log_path=served_log)
    if cfg.get("calibrate"):                         # X72: router counts over the whole agent run, for pruning
        import flash_calib
        return flash_calib.run(cfg, out, save, base, rung_used, server, restart=restart, log_path=served_log)

    from client import ChatClient
    from run_direct import answer
    import metrics
    req = urllib.request.Request(base + "/chat/completions", data=json.dumps({
        "model": "qwen", "messages": [{"role": "user", "content": "What is 2+2? One word."}], "max_tokens": 64,
        "chat_template_kwargs": {"enable_thinking": False}}).encode(), headers={"Content-Type": "application/json"})
    try:
        out["smoke_reply"] = json.loads(urllib.request.urlopen(req, timeout=600).read())["choices"][0]["message"]
    except Exception as e:
        out["smoke_reply"] = {"error": f"{type(e).__name__}: {e}"[:300]}
    print("SMOKE REPLY", out["smoke_reply"], flush=True)
    save()
    ms = rung_used["max_seqs"]
    out["bench"] = benched[rung_used["name"]][1] if rung_used["name"] in benched else bench_suite(server, rung_used)
    if cfg.get("agent_probe"):                       # X62: the python-tool agent (X44's harness) instead of direct
        return agent_eval(cfg, out, save, base, rung_used, server)
    sampler = metrics.Sampler(base.rsplit("/v1", 1)[0] + "/metrics", every=15)
    sampler.start()
    cap = rung_used.get("direct_tokens", cfg.get("direct_tokens", 32768))     # X56: a smaller KV pool -> a smaller cap
    client = ChatClient(base, "qwen", max_tokens=cap, temperature=1.0, top_p=0.95,
                        top_k=20, thinking=True, template_kwargs={"reasoning_effort": "xhigh"}, timeout=3 * 3600)
    deadline = cfg["deadline_epoch"] - 600
    # X41 v2: an accuracy run -- cfg answer_tasks (in order, until the deadline) at answer_concurrency; default = X28's
    # 4 quick tasks at once. An answer is not started with less than min_start_s left.
    tasks = [t for t in cfg.get("answer_tasks", TASKS) if t in ch]
    if "answer_tasks" not in cfg:
        tasks = tasks[:min(4, ms)]
    conc = min(rung_used.get("answer_concurrency", cfg.get("answer_concurrency", len(tasks) or 1)), ms)

    def one_task(t):
        if time.time() > deadline - cfg.get("min_start_s", 0):
            return {"task_id": t, "status": "skipped: deadline", "gen_tokens": 0, "wall": 0.0, "correct": [False]}
        ts = time.time()
        rec = answer(client, ch[t], 7, deadline, max_tokens=cap, commit=cfg.get("answer_commit", "none"))
        g = rec.get("grids") or []
        rec.update(task_id=t, wall=round(time.time() - ts, 1),
                   correct=[j < len(g) and g[j] == s for j, s in enumerate(sol[t])])
        rec.pop("grids", None)
        print("DIRECT", t, rec.get("status"), rec.get("gen_tokens"), rec["wall"], rec["correct"], flush=True)
        return rec
    with ThreadPoolExecutor(max_workers=conc or 1) as ex:
        out["direct"] = list(ex.map(one_task, tasks))
    sampler.stop()
    try:                                             # vLLM metric names; SGLang's differ -> may summarise nothing
        out["direct_metrics"] = metrics.write(sampler, str(work / "server_metrics.json"))
    except Exception as e:
        out["direct_metrics"] = {"error": f"{type(e).__name__}: {e}"[:300]}
    toks = sum(r.get("gen_tokens", 0) for r in out["direct"])
    span = sum(r["wall"] for r in out["direct"]) / max(conc, 1)        # answers ran conc at a time
    out["direct_metrics"]["client_gen_tok_s"] = round(toks / max(span, 1), 1)
    started = [r for r in out["direct"] if not str(r.get("status", "")).startswith("skipped")]
    per = {r["task_id"]: sum(r["correct"]) / max(len(r["correct"]), 1) for r in started}
    out["accuracy"] = {"attempted": len(started), "solved": round(sum(per.values()), 2), "per_task": per,
                       "mean_gen_tokens": round(toks / max(len(started), 1)),
                       "cut": sum(1 for r in started if r.get("finish") == "length"), "cap": cap}
    ref = (cfg.get("ref_solved") or {}).get(str(cap))    # X56: the 27B (X13) on the SAME attempted tasks, same cap
    if ref is not None:                              # X61: per-task credit (X13 scores half for 1 of 2 outputs)
        out["accuracy"]["ref_27b_same_tasks"] = round(sum(float(ref.get(t, 0)) for t in per), 2)
    out["summary"] = {
        "served": True, "rung": rung_used["name"], "start_s": out["rungs"][-1]["start_s"],
        "mem_total_gib": (round(out["machine"]["mem"]["MemTotal"] / 1024, 1) if out["machine"]["mem"].get("MemTotal")
                          else None),
        "rss_after_load": {k: out["rungs"][-1]["mem_trace"][-1][k] for k in ("rss_anon_gib", "rss_file_gib")},
        "tok_s": {b["n"]: b["tok_per_s"] for b in out["bench"]},
        "per_stream_tok_s": {b["n"]: b["per_stream"] for b in out["bench"]},
        **{k: out["rungs"][-1].get(k) for k in ("kv_tokens", "avail_mem_gb", "mamba_state_mb", "mamba_states",
                                                "accept_len_mean")},
        "direct": [(r["task_id"], r.get("status"), r.get("gen_tokens"), all(r["correct"])) for r in out["direct"]],
        "accuracy": {k: out["accuracy"][k] for k in ("attempted", "solved", "mean_gen_tokens", "cut", "cap",
                                                     "ref_27b_same_tasks") if k in out["accuracy"]},
        "serve_check": out["rungs"][-1].get("serve_check"),
        "rungs_tried": [(r["name"], r.get("skipped") or ("served" if r.get("started") else "failed")) for r in out["rungs"]],
        "direct_gen_tok_s": out["direct_metrics"].get("gen_tok_per_s"),
        "direct_client_tok_s": out["direct_metrics"].get("client_gen_tok_s"), "backend": cfg.get("backend", "vllm")}
    save()
    print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
    return out


def mix_all(cfg, out, save, prepared, try_rung, ch):
    """X68 (2026-10-27): would SGLang serve the submission's direct answers faster than vLLM 0.19? Each rung in turn:
    start (+ the serve check), the quality check, X54's decode-only loads (version_bench.mix_load: real X39 traces at
    depths 2k-72k, tokens counted from the streams), stop. A rung with only_if_failed runs only if that rung did not
    serve. The vLLM 0.19 side runs in its own cell with the same loads."""
    import version_bench as vb
    mb = cfg["mix_bench"]
    items = vb.load_traces(mb["traces"])
    out["mix"] = {}
    for rung0 in cfg["rungs"]:
        if rung0.get("only_if_failed") and rung0["only_if_failed"] in out["mix"]:
            continue
        if rung0.get("only_if_served") and rung0["only_if_served"] not in out["mix"]:    # X75: same flag would fail
            out["rungs"].append({"name": rung0["name"], "skipped": f"{rung0['only_if_served']} did not serve"})
            save()
            continue
        if time.time() > cfg["deadline_epoch"] - 1500:
            out["rungs"].append({"name": rung0["name"], "skipped": "deadline"})
            save()
            continue
        rung, why = prepared(rung0)
        if rung is None:
            out["rungs"].append({"name": rung0["name"], "skipped": why})
            save()
            continue
        proc, up, rec = try_rung(rung)
        if up:
            rec["quality"] = vb.quality_check()
            print("QUALITY", rung["name"], json.dumps(rec["quality"]), flush=True)
            cache, loads = {}, []
            # X75: KV tokens a load needs = its depths + per-stream prompt + growth over the window [ESTIMATE];
            # skip_unfit drops loads the rung's pool cannot hold, fit_load adds the largest n <= max_seqs that fits
            need = lambda n: (sum(vb.depths(n, mb.get("depth_lo", 2000), mb.get("depth_span", 70000)))
                              + n * (mb.get("prompt_est", 5000) + int(mb.get("window_s", 240) * mb.get("growth_tok_s", 30))))
            kv, plan = rec.get("kv_tokens"), list(rung.get("loads", mb.get("loads", [16, 20])))
            if mb.get("fit_load") and kv:
                fit = [n for n in range(1, rung["max_seqs"] + 1) if need(n) <= kv]
                if fit and fit[-1] not in plan:
                    plan = sorted(plan + [fit[-1]])
            for n in plan:
                if time.time() > cfg["deadline_epoch"] - 900:
                    loads.append({"n": n, "skipped": "deadline"})
                    continue
                if mb.get("skip_unfit") and kv and need(n) > kv:
                    loads.append({"n": n, "skipped": f"does not fit: needs ~{need(n):,} of {kv:,} KV tokens"})
                    print("MIX", rung["name"], json.dumps(loads[-1]), flush=True)
                    continue
                try:                                     # X75: depth_lo / depth_span = the agents' context range
                    r = vb.mix_load(n, items, ch, mb.get("window_s", 240), cache, poll=mb.get("poll_s", 5.0),
                                    prefill_timeout=mb.get("prefill_timeout_s", 1800), client_count=True,
                                    depth_lo=mb.get("depth_lo", 2000), depth_span=mb.get("depth_span", 70000))
                except Exception as e:                   # noqa: BLE001
                    r = {"n": n, "error": repr(e)[:400]}
                loads.append(r)
                print("MIX", rung["name"], json.dumps(r), flush=True)
                out["mix"][rung["name"]] = loads
                save()
            rec["mix"] = loads
            out["mix"][rung["name"]] = loads
        stop(proc)
        save()
    out["summary"] = {"served": bool(out["mix"]), "mode": "mix_bench", "backend": cfg.get("backend"),
                      "tok_s": {name: {str(x["n"]): x.get("decode_tok_per_s") for x in loads}
                                for name, loads in out["mix"].items()},
                      "rungs_tried": [(r["name"], r.get("skipped") or ("served" if r.get("started") else "failed"))
                                      for r in out["rungs"]],
                      # X75: per rung, the KV pool it got and what each load did with it
                      "per_rung": {r["name"]: {
                          "kv_tokens": r.get("kv_tokens"), "mamba_states": r.get("mamba_states"),
                          "avail_mem_gb": r.get("avail_mem_gb"), "quality_ok": (r.get("quality") or {}).get("ok"),
                          "early_head": next((l.split("X75 EARLY HEAD", 1)[1][:90] for l in r.get("log_facts", [])
                                              if "X75 EARLY HEAD" in l), None),
                          "loads": {str(x["n"]): {k: x.get(k) for k in (
                              "decode_tok_per_s", "per_stream", "preemptions", "kv_peak_pct", "running_min",
                              "accept_length", "context_median", "error", "skipped") if x.get(k) is not None}
                                    for x in r.get("mix", [])},
                          "error_tail": (r.get("error_tail") or [])[-3:]} for r in out["rungs"] if "rung" in r}}
    save()
    print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
    return out


def wait_cpu(w, poll=5.0):
    """c23f: with Flash-Next loaded the server holds ~156 GiB of the container's ~173 GiB (X60 rss), so the CPU solver
    runs ALONE during install + preload. Wait for its process group (pid in w["pid_file"]) to finish or until
    w["until_epoch"], then kill whatever is left. Never raises."""
    import signal
    t0 = time.time()
    try:
        pid = int(Path(w["pid_file"]).read_text().strip())
    except Exception as e:                           # noqa: BLE001  no solver started -> nothing to wait for
        return {"waited_s": 0, "pid": None, "note": f"{type(e).__name__}: {e}"[:200]}

    def alive():
        try:
            os.kill(pid, 0)
        except OSError:
            return False
        try:                                         # a zombie (exited, not reaped by the notebook) counts as gone
            return Path(f"/proc/{pid}/stat").read_text().split(")")[-1].split()[0] != "Z"
        except Exception:                            # noqa: BLE001  no /proc (not Linux): os.kill said it exists
            return True
    while alive() and time.time() < w["until_epoch"]:
        time.sleep(poll)
    killed = alive()
    if killed:
        sig = getattr(signal, "SIGKILL", signal.SIGTERM)
        try:
            os.killpg(pid, sig)
        except Exception:                            # noqa: BLE001
            try:
                os.kill(pid, sig)
            except Exception:                        # noqa: BLE001
                pass
    return {"waited_s": round(time.time() - t0), "pid": pid, "killed": killed}


def tool_check(base, timeout=900):
    """X62: one thinking-off request offering the agent's python tool; ok when a call comes back -- parsed by the server
    (--tool-call-parser) or recovered from the text by the client, as the agent itself would."""
    from client import ChatClient
    from repl_agent import TOOLS
    try:
        r = ChatClient(base, "qwen", max_tokens=512, thinking=False, timeout=timeout).chat(
            [{"role": "user", "content": "Use the python tool to print 17 * 23. Call the tool; do not answer in text."}],
            tools=TOOLS)
    except Exception as e:                           # noqa: BLE001
        return {"ok": False, "error": f"{type(e).__name__}: {e}"[:300]}
    calls = r.get("tool_calls") or []
    return {"ok": any(c.get("name") == "python" for c in calls), "calls": [c.get("name") for c in calls],
            "content": (r.get("content") or "")[:200], "error": r.get("error")}


def reuse_stats(tx_path):
    """X63: prefix reuse and token-estimate accuracy from the agent's per-turn records (probe _transcripts.json). X44
    (vLLM 0.19, from its server counters) re-processed 5,604 prompt tokens per request against 7,335 generated: the
    previous turn's own output came back as fresh prefill (76%)."""
    try:
        tx = json.load(open(tx_path))
    except Exception as e:                           # noqa: BLE001
        return {"error": f"{type(e).__name__}: {e}"[:200]}
    req = prompt = cached = gen = 0
    reprocessed = prev_gen = 0
    under, ratios = 0, []
    for turns in tx.values():
        last = None
        for e in turns:
            if "prompt" not in e:
                continue
            req += 1
            prompt += e["prompt"]
            cached += e.get("cached", 0)
            gen += e.get("tokens", 0)
            if last is not None:                     # what this turn had to prefill vs what the last one generated
                reprocessed += e["prompt"] - e.get("cached", 0)
                prev_gen += last.get("tokens", 0)
            if e.get("est"):
                ratios.append(e["prompt"] / e["est"])
                under += e["prompt"] > e["est"]
            last = e
    ratios.sort()
    return {"requests": req, "prompt_tokens": prompt, "cached_tokens": cached,
            "hit_rate": round(cached / max(prompt, 1), 3), "uncached_per_request": round((prompt - cached) / max(req, 1)),
            "generated_per_request": round(gen / max(req, 1)),
            "reprefill_per_prev_generated": round(reprocessed / max(prev_gen, 1), 3),
            "est_undercounts": under, "est_ratio_p50_p95_max": [round(ratios[int(q * (len(ratios) - 1))], 3)
                                                                for q in (0.5, 0.95, 1.0)] if ratios else None}


def agent_eval(cfg, out, save, base, rung, server):
    """X62 (user, 2026-10-26): X44's python-tool agent (thinking kept across turns, THINK CHECK) on this server, every
    attempt's transcript saved, scored per task against references (cfg agent_refs: {name: {task: credit}}, e.g. the 27B
    agent X44 and the 27B direct X13) on the tasks this run attempted."""
    import run_probe
    work = Path(cfg.get("work_dir", "/kaggle/working"))
    pcfg = dict(cfg["agent_probe"], base_url=base, model="qwen", data_dir=cfg.get("data_root", "/kaggle/input"),
                deadline_epoch=cfg["deadline_epoch"] - 600, out=str(work / "probe_results.json"),
                trace_dir=str(work / cfg.get("trace_subdir", "agent_traces")))
    pcfg["concurrency"] = rung.get("answer_concurrency", pcfg.get("concurrency", 4))
    out["agent_probe_cfg"] = {k: v for k, v in pcfg.items() if k != "task_ids"}
    out["tool_check"] = tool_check(base)
    print("TOOL CHECK", json.dumps(out["tool_check"]), flush=True)
    save()
    if not out["tool_check"].get("ok"):              # no usable tool call -> the agent could never run python
        out["summary"] = {"served": True, "rung": rung["name"], "mode": "agent",
                          "reason": "tool check failed: no python tool call parsed", "tool_check": out["tool_check"]}
        save()
        print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
        return out
    if cfg.get("agent_smoke"):                       # X63: the agent end to end on 1 task, 3 short turns, first
        scfg = dict(pcfg, task_ids=list(pcfg["task_ids"])[:1], max_turns=3, max_gen_tokens=6144, turn_max_tokens=2048,
                    max_tokens=2048, concurrency=1, out=str(work / "agent_smoke.json"),
                    trace_dir=str(work / "agent_smoke_traces"))
        ts = time.time()
        try:
            sres = run_probe.main(scfg) or {}
        except Exception as e:                       # noqa: BLE001
            sres = {"error": f"{type(e).__name__}: {e}"[-1500:]}
        traces = sorted(os.listdir(scfg["trace_dir"])) if os.path.isdir(scfg["trace_dir"]) else []
        out["agent_smoke"] = {"s": round(time.time() - ts), "error": sres.get("error"),
                              "think_check": sres.get("think_check"), "traces": traces,
                              "reuse": reuse_stats(scfg["out"].replace(".json", "_transcripts.json"))}
        print("AGENT SMOKE", json.dumps(out["agent_smoke"]), flush=True)
        save()
        if sres.get("error") or not traces or (out["agent_smoke"]["reuse"].get("requests") or 0) < 2:
            out["summary"] = {"served": True, "rung": rung["name"], "mode": "agent", "reason": "agent smoke failed",
                              "agent_smoke": out["agent_smoke"]}
            save()
            print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
            return out
    t = time.time()
    try:
        res = run_probe.main(pcfg) or {}
    except Exception as e:                           # the server-side record must survive a probe crash
        res = {"error": f"{type(e).__name__}: {e}"[-1500:]}
    per = {k.split("@")[0]: v for k, v in (res.get("per_task") or {}).items()}
    attempts = (res.get("report") or {}).get("attempts", [])
    started = sorted({a["task_id"].split("@")[0] for a in attempts if a.get("status") != "stopped" or a.get("turns")})
    acc = {"attempted": len(started), "solved": round(sum(per.get(t_, 0.0) for t_ in started), 2),
           "think_check": res.get("think_check"), "gen_tokens": res.get("gen_tokens"),
           "mean_gen_tokens": round((res.get("gen_tokens") or 0) / max(len(attempts), 1)),
           "statuses": {s: sum(1 for a in attempts if a.get("status") == s) for s in {a.get("status") for a in attempts}},
           "verified_wrong": [a["task_id"] for a in attempts if a.get("status") == "verified"
                              and per.get(a["task_id"].split("@")[0], 0) < 1],
           "wall_s": round(time.time() - t), "error": res.get("error"),
           "overflows": sum(a.get("overflows", 0) for a in attempts),           # X63: context-length rejections
           "trims": sum(a.get("evictions", 0) for a in attempts),               # X64: history trims (X63: 25)
           "reuse": reuse_stats(pcfg["out"].replace(".json", "_transcripts.json"))}
    acc["client_gen_tok_s"] = round((res.get("gen_tokens") or 0) / max(acc["wall_s"], 1), 1)
    for name, ref in (cfg.get("agent_refs") or {}).items():
        acc[f"ref_{name}_same_tasks"] = round(sum(float(ref.get(t_, 0)) for t_ in started), 2)
    label = cfg.get("model_label", "flash")
    acc["per_task"] = {t_: {label: per.get(t_, 0.0), **{n: float(r.get(t_, 0)) for n, r in
                                                         (cfg.get("agent_refs") or {}).items()}} for t_ in started}
    out["accuracy"] = acc
    out["summary"] = {"served": True, "rung": rung["name"], "mode": "agent",
                      "tok_s": {b["n"]: b["tok_per_s"] for b in out["bench"]},
                      **{k: out["rungs"][-1].get(k) for k in ("kv_tokens", "accept_len_mean")},
                      "accuracy": {k: v for k, v in acc.items() if k not in ("per_task",)},
                      "serve_check": out["rungs"][-1].get("serve_check"),
                      "rungs_tried": [(r["name"], r.get("skipped") or ("served" if r.get("started") else "failed"))
                                      for r in out["rungs"]], "backend": cfg.get("backend", "vllm")}
    save()
    print("FLASH SMOKE SUMMARY", json.dumps(out["summary"]), flush=True)
    return out


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))
    sys.stdout.flush()
    os._exit(0)

In [ ]:
%%writefile /kaggle/working/agent/sglang_launch.py
"""SGLang backend for the Flash-Next smoke (X29, 2026-10-17): the fallback if vLLM 0.30 cannot serve the checkpoint.

Follows the approach of dfranzen's ARC-AGI-3 Milestone 2 notebook (Pennyroyal = an SGLang 0.5.19 fork, offline wheels
in dfranzen/pennyroyal-v253) but for 4x L4 instead of one RTX PRO 6000: tensor parallel 4, compute capability 8.9,
the fork's sm120-only switches off, no MTP drafter. Steps: install the bundle into its own uv venv (offline), point the
JIT toolchain at the bundled CUDA 13, report versions / GPU support / PLE-offload code, build the `sglang serve`
command for a rung.
"""
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-l4"


def _run(cmd, env=None, timeout=3600):
    r = subprocess.run(list(map(str, cmd)), text=True, capture_output=True, env=env, timeout=timeout)
    if r.returncode:
        raise RuntimeError(f"{' '.join(map(str, cmd))[:300]} -> {r.returncode}\n{(r.stdout + r.stderr)[-3000:]}")
    return r.stdout


def find_bundle(root_glob):
    """The wheelhouse root: the directory holding requirements.lock and wheels/."""
    for lock in sorted(glob.glob(f"{root_glob}/**/requirements.lock", recursive=True)):
        d = Path(lock).parent
        if (d / "wheels").is_dir():
            return d
    raise FileNotFoundError(f"no requirements.lock + wheels/ under {root_glob}")


def mount_of(path):
    """The /proc/mounts entry (fs type, source) holding `path` -- 10-22 forum: some inputs moved from NFS to GCSFUSE."""
    best = ("", "?", "?")
    try:
        for line in open("/proc/mounts"):
            src, mnt, fs = line.split()[:3]
            if str(path).startswith(mnt) and len(mnt) > len(best[0]):
                best = (mnt, fs, src)
    except Exception:
        pass
    return {"mount": best[0], "fs": best[1], "source": best[2][:120]}


def stage_bundle(bundle, dest, threads=16):
    """X56 v4 (2026-10-25): copy the bundle (requirements.lock + wheels/) to local disk with parallel streams before uv
    reads it. X56 v3's install from /kaggle/input timed out after 3,600 s (X29 / X41: 90 s) -- the input storage was slow
    this week (forum 751344: GCSFUSE mounts, per-stream reads ~10-25 MB/s)."""
    from concurrent.futures import ThreadPoolExecutor
    bundle, dest = Path(bundle), Path(dest)
    files = [bundle / "requirements.lock"] + sorted((bundle / "wheels").glob("*.whl"))
    (dest / "wheels").mkdir(parents=True, exist_ok=True)
    t = time.time()

    def cp(p):
        q = dest / p.relative_to(bundle)
        if not (q.exists() and q.stat().st_size == p.stat().st_size):
            shutil.copyfile(p, q)
        return q.stat().st_size
    with ThreadPoolExecutor(threads) as ex:
        n = sum(ex.map(cp, sorted(files, key=lambda p: -p.stat().st_size)))      # biggest first: they set the wall time
    dt = time.time() - t
    return {"files": len(files), "gb": round(n / 1e9, 2), "s": round(dt), "mb_s": round(n / 1e6 / max(dt, 1e-6), 1),
            "from": mount_of(bundle)}


def install(bundle, prefix=PREFIX):
    """uv (from its own wheel) -> venv on the image's Python -> the locked wheels -> the SGLang wheel itself."""
    bundle, prefix = Path(bundle), Path(prefix)
    wheels = bundle / "wheels"
    venv = prefix / "venv"
    py = venv / "bin" / "python"
    env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
    env.pop("PYTHONPATH", None)
    uvw = sorted(wheels.glob("uv-*.whl"))
    sgl = sorted(wheels.glob("sglang-*.whl"))
    assert len(uvw) == 1 and len(sgl) == 1, (uvw, sgl)
    uv = prefix / "bin" / "uv"
    uv.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uvw[0]) as z:
        member = [n for n in z.namelist() if n.endswith("/uv") or n == "uv"]
        assert len(member) == 1, member
        data = z.read(member[0])
    assert data[:4] == b"\x7fELF", "the uv wheel does not hold a Linux binary"
    uv.write_bytes(data)
    uv.chmod(0o755)
    if not py.exists():
        _run([uv, "venv", "--python", sys.executable, venv], env)
    pip = [uv, "pip", "install", "--python", py, "--no-index", "--find-links", wheels]
    _run(pip + ["-r", bundle / "requirements.lock"], env)
    _run(pip + ["--reinstall", "--no-deps", sgl[0]], env)
    return {"python": str(py), "sglang": str(venv / "bin" / "sglang"), "venv": str(venv), "wheel": sgl[0].name}


def toolchain(venv):
    """The bundled CUDA 13 (nvcc + runtime) as CUDA_HOME, unversioned .so links, libcuda from the driver."""
    cu = sorted(Path(venv).glob("lib/python*/site-packages/nvidia/cu13"))
    assert cu and (cu[0] / "bin" / "nvcc").is_file(), "bundled CUDA 13 nvcc missing"
    home = cu[0]
    lib = home / "lib"
    if not (home / "lib64").exists():
        (home / "lib64").symlink_to("lib")
    for so in sorted(lib.glob("*.so.*")):
        link = lib / re.sub(r"\.so\..*$", ".so", so.name)
        if not link.exists() and not link.is_symlink():
            link.symlink_to(so.name)
    if not (lib / "libcuda.so").exists():
        hits = []
        for d in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            hits += sorted(p for p in Path(d).glob("libcuda.so*") if p.is_file())
        assert hits, "driver libcuda.so not found"
        (lib / "libcuda.so").symlink_to(hits[0].resolve())
    cxx = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
    assert cxx and shutil.which(cxx.replace("g++", "gcc")), "no host C/C++ compiler"
    return str(home), cxx


def env_for(venv, cuda_home, cxx, prefix=PREFIX):
    e = dict(os.environ)
    e.pop("PYTHONPATH", None)
    cache = Path(prefix) / "cache"
    for d in ("hf", "torch", "inductor", "triton", "flashinfer", "cuda", "sglang/jit"):
        (cache / d).mkdir(parents=True, exist_ok=True)
    e.update({
        "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "UV_OFFLINE": "1",
        "CUDA_HOME": cuda_home, "CUDACXX": f"{cuda_home}/bin/nvcc",
        "PATH": f"{cuda_home}/bin:{venv}/bin:" + e.get("PATH", ""),
        "LD_LIBRARY_PATH": f"{cuda_home}/lib64:{cuda_home}/lib:" + e.get("LD_LIBRARY_PATH", ""),
        "CC": cxx.replace("g++", "gcc"), "CXX": cxx, "CUDAHOSTCXX": cxx,
        "TORCH_CUDA_ARCH_LIST": "8.9",                               # L4 (the reference used 12.0)
        "HF_HOME": f"{cache}/hf", "XDG_CACHE_HOME": str(cache), "TORCH_HOME": f"{cache}/torch",
        "CUDA_CACHE_PATH": f"{cache}/cuda", "TORCHINDUCTOR_CACHE_DIR": f"{cache}/inductor",
        "TRITON_CACHE_DIR": f"{cache}/triton", "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
        "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
        "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
        "OMP_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false", "MAX_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2",
        # the fork's Blackwell-only paths stay off on sm89
        "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "0", "SGLANG_SM120_ONLINE_MXFP8": "0",
        "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
        "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false", "SGLANG_MM_PREPROCESS_DEVICE": "cpu",
    })
    return e


FACTS = r'''
import json, torch, sglang
out = {"sglang": sglang.__version__, "torch": torch.__version__, "cuda": torch.version.cuda,
       "arch_list": torch.cuda.get_arch_list(), "gpus": torch.cuda.device_count(),
       "capability": list(torch.cuda.get_device_capability(0))}
try:
    import sgl_kernel; out["sgl_kernel"] = getattr(sgl_kernel, "__version__", "?")
except Exception as e:
    out["sgl_kernel_error"] = repr(e)[:300]
try:
    import flashinfer
    p = torch.softmax(torch.randn(4, 1000, device="cuda"), -1)
    flashinfer.sampling.top_p_sampling_from_probs(p, 0.9); torch.cuda.synchronize(); out["flashinfer_jit"] = "ok"
except Exception as e:
    out["flashinfer_jit"] = repr(e)[:400]
print("SGL_FACTS " + json.dumps(out))
'''


def facts(py, env, venv):
    out = {}
    try:
        txt = _run([py, "-c", FACTS], env, timeout=1200)
        out.update(json.loads(txt.split("SGL_FACTS ", 1)[1].splitlines()[0]))
    except Exception as e:
        out["error"] = str(e)[-1500:]
    src = sorted(glob.glob(f"{venv}/lib/python*/site-packages/sglang"))
    hits = []
    for p in (glob.glob(f"{src[0]}/**/*.py", recursive=True) if src else []):
        if not re.search(r"qwen4|ple|ngram", os.path.basename(p), re.I):
            continue
        try:
            for i, line in enumerate(open(p, errors="replace")):
                if re.search(r"sm120|SM120|is_sm1[02]0|is_blackwell|ple_offload|pin_memory|cpu_offload", line):
                    hits.append(f"{p[len(src[0]) + 1:]}:{i + 1}: {line.strip()[:150]}")
        except Exception:
            pass
    out["model_code_hits"] = hits[:60]
    return out


def serve_cmd(sglang_bin, model, rung, port):
    """`sglang serve` for one rung. Base = the reference's AutoRound + PLE-offload flags adapted to TP 4; a rung adds
    its own (`args`) and may drop the fork-specific extras (`minimal`)."""
    n = rung["max_seqs"]
    # X41 (2026-10-18) rung options, defaults = X29's flags: chunk / max_prefill (prefill activation peak), mamba_cache
    # (linear-attention states kept; 27.5 MB per GPU each in X29), radix False (no prompt reuse: direct answers never
    # share a prefix, so no extra_buffer states are needed).
    # X63: the dense Qwen3.8-27B (block FP8, no PLE table) through the same launcher -> rung quantization / ple_offload
    quant = rung.get("quantization", "auto-round")
    cmd = [sglang_bin, "serve", "--model-path", str(model), "--load-format", "safetensors",
           "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", str(port),
           "--tensor-parallel-size", "4", "--dtype", "bfloat16", *(["--quantization", quant] if quant else []),
           # X56: "auto" = a BF16 cache, the one the sparse-attention decode kernel accepts on SM89 (X41b)
           "--kv-cache-dtype", rung.get("kv_dtype", "fp8_e4m3"), "--mem-fraction-static", str(rung["gpu_util"]),
           "--context-length", str(rung["max_len"]), "--max-running-requests", str(n),
           "--chunked-prefill-size", str(rung.get("chunk", 8192)),
           "--max-prefill-tokens", str(rung.get("max_prefill", 16384)),
           *(["--ple-offload-embedding"] if rung.get("ple_offload", True) else []),
           "--trust-remote-code", "--reasoning-parser", "qwen3", "--watchdog-timeout", "1800", "--enable-metrics"]
    tmpl = Path(model) / "chat_template.jinja"
    if tmpl.is_file():
        cmd += ["--chat-template", str(tmpl)]
    if rung.get("override_args"):                  # X57: e.g. {"language_model_only": true} -> no vision tower loaded
        cmd += ["--json-model-override-args", json.dumps(rung["override_args"])]
    if rung.get("tool_parser"):                    # X62: the agent's python / submit tools (dfranzen: qwen3_coder)
        cmd += ["--tool-call-parser", rung["tool_parser"]]
    if rung.get("schedule_policy"):                # X62: lpm = longest prefix match first (prefix-cache hits)
        cmd += ["--schedule-policy", rung["schedule_policy"]]
    spec = rung.get("spec")
    if spec:                                       # X59: MTP speculative decoding with the INT4 drafter (see spec_args)
        cmd += spec_args(spec, rung.get("kv_dtype", "fp8_e4m3"))
    if not rung.get("minimal"):
        bs = sorted(b for b in {1, 2, 4, 8, n} if b <= n)          # the reference keeps graph sizes <= max running
        radix = (["--mamba-radix-cache-strategy", "extra_buffer", "--mamba-track-interval", "64"]
                 if rung.get("radix", True) else ["--disable-radix-cache"])
        # X41: FlashInfer's GDN kernels need SM90+ ("FlashInfer GDN kernels are not available. Requires SM90+");
        # on L4 the linear-attention layers run on SGLang's Triton kernels (its default there).
        lin = rung.get("linear_backend", "flashinfer")
        # X64 v2 (2026-10-27): the recurrent state's precision. bfloat16 = dfranzen's Flash-Next recipe; Qwen3.8-27B's
        # checkpoint declares float32 (SGLang cookbook: "treat bfloat16 as an accuracy gate"), and vLLM 0.19 keeps it in
        # float32 (X44 log: mamba page ~812 KB/layer/GPU = 12 heads x 128 x 128 x 4 B + conv state) -> X63 ran bf16.
        cmd += ["--page-size", "64", "--mamba-ssm-dtype", rung.get("ssm_dtype", "bfloat16"),
                "--max-mamba-cache-size", str(rung.get("mamba_cache", 3 * n)), *radix,
                "--mamba-backend", lin, "--linear-attn-decode-backend", lin,
                "--linear-attn-prefill-backend", lin, "--moe-runner-backend", "auto",
                "--cuda-graph-max-bs-decode", str(n), "--cuda-graph-bs-decode", *map(str, bs),
                "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
                "--weight-loader-prefetch-checkpoints", "--weight-loader-drop-cache-after-load"]
    return cmd + list(rung.get("args", []))


def spec_args(spec, kv_dtype="fp8_e4m3"):
    """X59 (2026-10-25), from dfranzen's ARC-AGI-3 notebook (RTX PRO 6000, same Pennyroyal v2.5.3 + AutoRound target):
    NEXTN with the albucino MTP drafter (MTP experts INT4 group 32, compressed-tensors; 1.4 GB instead of the checkpoint's
    4.9 GB BF16 MTP), a linear draft chain (topk 1) of `steps`, strict acceptance, and optionally FR-Spec's 64k hot-token
    map (the drafter's output layer restricted to frequent tokens). `spec` carries the runtime paths: draft_view (from
    prepare_draft_view) and token_map. They run --gdn-mtp-cache-mode none, whose verify uses a FlashInfer GDN kernel
    (SM90+); L4 keeps the default "full" unless the rung says otherwise."""
    if spec.get("algo") == "DFLASH":
        # X59: incoai's DFlash2 drafter for Flash-Next (BF16, 5 layers, block 8). Their model card serves it with
        # --speculative-num-draft-tokens 4 ("limited by the current QSA implementation"), an unquantized draft, Triton
        # verify for the linear-attention layers. The draft cache has one slot per target token (shared index space):
        # BF16 = 5 layers x 2 KV heads/GPU x 128 x 2 x 2 B ~ 5 KB/token/GPU next to the target's ~5.7 KB FP8 -- kept BF16
        # anyway ("auto" would follow the FP8 target cache, untested in FlashInfer's non-causal draft attention on SM89).
        return ["--speculative-algorithm", "DFLASH", "--speculative-draft-model-path", str(spec["draft_path"]),
                "--speculative-draft-model-quantization", "unquant",
                "--speculative-num-draft-tokens", str(spec.get("block", 4)),
                "--speculative-draft-kv-cache-dtype", spec.get("draft_kv_dtype", "bf16"),
                "--speculative-draft-model-override-args", "{}", "--linear-attn-verify-backend", "triton"]
    steps = int(spec.get("steps", 3))
    # X63: the 27B's own MTP head sits in its checkpoint (draft path = the model, quantized like it) -> draft_quant None
    dq = spec.get("draft_quant", "compressed-tensors")
    out = ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(steps), "--speculative-eagle-topk", "1",
           "--speculative-num-draft-tokens", str(steps + 1), "--speculative-draft-model-path", str(spec["draft_view"]),
           *(["--speculative-draft-model-quantization", dq] if dq else []), "--speculative-moe-runner-backend", "auto",
           "--speculative-draft-kv-cache-dtype", kv_dtype, "--speculative-accept-threshold-single", "1.0",
           "--speculative-accept-threshold-acc", "1.0"]
    if spec.get("token_map"):
        out += ["--speculative-token-map", str(spec["token_map"])]
    if spec.get("gdn_mtp_cache_mode"):
        out += ["--gdn-mtp-cache-mode", spec["gdn_mtp_cache_mode"]]
    return out


DRAFT_EXPERTS = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DRAFT_DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def prepare_draft_view(target, draft_root, work_root):
    """X59: SGLang reads the albucino drafter only once its compressed-tensors config names the MTP expert layers by
    regex (the checkpoint says "RoutedExperts", ignore []). Build a directory holding a rewritten config.json plus
    symlinks to the read-only weights / index and the target's tokenizer + chat template (dfranzen's prepare_draft_view,
    same checks). Returns the view path."""
    target, draft_root, work_root = Path(target), Path(draft_root), Path(work_root)
    found = []
    for p in sorted(draft_root.rglob("config.json")):
        c = json.loads(p.read_text())
        q = c.get("quantization_config") or {}
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            found.append((p.parent, c))
    if len(found) != 1:
        raise RuntimeError(f"expected one INT4 g32 MTP drafter under {draft_root}, found {len(found)}")
    src, c = found[0]
    index = json.loads((src / "model.safetensors.index.json").read_text())
    shards = sorted(set(index["weight_map"].values()))
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("the drafter index holds no MTP experts")
    q = c["quantization_config"]
    g = q["config_groups"]["mtp_routed_experts"]
    w = g["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError(f"expected symmetric INT4 group-32 drafter experts, got {w}")
    if g["targets"] not in (["RoutedExperts"], [DRAFT_EXPERTS]) or q.get("ignore", []) not in ([], [DRAFT_DENSE_IGNORE]):
        raise RuntimeError(f"unexpected drafter target / ignore rules: {g['targets']} / {q.get('ignore')}")
    g["targets"], q["ignore"] = [DRAFT_EXPERTS], [DRAFT_DENSE_IGNORE]
    view = work_root / "draft-view"
    view.mkdir(parents=True, exist_ok=True)
    (view / "config.json").write_text(json.dumps(c, indent=2) + "\n")
    links = {n: src / n for n in shards}
    links["model.safetensors.index.json"] = src / "model.safetensors.index.json"
    for n in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json",
              "generation_config.json"):
        if (target / n).is_file():
            links[n] = target / n
    for n, s in links.items():
        d = view / n
        if d.is_symlink() or d.exists():
            d.unlink()
        d.symlink_to(s.resolve())
    return view


SPARSE_REL = "sglang/srt/layers/attention/qwen_sparse_attn_backend.py"
SCRATCH_DTYPE = re.compile(r"^(\s*)k_buffer\.dtype,\s*$")
SCRATCH_FP8 = ("(q.dtype if k_buffer.dtype in (torch.float8_e4m3fn, torch.float8_e5m2) else k_buffer.dtype),"
               "  # X56 patch: an FP8 cache packs into q-dtype scratch (the compaction store casts)")


def _sparse_file(venv):
    hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{SPARSE_REL}"))
    return hits[0] if len(hits) == 1 else None


def patch_sparse_fp8(venv):
    """X56 (2026-10-24; Pennyroyal's own source + SGLang #36545 / #36644 / #36968): on SM89 the QSA decode
    (_forward_paged_attention) gathers each query's top-k KV entries into scratch of the CACHE's dtype, then calls the
    varlen kernel -- classic FA2 when importable, else FA4 cute. Neither takes bf16 q with FP8 k / v (X41b's assert).
    With an FP8 cache, allocate that scratch in q's dtype instead: _compact_kv's tl.store casts on write, so the ~2,048
    gathered entries arrive dequantized (Flash-Next ships no KV scales -> scale 1.0), as #36644 does for its FlashAttention
    fallback. Text patch of the scratch call inside _forward_paged_attention only, with checks."""
    p = _sparse_file(venv)
    if p is None:
        return {"ok": False, "why": f"{SPARSE_REL} not found exactly once"}
    src = open(p, encoding="utf-8").read()
    if "X56 patch" in src:
        return {"ok": True, "already": True, "file": p}
    if not re.search(r"^import torch\s*$", src, re.M):
        return {"ok": False, "why": "torch is not imported at module level", "file": p}
    lines = src.split("\n")
    i = next((k for k, l in enumerate(lines) if l.lstrip().startswith("def _forward_paged_attention")), None)
    if i is None:
        return {"ok": False, "why": "no def _forward_paged_attention", "file": p}
    ind = len(lines[i]) - len(lines[i].lstrip())

    def ends(line):                    # the next method / class (a multi-line signature's ") -> T:" does not count)
        dent = len(line) - len(line.lstrip())
        return bool(line.strip()) and (dent < ind or (dent == ind and bool(re.match(r"(async\s+def|def|class)\s|@",
                                                                                   line.strip()))))
    end = next((k for k in range(i + 1, len(lines)) if ends(lines[k])), len(lines))
    j = next((k for k in range(i, end) if "self._get_fa2_scratch(" in lines[k]), None)
    if j is None:
        return {"ok": False, "why": "no _get_fa2_scratch call in _forward_paged_attention", "file": p}
    k = next((k for k in range(j + 1, min(end, j + 8)) if SCRATCH_DTYPE.match(lines[k])), None)
    if k is None:
        return {"ok": False, "why": "no 'k_buffer.dtype,' argument in that call", "file": p,
                "context": "\n".join(lines[j:j + 8])}
    lines[k] = SCRATCH_DTYPE.match(lines[k]).group(1) + SCRATCH_FP8
    src = "\n".join(lines)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", "qwen_sparse_attn_backend.*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p, "line": k + 1, "context": "\n".join(lines[j:k + 3])}


def install_fa2(venv, wheel_glob, prefix=PREFIX):
    """X56: classic flash-attn 2 into the SGLang venv (offline, no deps). SGLang's QSA resolver prefers it over FA4 cute
    on non-SM12x GPUs; without it SM89 falls to cute and the first sparse decode fails (#36968). The RTX 4090 run that
    served Flash-Next used flash-attn 2.8.3.post1 on torch 2.13 cu130 -- the Pennyroyal bundle's torch."""
    hits = sorted(glob.glob(wheel_glob, recursive=True))
    if not hits:
        return {"ok": False, "why": f"no wheel at {wheel_glob}"}
    uv = Path(prefix) / "bin" / "uv"
    env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
    env.pop("PYTHONPATH", None)
    try:
        t = time.time()                                # X56 v4: local copy first (slow input storage)
        local = Path(prefix) / os.path.basename(hits[0])
        shutil.copyfile(hits[0], local)
        copy_s = round(time.time() - t)
        _run([uv, "pip", "install", "--python", Path(venv) / "bin" / "python", "--no-index", "--no-deps", local], env)
    except Exception as e:
        return {"ok": False, "wheel": os.path.basename(hits[0]), "why": str(e)[-1500:]}
    return {"ok": True, "wheel": os.path.basename(hits[0]), "copy_s": copy_s, "from": mount_of(hits[0])}


FA2_CHECK = r"""
import json, torch
out = {}
try:
    import flash_attn
    from flash_attn import flash_attn_varlen_func
    out["flash_attn"] = flash_attn.__version__
    torch.manual_seed(0)
    b, hq, hk, d, n = 3, 6, 1, 256, 2048          # one TP rank of Flash-Next's QSA decode: GQA 6:1, head 256, top-k 2048
    q = torch.randn(b, hq, d, device="cuda", dtype=torch.bfloat16)
    k = torch.randn(b * n, hk, d, device="cuda", dtype=torch.bfloat16)
    v = torch.randn_like(k)
    cu_q = torch.arange(b + 1, dtype=torch.int32, device="cuda")
    cu_k = torch.arange(0, b * n + 1, n, dtype=torch.int32, device="cuda")
    o = flash_attn_varlen_func(q=q, k=k, v=v, cu_seqlens_q=cu_q, cu_seqlens_k=cu_k, max_seqlen_q=1, max_seqlen_k=n,
                               softmax_scale=d ** -0.5, causal=True)
    torch.cuda.synchronize()
    err = 0.0
    for i in range(b):
        kk, vv = k[i * n:(i + 1) * n, 0].float(), v[i * n:(i + 1) * n, 0].float()
        ref = torch.softmax(q[i].float() @ kk.T * d ** -0.5, -1) @ vv
        err = max(err, (o[i].float() - ref).abs().max().item())
    out["fa2_max_err"] = round(err, 4)
    out["fa2_ok"] = err < 0.05
except Exception as e:
    out["fa2_error"] = repr(e)[:500]
try:
    from sglang.srt.layers.attention.qwen_sparse_attn_backend import _resolve_flash_attn_varlen_func
    f = _resolve_flash_attn_varlen_func()
    out["qsa_varlen"] = f"{f.__module__}.{f.__name__}"
except Exception as e:
    out["qsa_varlen_error"] = repr(e)[:500]
print("FA2_CHECK " + json.dumps(out))
"""


def fa2_check(py, env):
    """X56: FA2 runs Flash-Next's decode shape on this GPU (vs a float32 reference) AND SGLang's QSA resolver picks it."""
    try:
        txt = _run([py, "-c", FA2_CHECK], env, timeout=1200)
        out = json.loads(txt.split("FA2_CHECK ", 1)[1].splitlines()[0])
    except Exception as e:
        return {"ok": False, "error": str(e)[-1500:]}
    out["ok"] = bool(out.get("fa2_ok")) and "flash_attn_interface" in out.get("qsa_varlen", "")
    return out


def sparse_source(venv, around=(("def _resolve_flash_attn_varlen_func", 36), ("def _forward_paged_attention", 95))):
    """X56: the sparse-attention backend's own code -- the varlen resolver, the paged decode, and every line naming the
    kernel choice, FP8 or the GPU generation -- so a failure here says what to change next without another run."""
    p = _sparse_file(venv)
    if p is None:
        return {"error": f"{SPARSE_REL} not found exactly once"}
    lines = open(p, encoding="utf-8", errors="replace").read().split("\n")
    out = {"file": p, "lines": len(lines)}
    for name, n in around:
        i = next((k for k, l in enumerate(lines) if l.lstrip().startswith(name)), None)
        out[name] = None if i is None else "\n".join(f"{k + 1}: {lines[k]}" for k in range(i, min(len(lines), i + n)))
    pat = re.compile(r"cute|flash_attn|fp8|float8|kv_cache_dtype|is_sm|capability|environ|flashinfer|scratch")
    out["hits"] = [f"{k + 1}: {l.strip()[:160]}" for k, l in enumerate(lines) if pat.search(l)][:120]
    return out


QWEN4_REL = "sglang/srt/models/qwen4_exp.py"
# X59: incoai's qwen38-dflash2-capture.patch (Qwen3.8-Flash-Next-DFlash2 repo; written for upstream SGLang 3c56008, TP1)
# ported to Pennyroyal's qwen4_exp.py. Without it the Flash-Next forward hands speculative workers its hyper-connection
# state (4 x 2560) instead of the 5 captured layer outputs DFlash2 was trained on. Each (anchor, insert-before-or-after):
_DFLASH_LOOP_ANCHOR = ("                    ),\n                )\n\n        _commit_ple_batch(ple_batch, forward_batch)\n")
_DFLASH_LOOP = ("                    ),\n                )\n"
                "            if i in getattr(self, \"_dflash_completed_layer_ids\", ()):  # X59 DFlash2 capture patch\n"
                "                # DFlash2 was trained on completed, zero-based layer outputs averaged over the 4 streams\n"
                "                assert hidden_states.shape[-1] == self.hc_count * self.hidden_size\n"
                "                aux_hidden_states.append(\n"
                "                    hidden_states.unflatten(-1, (self.hc_count, self.hidden_size)).mean(dim=-2)\n"
                "                )\n"
                "\n        _commit_ple_batch(ple_batch, forward_batch)\n")
_DFLASH_RET_ANCHOR = ("        hidden_states, _ = self.hyper_connection_mixer.mix(hidden_states)\n"
                      "        if not forward_batch.forward_mode.is_idle():\n")
_DFLASH_RET = ("        hidden_states, _ = self.hyper_connection_mixer.mix(hidden_states)\n"
               "        if getattr(self, \"_dflash_completed_layer_ids\", ()):  # X59 DFlash2 capture patch\n"
               "            return hidden_states, aux_hidden_states\n"
               "        if not forward_batch.forward_mode.is_idle():\n")
_DFLASH_VL_ANCHOR = ("            inputs_embeds=input_embeds,\n        )\n        if isinstance(model_output, tuple):\n")
_DFLASH_VL = ("            inputs_embeds=input_embeds,\n        )\n"
              "        if getattr(self, \"_dflash_completed_layer_ids\", ()):  # X59 DFlash2 capture patch\n"
              "            return model_output\n"
              "        if isinstance(model_output, tuple):\n")
_DFLASH_SET_ANCHOR = ("            self.visual.deepstack_visual_indexes if self.visual is not None else []\n        )\n\n"
                      "    @torch.no_grad()\n    def forward(self, *args, **kwargs):\n")
_DFLASH_SET = ("            self.visual.deepstack_visual_indexes if self.visual is not None else []\n        )\n\n"
               "    def set_dflash_layers_to_capture(self, layer_ids):  # X59 DFlash2 capture patch\n"
               "        assert self.pp_group.is_last_rank\n"
               "        assert list(layer_ids) == [5, 15, 25, 35, 45], layer_ids\n"
               "        self.capture_aux_hidden_states = True\n"
               "        self.model._dflash_completed_layer_ids = tuple(layer_ids)\n"
               "        for layer in self.model.layers:\n"
               "            layer._is_layer_to_capture = False\n"
               "        print(\"X59_DFLASH_CAPTURE completed_layers=[5,15,25,35,45] hc_mean=True features=12800\", "
               "flush=True)\n\n"
               "    @torch.no_grad()\n    def forward(self, *args, **kwargs):\n")


def patch_dflash_capture(venv):
    """X59: apply the DFlash2 feature-capture port to Pennyroyal's qwen4_exp.py (4 text edits, each anchor exactly once,
    compiled before writing). Inert unless SGLang calls set_dflash_layers_to_capture, i.e. only under DFLASH."""
    hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{QWEN4_REL}"))
    if len(hits) != 1:
        return {"ok": False, "why": f"{QWEN4_REL} found {len(hits)} times"}
    p = hits[0]
    src = open(p, encoding="utf-8").read()
    if "X59 DFlash2 capture patch" in src:
        return {"ok": True, "already": True, "file": p}
    for name, (anchor, new) in (("loop", (_DFLASH_LOOP_ANCHOR, _DFLASH_LOOP)), ("return", (_DFLASH_RET_ANCHOR, _DFLASH_RET)),
                                ("vl", (_DFLASH_VL_ANCHOR, _DFLASH_VL)), ("set", (_DFLASH_SET_ANCHOR, _DFLASH_SET))):
        n = src.count(anchor)
        if n != 1:
            return {"ok": False, "why": f"anchor '{name}' found {n} times", "file": p}
        src = src.replace(anchor, new)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", "qwen4_exp.*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p}


MARLIN_REL = "sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py"
_MARLIN_SUM_ANCHOR = ("        moe_sum_reduce(\n            intermediate_cache3,\n            output,\n"
                      "            routed_scaling_factor,\n        )\n        return output")
_MARLIN_SUM = ("        # X59 patch: sgl_kernel's moe_sum_reduce (small-token) failed to launch on SM89 (X58 e0, CUDA graph\n"
               "        # capture); a plain sum over the top-k outputs (top-k weights already applied by the GEMM)\n"
               "        torch.sum(intermediate_cache3, dim=1, out=output)\n"
               "        if routed_scaling_factor != 1.0:\n"
               "            output.mul_(routed_scaling_factor)\n"
               "        return output")


def patch_marlin_sum(venv):
    """X59 (2026-10-25): with expert parallelism the experts run on Marlin (X58 e0: GPTQMarlinMoEMethod, scales 4x
    smaller), whose last step -- sgl_kernel.moe_sum_reduce over the 10 routed outputs -- raised "moe_sum_reduce CUDA
    kernel (small-token) launch failed" on L4 while capturing decode graphs. Replace that call with torch.sum (same
    math; the source's own comment puts the generic reduce at ~6.7 us vs ~5.7). Text patch with checks."""
    hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{MARLIN_REL}"))
    if len(hits) != 1:
        return {"ok": False, "why": f"{MARLIN_REL} found {len(hits)} times"}
    p = hits[0]
    src = open(p, encoding="utf-8").read()
    if "X59 patch" in src:
        return {"ok": True, "already": True, "file": p}
    n = src.count(_MARLIN_SUM_ANCHOR)
    if n != 1:
        return {"ok": False, "why": f"moe_sum_reduce anchor found {n} times", "file": p}
    src = src.replace(_MARLIN_SUM_ANCHOR, _MARLIN_SUM)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", "fused_marlin_moe.*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p}


# X60 (2026-10-25): X59's EP rung served "!!!!" (logits garbage). Under EP the StandardDispatcher hands the MoE runner
# LOCAL expert ids with -1 for the other ranks' experts ("-1 is the drop sentinel both the triton and deep_gemm runners
# honor"). fused_marlin_moe honors it only when given an expert_map (is_ep -> the kernel skips -1 blocks; masked rows of
# the shared cache zeroed), but GPTQ-Marlin's MarlinMoeQuantInfo never sets one, nor does compressed-tensors' runner path
# (its direct apply_weights path does: dispatcher.local_expert_mapping + num_experts). So Marlin read expert -1 (X58:
# "launch failed", a sticky error from an out-of-bounds read) or summed stale rows (X59: garbage). Pass the mapping.
_EP_KWARGS = ("            # X60 patch: under EP give Marlin the dispatcher's mapping, so the -1 (other ranks') experts are\n"
              "            # skipped and their rows zeroed instead of read / summed (X58 launch failure, X59 garbage)\n"
              "            expert_map=(\n"
              "                getattr(getattr(layer, \"dispatcher\", None), \"local_expert_mapping\", None)\n"
              "                if getattr(layer, \"moe_ep_size\", 1) > 1 else None\n"
              "            ),\n"
              "            global_num_experts=(layer.num_experts if getattr(layer, \"moe_ep_size\", 1) > 1 else -1),\n")
_EP_PATCHES = (
    ("sglang/srt/hardware_backend/gpu/quantization/gptq_kernels.py",
     "            weight_bits=self.quant_config.weight_bits,\n            is_k_full=self.is_k_full,\n        )\n"),
    ("sglang/srt/layers/quantization/compressed_tensors/schemes/compressed_tensors_wNa16_moe.py",
     "            w2_qzeros=layer.w2_weight_zero_point if not self.sym else None,\n        )\n"),
)


def patch_marlin_ep(venv):
    """X60: insert expert_map / global_num_experts into both Marlin MoE quant-info builders (anchor exactly once in
    each, compiled before writing). Inert without expert parallelism (moe_ep_size 1 -> None / -1, as before)."""
    out = {"ok": True, "files": []}
    for rel, anchor in _EP_PATCHES:
        hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{rel}"))
        if len(hits) != 1:
            return {"ok": False, "why": f"{rel} found {len(hits)} times"}
        p = hits[0]
        src = open(p, encoding="utf-8").read()
        if "X60 patch" in src:
            out["files"].append({"file": p, "already": True})
            continue
        n = src.count(anchor)
        if n != 1:
            return {"ok": False, "why": f"anchor found {n} times in {rel}"}
        closing = anchor.rsplit("        )\n", 1)
        src = src.replace(anchor, closing[0] + _EP_KWARGS + "        )\n" + closing[1])
        compile(src, p, "exec")
        open(p, "w", encoding="utf-8").write(src)
        for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", os.path.basename(p)[:-3] + ".*.pyc")):
            os.remove(c)
        out["files"].append({"file": p})
    return out


# c23fp (2026-10-31; reh2 crashed on every pruned rung): EP builds ONE expert-location table, in the target's worker
# from the target's config -- 384 experts once pruned. albucino's MTP drafter keeps 512, and its weight loader looked
# expert 384 up in that table (IndexError ... size 384). With init_expert_location "trivial", no redundant experts and no
# dispatch algorithm, the table is the identity and routing never reads it, so a MoE layer whose routed-expert count
# differs from the table's loads as if there were no table: SGLang's own path, by the layer's own numbering (read in
# the Pennyroyal wheel, sglang 0.5.19+gd00d88efc8d6). Target layers match the table and keep the stock path; with an
# unpruned target (c23f) the guard never fires.
_DRAFT_MAP = ("sglang/srt/layers/moe/fused_moe_triton/layer.py",
              "        global_expert_location_metadata = get_global_expert_location_metadata()\n"
              "        if global_expert_location_metadata is None:\n")
_DRAFT_MAP_ADD = (
    "        if (global_expert_location_metadata is not None   # c23fp patch: a drafter with its own expert count\n"
    "                and global_expert_location_metadata.num_logical_experts != self._num_global_routed\n"
    '                and get_exec().moe.init_expert_location == "trivial"\n'
    "                and get_exec().moe.ep_num_redundant_experts == 0\n"
    "                and get_exec().moe.ep_dispatch_algorithm is None):\n"
    "            global_expert_location_metadata = None\n")


def patch_draft_expert_map(venv):
    """c23fp: insert the guard right after FusedMoE.weight_loader reads the table (anchor exactly once, compiled before
    writing)."""
    rel, anchor = _DRAFT_MAP
    hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{rel}"))
    if len(hits) != 1:
        return {"ok": False, "why": f"{rel} found {len(hits)} times"}
    p = hits[0]
    src = open(p, encoding="utf-8").read()
    if "c23fp patch" in src:
        return {"ok": True, "file": p, "already": True}
    n = src.count(anchor)
    if n != 1:
        return {"ok": False, "why": f"anchor found {n} times in {rel}"}
    first, rest = anchor.split("\n", 1)
    src = src.replace(anchor, first + "\n" + _DRAFT_MAP_ADD + rest)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", os.path.basename(p)[:-3] + ".*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p}


# X75 (2026-10-30): the MTP drafter loads its own embed + lm_head (0.3 + 0.3 GB/GPU on 4x L4, X60 log) and swaps them
# for the target's in EagleDraftWorker.alloc_memory_pool -> init_lm_head -> set_embed_and_head, which the scheduler calls
# AFTER the target's KV pool is sized: the copy is counted as used, then freed into nothing. Share them before the pool
# is sized instead. Gated by the server's env (ARC_X75_EARLY_HEAD=1), so one install serves rungs with and without it.
_HEAD_SCHED = ("sglang/srt/managers/scheduler.py",
               '    def init_memory_pools(self):\n        """Allocate KV cache pools for target and draft workers."""\n')
_HEAD_SCHED_ADD = (
    "        # X75 patch (ARC_X75_EARLY_HEAD=1): the MTP drafter takes the target's embed + lm_head BEFORE the target KV\n"
    "        # pool is sized; stock does it after (EagleDraftWorker.alloc_memory_pool -> init_lm_head), so the drafter's\n"
    "        # own copy is counted as used when the pool is sized and then freed into nothing.\n"
    '        if os.environ.get("ARC_X75_EARLY_HEAD") == "1" and self.draft_worker is not None:\n'
    "            _x75_dw = self.draft_worker\n"
    '            if not hasattr(_x75_dw, "init_lm_head"):          # spec v2: EAGLEWorkerV2 -> its EagleDraftWorker\n'
    '                _x75_dw = getattr(_x75_dw, "draft_worker", None)\n'
    '            if hasattr(_x75_dw, "init_token_map") and hasattr(_x75_dw, "init_lm_head"):\n'
    "                import torch as _x75_torch\n"
    "                _x75_free = _x75_torch.cuda.mem_get_info()[0]\n"
    "                _x75_dw.init_token_map()\n"
    "                _x75_dw.init_lm_head()\n"
    "                _x75_dw._x75_head_shared = True\n"
    '                logger.info("X75 EARLY HEAD: drafter embed + lm_head shared before the KV pool; free %.3f -> %.3f GB",\n'
    "                            _x75_free / 1e9, _x75_torch.cuda.mem_get_info()[0] / 1e9)\n"
    "            else:\n"
    '                logger.info("X75 EARLY HEAD: NOT applied -- no drafter with init_lm_head (%s)",\n'
    "                            type(self.draft_worker).__name__)\n")
_HEAD_EAGLE = ("sglang/srt/speculative/eagle_worker_v2.py",
               "        self.init_token_map()\n        self.init_lm_head()\n")
_HEAD_EAGLE_NEW = ('        if not getattr(self, "_x75_head_shared", False):   # X75 patch: shared before the KV pool\n'
                   "            self.init_token_map()\n            self.init_lm_head()\n")


def patch_early_draft_head(venv):
    """X75: insert the early share into Scheduler.init_memory_pools and make EagleDraftWorker.alloc_memory_pool skip
    its own share once done (each anchor exactly once, compiled before writing). Inert unless the server's env sets
    ARC_X75_EARLY_HEAD=1; the server log then says "X75 EARLY HEAD" with the free memory before and after (or "NOT
    applied" when no drafter has init_lm_head)."""
    out = {"ok": True, "files": []}
    for (rel, anchor), new in ((_HEAD_SCHED, _HEAD_SCHED[1] + _HEAD_SCHED_ADD), (_HEAD_EAGLE, _HEAD_EAGLE_NEW)):
        hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{rel}"))
        if len(hits) != 1:
            return {"ok": False, "why": f"{rel} found {len(hits)} times"}
        p = hits[0]
        src = open(p, encoding="utf-8").read()
        if "X75 patch" in src:
            out["files"].append({"file": p, "already": True})
            continue
        n = src.count(anchor)
        if n != 1:
            return {"ok": False, "why": f"anchor found {n} times in {rel}"}
        if rel == _HEAD_SCHED[0] and not (re.search(r"^import os$", src, re.M) and re.search(r"^logger = ", src, re.M)):
            return {"ok": False, "why": "scheduler.py lacks module-level `import os` / `logger`"}
        src = src.replace(anchor, new)
        compile(src, p, "exec")
        open(p, "w", encoding="utf-8").write(src)
        for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", os.path.basename(p)[:-3] + ".*.pyc")):
            os.remove(c)
        out["files"].append({"file": p})
    return out


RUNNER_REL = "sglang/srt/model_executor/model_runner.py"
MEM_ANCHOR = re.compile(r"^(\s*)report_online_quantization\($")
MEM_REPORT = r'''# X57 patch: GPU bytes per parameter group on TP rank 0 -- where the weight memory goes
try:
    if self.ps.tp_rank == 0:
        import collections as _c, json as _j, re as _r
        _agg, _dt, _seen = _c.Counter(), _c.Counter(), set()
        for _n, _t in list(self.model.named_parameters()) + list(self.model.named_buffers()):
            if _t.is_cuda and _t.data_ptr() not in _seen:
                _seen.add(_t.data_ptr())
                _b = _t.numel() * _t.element_size()
                _agg[_r.sub(r"\.\d+\.", ".N.", _n)] += _b
                _dt[str(_t.dtype)] += _b
        logger.info("X57 MEMREPORT " + _j.dumps({
            "total_gib": round(sum(_agg.values()) / 2**30, 3),
            "by_dtype_gib": {k: round(v / 2**30, 3) for k, v in _dt.items()},
            "torch_allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
            "torch_reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
            "top_mib": [[k, round(v / 2**20, 1)] for k, v in _agg.most_common(80)]}))
except Exception as _e:
    logger.info("X57 MEMREPORT failed: %r", _e)'''


def patch_mem_report(venv):
    """X57 (2026-10-25): X56 v5 loaded 19.79 GiB of weights per GPU where the checkpoint's GPU-side groups / 4 give
    ~17.1 (experts 58.9 + other 6.4 + embed/lm_head 2.4 + vision 0.8 GiB); ~2.5 GiB/GPU unexplained (replicated
    tensors?). Log the per-group GPU bytes on rank 0 right after the load ("X57 MEMREPORT {...}"). Diagnostic only."""
    hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/{RUNNER_REL}"))
    if len(hits) != 1:
        return {"ok": False, "why": f"{RUNNER_REL} found {len(hits)} times"}
    p = hits[0]
    src = open(p, encoding="utf-8").read()
    if "X57 patch" in src:
        return {"ok": True, "already": True, "file": p}
    lines = src.split("\n")
    found = [k for k, l in enumerate(lines) if MEM_ANCHOR.match(l)]
    if len(found) != 1:
        return {"ok": False, "why": f"{len(found)} 'report_online_quantization(' lines", "file": p}
    k = found[0]
    ind = MEM_ANCHOR.match(lines[k]).group(1)
    lines[k:k] = [ind + l if l else l for l in MEM_REPORT.split("\n")]
    src = "\n".join(lines)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", "model_runner.*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p, "line": k + 1}


def patch_triton_bench_cache(venv, mb=64):
    """X41 v1 (2026-10-18): the first request died in Triton's autotuner -- do_bench flushes L2 with a 256 MiB buffer
    (driver.get_empty_cache_for_benchmark) and only 160 MiB was free next to the weights + KV pool. Shrink that buffer
    to `mb` MiB (still > L4's 48 MB L2, so the flush still works). Text patch with checks; returns what it did."""
    hits = sorted(glob.glob(f"{venv}/lib/python*/site-packages/triton/backends/nvidia/driver.py"))
    if len(hits) != 1:
        return {"ok": False, "why": f"{len(hits)} driver.py files", "files": hits}
    p = hits[0]
    src = open(p, encoding="utf-8").read()
    if "X41 patch" in src:
        return {"ok": True, "already": True, "file": p}
    pat = re.compile(r"^(\s*)cache_size = 256 \* 1024 \* 1024\s*$", re.M)
    found = pat.findall(src)
    if len(found) != 1:
        return {"ok": False, "why": f"{len(found)} 'cache_size = 256 * 1024 * 1024' lines", "file": p}
    src = pat.sub(lambda m: f"{m.group(1)}cache_size = {int(mb)} * 1024 * 1024  # X41 patch: was 256 MiB", src)
    compile(src, p, "exec")
    open(p, "w", encoding="utf-8").write(src)
    for c in glob.glob(os.path.join(os.path.dirname(p), "__pycache__", "driver.*.pyc")):
        os.remove(c)
    return {"ok": True, "file": p, "mb": int(mb)}

In [ ]:
%%writefile /kaggle/working/agent/flash_sub.py
"""c23f (2026-10-28): Qwen3.8-Flash-Next + the python-first agent as the WHOLE submission.

Why (user, 10-28, after forum 750546 / 752709): the top 20 of the leaderboard are at 68-88 and the leading teams say
they run Qwen3.8-Flash-Next with the Duck harness (Tufa Labs' python-tool agent -- our repl_agent is modelled on it);
the organizers cleared Flash-Next. We serve it on 4x L4 (X60: EP4 + MTP drafter, 90.7 tok/s single, ~181 at 4) and
have the agent (X44) with X69's python-first changes; the two were never run together (X62 built, not run).

flash_smoke.main brings the server up (install, patches, rung ladder, serve check) and hands over to run() here:
  1. tool check + THINK CHECK (which message key carries earlier reasoning; none -> reasoning not kept);
  2. every task in prompt-length order (smallest first), attempt 1 for all, then attempt 2 for the unverified ones
     (repl_agent.solve_repl: attempt-major, stop_after=1), concurrency = min(max_seqs, KV pool // kv_per_agent);
  3. after EVERY attempt the task's candidates are re-ranked (verified programs first, then unverified ones that got at
     least one example right) and submission.json is rewritten atomically -- a crash or a timeout leaves a valid file;
  4. rehearsal (eval tasks with solutions): the C23F REPORT line scores it.
The CPU solver's answers are merged by the notebook afterwards (cpu_merge: fallback slots first).
"""
import gzip
import json
import os
import threading
import time
from collections import defaultdict
from pathlib import Path


def _key(g):
    return json.dumps(g, separators=(",", ":"))


def candidates(task, recs, min_ok=1):
    """Per test output: grids best first -- verified programs (by votes), then unverified ones that reproduced at least
    `min_ok` examples (by examples right, then votes). Same ranking as agent.aggregate, over attempt RECORDS."""
    out = []
    for j in range(len(task["test"])):
        ver, unv, grids = defaultdict(int), {}, {}
        for r in recs:
            for p in r.get("programs") or []:
                g = (p.get("test") or [None] * (j + 1))[j] if j < len(p.get("test") or []) else None
                if not g:
                    continue
                k = _key(g)
                grids[k] = g
                if p["n_ok"] == p["n_train"]:
                    ver[k] += 1
                elif p["n_ok"] >= min_ok:
                    best, votes = unv.get(k, (0, 0))
                    unv[k] = (max(best, p["n_ok"]), votes + 1)
        ranked = [grids[k] for k, _ in sorted(ver.items(), key=lambda kv: -kv[1])]
        ranked += [grids[k] for k in sorted((k for k in unv if k not in ver), key=lambda k: (-unv[k][0], -unv[k][1]))]
        out.append(ranked)
    return out


def fallback_sub(ch):
    return {t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in task["test"]] for t, task in ch.items()}


def write_atomic(path, obj):
    tmp = str(path) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


def apply(sub, tid, task, cands):
    """Agent candidates into the task's rows: attempt_1 / attempt_2 from the ranked list; a slot without a candidate
    keeps what it has (the fallback, later filled by the CPU solver)."""
    for j, c in enumerate(cands):
        if j >= len(sub[tid]):
            break
        if c:
            sub[tid][j]["attempt_1"] = c[0]
        if len(c) > 1:
            sub[tid][j]["attempt_2"] = c[1]


def score(sub, sol, ids):
    return sum(sum(any(sub[t][j][a] == s for a in ("attempt_1", "attempt_2")) for j, s in enumerate(sol[t]))
               / len(sol[t]) for t in ids)


def agent_prep(base, sc, rep, rung, out):
    """Tool check + THINK CHECK + the agent's options, concurrency and client (shared with X72's calibration).
    Returns (opts, conc, client) or None when no python tool call parses."""
    import flash_smoke
    import run_probe
    from client import ChatClient
    from repl_agent import AGENT_OPTS
    rep["tool_check"] = flash_smoke.tool_check(base)
    print("TOOL CHECK", json.dumps(rep["tool_check"]), flush=True)
    opts = {k: v for k, v in sc["agent"].items() if k in AGENT_OPTS or k in ("max_turns", "max_gen_tokens",
                                                                             "context_budget", "nudge_every",
                                                                             "attempt_views")}
    tk = {"reasoning_effort": sc.get("reasoning_effort", "xhigh"), "preserve_thinking": True}
    if opts.get("keep_reasoning"):
        try:
            key, res = run_probe.think_check(base, "qwen", tk)
        except Exception as e:                       # noqa: BLE001
            key, res = None, {"error": f"{type(e).__name__}: {e}"[:300]}
        rep["think_check"] = dict(res, key=key)
        print("THINK CHECK", json.dumps(rep["think_check"]), flush=True)
        if key is None:                              # earlier reasoning would not reach the model: do not send it
            opts["keep_reasoning"] = opts["keep_cut_reasoning"] = False
        else:
            opts["reasoning_key"] = key
    if not rep["tool_check"].get("ok"):
        return None
    kv = (out.get("rungs") or [{}])[-1].get("kv_tokens") or 0
    conc = min(rung.get("max_seqs", 4), sc.get("max_concurrency", 8),
               max(sc.get("min_concurrency", 2), kv // sc.get("kv_per_agent", 24000)) if kv else rung.get("max_seqs", 4))
    client = ChatClient(base, "qwen", max_tokens=opts.get("turn_max_tokens", 12288),
                        temperature=sc.get("temperature", 0.6), top_p=sc.get("top_p", 0.95), top_k=sc.get("top_k", 20),
                        thinking=True, template_kwargs=tk, timeout=3600)
    return opts, conc, client


class TrackedClient:
    """The agent's client, counting requests in flight and when one last returned (the watchdog's inputs). While the
    watchdog restarts the server no request goes out; a request that fails without an HTTP answer (connection refused /
    reset, timeout) waits up to grace_s for a restart to begin, then for it to end, and is sent again on the new server
    -- so the tasks running when the server died keep their conversations, and the queue does not burn through
    instant failures. Nothing waits past the agent deadline."""

    def __init__(self, client, deadline=None, grace_s=60, retries=2):
        self.client, self.deadline, self.grace_s, self.retries = client, deadline or float("inf"), grace_s, retries
        self.cond, self.inflight, self.last_return = threading.Condition(), 0, time.time()
        self.gen, self.restarting, self.dead, self.resent = 0, False, False, 0

    def _left(self):
        return max(0.0, min(self.deadline + 60 - time.time(), 1e6))

    def chat(self, *a, **k):
        r = None
        for _ in range(self.retries + 1):
            with self.cond:
                self.cond.wait_for(lambda: not self.restarting or self.dead, timeout=self._left())
                if self.restarting and not self.dead:
                    return {"error": "server restart outlived the deadline", "context_overflow": False}
                gen = self.gen
                self.inflight += 1
            if k.get("timeout") and self.deadline < float("inf"):
                k["timeout"] = max(60.0, min(k["timeout"], self.deadline - time.time() + 60.0))
            try:
                r = self.client.chat(*a, **k)
            finally:
                with self.cond:
                    self.inflight -= 1
                    self.last_return = time.time()
            err = isinstance(r, dict) and r.get("error")
            if not err or str(err).startswith("HTTP") or self.dead:
                return r
            with self.cond:
                self.cond.wait_for(lambda: self.gen != gen or self.restarting or self.dead,
                                   timeout=min(self.grace_s, self._left()))
                self.cond.wait_for(lambda: not self.restarting or self.dead, timeout=self._left())
                if self.gen == gen or self.dead:
                    return r
                self.resent += 1
        return r

    def __getattr__(self, name):
        return getattr(self.client, name)


class Watchdog:
    """X62 (10-26): Flash-Next served the agent for 1.7 h, then a request errored and the scheduler went SILENT for
    9.3 h until the deadline. Every poll_s: if the server process has exited, or requests are open and neither the
    server log has grown nor a request returned for hang_s, call restart(rung_name) -> (proc, name, log_path) or None
    (it stops the old server, starts a rung, runs the serve check). At most max_restarts, and none with less than
    min_left_s to the deadline (a restart takes ~15-25 min); after that the client stops waiting (dead)."""

    def __init__(self, client, proc, log_path, restart, hang_s=600, poll_s=15, max_restarts=3, restart_rung=None,
                 deadline=None, min_left_s=1800):
        self.client, self.proc, self.log_path, self.restart = client, proc, log_path, restart
        self.hang_s, self.poll_s, self.max_restarts, self.restart_rung = hang_s, poll_s, max_restarts, restart_rung
        self.deadline, self.min_left_s = deadline or float("inf"), min_left_s
        self.events, self.stop_ev, self.thread = [], threading.Event(), None
        self.last_size, self.last_growth = self._size(), time.time()

    def _size(self):
        try:
            return os.path.getsize(self.log_path) if self.log_path else 0
        except OSError:
            return 0

    def check(self, now=None):
        """The reason to restart now, or None."""
        now = now or time.time()
        size = self._size()
        if size > self.last_size:
            self.last_size, self.last_growth = size, now
        if self.proc is not None and getattr(self.proc, "poll", lambda: None)() is not None:
            return "server process exited"
        quiet = now - max(self.last_growth, self.client.last_return)
        if self.client.inflight > 0 and quiet > self.hang_s:
            return f"no server progress for {quiet:.0f} s with {self.client.inflight} requests open"
        return None

    def _set(self, **kw):
        with self.client.cond:
            for k, v in kw.items():
                setattr(self.client, k, v)
            self.client.cond.notify_all()

    def step(self):
        reason = self.check()
        if not reason or self.client.dead:
            return None
        if len(self.events) >= self.max_restarts or self.deadline - time.time() < self.min_left_s:
            why = "max restarts reached" if len(self.events) >= self.max_restarts else "too close to the deadline"
            self.events.append({"reason": reason, "gave_up": why})
            print("WATCHDOG gave up:", reason, "|", why, flush=True)
            self._set(dead=True)
            return self.events[-1]
        t = time.time()
        print(f"WATCHDOG restart: {reason}", flush=True)
        self._set(restarting=True)
        try:
            res = self.restart(self.restart_rung)
        except Exception as e:                       # noqa: BLE001
            res = None
            reason += f" | restart raised {type(e).__name__}: {e}"[:300]
        ev = {"reason": reason, "s": round(time.time() - t), "ok": res is not None}
        if res is not None:
            self.proc, ev["rung"], self.log_path = res
        self.events.append(ev)
        self.last_size, self.last_growth = self._size(), time.time()
        self.client.last_return = time.time()
        self._set(restarting=False, gen=self.client.gen + (res is not None))
        print("WATCHDOG", json.dumps(ev), flush=True)
        return ev

    def _loop(self):
        while not self.stop_ev.wait(self.poll_s):
            self.step()

    def start(self):
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_ev.set()
        self._set(dead=True)


def run(cfg, out, save, base, rung, server, restart=None, log_path=None):
    from repl_agent import solve_repl
    sc = cfg["submission"]
    work = Path(cfg.get("work_dir", "/kaggle/working"))
    ch = json.load(open(sc["challenges"]))
    sol = json.load(open(sc["solutions"])) if sc.get("solutions") else None
    sub_path = Path(sc["out_path"])
    sub = json.load(open(sub_path)) if sub_path.exists() else fallback_sub(ch)
    for t in ch:                                     # every task present and well-formed from the start
        if t not in sub or len(sub[t]) != len(ch[t]["test"]):
            sub[t] = fallback_sub({t: ch[t]})[t]
    write_atomic(sub_path, sub)
    rep = {"rung": rung["name"], "kv_tokens": (out.get("rungs") or [{}])[-1].get("kv_tokens"), "tasks": len(ch)}
    out["submission_run"] = rep
    prep = agent_prep(base, sc, rep, rung, out)
    save()
    if prep is None:
        rep["error"] = "tool check failed: the agent cannot call python -- submission stays fallback + CPU"
        save()
        print("C23F REPORT", json.dumps(rep), flush=True)
        return out
    opts, conc, client = prep
    client = TrackedClient(client, sc["agent_deadline_epoch"], sc.get("resend_grace_s", 60))
    wd = (Watchdog(client, server, log_path, restart, sc.get("hang_s", 600), sc.get("watch_poll_s", 15),
                   sc.get("max_restarts", 3), sc.get("hang_restart_rung"), sc["agent_deadline_epoch"],
                   sc.get("restart_min_left_s", 1800)).start()
          if restart and sc.get("hang_s", 600) else None)
    kv = rep["kv_tokens"] or 0
    rep["concurrency"] = conc
    order = sorted(ch, key=lambda t: len(json.dumps(ch[t])))           # X51: shortest task first
    lock = threading.Lock()
    recs = defaultdict(list)
    att_log = work / sc.get("log_name", "flash_attempts.jsonl")
    trace_dir = work / sc["trace_subdir"] if sc.get("trace_subdir") else None
    if trace_dir:
        trace_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    stats = {"done": 0, "verified": 0, "gen_tokens": 0}

    def on_done(rec):
        tid = rec["task_id"]
        with lock:
            recs[tid].append(rec)
            apply(sub, tid, ch[tid], candidates(ch[tid], recs[tid], sc.get("min_ok", 1)))
            write_atomic(sub_path, sub)
            stats["done"] += 1
            stats["verified"] += rec["status"] == "verified"
            stats["gen_tokens"] += rec.get("gen_tokens", 0)
            slim = {k: v for k, v in rec.items() if k != "transcript"}
            if sol is not None:
                slim["test_correct"] = [p["test"] == sol[tid] for p in rec.get("programs") or []][-1:]
            with open(att_log, "a") as f:
                f.write(json.dumps(slim) + "\n")
            if trace_dir and rec.get("turns"):           # an attempt stopped before its first turn has nothing to read
                with gzip.open(trace_dir / f"{tid}_{rec.get('idx', 0)}.json.gz", "wt", encoding="utf-8") as g:
                    json.dump(rec, g)
        el = time.time() - t0
        print(f"[{el:6.0f}s] {tid}#{rec.get('idx')}{'@' + rec['view'] if rec.get('view') else ''} {rec['status']:8s} tok {rec.get('gen_tokens', 0):6d} | done "
              f"{stats['done']} verified {stats['verified']} | {stats['gen_tokens'] / max(el, 1):.0f} tok/s", flush=True)

    deadline = sc["agent_deadline_epoch"]
    rep["agent_s_available"] = round(deadline - time.time())
    print(f"AGENT: {len(order)} tasks, {conc} at once (KV {kv:,}), {sc.get('attempts', 2)} attempts, "
          f"{rep['agent_s_available'] / 3600:.2f} h, opts {json.dumps(opts)}", flush=True)
    save()
    try:
        _, report = solve_repl({t: ch[t] for t in order}, client, attempts=sc.get("attempts", 2), concurrency=conc,
                               deadline=deadline, stop_after=1, log=lambda m: None, on_done=on_done, **opts)
        atts = report["attempts"]
    except Exception as e:                           # the submission file is already as good as the work done
        rep["error"] = f"{type(e).__name__}: {e}"[-1500:]
        atts = [r for v in recs.values() for r in v]
    if wd is not None:
        wd.stop()
        rep["restarts"], rep["requests_resent"] = wd.events, client.resent
    started = {a["task_id"] for a in atts if a.get("turns")}
    rep.update({"agent_wall_s": round(time.time() - t0), "attempts_started": sum(1 for a in atts if a.get("turns")),
                "tasks_started": len(started), "verified_tasks": sum(1 for t in ch if any(
                    r["status"] == "verified" for r in recs.get(t, []))),
                "gen_tokens": stats["gen_tokens"], "gen_tok_s": round(stats["gen_tokens"] / max(time.time() - t0, 1), 1),
                "statuses": {s: sum(1 for a in atts if a.get("status") == s) for s in {a.get("status") for a in atts}},
                "name_errors": sum(a.get("name_errors", 0) for a in atts),
                "session_restarts": sum(a.get("session_restarts", 0) for a in atts)})
    if sol is not None:
        ids = sorted(ch)
        rep["score_agent"] = round(score(sub, sol, ids), 2)
        rep["score_started"] = round(score(sub, sol, sorted(started)), 2)
        rep["verified_wrong"] = sorted(t for t in ch if any(r["status"] == "verified" for r in recs.get(t, []))
                                       and score(sub, sol, [t]) < 1)
        # X79: drop every program submitted at or after a force / rescue order -> reh2's harness in the same run
        sub0 = fallback_sub(ch)
        for t in ch:
            r0 = [dict(r, programs=[p for p in r.get("programs") or []
                                    if not r.get("force_turn") or p.get("turn", 0) < r["force_turn"]])
                  for r in recs.get(t, [])]
            apply(sub0, t, ch[t], candidates(ch[t], r0, sc.get("min_ok", 1)))
        rep["score_agent_noforce"] = round(score(sub0, sol, ids), 2)
        fa = [r for v in recs.values() for r in v if r.get("force_turn")]
        after = lambda r: [p for p in r.get("programs") or [] if p.get("turn", 0) >= r["force_turn"]]
        rep["force"] = {
            "ordered": len(fa), "rescue_turns": sum(1 for r in fa if r.get("rescued")),
            "submitted_after": sum(1 for r in fa if after(r)),
            "verified_after": sorted(f"{r['task_id']}#{r['idx']}" for r in fa if r["status"] == "verified"),
            "best_ok_after": sorted((max(p["n_ok"] for p in after(r)), r["n_train"]) for r in fa if after(r)),
            "tasks_fixed": sorted(t for t in ch if score(sub, sol, [t]) > score(sub0, sol, [t])),
            "tasks_lost": sorted(t for t in ch if score(sub, sol, [t]) < score(sub0, sol, [t])),
            "gain": round(rep["score_agent"] - rep["score_agent_noforce"], 2)}
    save()
    print("C23F REPORT", json.dumps(rep), flush=True)
    return out

In [ ]:
%%writefile /kaggle/working/agent/prune_moe.py
"""Prune Qwen3.8-Flash-Next's routed experts from router statistics (teammate arunodhayan's X59 prune_moe.py, 2026-10-26,
which served correctly after pruning 512 -> 384; adapted 2026-10-28 for c23fp).

    python prune_moe.py '<json config>'

Config: model_glob; counts_json (X72's flash_calib_counts.json: {"counts": [[...512] x layers]}) OR files (raw SGLang
expert-distribution recorder dumps); keep (experts kept per layer, a multiple of 4 for expert parallelism 4); drop_mtp
(the checkpoint's own MTP weights -- c23fp's drafter is albucino's separate model); out_roots; report.

Per MoE layer the `keep` experts with the most router picks are kept (frequency pruning), renumbered 0..keep-1 in their
original order; every [num_experts, ...] tensor of that layer's router is sliced to the same rows; config num_experts =
keep. Shards with no expert / router / MTP tensor (the 51B n-gram table) are symlinked, not copied.

Changed from X59: counts can come from X72's summed counts (X59 used the recorder's last 1,000 forward passes only, ~4k
tokens: EXPERIMENTS 10-28); recorder dumps load with weights_only=True (tensors only, no pickled code); the report says
which counts were used and how many tokens they represent.
"""
import json
import os
import re
import shutil
import sys
import time
from collections import defaultdict
from pathlib import Path

EXPERT = re.compile(r"^(.*\.layers\.(\d+)\.mlp\.experts\.)(\d+)(\..+)$")
MLP = re.compile(r"\.layers\.(\d+)\.mlp\.")


def find_num_experts(conf):
    if isinstance(conf, dict):
        if isinstance(conf.get("num_experts"), int):
            return conf["num_experts"]
        for v in conf.values():
            n = find_num_experts(v)
            if n:
                return n
    return None


def set_num_experts(conf, old, new):
    if isinstance(conf, dict):
        for k, v in conf.items():
            if k == "num_experts" and v == old:
                conf[k] = new
            else:
                set_num_experts(v, old, new)


def _arrays(obj, n_exp):
    """Every (rows x n_exp) array inside a recorder dump (dicts / lists / tensors)."""
    import numpy as np
    out = []
    if isinstance(obj, dict):
        for v in obj.values():
            out += _arrays(v, n_exp)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            out += _arrays(v, n_exp)
    elif hasattr(obj, "shape") and len(obj.shape) >= 2 and obj.shape[-1] == n_exp:
        a = obj.detach().cpu().double().numpy() if hasattr(obj, "detach") else np.asarray(obj, dtype=float)
        out.append(a.reshape(-1, a.shape[-2], n_exp).sum(0))
    return out


def load_counts(files, n_exp):
    """Sum of the (layers x experts) count arrays in recorder dumps; the most common shape wins."""
    import torch
    found = []
    for f in files:
        try:
            obj = torch.load(f, map_location="cpu", weights_only=True)
        except Exception:
            continue
        found += _arrays(obj, n_exp)
    if not found:
        return None
    shapes = defaultdict(list)
    for a in found:
        shapes[a.shape].append(a)
    best = max(shapes.values(), key=lambda v: (len(v), sum(x.sum() for x in v)))
    return sum(best)


def load_counts_json(path, n_exp):
    import numpy as np
    d = json.load(open(path))
    a = np.asarray(d["counts"], dtype=float)
    if a.ndim != 2 or a.shape[1] != n_exp:
        raise RuntimeError(f"counts in {path} have shape {a.shape}, expected (layers, {n_exp})")
    return a, {k: d.get(k) for k in ("tokens", "dumps", "full_dumps", "source")}


def choose_keep(counts, layers, k):
    """{layer: sorted kept expert ids}, top-k by count (ties: lower id), plus the routed share each layer keeps."""
    keep, share = {}, {}
    for row, L in zip(counts, layers):
        order = sorted(range(len(row)), key=lambda e: (-row[e], e))
        keep[L] = sorted(order[:k])
        tot = float(row.sum())
        share[L] = float(row[keep[L]].sum()) / tot if tot else 0.0
    return keep, share


def prune(src, dst, keep, n_exp, drop_mtp=True, log=print):
    import torch
    from safetensors import safe_open
    from safetensors.torch import save_file
    src, dst = Path(src), Path(dst)
    idx = json.load(open(src / "model.safetensors.index.json"))
    wm = idx["weight_map"]
    by_file = defaultdict(list)
    for n, f in wm.items():
        by_file[f].append(n)
    remap = {L: {old: new for new, old in enumerate(ks)} for L, ks in keep.items()}
    rows = {L: torch.tensor(ks) for L, ks in keep.items()}
    stats, new_map = defaultdict(int), {}
    dst.mkdir(parents=True, exist_ok=True)
    for f, names in sorted(by_file.items()):
        touch = any(MLP.search(n) or n.startswith("mtp.") for n in names)
        if not touch:                                    # e.g. the n-gram table shards: link, never copy
            os.symlink(src / f, dst / f)
            for n in names:
                new_map[n] = f
            stats["linked_files"] += 1
            continue
        tensors = {}
        with safe_open(str(src / f), framework="pt") as sf:
            meta = sf.metadata()
            for n in names:
                if drop_mtp and n.startswith("mtp."):
                    stats["mtp_dropped"] += 1
                    continue
                m = EXPERT.match(n)
                if m:
                    L, e = int(m.group(2)), int(m.group(3))
                    if L in remap and e not in remap[L]:
                        stats["expert_tensors_dropped"] += 1
                        continue
                    new = f"{m.group(1)}{remap[L][e]}{m.group(4)}" if L in remap else n
                    tensors[new] = sf.get_tensor(n)
                    stats["expert_tensors_kept"] += 1
                    continue
                t = sf.get_tensor(n)
                mm = MLP.search(n)
                if mm and int(mm.group(1)) in rows and t.dim() >= 1 and t.shape[0] == n_exp:
                    if not t.is_floating_point():
                        raise RuntimeError(f"router tensor {n} is quantized ({t.dtype}); cannot slice rows")
                    t = t.index_select(0, rows[int(mm.group(1))])
                    stats["router_tensors_sliced"] += 1
                tensors[n] = t.contiguous()
        if not tensors:                                  # e.g. an MTP-only shard: nothing left to write
            stats["emptied_files"] += 1
            continue
        save_file(tensors, str(dst / f), metadata=meta or {"format": "pt"})
        for n in tensors:
            new_map[n] = f
        stats["rewritten_files"] += 1
        log(f"[prune] {f}: {len(tensors)} tensors written", flush=True)
    total = sum(os.path.getsize(dst / f) for f in set(new_map.values()))
    json.dump({"metadata": dict(idx.get("metadata", {}), total_size=total), "weight_map": new_map},
              open(dst / "model.safetensors.index.json", "w"), indent=1)
    return dict(stats)


def main(cfg):
    report = {"ok": False, "keep": cfg.get("keep")}
    t0 = time.time()
    try:
        import glob
        model = Path(sorted(glob.glob(cfg["model_glob"], recursive=True))[0]).parent
        conf = json.load(open(model / "config.json"))
        n_exp = find_num_experts(conf)
        k = int(cfg["keep"])
        assert n_exp and 0 < k < n_exp, (n_exp, k)
        if cfg.get("counts_json"):
            hits = sorted(glob.glob(cfg["counts_json"], recursive=True))
            if not hits:
                raise RuntimeError(f"no calibration counts at {cfg['counts_json']}")
            counts, meta = load_counts_json(hits[0], n_exp)
            report["counts"] = dict(meta, path=hits[0])
        else:
            counts = load_counts(cfg.get("files", []), n_exp)
            report["counts"] = {"source": "recorder dumps", "files": len(cfg.get("files", []))}
        if counts is None:
            raise RuntimeError(f"no (layers x {n_exp}) counts in {cfg.get('files')}")
        idx = json.load(open(model / "model.safetensors.index.json"))
        layers = sorted({int(m.group(2)) for n in idx["weight_map"] if (m := EXPERT.match(n))})
        if counts.shape[0] < len(layers):
            raise RuntimeError(f"counts have {counts.shape[0]} rows for {len(layers)} MoE layers")
        counts = counts[:len(layers)]
        report.update(n_experts=n_exp, moe_layers=len(layers), routed_total=float(counts.sum()),
                      unused_experts_per_layer_mean=float((counts == 0).sum(1).mean()))
        if counts.sum() <= 0:
            raise RuntimeError("all counts are zero: the recorder saw no traffic")
        keep, share = choose_keep(counts, layers, k)
        report["kept_routing_share"] = {"mean": round(sum(share.values()) / len(share), 4),
                                        "min": round(min(share.values()), 4)}
        files = sorted(set(idx["weight_map"].values()))
        touched = {f for n, f in idx["weight_map"].items() if MLP.search(n) or n.startswith("mtp.")}
        need = sum(os.path.getsize(model / f) for f in touched) * (k / n_exp + 0.12) + 5e9
        root = None
        for r in cfg.get("out_roots", ["/tmp"]):
            try:
                Path(r).mkdir(parents=True, exist_ok=True)
                if shutil.disk_usage(r).free > need:
                    root = r
                    break
            except Exception:
                continue
        report["disk_needed_gb"] = round(need / 1e9, 1)
        report["disk_free_gb"] = {r: round(shutil.disk_usage(r).free / 1e9, 1)
                                  for r in cfg.get("out_roots", ["/tmp"]) if os.path.isdir(r)}
        if root is None:
            raise RuntimeError("no output location with enough free disk")
        dst = Path(root) / f"flash_pruned_{k}"
        shutil.rmtree(dst, ignore_errors=True)
        report["stats"] = prune(model, dst, keep, n_exp, cfg.get("drop_mtp", True))
        set_num_experts(conf, n_exp, k)
        json.dump(conf, open(dst / "config.json", "w"), indent=1)
        for p in model.iterdir():                         # tokenizer, chat template, generation config, ...
            if p.name not in ("config.json", "model.safetensors.index.json") and not p.name.endswith(".safetensors"):
                if not (dst / p.name).exists():
                    os.symlink(p, dst / p.name)
        json.dump({str(L): v for L, v in keep.items()}, open(dst / "kept_experts.json", "w"))
        report.update(ok=True, dst=str(dst), files=len(files), seconds=round(time.time() - t0))
    except Exception as e:
        report["reason"] = f"{type(e).__name__}: {e}"[:600]
    if cfg.get("report"):
        json.dump(report, open(cfg["report"], "w"), indent=1)
    print("PRUNE REPORT", json.dumps(report), flush=True)
    return report


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))

In [ ]:
%%writefile /kaggle/working/agent/augment.py
"""X70 (2026-10-28, user: "maybe more augmentations can help, similar to what nvarc does?"): show Qwen the task in
another orientation or colouring and map its answer back.

NVARC answers each task in 16 views (dihedral turns and flips x colour permutations) and votes. The trace review found
Qwen repeats the SAME wrong reading across draws (dbff022c: wrong in 12 of 14 finished answers, the colour key read in
the wrong direction). A view changes what the model sees -- a vertical key becomes horizontal -- so it tests whether
such misreadings are tied to the presentation. Each view is a bijection on grids; `back` undoes it on an answer.

  transpose  rows <-> columns (its own inverse)
  colour     a fixed permutation of colours 1-9 per task (0, the usual background, stays 0)
  identity   unchanged (a same-session control)
"""
import random

VIEWS = ("identity", "transpose", "colour")


def _transpose(g):
    return [list(r) for r in zip(*g)]


def colour_map(task_id):
    """Deterministic per task: a shuffle of 1-9 with no fixed point, 0 -> 0."""
    rng = random.Random(f"x70-colour-{task_id}")
    while True:
        perm = list(range(1, 10))
        rng.shuffle(perm)
        if all(a != b for a, b in zip(range(1, 10), perm)):
            return {0: 0, **dict(zip(range(1, 10), perm))}


def forward(view, task_id):
    if view == "identity":
        return lambda g: [list(r) for r in g]
    if view == "transpose":
        return _transpose
    if view == "colour":
        m = colour_map(task_id)
        return lambda g: [[m[v] for v in r] for r in g]
    raise ValueError(view)


def back(view, task_id):
    if view == "colour":
        inv = {v: k for k, v in colour_map(task_id).items()}
        return lambda g: [[inv[v] for v in r] for r in g]
    return forward(view, task_id)                        # identity and transpose are their own inverse


def report(recs, ids, views, sol, groups):
    """Per view: solved per task group; paired against the identity view (fixed = identity wrong / view right, broken
    = the reverse); the union of views; and a majority vote (a grid >= 2 views agree on, else identity's)."""
    # 10-31: a resumed run asks only the unfinished tasks but keeps the full groups (KeyError 'd35bdbdc' in X70 run 3)
    groups = {g: [t for t in ts if t in ids] for g, ts in groups.items()}

    def grids(t, v):
        r = next((r for r in recs if r["task_id"] == t and r.get("view") == v), None)
        return r.get("grids") if r else None

    def score(t, gs):
        return sum(g == s for g, s in zip(gs, sol[t])) / len(sol[t]) if gs else 0.0

    per = {v: {t: score(t, grids(t, v)) for t in ids} for v in views}
    out = {"solved": {v: {g: round(sum(per[v][t] for t in ts), 2) for g, ts in groups.items()} for v in views}}
    if "identity" in views:
        out["vs_identity"] = {v: {"fixed": sorted(t for t in ids if per[v][t] > per["identity"][t]),
                                  "broken": sorted(t for t in ids if per[v][t] < per["identity"][t])}
                              for v in views if v != "identity"}
    union, vote = {}, {}
    for t in ids:
        union[t] = max(per[v][t] for v in views)
        n = len(sol[t])
        voted = []
        for j in range(n):
            cands = [gs[j] for gs in (grids(t, v) for v in views) if gs and j < len(gs)]
            best = next((g for g in cands if sum(g == h for h in cands) >= 2), None)
            if best is None:
                ident = grids(t, "identity")
                best = ident[j] if ident and j < len(ident) else (cands[0] if cands else None)
            voted.append(best)
        vote[t] = sum(g == s for g, s in zip(voted, sol[t])) / n
    out["union"] = {g: round(sum(union[t] for t in ts), 2) for g, ts in groups.items()}
    out["vote"] = {g: round(sum(vote[t] for t in ts), 2) for g, ts in groups.items()}
    out["answered"] = {v: sum(1 for t in ids if grids(t, v)) for v in views}
    return out


def task_view(task, view, task_id):
    """The whole task (every example input and output, every test input) in `view`."""
    f = forward(view, task_id)
    return {"train": [{"input": f(p["input"]), "output": f(p["output"])} for p in task["train"]],
            "test": [{"input": f(t["input"])} for t in task["test"]]}

In [ ]:
# c23fp: prune Flash-Next's experts in the background (agent/prune_moe.py, ~11 min, to /tmp) from X72's router counts
# while the CPU solver runs. The run cell serves the pruned model if this worked, else the full model (= c23f).
import glob, json, os, subprocess, sys
PRUNE_PROC = None
if RUN and FLX.get("prune"):
    _p = FLX["prune"]
    _hits = sorted(glob.glob(_p["counts_glob"], recursive=True))
    if _hits:
        _pc = {"model_glob": FLX["flash"]["model_glob"], "counts_json": _hits[0], "keep": _p["keep"], "drop_mtp": True,
               "out_roots": _p["out_roots"], "report": "/kaggle/working/prune_report.json"}
        PRUNE_PROC = subprocess.Popen([sys.executable, "/kaggle/working/agent/prune_moe.py", json.dumps(_pc)],
                                      stdout=open("/kaggle/working/prune.out", "w"), stderr=subprocess.STDOUT,
                                      cwd="/kaggle/working/agent", start_new_session=True)
        print("prune started from", _hits[0], "keep", _p["keep"])
    else:
        print("no calibration counts attached -> the full model (c23f)")

In [ ]:
# c23fp: the pruned model (more agents at once) if the prune worked; if it did not, or no pruned rung served and >= 3 h
# remain, the full model with c23f's rungs. Each flash_smoke run: server ladder -> agent/flash_sub.py.
import json, os, subprocess, sys, time
def _flash(over, out_name, max_conc, restart_rung):
    cfg = dict(FLX["flash"], site="/kaggle/working/vllm-site-packages", deadline_epoch=AGENT_DEADLINE,
               out="/kaggle/working/" + out_name,
               cpu_wait={"pid_file": "/kaggle/working/cpu_solver.pid", "until_epoch": START + FLX["cpu_alone_min"] * 60},
               submission={"challenges": CHALLENGES, "solutions": SOLUTIONS, "out_path": "/kaggle/working/submission.json",
                           "agent": FLX["agent"], "agent_deadline_epoch": AGENT_DEADLINE,
                           "trace_subdir": None if RERUN else "agent_traces", "max_concurrency": max_conc,
                           "hang_restart_rung": restart_rung,
                           **{k: FLX[k] for k in ("attempts", "temperature", "top_p", "top_k", "reasoning_effort",
                                                  "kv_per_agent", "min_concurrency", "min_ok", "hang_s", "watch_poll_s",
                                                  "max_restarts")}}, **over)
    try:
        rc = subprocess.run([sys.executable, "/kaggle/working/agent/flash_smoke.py", json.dumps(cfg)],
                            cwd="/kaggle/working/agent", timeout=max(600, AGENT_DEADLINE + FLX["agent_kill_after_s"] - time.time())).returncode
    except subprocess.TimeoutExpired:
        rc = "timeout"
    try:
        res = json.load(open("/kaggle/working/" + out_name))
    except Exception:
        res = {}
    return rc, "submission_run" in res
if RUN:
    _pr = {"ok": False, "reason": "no prune started"}
    if PRUNE_PROC is not None:
        try:
            PRUNE_PROC.wait(timeout=max(60, START + FLX["prune"]["wait_min"] * 60 - time.time()))
        except subprocess.TimeoutExpired:
            PRUNE_PROC.kill()
        try:
            _pr = json.load(open("/kaggle/working/prune_report.json"))
        except Exception as e:
            _pr = {"ok": False, "reason": repr(e)}
    TIMING["prune"] = {k: v for k, v in _pr.items() if k != "stats"}
    print("PRUNE", json.dumps(TIMING["prune"]))
    _ran = False
    if _pr.get("ok"):
        rc, _ran = _flash(dict(FLX["prune"].get("flash_over") or {}, model_glob=_pr["dst"] + "/config.json",
                               rungs=FLX["pruned_rungs"]),
                          "flash_smoke_pruned.json", FLX["prune"]["max_concurrency"], FLX["prune"]["hang_restart_rung"])
        TIMING["flash_rc_pruned"] = rc
    if not _ran and AGENT_DEADLINE - time.time() > FLX["prune"]["fallback_min_h"] * 3600:
        rc, _ran = _flash({}, "flash_smoke.json", FLX["max_concurrency"], FLX["hang_restart_rung"])
        TIMING["flash_rc_full"] = rc
    TIMING["agent_ran"] = _ran
    TIMING["agent_end"] = round(time.time() - START)
    print("agent ran:", _ran, "at +", round((time.time() - START) / 3600, 2), "h")

In [ ]:
# Merge the CPU solver's verified answers (cpu_solver/cpu_merge.py: fallback slots first, then attempt_2; atomic).
import json, os, signal, sys
TIMING["cpu"] = {}
if RUN and FLX.get("cpu"):
    try:
        _log = [json.loads(l) for l in open("/kaggle/working/cpu_log.jsonl")] if os.path.exists("/kaggle/working/cpu_log.jsonl") else []
        TIMING["cpu"]["tasks_done"] = sum(1 for r in _log if "task" in r)
        if os.path.exists("/kaggle/working/cpu_answers.json"):
            sys.path.insert(0, "/kaggle/working/cpu_solver")
            import cpu_merge
            TIMING["cpu"].update(cpu_merge.merge_files("/kaggle/working/submission.json",
                                                       "/kaggle/working/cpu_answers.json", CHALLENGES, None))
    except Exception as e:
        print("CPU merge failed (submission unchanged):", repr(e))
    if CPU_PROC is not None:
        try:
            os.killpg(CPU_PROC.pid, signal.SIGKILL)
        except Exception:
            pass
    print("CPU solver:", json.dumps(TIMING["cpu"]))

In [ ]:
# Stop the server; validate the submission (bad rows -> fallback); the rehearsal's score; timing.
import json, os, glob, signal, subprocess, time
from pathlib import Path
pid = Path("/kaggle/working/vllm-server.pid")
if pid.exists():
    try:
        os.killpg(int(pid.read_text()), signal.SIGKILL)
    except Exception as e:
        print("kill:", e)
try:                       # a watchdog restart still loading at the agent deadline has no pid file yet
    subprocess.run(["pkill", "-9", "-f", "sglang"], capture_output=True, timeout=30)
except Exception as e:
    print("pkill:", e)
ch = json.load(open(CHALLENGES))
sub = json.load(open("/kaggle/working/submission.json"))
def ok_grid(g):
    return (isinstance(g, list) and g and all(isinstance(r, list) and r and len(r) == len(g[0]) for r in g)
            and all(isinstance(v, int) and 0 <= v <= 9 for r in g for v in r))
bad = [t for t, task in ch.items() if t not in sub or len(sub[t]) != len(task["test"])
       or not all(ok_grid(r.get("attempt_1")) and ok_grid(r.get("attempt_2")) for r in sub[t])]
if bad:
    print("INVALID rows for", len(bad), "tasks -> fallback for those")
    for t in bad:
        sub[t] = [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in ch[t]["test"]]
sub = {t: sub[t] for t in ch}
json.dump(sub, open("/kaggle/working/submission.json", "w"))
print(f"submission.json valid: {len(sub)} tasks")
if RUN and SOLUTIONS:
    sol = json.load(open(SOLUTIONS))
    sc = sum(sum(any(r[a] == s for a in ("attempt_1", "attempt_2")) for r, s in zip(sub[t], sol[t])) / len(sol[t])
             for t in ch)
    TIMING["rehearsal_score"] = round(sc, 2)
    print(f"REHEARSAL SCORE {sc:.2f} / {len(ch)} ({100 * sc / len(ch):.1f}%) -- agent + CPU solver")
TIMING["total_s"] = round(time.time() - START)
json.dump(TIMING, open("/kaggle/working/c23f_timing.json", "w"), indent=1)
print("C23F TIMING", json.dumps(TIMING))
subprocess.run("rm -rf /tmp/sgl-l4 /tmp/sgl-bundle", shell=True)